# PRAM: supplementary analyses (notebook 03)

This notebook has three parts that run in one kernel: **(1)** extra analyses of stage A (33-column full-bank test, 42-column single-bank 100-draw interval), stage B (importance weighting, CORAL, anchor regression) and extra computations (TabPFN sample row count back-calculation, 33-column frozen versus 42-column PRAM test, partner prevalence halved and doubled); **(2)** four cells for the within-hospital bootstrap of Supplementary S8 (5,000 repetitions); **(3)** thirteen cells for the supplementary computations (mixing-weight selection criterion N1, bank-difference interval N2, temperature record fraction N5, mixing-weight grid up to 1.00, federated analyses).

**How to run**: open this notebook in the working folder that notebooks 01 and 02 used (where `prepared_data` and `out_pram/` are visible) and run Kernel > Restart & Run All once. At the end the file `pram_results_notebook03.zip` is written to the working folder. If results of notebooks 01 and 02 are present they are read; the rest is recomputed with the same functions and keys.

- **Expected time and memory** (rough estimates): the supplementary computations alone take about 0.5 to 2 hours. If all earlier stages have to be recomputed from scratch, the time is much longer (about 8 to 15 hours for part 1 on its own). The expected peak memory is about 3 GiB or less (a simulated 200,000-patient neighbor search measured 2.56 GiB; the full bank is about 68,000 patients).
- **The cells form three parts**: (1) the cells of part 1 (cells 1 to 78), (2) the four S8 cells (80 to 83), (3) the thirteen supplementary cells (titled "supplementary cell n/13"). The constants of the supplementary cells are collected in one place above supplementary cell 1/13.
- **Thread setting**: the BLAS thread limit defaults to `N_THREADS = 0` (no limit). If a different value is set, the fingerprint of stored supplementary results changes, those units are recomputed, and they match stored results only within the tolerance below.
- **Reproduction judgment**: the metric tolerance of the reproduction check is 1e-6 absolute. A difference up to 1e-12 is shown as within 1e-12, a difference over 1e-12 and up to 1e-6 as within thread-order noise (the value is read as it is), and a difference over 1e-6 as not read. The per-fold alpha lists must be equal without tolerance. Even if the reproduction check mismatches, `R7_repro_diff_summary.csv` is kept in the zip (counts of cells, maximum difference and median, without hospital ids; for the full-bank rows also the number of queries over 1e-6).
- **Count cells of the N5 table**: if any cell whose value is 1 to 4, or any sum or difference piece whose value is 1 to 4, exists, all six count cells are masked (a rule independent of values, so the masking position does not reveal values); otherwise the numbers are given as they are. Fractions are given at the first decimal place. Names and positions of the pieces involved are not written. For a table with masking, the gap line and the 20-point threshold judgment use the rounded fractions shown on the screen.
- **Last line of the screen**: the status line may print "partial" because the federated analyses A and B come back with the reproduction checks partly mismatched. This is expected. If the reproduction judgment line of the summary md is "within 1e-12" or "within thread-order noise", the hospital-level and full-bank parts are fine.
- **If interrupted**: if the kernel dies, running the same notebook again with Restart & Run All continues from where it was cut. Adding cells and running everything again does not recompute earlier steps.
- **If memory is insufficient**: do not change `R8_WORKING_MEMORY_MB = 1024` in supplementary cell 1/13 (it is part of the fingerprint of the supplementary units, so they would be recomputed from the start). For the interval parts there is no setting to reduce. Only if N5 has to be read again, lower `R6_N5_CHUNK, R6_N5_CK_EVERY = 500000, 40` to `200000, 100`.
- If the run stops with a red error line (in any part), save the `.ipynb` that holds the output of the cell that stopped. Do not delete `out_pram/`.
- No external API is called.


# PRAM: extra analyses. Stage A (33-column full-bank test and 42-column single-bank 100-draw interval), stage B (importance weighting, CORAL, anchor regression) and extra computations

| | Values computed | Definition |
|---|---|---|
| (a) 33-column full-bank test | the six 33-column q of Table 3 Panel A and the 33-column Whole pool P and q of Supplementary S5 Panel B | the same as for 42 columns. In the full bank (one result pooling the 5-fold test predictions), the paired DeLong (standard implementation) of retrieval (`pram_b1`) and frozen logistic (`b1_base`) for 6 units of outcome x target, with Benjamini-Hochberg q |
| (b) 42-column single-bank 100-draw interval | the 24 42-column cells of Interval below -0.01 of Supplementary S5 Panel B | the same as the 33-column rows. Bank sizes 500, 1,000, 2,000, 5,000; 100 draws per cell (5 folds x 20 per fold); one draw is one bank and one test fold of the hospital that received it; interval = difference +- 1.96 x standard DeLong standard error; fraction of draws whose whole interval is below -0.01 |
| same definition, supplement | Single banks of 500 that lost more than 0.01 for 33-column MIMIC-III and MIMIC-IV in Table 3 Panel A and the 33-column MIMIC rows of S5 Panel B | the same computation as (b) on the two 33-column MIMIC cohorts (the 33-column eICU is read from the stored results) |

## Definitions of the outputs

**(a) 33-column full-bank q**

1. Per cell, p is the paired DeLong p of one full-bank run (standard variance, two-sided). q is the Benjamini-Hochberg adjusted p over the six 33-column cells as one set. This is the same set definition as the 42-column q, which was one set of six cells per feature set, and it is the **main value**. The q adjusted over the twelve cells of both sets is also given as a **reference column**, because there are two standard ways of defining the set.
2. Every cell is reported with its q value; no cell is dropped.
3. A cell with q below 0.05 and a negative AUROC change is marked separately.
4. If the reproduction check in this notebook (below) mismatches, the values of (a) must not be used.

**(b) 42-column single-bank 100 draws**

1. Per cell, the number k of draws among 100 whose DeLong 95% interval upper bound is below -0.01, its fraction and the Clopper-Pearson 95% interval are given. The numbers of draws whose whole interval is above 0 and below 0 are given with the same definition. (For k = 0 in a cell the exact 95% upper bound is 3.6%.)
2. The 100 draws share 5 test folds (20 banks per fold) and are not independent. The fractions are descriptive values and the interval measures only the bank variation.
3. The 10 draws of the 42-column grid (one draw is a 5-fold pooled point, S5 Panel C) and the 100 draws here (single bank) are different units; they are not mixed in the same column.

A small q can reflect a large sample rather than a large effect, and k = 0 can reflect low power in small banks with wide intervals. The interval width (standard error) of the same cell and the fraction of point estimates below -0.01 (`share_d_below_m01`) are given together.

## What this part does

1. **Where to open it: in the working folder of notebooks 01 and 02.** The folder shows `prepared_data`, and the eICU patient table and MIMIC-IV tables that notebooks 01 and 02 found as hospital-level input are found at the same candidate paths (the candidate lists are `EICU_PATIENT_CANDIDATES` and `_M4DIRS` of the settings cell, the third cell from the top). If the eICU patient table is not found, the hospital level, the hospital anchor extension and E-2 are marked "no data", and the last cell counts these sections as not completed. If the MIMIC-IV tables are not found, AR-period is dropped and the B-3 state is partial (no AR-period). If results of notebooks 01 and 02 (`resume/` and `github_csv/`) are in this folder, they are read per unit (`[SKIP read]`); if they are absent or insufficient, they are recomputed in this notebook with the same function, the same draw key and the same settings constants. The notebook does not stop because earlier results are absent.
2. **Only names starting with `R5_` are newly written.** An attempt to write an existing resume stops with `R5Stop`, and the sha256 hashes of existing files are compared before the start and after the end. Recomputed values are written to the resume starting with `R5_p_`.
3. **What was read and what was computed is printed as counts** (input collection table, also in the summary md). Read or recomputed, the values are compared with aggregate constants taken from the csv files of notebooks 01 and 02 (constants in the cell; the source csv name and sha256 are in the comments).
4. **Finished units are not rerun.** When run again in the same folder, finished units are skipped. The new computations are (a) 6 units of 33 columns and (b) 24 units of 42 columns (and 16 units of 33-column MIMIC).
5. A reproduction check tests whether the same computation gives the stored values again (2 full-bank 42-column units and 2 33-column eICU tail units are rerun).
6. At the end, `out_pram_part1.zip` is made in the current working folder. If `prepared_data` was not found and nothing was computed, no zip is made and the run ends with "not computed because the input is missing". **After the run, do not delete or move the `out_pram` folder** (it is read again when continuing; the zip does not contain the resume).

**Not needed**: the `resume/` and `github_csv/` of notebooks 01 and 02 (read if present, otherwise recomputed), and the closed-testing results. **Needed**: the same data `prepared_data` read by notebooks 01 and 02, the eICU patient table for the hospital level and partner scenarios, and the two MIMIC-IV tables `patients` and `icustays` for the development period anchor (if absent, only that version is dropped and goes into the sections not completed). TabPFN is not rerun (if the `a9_tabpfn` resume is absent, the sample row count is back-calculated from the aggregate constants).

**First thing to look at within a minute of starting**: running the settings cell (the third cell from the top) prints the input check table (`| input | path | status |`). If in that table the two eICU patient table lines and the two MIMIC-IV lines (patients, icustays) show the path "not found" or the status "column absent" or "absent", that input was not found at this location. Left as it is, this would be noticed only in the last cell 8 to 15 hours later, so stop the run, move the notebook to the location where notebooks 01 and 02 were run and open it again. If the top six lines (files inside `prepared_data`) are absent, all later computation cells are skipped. The absence of the `tabpfn package` and `GPU (CUDA)` lines is normal (this part does not rerun TabPFN).

**Run time (a folder where `out_pram` is empty, so no stored results exist): about 8 to 15 hours in total** (rough estimate; up to about 20 hours if one fit call exceeds 3 seconds). It is the sum of 6 to 11 hours excluding input collection, 1.5 to 3 hours of input collection and about 0.2 hours for the two AR-period rules. The pre-check cells measure on this machine **within 5 minutes of the start** and print the same total again; that value takes priority over these rough estimates.

**Total expected time.** The pre-check cells of A, B, input collection and the extras print the expected minutes of A plus input collection, B and the extra computations, and the total, within 5 minutes of the start (values counted from the measured neighbor search speed and fit time of this machine; rough estimates). TabPFN is not rerun. This part has no A9 cell.

**In a folder without stored results the time increases.** The estimates above assume that stored results exist (input collection is then only reading). If this folder has none of them, input collection recomputes with the same function 30 rescoring units, 60 grid units, 8 33-column eICU tail units, 252 hospital-level hospitals and 252 network factor-1 hospitals, about 1.5 to 3 hours in total (rough estimate). If only part of the results exist, existing units are read and only the missing units are computed. The definitions, the number of draws and the seeds are not changed to save time.

**To leave out the hospital anchor extension.** It is a reference upper bound that does not enter the pattern and the slowest part of B-3. Setting `RUN_B_ANCHOR_HOSP = True` to `False` in the B settings cell leaves out only this part (the two-environment version, the period version, PRAM on top and the pattern still run); setting it back to `True` and running again from the start in the same folder computes only this part. The default is on, and running everything in one go is fine.

**If interrupted (the kernel died or was closed).** Restart the kernel in the same folder and **run again from the start to the end**. Finished units are skipped (`[SKIP]`) and only the rest is computed. Do not delete the `out_pram` folder (resume). Do not change the settings.

**If stopped (red error line, `R5Stop`, status stop or reproduction mismatch).** Save the notebook file (.ipynb) that holds the screen output (the error line and the preceding status table). The summary md is written only in the last cell, so if the run stops midway it is not made. The `R5_` files made in `out_pram/github_csv` until then are not deleted.

**If run again after everything finished**, the hospital-level PRAM recomputation (3 folds per hospital) runs again (only that part has no resume; minutes to tens of minutes). The rest is skipped.


## Stage B. Domain adaptation family and anchor regression

Stage B compares three further methods with PRAM in the same cells, with the same metrics: importance weighting (B-1), CORAL (B-2) and anchor regression (B-3). Importance weighting has two versions: the **version without rescaling** and the **version with rescaling** (density ratio multiplied by the sample size ratio n_dev/n_bank), which is the main result. The three methods are switched on and off separately by `RUN_B_IW`, `RUN_B_CORAL` and `RUN_B_ANCHOR` (B settings cell). The eICU hospital extension inside anchor (AR-hosp, a reference upper bound, hospital level only) can be switched off separately by `RUN_B_ANCHOR_HOSP`: if it is off, the two-environment version, the development split period version and PRAM on top still run and the patterns of analyses P and F do not change (the hospital extension is the slowest part of B-3; the default is on). Each can be resumed and writes only to new names (resume starting with `R5_b_`).

**Limitations**

1. **The development environments of anchor regression are only two.** The environments of the two-environment version (AR-2db) are the MIMIC-IV training split and MIMIC-III. With a single anchor environment indicator, what gamma does is only to reduce the difference of the mean residuals of two environments. The cited reference ran 5 to 6 development environments and has no experiment that varies the number of environments. So the effect of anchor is also examined in the development split period version (AR-period, about 5 environments) and the eICU hospital anchor extension (AR-hosp, about 60 environments). AR-period has two environment-period rules, and both are given: strict (only stays that fall entirely inside one period; main rule) and majority (sensitivity). With two environments the main value of gamma in AR-2db resolves by construction into gamma = 1 (the pooled version).
2. **The truncation 0.05 to 20 of the importance weights has no literature basis.** It is a choice made here (it is not taken from Sugiyama 2007 or Bickel 2009). The two versions use the same truncation.
3. **Importance weighting without rescaling lacks the sample size ratio.** The shift discriminator is trained on about 18,700 development rows (0) and 500 to 5,000 bank rows (1), so q/(1-q) contains the prior ratio n_bank/n_dev. The estimate of the density ratio p_bank/p_dev is q/(1-q) x n_dev/n_bank (Bickel 2009 equations 12 to 17). The version without rescaling does not multiply by this ratio, so at banks of 500 and 1,000 the mean weight piles up near the lower bound 0.05 (0.05 to 0.085), the same ridge penalty becomes relatively stronger, and this can make a structure in which retraining comes out lower than frozen. The two versions are run in the same cells and separated. The weight summaries of the two versions (mean weight, fractions stuck at the lower and upper bounds, ESS) are stored for both in the cell information.
4. **The version with rescaling also does not normalize the weights to mean 1.** In cells where the mean of the truncated weights is below 1, the same ridge penalty (C = 1) is relatively strong for the weighted loss. To show the size of this effect, a diagnostic variant `iwn_erm` with the correction weights divided to mean 1 is given in the same cells (diagnostic only).
5. **CORAL has two versions.** Sun et al. (2016) assume that the features of both domains are normalized to mean 0 and match only the covariance. `coral` is the version without matching means, so it does not satisfy this premise; `coral_z` is the version re-standardized with the bank mean and standard deviation, which fits the premise. Only `coral_z` can be read as a limitation of the method.
6. **The anchor regression of this notebook is not the method of the cited reference as it is.** Three differences: (1) the gamma selection criterion (the reference uses the negative log likelihood of the held-out environment; this notebook gives two, `selected` by held-out AUROC and `selected_nll` by negative log likelihood); (2) the form for a binary outcome (the reference fits a probit-link likelihood anchor by boosting and uses the linear anchor only for continuous outcomes; the probit version here uses a linear predictor with the same loss, and the linear probability version is a use not in the reference); (3) the general penalty (the reference adds an elastic-net penalty; this notebook has none).

**Stage C (adversarial domain adaptation) is not in this notebook.** It would need torch and its computation is large. The stage B code uses only numpy, scipy and scikit-learn.

### Definitions of the stage B outputs

**Comparison.** Every new method is paired in the same cells (feature set F42 and F33 x outcome 2 x target 3 x bank size), between the same draws, with the frozen logistic, PRAM, logistic recalibration and local retraining (these four are read from the stored `matrix_F42` and `matrix_F33` resume if present, otherwise recomputed by the input-collection section with the same function and the same draw key). The banks of the new methods use the same split and the same draw key as the grid, so the same patients are drawn (the reproduction check below compares them). The metrics are AUROC, ECE, net benefit at the 0.20 threshold, calibration slope and intercept; the first three are used for the cell classification.

**Cell-level definitions** (X = new method, paired median between draws): D+ = delta AUROC (X - PRAM) > +0.01. DOM = delta AUROC (X - PRAM) >= -0.01 and median delta ECE (X - PRAM) < 0 and 90% or more of the draws with negative delta ECE (for the full bank with one draw only the sign) and delta NB20 (X - PRAM) >= +0.001. NULL = |delta AUROC (X - frozen)| <= 0.01 and |delta NB20| <= 0.001 and |delta ECE| <= 0.005. The AUROC and net benefit margins are post hoc definitions, and the ECE margin 0.005 is predefined but weakly based. As sensitivity, two further margin sets (0.005, 0.0005, 0.0025) and (0.02, 0.002, 0.01) are given. The cell classification is a distribution between bank draws and not a test.

**Cell pattern of analysis P** (per method X). The patterns P0 to P4 name where the cell counts fall and nothing else. For B-1 the three variants of the **version with rescaling** (iwc_erm, iwc_platt, iwc_temp) are the methods; the version without rescaling (iw_erm, iw_platt, iw_temp) is a comparison and does not enter the overall pattern. If the two versions split widely (6 or more cells with different decisions, or a pattern that differs under one of the three margin sets), both are shown. Counts use the 20 eICU cells and all 60 cells. P1 = eICU DOM 0, D+ 0 and NULL below 90% of all cells. P2 = eICU D+ 1 or more, or DOM 1 to 15. P3 = DOM 0 and D+ 0 and NULL at 90% or more of all cells. P4 = the same X is DOM in 16 or more of the 20 eICU cells and also in 60% or more of the hospitals for both outcomes at the hospital level. If none of these conditions applies, the result is marked P0 (no listed pattern). The conditions are checked in the order P4, P2, P0 (DOM above 15 cells), P3, P1. The gamma of anchor is chosen without target labels by two criteria (maximum held-out environment AUROC and minimum held-out environment negative log likelihood) and each is given in the same table; the post hoc optimum that defines a favorable gamma per metric is not used, and the whole grid is in the csv. If the pattern splits under the three margin sets (strict, base, loose), base is the main decision and the number of split cells is given with it (`R5_B_margin_split.csv`). The environment-version and hospital-version anchors that use labels (AR-2db and AR-hosp) are reference upper bounds and do not enter the pattern.

**Cell pattern of analysis F** (PRAM on the frozen model trained by anchor): G_X = median delta AUROC (pram_on_X - X). F1 = in 90% or more of all cells G_X >= -0.01, and in 50% or more the ECE decreased. F2 = one of the two is 50 to 90%. F3 = 90% or more cells with G_X >= -0.01, |delta ECE| <= 0.005 and |delta NB20| <= 0.001. F4 = G_X < -0.01 and ECE increased in 50% or more of the cells. If none applies, F0 (no listed pattern). The conditions are checked in the order F4, F3, then F1 or F2.

**Implementation checks.** They differ by version. (a) **Version without rescaling**: does this implementation have the same direction and size as the reference values: the eICU death IW-Platt slope within 0.40 to 0.55 (reference 0.440 to 0.485), and cells where IW-ERM AUROC is lower than frozen at 18/24 or more on the reference grid (bank 500, 1,000, 5,000, full; reference 22/24). If the result is outside, the version without rescaling is used only as an implementation check (the version with rescaling is unaffected). Even if it passes, this is not evidence that importance weighting does not work. (b) **Version with rescaling**: in all 24 cells of banks 500 and 1,000 the fraction stuck at the lower bound is at or below that of the version without rescaling and more than half of the cells are lower (if not, it is treated as a code error and the values of the version with rescaling must not be used), and the cell median of the mean weight at banks 500 and 1,000 is 0.5 to 2.0 (near 1; being outside is not evidence of an implementation error).


In [ ]:
# ================================================================
# PRAM Cell 1. Settings and input check (all constants live in this cell only)
# ================================================================
import os, sys, io, gc, json, time, math, zipfile, hashlib, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from scipy.optimize import minimize, minimize_scalar

import sklearn
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.exceptions import ConvergenceWarning
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


def _ov(name, default):
    """Used only when a test overrides values. Defaults apply in a normal run."""
    return globals().get(name + "_OVERRIDE", default)


# ---- run profile ---------------------------------------------------
#   "full"  : for the paper (default). The repetition counts are set in one place below.
#   "check" : pipeline check. Only verifies that the run completes on real data. Results are not used in the paper.
RUN_PROFILE = _ov("RUN_PROFILE", "full")
_PROFILES = {
    "full":  dict(N_DRAWS_MAIN=10, N_DRAWS_TAIL=100, BANK_SIZES=[500, 1000, 2000, 5000, "full"],
                  TAIL_BANKS=[500, 1000, 2000, 5000], LIGHT_BANKS=[1000, "full"], SITE_LIMIT=None,
                  TABPFN_BANKS=[500, 1000, 5000, "full"], COLDSTART_BANKS=[0, 100, 300, 500, 1000, 2000, 5000, "full"]),
    "check": dict(N_DRAWS_MAIN=2, N_DRAWS_TAIL=10, BANK_SIZES=[500, "full"],
                  TAIL_BANKS=[500], LIGHT_BANKS=["full"], SITE_LIMIT=8, TABPFN_BANKS=[500, "full"], COLDSTART_BANKS=[0, 500, "full"]),
}
_P = _PROFILES[RUN_PROFILE]

# ---- shared settings (seed, bootstrap repetitions, input folders) are defined once in code/pram_config.py ----
_cfg_dir = next((p / "code" for p in [Path.cwd(), *Path.cwd().parents] if (p / "code" / "pram_config.py").is_file()), None)
if _cfg_dir is None:
    raise FileNotFoundError("code/pram_config.py not found: open this notebook inside the repository checkout")
sys.path.insert(0, str(_cfg_dir))
from pram_config import SEED, BOOT_SEED, N_BOOT, THRESH_RANGE, DATA_ROOT, MIMIC4_ROOT, EICU_ROOT  # noqa: E402

# ---- reproducibility and repetition counts (one value for the whole analysis) --------------------------
ALPHA_GRID = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]   # mixing weight grid
K_NEIGHBORS = 50                            # default of retrieval_R1_cosine (Methods)
N_FOLDS = 5                                 # number of mock-batch folds
N_INNER = 3                                 # number of inner cross-validation folds within a bank (inner_cv_bank)
N_SITE_SPLITS = 3                           # number of folds in the hospital-level analysis
N_DRAWS_MAIN = _ov("N_DRAWS_MAIN", _P["N_DRAWS_MAIN"])    # number of draws per bank size
N_DRAWS_TAIL = _ov("N_DRAWS_TAIL", _P["N_DRAWS_TAIL"])    # tail risk: bank draws per cell = 5 folds x 20 draws per fold (100, as in Supplementary S7 Panel D)
BANK_SIZES = _ov("BANK_SIZES", _P["BANK_SIZES"])          # bank sizes of the strategy grid
TAIL_BANKS = _ov("TAIL_BANKS", _P["TAIL_BANKS"])          # the four bank sizes of Supplementary S7 Panel D
TABPFN_BANKS = _ov("TABPFN_BANKS", _P["TABPFN_BANKS"])
LIGHT_BANKS = _ov("LIGHT_BANKS", _P["LIGHT_BANKS"])       # bank sizes of the variants without temperature only and without INR only
SITE_LIMIT = _ov("SITE_LIMIT", _P["SITE_LIMIT"])          # None means all eligible hospitals
DEV_CB_ITER = _ov("DEV_CB_ITER", 500)       # frozen CatBoost iterations
LOCAL_CB_ITER = _ov("LOCAL_CB_ITER", 300)   # local CatBoost iterations
LAM_GRID = np.logspace(-4, 4, 17)           # anchored ridge regularization grid
ANCHORED_MAX_BANK = None                    # largest bank size at which the anchored ridge (model revision step) is fitted. None also fits the full bank
REVISION_RIDGE = 1e-5                       # tiny regularization for numerical stability of the model-revision fit (mean-loss units)
CLOSED_TEST_ALPHA = 0.05                    # closed-testing significance level
N_EVENT_MIN = 2                             # minimum number of events and non-events in a bank (below this the update fit is skipped)
N_TIME_SPLITS = 2                           # A1 temporal validation: number of folds that split the test period by outcome stratum (test fold and same-period bank alternate)

# ---- population and outcome definitions ----------------------
DROP_COLS = ["crp_min", "crp_max", "crp_last", "albumin_last", "albumin_max", "albumin_min",
             "bmi", "height_cm_min", "height_cm_last", "height_cm_max"]
OUTCOME_MAP = {"AKI_168h": "aki_168h", "mortality_168h": "death_168h"}
OUTCOMES = ["AKI_168h", "mortality_168h"]
TARGETS = ["MIMIC-IV", "MIMIC-III", "eICU"]
MISS_CUT_PCT = 30                           # variables whose maximum missingness over the three databases exceeds this value are dropped from the 33-column set
EXPECTED_SPARSE_9 = ["inr_last", "inr_max", "inr_min", "temp_c_last", "temp_c_max", "temp_c_min",
                     "weight_kg_last", "weight_kg_max", "weight_kg_min"]      # the nine variables expected to be dropped

# ---- hospital level and network ---------------------------
MIN_SITE_N = _ov("MIN_SITE_N", 300)
MIN_SITE_EVENTS = _ov("MIN_SITE_EVENTS", 15)    # minimum number of events and of non-events each
MAX_PARTNERS = 20
TEST_FRAC_NET = 0.30
NET_ALPHA_FIXED = 0.30                      # fixed mixing weight of an unlabeled hospital
NET_ALPHA_GRID = [round(a, 2) for a in np.arange(0.0, 1.0001, 0.05)]   # 0 to 1.0 in steps of 0.05
PARTNER_PREV_FACTORS = _ov("PARTNER_PREV_FACTORS", [1.0])                # partner bank prevalence multipliers. With 1.0 only, the whole partner cohort is used. Changing to [0.5, 1.0, 2.0] turns on the partner prevalence scenarios of A7

# ---- net benefit thresholds (A3) --------------------------------------------------
THRESH_GRID = np.round(np.arange(0.01, 0.5001, 0.01), 2)      # full 1 to 50% grid, stored in all tables
THRESH_MS = 0.20                            # main threshold of the main analysis
#   THRESH_RANGE (outcome-specific threshold ranges) is defined in code/pram_config.py; the summary changes when the ranges are edited.
#   the full-grid (1 to 50%) net benefit is written to csv, so changing the range needs no rerun.
T_SITE = {oc: round(float(np.mean(r)), 2) for oc, r in THRESH_RANGE.items()}   # midpoints of the outcome-specific ranges (representative thresholds of A2 and A11)
RANGE_ORIGINAL = (0.02, 0.30)               # range 2 to 30%

# ---- A2 tail risk and non-inferiority margin (post hoc definitions, labeled post hoc in the summary) ---------
NI_MARGINS_AUROC = [0.0, 0.005, 0.01, 0.02]
NI_MARGIN_AUROC = 0.01
NI_MARGINS_NB = [0.0, 0.0005, 0.001, 0.002]
NI_MARGIN_NB = 0.001                        # net benefit 0.001 = one net true positive per 1,000 patients
CP_LEVEL = 0.95                             # Clopper-Pearson interval level
SAFE_UPPER_BOUND = 0.05                     # "safest minimum bank size" = first size at which the upper bound of the probability of losing is at or below this value

A2_TARGETS_F33 = ["eICU"]                   # targets for which the 33-column tail is computed anew (eICU)
A2_RECOMPUTE_F42 = _ov("A2_RECOMPUTE_F42", False)                    # if False and the reference 42-column tail file (`a54_alpha_stability.jsonl`) exists in REFERENCE_RESUME_DIR, the 42-column tail is read from it (AUROC gain only, no model fit)
REFERENCE_RESUME_DIR = Path("reference") / "resume"                            # folder of stored reference result files (optional)
A54_PATH = REFERENCE_RESUME_DIR / "a54_alpha_stability.jsonl"
A29_PATH = REFERENCE_RESUME_DIR / "a29_real_sites.jsonl"
A51_PATH = REFERENCE_RESUME_DIR / "a51_hospital_thresholds.jsonl"
A59_PATH = REFERENCE_RESUME_DIR / "a59_hospital_diag.jsonl"
REPRO_UNITS = _ov("REPRO_UNITS", [("AKI_168h", "eICU", 1000), ("mortality_168h", "eICU", "full"),
                                  ("AKI_168h", "MIMIC-III", 500), ("mortality_168h", "MIMIC-IV", 2000)])   # units compared with the reference values before the 42-column grid
REPRO_TOL = 0.00005                         # the reference values have four decimals, so a rounding tolerance is allowed

# ---- rescoring: per-draw paired DeLong (Table 2 q column, Supplementary S7 Panels A and B, development-bank starting table p column) -------------
#   Only the two sections below (rescoring and starting bank) compute anew. The other sections skip units already finished in resume.
RUN_RESCORE = _ov("RUN_RESCORE", True)               # turns on the rescoring section (pooled DeLong, 42 columns) and its pre-check
RESCORE_COLDSTART = _ov("RESCORE_COLDSTART", True)   # p column of the development-bank starting table. Turning it off skips the computation of that column
RESCORE_ONLY_GUARD = _ov("RESCORE_ONLY_GUARD", False)  # if True, stops before computing when a unit outside the rescoring would run anew (guards against rerunning 12 to 16 hours in a folder that already holds finished results). Off by default so that the notebook runs from scratch
RESCORE_NAMES = ("rescore_F42", "coldstart_F42")      # only the resume file with this name is written anew
COLDSTART_BANKS = _ov("COLDSTART_BANKS", _P["COLDSTART_BANKS"])   # 0 = source bank only, number = source bank + local bank
COLDSTART_SRC_OOF_SPLITS = 3                # number of cross-validation folds for the source-bank base prediction of _source_bank
SIG_LEVEL = 0.05                            # significance level of the per-draw test (criterion of Supplementary S7 Panel B)

# ---- A1 temporal validation ------------------------------------------------------
A1_POLICIES = _ov("A1_POLICIES", ("P", "S"))                    # P past bank, S same-period bank. Changing to ("P", "S", "R") also gives a random (periods mixed) bank R
HOSP_MIN_N_PER_YEAR = _ov("HOSP_MIN_N_PER_YEAR", 300)    # hospital-level temporal analysis: minimum number of ICU stays in each of the two years
HOSP_MIN_EVENTS_PER_YEAR = _ov("HOSP_MIN_EVENTS_PER_YEAR", 15)
HOSP_TIME_MIN_K = 10                        # if fewer hospitals qualify, only descriptive statistics instead of meta-analysis
RUN_A1C = _ov("RUN_A1C", False)                             # A1c: refit the development model on the earlier MIMIC-IV period (turned off)
YEAR_RULES = ("strict", "majority")         # MIMIC-IV year-period assignment: only when entirely inside a period / period covering the majority
RUN_A1D = _ov("RUN_A1D", False)                             # A1d: hospital-level temporal validation (turned off; the number of hospitals above the criterion in both years is unknown)

# ---- A9 TabPFN ----------------------------------------------------------
TABPFN_N_ESTIMATORS = _ov("TABPFN_N_ESTIMATORS", 4)
TABPFN_CTX_GPU = _ov("TABPFN_CTX_GPU", 10000)       # development subsample size (context limit of v2: 10,000 rows)
TABPFN_CTX_CPU = _ov("TABPFN_CTX_CPU", 2000)
TABPFN_SUBSETS_GPU = _ov("TABPFN_SUBSETS_GPU", 3)   # number of subsample ensemble members
TABPFN_SUBSETS_CPU = _ov("TABPFN_SUBSETS_CPU", 1)
TABPFN_CPU_TARGET_CAP = _ov("TABPFN_CPU_TARGET_CAP", 6000)   # maximum number of rows per target cohort on CPU
TABPFN_MAX_MINUTES = _ov("TABPFN_MAX_MINUTES", 240)          # if the expected time exceeds this, the sample is reduced
TABPFN_BATCH = _ov("TABPFN_BATCH", 128)   # rows per prediction batch. On an 8 GB GPU a batch of 10,000 context rows x 5,000 test rows requires 72 GiB and runs out of memory. Start at 128 and halve and retry on out-of-memory (tabpfn_predict)
TABPFN_TARGETS = ["eICU", "MIMIC-III"]              # eICU and MIMIC-III external validation

# ---- A11 strategy selection rule -------------------------------------------------
SEL_CANDIDATES = ["b1_base", "b5_base", "platt_b1", "recal_iso", "pram_b1", "pram_b5", "local_lr"]
#   pooled_lr needs the development record and cannot be chosen by a hospital, and local_cb and anchored ridge are left out of the candidates because their inner cross-validation within a bank is costly.

# ---- sections to run (each can be switched on or off; off removes both its computation and its report) -------------
RUN_MATRIX = _ov("RUN_MATRIX", True)       # strategy grid (main analysis, the 33-column set of A8, A4, A5 and A11 are derived from it)
RUN_A1 = _ov("RUN_A1", True)           # temporal validation
RUN_A2 = _ov("RUN_A2", True)           # tail risk
RUN_HOSPITAL = _ov("RUN_HOSPITAL", True)     # hospital-level computation (the hospital parts of A3, A6 and A8 are derived from it)
RUN_A3 = _ov("RUN_A3", True)           # decision-curve threshold report (on top of the hospital-level results)
RUN_A4 = _ov("RUN_A4", True)           # update hierarchy and closed testing (hier and anch groups from the grid and the hospital level)
RUN_A5 = _ov("RUN_A5", True)           # retrieval + Platt combination order
RUN_A6 = _ov("RUN_A6", True)           # hospital-level meta-analysis (tau, prediction interval, event counts)
RUN_A7 = _ov("RUN_A7", True)           # network of unlabeled hospitals and partner prevalence (including the network part of A8). The 33-column network is also derived from it
RUN_A9 = _ov("RUN_A9", True)           # TabPFN
RUN_REAGG = _ov("RUN_REAGG", True)      # if the reference hospital-level files (`a29_real_sites`, `a51_hospital_thresholds`, `a59_hospital_diag`) exist in REFERENCE_RESUME_DIR, recount them and compare with the hospital computation of this notebook (skipped if absent)
RUN_A11 = _ov("RUN_A11", False)         # strategy selection rule within a bank (turned off)

# ---- strategy group ----------------------------------------------------------
GROUPS_FULL = (("frozen", "pram", "recal", "local", "localcb") + (("anch", "hier") if RUN_A4 else ())
               + (("combo",) if RUN_A5 else ()) + (("sel",) if RUN_A11 else ()))
GROUPS_LIGHT = ("frozen", "pram", "recal")
GROUPS_HOSP = ("frozen", "pram", "recal", "local", "localcb") + (("anch", "hier") if RUN_A4 else ()) + (("combo",) if RUN_A5 else ())
M_EIGHT = ["b1_base", "b5_base", "recal_iso", "pram_b1", "pram_b5", "local_lr", "pooled_lr", "local_cb"]   # the eight strategies of the manuscript
FS_SPEC = {
    "F42":        dict(drop_rule=None, drop_prefix=(), targets=TARGETS, banks=BANK_SIZES,
                       groups=GROUPS_FULL, hospital=True, network=True),
    "F33":        dict(drop_rule="miss", drop_prefix=(), targets=TARGETS, banks=BANK_SIZES,
                       groups=GROUPS_FULL, hospital=True, network=True),
    "F42_noTemp": dict(drop_rule=None, drop_prefix=("temp_c_",), targets=["eICU"], banks=LIGHT_BANKS,
                       groups=GROUPS_LIGHT, hospital=False, network=False),
    "F42_noINR":  dict(drop_rule=None, drop_prefix=("inr_",), targets=["eICU"], banks=LIGHT_BANKS,
                       groups=GROUPS_LIGHT, hospital=False, network=False),
}
FS_RUN = _ov("FS_RUN", ["F42", "F33"])      # "F42_noTemp" and "F42_noINR" (variants without temperature only or INR only) run when included

# ---- output --------------------------------------------------------------
OUT_NAME = "out_pram" + ("" if RUN_PROFILE == "full" else "_CHECK")
OUT_DIR = Path(OUT_NAME)
CSV_DIR = OUT_DIR / "github_csv"
RESUME_DIR = OUT_DIR / "resume"
for _d in (OUT_DIR, CSV_DIR, RESUME_DIR):
    _d.mkdir(parents=True, exist_ok=True)
ZIP_NAME = OUT_NAME + ".zip"

# ---- candidate paths of the temporal and hospital raw data --------------------------
#   candidates for the eICU patient table are the list below plus the .gz variants.
#   MIMIC-IV candidates are listed below. If not found, write the actual path in EXTRA_RAW_PATHS below.
EXTRA_RAW_PATHS = {"eicu_patient": [EICU_ROOT / "patient.csv", EICU_ROOT / "patient.csv.gz"], "mimic4_patients": [], "mimic4_icustays": []}
#   MIMIC-IV folder (patients and icustays are expected under hosp and icu in this folder).
#   Below, hosp/patients.csv(.gz) and icu/icustays.csv(.gz) are searched. If not found, the paths searched are printed in the input check table and only A1b is skipped.
_E = ["./eICU/patient.csv", "eICU/patient.csv", "./eicu/patient.csv",
      DATA_ROOT / "eICU" / "patient.csv", DATA_ROOT / "eicu" / "patient.csv", DATA_ROOT / "eicu" / "hospital_map.csv"]
EICU_PATIENT_CANDIDATES = [Path(p) for p in _E + [str(p) + ".gz" for p in _E[:5]]
                           + EXTRA_RAW_PATHS["eicu_patient"]]
_M4DIRS = [MIMIC4_ROOT, DATA_ROOT / "mimic4", "."]
MIMIC4_PATIENTS_CANDIDATES = [Path(d) / s for d in _M4DIRS for s in ("hosp/patients.csv", "hosp/patients.csv.gz", "patients.csv")] \
    + [Path(p) for p in EXTRA_RAW_PATHS["mimic4_patients"]]
MIMIC4_ICUSTAYS_CANDIDATES = [Path(d) / s for d in _M4DIRS for s in ("icu/icustays.csv", "icu/icustays.csv.gz", "icustays.csv")] \
    + [Path(p) for p in EXTRA_RAW_PATHS["mimic4_icustays"]]


def find_first(cands):
    """the first existing path among the candidates, and the list of absolute paths searched."""
    tried = []
    for p in cands:
        try:
            ap = Path(p).resolve()
        except OSError:
            ap = Path(p)
        tried.append(str(ap))
        try:
            if Path(p).is_file():
                return Path(p), tried
        except OSError:
            continue
    return None, tried


def header_cols(path):
    try:
        return [str(c) for c in pd.read_csv(path, nrows=3).columns]
    except Exception as exc:  # noqa: BLE001
        print("  [read failed]", path, type(exc).__name__, str(exc)[:80])
        return []


# ---- input check table ---------------------------------------------------------
MAIN_FILES = {
    "MIMIC-IV features": DATA_ROOT / "mimic4" / "X_folded.parquet",
    "MIMIC-IV outcomes": DATA_ROOT / "mimic4" / "binary_outcomes_mimic4.csv",
    "MIMIC-III features": DATA_ROOT / "mimic3" / "X_external.parquet",
    "MIMIC-III outcomes": DATA_ROOT / "mimic3" / "binary_outcomes_mimic3.csv",
    "eICU features": DATA_ROOT / "eicu" / "X_external.parquet",
    "eICU outcomes": DATA_ROOT / "eicu" / "binary_outcomes_eicu.csv",
}
INPUT_ROWS = []
for _k, _p in MAIN_FILES.items():
    INPUT_ROWS.append((_k, str(Path(_p).resolve()), "present" if Path(_p).is_file() else "absent"))
DATA_OK = all(r[2] == "present" for r in INPUT_ROWS)

EICU_PATIENT_PATH, _tried_e = find_first(EICU_PATIENT_CANDIDATES)
EICU_COLS = header_cols(EICU_PATIENT_PATH) if EICU_PATIENT_PATH else []
_lowE = {c.lower(): c for c in EICU_COLS}
EICU_COL_STAY = _lowE.get("patientunitstayid")
EICU_COL_HOSP = _lowE.get("hospitalid")
EICU_COL_YEAR = _lowE.get("hospitaldischargeyear")
HAVE_HOSPITALID = bool(EICU_COL_STAY and EICU_COL_HOSP)
HAVE_EICU_YEAR = bool(EICU_COL_STAY and EICU_COL_YEAR)
INPUT_ROWS.append(("eICU patient table (hospital id)", str(EICU_PATIENT_PATH) if EICU_PATIENT_PATH else "not found",
                   "column present" if HAVE_HOSPITALID else "column absent"))
INPUT_ROWS.append(("eICU patient table (hospitaldischargeyear)", str(EICU_PATIENT_PATH) if EICU_PATIENT_PATH else "not found",
                   "column present" if HAVE_EICU_YEAR else "column absent"))

MIMIC4_PATIENTS_PATH, _tried_p = find_first(MIMIC4_PATIENTS_CANDIDATES)
MIMIC4_ICUSTAYS_PATH, _tried_i = find_first(MIMIC4_ICUSTAYS_CANDIDATES)
_cp = {c.lower(): c for c in (header_cols(MIMIC4_PATIENTS_PATH) if MIMIC4_PATIENTS_PATH else [])}
_ci = {c.lower(): c for c in (header_cols(MIMIC4_ICUSTAYS_PATH) if MIMIC4_ICUSTAYS_PATH else [])}
_m4_miss_p = [k for k in ("subject_id", "anchor_year_group") if k not in _cp]
_m4_miss_i = [k for k in ("stay_id", "subject_id") if k not in _ci]
HAVE_MIMIC4_YEAR = bool(MIMIC4_PATIENTS_PATH and MIMIC4_ICUSTAYS_PATH and not _m4_miss_p and not _m4_miss_i)
#   the unit-transfer adjustment works only when patients has anchor_year and icustays has intime.
#   for a copy without anchor_year (anonymized), the periods are split by a patient-level approximation (see read_mimic4_periods and the A1b cell).
M4_HAS_SHIFT = bool(HAVE_MIMIC4_YEAR and "anchor_year" in _cp and "intime" in _ci)
M4_MODE = "unit-transfer adjusted" if M4_HAS_SHIFT else ("patient-level approximation" if HAVE_MIMIC4_YEAR else "absent")
M4_APPROX_WHY = "" if (M4_HAS_SHIFT or not HAVE_MIMIC4_YEAR) else ("anchor_year absent" if "anchor_year" not in _cp else "intime absent")
M4_WHY = []                                  # reason A1b cannot run (human-readable). It runs when empty
if not MIMIC4_PATIENTS_PATH:
    M4_WHY.append("not found: patients")
elif _m4_miss_p:
    M4_WHY.append("column absent: " + ", ".join(_m4_miss_p) + " (patients)")
if not MIMIC4_ICUSTAYS_PATH:
    M4_WHY.append("not found: icustays")
elif _m4_miss_i:
    M4_WHY.append("column absent: " + ", ".join(_m4_miss_i) + " (icustays)")
INPUT_ROWS.append(("MIMIC-IV patients(anchor_year_group)", str(MIMIC4_PATIENTS_PATH) if MIMIC4_PATIENTS_PATH else "not found",
                   ("column present" if not _m4_miss_p else "column absent: " + ", ".join(_m4_miss_p)) if MIMIC4_PATIENTS_PATH else "not found"))
INPUT_ROWS.append(("MIMIC-IV icustays(stay_id · subject_id)", str(MIMIC4_ICUSTAYS_PATH) if MIMIC4_ICUSTAYS_PATH else "not found",
                   ("column present" if not _m4_miss_i else "column absent: " + ", ".join(_m4_miss_i)) if MIMIC4_ICUSTAYS_PATH else "not found"))
INPUT_ROWS.append(("MIMIC-IV period assignment method (A1b)", "unit-transfer adjustment if anchor_year and intime exist, otherwise patient-level approximation", M4_MODE + (" (" + M4_APPROX_WHY + ")" if M4_APPROX_WHY else "")))

try:
    import tabpfn as _tabpfn_mod            # noqa: F401
    HAVE_TABPFN = True
except Exception:  # noqa: BLE001
    HAVE_TABPFN = False
TABPFN_PKG_VERSION = "not installed"
if HAVE_TABPFN:
    try:
        from importlib.metadata import version as _pkg_version
        TABPFN_PKG_VERSION = _pkg_version("tabpfn")
    except Exception:  # noqa: BLE001
        TABPFN_PKG_VERSION = str(getattr(_tabpfn_mod, "__version__", "unknown"))
HAVE_CUDA = False
if HAVE_TABPFN:                              # torch is imported only when TabPFN is present, to look at the GPU (avoids import conflicts of torch in environments without it)
    try:
        import torch as _torch_mod
        HAVE_CUDA = bool(_torch_mod.cuda.is_available())
    except Exception:  # noqa: BLE001
        HAVE_CUDA = False
INPUT_ROWS.append(("tabpfn package", "pip install tabpfn . A9 selects the v2 weights explicitly (create_default_for_version(ModelVersion.V2)). The v2 weights are downloaded on the first run",
                   ("installed " + TABPFN_PKG_VERSION) if HAVE_TABPFN else "absent"))
INPUT_ROWS.append(("GPU(CUDA)", "used by A9 only", "present" if HAVE_CUDA else "absent"))

print("=" * 100)
print("  PRAM profile =", RUN_PROFILE, "(use full when the results go into the paper)" if RUN_PROFILE == "full" else "(for checking only; results are not used in the paper)")
print("=" * 100)
print("| input | path | status |")
print("|---|---|---|")
for _r in INPUT_ROWS:
    print("| %s | %s | %s |" % _r)
if not DATA_OK:
    print("\n  [not a stop] one of the six main inputs is missing. All analysis cells below are skipped. Fix DATA_ROOT and run again ->", DATA_ROOT.resolve())
if not HAVE_HOSPITALID:
    print("\n  The hospital id column was not found. Hospital-level analysis, network and hospital-level temporal analysis are marked not available and skipped. Paths searched:")
    for _t in _tried_e:
        print("    ", _t)
if not HAVE_EICU_YEAR:
    print("\n  The eICU discharge year column (hospitaldischargeyear) was not found. The eICU temporal validation of A1 is skipped as not available.")
if HAVE_MIMIC4_YEAR and not M4_HAS_SHIFT:
    print("\n  The MIMIC-IV period assignment is a patient-level approximation (%s). All stays of the same patient are placed in that patient anchor_year_group period. intime is not used to order patients." % M4_APPROX_WHY)
if not HAVE_MIMIC4_YEAR:
    print("\n  The MIMIC-IV time chain (A1b) cannot run , " + " · ".join(M4_WHY) + ". The MIMIC-IV time chain of A1 is skipped as not available. Paths searched (patients):")
    for _t in _tried_p:
        print("    ", _t)
    print("  paths searched (icustays):")
    for _t in _tried_i:
        print("    ", _t)
print("\n  python %s | numpy %s | pandas %s | sklearn %s" % (platform.python_version(), np.__version__, pd.__version__, sklearn.__version__))
print("  SEED=%d | folds %d | inner folds %d | hospital folds %d | draws %s (tail %s) | bank sizes %s" % (
    SEED, N_FOLDS, N_INNER, N_SITE_SPLITS, N_DRAWS_MAIN, N_DRAWS_TAIL, BANK_SIZES))
print("  output folder %s | zip %s" % (OUT_DIR.resolve(), ZIP_NAME))


In [ ]:
# ================================================================
# PRAM extra analyses  new settings of this notebook (existing constants use those of the settings cell, the third cell from the top: seed, number of folds, number of draws, significance level, margin)
# ================================================================
import re

R5_WHOLE_FS = "F33"                                  # (a) feature sets for which the full-bank paired DeLong is run here. The 42-column set is run in notebook 01
R5_TAIL_FS = ("F42",)                                # (b) feature sets for which the single-bank 100-draw interval is newly run (three cohorts)
R5_TAIL_EXTRA_FS = ("F33",)                          # 33-column MIMIC cells (the 33-column tail of notebook 01 covers eICU only)
R5_TAIL_EXTRA_TARGETS = [t for t in TARGETS if t not in A2_TARGETS_F33]      # the stored 33-column eICU results are read and used
R5_TAIL_GROUPS = ("frozen", "pram")                  # strategy group needed for the paired DeLong. The prediction of pram_b1 does not depend on the group composition (the reproduction check looks at it)
R5_TAIL_KEEP = ["b1_base", "pram_b1"]
R5_TAIL_PAIRS = {"pram_b1": "b1_base"}
R5_Z = 1.96                                          # 95% normal interval (same value as tail_gaps of the new-definitions cell)
R5_CHECK_TOL = 1e-9                                  # tolerance of the reproduction check (same as the grid comparison)
R5_CSV_RTOL = 1e-4                                   # the stored csv has 6 significant digits, so a relative error is allowed
R5_REPRO_WHOLE = [("F42", "AKI_168h", "MIMIC-IV"), ("F42", "mortality_168h", "MIMIC-III")]       # full-bank units (small cohorts) to rerun as a reproduction check against notebook 01
R5_REPRO_TAIL = [("F33", "AKI_168h", "eICU", TAIL_BANKS[0]), ("F33", "mortality_168h", "eICU", TAIL_BANKS[min(1, len(TAIL_BANKS) - 1)])]   # 33-column eICU tail units of the stored results
R5_RES_WHOLE = f"R5_whole_{R5_WHOLE_FS}"
R5_RES_TAIL = {fs: f"R5_tail_{fs}" for fs in dict.fromkeys(tuple(R5_TAIL_FS) + tuple(R5_TAIL_EXTRA_FS))}
R5_RES_REPRO = ("R5_repro_whole", "R5_repro_tail")
R5_NEW_NAMES = tuple([R5_RES_WHOLE] + list(R5_RES_TAIL.values()) + list(R5_RES_REPRO))      # all resume names newly written by this notebook
R5_CSV_PREFIX = "R5_"                                # prefix of new csv files (a csv without this prefix cannot be written)
R5_SUM_NAME = "PRAM-extra-summary.md"
R5_RES_NAME = "PRAM-extra-results.md"
R5_ZIP_NAME = OUT_NAME + "_part1.zip"
R5_WHOLE_UNITS = [(R5_WHOLE_FS, oc, tg, "full") for oc in OUTCOMES for tg in FS_SPEC[R5_WHOLE_FS]["targets"]]
R5_TAIL_UNITS = ([(fs, oc, tg, bk) for fs in R5_TAIL_FS for oc in OUTCOMES for tg in FS_SPEC[fs]["targets"] for bk in TAIL_BANKS]
                 + [(fs, oc, tg, bk) for fs in R5_TAIL_EXTRA_FS for oc in OUTCOMES for tg in R5_TAIL_EXTRA_TARGETS for bk in TAIL_BANKS])
R5_INVENTORY_NEED = dict([(f"matrix_{f}", len(OUTCOMES) * len(FS_SPEC[f]["targets"]) * len(FS_SPEC[f]["banks"])) for f in ("F42", "F33")]
                         + [("rescore_F42", len(OUTCOMES) * len(FS_SPEC["F42"]["targets"]) * len(FS_SPEC["F42"]["banks"])),
                            ("a2_tail_F33", len(OUTCOMES) * len(A2_TARGETS_F33) * len(TAIL_BANKS))])    # number of units read if the folder holds them (only printed in the check table, not a requirement. If insufficient, the input-collection section recomputes with the same function)
print("settings: (a) %s full bank %d units | (b) %s single bank %d units (banks %s, %d draws per cell, including the 33-column MIMIC supplement) | significance level %s | margin %s" % (
    R5_WHOLE_FS, len(R5_WHOLE_UNITS), list(R5_TAIL_FS), len(R5_TAIL_UNITS), TAIL_BANKS, N_DRAWS_TAIL, SIG_LEVEL, NI_MARGIN_AUROC))
print("  resume names newly written: %s" % (list(R5_NEW_NAMES),))
print("  new zip: %s" % R5_ZIP_NAME)


In [ ]:
# ================================================================
# PRAM extra analyses, stage B  settings (existing constants use those of the settings cell, the third cell from the top: seed, number of folds, number of draws, bank sizes, margin)
# ================================================================
RUN_B_IW = True                                      # B-1 importance-weighted retraining and recalibration
RUN_B_CORAL = True                                   # B-2 CORAL
RUN_B_ANCHOR = True                                  # B-3 anchor regression (two-environment version, development split period version, eICU hospital extension and PRAM on top)
RUN_B_ANCHOR_HOSP = True                             # switches on and off only the eICU hospital extension of B-3 (AR-hosp, hospital level only, reference upper bound). If off, the two-environment version, the period version and PRAM on top still run (meaningful only when RUN_B_ANCHOR is on)

R5B_IW_CLIP = (0.05, 20.0)                           # importance weight truncation (a choice made here, not taken from the literature)
R5B_DISC_C = 1.0                                     # regularization of the shift-discrimination logistic (same as the version without rescaling)
R5B_LR_C = 1.0                                       # regularization of the logistic refitted after weighting and alignment (same as the frozen model)
R5B_PLATT_C = 1e6                                    # regularization of the weighted recalibration
R5B_Q_EPS = 1e-6                                     # truncation when converting discrimination probabilities to weights (same as the version without rescaling)
R5B_CORAL_LAMBDA = 1.0                               # multiple of the identity matrix added to the CORAL covariance (value of Sun et al. 2016)
R5B_AR_GAMMAS = (1, 2, 5, 10, 20, 50, 100)           # anchor regression gamma grid (ascending)
R5B_CV_FOLDS = 5                                     # number of environment-level folds (if there are fewer environments, the number of environments. With two environments, leave-one-environment-out)
R5B_TIE_TOL = 1e-6                                   # a tie if the fold AUROC is within this value (on a tie, the smaller gamma)
R5B_MIN_ENV_N = 30                                   # minimum number of stays of a period environment (below this the period is dropped, and printed)
R5B_PROBIT_MAXITER = _ov("R5B_PROBIT_MAXITER", 3000)    # maximum iterations of probit anchor L-BFGS (only an internal check overrides this value to test stopping)
R5B_PROBIT_FTOL = 1e-15
R5B_PROBIT_GTOL = 1e-9
R5B_PROBIT_GRAD_TOL = 1e-6                           # convergence criterion: maximum absolute value of the mean-loss gradient. If exceeded, stops without a value
R5B_BETA_MAX = 30.0                                  # upper bound of the absolute probit coefficient. If exceeded on standardized features, it is complete separation (only the slope approaches 0), so stops without a value
R5B_EIG_REL = 1e-12                                  # lower bound of the minimum eigenvalue / maximum eigenvalue of CORAL covariance + lam I (below this it is singular and stops)
R5B_ECE_MARGIN = 0.005                               # ECE margin (predefined, weak basis)
R5B_MARGIN_SETS = {"strict": (0.005, 0.0005, 0.0025), "base": (NI_MARGIN_AUROC, NI_MARGIN_NB, R5B_ECE_MARGIN), "loose": (0.02, 0.002, 0.01)}      # (AUROC, net benefit, ECE) three sensitivity sets
R5B_DOM_SHARE = 0.90                                 # DOM: fraction of draws in which ECE decreased
R5B_NULL_SHARE = 0.90                                # P3: fraction of NULL cells
R5B_P4_CELL_SHARE = 0.80                           # P4: DOM fraction among eICU cells
R5B_P4_HOSP_SHARE = 0.60                           # P4: DOM fraction among hospitals
R5B_P2_DOM_MAX = 15                                # P2: upper limit of the number of DOM cells (1 to 15 of the rule based on the 20 eICU cells)
R5B_F_KEEP_SHARE = 0.90                              # F1: fraction of cells with G_X >= -margin
R5B_F_ECE_SHARE = 0.50                               # F1: fraction of cells in which ECE decreased
R5B_IW_CHECK = {"slope_lo": 0.40, "slope_hi": 0.55, "min_cells": 18, "n_cells": 24, "banks": (500, 1000, 5000, "full")}   # check criterion for whether the version without rescaling reproduces the reference values (not used for the version with rescaling)
R5B_IWC_CHECK = {"banks": (500, 1000), "w_mean_lo": 0.5, "w_mean_hi": 2.0}   # check criterion of the version with rescaling: the lower-bound fraction at banks 500 and 1,000 is at or below that of the version without rescaling (more than half of the cells lower), the mean weight is near 1
R5B_SPLIT_CELLS = 6                                  # the two versions split widely: number of cells with different D+, DOM and NULL (10% of 60 cells)
R5B_REPRO_BANKS = ("full",)                          # bank size of the cells compared to check whether the draws are the same as in the grid of notebook 01 (the result columns platt_b1 and local_lr use the bank label)
R5B_REPRO_UNITS = [("F42", "AKI_168h", "MIMIC-III"), ("F42", "mortality_168h", "MIMIC-IV")]       # cells to rerun to see whether the same values as the stored resume come out (small cohort)
R5B_REPRO_TOL = 1e-9
R5B_HOSP_BOOT_N = N_BOOT                             # hospital-level interval (same kind as the main text): number of hospital bootstrap repetitions. Uses the single N_BOOT of the settings cell (5,000, the same as N_BOOT of the aggregation script that produced the main-text values). This notebook bootstraps only here
R5B_HOSP_BOOT_SEED = BOOT_SEED                       # the single BOOT_SEED of the settings cell (20261003, SEED of the same script)
# the random stream also matches the same script (aggregate_claims.py run_B): one generator (seed above) is used in sequence over the feature set (F42 -> F33) -> comparison pairs (4 HOSP_PAIRS) -> outcome,
# and in each call one (B x k) replicate matrix for k hospitals is drawn. k is the k of the script output B_hospital_stats.csv (AKI 67 and death 59 in all pairs). The hospital order is the order of the PRAM rows of the hospital resume of the stored results (largest hospital first)
R5B_HOSP_BOOT_FS = ("F42", "F33")
R5B_HOSP_BOOT_PAIRS = 4
R5B_HOSP_BOOT_K = (("AKI_168h", 67), ("mortality_168h", 59))
# main-text hospital-level interval = the pram_b1|b1_base dAUC row of the same script output B_hospital_stats.csv: (stay-weighted mean, interval lower, interval upper). The PRAM row of the notebook must equal these values for the stream to be the same
R5B_HOSP_BOOT_REF = {("F42", "AKI_168h"): (0.0091850558524123, 0.003979521791381, 0.0149246521552983), ("F42", "mortality_168h"): (0.0201485347503233, 0.0134956416048028, 0.0267020220528477),
                     ("F33", "AKI_168h"): (0.0083724231834314, 0.0030835396229329, 0.0140423611854596), ("F33", "mortality_168h"): (0.0015488097492803, -0.0014804358565226, 0.0049148952199482)}
R5B_HOSP_BOOT_REF_TOL = 5e-6                         # that script counted the difference with the 6-decimal AUROC of the stored csv (within 1e-6 per hospital). The weighted mean and percentiles must agree within that
R5B_IW_REF_TOL = {"w_rel": 0.25, "auc_abs": 0.03} # tolerance of the reproduction checks 2-2 and 2-3 of the version without rescaling (a predefined value with a weak basis)
# 24 reference cells: (outcome, target, bank, mean weight iw_w_mean, shift-discrimination AUC). Mean weight = reference values of the version without rescaling, discrimination AUC = Supplementary S8 Panel H (results_csv/legacy)
R5B_IW_REF = [
    ("AKI_168h", "MIMIC-IV", "500", 0.0501, 0.579), ("AKI_168h", "MIMIC-IV", "1000", 0.0559, 0.552), ("AKI_168h", "MIMIC-IV", "5000", 0.2669, 0.526), ("AKI_168h", "MIMIC-IV", "full", 0.4286, 0.521),
    ("AKI_168h", "MIMIC-III", "500", 0.0585, 0.835), ("AKI_168h", "MIMIC-III", "1000", 0.0778, 0.830), ("AKI_168h", "MIMIC-III", "5000", 0.2698, 0.830), ("AKI_168h", "MIMIC-III", "full", 0.6103, 0.828),
    ("AKI_168h", "eICU", "500", 0.0632, 0.921), ("AKI_168h", "eICU", "1000", 0.0846, 0.930), ("AKI_168h", "eICU", "5000", 0.2612, 0.927), ("AKI_168h", "eICU", "full", 1.8720, 0.924),
    ("mortality_168h", "MIMIC-IV", "500", 0.0502, 0.571), ("mortality_168h", "MIMIC-IV", "1000", 0.0556, 0.552), ("mortality_168h", "MIMIC-IV", "5000", 0.2669, 0.527), ("mortality_168h", "MIMIC-IV", "full", 0.4286, 0.523),
    ("mortality_168h", "MIMIC-III", "500", 0.0593, 0.825), ("mortality_168h", "MIMIC-III", "1000", 0.0769, 0.830), ("mortality_168h", "MIMIC-III", "5000", 0.2693, 0.828), ("mortality_168h", "MIMIC-III", "full", 0.6081, 0.829),
    ("mortality_168h", "eICU", "500", 0.0635, 0.927), ("mortality_168h", "eICU", "1000", 0.0840, 0.927), ("mortality_168h", "eICU", "5000", 0.2581, 0.928), ("mortality_168h", "eICU", "full", 1.8426, 0.924)]
R5B_KEEP = {"iw": ("b1_base", "iw_erm", "iw_platt", "iw_temp", "dev_platt", "dev_temp"), "iwc": ("b1_base", "iwc_erm", "iwn_erm", "iwc_platt", "iwc_temp", "dev_platt", "dev_temp"), "coral": ("b1_base", "coral", "coral_z")}      # methods by which the cell-level call gives metrics (the metric-computation share of the expected time is also counted with this number)
R5B_CHECK_METHOD = "platt_b1"                        # columns added only to the cell-level call for direct comparison: fitted with the bank label, so the values differ if the bank draws and split differ from the stored results (recal group)
R5B_FS = [f for f in ("F42", "F33") if f in FS_RUN]  # feature sets in which stage B runs (same as stage A and the stored results)
R5B_AR_METHODS = {"lpm": "linear probability + Platt", "probit": "probit anchor classification"}
R5B_PERIOD_MAIN_RULE = "strict"                        # main rule among the environment period rules of AR-period (env name `period`). The other rule is a sensitivity (env name `period_alt`, differing in only the environment period rule)
#   basis: Table 4, Supplementary S9 and A1 YEAR_RULES list strict first, and `read_mimic4_periods` defines strict (only stays that fall entirely inside one 3-year period) as the primary rule
R5B_PERIOD_ALL_RULES = ("strict", "majority")           # two rules computed in the unit-transfer adjustment method (anchor_year and intime present). In the patient-level approximation method the two rules are the same, so only patient_group is computed
R5B_AR_ENVS = ("2db", "period", "period_alt")           # environment versions that run at the cell level. The hospital anchor extension (hosp) is at the hospital level only. period = main rule, period_alt = the other rule (sensitivity)
R5B_TARGETS_BY_ENV = {"2db": ("MIMIC-IV", "eICU"), "period": tuple(TARGETS), "period_alt": tuple(TARGETS)}
R5B_NAMES = {"iw": "R5_b_iw", "iwinfo": "R5_b_iwinfo", "coral": "R5_b_coral", "coralinfo": "R5_b_coralinfo", "iwh": "R5_b_iwh", "coralh": "R5_b_coralh",
             "arfit": "R5_b_arfit", "areval": "R5_b_areval", "arh": "R5_b_arh", "fus": "R5_b_fus", "fusinfo": "R5_b_fusinfo",
             "iwc": "R5_b_iwc", "iwcinfo": "R5_b_iwcinfo", "iwch": "R5_b_iwch",
             "repro": "R5_b_repro", "reproh": "R5_b_reproh"}


def r5b_rn(kind, fs=None):
    """resume names of stage B. When a feature set is attached, it is appended to the end of the name as `_F42`."""
    return R5B_NAMES[kind] + (("_" + fs) if fs else "")


R5B_NEW_NAMES = tuple([r5b_rn(k, fs) for k in ("iw", "iwinfo", "coral", "coralinfo", "iwh", "coralh", "arh", "fus", "fusinfo", "reproh", "iwc", "iwcinfo", "iwch") for fs in R5B_FS]
                      + [r5b_rn("arfit"), r5b_rn("areval"), r5b_rn("repro")])
R5_NEW_NAMES += R5B_NEW_NAMES                        # the write lock of stage A also permits the new names of B (other names stay blocked)
print("stage B settings: on IW %s | CORAL %s | ANCHOR %s (hospital extension %s) | gamma grid %s | truncation %s | CORAL regularization %s | feature sets %s" % (RUN_B_IW, RUN_B_CORAL, RUN_B_ANCHOR, RUN_B_ANCHOR_HOSP, list(R5B_AR_GAMMAS), R5B_IW_CLIP, R5B_CORAL_LAMBDA, R5B_FS))
print("  new resume names written (B): %d, all starting with R5_b_" % len(R5B_NEW_NAMES))


In [ ]:
# ================================================================
# PRAM extra computations settings (existing constants use those of the settings cell, A settings and B settings: seed, number of folds, number of draws, significance level, margin)
# ================================================================
RUN_X_A9INV = True                                   # back-calculation of the TabPFN evaluation sample row count (no new computation, read only)
RUN_X_S5A = True                                     # values of the 42-column and 33-column MIMIC of Supplementary S5 Panel A from the new 100 draws (no new computation)
RUN_X_FZ = True                                      # E-1: paired DeLong of the 33-column frozen logistic versus 42 columns + PRAM (6 full-bank units)
RUN_X_NET = True                                     # E-2: partner bank prevalence 0.5x and 2x (unlabeled hospital network)
X_FZ_FROZEN_FS = "F33"                               # read the frozen logistic in the 33-column context
X_FZ_PRAM_FS = "F42"                                 # read PRAM in the 42-column context
X_FZ_UNITS = [(oc, tg) for oc in OUTCOMES for tg in TARGETS]       # 2 outcomes x 3 targets = 6 units
X_FZ_PRIMARY = ("mortality_168h", "eICU")            # pre-specified comparison (the remaining 5 units are descriptive)
X_FZ_TOL = 1e-9                                      # tolerance of the reproduction check against grid values of the stored results
X_NET_FACTORS = [0.5, 2.0]                           # partner bank prevalence multipliers (1.0 is read from the stored results)
X_NET_FS = tuple(f for f in ("F42", "F33") if f in FS_RUN and FS_SPEC[f]["network"])
X_NET_REPRO_N = 2                                    # number of hospitals (per outcome) to compare with the stored network values (factor 1)
X_NET_REPRO_RTOL = R5_CSV_RTOL                       # relative tolerance of the E-2 reproduction check (1e-4, the same as the comparison with the stored csv)
X_NET_REPRO_ATOL = {"d_auroc": 1e-7, "d_ece": 1e-7, "intercept": 1e-4, "slope": 1e-4}   # absolute tolerance per column. Basis: the noise of running the same computation with only the number of BLAS threads changed is at most d_auroc 0, d_ece 4.1e-10, intercept 7.6e-6, slope 5.1e-6, and the differences of a wrong computation (different random stream key or neighbor count K) are at least d_auroc 2.7e-3, d_ece 1.4e-4, intercept 5.0e-3, slope 1.8e-3 (20 simulated units)
X_NET_ECE_SHARE = 0.50                               # E2-1: lower bound of the fraction of hospitals with better ECE than frozen
X_NET_E23_SHARE = 0.50                              # E2-3: fraction of harmful cells (delta AUROC negative and delta ECE positive)
X_NET_ALPHA_LO, X_NET_ALPHA_HI = 0.05, 0.50          # range summarized separately inside the mixing-weight grid (0.05 to 0.50 of A7)
X_NAMES = {"fz": "R5_x_fz", "net": "R5_x_net"}
X_NEW_NAMES = tuple([X_NAMES["fz"]] + [f"{X_NAMES['net']}_{fs}" for fs in X_NET_FS])
R5_NEW_NAMES += X_NEW_NAMES                          # the write lock of stage A also permits the new names (other names stay blocked)
R5X_MAP_ROWS = []
print("extra computation settings: on N back-calculation %s | S5 Panel A %s | E-1 %s | E-2 %s | partner prevalence multipliers %s | feature sets %s" % (RUN_X_A9INV, RUN_X_S5A, RUN_X_FZ, RUN_X_NET, X_NET_FACTORS, list(X_NET_FS)))
print("  new resume names written (extra): %s" % (list(X_NEW_NAMES),))


In [ ]:
# ================================================================
# PRAM input collection  settings (values of the stored results are read if present, otherwise recomputed in this notebook with the same function)
# ================================================================
R5P_NAMES = {"rescore": "R5_p_rescore_F42", "a2tail": "R5_p_a2tail_F33", "matrix_F42": "R5_p_matrix_F42", "matrix_F33": "R5_p_matrix_F33",
             "hospital_F42": "R5_p_hospital_F42", "hospital_F33": "R5_p_hospital_F33", "network_F42": "R5_p_network_F42", "network_F33": "R5_p_network_F33"}
R5_NEW_NAMES += tuple(R5P_NAMES.values())            # the write lock also permits the new names (other names stay blocked)
R5P_GROUPS = ("frozen", "pram", "recal", "local")    # groups that produce the four methods B reads for comparison. Other groups do not affect the values of these four methods
R5P_KEEP = ("b1_base", "pram_b1", "platt_b1", "local_lr")
R5P_ALL_BANKS = bool(RUN_B_IW or RUN_B_CORAL or RUN_B_ANCHOR)       # if any B is on, all grid bank sizes, otherwise only the full bank ("full") read by the (a) comparison and E-1
R5P_NEED_HOSP = bool(RUN_B_IW or RUN_B_CORAL or RUN_B_ANCHOR)       # the hospital-level comparison targets are read only by B
R5P_NET_FACTOR = 1.0                                 # observed-value multipliers compared by E-2 (the values used by A7 of notebook 01)
R5P_A9_ROUND = 6                                     # the csv constants have 6 significant digits. When back-calculating with csv constants, the candidate values are also rounded to the same digits before comparing
R5P_RESCORE_FS = "F42"
R5P_A2_FS = "F33"
print("input collection settings: values of the stored results are read if present (per unit) and otherwise recomputed with the same function | computation groups %s | methods %s | if B is on, grid full bank %s | hospital level %s" % (
    list(R5P_GROUPS), list(R5P_KEEP), R5P_ALL_BANKS, R5P_NEED_HOSP))
print("  new resume names written (input collection): %s" % (list(R5P_NAMES.values()),))


In [ ]:
# ================================================================
# PRAM constants for comparison taken from the result csv of the stored results (aggregate values only)
# ================================================================
# From the github_csv of the stored results, only the median over draws per cell (outcome x target x bank size x strategy), the distribution summary over all hospitals, Table 2 q and the 33-column eICU tail summary were taken.
# No patient-level rows, hospital ids or hospital codes are included. When the stored resume is absent from this folder and the values are recomputed, these are used only to check whether the recomputed values equal the stored values.
# source csv and sha256:
#   A2_tail_draws_F33.csv  a99003c64a7b2609c38f857a003c99288011e4d4856f8b5df353e3e4e5f6df3c
#   A2_tail_summary.csv  3ad4c28619a673ee74ab6a740e94ab18be76f78366377881a8b50aea7aad1a6d
#   A7_network_F33.csv  458c05519580113ee5cbb404c8b39272da4e1b3d72b8f3b9a8405ce05660a76a
#   A7_network_F42.csv  18697875d54df31b4daf91871294031d14cdd4c346828a245e79ac1846aa4695
#   A9_tabpfn_draws.csv  338d449186bffe2ee48b7884ff7f5a400ca27ae28ed434664b442adf955da352
#   RESCORE_draws_F42.csv  92efcceebc354167c4e4100e345bd0648c84788045a62711d510cddb64f1679d
#   RESCORE_table2_q.csv  4442cebd2211c59de32dd0a700dc1f3e072850f03526b19f9e6c4c647a4f4561
#   hospital_results_F33.csv  15a4c6ade3fbe1d4cc0d484713825d62e30cb41e79bc990339916dcb00ac5419
#   hospital_results_F42.csv  f23536f956865fc37f3a51c737f637e06ded670755a9ff42f03aa4e98e44aff2
#   matrix_draws_F33.csv  8eea4ec03dc6020c60c9b90cf7ad304a1b4b53224258266183c04b5f0cd73504
#   matrix_draws_F42.csv  54f56c2b14c44340d0ee1d05599352549116fc80a111f23463106411e15e0e5d
R5_REF_SOURCES = {"A2_tail_draws_F33.csv": "a99003c64a7b2609c38f857a003c99288011e4d4856f8b5df353e3e4e5f6df3c", "A2_tail_summary.csv": "d42a44077b65b052c3e014ccc2b75ba846e47b6c3f25d03c53a7d8ca9399f31f", "A7_network_F33.csv": "0e7234a8756017909a51d8d0c4b655991d2253988b6e909b9e781989b467888d", "A7_network_F42.csv": "1d79f181ae778639396f74f3840882e2e71f8a60efc818feef94de46266a629e", "A9_tabpfn_draws.csv": "338d449186bffe2ee48b7884ff7f5a400ca27ae28ed434664b442adf955da352", "RESCORE_draws_F42.csv": "92efcceebc354167c4e4100e345bd0648c84788045a62711d510cddb64f1679d", "RESCORE_table2_q.csv": "e2b95c5d6fa1d9e5067f2157a50974a000e474f2a054d5a0cbc50e198d00523d", "hospital_results_F33.csv": "1b6354107b472b9dc68537c5e0fb67f62f72f0debaf4acae117c0a18d538a2d0", "hospital_results_F42.csv": "8598ed879bf8c4cdf94d2f409725fb42fae2357c1575b068bf699fe1b7b1b502", "matrix_draws_F33.csv": "8eea4ec03dc6020c60c9b90cf7ad304a1b4b53224258266183c04b5f0cd73504", "matrix_draws_F42.csv": "54f56c2b14c44340d0ee1d05599352549116fc80a111f23463106411e15e0e5d"}

_R5_REF_DEFAULT = {
    'matrix_cols': ['auroc', 'ece', 'slope', 'brier', 'nb@0.2', 'n_rep'],
    'matrix': {
        'F33|AKI_168h|MIMIC-III|1000|b1_base': [0.714687, 0.0139087, 1.00939, 0.120529, 0.0428724, 10],
        'F33|AKI_168h|MIMIC-III|1000|local_lr': [0.687896, 0.0196275, 0.784133, 0.125016, 0.0348026, 10],
        'F33|AKI_168h|MIMIC-III|1000|platt_b1': [0.714316, 0.0114646, 1.00203, 0.1204435, 0.04244175, 10],
        'F33|AKI_168h|MIMIC-III|1000|pram_b1': [0.717446, 0.0173048, 1.156875, 0.120866, 0.04268295, 10],
        'F33|AKI_168h|MIMIC-III|2000|b1_base': [0.714687, 0.0139087, 1.00939, 0.120529, 0.0428724, 10],
        'F33|AKI_168h|MIMIC-III|2000|local_lr': [0.702855, 0.0131448, 0.8988915, 0.122333, 0.03887625, 10],
        'F33|AKI_168h|MIMIC-III|2000|platt_b1': [0.7142595, 0.0113251, 0.9926805, 0.1203495, 0.0425882, 10],
        'F33|AKI_168h|MIMIC-III|2000|pram_b1': [0.719186, 0.01746445, 1.172615, 0.120693, 0.04318245, 10],
        'F33|AKI_168h|MIMIC-III|5000|b1_base': [0.714687, 0.0139087, 1.00939, 0.120529, 0.0428724, 10],
        'F33|AKI_168h|MIMIC-III|5000|local_lr': [0.71219, 0.0118546, 0.974447, 0.1204625, 0.04146855, 10],
        'F33|AKI_168h|MIMIC-III|5000|platt_b1': [0.7145145, 0.0112453, 0.9976355, 0.1203145, 0.0424418, 10],
        'F33|AKI_168h|MIMIC-III|5000|pram_b1': [0.723223, 0.01676555, 1.2119, 0.1204495, 0.04353555, 10],
        'F33|AKI_168h|MIMIC-III|500|b1_base': [0.714687, 0.0139087, 1.00939, 0.120529, 0.0428724, 10],
        'F33|AKI_168h|MIMIC-III|500|local_lr': [0.664722, 0.03435225, 0.5618615, 0.130421, 0.02927345, 10],
        'F33|AKI_168h|MIMIC-III|500|platt_b1': [0.713742, 0.01264665, 0.976679, 0.120481, 0.04232985, 10],
        'F33|AKI_168h|MIMIC-III|500|pram_b1': [0.7149095, 0.0175415, 1.14567, 0.1212515, 0.04282075, 10],
        'F33|AKI_168h|MIMIC-III|full|b1_base': [0.714687, 0.0139087, 1.00939, 0.120529, 0.0428724, 1],
        'F33|AKI_168h|MIMIC-III|full|local_lr': [0.717034, 0.0100781, 0.991118, 0.119836, 0.0426313, 1],
        'F33|AKI_168h|MIMIC-III|full|platt_b1': [0.714575, 0.0113607, 0.999351, 0.120311, 0.0423729, 1],
        'F33|AKI_168h|MIMIC-III|full|pram_b1': [0.728638, 0.0189298, 1.24335, 0.119955, 0.044681, 1],
        'F33|AKI_168h|MIMIC-IV|1000|b1_base': [0.731495, 0.00842355, 1.0395, 0.111649, 0.0362658, 10],
        'F33|AKI_168h|MIMIC-IV|1000|local_lr': [0.6963395, 0.01724465, 0.767737, 0.116065, 0.0295407, 10],
        'F33|AKI_168h|MIMIC-IV|1000|platt_b1': [0.730854, 0.00756247, 0.992991, 0.111697, 0.0361662, 10],
        'F33|AKI_168h|MIMIC-IV|1000|pram_b1': [0.7307155, 0.0152262, 1.135755, 0.11191, 0.0365398, 10],
        'F33|AKI_168h|MIMIC-IV|2000|b1_base': [0.731495, 0.00842355, 1.0395, 0.111649, 0.0362658, 10],
        'F33|AKI_168h|MIMIC-IV|2000|local_lr': [0.715349, 0.00999392, 0.910506, 0.113382, 0.03322705, 10],
        'F33|AKI_168h|MIMIC-IV|2000|platt_b1': [0.7310225, 0.00799018, 1.008481, 0.111734, 0.03609145, 10],
        'F33|AKI_168h|MIMIC-IV|2000|pram_b1': [0.733063, 0.0168385, 1.15715, 0.111704, 0.03733685, 10],
        'F33|AKI_168h|MIMIC-IV|5000|b1_base': [0.731495, 0.00842355, 1.0395, 0.111649, 0.0362658, 10],
        'F33|AKI_168h|MIMIC-IV|5000|local_lr': [0.7274345, 0.00655444, 0.983728, 0.111701, 0.0362409, 10],
        'F33|AKI_168h|MIMIC-IV|5000|platt_b1': [0.731259, 0.00705125, 1.001855, 0.1116795, 0.0361164, 10],
        'F33|AKI_168h|MIMIC-IV|5000|pram_b1': [0.7356095, 0.01725435, 1.18976, 0.111484, 0.03747385, 10],
        'F33|AKI_168h|MIMIC-IV|500|b1_base': [0.731495, 0.00842355, 1.0395, 0.111649, 0.0362658, 10],
        'F33|AKI_168h|MIMIC-IV|500|local_lr': [0.668792, 0.03187615, 0.570045, 0.120818, 0.0224046, 10],
        'F33|AKI_168h|MIMIC-IV|500|platt_b1': [0.730014, 0.009512335, 1.00441, 0.1120495, 0.03640285, 10],
        'F33|AKI_168h|MIMIC-IV|500|pram_b1': [0.730104, 0.01531845, 1.14782, 0.112023, 0.03652735, 10],
        'F33|AKI_168h|MIMIC-IV|full|b1_base': [0.731495, 0.00842355, 1.0395, 0.111649, 0.0362658, 1],
        'F33|AKI_168h|MIMIC-IV|full|local_lr': [0.730678, 0.00585275, 0.991509, 0.111404, 0.0366643, 1],
        'F33|AKI_168h|MIMIC-IV|full|platt_b1': [0.731324, 0.006394, 0.999974, 0.111657, 0.0359171, 1],
        'F33|AKI_168h|MIMIC-IV|full|pram_b1': [0.737037, 0.0186363, 1.19789, 0.11135, 0.0381588, 1],
        'F33|AKI_168h|eICU|1000|b1_base': [0.703355, 0.0365532, 1.17787, 0.0558203, 0.00270425, 10],
        'F33|AKI_168h|eICU|1000|local_lr': [0.6525205, 0.01620835, 0.564402, 0.057171, -0.000848753, 10],
        'F33|AKI_168h|eICU|1000|platt_b1': [0.7012675, 0.003155565, 0.9925705, 0.05456935, 0.003066165, 10],
        'F33|AKI_168h|eICU|1000|pram_b1': [0.702634, 0.027156, 1.24373, 0.0553327, 0.00312128, 10],
        'F33|AKI_168h|eICU|2000|b1_base': [0.703355, 0.0365532, 1.17787, 0.0558203, 0.00270425, 10],
        'F33|AKI_168h|eICU|2000|local_lr': [0.682474, 0.0066412, 0.7628785, 0.0557247, 0.001181275, 10],
        'F33|AKI_168h|eICU|2000|platt_b1': [0.7027175, 0.002422255, 1.00949, 0.0544878, 0.00323518, 10],
        'F33|AKI_168h|eICU|2000|pram_b1': [0.706545, 0.02647965, 1.273195, 0.0552004, 0.00324988, 10],
        'F33|AKI_168h|eICU|5000|b1_base': [0.703355, 0.0365532, 1.17787, 0.0558203, 0.00270425, 10],
        'F33|AKI_168h|eICU|5000|local_lr': [0.7087175, 0.00229118, 0.917321, 0.05458185, 0.00285122, 10],
        'F33|AKI_168h|eICU|5000|platt_b1': [0.702966, 0.002296155, 1.0042285, 0.0544685, 0.0032756, 10],
        'F33|AKI_168h|eICU|5000|pram_b1': [0.7099715, 0.0260844, 1.309975, 0.0550277, 0.00325539, 10],
        'F33|AKI_168h|eICU|500|b1_base': [0.703355, 0.0365532, 1.17787, 0.0558203, 0.00270425, 10],
        'F33|AKI_168h|eICU|500|local_lr': [0.6051065, 0.02619705, 0.342637, 0.05961315, -0.00378632, 10],
        'F33|AKI_168h|eICU|500|platt_b1': [0.7004295, 0.005535845, 1.171495, 0.0546825, 0.002522375, 10],
        'F33|AKI_168h|eICU|500|pram_b1': [0.703199, 0.02828905, 1.227185, 0.05537905, 0.0029541, 10],
        'F33|AKI_168h|eICU|full|b1_base': [0.703355, 0.0365532, 1.17787, 0.0558203, 0.00270425, 1],
        'F33|AKI_168h|eICU|full|local_lr': [0.727542, 0.00259053, 0.994818, 0.0539923, 0.00419967, 1],
        'F33|AKI_168h|eICU|full|platt_b1': [0.70321, 0.00146467, 0.998801, 0.0544425, 0.00321497, 1],
        'F33|AKI_168h|eICU|full|pram_b1': [0.715848, 0.0249221, 1.32808, 0.0548525, 0.00335827, 1],
        'F33|mortality_168h|MIMIC-III|1000|b1_base': [0.855046, 0.0136973, 0.894915, 0.0409163, 0.005288, 10],
        'F33|mortality_168h|MIMIC-III|1000|local_lr': [0.8074095, 0.01625185, 0.683796, 0.0434985, 0.0025062, 10],
        'F33|mortality_168h|MIMIC-III|1000|platt_b1': [0.8539835, 0.006915355, 0.9846735, 0.04072085, 0.00543441, 10],
        'F33|mortality_168h|MIMIC-III|1000|pram_b1': [0.8552585, 0.01110565, 0.960192, 0.0406704, 0.005124365, 10],
        'F33|mortality_168h|MIMIC-III|2000|b1_base': [0.855046, 0.0136973, 0.894915, 0.0409163, 0.005288, 10],
        'F33|mortality_168h|MIMIC-III|2000|local_lr': [0.8338085, 0.010309, 0.8137755, 0.0421322, 0.00381528, 10],
        'F33|mortality_168h|MIMIC-III|2000|platt_b1': [0.8545365, 0.006302655, 1.01881, 0.04067575, 0.00558082, 10],
        'F33|mortality_168h|MIMIC-III|2000|pram_b1': [0.8554335, 0.01077155, 0.9592485, 0.0405981, 0.005244935, 10],
        'F33|mortality_168h|MIMIC-III|5000|b1_base': [0.855046, 0.0136973, 0.894915, 0.0409163, 0.005288, 10],
        'F33|mortality_168h|MIMIC-III|5000|local_lr': [0.8503535, 0.007025985, 0.9212755, 0.0409841, 0.00486599, 10],
        'F33|mortality_168h|MIMIC-III|5000|platt_b1': [0.8549575, 0.00587128, 0.99568, 0.04063845, 0.00580474, 10],
        'F33|mortality_168h|MIMIC-III|5000|pram_b1': [0.8583925, 0.01006839, 0.979567, 0.04044905, 0.00539996, 10],
        'F33|mortality_168h|MIMIC-III|500|b1_base': [0.855046, 0.0136973, 0.894915, 0.0409163, 0.005288, 10],
        'F33|mortality_168h|MIMIC-III|500|local_lr': [0.7774755, 0.0218499, 0.540099, 0.04596985, 0.00140382, 10],
        'F33|mortality_168h|MIMIC-III|500|platt_b1': [0.8527415, 0.007658365, 1.014195, 0.04074675, 0.005580815, 10],
        'F33|mortality_168h|MIMIC-III|500|pram_b1': [0.852478, 0.0109725, 0.967146, 0.0406847, 0.005029625, 10],
        'F33|mortality_168h|MIMIC-III|full|b1_base': [0.855046, 0.0136973, 0.894915, 0.0409163, 0.005288, 1],
        'F33|mortality_168h|MIMIC-III|full|local_lr': [0.854588, 0.00541348, 0.968468, 0.0406065, 0.00551192, 1],
        'F33|mortality_168h|MIMIC-III|full|platt_b1': [0.855012, 0.00558538, 1.00013, 0.0406131, 0.00571862, 1],
        'F33|mortality_168h|MIMIC-III|full|pram_b1': [0.860887, 0.00957324, 0.999052, 0.0402573, 0.00568417, 1],
        'F33|mortality_168h|MIMIC-IV|1000|b1_base': [0.876298, 0.00382765, 1.02719, 0.0356134, 0.00670021, 10],
        'F33|mortality_168h|MIMIC-IV|1000|local_lr': [0.8349675, 0.0147281, 0.691689, 0.03877515, 0.004408685, 10],
        'F33|mortality_168h|MIMIC-IV|1000|platt_b1': [0.8755955, 0.00536311, 0.999441, 0.03570755, 0.00670021, 10],
        'F33|mortality_168h|MIMIC-IV|1000|pram_b1': [0.876344, 0.00453251, 1.077875, 0.03561675, 0.006725115, 10],
        'F33|mortality_168h|MIMIC-IV|2000|b1_base': [0.876298, 0.00382765, 1.02719, 0.0356134, 0.00670021, 10],
        'F33|mortality_168h|MIMIC-IV|2000|local_lr': [0.8516055, 0.009316365, 0.8160825, 0.03751045, 0.00549218, 10],
        'F33|mortality_168h|MIMIC-IV|2000|platt_b1': [0.8759955, 0.00506932, 1.0003335, 0.0357031, 0.00655076, 10],
        'F33|mortality_168h|MIMIC-IV|2000|pram_b1': [0.87655, 0.00465362, 1.081875, 0.0355544, 0.006750025, 10],
        'F33|mortality_168h|MIMIC-IV|5000|b1_base': [0.876298, 0.00382765, 1.02719, 0.0356134, 0.00670021, 10],
        'F33|mortality_168h|MIMIC-IV|5000|local_lr': [0.8642995, 0.00712023, 0.905457, 0.03667245, 0.005616715, 10],
        'F33|mortality_168h|MIMIC-IV|5000|platt_b1': [0.876018, 0.00483812, 0.9981705, 0.0356741, 0.00663794, 10],
        'F33|mortality_168h|MIMIC-IV|5000|pram_b1': [0.8787905, 0.00597133, 1.09999, 0.0355006, 0.006812295, 10],
        'F33|mortality_168h|MIMIC-IV|500|b1_base': [0.876298, 0.00382765, 1.02719, 0.0356134, 0.00670021, 10],
        'F33|mortality_168h|MIMIC-IV|500|local_lr': [0.802883, 0.02305115, 0.522773, 0.0424706, 0.00193036, 10],
        'F33|mortality_168h|MIMIC-IV|500|platt_b1': [0.8736975, 0.006438885, 1.001988, 0.03589375, 0.00679984, 10],
        'F33|mortality_168h|MIMIC-IV|500|pram_b1': [0.8755795, 0.003824345, 1.064355, 0.03558755, 0.00677493, 10],
        'F33|mortality_168h|MIMIC-IV|full|b1_base': [0.876298, 0.00382765, 1.02719, 0.0356134, 0.00670021, 1],
        'F33|mortality_168h|MIMIC-IV|full|local_lr': [0.868845, 0.00569154, 0.938158, 0.0364345, 0.00612733, 1],
        'F33|mortality_168h|MIMIC-IV|full|platt_b1': [0.876107, 0.0046255, 0.99964, 0.0356827, 0.00657567, 1],
        'F33|mortality_168h|MIMIC-IV|full|pram_b1': [0.879957, 0.00593733, 1.0989, 0.035486, 0.00692438, 1],
        'F33|mortality_168h|eICU|1000|b1_base': [0.811718, 0.0247324, 0.743675, 0.04505, -0.000804662, 10],
        'F33|mortality_168h|eICU|1000|local_lr': [0.773129, 0.0163739, 0.652195, 0.04340365, 0.0006154375, 10],
        'F33|mortality_168h|eICU|1000|platt_b1': [0.8094065, 0.004338015, 0.988534, 0.04099685, 0.002070445, 10],
        'F33|mortality_168h|eICU|1000|pram_b1': [0.8131695, 0.0195711, 0.835144, 0.04302, 0.0002921035, 10],
        'F33|mortality_168h|eICU|2000|b1_base': [0.811718, 0.0247324, 0.743675, 0.04505, -0.000804662, 10],
        'F33|mortality_168h|eICU|2000|local_lr': [0.79581, 0.0088944, 0.818526, 0.041765, 0.001998795, 10],
        'F33|mortality_168h|eICU|2000|platt_b1': [0.811, 0.004203815, 1.0067, 0.0409378, 0.00194552, 10],
        'F33|mortality_168h|eICU|2000|pram_b1': [0.815024, 0.018297, 0.8698755, 0.0423678, 0.0010012325, 10],
        'F33|mortality_168h|eICU|5000|b1_base': [0.811718, 0.0247324, 0.743675, 0.04505, -0.000804662, 10],
        'F33|mortality_168h|eICU|5000|local_lr': [0.8121595, 0.00603807, 0.903687, 0.04102175, 0.00281264, 10],
        'F33|mortality_168h|eICU|5000|platt_b1': [0.810704, 0.00426792, 0.994968, 0.04095335, 0.0019694, 10],
        'F33|mortality_168h|eICU|5000|pram_b1': [0.817548, 0.0191428, 0.8568255, 0.0425853, 0.000834056, 10],
        'F33|mortality_168h|eICU|500|b1_base': [0.811718, 0.0247324, 0.743675, 0.04505, -0.000804662, 10],
        'F33|mortality_168h|eICU|500|local_lr': [0.7420665, 0.02403345, 0.5027875, 0.04552755, -0.0004850015, 10],
        'F33|mortality_168h|eICU|500|platt_b1': [0.808558, 0.00398559, 0.995333, 0.0411284, 0.00207963, 10],
        'F33|mortality_168h|eICU|500|pram_b1': [0.812256, 0.0180697, 0.8502025, 0.0427297, 0.000734851, 10],
        'F33|mortality_168h|eICU|full|b1_base': [0.811718, 0.0247324, 0.743675, 0.04505, -0.000804662, 1],
        'F33|mortality_168h|eICU|full|local_lr': [0.822702, 0.00350421, 0.988911, 0.0404223, 0.00374039, 1],
        'F33|mortality_168h|eICU|full|platt_b1': [0.811473, 0.00419614, 0.999886, 0.0409175, 0.00203554, 1],
        'F33|mortality_168h|eICU|full|pram_b1': [0.820733, 0.0162643, 0.921248, 0.0413376, 0.00196205, 1],
        'F42|AKI_168h|MIMIC-III|1000|b1_base': [0.722864, 0.0112148, 1.01715, 0.120087, 0.0435097, 10],
        'F42|AKI_168h|MIMIC-III|1000|local_lr': [0.6917795, 0.0221963, 0.696607, 0.125798, 0.0363873, 10],
        'F42|AKI_168h|MIMIC-III|1000|platt_b1': [0.7223235, 0.008603965, 0.9878035, 0.120064, 0.04289825, 10],
        'F42|AKI_168h|MIMIC-III|1000|pram_b1': [0.7264285, 0.0177834, 1.18816, 0.1203095, 0.0448274, 10],
        'F42|AKI_168h|MIMIC-III|2000|b1_base': [0.722864, 0.0112148, 1.01715, 0.120087, 0.0435097, 10],
        'F42|AKI_168h|MIMIC-III|2000|local_lr': [0.7093395, 0.0128692, 0.8601685, 0.122166, 0.0396772, 10],
        'F42|AKI_168h|MIMIC-III|2000|platt_b1': [0.7226255, 0.008769025, 0.9950875, 0.119944, 0.0427518, 10],
        'F42|AKI_168h|MIMIC-III|2000|pram_b1': [0.7282105, 0.0174822, 1.19476, 0.1200535, 0.0448274, 10],
        'F42|AKI_168h|MIMIC-III|5000|b1_base': [0.722864, 0.0112148, 1.01715, 0.120087, 0.0435097, 10],
        'F42|AKI_168h|MIMIC-III|5000|local_lr': [0.720871, 0.01034035, 0.9557415, 0.119831, 0.042881, 10],
        'F42|AKI_168h|MIMIC-III|5000|platt_b1': [0.722682, 0.00717728, 0.995589, 0.11992, 0.0428293, 10],
        'F42|AKI_168h|MIMIC-III|5000|pram_b1': [0.730015, 0.0184988, 1.21504, 0.1198165, 0.04556805, 10],
        'F42|AKI_168h|MIMIC-III|500|b1_base': [0.722864, 0.0112148, 1.01715, 0.120087, 0.0435097, 10],
        'F42|AKI_168h|MIMIC-III|500|local_lr': [0.6663975, 0.04192925, 0.484811, 0.131555, 0.0289548, 10],
        'F42|AKI_168h|MIMIC-III|500|platt_b1': [0.7217855, 0.00925941, 0.9803995, 0.1200965, 0.0430188, 10],
        'F42|AKI_168h|MIMIC-III|500|pram_b1': [0.723631, 0.0160388, 1.16162, 0.120502, 0.0438456, 10],
        'F42|AKI_168h|MIMIC-III|full|b1_base': [0.722864, 0.0112148, 1.01715, 0.120087, 0.0435097, 1],
        'F42|AKI_168h|MIMIC-III|full|local_lr': [0.725977, 0.00978656, 0.982362, 0.119022, 0.0439576, 1],
        'F42|AKI_168h|MIMIC-III|full|platt_b1': [0.722714, 0.00761839, 0.999285, 0.11991, 0.0427346, 1],
        'F42|AKI_168h|MIMIC-III|full|pram_b1': [0.733838, 0.0204262, 1.24075, 0.119414, 0.0467135, 1],
        'F42|AKI_168h|MIMIC-IV|1000|b1_base': [0.738873, 0.0105245, 1.03392, 0.110609, 0.0398276, 10],
        'F42|AKI_168h|MIMIC-IV|1000|local_lr': [0.698912, 0.0192152, 0.737685, 0.1158975, 0.0310601, 10],
        'F42|AKI_168h|MIMIC-IV|1000|platt_b1': [0.7382165, 0.00926677, 0.991143, 0.1106635, 0.0395661, 10],
        'F42|AKI_168h|MIMIC-IV|1000|pram_b1': [0.7383015, 0.01608645, 1.122645, 0.110826, 0.03980275, 10],
        'F42|AKI_168h|MIMIC-IV|2000|b1_base': [0.738873, 0.0105245, 1.03392, 0.110609, 0.0398276, 10],
        'F42|AKI_168h|MIMIC-IV|2000|local_lr': [0.721299, 0.01142495, 0.8587585, 0.112987, 0.0350578, 10],
        'F42|AKI_168h|MIMIC-IV|2000|platt_b1': [0.73844, 0.00906311, 0.991406, 0.1106915, 0.03988995, 10],
        'F42|AKI_168h|MIMIC-IV|2000|pram_b1': [0.7393255, 0.0155369, 1.145625, 0.110729, 0.0396782, 10],
        'F42|AKI_168h|MIMIC-IV|5000|b1_base': [0.738873, 0.0105245, 1.03392, 0.110609, 0.0398276, 10],
        'F42|AKI_168h|MIMIC-IV|5000|local_lr': [0.734021, 0.006924775, 0.965781, 0.110766, 0.0386196, 10],
        'F42|AKI_168h|MIMIC-IV|5000|platt_b1': [0.7386225, 0.00953196, 0.9987855, 0.110647, 0.0397778, 10],
        'F42|AKI_168h|MIMIC-IV|5000|pram_b1': [0.743232, 0.0188283, 1.186065, 0.110414, 0.04089865, 10],
        'F42|AKI_168h|MIMIC-IV|500|b1_base': [0.738873, 0.0105245, 1.03392, 0.110609, 0.0398276, 10],
        'F42|AKI_168h|MIMIC-IV|500|local_lr': [0.667137, 0.0384032, 0.505296, 0.1222495, 0.02358775, 10],
        'F42|AKI_168h|MIMIC-IV|500|platt_b1': [0.7369105, 0.01166945, 0.9931655, 0.11106, 0.03950385, 10],
        'F42|AKI_168h|MIMIC-IV|500|pram_b1': [0.737321, 0.01479235, 1.10924, 0.110907, 0.03971555, 10],
        'F42|AKI_168h|MIMIC-IV|full|b1_base': [0.738873, 0.0105245, 1.03392, 0.110609, 0.0398276, 1],
        'F42|AKI_168h|MIMIC-IV|full|local_lr': [0.737461, 0.0072861, 0.983126, 0.110312, 0.0394291, 1],
        'F42|AKI_168h|MIMIC-IV|full|platt_b1': [0.73867, 0.00916761, 0.999382, 0.110629, 0.0397031, 1],
        'F42|AKI_168h|MIMIC-IV|full|pram_b1': [0.744225, 0.0196691, 1.19623, 0.110297, 0.041347, 1],
        'F42|AKI_168h|eICU|1000|b1_base': [0.690389, 0.0425957, 0.834564, 0.0582808, -0.00146235, 10],
        'F42|AKI_168h|eICU|1000|local_lr': [0.6452715, 0.02025205, 0.4611585, 0.0583783, -0.001570745, 10],
        'F42|AKI_168h|eICU|1000|platt_b1': [0.689097, 0.003360975, 0.9515725, 0.05539675, 0.00148991, 10],
        'F42|AKI_168h|eICU|1000|pram_b1': [0.695611, 0.02871075, 0.9649305, 0.0564141, 0.0006099265, 10],
        'F42|AKI_168h|eICU|2000|b1_base': [0.690389, 0.0425957, 0.834564, 0.0582808, -0.00146235, 10],
        'F42|AKI_168h|eICU|2000|local_lr': [0.679637, 0.00974796, 0.6842345, 0.05621995, 0.0009755145, 10],
        'F42|AKI_168h|eICU|2000|platt_b1': [0.6897885, 0.004110075, 1.00455, 0.0553107, 0.001466025, 10],
        'F42|AKI_168h|eICU|2000|pram_b1': [0.698069, 0.02734625, 1.01023, 0.0560548, 0.0009920475, 10],
        'F42|AKI_168h|eICU|5000|b1_base': [0.690389, 0.0425957, 0.834564, 0.0582808, -0.00146235, 10],
        'F42|AKI_168h|eICU|5000|local_lr': [0.709372, 0.00341441, 0.8757735, 0.0546836, 0.0029688, 10],
        'F42|AKI_168h|eICU|5000|platt_b1': [0.6899855, 0.004033595, 1.001265, 0.0553091, 0.001480725, 10],
        'F42|AKI_168h|eICU|5000|pram_b1': [0.702758, 0.0257702, 1.05063, 0.0557694, 0.00127313, 10],
        'F42|AKI_168h|eICU|500|b1_base': [0.690389, 0.0425957, 0.834564, 0.0582808, -0.00146235, 10],
        'F42|AKI_168h|eICU|500|local_lr': [0.595421, 0.0343937, 0.2532185, 0.0613738, -0.00473795, 10],
        'F42|AKI_168h|eICU|500|platt_b1': [0.687742, 0.004467625, 1.07118, 0.05559, 0.00063932, 10],
        'F42|AKI_168h|eICU|500|pram_b1': [0.692896, 0.0299434, 0.9489545, 0.05644315, 0.000637483, 10],
        'F42|AKI_168h|eICU|full|b1_base': [0.690389, 0.0425957, 0.834564, 0.0582808, -0.00146235, 1],
        'F42|AKI_168h|eICU|full|local_lr': [0.731536, 0.00214903, 0.989258, 0.0538317, 0.00450831, 1],
        'F42|AKI_168h|eICU|full|platt_b1': [0.690235, 0.00376709, 1.00012, 0.0552735, 0.0015836, 1],
        'F42|AKI_168h|eICU|full|pram_b1': [0.710004, 0.0224868, 1.09394, 0.0553088, 0.00176364, 1],
        'F42|mortality_168h|MIMIC-III|1000|b1_base': [0.860025, 0.0144115, 0.906737, 0.0407909, 0.00582196, 10],
        'F42|mortality_168h|MIMIC-III|1000|local_lr': [0.803556, 0.0200647, 0.612966, 0.04455895, 0.00203252, 10],
        'F42|mortality_168h|MIMIC-III|1000|platt_b1': [0.8584715, 0.006886825, 0.973155, 0.04058565, 0.005890865, 10],
        'F42|mortality_168h|MIMIC-III|1000|pram_b1': [0.8603205, 0.0121904, 0.997746, 0.04049955, 0.00559804, 10],
        'F42|mortality_168h|MIMIC-III|2000|b1_base': [0.860025, 0.0144115, 0.906737, 0.0407909, 0.00582196, 10],
        'F42|mortality_168h|MIMIC-III|2000|local_lr': [0.835149, 0.0113511, 0.7920035, 0.0422817, 0.003711935, 10],
        'F42|mortality_168h|MIMIC-III|2000|platt_b1': [0.859492, 0.00649768, 1.011625, 0.040502, 0.00602005, 10],
        'F42|mortality_168h|MIMIC-III|2000|pram_b1': [0.8617585, 0.01220265, 0.991573, 0.04046025, 0.00575307, 10],
        'F42|mortality_168h|MIMIC-III|5000|b1_base': [0.860025, 0.0144115, 0.906737, 0.0407909, 0.00582196, 10],
        'F42|mortality_168h|MIMIC-III|5000|local_lr': [0.854071, 0.00810918, 0.8950235, 0.04109185, 0.00478848, 10],
        'F42|mortality_168h|MIMIC-III|5000|platt_b1': [0.859842, 0.005975665, 0.9935745, 0.04047915, 0.00597699, 10],
        'F42|mortality_168h|MIMIC-III|5000|pram_b1': [0.8640025, 0.01133275, 1.01092, 0.04028465, 0.005865025, 10],
        'F42|mortality_168h|MIMIC-III|500|b1_base': [0.860025, 0.0144115, 0.906737, 0.0407909, 0.00582196, 10],
        'F42|mortality_168h|MIMIC-III|500|local_lr': [0.7670765, 0.02842395, 0.464642, 0.0477176, 0.000232534, 10],
        'F42|mortality_168h|MIMIC-III|500|platt_b1': [0.8576195, 0.00760961, 1.01935, 0.0405057, 0.005572205, 10],
        'F42|mortality_168h|MIMIC-III|500|pram_b1': [0.858952, 0.01248215, 0.9818855, 0.04061165, 0.005486085, 10],
        'F42|mortality_168h|MIMIC-III|full|b1_base': [0.860025, 0.0144115, 0.906737, 0.0407909, 0.00582196, 1],
        'F42|mortality_168h|MIMIC-III|full|local_lr': [0.861125, 0.00689547, 0.961083, 0.0406172, 0.00595976, 1],
        'F42|mortality_168h|MIMIC-III|full|platt_b1': [0.859931, 0.00578186, 1.00008, 0.0404561, 0.00608034, 1],
        'F42|mortality_168h|MIMIC-III|full|pram_b1': [0.867523, 0.0109474, 1.02927, 0.0401186, 0.00608034, 1],
        'F42|mortality_168h|MIMIC-IV|1000|b1_base': [0.883348, 0.00629829, 1.00636, 0.0350369, 0.00769652, 10],
        'F42|mortality_168h|MIMIC-IV|1000|local_lr': [0.831435, 0.01677515, 0.628994, 0.03952645, 0.004558135, 10],
        'F42|mortality_168h|MIMIC-IV|1000|platt_b1': [0.8823535, 0.006255485, 1.0028595, 0.0351151, 0.00749726, 10],
        'F42|mortality_168h|MIMIC-IV|1000|pram_b1': [0.8833805, 0.006182245, 1.081645, 0.0349824, 0.007808605, 10],
        'F42|mortality_168h|MIMIC-IV|2000|b1_base': [0.883348, 0.00629829, 1.00636, 0.0350369, 0.00769652, 10],
        'F42|mortality_168h|MIMIC-IV|2000|local_lr': [0.853882, 0.01070455, 0.7649275, 0.0378433, 0.006052605, 10],
        'F42|mortality_168h|MIMIC-IV|2000|platt_b1': [0.8830475, 0.00615061, 1.00169, 0.0351097, 0.007708975, 10],
        'F42|mortality_168h|MIMIC-IV|2000|pram_b1': [0.883868, 0.00638048, 1.07894, 0.03498625, 0.00759689, 10],
        'F42|mortality_168h|MIMIC-IV|5000|b1_base': [0.883348, 0.00629829, 1.00636, 0.0350369, 0.00769652, 10],
        'F42|mortality_168h|MIMIC-IV|5000|local_lr': [0.8682615, 0.00656079, 0.8841645, 0.03642205, 0.007223275, 10],
        'F42|mortality_168h|MIMIC-IV|5000|platt_b1': [0.8830395, 0.006180515, 0.998156, 0.03506975, 0.007733885, 10],
        'F42|mortality_168h|MIMIC-IV|5000|pram_b1': [0.886729, 0.00651812, 1.105395, 0.0348893, 0.00787088, 10],
        'F42|mortality_168h|MIMIC-IV|500|b1_base': [0.883348, 0.00629829, 1.00636, 0.0350369, 0.00769652, 10],
        'F42|mortality_168h|MIMIC-IV|500|local_lr': [0.8025855, 0.0274719, 0.4669315, 0.0438399, 0.002341335, 10],
        'F42|mortality_168h|MIMIC-IV|500|platt_b1': [0.881129, 0.0075898, 1.015378, 0.03526005, 0.007771245, 10],
        'F42|mortality_168h|MIMIC-IV|500|pram_b1': [0.883048, 0.006479115, 1.06855, 0.0349912, 0.007721435, 10],
        'F42|mortality_168h|MIMIC-IV|full|b1_base': [0.883348, 0.00629829, 1.00636, 0.0350369, 0.00769652, 1],
        'F42|mortality_168h|MIMIC-IV|full|local_lr': [0.874527, 0.00633798, 0.916247, 0.036056, 0.00737272, 1],
        'F42|mortality_168h|MIMIC-IV|full|platt_b1': [0.883181, 0.0060652, 0.999723, 0.0350663, 0.00759689, 1],
        'F42|mortality_168h|MIMIC-IV|full|pram_b1': [0.886562, 0.00605129, 1.09423, 0.034893, 0.00782106, 1],
        'F42|mortality_168h|eICU|1000|b1_base': [0.774476, 0.0429705, 0.406383, 0.059085, -0.00686351, 10],
        'F42|mortality_168h|eICU|1000|local_lr': [0.7612335, 0.0206808, 0.549272, 0.04457805, 0.000529093, 10],
        'F42|mortality_168h|eICU|1000|platt_b1': [0.772643, 0.005353035, 0.949117, 0.0430492, -0.000288429, 10],
        'F42|mortality_168h|eICU|1000|pram_b1': [0.7943725, 0.0257413, 0.68743, 0.0465414, -0.003119445, 10],
        'F42|mortality_168h|eICU|2000|b1_base': [0.774476, 0.0429705, 0.406383, 0.059085, -0.00686351, 10],
        'F42|mortality_168h|eICU|2000|local_lr': [0.7954745, 0.0110928, 0.749279, 0.0421301, 0.00241766, 10],
        'F42|mortality_168h|eICU|2000|platt_b1': [0.773454, 0.005153245, 0.986226, 0.04291975, -0.0002700575, 10],
        'F42|mortality_168h|eICU|2000|pram_b1': [0.7989745, 0.023096, 0.760755, 0.0446736, -0.002346015, 10],
        'F42|mortality_168h|eICU|5000|b1_base': [0.774476, 0.0429705, 0.406383, 0.059085, -0.00686351, 10],
        'F42|mortality_168h|eICU|5000|local_lr': [0.8132965, 0.006395265, 0.87611, 0.0408249, 0.003687115, 10],
        'F42|mortality_168h|eICU|5000|platt_b1': [0.7735825, 0.00528644, 0.978089, 0.04291795, -0.0003361945, 10],
        'F42|mortality_168h|eICU|5000|pram_b1': [0.802877, 0.02253405, 0.77024, 0.0443022, -0.001956545, 10],
        'F42|mortality_168h|eICU|500|b1_base': [0.774476, 0.0429705, 0.406383, 0.059085, -0.00686351, 10],
        'F42|mortality_168h|eICU|500|local_lr': [0.721191, 0.03011055, 0.3929005, 0.04746465, -0.0010857415, 10],
        'F42|mortality_168h|eICU|500|platt_b1': [0.769555, 0.005258955, 0.9238195, 0.04320775, -0.0003086375, 10],
        'F42|mortality_168h|eICU|500|pram_b1': [0.7912805, 0.02530695, 0.689172, 0.04665795, -0.00319109, 10],
        'F42|mortality_168h|eICU|full|b1_base': [0.774476, 0.0429705, 0.406383, 0.059085, -0.00686351, 1],
        'F42|mortality_168h|eICU|full|local_lr': [0.826308, 0.0031857, 0.985655, 0.0399787, 0.00467733, 1],
        'F42|mortality_168h|eICU|full|platt_b1': [0.774243, 0.00508858, 0.997468, 0.0428697, -0.000411517, 1],
        'F42|mortality_168h|eICU|full|pram_b1': [0.810257, 0.0230365, 0.777538, 0.0438866, -0.00144398, 1],
    },
    'rescore_cols': ['d_auroc', 'var_standard', 'p_standard', 'n_rep'],
    'rescore': {
        'AKI_168h|MIMIC-III|1000|main': [0.003565265, 2.581165e-06, 0.0496513, 10],
        'AKI_168h|MIMIC-III|2000|main': [0.00534717, 2.978475e-06, 0.0010392285, 10],
        'AKI_168h|MIMIC-III|5000|main': [0.007151255, 3.472815e-06, 3.7142e-05, 10],
        'AKI_168h|MIMIC-III|500|main': [0.0007676915, 2.43197e-06, 0.3758695, 10],
        'AKI_168h|MIMIC-III|full|key08': [0.0109038, 4.34082e-06, 1.66328e-07, 1],
        'AKI_168h|MIMIC-III|full|main': [0.010975, 4.58629e-06, 2.9794e-07, 1],
        'AKI_168h|MIMIC-IV|1000|main': [-0.0005713935, 2.2563e-06, 0.228718, 10],
        'AKI_168h|MIMIC-IV|2000|main': [0.0004527625, 1.95372e-06, 0.436281, 10],
        'AKI_168h|MIMIC-IV|5000|main': [0.004359095, 2.504545e-06, 0.00456117, 10],
        'AKI_168h|MIMIC-IV|500|main': [-0.00155195, 2.2481e-06, 0.201953, 10],
        'AKI_168h|MIMIC-IV|full|key08': [0.00492054, 3.10868e-06, 0.00525821, 1],
        'AKI_168h|MIMIC-IV|full|main': [0.00535222, 2.96182e-06, 0.00187115, 1],
        'AKI_168h|eICU|1000|main': [0.00522261, 2.842525e-06, 0.001110717, 10],
        'AKI_168h|eICU|2000|main': [0.00768045, 3.22496e-06, 7.449415e-06, 10],
        'AKI_168h|eICU|5000|main': [0.012369, 3.44112e-06, 7.31696e-10, 10],
        'AKI_168h|eICU|500|main': [0.002507285, 2.50298e-06, 0.06566645, 10],
        'AKI_168h|eICU|full|key08': [0.0193588, 4.90959e-06, 2.39679e-18, 1],
        'AKI_168h|eICU|full|main': [0.0196158, 4.67587e-06, 1.17484e-19, 1],
        'mortality_168h|MIMIC-III|1000|main': [0.0002959328, 3.642995e-06, 0.3165145, 10],
        'mortality_168h|MIMIC-III|2000|main': [0.00173381, 2.47827e-06, 0.2579475, 10],
        'mortality_168h|MIMIC-III|5000|main': [0.003977855, 3.86674e-06, 0.03310695, 10],
        'mortality_168h|MIMIC-III|500|main': [-0.0010726135, 2.928365e-06, 0.415847, 10],
        'mortality_168h|MIMIC-III|full|key08': [0.00787497, 4.48201e-06, 0.000199428, 1],
        'mortality_168h|MIMIC-III|full|main': [0.00749804, 4.41917e-06, 0.000361383, 1],
        'mortality_168h|MIMIC-IV|1000|main': [3.277555e-05, 2.718335e-06, 0.6188225, 10],
        'mortality_168h|MIMIC-IV|2000|main': [0.000519999, 2.51473e-06, 0.463903, 10],
        'mortality_168h|MIMIC-IV|5000|main': [0.003381125, 3.188495e-06, 0.06248765, 10],
        'mortality_168h|MIMIC-IV|500|main': [-0.000300105, 1.968105e-06, 0.406847, 10],
        'mortality_168h|MIMIC-IV|full|key08': [0.00341986, 2.84472e-06, 0.0425982, 1],
        'mortality_168h|MIMIC-IV|full|main': [0.00321367, 2.4455e-06, 0.0398763, 1],
        'mortality_168h|eICU|1000|main': [0.0198965, 3.797285e-06, 1.462847e-24, 10],
        'mortality_168h|eICU|2000|main': [0.02449875, 4.603895e-06, 3.688735e-30, 10],
        'mortality_168h|eICU|5000|main': [0.02840115, 4.8715e-06, 7.242555e-39, 10],
        'mortality_168h|eICU|500|main': [0.0168049, 3.3642e-06, 8.457559e-22, 10],
        'mortality_168h|eICU|full|key08': [0.0357811, 5.40211e-06, 1.7756e-53, 1],
        'mortality_168h|eICU|full|main': [0.0357811, 5.40211e-06, 1.7756e-53, 1],
    },
    'table2_q': {
        'AKI_168h|MIMIC-III': 5.9588e-07,
        'AKI_168h|MIMIC-IV': 0.00224538,
        'AKI_168h|eICU': 3.52451e-19,
        'mortality_168h|MIMIC-III': 0.000542075,
        'mortality_168h|MIMIC-IV': 0.0398763,
        'mortality_168h|eICU': 1.06536e-52,
    },
    'a2tail_cols': ['auroc', 'd_auroc_base', 'var_auroc_base', 'n_rows'],
    'a2tail': {
        'F33|AKI_168h|eICU|1000|b1_base': [0.70873, None, None, 100],
        'F33|AKI_168h|eICU|1000|pram_b1': [0.7097135, 0.002247805, 8.035925e-06, 100],
        'F33|AKI_168h|eICU|2000|b1_base': [0.70873, None, None, 100],
        'F33|AKI_168h|eICU|2000|pram_b1': [0.711137, 0.003834085, 1.06757e-05, 100],
        'F33|AKI_168h|eICU|5000|b1_base': [0.70873, None, None, 100],
        'F33|AKI_168h|eICU|5000|pram_b1': [0.714427, 0.007249815, 1.0016965e-05, 100],
        'F33|AKI_168h|eICU|500|b1_base': [0.70873, None, None, 100],
        'F33|AKI_168h|eICU|500|pram_b1': [0.707945, 0.0, 4.805105e-06, 100],
        'F33|mortality_168h|eICU|1000|b1_base': [0.808278, None, None, 100],
        'F33|mortality_168h|eICU|1000|pram_b1': [0.8116255, 0.002011655, 2.638665e-06, 100],
        'F33|mortality_168h|eICU|2000|b1_base': [0.808278, None, None, 100],
        'F33|mortality_168h|eICU|2000|pram_b1': [0.811986, 0.00342764, 4.10079e-06, 100],
        'F33|mortality_168h|eICU|5000|b1_base': [0.808278, None, None, 100],
        'F33|mortality_168h|eICU|5000|pram_b1': [0.8141565, 0.00584249, 5.524765e-06, 100],
        'F33|mortality_168h|eICU|500|b1_base': [0.808278, None, None, 100],
        'F33|mortality_168h|eICU|500|pram_b1': [0.8093055, 0.000911944, 1.5058e-06, 100],
    },
    'a2tail_share': {
        'AKI_168h|eICU|1000': [0.02, 0.32],
        'AKI_168h|eICU|2000': [0.0, 0.43],
        'AKI_168h|eICU|500': [0.08, 0.18],
        'AKI_168h|eICU|5000': [0.0, 0.68],
        'mortality_168h|eICU|1000': [0.01, 0.36],
        'mortality_168h|eICU|2000': [0.0, 0.52],
        'mortality_168h|eICU|500': [0.05, 0.26],
        'mortality_168h|eICU|5000': [0.0, 0.77],
    },
    'a54': {
        'AKI_168h|MIMIC-III|1000': [100, 0.00322421, -0.00247304, -0.00535022, 0.01, 0.0, 0.0],
        'AKI_168h|MIMIC-III|2000': [100, 0.00572053, 0.0, -0.00441742, 0.0, 0.0, 0.0],
        'AKI_168h|MIMIC-III|500': [100, 0.00130618, -0.00289688, -0.00949473, 0.03, 0.0, 0.0],
        'AKI_168h|MIMIC-III|5000': [100, 0.00753084, 0.00319098, -0.00241229, 0.0, 0.0, 0.0],
        'AKI_168h|MIMIC-IV|1000': [100, 0.0, -0.00846444, -0.0106965, 0.08, 0.01, 0.0],
        'AKI_168h|MIMIC-IV|2000': [100, 0.0, -0.00505996, -0.0125832, 0.06, 0.02, 0.0],
        'AKI_168h|MIMIC-IV|500': [100, 0.0, -0.00705906, -0.0206655, 0.11, 0.03, 0.01],
        'AKI_168h|MIMIC-IV|5000': [100, 0.00466606, -0.00117358, -0.00251516, 0.0, 0.0, 0.0],
        'AKI_168h|eICU|1000': [100, 0.0072679, 0.0, -0.00164201, 0.0, 0.0, 0.0],
        'AKI_168h|eICU|2000': [100, 0.00885704, 0.0, -0.000471408, 0.0, 0.0, 0.0],
        'AKI_168h|eICU|500': [100, 0.00376046, -8.06487e-05, -0.00996787, 0.02, 0.0, 0.0],
        'AKI_168h|eICU|5000': [100, 0.0122787, 0.00597597, 0.00219109, 0.0, 0.0, 0.0],
        'mortality_168h|MIMIC-III|1000': [100, 0.0, -0.00147352, -0.0150658, 0.02, 0.01, 0.0],
        'mortality_168h|MIMIC-III|2000': [100, 0.00290065, -1.78192e-05, -0.0118663, 0.01, 0.01, 0.0],
        'mortality_168h|MIMIC-III|500': [100, 0.0, -0.00850316, -0.0164688, 0.13, 0.05, 0.0],
        'mortality_168h|MIMIC-III|5000': [100, 0.00515371, 0.000476498, -0.00223026, 0.0, 0.0, 0.0],
        'mortality_168h|MIMIC-IV|1000': [100, 0.0, -0.00623668, -0.011147, 0.07, 0.01, 0.0],
        'mortality_168h|MIMIC-IV|2000': [100, 0.000256199, -0.00177797, -0.0165009, 0.03, 0.01, 0.0],
        'mortality_168h|MIMIC-IV|500': [100, 0.0, -0.0135781, -0.0229945, 0.22, 0.07, 0.01],
        'mortality_168h|MIMIC-IV|5000': [100, 0.00302972, -0.000800485, -0.00136719, 0.0, 0.0, 0.0],
        'mortality_168h|eICU|1000': [100, 0.0220029, 0.0074715, 0.0, 0.0, 0.0, 0.0],
        'mortality_168h|eICU|2000': [100, 0.0247737, 0.0158801, 0.0105108, 0.0, 0.0, 0.0],
        'mortality_168h|eICU|500': [100, 0.0178039, 0.0, 0.0, 0.0, 0.0, 0.0],
        'mortality_168h|eICU|5000': [100, 0.0289135, 0.0203495, 0.0185362, 0.0, 0.0, 0.0],
    },
    'hospital_stats': ['n', 'mean', 'p10', 'median', 'p90', 'min', 'max'],
    'hospital': {
        'F33|AKI_168h|b1_base|auroc': [67, 0.69850699, 0.6118762, 0.709605, 0.7742212, 0.510417, 0.816286],
        'F33|AKI_168h|b1_base|d_auroc_b1': [67, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        'F33|AKI_168h|b1_base|ece': [67, 0.038139203, 0.02147178, 0.0350032, 0.06041728, 0.0153134, 0.0704044],
        'F33|AKI_168h|b1_base|nb@0.2': [67, 0.0029306987, -0.00456169, 0.00154639, 0.009338414, -0.0110577, 0.0287162],
        'F33|AKI_168h|local_lr|auroc': [67, 0.63781428, 0.5333886, 0.639737, 0.7083076, 0.40877, 0.834688],
        'F33|AKI_168h|local_lr|d_auroc_b1': [67, -0.060692704, -0.1391186, -0.0676153, 0.01683668, -0.208488, 0.150745],
        'F33|AKI_168h|local_lr|ece': [67, 0.034566856, 0.01440666, 0.0328668, 0.0575111, 0.00899773, 0.103715],
        'F33|AKI_168h|local_lr|nb@0.2': [67, -0.0024964737, -0.009391328, -0.00256598, 0.004357084, -0.0323625, 0.0297897],
        'F33|AKI_168h|platt_b1|auroc': [67, 0.68796894, 0.6060182, 0.702268, 0.770763, 0.395852, 0.812217],
        'F33|AKI_168h|platt_b1|d_auroc_b1': [67, -0.0105381, -0.02326012, -0.00396438, -0.0004792072, -0.154397, 0.00123686],
        'F33|AKI_168h|platt_b1|ece': [67, 0.0097595855, 0.003485378, 0.00808811, 0.01823498, 2.79067e-05, 0.030621],
        'F33|AKI_168h|platt_b1|nb@0.2': [67, 0.0039733628, -0.0010017378, 0.00263713, 0.010407426, -0.00466667, 0.0274493],
        'F33|AKI_168h|pram_b1|auroc': [67, 0.70827812, 0.6302258, 0.710474, 0.7688698, 0.566784, 0.822093],
        'F33|AKI_168h|pram_b1|d_auroc_b1': [67, 0.0097711436, -0.01618718, 0.00331942, 0.04556568, -0.0330652, 0.0969909],
        'F33|AKI_168h|pram_b1|ece': [67, 0.030333996, 0.01867824, 0.0278006, 0.04561006, 0.0100289, 0.0697987],
        'F33|AKI_168h|pram_b1|nb@0.2': [67, 0.0040148098, -0.002384138, 0.00347222, 0.009614958, -0.00623701, 0.0299831],
        'F33|mortality_168h|b1_base|auroc': [59, 0.81250781, 0.7321676, 0.820085, 0.8643294, 0.645386, 0.901742],
        'F33|mortality_168h|b1_base|d_auroc_b1': [59, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        'F33|mortality_168h|b1_base|ece': [59, 0.032507594, 0.01426896, 0.0298264, 0.05540184, 0.00837714, 0.0851895],
        'F33|mortality_168h|b1_base|nb@0.2': [59, 0.0012583123, -0.009573788, 0.00242981, 0.0129338, -0.0384283, 0.0221519],
        'F33|mortality_168h|local_lr|auroc': [59, 0.73805292, 0.631097, 0.746205, 0.8225274, 0.496953, 0.89266],
        'F33|mortality_168h|local_lr|d_auroc_b1': [59, -0.074454867, -0.1607006, -0.0610866, -0.01490108, -0.212527, 0.0510891],
        'F33|mortality_168h|local_lr|ece': [59, 0.030851334, 0.01421012, 0.0263037, 0.05309848, 0.011187, 0.0804636],
        'F33|mortality_168h|local_lr|nb@0.2': [59, 0.0016089032, -0.004781976, 0.00131694, 0.007204362, -0.0105882, 0.0229111],
        'F33|mortality_168h|platt_b1|auroc': [59, 0.806884, 0.7245964, 0.811922, 0.8619112, 0.641123, 0.888746],
        'F33|mortality_168h|platt_b1|d_auroc_b1': [59, -0.0056238688, -0.0129602, -0.0036513, -0.0008108946, -0.0315719, 0.000436904],
        'F33|mortality_168h|platt_b1|ece': [59, 0.011411203, 0.004584212, 0.0105426, 0.02042446, 0.00185899, 0.0243867],
        'F33|mortality_168h|platt_b1|nb@0.2': [59, 0.0052868062, -0.0010658376, 0.00317528, 0.01334688, -0.00359195, 0.0232068],
        'F33|mortality_168h|pram_b1|auroc': [59, 0.81383141, 0.740472, 0.823355, 0.8671202, 0.649541, 0.896772],
        'F33|mortality_168h|pram_b1|d_auroc_b1': [59, 0.0013235853, -0.01318834, -0.0010101, 0.02259442, -0.0197035, 0.0552008],
        'F33|mortality_168h|pram_b1|ece': [59, 0.025339175, 0.01224982, 0.0250222, 0.0424548, 0.00626075, 0.061366],
        'F33|mortality_168h|pram_b1|nb@0.2': [59, 0.0031338495, -0.005944082, 0.00280505, 0.01459398, -0.0189983, 0.021097],
        'F42|AKI_168h|b1_base|auroc': [67, 0.68895763, 0.6201516, 0.699173, 0.7448278, 0.480427, 0.782266],
        'F42|AKI_168h|b1_base|d_auroc_b1': [67, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        'F42|AKI_168h|b1_base|ece': [67, 0.041615404, 0.0221103, 0.0396973, 0.06487844, 0.0110213, 0.0825626],
        'F42|AKI_168h|b1_base|nb@0.2': [67, -0.00042330101, -0.009501022, -0.00144231, 0.008430706, -0.0208895, 0.0268879],
        'F42|AKI_168h|local_lr|auroc': [67, 0.63083152, 0.535661, 0.637871, 0.7014808, 0.421815, 0.838302],
        'F42|AKI_168h|local_lr|d_auroc_b1': [67, -0.058126003, -0.1409962, -0.0453761, 0.01722354, -0.264171, 0.167543],
        'F42|AKI_168h|local_lr|ece': [67, 0.041254585, 0.01735346, 0.0386856, 0.06685918, 0.011354, 0.122031],
        'F42|AKI_168h|local_lr|nb@0.2': [67, -0.0034106554, -0.01100162, -0.00367647, 0.006012938, -0.0347896, 0.0257009],
        'F42|AKI_168h|platt_b1|auroc': [67, 0.67975079, 0.61779, 0.695235, 0.7352558, 0.346334, 0.777547],
        'F42|AKI_168h|platt_b1|d_auroc_b1': [67, -0.0092068441, -0.01766186, -0.00509338, -0.0002329884, -0.134093, 0.0033526],
        'F42|AKI_168h|platt_b1|ece': [67, 0.0097870769, 0.002488446, 0.00861946, 0.01777514, 0.000550945, 0.0381657],
        'F42|AKI_168h|platt_b1|nb@0.2': [67, 0.0026020914, -0.002157034, 0.000877963, 0.009040226, -0.00597426, 0.0236486],
        'F42|AKI_168h|pram_b1|auroc': [67, 0.69765624, 0.6210724, 0.70512, 0.7559068, 0.565294, 0.808722],
        'F42|AKI_168h|pram_b1|d_auroc_b1': [67, 0.0086986703, -0.0118898, 0.00395756, 0.03337668, -0.0283644, 0.0848671],
        'F42|AKI_168h|pram_b1|ece': [67, 0.029503124, 0.01820424, 0.0273173, 0.04434278, 0.0109133, 0.068256],
        'F42|AKI_168h|pram_b1|nb@0.2': [67, 0.002228719, -0.004818146, 0.00123457, 0.009176826, -0.0117647, 0.0268879],
        'F42|mortality_168h|b1_base|auroc': [59, 0.77781914, 0.704434, 0.780214, 0.835408, 0.679376, 0.898516],
        'F42|mortality_168h|b1_base|d_auroc_b1': [59, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        'F42|mortality_168h|b1_base|ece': [59, 0.051266458, 0.02811566, 0.0478858, 0.08105754, 0.0147605, 0.113212],
        'F42|mortality_168h|b1_base|nb@0.2': [59, -0.004983852, -0.01912066, -0.00356415, 0.006684688, -0.0301455, 0.0234568],
        'F42|mortality_168h|local_lr|auroc': [59, 0.7330749, 0.6315074, 0.745083, 0.8258934, 0.480807, 0.847602],
        'F42|mortality_168h|local_lr|d_auroc_b1': [59, -0.044744204, -0.1407056, -0.0330181, 0.03670486, -0.236027, 0.0644808],
        'F42|mortality_168h|local_lr|ece': [59, 0.035107602, 0.01726114, 0.0319202, 0.05591382, 0.0105172, 0.0881684],
        'F42|mortality_168h|local_lr|nb@0.2': [59, 0.00096864617, -0.005949552, 0.00105485, 0.008136, -0.0129412, 0.0154987],
        'F42|mortality_168h|platt_b1|auroc': [59, 0.77073258, 0.6922464, 0.772127, 0.831288, 0.660434, 0.894184],
        'F42|mortality_168h|platt_b1|d_auroc_b1': [59, -0.0070865164, -0.02059394, -0.00430063, 0.0002630366, -0.0300045, 0.00221571],
        'F42|mortality_168h|platt_b1|ece': [59, 0.012451668, 0.004657374, 0.010299, 0.02367684, 0.00237858, 0.0374262],
        'F42|mortality_168h|platt_b1|nb@0.2': [59, 0.0011366515, -0.003113236, 0.0, 0.005870874, -0.00588235, 0.0209877],
        'F42|mortality_168h|pram_b1|auroc': [59, 0.7964261, 0.728313, 0.806211, 0.8416692, 0.707677, 0.889852],
        'F42|mortality_168h|pram_b1|d_auroc_b1': [59, 0.01860695, -0.00981854, 0.0151526, 0.05607582, -0.0211441, 0.0758654],
        'F42|mortality_168h|pram_b1|ece': [59, 0.0317385, 0.01604468, 0.0306776, 0.04992964, 0.0107299, 0.0891173],
        'F42|mortality_168h|pram_b1|nb@0.2': [59, -0.0014971475, -0.01056298, -0.00175316, 0.009871408, -0.027027, 0.0179012],
    },
    'network': {
        'F33|AKI_168h|base_slope': [67, 1.2370619, 0.5854864, 1.17969, 2.191082, -0.20879, 2.62427],
        'F33|AKI_168h|d_auroc': [67, 0.013161624, -0.004664506, 0.010101, 0.03162748, -0.0615079, 0.0896552],
        'F33|AKI_168h|d_ece': [67, -0.0055906375, -0.0152066, -0.00710827, 0.006090118, -0.0242503, 0.0203435],
        'F33|AKI_168h|d_nb_rep': [67, 0.0027486345, -0.003745816, 0.00314654, 0.008185336, -0.00926989, 0.0123419],
        'F33|AKI_168h|intercept': [67, 0.7124307, -0.8950872, 0.659351, 2.326074, -2.77238, 4.11188],
        'F33|AKI_168h|prev_diff': [67, -0.00012515299, -0.01853754, 0.00374943, 0.02265106, -0.0609013, 0.0302948],
        'F33|AKI_168h|slope': [67, 1.5124372, 0.7486976, 1.4115, 2.558026, -0.0569218, 2.96658],
        'F33|mortality_168h|base_slope': [59, 0.81605122, 0.5789666, 0.795331, 1.14089, 0.260363, 1.56703],
        'F33|mortality_168h|d_auroc': [59, 0.0071856791, -0.005882728, 0.00428571, 0.02496348, -0.0362369, 0.0403509],
        'F33|mortality_168h|d_ece': [59, -0.0066037089, -0.01721038, -0.0061834, 0.002903718, -0.0327349, 0.0107614],
        'F33|mortality_168h|d_nb_rep': [59, 0.0011827835, -0.003951446, 0.00153374, 0.00663063, -0.0192308, 0.0103306],
        'F33|mortality_168h|intercept': [59, -0.34023909, -1.312168, -0.290002, 0.5284842, -2.00221, 1.31222],
        'F33|mortality_168h|prev_diff': [59, -0.0003016273, -0.01843756, 0.00132887, 0.01942866, -0.051248, 0.0369645],
        'F33|mortality_168h|slope': [59, 1.029544, 0.7456482, 1.00187, 1.380304, 0.319746, 2.028],
        'F42|AKI_168h|base_slope': [67, 0.86706693, 0.3657442, 0.86185, 1.516638, -0.247322, 1.9556],
        'F42|AKI_168h|d_auroc': [67, 0.017576224, -0.006804438, 0.0167931, 0.03730846, -0.0277778, 0.113793],
        'F42|AKI_168h|d_ece': [67, -0.008771876, -0.01717854, -0.00959804, 0.001441952, -0.0301185, 0.0126205],
        'F42|AKI_168h|d_nb_rep': [67, 0.0027658511, -0.0002886772, 0.00242392, 0.00731011, -0.00776905, 0.0142575],
        'F42|AKI_168h|intercept': [67, -0.12919573, -1.405504, -0.258795, 1.477774, -3.23668, 2.77927],
        'F42|AKI_168h|prev_diff': [67, -0.00012515299, -0.01853754, 0.00374943, 0.02265106, -0.0609013, 0.0302948],
        'F42|AKI_168h|slope': [67, 1.1382659, 0.5762382, 1.05944, 1.904432, -0.162618, 2.36238],
        'F42|mortality_168h|base_slope': [59, 0.46520637, 0.2526004, 0.480514, 0.6677342, 0.153418, 0.955799],
        'F42|mortality_168h|d_auroc': [59, 0.02759799, -0.0005698, 0.0173611, 0.07008276, -0.0170648, 0.150485],
        'F42|mortality_168h|d_ece': [59, -0.012587086, -0.02289384, -0.0117947, -0.0001184204, -0.0335053, 0.00516258],
        'F42|mortality_168h|d_nb_rep': [59, 0.0020745704, -0.005964078, 0.00307377, 0.007869434, -0.012069, 0.0138889],
        'F42|mortality_168h|intercept': [59, -1.102689, -1.841872, -1.0763, -0.4025352, -2.40126, 0.120128],
        'F42|mortality_168h|prev_diff': [59, -0.0003016273, -0.01843756, 0.00132887, 0.01942866, -0.051248, 0.0369645],
        'F42|mortality_168h|slope': [59, 0.74156093, 0.4426762, 0.756895, 0.9898132, 0.326338, 1.17011],
    },
    'a9': {'AKI_168h|MIMIC-III': {'obs_rate': 0.161086, 'auroc': 0.722864, 'brier': 0.120087, 'ece': 0.0112148, 'slope': 1.01715, 'intercept': 0.113366, 'mean_pred': 0.150915, 'banks': ['1000', '500', '5000', 'full'], 'n_rep': 10, 'reduced': True}, 'AKI_168h|eICU': {'obs_rate': 0.0609208, 'auroc': 0.686022, 'brier': 0.0586151, 'ece': 0.0423256, 'slope': 0.809234, 'intercept': -0.981859, 'mean_pred': 0.103246, 'banks': ['1000', '500', '5000', 'full'], 'n_rep': 10, 'reduced': True}, 'mortality_168h|MIMIC-III': {'obs_rate': 0.0474025, 'auroc': 0.860025, 'brier': 0.0407909, 'ece': 0.0144115, 'slope': 0.906737, 'intercept': 0.0479777, 'mean_pred': 0.0376989, 'banks': ['1000', '500', '5000', 'full'], 'n_rep': 10, 'reduced': True}, 'mortality_168h|eICU': {'obs_rate': 0.0464612, 'auroc': 0.772893, 'brier': 0.058052, 'ece': 0.0423213, 'slope': 0.420848, 'intercept': -1.85779, 'mean_pred': 0.0888297, 'banks': ['1000', '500', '5000', 'full'], 'n_rep': 10, 'reduced': True}},
}
R5_REF = _ov("R5_REF", _R5_REF_DEFAULT)                 # a test replaces these with constants made from a simulated stored state (in a real run, the constants above are used as they are)
print("constants for comparison (aggregates of the stored csv, aggregate values only): grid %d cells | rescoring %d cells | 33-column eICU tail %d cells | hospital distribution summary %d lines | network summary %d lines | A9 %d cells | source csv %d files" % (
    len(R5_REF["matrix"]), len(R5_REF["rescore"]), len(R5_REF["a2tail"]), len(R5_REF["hospital"]), len(R5_REF["network"]), len(R5_REF["a9"]), len(R5_REF_SOURCES)))

## Definitions

Preprocessors (`PreprocessorGeneral`, `PreprocessorCatBoost`), data loading (`_load_x`, `_load_y`, `load_mimic4`, `load_external`), cosine retrieval (`retrieval_R1_cosine`), `compute_ece`, `_logit`, the DeLong functions (`_midrank`, `_delong_core_original`), `inner_cv_bank`, `sample_bank`, the anchored ridge (`_anchored_ridge`, `_pick_lam`) and `Resumable`, the store of finished units.

| Definition | Note |
|---|---|
| `load_external`, `load_mimic4` | `load_external` also returns `stay_ids`. `load_mimic4` counts outcomes that are missing and fills them with 0 (a warning is printed) |
| `_midrank` | uses `scipy.stats.rankdata`, which gives the same values as a loop implementation |
| `_delong_core_original` | DeLong core in which `np.arange` is used in place of the within-class midranks. It overestimates the variance of the paired AUROC difference when scores within a class are not sorted (see A6 for its size in the hospital-level data). It is used only by diagnostic cells; every analysis uses the standard fastDeLong `_delong_core` defined with the new functions |
| `inner_cv_bank` | the number of folds defaults to `N_INNER`. Retrieval rule R1 (cosine) does not use mutual information, so no mutual-information computation is made |
| `_anchored_ridge`, `_pick_lam` | the number of folds is `N_INNER` |
| `Resumable` | stores finished units under the resume folder (`out_pram/resume/`) |


In [ ]:
# ================================================================
# PRAM definitions: preprocessors, data loading, retrieval, DeLong, inner cross-validation, anchored ridge, Resumable
# ================================================================
def _log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def stable_seed(*parts):
    s = "|".join(str(p) for p in parts) + f"|SEED={SEED}"
    return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16) % (2**31 - 1)


def rng_of(*parts):
    return np.random.RandomState(stable_seed(*parts))


def _jsafe(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (Path,)):
        return str(o)
    return str(o)


class Resumable:
    """append per row + completion marker per unit. Even if the kernel dies, units already completed are not recomputed."""

    def __init__(self, name):
        self.name = name
        self.rows_path = RESUME_DIR / f"{name}.jsonl"
        self.units_path = RESUME_DIR / f"{name}.units.jsonl"
        self.rows, self.units = [], set()
        self.n_dropped = 0                                       # counts the incomplete lines that are discarded
        if self.rows_path.exists():
            with open(self.rows_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.rows.append(json.loads(line))
                    except json.JSONDecodeError:
                        self.n_dropped += 1                      # discard the incomplete last line at the time of interruption
            if self.n_dropped:
                print(f"  [{name}] incomplete line {self.n_dropped} lines discarded (last line at the time of interruption)")
        if self.units_path.exists():
            with open(self.units_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        self.units.add(line)

    @staticmethod
    def _ukey(unit):
        return "::".join(str(x) for x in unit)

    def unit_done(self, unit):
        return self._ukey(unit) in self.units

    def add_row(self, row):
        with open(self.rows_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, default=_jsafe) + "\n")
            f.flush()
        self.rows.append(row)

    def mark_unit(self, unit):
        k = self._ukey(unit)
        with open(self.units_path, "a", encoding="utf-8") as f:
            f.write(k + "\n")
            f.flush()
        self.units.add(k)

    def df(self):
        return pd.DataFrame(self.rows) if self.rows else pd.DataFrame()


class PreprocessorGeneral:
    def __init__(self, cat_cols, num_cols, random_state=SEED):
        self.cat_cols, self.num_cols = cat_cols, num_cols
        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        self.num_imputer = IterativeImputer(max_iter=10, random_state=random_state, sample_posterior=False)
        self.scaler = StandardScaler()
        self._fitted = False
        self._all_nan_cols = []

    def fit(self, X_df):
        if len(self.cat_cols) > 0:
            Xi = self.cat_imputer.fit_transform(X_df[self.cat_cols].values)
            self.ohe.fit(Xi)
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values
            self._all_nan_cols = [i for i in range(X_num.shape[1]) if np.all(np.isnan(X_num[:, i]))]
            if self._all_nan_cols:
                X_num = X_num.copy()
                X_num[:, self._all_nan_cols] = 0.0
            Xni = self.num_imputer.fit_transform(X_num)
            self.scaler.fit(Xni)
        self._fitted = True
        return self

    def transform(self, X_df):
        parts = []
        if len(self.cat_cols) > 0:
            parts.append(self.ohe.transform(self.cat_imputer.transform(X_df[self.cat_cols].values)))
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values.copy()
            if self._all_nan_cols:
                X_num[:, self._all_nan_cols] = 0.0
            parts.append(self.scaler.transform(self.num_imputer.transform(X_num)))
        return np.hstack(parts).astype(np.float32) if parts else np.empty((len(X_df), 0), dtype=np.float32)

    def fit_transform(self, X_df):
        self.fit(X_df)
        return self.transform(X_df)

    def column_names(self):
        """column names after the transform (A10). An auxiliary function; not used in any computation."""
        names = []
        if len(self.cat_cols) > 0:
            names += [str(n) for n in self.ohe.get_feature_names_out(self.cat_cols)]
        names += list(self.num_cols)
        return names


class PreprocessorCatBoost:
    def __init__(self, cat_cols, num_cols, random_state=SEED):
        self.cat_cols, self.num_cols = cat_cols, num_cols
        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.num_imputer = IterativeImputer(max_iter=10, random_state=random_state, sample_posterior=False)
        self.cat_feature_indices = list(range(len(cat_cols)))
        self._all_nan_cols = []

    def fit(self, X_df):
        if len(self.cat_cols) > 0:
            self.cat_imputer.fit(X_df[self.cat_cols].values)
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values
            self._all_nan_cols = [i for i in range(X_num.shape[1]) if np.all(np.isnan(X_num[:, i]))]
            if self._all_nan_cols:
                X_num = X_num.copy()
                X_num[:, self._all_nan_cols] = 0.0
            self.num_imputer.fit(X_num)
        return self

    def transform(self, X_df):
        parts = []
        if len(self.cat_cols) > 0:
            parts.append(self.cat_imputer.transform(X_df[self.cat_cols].values))
        if len(self.num_cols) > 0:
            X_num = X_df[self.num_cols].values.copy()
            if self._all_nan_cols:
                X_num[:, self._all_nan_cols] = 0.0
            parts.append(self.num_imputer.transform(X_num))
        if not parts:
            return pd.DataFrame()
        out = pd.DataFrame(np.hstack(parts), columns=self.cat_cols + self.num_cols)
        for c in self.cat_cols:
            out[c] = out[c].round().astype(int)
        return out

    def fit_transform(self, X_df):
        self.fit(X_df)
        return self.transform(X_df)


# ---- data loading ------------------------------------------
def _detect_cols(X_df):
    cat = [c for c in X_df.columns if X_df[c].dropna().isin([0, 1]).all() and X_df[c].nunique(dropna=True) <= 2]
    num = [c for c in X_df.columns if c not in cat]
    return cat, num


def _load_x(db_name):
    """reads the feature matrix and returns (X without stay_id and fold_id, stay_id array)."""
    if db_name == "mimic4":
        fp = DATA_ROOT / "mimic4" / "X_folded.parquet"
    else:
        fp = DATA_ROOT / db_name / "X_external.parquet"
    X_f = pd.read_parquet(fp)
    stay_ids = X_f["stay_id"].values
    X_df = X_f.drop(columns=[c for c in ["stay_id", "fold_id"] if c in X_f.columns])
    X_df = X_df.drop(columns=[c for c in DROP_COLS if c in X_df.columns])
    return X_df, stay_ids


def _load_y(db_name):
    fp = DATA_ROOT / db_name / f"binary_outcomes_{db_name}.csv"
    return pd.read_csv(fp)


def split_56_14_30(y):
    """70/30 stratified split, then the 70 part is split 80/20 stratified. Train 56, validation 14, test 30."""
    idx_all = np.arange(len(y))
    idx_tr70, idx_te, _, _ = train_test_split(idx_all, y, test_size=0.30, stratify=y, random_state=SEED)
    idx_tr, idx_va, _, _ = train_test_split(idx_tr70, y[idx_tr70], test_size=0.20, stratify=y[idx_tr70], random_state=SEED)
    return idx_tr, idx_va, idx_te


def load_mimic4(outcome="AKI_168h"):
    X_df, stay_ids = _load_x("mimic4")
    y_df = _load_y("mimic4")
    y_col = OUTCOME_MAP[outcome]
    merged = pd.DataFrame({"stay_id": stay_ids}).merge(y_df[["stay_id", y_col]], on="stay_id", how="left")
    n_miss = int(merged[y_col].isna().sum())                     # the number of missing outcomes is counted; they are filled with 0
    if n_miss:
        print(f"  [warning] MIMIC-IV {outcome}: stays without outcome {n_miss} rows were filled with 0 (missing outcomes are filled with 0).")
    y = merged[y_col].fillna(0).values.astype(int)
    cat_cols, num_cols = _detect_cols(X_df)
    idx_tr, idx_va, idx_te = split_56_14_30(y)
    return {"X_df": X_df, "y": y, "stay_ids": stay_ids,
            "idx_train": idx_tr, "idx_val": idx_va, "idx_test": idx_te,
            "cat_cols": cat_cols, "num_cols": num_cols,
            "name": "MIMIC-IV", "outcome": outcome, "n_outcome_missing": n_miss}


def load_external(db_name, outcome):
    X_ref, _ = _load_x("mimic4")
    cat_cols, num_cols = _detect_cols(X_ref)
    X_df, stay_ids = _load_x(db_name)
    y_df = _load_y(db_name)
    y_col = OUTCOME_MAP[outcome]
    X_aligned = pd.DataFrame(index=range(len(stay_ids)))
    for col in cat_cols + num_cols:
        X_aligned[col] = X_df[col].values if col in X_df.columns else np.nan
    merged = X_aligned.copy()
    merged["stay_id"] = stay_ids
    merged = merged.merge(y_df[["stay_id", y_col]], on="stay_id", how="inner")
    y = merged[y_col].values.astype(int)
    stay_out = merged["stay_id"].values                          # stay_id is kept after the merge (for linking to time and hospital)
    X_out = merged.drop(columns=["stay_id", y_col])
    missing_cols = [c for c in cat_cols + num_cols if c not in X_df.columns]
    if missing_cols:
        _log(f"  {db_name}: structurally missing -> {missing_cols}")
    display_name = {"mimic3": "MIMIC-III", "eicu": "eICU"}[db_name]
    return {"X_df": X_out, "y": y, "stay_ids": stay_out, "cat_cols": cat_cols, "num_cols": num_cols,
            "name": display_name, "outcome": outcome}


# ---- retrieval and helper functions ----------------
def retrieval_R1_cosine(c, k=K_NEIGHBORS):
    k_ = min(k, len(c["X_tr_s"]))
    if k_ < 1:
        return np.full(len(c["X_te_s"]), 0.5)
    nn = NearestNeighbors(n_neighbors=k_, metric="cosine", algorithm="brute", n_jobs=-1)
    nn.fit(c["X_tr_s"])
    d, idx = nn.kneighbors(c["X_te_s"])
    w = 1.0 / (d + 1e-6)
    return (w * c["y_tr"][idx]).sum(1) / w.sum(1)


RETRIEVAL_FN = retrieval_R1_cosine


def compute_ece(y, p, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (p >= lo) & (p < hi)
        if m.sum() == 0:
            continue
        ece += m.sum() * abs(y[m].mean() - p[m].mean())
    return ece / len(y)


def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


# ---- DeLong -----------------------------------------------
def _midrank(x):
    """Compute midranks for tied values (scipy implementation; gives the same values as a loop implementation)."""
    return scipy_stats.rankdata(np.asarray(x, dtype=float), method="average")


def _delong_core_original(preds_sorted_T, m):
    """The DeLong core that uses np.arange in place of the within-class midranks (the original core). Defect: if the scores within a class
    are not in sorted order, the variance of the paired AUROC difference is overestimated. It is correct only when sorted by score within each class.
    All analyses of this notebook use `_delong_core` below; this function is called only by diagnostic cells that give the original variance for comparison."""
    n = preds_sorted_T.shape[1] - m
    k = preds_sorted_T.shape[0]
    aucs = np.zeros(k)
    v01 = np.zeros((k, m))
    v10 = np.zeros((k, n))
    for i in range(k):
        tx = _midrank(preds_sorted_T[i, :])
        aucs[i] = (np.sum(tx[:m]) - m * (m + 1) / 2.0) / (m * n)
        v10[i] = (tx[m:] - np.arange(1, n + 1)) / m
        v01[i] = 1.0 - (tx[:m] - np.arange(1, m + 1)) / n
    if m > 1:
        S01 = np.cov(v01) if k > 1 else np.array([[np.var(v01[0], ddof=1)]])
    else:
        S01 = np.zeros((k, k))
    if n > 1:
        S10 = np.cov(v10) if k > 1 else np.array([[np.var(v10[0], ddof=1)]])
    else:
        S10 = np.zeros((k, k))
    S = S01 / m + S10 / n
    return aucs, S


def _delong_core(preds_sorted_T, m):
    """Standard fastDeLong (Sun and Xu 2014). Builds the structural components from midranks within positives and negatives (tx, ty) and overall midranks (tz).
    preds_sorted_T is (k, n_total), positives first. Returns (AUROCs, covariance matrix). An internal check compares with the O(mn) definition and with a bootstrap."""
    n = preds_sorted_T.shape[1] - m
    k = preds_sorted_T.shape[0]
    pos, neg = preds_sorted_T[:, :m], preds_sorted_T[:, m:]
    tx = np.array([_midrank(pos[i]) for i in range(k)])
    ty = np.array([_midrank(neg[i]) for i in range(k)])
    tz = np.array([_midrank(preds_sorted_T[i]) for i in range(k)])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    S01 = np.cov(v01) if k > 1 else np.array([[np.var(v01[0], ddof=1)]])
    S10 = np.cov(v10) if k > 1 else np.array([[np.var(v10[0], ddof=1)]])
    return aucs, S01 / m + S10 / n


# ---- mixing-weight selection within a bank and stratified draw ----------------------
def inner_cv_bank(X_bank, y_bank, p_base_bank, r_fn, n_splits=N_INNER, key=""):
    """Select (a) alpha and (b) produce OOF PRAM predictions by cross-validation inside the bank. OOF predictions avoid self-retrieval.
    compute_mi is not called, because R1 cosine does not read mutual information."""
    n = len(y_bank)
    if n < n_splits * 5 or len(np.unique(y_bank)) < 2:
        return {"alpha": 0.0, "oof_pram": p_base_bank.copy(), "ok": False}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                          random_state=stable_seed("innercv", key))
    scores = {a: [] for a in ALPHA_GRID}
    oof_retr = np.full(n, np.nan)
    for itr, iva in skf.split(np.zeros(n), y_bank):
        mi_i, mp_i = None, None
        pr = r_fn({"X_tr_s": X_bank[itr], "y_tr": y_bank[itr],
                   "X_te_s": X_bank[iva], "y_te": y_bank[iva],
                   "mi": mi_i, "mi_prob": mp_i})
        oof_retr[iva] = pr
        if len(np.unique(y_bank[iva])) > 1:
            for a in ALPHA_GRID:
                scores[a].append(roc_auc_score(y_bank[iva],
                                               (1 - a) * p_base_bank[iva] + a * pr))
    best_a = max(ALPHA_GRID, key=lambda a: np.nanmean(scores[a]) if scores[a] else -1)
    oof_retr = np.where(np.isnan(oof_retr), p_base_bank, oof_retr)
    return {"alpha": best_a, "oof_pram": (1 - best_a) * p_base_bank + best_a * oof_retr,
            "ok": True}


def sample_bank(pool_y, size, key):
    """stratified draw. Uses round()."""
    r = rng_of("bank", key)
    if size == "full":
        return np.arange(len(pool_y))
    prev = pool_y.mean()
    n_pos = max(1, int(round(size * prev)))
    n_neg = max(1, size - n_pos)
    pi, ni = np.where(pool_y == 1)[0], np.where(pool_y == 0)[0]
    sp = r.choice(pi, min(n_pos, len(pi)), replace=len(pi) < n_pos)
    sn = r.choice(ni, min(n_neg, len(ni)), replace=len(ni) < n_neg)
    return np.sort(np.concatenate([sp, sn]))


# ---- anchored ridge -------------------------------------------
def _anchored_ridge(X, y, w_src, b_src, lam, anchor_intercept=True, maxiter=500):
    """minimizes mean log loss + lam*||w - w_src||^2. Returns (w, b, ||u||, converged)"""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, d = X.shape
    off = X @ w_src + b_src

    def fun(t):
        u = t[:d]
        db = 0.0 if anchor_intercept else t[d]
        z = off + X @ u + db
        loss = float(np.mean(np.logaddexp(0.0, z) - y * z)) + lam * float(u @ u)
        g = (1.0 / (1.0 + np.exp(-z)) - y) / n
        gu = X.T @ g + 2.0 * lam * u
        return (loss, gu) if anchor_intercept else (loss, np.concatenate([gu, [g.sum()]]))

    t0 = np.zeros(d if anchor_intercept else d + 1)
    r = minimize(fun, t0, jac=True, method="L-BFGS-B",
                 options={"maxiter": maxiter, "ftol": 1e-12, "gtol": 1e-9})
    u = r.x[:d]
    db = 0.0 if anchor_intercept else float(r.x[d])
    return w_src + u, b_src + db, float(np.sqrt(u @ u)), bool(r.success)


def _pick_lam(Xb, yb, w_src, b_src, anchor_intercept, key, n_splits=N_INNER):
    """lam is chosen within the bank only."""
    if len(np.unique(yb)) < 2 or len(yb) < n_splits * 5:
        return float(LAM_GRID[-1])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                          random_state=stable_seed("a53", key))
    scores = {float(l): [] for l in LAM_GRID}
    for itr, iva in skf.split(np.zeros(len(yb)), yb):
        if len(np.unique(yb[iva])) < 2:
            continue
        for l in LAM_GRID:
            w, b, _, _ = _anchored_ridge(Xb[itr], yb[itr], w_src, b_src,
                                         float(l), anchor_intercept, maxiter=200)
            scores[float(l)].append(roc_auc_score(yb[iva], Xb[iva] @ w + b))
    best = max(scores, key=lambda l: np.nanmean(scores[l]) if scores[l] else -1)
    return float(best)


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))


## New functions

Every analysis of this notebook calls one engine (`bank_predictions`) that returns the test predictions of the strategies for one bank and one set of test patients, so the main analysis, temporal validation, hospital level and tail risk share the same code.

| Definition | What |
|---|---|
| `full_metrics` | AUROC, AUPRC, Brier score, ECE, calibration slope and intercept, and net benefit over the full 1 to 50% threshold grid. The slope is an unpenalized Newton fit |
| `update_hierarchy` | update steps 0 to 3 and closed testing, on the frozen logistic model only: intercept only (1, maximum likelihood), intercept and slope (2, unpenalized), model revision (3). The test is a likelihood-ratio test of the unpenalized revision; when step 3 is chosen, the prediction is the **anchored ridge** with the penalty chosen within the bank. The order of the closed-testing steps follows Vergouwe et al. (2017) |
| `bank_predictions` | test predictions per strategy group for one bank and one set of test patients |
| `run_tail_unit` | tail risk: one (bank, test fold) pair; 5 folds x 20 draws per fold |
| `run_temporal` | temporal validation: the test period is split into 2 folds that alternate as test; the past bank (P) and the same-period bank (S) are compared at the same size |
| `hospital_oof`, `hospital_unit` | one hospital, 3 folds |
| `network_unit` | an unlabeled hospital receives the mean probability of its partner hospitals |
| `meta_reml_hk` | REML tau squared, Hartung-Knapp interval (untruncated and truncated forms), prediction interval, inverse-variance I squared, Q and degrees of freedom |
| `delong_delta_var`, `_delong_core` | paired AUROC difference and its variance (standard fastDeLong, Sun and Xu 2014). Checked against the O(mn) definition and a bootstrap |


In [ ]:
# ================================================================
# PRAM Cell 3. Newly written definitions (reporting, metrics, update hierarchy)
# ================================================================
try:
    from IPython.display import display, Markdown
    from IPython import get_ipython as _get_ipy
    _IN_IPY = _get_ipy() is not None
except Exception:  # noqa: BLE001
    _IN_IPY = False

REPORT_LINES = []                       # every table shown on screen is also accumulated in the result md


def emit(text=""):
    REPORT_LINES.append(str(text))
    if _IN_IPY:
        display(Markdown(str(text)))
    else:
        print(text)


def md_table(df, digits=4, max_rows=80):
    if df is None or len(df) == 0:
        return "(empty)"
    d = df.head(max_rows)
    cols = [str(c) for c in d.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in d.itertuples(index=False):
        cells = []
        for v in r:
            if isinstance(v, (float, np.floating)):
                cells.append("" if not np.isfinite(v) else f"{v:.{digits}f}")
            else:
                cells.append(str(v))
        lines.append("| " + " | ".join(cells) + " |")
    if len(df) > max_rows:
        lines.append(f"... {len(df) - max_rows} rows omitted (full table in csv)")
    return "\n".join(lines)


def emit_table(title, df, digits=4, max_rows=80):
    emit("\n**" + title + "**\n")
    emit(md_table(df, digits=digits, max_rows=max_rows))


def save_csv(df, name):
    path = CSV_DIR / (name + ".csv")
    df.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.6g")
    emit(f"- saved `{path.as_posix()}` ({len(df)}rows)")
    return path


# ---- metrics ------------------------------------------------------------------
NB_COLS = [f"nb@{float(t)}" for t in THRESH_GRID]


def _newton_2(y, x, iters=60):
    """unpenalized maximum likelihood of p = sigmoid(a + b x). Starts at (0, 1). Returns (nan, nan) on failure."""
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    if (not np.isfinite(x).all()) or x.std() < 1e-9 or len(np.unique(y)) < 2:
        return np.nan, np.nan

    def nll(a_, b_):
        z = a_ + b_ * x
        return float(np.sum(np.logaddexp(0.0, z) - y * z))

    a, b = 0.0, 1.0
    cur = nll(a, b)
    for _ in range(iters):
        pr = _sigmoid(a + b * x)
        w = pr * (1 - pr) + 1e-12
        g = np.array([np.sum(y - pr), np.sum((y - pr) * x)])
        H = np.array([[w.sum(), (w * x).sum()], [(w * x).sum(), (w * x * x).sum()]]) + 1e-10 * np.eye(2)
        step = np.linalg.solve(H, g)
        t = 1.0
        while t > 1e-4 and nll(a + t * step[0], b + t * step[1]) > cur + 1e-12:
            t *= 0.5
        a, b = a + t * step[0], b + t * step[1]
        new = nll(a, b)
        done = abs(cur - new) < 1e-10 or max(abs(t * step[0]), abs(t * step[1])) < 1e-9
        cur = new
        if done:
            break
    return float(a), float(b)


def calib_slope_intercept(y, p):
    """(slope, intercept). Unpenalized calibration line on the logit."""
    a, b = _newton_2(y, _logit(np.asarray(p, dtype=float)))
    return b, a


def nb_curve(y, p, grid=THRESH_GRID):
    """net benefit over the whole threshold grid. One sort counts all thresholds."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    n = len(y)
    order = np.argsort(-p, kind="stable")
    ps, ys = p[order], y[order]
    ctp = np.cumsum(ys)
    cfp = np.cumsum(1 - ys)
    k = np.searchsorted(-ps, -np.asarray(grid, dtype=float), side="right")      # number of patients with p >= t
    tp = np.where(k > 0, ctp[np.maximum(k - 1, 0)], 0)
    fp = np.where(k > 0, cfp[np.maximum(k - 1, 0)], 0)
    g = np.asarray(grid, dtype=float)
    return tp / n - fp / n * (g / (1 - g))


def full_metrics(y, p, orig_slope=False):
    """if orig_slope=True, adds the slope of a scikit-learn C=1e6 logistic fit as slope_orig. For reproduction checks only."""
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    out = {}
    try:
        out["auroc"] = float(roc_auc_score(y, p))
        out["auprc"] = float(average_precision_score(y, p))
        out["brier"] = float(brier_score_loss(y, p))
    except ValueError:
        out["auroc"] = out["auprc"] = out["brier"] = np.nan
    out["ece"] = float(compute_ece(y, p))
    sl, ic = calib_slope_intercept(y, p)
    out["slope"], out["intercept"] = sl, ic
    if orig_slope:
        lp = _logit(p)
        out["slope_orig"] = float(LogisticRegression(max_iter=1000, C=1e6).fit(lp.reshape(-1, 1), y).coef_[0][0])             if (np.isfinite(lp).all() and lp.std() > 1e-9 and len(np.unique(y)) > 1) else np.nan
    out["mean_pred"] = float(p.mean())
    out["obs_rate"] = float(y.mean())
    for c, v in zip(NB_COLS, nb_curve(y, p)):
        out[c] = float(v)
    return out


def delong_delta_var(y, pa, pb, original=False):
    """paired DeLong. (AUROC_a, AUROC_b, difference, variance of the difference). nan if one class is empty. If original=True, the original implementation (for diagnosis)."""
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan, np.nan
    order = np.argsort(y)[::-1]
    m = int(y.sum())
    if m < 2 or len(y) - m < 2:
        return np.nan, np.nan, np.nan, np.nan
    preds = np.vstack([np.asarray(pa, dtype=float)[order], np.asarray(pb, dtype=float)[order]])
    aucs, S = (_delong_core_original if original else _delong_core)(preds, m)
    var = float(S[0, 0] + S[1, 1] - 2 * S[0, 1])
    return float(aucs[0]), float(aucs[1]), float(aucs[0] - aucs[1]), max(var, 0.0)


def cp_interval(k, n, level=CP_LEVEL):
    """Clopper-Pearson exact interval."""
    if n <= 0:
        return np.nan, np.nan
    a = 1 - level
    lo = 0.0 if k == 0 else scipy_stats.beta.ppf(a / 2, k, n - k + 1)
    hi = 1.0 if k == n else scipy_stats.beta.ppf(1 - a / 2, k + 1, n - k)
    return float(lo), float(hi)


def area_under_curve(grid, vals, lo, hi):
    """trapezoid area within [lo, hi]."""
    g = np.asarray(grid, dtype=float)
    v = np.asarray(vals, dtype=float)
    m = (g >= lo - 1e-9) & (g <= hi + 1e-9)
    if m.sum() < 2:
        return np.nan
    trap = getattr(np, "trapezoid", None) or np.trapz
    return float(trap(v[m], g[m]))


# ---- update hierarchy and closed testing (A4) ---------------------------------------------
def _nll_sum(y, z):
    return float(np.sum(np.logaddexp(0.0, z) - y * z))


def fit_intercept_only(y, off):
    """re-estimates the intercept only (recalibration-in-the-large). The slope is fixed at 1."""
    a = 0.0
    for _ in range(60):
        pr = _sigmoid(off + a)
        step = float(np.sum(y - pr) / (np.sum(pr * (1 - pr)) + 1e-12))
        step = max(-5.0, min(5.0, step))
        a += step
        if abs(step) < 1e-10:
            break
    return a


def fit_revision(X, y, off, ridge=REVISION_RIDGE):
    """model revision: the frozen linear predictor is an offset; the intercept and all coefficients are re-estimated. The tiny regularization is for numerical stability."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, d = X.shape

    def fun(t):
        a, u = t[0], t[1:]
        z = off + a + X @ u
        loss = float(np.mean(np.logaddexp(0.0, z) - y * z)) + ridge * float(u @ u)
        g = (_sigmoid(z) - y) / n
        return loss, np.concatenate([[g.sum()], X.T @ g + 2.0 * ridge * u])

    r = minimize(fun, np.zeros(d + 1), jac=True, method="L-BFGS-B",
                 options={"maxiter": 300, "ftol": 1e-12, "gtol": 1e-8})
    return float(r.x[0]), r.x[1:].copy(), bool(r.success)


def update_hierarchy(Xb, yb, offb, Xt, offt, p_frozen_t, p_rev_alt=None):
    """fits update steps 0 to 3 in the bank and chooses one step by closed testing.
    0 no update | 1 intercept only | 2 intercept + slope | 3 model revision. Closed testing (likelihood ratio, significance level CLOSED_TEST_ALPHA):
    (1) if revision versus no update (d+1 degrees of freedom) is not significant, 0. (2) if significant and revision versus step 2 (d-1 degrees of freedom) is significant, 3.
    (3) otherwise, if step 2 versus step 1 (1 degree of freedom) is significant, 2, otherwise 1.
    if p_rev_alt is given, its prediction (anchored ridge with the penalty chosen within the bank) is used when step 3 is chosen. The test always uses the likelihood of the unpenalized revision."""
    res = {"upd1": p_frozen_t, "upd2": p_frozen_t, "upd3": p_frozen_t, "ct": p_frozen_t,
           "step": 0, "p_all": np.nan, "p_rev_vs_recal": np.nan, "p_slope": np.nan,
           "slope2": np.nan, "ok": False}
    yb = np.asarray(yb).astype(float)
    n1 = int(yb.sum())
    n0 = int(len(yb) - n1)
    if min(n1, n0) < N_EVENT_MIN or not np.isfinite(offb).all() or not np.isfinite(offt).all():
        return res
    d = Xb.shape[1]
    ll0 = -_nll_sum(yb, offb)
    a1 = fit_intercept_only(yb, offb)
    ll1 = -_nll_sum(yb, offb + a1)
    a2, b2 = _newton_2(yb, offb)
    if not np.isfinite(a2):
        return res
    ll2 = -_nll_sum(yb, a2 + b2 * offb)
    a3, u3, _ok3 = fit_revision(Xb, yb, offb)
    ll3 = -_nll_sum(yb, offb + a3 + Xb @ u3)
    res["upd1"] = _sigmoid(offt + a1)
    res["upd2"] = _sigmoid(a2 + b2 * offt)
    res["upd3"] = _sigmoid(offt + a3 + Xt @ u3)
    res["slope2"] = float(b2)
    lr_all = max(0.0, 2 * (ll3 - ll0))
    lr_rev = max(0.0, 2 * (ll3 - ll2))
    lr_slope = max(0.0, 2 * (ll2 - ll1))
    res["p_all"] = float(scipy_stats.chi2.sf(lr_all, d + 1))
    res["p_rev_vs_recal"] = float(scipy_stats.chi2.sf(lr_rev, max(d - 1, 1)))
    res["p_slope"] = float(scipy_stats.chi2.sf(lr_slope, 1))
    if res["p_all"] >= CLOSED_TEST_ALPHA:
        step = 0
    elif res["p_rev_vs_recal"] < CLOSED_TEST_ALPHA:
        step = 3
    elif res["p_slope"] < CLOSED_TEST_ALPHA:
        step = 2
    else:
        step = 1
    res["step"] = step
    res["ct"] = [p_frozen_t, res["upd1"], res["upd2"], res["upd3"] if p_rev_alt is None else p_rev_alt][step]
    res["ok"] = True
    return res


# ---- auxiliary fits -----------------------------------------------------------
def platt_sklearn(p_fit, y_fit, p_apply):
    """Platt scaling: logistic regression with default regularization on the logit."""
    lr_p = LogisticRegression(max_iter=1000, random_state=SEED)
    lr_p.fit(_logit(p_fit).reshape(-1, 1), y_fit)
    return lr_p.predict_proba(_logit(p_apply).reshape(-1, 1))[:, 1]


def _logloss(y, p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, dtype=float)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def inner_oof_candidates(Xb, yb, p1b, p5b, icv1, icv5, key, cand):
    """A11: builds the within-bank predictions of each candidate strategy by inner cross-validation. Only strategies that a hospital can actually choose are candidates."""
    out = {}
    if "b1_base" in cand:
        out["b1_base"] = p1b
    if "b5_base" in cand:
        out["b5_base"] = p5b
    if "pram_b1" in cand:
        out["pram_b1"] = icv1["oof_pram"]
    if "pram_b5" in cand:
        out["pram_b5"] = icv5["oof_pram"]
    need_fit = [c for c in ("platt_b1", "recal_iso", "local_lr") if c in cand]
    if need_fit:
        oofs = {c: p1b.copy() for c in need_fit}
        skf = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=stable_seed("selcv", key))
        for itr, iva in skf.split(np.zeros(len(yb)), yb):
            if len(np.unique(yb[itr])) < 2:
                continue
            if "platt_b1" in oofs:
                oofs["platt_b1"][iva] = platt_sklearn(p1b[itr], yb[itr], p1b[iva])
            if "recal_iso" in oofs:
                iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(p1b[itr], yb[itr])
                oofs["recal_iso"][iva] = iso.predict(p1b[iva])
            if "local_lr" in oofs:
                lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xb[itr], yb[itr])
                oofs["local_lr"][iva] = lr.predict_proba(Xb[iva])[:, 1]
        out.update(oofs)
    return out


# ---- strategy engine -----------------------------------------------------------------
def bank_predictions(C, bank, test, key, groups, cb_iter=None):
    """test predictions of the strategies fitted on one bank (row indices). Same rule as run_matrix_unit.
    C is the target cohort context (build_ctx). groups: frozen pram recal local localcb anch hier combo sel."""
    cb_iter = LOCAL_CB_ITER if cb_iter is None else cb_iter
    Xg, y, pb1, pb5 = C["Xg"], C["y"], C["pb1"], C["pb5"]
    Xb, yb, p1b, p5b = Xg[bank], y[bank], pb1[bank], pb5[bank]
    Xt, yt, p1t, p5t = Xg[test], y[test], pb1[test], pb5[test]
    P = {"b1_base": p1t, "b5_base": p5t}
    info = {}
    two = len(np.unique(yb)) > 1
    icv1 = icv5 = p_retr = None

    if "pram" in groups or "combo" in groups or "sel" in groups:
        p_retr = RETRIEVAL_FN({"X_tr_s": Xb, "y_tr": yb, "X_te_s": Xt, "y_te": yt, "mi": None, "mi_prob": None})
        icv1 = inner_cv_bank(Xb, yb, p1b, RETRIEVAL_FN, key="b1|" + key)
        icv5 = inner_cv_bank(Xb, yb, p5b, RETRIEVAL_FN, key="b5|" + key)
        P["pram_b1"] = (1 - icv1["alpha"]) * p1t + icv1["alpha"] * p_retr
        P["pram_b5"] = (1 - icv5["alpha"]) * p5t + icv5["alpha"] * p_retr
        info["alpha_b1"], info["alpha_b5"] = icv1["alpha"], icv5["alpha"]

    if "recal" in groups:
        if two:
            iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(p1b, yb)
            P["recal_iso"] = iso.predict(p1t)
            P["platt_b1"] = platt_sklearn(p1b, yb, p1t)
            P["platt_b5"] = platt_sklearn(p5b, yb, p5t)          # CatBoost gets Platt only (the update hierarchy is defined on the logistic model)
        else:
            P["recal_iso"], P["platt_b1"], P["platt_b5"] = p1t, p1t, p5t

    if "local" in groups:
        if two:
            lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xb, yb)
            P["local_lr"] = lr.predict_proba(Xt)[:, 1]
            pl = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
            pl.fit(np.vstack([C["Xs_src"], Xb]), np.concatenate([C["ys_src"], yb]))
            P["pooled_lr"] = pl.predict_proba(Xt)[:, 1]
        else:
            P["local_lr"], P["pooled_lr"] = p1t, p1t

    if "localcb" in groups:
        if two:
            cb = CatBoostClassifier(iterations=cb_iter, depth=6, learning_rate=0.05, verbose=0, random_seed=SEED)
            cb.fit(C["Xc"].iloc[bank], yb, cat_features=C["cb_cat_idx"])
            P["local_cb"] = cb.predict_proba(C["Xc"].iloc[test])[:, 1]
        else:
            P["local_cb"] = p1t

    if "anch" in groups and two and (ANCHORED_MAX_BANK is None or len(bank) <= ANCHORED_MAX_BANK):
        lam = _pick_lam(Xb, yb, C["w_src"], C["b_src"], True, "anch|" + key)
        w_a, b_a, _, _ = _anchored_ridge(Xb, yb, C["w_src"], C["b_src"], lam, True)
        P["anchored_ridge"] = _sigmoid(Xt @ w_a + b_a)
        info["anch_lam"] = lam

    if "hier" in groups:                       # the update hierarchy only on the frozen logistic model (for CatBoost see platt_b5 above)
        u = update_hierarchy(Xb, yb, _logit(p1b), Xt, _logit(p1t), p1t, p_rev_alt=P.get("anchored_ridge"))
        P["upd1_b1"], P["upd2_b1"], P["upd3_b1"], P["ct_b1"] = u["upd1"], u["upd2"], u["upd3"], u["ct"]
        info["ct_step_b1"] = u["step"]
        info["ct_p_all_b1"] = u["p_all"]
        info["ct_p_rev_vs_recal_b1"] = u["p_rev_vs_recal"]
        info["slope2_b1"] = u["slope2"]

    if "combo" in groups and two:
        # retrieval -> Platt: Platt is fitted on the inner cross-validation OOF mixture predictions in the bank and applied to the test mixture prediction 
        P["pram_platt_b1"] = platt_sklearn(icv1["oof_pram"], yb, P["pram_b1"])
        P["pram_platt_b5"] = platt_sklearn(icv5["oof_pram"], yb, P["pram_b5"])
    elif "combo" in groups:
        P["pram_platt_b1"], P["pram_platt_b5"] = p1t, p5t

    if "sel" in groups:
        cand = [c for c in SEL_CANDIDATES if c in P]
        oof_c = inner_oof_candidates(Xb, yb, p1b, p5b, icv1, icv5, key, cand)
        cand = [c for c in cand if c in oof_c]
        t_site = T_SITE[C["outcome"]]
        ll = {c: _logloss(yb, oof_c[c]) for c in cand}
        nbv = {c: float(nb_curve(yb, oof_c[c], np.array([t_site]))[0]) for c in cand}
        pick_ll = min(cand, key=lambda c: ll[c])
        pick_nb = max(cand, key=lambda c: nbv[c])
        P["sel_logloss"], P["sel_nb"] = P[pick_ll], P[pick_nb]
        info["sel_logloss"], info["sel_nb"] = pick_ll, pick_nb
    return P, info


def run_matrix_unit(C, fs, outcome, target, bank_size, n_draws, groups, key_tag="m23", keep=None, delong_pairs=None, orig_slope=False):
    """one unit (feature set, outcome, target, bank size). For each draw (rep), the 5 folds are pooled and the metrics are computed once per bank result (pooled).
    the draw keys use key_tag m23, so the same patients are drawn in every run.
    if keep is given, only those strategies get metrics. If delong_pairs={strategy: reference}, the paired DeLong difference and its variance for that strategy are attached to the row."""
    y = C["y"]
    n = len(y)
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(n), y))
    n_rep = 1 if bank_size == "full" else n_draws
    rows, info_rows = [], []
    for rep in range(n_rep):
        oof = {}
        for fi, (pool_idx, test_idx) in enumerate(folds):
            key = f"{outcome}|{target}|{bank_size}|{fi}|{rep}"
            if bank_size == "full":
                sel = np.arange(len(pool_idx))
            else:
                sel = sample_bank(y[pool_idx], bank_size, key_tag + "|" + key)
            bank = pool_idx[sel]
            P, info = bank_predictions(C, bank, test_idx, key, set(groups))
            for m, v in P.items():
                if m not in oof:
                    oof[m] = np.full(n, np.nan, dtype=np.float32)
                oof[m][test_idx] = v
            info_rows.append({"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size),
                              "rep": rep, "fold": fi, "n_bank": int(len(bank)),
                              "events_bank": int(y[bank].sum()), **info})
        for m, pm in oof.items():
            if np.isnan(pm).any() or (keep is not None and m not in keep):
                continue
            row = {"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size),
                   "rep": rep, "method": m, **full_metrics(y, pm.astype(float), orig_slope=orig_slope)}
            if delong_pairs and m in delong_pairs and delong_pairs[m] in oof:
                _a, _b, dd, vv = delong_delta_var(y, pm.astype(float), oof[delong_pairs[m]].astype(float))
                row["d_auroc_base"], row["var_auroc_base"] = dd, vv
            rows.append(row)
    return rows, info_rows


# ---- development model and target context -------------------------------------------------------
def build_dev(X_train_df, y_train, cols, cat_ref, num_ref, tag):
    """fits the two frozen models (logistic and CatBoost) and the preprocessor on the development training data only."""
    cc = [c for c in cat_ref if c in cols]
    nc = [c for c in num_ref if c in cols]
    X_train_df = X_train_df.reset_index(drop=True)
    y_train = np.asarray(y_train).astype(int)
    pg = PreprocessorGeneral(cc, nc)
    Xtr_g = pg.fit_transform(X_train_df)
    pc = PreprocessorCatBoost(cc, nc)
    Xtr_c = pc.fit_transform(X_train_df)
    m1 = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xtr_g, y_train)
    m5 = CatBoostClassifier(iterations=DEV_CB_ITER, depth=6, learning_rate=0.05, verbose=0, random_seed=SEED)
    m5.fit(Xtr_c, y_train, cat_features=pc.cat_feature_indices)
    return {"pg": pg, "pc": pc, "m1": m1, "m5": m5, "Xs_src": Xtr_g, "ys_src": y_train,
            "cb_cat_idx": pc.cat_feature_indices, "cols": cc + nc, "tag": tag,
            "w_src": np.asarray(m1.coef_).ravel().astype(float), "b_src": float(np.asarray(m1.intercept_).ravel()[0])}


def make_ctx(dev, X_df, y, stay_ids, outcome):
    """moves the target cohort through the development preprocessor and attaches the frozen model outputs."""
    X_df = X_df.reset_index(drop=True)
    Xg = dev["pg"].transform(X_df)
    Xc = dev["pc"].transform(X_df)
    return {"Xg": Xg, "Xc": Xc, "y": np.asarray(y).astype(int),
            "pb1": dev["m1"].predict_proba(Xg)[:, 1], "pb5": dev["m5"].predict_proba(Xc)[:, 1],
            "stay_ids": np.asarray(stay_ids), "outcome": outcome,
            "Xs_src": dev["Xs_src"], "ys_src": dev["ys_src"], "cb_cat_idx": dev["cb_cat_idx"],
            "w_src": dev["w_src"], "b_src": dev["b_src"], "d": int(Xg.shape[1])}


# ---- hospital id and discharge year ---------------------------------------------------------
def load_eicu_patient_table():
    """reads only the stay, hospital and discharge year columns of the patient table (those that exist)."""
    cols = [c for c in (EICU_COL_STAY, EICU_COL_HOSP, EICU_COL_YEAR) if c]
    df = pd.read_csv(EICU_PATIENT_PATH, usecols=cols)
    ren = {EICU_COL_STAY: "stay_id"}
    if EICU_COL_HOSP:
        ren[EICU_COL_HOSP] = "hospitalid"
    if EICU_COL_YEAR:
        ren[EICU_COL_YEAR] = "year"
    df = df.rename(columns=ren)
    df["stay_id"] = pd.to_numeric(df["stay_id"], errors="coerce")
    df = df.dropna(subset=["stay_id"]).drop_duplicates("stay_id")
    df["stay_id"] = df["stay_id"].astype("int64")
    return df.set_index("stay_id")


def map_to_stays(table, col, stay_ids):
    s = pd.to_numeric(pd.Series(np.asarray(stay_ids)), errors="coerce")
    return s.map(table[col]).to_numpy(dtype=float)


def eligible_sites(y, hid):
    """eligible hospitals: at least MIN_SITE_N stays, at least MIN_SITE_EVENTS events and non-events each. Largest hospitals first."""
    keep = []
    for h in pd.unique(hid[pd.notna(hid)]):
        m = (hid == h)
        if m.sum() >= MIN_SITE_N and y[m].sum() >= MIN_SITE_EVENTS and (m.sum() - y[m].sum()) >= MIN_SITE_EVENTS:
            keep.append(int(h))
    keep.sort(key=lambda h: -int((hid == h).sum()))
    return keep


# ---- hospital level ----------------------------
def hospital_oof(C, outcome, h, hid, groups):
    """one set of 3-fold test predictions (OOF) of one hospital and the per-fold event counts. (None, None, None) if not eligible."""
    y = C["y"]
    idx = np.where(hid == h)[0]
    ys = y[idx]
    n_sp = min(N_SITE_SPLITS, int(ys.sum()), int(len(ys) - ys.sum()))
    if n_sp < 2:
        return None, None, None
    skf = StratifiedKFold(n_splits=n_sp, shuffle=True, random_state=stable_seed("a29", outcome, int(h)))
    oof, fold_rows = {}, []
    for fi, (bank_i, test_i) in enumerate(skf.split(np.zeros(len(ys)), ys)):
        bank, test = idx[bank_i], idx[test_i]
        yb, yte = y[bank], y[test]
        if len(np.unique(yb)) < 2 or len(np.unique(yte)) < 2:
            return None, None, None
        P, info = bank_predictions(C, bank, test, f"a29|{outcome}|{h}|{fi}", set(groups))
        for m, v in P.items():
            if m not in oof:
                oof[m] = np.full(len(ys), np.nan)
            oof[m][test_i] = v
        fold_rows.append({"outcome": outcome, "hospitalid": int(h), "fold": fi,
                          "n_test": int(len(test)), "events_test": int(yte.sum()),
                          "n_bank": int(len(bank)), "events_bank": int(yb.sum()), **info})
    return ys, oof, fold_rows


def hospital_unit(C, outcome, h, hid, groups):
    ys, oof, fold_rows = hospital_oof(C, outcome, h, hid, groups)
    if ys is None:
        return None, None
    rows = []
    base1, base5 = oof["b1_base"], oof["b5_base"]
    for m, v in oof.items():
        if np.isnan(v).any():
            continue
        _a, _b, d1, v1 = delong_delta_var(ys, v, base1)
        _a, _b, d5, v5 = delong_delta_var(ys, v, base5)
        rows.append({"outcome": outcome, "hospitalid": int(h), "method": m, "n_site": int(len(ys)),
                     "n_events": int(ys.sum()), "prev": float(ys.mean()), **full_metrics(ys, v),
                     "d_auroc_b1": d1, "var_b1": v1, "d_auroc_b5": d5, "var_b5": v5})
    return rows, fold_rows


# ---- network of unlabeled hospitals ------
def _topk_cos(Xb, yb, Xq, k):
    kk = int(min(k, len(Xb)))
    nn = NearestNeighbors(n_neighbors=kk, metric="cosine").fit(Xb)
    d, i = nn.kneighbors(Xq)
    return d, yb[i]


def _site_reply(d, yk):
    """the one probability a hospital returns to the network."""
    w = 1.0 / (d + 1e-6)
    return (w * yk).sum(axis=1) / np.maximum(w.sum(axis=1), 1e-12)


def _resample_prev(gi, y, r, seed):
    """scales the prevalence of the partner bank by a factor r. When raising, all events + some non-events; when lowering, all non-events + some events."""
    rs = np.random.RandomState(seed)
    yg = y[gi]
    pos, neg = gi[yg == 1], gi[yg == 0]
    prev = len(pos) / len(gi)
    q = min(max(prev * r, 0.005), 0.95)
    if q > prev:
        n_neg = max(1, min(int(round(len(pos) * (1 - q) / q)), len(neg)))
        return np.sort(np.concatenate([pos, rs.choice(neg, n_neg, replace=False)]))
    n_pos = max(1, min(int(round(len(neg) * q / (1 - q))), len(pos)))
    return np.sort(np.concatenate([neg, rs.choice(pos, n_pos, replace=False)]))


def network_unit(C, outcome, h, hid, sites, factors, alphas):
    """mixes the mean probability of the partners (up to 20 hospitals) into the 30% test portion of hospital h.
    factors are the partner bank prevalence multipliers. With 1.0 the whole partner cohort is used."""
    Xg, y, pb1 = C["Xg"], C["y"], C["pb1"]
    idx = np.where(hid == h)[0]
    rng = rng_of("a32", outcome, int(h))
    perm = rng.permutation(len(idx))
    n_te = max(60, int(TEST_FRAC_NET * len(idx)))
    te = idx[perm[:n_te]]
    yte = y[te]
    if len(np.unique(yte)) < 2 or len(idx) - n_te < 60:
        return None
    Xte, p1te = Xg[te], pb1[te]
    partners = [g for g in sites if g != h]
    rng.shuffle(partners)
    partners = partners[:MAX_PARTNERS]
    if len(partners) < 4:
        return None
    own_prev = float(y[idx].mean())
    base = full_metrics(yte, p1te)
    nb_c = f"nb@{float(T_SITE[outcome])}"          # column name of the net benefit at the outcome-specific representative threshold (midpoint of the outcome-specific range) (same as nbcol)
    rows = []
    for r in factors:
        P_list, prevs = [], []
        for g in partners:
            gi = np.where(hid == g)[0]
            if r != 1.0:
                gi = _resample_prev(gi, y, r, stable_seed("a7", outcome, int(h), int(g), r))
            d, yk = _topk_cos(Xg[gi], y[gi].astype(float), Xte, K_NEIGHBORS)
            P_list.append(_site_reply(d, yk))
            prevs.append(float(y[gi].mean()))
        p_net = np.column_stack(P_list).mean(axis=1)
        for a in alphas:
            pm = (1 - a) * p1te + a * p_net
            m = full_metrics(yte, pm)
            rows.append({"outcome": outcome, "hospitalid": int(h), "factor": float(r), "alpha": float(a),
                         "n_test": int(len(te)), "n_partners": int(len(partners)),
                         "prev_own": own_prev, "prev_partner": float(np.mean(prevs)),
                         "prev_diff": float(np.mean(prevs)) - own_prev,
                         "d_auroc": m["auroc"] - base["auroc"], "base_auroc": base["auroc"],
                         "base_slope": base["slope"], "base_intercept": base["intercept"],
                         "base_ece": base["ece"], "d_ece": m["ece"] - base["ece"],
                         "rep_threshold": float(T_SITE[outcome]), "base_nb_rep": base[nb_c], "nb_rep": m[nb_c], "d_nb_rep": m[nb_c] - base[nb_c], **m})
    return rows


# ---- meta-analysis (A6) ------------------------------------------------------------
def meta_reml_hk(d, v, exclude_zero_var=True):
    """random-effects meta-analysis of per-hospital estimates d and variances v. REML tau squared, Hartung-Knapp interval (original and truncated forms: q lower-bounded at 1),
    95% prediction interval (t distribution, k-2 degrees of freedom, truncated SE), inverse-variance Higgins I squared, Q and degrees of freedom. None if k<3. A hospital with any nan is dropped.
    hospitals with zero variance (v <= 1e-12) are dropped by default (exclude_zero_var=True). For such a hospital the retrieval prediction equals the frozen prediction, so the gain is also 0,
    and a variance floor of 1e-12 would give weight 1e12 and take over the whole pooled value (one hospital of F42 AKI retrieval held 99.9998% of the weight).
    exclude_zero_var=False keeps the variance floor 1e-12 in the weights (for the diagnostic columns *_orig and the re-aggregation of the reference hospital file a59).
    k_zero_var is the number of dropped hospitals, top1_weight_share is the largest share of weight held by one hospital."""
    d = np.asarray(d, dtype=float)
    v = np.asarray(v, dtype=float)
    zero = np.isfinite(d) & np.isfinite(v) & (v <= 1e-12)
    k_zero_var = int(zero.sum())
    ok = np.isfinite(d) & np.isfinite(v)
    if exclude_zero_var:
        ok &= ~zero
    d, v = d[ok], np.maximum(v[ok], 1e-12)
    k = len(d)
    if k < 3:
        return None
    w = 1.0 / v
    top1_weight_share = float(w.max() / w.sum())
    mu_fe = float((w * d).sum() / w.sum())
    Q = float((w * (d - mu_fe) ** 2).sum())
    I2 = max(0.0, (Q - (k - 1)) / Q) * 100 if Q > 0 else 0.0

    def nll(t2):
        wt = 1.0 / (v + t2)
        mu = (wt * d).sum() / wt.sum()
        return 0.5 * (np.log(v + t2).sum() + np.log(wt.sum()) + (wt * (d - mu) ** 2).sum())

    upper = max(10.0 * float(np.var(d, ddof=1)), 10.0 * float(v.max()), 1e-6)
    r = minimize_scalar(nll, bounds=(0.0, upper), method="bounded", options={"xatol": 1e-12})
    t2 = float(r.x) if nll(float(r.x)) < nll(0.0) - 1e-12 else 0.0
    wt = 1.0 / (v + t2)
    mu = float((wt * d).sum() / wt.sum())
    se_re = float(np.sqrt(1.0 / wt.sum()))
    q = float((wt * (d - mu) ** 2).sum() / (k - 1))
    se_hk = float(np.sqrt(q / wt.sum()))
    se_hkf = float(np.sqrt(max(1.0, q) / wt.sum()))
    tc = float(scipy_stats.t.ppf(0.975, k - 1))
    tp = float(scipy_stats.t.ppf(0.975, k - 2))
    s2 = float((k - 1) * w.sum() / (w.sum() ** 2 - (w ** 2).sum()))
    half_pi = tp * float(np.sqrt(t2 + se_hkf ** 2))
    return {"k": k, "mu_fixed": mu_fe, "Q": Q, "I2_invvar": I2, "tau2": t2, "tau": float(np.sqrt(t2)),
            "I2_reml": float(100 * t2 / (t2 + s2)) if (t2 + s2) > 0 else 0.0,
            "mu_random": mu, "se_random": se_re,
            "ci_z_lo": mu - 1.96 * se_re, "ci_z_hi": mu + 1.96 * se_re,
            "ci_hk_lo": mu - tc * se_hk, "ci_hk_hi": mu + tc * se_hk,
            "ci_hk_trunc_lo": mu - tc * se_hkf, "ci_hk_trunc_hi": mu + tc * se_hkf,
            "df": k - 1, "Q_p": float(scipy_stats.chi2.sf(Q, k - 1)), "q_hk": q,
            "pi_lo": mu - half_pi, "pi_hi": mu + half_pi,
            "k_zero_var": k_zero_var, "top1_weight_share": top1_weight_share}


def i2_stays_weighted(d, n):
    """Stay-weighted I squared (Q with stays as weights). The weight is not the inverse variance, so it is non-standard."""
    d = np.asarray(d, dtype=float)
    w = np.asarray(n, dtype=float)
    if len(d) < 3:
        return np.nan
    mu = float(np.average(d, weights=w))
    Q = float(((d - mu) ** 2 * w).sum())
    return max(0.0, (Q - (len(d) - 1)) / Q * 100) if Q > 0 else 0.0


# ---- actual MIMIC-IV year periods (A1) -----------------------------------------------
def read_mimic4_periods():
    """actual year period of a stay (MIMIC-IV). Two methods.
    (a) unit-transfer adjustment (M4_HAS_SHIFT): when patients has anchor_year and icustays has intime. anchor_year_group of patients is
        the actual 3-year period, and the period is shifted by the difference between the shifted year of the stay (intime) and anchor_year (PhysioNet MIMIC-IV documentation).
        if the stay falls entirely inside one 3-year period it is strict; if it straddles, it is dropped from strict. majority puts it in the period with most of its years.
    (b) patient-level approximation (copy without anchor_year): all stays of the same patient get the anchor_year_group period of that patient. There is no unit-transfer
        adjustment and intime is not used. MIMIC-IV shifts dates by a different amount per patient, so the intime order between patients is not the real calendar order.
        the patient_group column of the table is the period number (A1b uses only patient_group in this method)."""
    c_st = [_ci["stay_id"], _ci["subject_id"]] + ([_ci["intime"]] if M4_HAS_SHIFT else [])
    c_pt = [_cp["subject_id"], _cp["anchor_year_group"]] + ([_cp["anchor_year"]] if M4_HAS_SHIFT else [])
    # usecols returns columns in file order, so select again by name (naming by position would swap stay_id and subject_id)
    st = pd.read_csv(MIMIC4_ICUSTAYS_PATH, usecols=c_st)[c_st]
    st.columns = ["stay_id", "subject_id"] + (["intime"] if M4_HAS_SHIFT else [])
    pt = pd.read_csv(MIMIC4_PATIENTS_PATH, usecols=c_pt)[c_pt]
    pt.columns = ["subject_id", "anchor_year_group"] + (["anchor_year"] if M4_HAS_SHIFT else [])
    pt = pt.drop_duplicates("subject_id")
    m = st.merge(pt, on="subject_id", how="left")
    grp = m["anchor_year_group"].astype(str).str.extract(r"(\d{4})\s*-\s*(\d{4})")
    lo = pd.to_numeric(grp[0], errors="coerce")
    hi = pd.to_numeric(grp[1], errors="coerce")
    pairs = sorted({(int(a), int(b)) for a, b in zip(lo.dropna(), hi.dropna())})
    labels = [f"{a}-{b}" for a, b in pairs]
    if not M4_HAS_SHIFT:
        at = {p_: i_ for i_, p_ in enumerate(pairs)}
        pg = np.full(len(m), -1, dtype=int)
        for i, (a, b) in enumerate(zip(lo.to_numpy(dtype=float), hi.to_numpy(dtype=float))):
            if np.isfinite(a) and np.isfinite(b):
                pg[i] = at[(int(a), int(b))]
        out = pd.DataFrame({"stay_id": pd.to_numeric(m["stay_id"], errors="coerce").astype("int64"),
                            "shift": np.nan, "strict": pg, "majority": pg, "patient_group": pg})
        return out.set_index("stay_id"), labels
    shift = pd.to_datetime(m["intime"], errors="coerce").dt.year - pd.to_numeric(m["anchor_year"], errors="coerce")
    r_lo, r_hi = (lo + shift).to_numpy(dtype=float), (hi + shift).to_numpy(dtype=float)

    def which(y_):
        for gi_, (a, b) in enumerate(pairs):
            if a <= y_ <= b:
                return gi_
        return -1

    strict = np.full(len(m), -1, dtype=int)
    major = np.full(len(m), -1, dtype=int)
    for i in range(len(m)):
        if not (np.isfinite(r_lo[i]) and np.isfinite(r_hi[i])):
            continue
        yrs = range(int(r_lo[i]), int(r_hi[i]) + 1)
        gs = [which(y_) for y_ in yrs]
        if all(g_ == gs[0] for g_ in gs) and gs[0] >= 0:
            strict[i] = gs[0]
        cnt = {}
        for g_ in gs:
            if g_ >= 0:
                cnt[g_] = cnt.get(g_, 0) + 1
        if cnt:
            major[i] = sorted(cnt, key=lambda g_: (-cnt[g_], g_))[0]
    out = pd.DataFrame({"stay_id": pd.to_numeric(m["stay_id"], errors="coerce").astype("int64"),
                        "shift": shift.to_numpy(dtype=float), "strict": strict, "majority": major})
    return out.set_index("stay_id"), labels


# ---- temporal validation runner (A1) -------------------------------------------------------
def mean_abs_smd(XA, XB):
    """mean absolute standardized mean difference over columns of the standardized feature matrices of two periods."""
    ma, mb = XA.mean(0), XB.mean(0)
    sa, sb = XA.std(0, ddof=1), XB.std(0, ddof=1)
    sp = np.sqrt((sa ** 2 + sb ** 2) / 2.0)
    sp = np.where(sp < 1e-9, np.nan, sp)
    return float(np.nanmean(np.abs(ma - mb) / sp))


def run_temporal(C, past_mask, test_mask, name, outcome, banks, n_draws, groups):
    """splits the test-period patients into N_TIME_SPLITS folds stratified by outcome and tests fold by fold in turn (role alternation).
    P past bank (past period only) | S same-period bank (test period outside the test fold) | R (optional) everything outside the test fold (periods mixed).
    the three policies are scored on the same whole test patients, so differences between policies are paired. Compare at the same size only:
    if bank is "full", the policies are reduced to the largest size all can supply (n_max) and labeled "max"."""
    y = C["y"]
    test_ids = np.where(test_mask)[0]
    past_ids = np.where(past_mask)[0]
    notes = []
    if len(test_ids) < 4 * N_TIME_SPLITS or len(past_ids) < 50 or len(np.unique(y[test_ids])) < 2:
        notes.append({"analysis": name, "outcome": outcome, "note": f"insufficient sample (test period {len(test_ids)}, past {len(past_ids)})"})
        return [], [], notes
    folds = list(StratifiedKFold(n_splits=N_TIME_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(test_ids)), y[test_ids]))
    n_same = min(len(tr) for tr, _ in folds)                       # pool size of policy S (smallest fold)
    n_max = min(len(past_ids), n_same)
    rows, info_rows = [], []
    for bank_size in banks:
        size = n_max if bank_size == "full" else bank_size
        label = "max" if bank_size == "full" else str(bank_size)
        if size > n_max or size < 30:
            notes.append({"analysis": name, "outcome": outcome, "bank": label,
                          "note": f"insufficient pool (past {len(past_ids)} , same period {n_same} the smaller of {n_max} < bank {size})"})
            continue
        n_rep = 1 if bank_size == "full" else n_draws
        for pol in A1_POLICIES:
            for rep in range(n_rep):
                oof = {}
                for fi, (tr_i, te_i) in enumerate(folds):
                    test, other = test_ids[te_i], test_ids[tr_i]
                    pool = {"P": past_ids, "S": other, "R": np.concatenate([past_ids, other])}[pol]
                    key = f"a1|{name}|{pol}|{outcome}|{label}|{fi}|{rep}"
                    sel = np.arange(len(pool)) if size >= len(pool) else sample_bank(y[pool], size, key)
                    bank = pool[sel]
                    if np.intersect1d(bank, test).size or (pol == "P" and not past_mask[bank].all()):
                        raise AssertionError("invariant violation: the bank contains test patients or the P bank contains non-past patients " + key)
                    P, info = bank_predictions(C, bank, test, key, set(groups))
                    for m, v in P.items():
                        if m not in oof:
                            oof[m] = np.full(len(y), np.nan, dtype=np.float32)
                        oof[m][test] = v
                    info_rows.append({"analysis": name, "outcome": outcome, "bank": label, "policy": pol,
                                      "rep": rep, "fold": fi, "n_bank": int(len(bank)),
                                      "events_bank": int(y[bank].sum()), **info})
                for m, pm in oof.items():
                    pt = pm[test_ids].astype(float)
                    if np.isnan(pt).any():
                        continue
                    rows.append({"analysis": name, "outcome": outcome, "bank": label, "policy": pol,
                                 "rep": rep, "method": m, "n_test": int(len(test_ids)), "n_bank": int(size),
                                 **full_metrics(y[test_ids], pt)})
    return rows, info_rows, notes


def period_summary(C, past_mask, test_mask, name, outcome):
    """size, prevalence and feature shift of the past and test periods (for A1 description)."""
    y = C["y"]
    pa, te = np.where(past_mask)[0], np.where(test_mask)[0]
    smd = mean_abs_smd(C["Xg"][pa], C["Xg"][te]) if len(pa) > 2 and len(te) > 2 else np.nan
    return {"analysis": name, "outcome": outcome, "n_past": int(len(pa)), "n_test": int(len(te)),
            "prev_past": float(y[pa].mean()) if len(pa) else np.nan,
            "prev_test": float(y[te].mean()) if len(te) else np.nan, "mean_abs_smd": smd}


# ---- TabPFN connection (A9) ------------------------------------------------------------
TABPFN_INFO = {}                  # filled once when the first classifier is created: package version, v2 selection method, model_version, weight path argument


def tabpfn_classifier(device, seed):
    """creates the classifier by selecting the TabPFN v2 weights explicitly. Only the arguments accepted by the installed tabpfn are passed (the signature is read).
    behaviour by version (read from the sources of tabpfn 2.0.9, 2.1.3, 2.2.1, 6.0.0 and 9.1.0):
      - in 6.0.0 and 9.1.0, TabPFNClassifier.create_default_for_version(ModelVersion.V2, **kw) puts the v2 weight path in as model_path
        (classifier.py of 9.1.0 and 6.0.0. ModelVersion is in tabpfn.constants).
        versions between the two have not been tested. If the function is missing or fails, the fallback below is used and that fact is recorded in model_version.
        the default of 9.1.0 (when model_path is auto, settings.py, model_version=V3_5) is v3.5, not v2. So the default constructor is not used.
        the v2 weight file name has no v2.5, v2.6, v3 or v3.5 marker, so when 9.1.0 reads the version from the file name (model_loading.py) it reads V2.
      - 2.x has no selection function and only v2 weights, so the default constructor is v2 (ModelVersion in 2.2.1 model_loading.py has V2 only).
    if v2 cannot be selected, the run does not stop but continues with the default weights and records that fact in model_version."""
    from tabpfn import TabPFNClassifier
    params = inspect.signature(TabPFNClassifier.__init__).parameters
    kw = {}
    if "device" in params:
        kw["device"] = device
    if "n_estimators" in params:
        kw["n_estimators"] = TABPFN_N_ESTIMATORS
    if "random_state" in params:
        kw["random_state"] = seed
    factory = getattr(TabPFNClassifier, "create_default_for_version", None)
    clf, how, ver = None, "default constructor (this installation has no function to select the model version)", None
    if factory is not None:
        try:
            from tabpfn.constants import ModelVersion
            clf = factory(ModelVersion.V2, **kw)
            how, ver = "create_default_for_version(ModelVersion.V2)", "v2"
        except Exception as exc:  # noqa: BLE001
            how = f"v2 selection failed ({type(exc).__name__}: {str(exc)[:80]}), continuing with the default weights"
            ver = "unconfirmed: v2 selection failed, default weights"
    if clf is None:
        clf = TabPFNClassifier(**kw)
        if ver is None:
            ver = "v2 (tabpfn 2.x has v2 weights only)" if TABPFN_PKG_VERSION.startswith("2.") else "unconfirmed: no function to select the model version"
    if not TABPFN_INFO:
        TABPFN_INFO.update({"tabpfn_package_version": TABPFN_PKG_VERSION, "how_v2_selected": how, "model_version": ver,
                            "model_path_arg": str(getattr(clf, "model_path", "attribute absent")), "device": str(device),
                            "n_estimators": str(TABPFN_N_ESTIMATORS)})
    return clf


def tabpfn_info_rows():
    """installation and weight table printed by A9. Returns two sets (rows with full paths for the screen, rows with file names only for csv; the csv rows contain file names only, no folder paths)."""
    mp = TABPFN_INFO.get("model_path_arg", "")
    if mp in ("", "auto", "attribute absent"):
        wfile = "model_path is " + (mp if mp else "empty") + " (package default path, the file name is decided by this installation)"
    else:
        _f = Path(mp)
        wfile = _f.name + (" (present %.0f MB)" % (_f.stat().st_size / 1e6) if _f.is_file() else " (file not found)")
    items = [("tabpfn package version", TABPFN_INFO.get("tabpfn_package_version", "unknown")), ("v2 selection method", TABPFN_INFO.get("how_v2_selected", "unknown")),
             ("model_version", TABPFN_INFO.get("model_version", "unknown")), ("weight file", wfile),
             ("softmax_temperature (value after fitting)", TABPFN_INFO.get("softmax_temperature_after_fit", "before fitting")),
             ("device", TABPFN_INFO.get("device", "")), ("n_estimators", TABPFN_INFO.get("n_estimators", ""))]
    full = items + [("model_path argument (full path)", mp)]
    return full, items


def _is_oom(exc):
    """whether it is a GPU out-of-memory error (torch.OutOfMemoryError, TabPFNCUDAOutOfMemoryError, or out of memory in the message)."""
    return ("OutOfMemory" in type(exc).__name__) or ("out of memory" in str(exc).lower())


def tabpfn_predict(clf, X, batch=TABPFN_BATCH):
    """prediction in batches. On out of memory the batch is halved and the same batch is retried (minimum 8 rows). The predictions themselves do not depend on the batch size
    (TabPFN predicts each test row independently with the same context)."""
    out = np.empty(len(X), dtype=float)
    i, b = 0, max(int(batch), 1)
    while i < len(X):
        try:
            out[i:i + b] = clf.predict_proba(X[i:i + b])[:, 1]
            i += b
        except Exception as exc:  # noqa: BLE001
            if not _is_oom(exc) or b <= 8:
                raise
            b = max(b // 2, 8)
            try:
                import torch as _t
                _t.cuda.empty_cache()
            except Exception:  # noqa: BLE001
                pass
            print(f"  [A9] GPU out of memory , prediction batch reduced to {b} rows and retried", flush=True)
    return out


def tabpfn_fit_subsets(Xs, ys, n_ctx, n_sub, device, outcome):
    """fits TabPFN on n_sub stratified subsamples (n_ctx rows) of the development training data and freezes it (the model weights do not change)."""
    clfs = []
    for s in range(n_sub):
        idx = sample_bank(ys, n_ctx, f"tabpfn|{outcome}|{s}")
        clf = tabpfn_classifier(device, stable_seed("tabpfn", outcome, s))
        clf.fit(Xs[idx], ys[idx])
        TABPFN_INFO.setdefault("softmax_temperature_after_fit", str(getattr(clf, "softmax_temperature_", "attribute absent")))
        clfs.append(clf)
    return clfs


def tabpfn_frozen_probs(clfs, X):
    return np.mean([tabpfn_predict(c, X) for c in clfs], axis=0)


def stratified_cap(y, cap, key):
    """reduces the target cohort to cap rows (stratified). All rows if not reduced."""
    if cap is None or len(y) <= cap:
        return np.arange(len(y))
    return sample_bank(np.asarray(y), int(cap), key)


A9_RENAME = {"b1_base": "tabpfn_frozen", "b5_base": "logistic_frozen", "pram_b1": "pram_on_tabpfn",
             "pram_b5": "pram_on_logistic", "platt_b1": "platt_on_tabpfn", "recal_iso": "isotonic_on_tabpfn",
             "upd1_b1": "upd1_tabpfn", "upd2_b1": "upd2_tabpfn", "upd3_b1": "upd3_tabpfn", "ct_b1": "closed_test_tabpfn",
             "upd1_b5": "upd1_logistic", "upd2_b5": "upd2_logistic", "upd3_b5": "upd3_logistic", "ct_b5": "closed_test_logistic"}


# ---- reporting helpers (shared by the grid and hospital-level reporting cells) --------------------------------
def nbcol(t):
    return f"nb@{float(t)}"


def pick_report_banks(banks):
    pref = [b for b in (1000, "full") if b in banks]
    return pref if pref else [banks[0], banks[-1]]


def bank_sort_key(b):
    return 10 ** 9 if str(b) in ("full", "max") else int(b)


def draw_median_table(df, methods, extra_cols=()):
    """table of medians over draws (rep). Columns: AUROC, slope, intercept, ECE, net benefit at the representative threshold, net benefit at 20%."""
    d = df[df["method"].isin(methods)].copy()
    d["nb_site"] = [r[nbcol(T_SITE[oc])] for oc, r in zip(d["outcome"], d.to_dict("records"))]
    d["nb_20"] = d[nbcol(THRESH_MS)]
    cols = ["auroc", "slope", "intercept", "ece", "nb_site", "nb_20"] + list(extra_cols)
    g = d.groupby(["fs", "outcome", "target", "bank", "method"], sort=False)[cols].median().reset_index()
    g["n_draws"] = d.groupby(["fs", "outcome", "target", "bank", "method"], sort=False)["rep"].nunique().values
    return g


def paired_gap(df, method, base, col="auroc"):
    """summary of the distribution of the difference (col) between method and base in the same draws: median, 25%, 75%, minimum, fraction below 0."""
    keys = ["fs", "outcome", "target", "bank", "rep"]
    a = df[df["method"] == method].set_index(keys)[col]
    b = df[df["method"] == base].set_index(keys)[col]
    d = (a - b.reindex(a.index)).dropna()
    if len(d) == 0:
        return pd.DataFrame()
    g = d.groupby(["fs", "outcome", "target", "bank"], sort=False)
    out = g.agg(gap_median="median", gap_q25=lambda s: s.quantile(0.25), gap_q75=lambda s: s.quantile(0.75),
                gap_min="min", share_below_zero=lambda s: float((s < 0).mean()), n_draws="size").reset_index()
    out.insert(4, "method", method)
    out.insert(5, "base", base)
    return out


def hospital_time_unit(C, outcome, h, hid, year, y_past, y_test, groups):
    """A1 hospital level: the whole past year of hospital h is the bank and the whole next year is the test. Only hospitals above the criterion in both years."""
    y = C["y"]
    idx = np.where(hid == h)[0]
    bank, test = idx[year[idx] == y_past], idx[year[idx] == y_test]
    for part in (bank, test):
        ev = int(y[part].sum())
        if len(part) < HOSP_MIN_N_PER_YEAR or ev < HOSP_MIN_EVENTS_PER_YEAR or len(part) - ev < HOSP_MIN_EVENTS_PER_YEAR:
            return None
    P, info = bank_predictions(C, bank, test, f"a1h|{outcome}|{h}", set(groups))
    yt = y[test]
    rows = []
    for m, v in P.items():
        _a, _b, d1, v1 = delong_delta_var(yt, v, P["b1_base"])
        rows.append({"outcome": outcome, "hospitalid": int(h), "method": m, "n_bank": int(len(bank)), "n_test": int(len(test)),
                     "events_test": int(yt.sum()), **full_metrics(yt, v), "d_auroc_b1": d1, "var_b1": v1})
    return rows



def prepare_sites():
    """builds the hospital id array and the list of eligible hospitals (shared by the hospital-level cell and the network cell)."""
    hid, sites = {}, {}
    for oc in OUTCOMES:
        c = CTX[("F42", oc, "eICU")]
        hid[oc] = map_to_stays(EICU_TAB, "hospitalid", c["stay_ids"])
        s = eligible_sites(c["y"], hid[oc])
        sites[oc] = s[:SITE_LIMIT] if SITE_LIMIT else s
    return hid, sites

def as_table_frame(df, a="analysis", p="policy"):
    """temporarily renames the A1 result to the column names (fs, target) that draw_median_table and paired_gap read."""
    return df.rename(columns={a: "fs", p: "target"})


def policy_contrast(tab, method, base, col):
    """A1: puts the gains (col) of the same strategy side by side under policies P (past), S (same period) and R (if present), and gives P-S and P-R."""
    g = paired_gap(tab, method, base, col)
    if len(g) == 0:
        return pd.DataFrame()
    w = g.pivot_table(index=["fs", "outcome", "bank"], columns="target", values="gap_median", aggfunc="first")
    for pol in ("P", "S", "R"):
        if pol not in w.columns:
            w[pol] = np.nan
    w["P_minus_S"] = w["P"] - w["S"]
    w["P_minus_R"] = w["P"] - w["R"]
    w = w.reset_index().rename(columns={"fs": "analysis"})
    w.insert(3, "method", method)
    w.insert(4, "metric", col)
    return w


# ---- tail risk (A2): unit of (one bank, one test fold) -------------------------------------
def run_tail_unit(C, fs, outcome, target, bank_size, groups, keep, delong_pairs):
    """one cell (feature set, outcome, target, bank size). Makes N_FOLDS x N_DRAWS_TAIL // N_FOLDS (bank, test fold) pairs.
    one pair is one measurement in which one hospital receives one bank and measures on its own test patients."""
    y = C["y"]
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
    per_fold = max(1, N_DRAWS_TAIL // N_FOLDS)
    rows = []
    for fi, (pool_idx, test_idx) in enumerate(folds):
        if bank_size > len(pool_idx):
            continue
        yt = y[test_idx]
        for dr in range(per_fold):
            key = f"{outcome}|{target}|{bank_size}|{fi}|{dr}"
            bank = pool_idx[sample_bank(y[pool_idx], bank_size, "tail|" + key)]
            if len(np.unique(y[bank])) < 2:
                continue
            P, _info = bank_predictions(C, bank, test_idx, key, set(groups))
            for m in keep:
                if m not in P:
                    continue
                row = {"fs": fs, "outcome": outcome, "target": target, "bank": str(bank_size), "fold": fi, "draw": dr,
                       "method": m, "n_events_bank": int(y[bank].sum()), **full_metrics(yt, P[m])}
                if delong_pairs.get(m) in P:
                    _a, _b, dd, vv = delong_delta_var(yt, P[m], P[delong_pairs[m]])
                    row["d_auroc_base"], row["var_auroc_base"] = dd, vv
                rows.append(row)
    return rows


def tail_gaps(df, pairs):
    """turns the tail rows (metrics per method) into a table of (method - reference) pair differences. Columns: gap_auroc gap_ece gap_slope gap_nb_site gap_nb_20 dl_lo dl_hi."""
    t = df.copy()
    t["nb_site"] = [r[nbcol(T_SITE[oc])] for oc, r in zip(t["outcome"], t.to_dict("records"))]
    t["nb_20"] = t[nbcol(THRESH_MS)]
    keys = ["fs", "outcome", "target", "bank", "fold", "draw"]
    cols = ["auroc", "ece", "slope", "nb_site", "nb_20"]
    out = []
    for m, b in pairs.items():
        a = t[t["method"] == m].set_index(keys)
        bb = t[t["method"] == b].set_index(keys)
        if len(a) == 0 or len(bb) == 0:
            continue
        g = (a[cols] - bb[cols].reindex(a.index)).add_prefix("gap_")
        g["method"] = m
        if "d_auroc_base" in a.columns:
            sd = np.sqrt(a["var_auroc_base"].astype(float))
            g["dl_lo"], g["dl_hi"] = a["d_auroc_base"] - 1.96 * sd, a["d_auroc_base"] + 1.96 * sd
        out.append(g.reset_index())
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def tail_summary(G):
    """tail of the gain distribution per (feature set, source, outcome, target, bank, method) cell. Cells without a metric (such as net benefit in the rows read from the reference file) are left empty."""
    rows = []
    for (fs, src, oc, tg, bk, m), s in G.groupby(["fs", "source", "outcome", "target", "bank", "method"], sort=False):
        a = s["gap_auroc"].dropna().to_numpy()
        if len(a) == 0:
            continue
        n = len(a)
        row = {"fs": fs, "source": src, "outcome": oc, "target": tg, "bank": bk, "method": m, "n_draws": n,
               "dAUROC_median": float(np.median(a)), "dAUROC_p05": float(np.quantile(a, 0.05)),
               "dAUROC_p10": float(np.quantile(a, 0.10)), "dAUROC_worst": float(a.min()), "share_below_zero": float((a < 0).mean())}
        for mg in NI_MARGINS_AUROC:
            row[f"auroc_share_lost_more_than_{mg}"] = float((a < -mg).mean())
        k = int((a < -NI_MARGIN_AUROC).sum())
        row["auroc_ni_share"] = k / n
        row["auroc_ni_cp_lo"], row["auroc_ni_cp_hi"] = cp_interval(k, n)
        if "dl_hi" in s.columns and s["dl_hi"].notna().any():
            dl = s.dropna(subset=["dl_lo", "dl_hi"])
            row["draws_ci_wholly_below_zero"] = float((dl["dl_hi"] < 0).mean())
            row["draws_ci_wholly_above_zero"] = float((dl["dl_lo"] > 0).mean())
        for col, tag in (("gap_ece", "ece"), ("gap_slope", "slope")):
            v = s[col].dropna()
            if len(v):
                row[f"d_{tag}_median"] = float(v.median())
        v = s["gap_ece"].dropna()
        if len(v):
            row["share_ece_worse"] = float((v > 0).mean())
        for col, tag in (("gap_nb_site", "nb_site"), ("gap_nb_20", "nb_20")):
            v = s[col].dropna().to_numpy()
            if len(v) == 0:
                continue
            row[f"d_{tag}_median"], row[f"d_{tag}_p05"], row[f"d_{tag}_worst"] = float(np.median(v)), float(np.quantile(v, 0.05)), float(v.min())
            for mg in NI_MARGINS_NB:
                row[f"{tag}_share_lost_more_than_{mg}"] = float((v < -mg).mean())
            kn = int((v < -NI_MARGIN_NB).sum())
            row[f"{tag}_ni_share"] = kn / len(v)
            row[f"{tag}_ni_cp_lo"], row[f"{tag}_ni_cp_hi"] = cp_interval(kn, len(v))
        rows.append(row)
    return pd.DataFrame(rows)


## Write lock, check of stored results, hashes of existing files (done before reading the data)

1. **Write lock.** `Resumable.add_row` and `mark_unit` write only to names in `R5_NEW_NAMES`. An attempt to write another name (resume of notebooks 01 and 02) stops with `R5Stop`. `save_csv` writes only names starting with `R5_` (the saving of `frozen_model_auroc.csv` called by the data loading cell is also skipped).
2. **Check of stored results (does not stop).** A table prints how many grid, rescoring, hospital-level, network and 33-column tail resume files and csv files finished by notebook 01 exist in this folder. It does not stop because they are absent: the input-collection section below reads what exists and recomputes what is absent or insufficient with the same function.
3. **Hashes of existing files.** The sha256 of the resume, csv and md files that exist now are collected. The last cell recomputes them and prints whether they are unchanged.


In [ ]:
# ================================================================
# PRAM write lock | check of stored results (does not stop) | hashes of existing files
# ================================================================
class R5Stop(BaseException):
    """An attempt was made to write an existing file or the settings do not match. Stops before computing (it does not stop merely because stored results are absent). It is a BaseException, so `except Exception` does not swallow it."""


def r5_count_units(name):
    path = RESUME_DIR / f"{name}.units.jsonl"
    if not path.is_file():
        return 0
    with open(path, "r", encoding="utf-8") as _f:
        return sum(1 for _ln in _f if _ln.strip())


def r5_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as _f:
        for _blk in iter(lambda: _f.read(1 << 20), b""):
            h.update(_blk)
    return h.hexdigest()


def r5_snapshot():
    """relative path -> sha256 of the resume, csv and md files left by the stored results. New files of this notebook (R5_ prefix) are not included."""
    snap = {}
    for _d, _pat in ((RESUME_DIR, "*"), (CSV_DIR, "*"), (OUT_DIR, "*.md")):
        for _p in sorted(_d.glob(_pat)):
            if _p.is_file() and not _p.name.startswith(R5_CSV_PREFIX) and _p.name not in (R5_SUM_NAME, R5_RES_NAME):
                snap[_p.relative_to(OUT_DIR).as_posix()] = r5_sha256(_p)
    return snap


R5_SNAP0 = r5_snapshot()

if not hasattr(Resumable, "_r5_add_row_plain"):
    Resumable._r5_add_row_plain = Resumable.add_row
    Resumable._r5_mark_unit_plain = Resumable.mark_unit

    def _r5_guarded_add_row(self, row):
        if self.name not in R5_NEW_NAMES:
            raise R5Stop("stop , tried to write the existing resume file `%s`. This notebook writes only the new name %s. The existing file was not changed." % (self.name, list(R5_NEW_NAMES)))
        return Resumable._r5_add_row_plain(self, row)

    def _r5_guarded_mark_unit(self, unit):
        if self.name not in R5_NEW_NAMES:
            raise R5Stop("stop , tried to write unit completion to the existing resume file `%s`. This notebook writes only the new name %s. The existing file was not changed." % (self.name, list(R5_NEW_NAMES)))
        return Resumable._r5_mark_unit_plain(self, unit)
    Resumable.add_row = _r5_guarded_add_row
    Resumable.mark_unit = _r5_guarded_mark_unit

_save_csv_plain = save_csv


def save_csv(df, name):
    """This notebook writes only names starting with R5_. Other names (stored csv files) are not written and one line is left."""
    if not name.startswith(R5_CSV_PREFIX):
        print("  (not saved) %s , not a new file of this notebook. The existing file is not touched" % name)
        return None
    if df is None or len(getattr(df, "columns", [])) == 0:
        print("  (not saved) %s , an empty table with no column (the input is absent). A csv without columns is not made" % name)       # a table without columns (hospital id table absent) is not written, because the identifier column check of the last cell reads the saved csv files
        return None
    return _save_csv_plain(df, name)


R5_HAVE = {n: r5_count_units(n) for n in ("matrix_F42", "matrix_F33", "rescore_F42", "a2_tail_F33", "hospital_F42", "hospital_F33", "network_F42", "network_F33", "a9_tabpfn")}
print("check of stored results , folder %s" % RESUME_DIR.resolve())
print("| resume file | finished units | total readable | if absent or insufficient |")
print("|---|---|---|---|")
for _n, _v in R5_HAVE.items():
    _need = R5_INVENTORY_NEED.get(_n)
    print("| %s | %d | %s | %s |" % (_n, _v, _need if _need is not None else "counted after reading the data", "not read; recomputed with the same function" if _n != "a9_tabpfn" else "back-calculated from the aggregate constants of the stored csv (TabPFN is not rerun)"))
for _c in ("RESCORE_table2_q.csv", "A2_tail_summary.csv"):
    print("| github_csv/%s | %s | | if absent, compared with the aggregate constants of the stored csv in the cell |" % (_c, "present" if (CSV_DIR / _c).is_file() else "absent"))
print("hashes of %d existing files were collected (resume, github_csv, md). If 0, the folder is empty (or new): the input-collection section below recomputes the values needed" % len(R5_SNAP0))
if not ({R5_WHOLE_FS} | set(R5_TAIL_FS) | {"F42"}) <= set(FS_RUN):
    raise R5Stop("stop , the feature sets required are not in FS_RUN (required %s, FS_RUN %s)." % (sorted({R5_WHOLE_FS} | set(R5_TAIL_FS) | {"F42"}), FS_RUN))
print("check of stored results finished (does not stop) , write lock on (only %d new names are written)" % len(R5_NEW_NAMES))


## Data loading, feature sets, frozen models

The six input files described in the repository README are read from `prepared_data/`. The 33-column set drops the variables whose maximum missingness over the three databases exceeds 30% (the 9 columns are removed before input and the development cache is rebuilt). The notebook prints whether the dropped variables are the nine expected ones (temperature, INR and weight; minimum, maximum and last).

For each feature set the development models (frozen logistic regression and CatBoost) are refitted on the MIMIC-IV training 56%.

| Set | Columns | Targets | Use |
|---|---|---|---|
| F42 | 42 inputs (43 after transform) | three cohorts | main analysis and update hierarchy; reference check |
| F33 | 33 inputs (34 after transform) | three cohorts | 33-column analyses |
| F42_noTemp, F42_noINR | only the three temperature columns or only the three INR columns removed | eICU | off (runs if added to `FS_RUN`) |

The printout of the number of inputs and of columns after the transform (42, 43, 33 and 34) is the feature count check. The column names are listed in Supplementary S1 Panel B.


In [ ]:
# ================================================================
# PRAM Cell 4. Data loading, feature sets, frozen models, A10
# ================================================================
DS, DEV, CTX, FS_COLS = {}, {}, {}, {}
EICU_TAB, SITE_CODE = None, {}
CAT_REF, NUM_REF, ALLVARS, SPARSE = [], [], [], []
MISS_TAB = pd.DataFrame()
STATUS = {}                                   # status per section. Read by the closing table (was it finished) of the last cell


def status(section, state, note=""):
    STATUS[section] = (state, note)


if not DATA_OK:
    emit("**[skipped] The main inputs are missing.** Look at the input check table of cell 1, fix `DATA_ROOT` and run again. All analysis cells below are skipped.")
    status("data loading", "not available", "main inputs absent")
else:
    t_load = time.time()
    for _oc in OUTCOMES:
        DS[("MIMIC-IV", _oc)] = load_mimic4(_oc)
        DS[("MIMIC-III", _oc)] = load_external("mimic3", _oc)
        DS[("eICU", _oc)] = load_external("eicu", _oc)
    X_REF, _ = _load_x("mimic4")
    CAT_REF, NUM_REF = _detect_cols(X_REF)
    ALLVARS = CAT_REF + NUM_REF
    _rows = []
    for _name in TARGETS:
        _Xall = X_REF if _name == "MIMIC-IV" else DS[(_name, "AKI_168h")]["X_df"]
        for _c in ALLVARS:
            _rows.append({"database": _name, "variable": _c, "pct_missing": 100.0 * float(_Xall[_c].isna().mean())})
    MISS_TAB = pd.DataFrame(_rows)
    _mm = MISS_TAB.groupby("variable")["pct_missing"].max()
    SPARSE = sorted([c for c in ALLVARS if _mm[c] > MISS_CUT_PCT])
    emit("### Cohort")
    _coh = [{"database": n, "outcome": oc, "n": len(DS[(n, oc)]["y"]), "prevalence": float(DS[(n, oc)]["y"].mean()),
             "n_inputs": DS[(n, oc)]["X_df"].shape[1]} for n in TARGETS for oc in OUTCOMES]
    emit_table("cohort size and prevalence", pd.DataFrame(_coh))
    emit(f"- number of feature inputs {len(ALLVARS)} (expected 42) | categorical {CAT_REF} | continuous {len(NUM_REF)} columns")
    emit(f"- maximum missingness over the three databases is {MISS_CUT_PCT}% are dropped {len(SPARSE)} columns: {SPARSE}")
    emit("- the nine expected variables and " + ("**equal**" if SPARSE == sorted(EXPECTED_SPARSE_9) else f"**differ** (expected {sorted(EXPECTED_SPARSE_9)})"))
    for _fs in FS_RUN:
        _sp = FS_SPEC[_fs]
        _cols = [c for c in ALLVARS if not (_sp["drop_rule"] == "miss" and c in SPARSE)
                 and not any(c.startswith(p) for p in _sp["drop_prefix"])]
        FS_COLS[_fs] = _cols
    emit("- number of feature-set columns: " + ", ".join(f"{k} {len(v)}" for k, v in FS_COLS.items()))
    status("data loading", "done", f"{time.time() - t_load:.0f} s")

    t_dev = time.time()
    for _fs in FS_RUN:
        for _oc in OUTCOMES:
            _d4 = DS[("MIMIC-IV", _oc)]
            _tr = _d4["idx_train"]
            DEV[(_fs, _oc)] = build_dev(_d4["X_df"].iloc[_tr], _d4["y"][_tr], FS_COLS[_fs], CAT_REF, NUM_REF, f"{_fs}|{_oc}")
            for _tg in FS_SPEC[_fs]["targets"]:
                _d = DS[(_tg, _oc)]
                if _tg == "MIMIC-IV":
                    _te = _d["idx_test"]
                    CTX[(_fs, _oc, _tg)] = make_ctx(DEV[(_fs, _oc)], _d["X_df"].iloc[_te], _d["y"][_te], _d["stay_ids"][_te], _oc)
                else:
                    CTX[(_fs, _oc, _tg)] = make_ctx(DEV[(_fs, _oc)], _d["X_df"], _d["y"], _d["stay_ids"], _oc)
            _log(f"frozen model fit {_fs} {_oc}")
    _chk = []
    for (_fs, _oc, _tg), _c in CTX.items():
        assert 0.0 <= _c["pb1"].min() and _c["pb1"].max() <= 1.0 and 0.0 <= _c["pb5"].min() and _c["pb5"].max() <= 1.0, "the frozen model output is not a probability"
        assert len(_c["y"]) == len(_c["Xg"]) == len(_c["Xc"]) == len(_c["stay_ids"]), "the context array lengths do not match"
        _a1 = roc_auc_score(_c["y"], _c["pb1"])
        _a5 = roc_auc_score(_c["y"], _c["pb5"])
        _chk.append({"fs": _fs, "outcome": _oc, "target": _tg, "n": len(_c["y"]), "prev": float(_c["y"].mean()),
                     "d": _c["d"], "auroc_logistic": _a1, "auroc_catboost": _a5})
    EICU_TAB = load_eicu_patient_table() if (HAVE_HOSPITALID or HAVE_EICU_YEAR) else None
    if EICU_TAB is not None and "hospitalid" in EICU_TAB.columns:        # only this code, not the hospital id, goes to the result csv
        SITE_CODE = {int(h): f"H{k + 1:03d}" for k, h in enumerate(sorted(EICU_TAB["hospitalid"].dropna().unique()))}
    emit_table("frozen model performance", pd.DataFrame(_chk), max_rows=60)
    save_csv(pd.DataFrame(_chk), "frozen_model_auroc")
    status("frozen model", "done", f"{time.time() - t_dev:.0f} s")
    for _fs in [f for f in ("F42", "F33") if f in FS_COLS]:                  # feature count consistency: number of inputs and number of columns after the transform
        _pg = DEV[(_fs, "AKI_168h")]["pg"]
        emit(f"- feature count consistency {_fs}: input {len(FS_COLS[_fs])} -> model columns after the transform {len(_pg.column_names())} (categorical {len(_pg.cat_cols)} expanded as one-hot)")


## Computation functions (three taken unchanged from the rescoring cell of notebook 01, plus the interval summary)

The bodies of `delong_p`, `bh_adjust` and `rescore_unit` are identical, character by character, to those of the rescoring cell of notebook 01. `rescore_unit` collects, for one unit (outcome, target, bank size), the 5-fold test predictions with the same split and the same draw key as the grid, and gives the paired DeLong of retrieval and the frozen logistic. The tail computation calls `run_tail_unit` of the new-functions cell of notebook 01 as it is. Newly written are only the interval summary (`r5_ci_summary`) and the table arrangement.


In [ ]:
# ================================================================
# PRAM computation functions (the bodies of delong_p, bh_adjust and rescore_unit of the rescoring cell, as they are) + interval summary
# ================================================================
RS_GROUPS = ("frozen", "pram")                         # same values as the rescoring cell: only one retrieval is needed


def delong_p(d, var):
    """p rule of the DeLong paired test: p = 1 if the variance is below 1e-15, otherwise a two-sided normal test."""
    if not (np.isfinite(d) and np.isfinite(var)):
        return float("nan")
    if var < 1e-15:
        return 1.0
    return float(2.0 * scipy_stats.norm.sf(abs(d / np.sqrt(var))))


def bh_adjust(p):
    """Benjamini-Hochberg adjusted p (q). The monotone form that keeps the order. nan is skipped and kept as nan."""
    p = np.asarray(p, dtype=float)
    ok = np.isfinite(p)
    q = np.full(len(p), np.nan)
    if ok.sum() == 0:
        return q
    pv = p[ok]
    order = np.argsort(pv)
    ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1.0)
    q_sorted = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(len(pv))
    out[order] = np.minimum(q_sorted, 1.0)
    q[ok] = out
    return q


def rescore_unit(C, outcome, target, bank_size, n_draws):
    """one unit (outcome, target, bank size). With the same split, the same draw key (m23) and the same bank_predictions as run_matrix_unit, collects the 5-fold test predictions per draw (rep)
    and gives the paired DeLong of retrieval (pram_b1) and frozen logistic (b1_base) (standard implementation; the variance and p of the original implementation are given alongside for explanation). For the full bank, the retrieval prediction variant key08 made with the alternative mixing-weight selection key
    (no prefix b1|) is also given (a diagnosis to see whether using the original implementation on the same predictions reproduces the stored q)."""
    y = C["y"]
    n = len(y)
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(n), y))
    full = bank_size == "full"
    rows = []
    for rep in range(1 if full else n_draws):
        oof = {m: np.full(n, np.nan, dtype=np.float32) for m in ("pram_b1", "b1_base")}
        oof08 = np.full(n, np.nan, dtype=np.float32)
        for fi, (pool_idx, test_idx) in enumerate(folds):
            key = f"{outcome}|{target}|{bank_size}|{fi}|{rep}"
            bank = pool_idx[np.arange(len(pool_idx)) if full else sample_bank(y[pool_idx], bank_size, "m23|" + key)]
            P, _info = bank_predictions(C, bank, test_idx, key, set(RS_GROUPS))
            for m in oof:
                oof[m][test_idx] = P[m]
            if full:
                Xb, yb, p1b = C["Xg"][bank], y[bank], C["pb1"][bank]
                Xt, yt, p1t = C["Xg"][test_idx], y[test_idx], C["pb1"][test_idx]
                p_retr = RETRIEVAL_FN({"X_tr_s": Xb, "y_tr": yb, "X_te_s": Xt, "y_te": yt, "mi": None, "mi_prob": None})
                icv08 = inner_cv_bank(Xb, yb, p1b, RETRIEVAL_FN, key=key)
                oof08[test_idx] = ((1 - icv08["alpha"]) * p1t + icv08["alpha"] * p_retr).astype(np.float32)
        variants = [("main", oof["pram_b1"].astype(float), oof["b1_base"].astype(float))]
        if full:
            variants.append(("key08", oof08.astype(float), np.asarray(C["pb1"], dtype=float)))
        for variant, pm, base in variants:
            if np.isnan(pm).any():
                continue
            a_p, a_b, dd, v_std = delong_delta_var(y, pm, base)
            v_org = delong_delta_var(y, pm, base, original=True)[3]
            rows.append({"outcome": outcome, "target": target, "bank": str(bank_size), "rep": rep, "variant": variant,
                         "auroc_pram": a_p, "auroc_frozen": a_b, "d_auroc": dd, "var_standard": v_std, "var_original": v_org,
                         "p_standard": delong_p(dd, v_std), "p_original": delong_p(dd, v_org)})
    return rows


def r5_p_label(p):
    """notation of p and q used in tables."""
    if not np.isfinite(p):
        return "not computed"
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def r5_ci_summary(df, d_col, v_col, keys):
    """for each cell (keys), builds the interval d +- R5_Z x sqrt(v) from the paired AUROC difference d and variance v and counts draws by interval position.
    columns: n_draws (valid) n_excluded d_median d_p05 d_worst share_d_below_m01 (point estimate below the margin) k_ci_above0 k_ci_below0 k_ci_below_m01 (interval upper bound below the minus margin)
    that fraction and the Clopper-Pearson interval of k_ci_below_m01. Draws with negative or nan variance are excluded from the count."""
    rows = []
    for kk, s in df.groupby(keys, sort=False):
        kk = kk if isinstance(kk, tuple) else (kk,)
        d = s[d_col].to_numpy(float)
        v = s[v_col].to_numpy(float)
        ok = np.isfinite(d) & np.isfinite(v) & (v >= 0)
        d, v = d[ok], v[ok]
        n = len(d)
        row = dict(zip(keys, kk))
        row.update({"n_draws": int(n), "n_excluded": int((~ok).sum())})
        if n:
            se = np.sqrt(v)
            lo, hi = d - R5_Z * se, d + R5_Z * se
            k_a, k_b, k_m = int((lo > 0).sum()), int((hi < 0).sum()), int((hi < -NI_MARGIN_AUROC).sum())
            cp_lo, cp_hi = cp_interval(k_m, n)
            row.update({"d_median": float(np.median(d)), "d_p05": float(np.quantile(d, 0.05)), "d_worst": float(d.min()),
                        "share_d_below_m01": float((d < -NI_MARGIN_AUROC).mean()), "se_median": float(np.median(se)),
                        "k_ci_above0": k_a, "k_ci_below0": k_b, "k_ci_below_m01": k_m,
                        "share_ci_above0": k_a / n, "share_ci_below0": k_b / n, "share_ci_below_m01": k_m / n,
                        "cp_lo_below_m01": float(cp_lo), "cp_hi_below_m01": float(cp_hi)})
        rows.append(row)
    return pd.DataFrame(rows)


def r5_sort_banks(df):
    """sorts the bank column (string) by size."""
    out = df.copy()
    out["_o"] = [bank_sort_key(b) for b in out["bank"]]
    return out.sort_values([c for c in ("fs", "outcome", "target", "_o") if c in out.columns], kind="stable").drop(columns=["_o"]).reset_index(drop=True)


def r5_whole_frame(rows, n_vars):
    """turns the full-bank unit rows (variant == main) into a table: AUROC change, interval and p per cell. One row per cell."""
    m = rows[(rows["variant"] == "main") & (rows["bank"] == "full")]
    t = m.groupby(["outcome", "target"], sort=False).agg(
        auroc_pram=("auroc_pram", "first"), auroc_frozen=("auroc_frozen", "first"), d_auroc=("d_auroc", "first"),
        var_standard=("var_standard", "first"), p=("p_standard", "first"), n_rows=("rep", "size")).reset_index()
    t["n_vars"] = n_vars
    t["se"] = np.sqrt(t["var_standard"])
    t["ci_lo"], t["ci_hi"] = t["d_auroc"] - R5_Z * t["se"], t["d_auroc"] + R5_Z * t["se"]
    t["ni_at_margin"] = t["ci_lo"] > -NI_MARGIN_AUROC
    t["_o"] = [OUTCOMES.index(o) * 10 + TARGETS.index(g) for o, g in zip(t["outcome"], t["target"])]
    return t.sort_values("_o").drop(columns=["_o"]).reset_index(drop=True)


def r5_cells_for(n_test, n_bank, n_inner_cv):
    """number of (queries x reference rows) of the cosine neighbor search: 1 test query + n_inner_cv inner cross-validation passes (same formula as the pre-check cell)."""
    return n_test * n_bank + n_inner_cv * ((N_INNER - 1.0) / N_INNER) * n_bank * n_bank


def r5_knn_seconds_per_cell(Cx):
    """cosine neighbor search speed of this machine (seconds / (queries x reference rows)). A short measurement for the time estimate."""
    X = Cx["Xg"]
    n_ref, n_q = min(4000, max(len(X) // 2, 10)), min(2000, max(len(X) // 4, 10))
    t0 = time.time()
    nn = NearestNeighbors(n_neighbors=min(K_NEIGHBORS, n_ref), metric="cosine", algorithm="brute", n_jobs=-1).fit(X[:n_ref])
    nn.kneighbors(X[n_ref:n_ref + n_q])
    return (time.time() - t0) / float(n_ref * n_q)


def r5_whole_cells(n_rows):
    """neighbor search cost of one full-bank unit (full path of rescore_unit: two retrievals + the key 08 version)."""
    n_test = n_rows / N_FOLDS
    pool = n_rows - n_test
    return N_FOLDS * (r5_cells_for(n_test, pool, 2) + r5_cells_for(n_test, pool, 1))


def r5_tail_cells(n_rows, bank):
    """neighbor search cost of one tail unit: folds x draws per fold times (test query + two inner cross-validation)."""
    n_test = n_rows / N_FOLDS
    return N_FOLDS * max(1, N_DRAWS_TAIL // N_FOLDS) * r5_cells_for(n_test, bank, 2)


## Input collection functions (read the values of notebook 01 per unit, or recompute with the same function if absent)

This part handles six items (rescoring, 33-column eICU tail, grid, hospital level, network factor 1, TabPFN sample). If they are in this folder they are **read** (`[SKIP read]`); if they are absent or insufficient, they are **recomputed in this notebook with the same function, the same key and the same settings constants** and written to a new resume name starting with `R5_p_`. This is decided per unit: if only some units exist, existing units are read and only missing units are computed. If the number of draws or the number of methods of a unit read differs from the constants of this notebook, it is not read and is recomputed (whether the values are equal is not examined here; the aggregate comparison of input collection 4 examines the values). The same check is made when reading from the resume of this notebook, and for a unit interrupted just before the completion marker, with the same row written twice, only the last row is kept. Where each value was read from and what was computed is printed as counts in the input collection table below.


In [ ]:
# ================================================================
# PRAM input collection functions (read or recompute)
# ================================================================
R5P_LOG = []                                            # input collection record (one line per item)
R5P_SRC = {}                                            # item -> {unit: source}
R5P_EMPTY = pd.DataFrame()
R5P_NET_SEC = [1.0]                                     # seconds for one network hospital (factor 1.0). The pre-check cell measures one and replaces this
R5P_T_CALL_M = [float("nan")]                           # fixed time (seconds, excluding the neighbor search share) of one call that fits one bank with the groups frozen, pram, recal and local. Measured by the pre-check cell


def r5p_norm(x):
    """group key values as strings for comparison: a float read as an integer becomes an integer string."""
    try:
        f = float(x)
        if np.isfinite(f) and f == int(f):
            return str(int(f))
    except (TypeError, ValueError):
        pass
    return str(x)


def r5p_split(df, cols, rowkey=None):
    """a dictionary that splits a row table by the values (strings) of the unit column. If rowkey (the remaining columns that separate rows) is given, only the last of rows with the same (unit column + rowkey) is kept when there are two or more:
    for the case where rows of the same unit are present twice when re-reading a run interrupted after writing rows and before marking completion."""
    out = {}
    if df is None or not len(df) or not all(c in df.columns for c in cols):
        return out
    if rowkey and all(c in df.columns for c in rowkey):
        df = df.drop_duplicates(subset=list(cols) + [c for c in rowkey if c not in cols], keep="last").reset_index(drop=True)
    idx = {}
    for i, key in enumerate(zip(*[df[c].tolist() for c in cols])):
        idx.setdefault(tuple(r5p_norm(x) for x in key), []).append(i)
    return {k: df.iloc[ii] for k, ii in idx.items()}


def r5p_ukey(art, u):
    return tuple(r5p_norm(x) for x in art["uvals"](u))


def r5p_classify(art):
    """source per unit: read from stored results | resume of this notebook | computed (this run computes anew). Nothing is read or written (read only)."""
    R3, R5 = Resumable(art["name3"]), Resumable(art["name5"])
    g3, g5 = r5p_split(R3.df(), art["ucols"], art.get("rowkey")), r5p_split(R5.df(), art["ucols"], art.get("rowkey"))
    out = {}
    for u in art["units"]:
        k = r5p_ukey(art, u)
        if R3.unit_done(u) and art["valid"](g3.get(k, R5P_EMPTY), u):
            out[u] = "read from stored results"
        elif R5.unit_done(u) and art["valid"](g5.get(k, R5P_EMPTY), u):
            out[u] = "resume of this notebook"
        else:
            out[u] = "computed"
    return out


def r5p_provide(key, art):
    """secures all units of art. Returns: a row table (in unit order). The source per unit is in R5P_SRC[key], the counts in R5P_LOG."""
    R3, R5 = Resumable(art["name3"]), Resumable(art["name5"])
    g3, g5 = r5p_split(R3.df(), art["ucols"], art.get("rowkey")), r5p_split(R5.df(), art["ucols"], art.get("rowkey"))
    units, post = art["units"], art.get("post")
    got, src, todo = {}, {}, []
    for u in units:
        k = r5p_ukey(art, u)
        if R3.unit_done(u) and art["valid"](g3.get(k, R5P_EMPTY), u):
            got[u] = g3.get(k, R5P_EMPTY) if post is None else post(g3.get(k, R5P_EMPTY))
            src[u] = "read from stored results"
        elif R5.unit_done(u) and art["valid"](g5.get(k, R5P_EMPTY), u):          # the same check is made on the path that reads from the resume of this notebook
            got[u] = g5.get(k, R5P_EMPTY)
            src[u] = "resume of this notebook"
        else:
            todo.append(u)
    n3 = sum(1 for v in src.values() if v == "read from stored results")
    n5 = len(src) - n3
    if len(units) <= 80:
        for u in units:
            if src.get(u) == "read from stored results":
                print("  [SKIP read]", art["label"], u)
            elif src.get(u) == "resume of this notebook":
                print("  [SKIP]", art["label"], u)
    else:
        print(f"  [SKIP read] {art['label']}: units read from stored results {n3} | [SKIP] units read from the resume of this notebook {n5} (total {len(units)} )")
    cost = art.get("cost")
    w_all = {u: (cost(u) if cost else 1.0) for u in todo}
    w_left, w_done, t_loop = sum(w_all.values()), 0.0, time.time()
    for i, u in enumerate(todo):
        t0 = time.time()
        rows = art["compute"](u) or []
        for r in rows:
            R5.add_row(r)
        R5.mark_unit(u)
        got[u] = pd.DataFrame(rows)
        src[u] = "computed"
        w_done += w_all[u]
        eta = (time.time() - t_loop) * (w_left - w_done) / max(w_done, 1e-9)
        if len(todo) <= 80 or i % 10 == 0 or i == len(todo) - 1:
            print(f"  [DONE] input collection {art['label']} {u} ({time.time() - t0:.0f}s) {i + 1}/{len(todo)} remaining time about {eta / 60.0:.0f} min (estimated by cost ratio)", flush=True)
    parts = [got[u] for u in units if len(got[u])]
    df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    R5P_SRC[key] = src
    R5P_LOG.append({"item": art["label"], "units needed": len(units), "read from stored results (count)": n3, "read from the resume of this notebook (count)": n5, "computed anew this time (count)": len(todo), "resume written": art["name5"]})
    return df


def r5p_src_text(key, label=None):
    """source text of an input collection item. Without label, a sentence (number of units read from stored results and number recomputed); with label, a short tag (for the unit column of the table)."""
    s = R5P_SRC.get(key, {})
    if not s:
        return "not secured" if label is None else "not available"
    n3 = sum(1 for v in s.values() if v == "read from stored results")
    n = len(s)
    if label is not None:
        if n3 == n:
            return "read from %s" % label
        return ("recomputed in this run, same function as %s" % label) if n3 == 0 else ("read from %s where present, otherwise recomputed in this run" % label)
    if n3 == n:
        return "read from stored results (%d units)" % n
    return ("recomputed by this notebook with the same function (%d units)" % n) if n3 == 0 else ("read from the stored resume %d + recomputed with the same function %d" % (n3, n - n3))


# ---- whether the number of draws and number of methods of the units read equal the constants of this notebook (values are not looked at. The aggregate comparison of input collection 4 looks at the values) ----
def r5p_valid_matrix(g, u):
    if not len(g):
        return False
    exp = 1 if u[3] == "full" else N_DRAWS_MAIN
    cnt = g[g["method"].isin(R5P_KEEP)].groupby("method")["rep"].nunique()
    return all(int(cnt.get(m, 0)) == exp for m in R5P_KEEP)


def r5p_valid_rescore(g, u):
    if not len(g):
        return False
    exp = 1 if u[3] == "full" else N_DRAWS_MAIN
    ok = int(g[g["variant"] == "main"]["rep"].nunique()) == exp
    return bool(ok and (u[3] != "full" or (g["variant"] == "key08").any()))


def r5p_valid_a2tail(g, u):
    if not len(g):
        return False
    per_fold = max(1, N_DRAWS_TAIL // N_FOLDS)
    ps = g[g["method"] == "pram_b1"].drop_duplicates(["fold", "draw"])
    bs = g[g["method"] == "b1_base"].drop_duplicates(["fold", "draw"])
    return bool(len(ps) >= 1 and len(ps) == len(bs) and int(ps["draw"].max()) == per_fold - 1 and set(ps["fold"]) <= set(range(N_FOLDS)))


def r5p_valid_hospital(g, u):
    return (not len(g)) or set(R5P_KEEP) <= set(g["method"])


def r5p_valid_network(g, u):
    if not len(g):
        return True
    return int((g["factor"] == R5P_NET_FACTOR).sum()) == len(NET_ALPHA_GRID)


def r5p_post_methods(df):
    return df[df["method"].isin(R5P_KEEP)].reset_index(drop=True) if len(df) else df


def r5p_post_net(df):
    return df[df["factor"] == R5P_NET_FACTOR].reset_index(drop=True) if len(df) else df


def r5p_pooled_cells(n_rows, bank_size):
    """neighbor search cost of rescore_unit (queries x reference rows). Same formula as pooled_unit_cells of the pre-check cell."""
    n_test = n_rows / N_FOLDS
    pool = n_rows - n_test
    n_bank = pool if bank_size == "full" else min(bank_size, pool)
    per_call = r5_cells_for(n_test, n_bank, 2) + (r5_cells_for(n_test, n_bank, 1) if bank_size == "full" else 0.0)
    return (1 if bank_size == "full" else N_DRAWS_MAIN) * N_FOLDS * per_call


def r5p_matrix_sec(n_rows, bank_size):
    """expected seconds of run_matrix_unit (retrieval + recalibration + local retraining): fixed call time + neighbor search."""
    n_test = n_rows / N_FOLDS
    pool = n_rows - n_test
    n_bank = pool if bank_size == "full" else min(bank_size, pool)
    n_call = (1 if bank_size == "full" else N_DRAWS_MAIN) * N_FOLDS
    return n_call * ((R5P_T_CALL_M[0] if np.isfinite(R5P_T_CALL_M[0]) else R5_T_CALL) + r5_cells_for(n_test, n_bank, 2) * R5_C_CELL * 1.3)


def r5p_hosp_sec(oc, h):
    nh = float((R5B_HID[oc] == h).sum())
    return 3 * ((R5P_T_CALL_M[0] if np.isfinite(R5P_T_CALL_M[0]) else R5_T_CALL) + r5_cells_for(nh / 3.0, 2.0 * nh / 3.0, 2) * R5_C_CELL * 1.3)


def r5p_artifacts():
    """definition of input collection items. Called after the data are read (CTX, hospital table). Item name -> definition dictionary (no item is made for an absent input)."""
    arts = {}
    if R5P_RESCORE_FS in FS_RUN:
        arts["rescore"] = dict(
            label="rescoring pooled DeLong (rescore_F42 of notebook 01)", name3="rescore_F42", name5=R5P_NAMES["rescore"], ucols=["outcome", "target", "bank"],
            uvals=lambda u: (u[1], u[2], u[3]), valid=r5p_valid_rescore, post=None, rowkey=["rep", "variant"],
            units=[(R5P_RESCORE_FS, oc, tg, bk) for oc in OUTCOMES for tg in FS_SPEC[R5P_RESCORE_FS]["targets"] for bk in FS_SPEC[R5P_RESCORE_FS]["banks"]],
            compute=lambda u: rescore_unit(CTX[(u[0], u[1], u[2])], u[1], u[2], u[3], N_DRAWS_MAIN),
            cost=lambda u: r5p_pooled_cells(len(CTX[(u[0], u[1], u[2])]["y"]), u[3]) * R5_C_CELL * 1.3 + (1 if u[3] == "full" else N_DRAWS_MAIN) * N_FOLDS * 2 * R5_T_CALL)
    if R5P_A2_FS in FS_RUN:
        arts["a2tail"] = dict(
            label="33-column eICU tail (a2_tail_F33)", name3="a2_tail_F33", name5=R5P_NAMES["a2tail"], ucols=["fs", "outcome", "target", "bank"],
            uvals=lambda u: (u[0], u[1], u[2], u[3]), valid=r5p_valid_a2tail, post=None, rowkey=["fold", "draw", "method"],
            units=[(R5P_A2_FS, oc, tg, bk) for oc in OUTCOMES for tg in A2_TARGETS_F33 for bk in TAIL_BANKS],
            compute=lambda u: run_tail_unit(CTX[(u[0], u[1], u[2])], u[0], u[1], u[2], u[3], R5_TAIL_GROUPS, R5_TAIL_KEEP, R5_TAIL_PAIRS),
            cost=lambda u: N_FOLDS * max(1, N_DRAWS_TAIL // N_FOLDS) * (R5_T_CALL + r5_cells_for(len(CTX[(u[0], u[1], u[2])]["y"]) / N_FOLDS, u[3], 2) * R5_C_CELL * 1.3))
    for fs in R5B_FS:
        arts["matrix_" + fs] = dict(
            label=f"strategy grid (matrix_{fs})", name3=f"matrix_{fs}", name5=R5P_NAMES["matrix_" + fs], ucols=["fs", "outcome", "target", "bank"],
            uvals=lambda u: (u[0], u[1], u[2], u[3]), valid=r5p_valid_matrix, post=r5p_post_methods, rowkey=["rep", "method"],
            units=[(fs, oc, tg, bk) for oc in OUTCOMES for tg in FS_SPEC[fs]["targets"] for bk in (FS_SPEC[fs]["banks"] if R5P_ALL_BANKS else ["full"])],
            compute=lambda u: run_matrix_unit(CTX[(u[0], u[1], u[2])], u[0], u[1], u[2], u[3], N_DRAWS_MAIN, R5P_GROUPS, keep=R5P_KEEP)[0],
            cost=lambda u: r5p_matrix_sec(len(CTX[(u[0], u[1], u[2])]["y"]), u[3]))
    if R5P_NEED_HOSP and HAVE_HOSPITALID and EICU_TAB is not None and R5B_SITES:
        for fs in [f for f in R5B_FS if FS_SPEC[f]["hospital"]]:
            arts["hospital_" + fs] = dict(
                label=f"hospital level (hospital_{fs})", name3=f"hospital_{fs}", name5=R5P_NAMES["hospital_" + fs], ucols=["outcome", "hospitalid"],
                uvals=lambda u: (u[1], u[2]), valid=r5p_valid_hospital, post=r5p_post_methods, rowkey=["method"],
                units=[(fs, oc, int(h)) for oc in OUTCOMES for h in R5B_SITES[oc]],
                compute=lambda u: [r for r in (hospital_unit(CTX[(u[0], u[1], "eICU")], u[1], u[2], R5B_HID[u[1]], R5P_GROUPS)[0] or []) if r["method"] in R5P_KEEP],
                cost=lambda u: r5p_hosp_sec(u[1], u[2]))
    if RUN_X_NET and HAVE_HOSPITALID and EICU_TAB is not None and R5B_SITES:
        for fs in X_NET_FS:
            arts["network_" + fs] = dict(
                label=f"network factor 1 (network_{fs})", name3=f"network_{fs}", name5=R5P_NAMES["network_" + fs], ucols=["outcome", "hospitalid"],
                uvals=lambda u: (u[1], u[2]), valid=r5p_valid_network, post=r5p_post_net, rowkey=["factor", "alpha"],
                units=[(fs, oc, int(h)) for oc in OUTCOMES for h in R5B_SITES[oc]],
                compute=lambda u: network_unit(CTX[(u[0], u[1], "eICU")], u[1], u[2], R5B_HID[u[1]], R5B_SITES[u[1]], [R5P_NET_FACTOR], NET_ALPHA_GRID),
                cost=lambda u: R5P_NET_SEC[0])
    return arts


# ---- comparison (recomputed values versus the stored csv constants). The screen also prints the correct answer; csv and md hold only the value of this run, the difference and agreement ----
R5P_CHECK_ROWS = []
R5P_REF_SHOW = []                                       # (item, cell, column, this run, stored constant) of the mismatched cells. Used only for screen output


def r5p_close(new, ref):
    return bool(new is not None and ref is not None and np.isfinite(new) and np.isfinite(ref) and np.isclose(new, ref, rtol=R5_CSV_RTOL, atol=1e-12))


def r5p_add(item, key, col, new, ref):
    """adds one comparison line for a cell. If ref is absent (the cell is not in the constants), it is not counted."""
    if ref is None:
        return
    new = None if new is None else float(new)
    ok = r5p_close(new, float(ref))
    R5P_CHECK_ROWS.append({"item": item, "cell": key, "column": col, "this run": float("nan") if new is None else new,
                           "difference": abs(new - float(ref)) if (new is not None and np.isfinite(new)) else float("nan"), "agreement": ok})
    if not ok:
        R5P_REF_SHOW.append((item, key, col, new, float(ref)))


def r5p_dist(v):
    v = np.asarray(v, dtype=float)
    v = v[np.isfinite(v)]
    return [int(len(v)), float(v.mean()), float(np.quantile(v, 0.10)), float(np.median(v)), float(np.quantile(v, 0.90)), float(v.min()), float(v.max())] if len(v) else [0] + [float("nan")] * 6


# ---- expected counts of the comparison: the number compared is matched to the expected number, and if overlapping cells are insufficient it is not counted as complete ----
R5P_EXPECT = {}                                         # item -> {"expected": count, "cell columns": list of expected (cell, column), "outside plan": count, "reason": sentence}


def r5p_expect(item, key, cols, planned, why):
    """puts one cell (key) of the constants and the column names to compare (cols) in the expected list. If planned is False (a unit not in the plan of this run), it is counted only as outside the plan."""
    e = R5P_EXPECT.setdefault(item, {"expected": 0, "cell columns": [], "outside plan": 0, "reason": ""})
    if planned:
        e["expected"] += len(cols)
        e["cell columns"] += [(key, c) for c in cols]
    else:
        e["outside plan"] += len(cols)
        e["reason"] = why


def r5p_planned(key):
    """set of normalized keys of the units in the input collection plan (item key made by the pre-check cell). None if the item is not in the plan."""
    a = R5P_ARTS.get(key)
    return None if a is None else {tuple(r5p_norm(x) for x in u) for u in a["units"]}


def r5p_nn(vals):
    """indices of the non-empty (not None) positions in the list of constant values."""
    return [i for i, v in enumerate(vals) if v is not None]


## Stage B computation functions

The definitions are given at the level of formulas. Notation: X_s and y_s = development training (MIMIC-IV 56%) features and outcome, X_b = bank features (labels are not used), X_t = test features.

**B-1 importance weighting (two versions).** The shift discriminator g is a logistic regression (C = 1) that separates X_s (0, n_dev rows) from X_b (1, n_bank rows), and the two versions use one and the same fit. q(x) = g(1 | x). q/(1-q) is the density ratio of Bickel 2009 equations 12 to 17 multiplied by the prior ratio n_bank/n_dev. So the estimate of the density ratio p_bank/p_dev is q/(1-q) x n_dev/n_bank. **Version without rescaling**: w(x) = clip(q / (1 - q), 0.05, 20), without the sample size ratio. **Version with rescaling (main result)**: w_c(x) = clip(q / (1 - q) x n_dev / n_bank, 0.05, 20), with the same truncation. (a) **iw_erm** and **iwc_erm** = a logistic regression (C = 1) refitted on the development data with w or w_c as sample weights. (b) **iw_platt**, **iw_temp**, **iwc_platt**, **iwc_temp** = Platt scaling (with intercept) and temperature scaling (no intercept) fitted (C = 1e6) on the logit of the development prediction p_dev of the frozen logistic model with w as weights and applied to the logit of the test prediction. Control: the same recalibrations without weights, **dev_platt** and **dev_temp** (the same values for both versions). ESS = (sum w)^2 / sum w^2. The cell information stores per fold, for both versions, the mean weight, standard deviation, ESS, the fraction stuck at the lower bound (0.05) and the fraction stuck at the upper bound (20), and for the version with rescaling also n_dev/n_bank.

**B-2 CORAL (Sun et al. 2016, Algorithm 1, lambda = 1), two versions.** C_s = cov(X_s) + I, C_b = cov(X_b) + I (the sample covariance divided by n - 1). X_s* = X_s C_s^(-1/2) C_b^(1/2). The paper assumes that the features of both domains are normalized to mean 0 and matches only the covariance. So there are two standard choices, and both are given. (1) **`coral` = the version without matching means**: the development, bank and test features are used on the scale of the development preprocessor, a logistic regression (C = 1) is refitted on X_s* and y_s and applied to the test features **without a transform**. The development features have mean 0, but the bank and test features lie on the development scale and have a mean other than 0, so the premise of the paper is not satisfied. For a target that only shifts the mean, the transform becomes the identity and the result equals a refit of the frozen logistic regression. (2) **`coral_z` = the version re-standardized with the bank mean and standard deviation**: the development features are standardized with the development column means and standard deviations, and the bank and test features with the column means and standard deviations of the bank. For a column whose standard deviation is 0 in the bank (a category absent in a small bank), the bank mean is subtracted and the development standard deviation is used (if it were set to 1, the test rows of that category would enter much smaller than the development scale and the coefficient would weaken); only a column that is 0 also in the development gets 1. The number of such columns is in the cell information `coral_z_zero_sd_cols`. Columns whose standard deviation is 0 in the bank are **removed from the alignment in both versions** (the row and column of A are set to the identity and learned without a transform; if they entered the alignment, the target variance of the column would be only lambda and the learned coefficient would become about sqrt((s + lambda) / lambda) times that of the development fit). A logistic regression is fitted on the development features standardized and transformed with the same alignment (A = C_s^(-1/2) C_b^(1/2), covariance of the standardized features + I) and applied to the test features standardized with the bank statistics. Both domains then have mean 0, which satisfies the premise of the paper. The matrix functions are computed by eigendecomposition and checked against a direct computation with `scipy.linalg.sqrtm` and the inverse.

**B-3 anchor regression.** Environment indicator matrix A (environment one-hot; the intercept is in the column space), P_A = A (A'A)^(-1) A' (the projection that replaces values by environment means). n_e stays in one environment. (i) **Linear probability version** (Rothenhausler et al. 2021, Londschien et al., equations (1) and (2)): min over beta of ||y - X beta||^2 + (gamma - 1) ||P_A (y - X beta)||^2, gamma >= 1, the intercept included in X. With W = I - (1 - sqrt(gamma)) P_A, ||W (y - X beta)||^2 has the same value (P_A is symmetric and idempotent), so it is solved by least squares of W X and W y (`numpy.linalg.lstsq`, the minimum-norm solution if perfectly collinear). The closed solution X'(I + (gamma - 1) P_A) X beta = X'(I + (gamma - 1) P_A) y is derived from equation (2) and checked by solving it separately. The probability is obtained by fitting Platt once (the same default regularization as `platt_sklearn`) to the linear predictor over all development environments. (ii) **Probit version** (equation (4), for a binary outcome): y is set to -1 and +1, loss = -sum log Phi(y_i f_i) + (1/2)(gamma - 1) ||P_A r||^2, f = X beta, r = y phi(f) / Phi(y f) (likelihood score residual). The probability is Phi(f). The gradient X'[-r + (gamma - 1) r' * P_A r], r' = dr/df = -m (y f + m), m = phi(f)/Phi(y f), was derived here (independently of equations (6) and (7) of the reference) and checked by numerical differentiation. The initial value is the intercept f0 = Phi^(-1)(mean outcome). It is solved by L-BFGS, and if the maximum absolute gradient exceeds 1e-6 it stops without a value (`R5BStop`).

**gamma selection (without target labels, two criteria).** Training is done on folds that group the development environments (environment-level GroupKFold, number of folds = min(5, number of environments), leave-one-environment-out when there are two environments) and measured on the held-out environment: (criterion 1) AUROC, the gamma with the largest fold mean (`selected`); (criterion 2) the negative log likelihood, the gamma with the smallest fold mean (`selected_nll`, the criterion that the cited reference uses to choose gamma in classification; the fold means are in the `cv_nll` column). For the probit version the negative log likelihood of the held-out environment is measured with the probability Phi(score), and for the linear probability version with the probability from Platt scaling fitted to the linear predictor of the training folds. In each criterion, values within 1e-6 are a tie, and on a tie the smaller gamma is taken. If the two gammas are the same it is one value; if they differ both are given. The post hoc optimum that chooses gamma by the AUROC of the target test cell is not defined (the favorable gamma differs per metric). The target-cell values of the whole grid are in `R5_B3_ar_eval.csv`.

**Anchor is independent of the bank.** The models of AR-2db and AR-period are determined by the development environments alone and are independent of the bank draws. So in all draws of the same cell the metrics of X have the same value (the bank size is also the same), and the variation between draws comes only from the comparison targets (PRAM, local retraining). AR-hosp has the environments of the MIMIC-IV training split and each eligible hospital other than the target hospital, and is given only at the hospital level.


In [ ]:
# ================================================================
# PRAM stage B  computation functions (only numpy, scipy and sklearn are used)
# ================================================================
import hashlib
from sklearn.model_selection import GroupKFold


class R5BStop(BaseException):
    """stage B numerical computation did not converge or the computation cannot be defined. It stops without a value instead of continuing with NaN. It is a BaseException, so `except Exception` does not swallow it."""


# ---- B-1 importance weighting ----
R5B_DISC_MEMO = {}                                   # memory of the shift discriminator fit: the two versions (version without rescaling and version with rescaling) use one and the same fit (the values are the same and it is computed only once). Cleared per unit
R5B_DISC_STATS = {"fit": 0, "hit": 0}                # number of fits | number of retrievals from memory (counted by an internal check)
R5B_DISC_MEMO_MAX = 400                              # a cap so that memory does not grow without limit on call paths that do not clear it


def r5b_disc_memo_clear():
    R5B_DISC_MEMO.clear()


def r5b_disc_key(C, bank):
    """key that separates discriminator fits: the object id of the development and cohort feature arrays (the same object) and the bank row ids. The development array is held by the memory value (r5b_disc_q) so its object id is not reused for another array."""
    return (id(C["Xs_src"]), id(C["Xg"]), hashlib.blake2b(np.ascontiguousarray(bank, dtype=np.int64).tobytes(), digest_size=16).hexdigest())


def r5b_disc_q(X_src, X_tgt, memo_key=None):
    """logistic that separates the development sample (0) from the target unlabeled sample (1). Returns: (probability q (truncated) that a development sample is on the target side, discrimination AUC). Target labels are not used.
    if memo_key is given, the fit with the same key is not redone and is taken from memory (a deterministic fit, so the values are the same)."""
    if memo_key is not None and memo_key in R5B_DISC_MEMO:
        R5B_DISC_STATS["hit"] += 1
        return R5B_DISC_MEMO[memo_key][:2]
    Xd = np.vstack([X_src, X_tgt])
    yd = np.r_[np.zeros(len(X_src)), np.ones(len(X_tgt))]
    disc = LogisticRegression(max_iter=1000, C=R5B_DISC_C, random_state=SEED).fit(Xd, yd)
    q = np.clip(disc.predict_proba(X_src)[:, 1], R5B_Q_EPS, 1 - R5B_Q_EPS)
    try:
        auc = float(roc_auc_score(yd, disc.predict_proba(Xd)[:, 1]))
    except ValueError:
        auc = float("nan")
    R5B_DISC_STATS["fit"] += 1
    if memo_key is not None:
        if len(R5B_DISC_MEMO) >= R5B_DISC_MEMO_MAX:
            R5B_DISC_MEMO.clear()
        R5B_DISC_MEMO[memo_key] = (q, auc, X_src)
    return q, auc


def r5b_density_ratio(X_src, X_tgt, memo_key=None):
    """version without rescaling: w = clip(q/(1-q), truncation). The sample size ratio n_dev/n_bank is not multiplied."""
    q, auc = r5b_disc_q(X_src, X_tgt, memo_key)
    return np.clip(q / (1 - q), R5B_IW_CLIP[0], R5B_IW_CLIP[1]), auc


def r5b_density_ratio_rescaled(X_src, X_tgt, memo_key=None):
    """version with rescaling: w_c = clip(q/(1-q) x n_dev/n_bank, same truncation). An estimate of the density ratio p_bank/p_dev, dividing out the prior ratio n_bank/n_dev of the discriminator training sample (Bickel 2009 equations 12 to 17)."""
    q, auc = r5b_disc_q(X_src, X_tgt, memo_key)
    return np.clip(q / (1 - q) * (len(X_src) / len(X_tgt)), R5B_IW_CLIP[0], R5B_IW_CLIP[1]), auc


def r5b_platt_weighted(p_fit, y_fit, p_apply, w=None, temperature_only=False):
    """logistic on the logit. If temperature_only, the intercept is fixed."""
    p_apply = np.asarray(p_apply, dtype=float)
    if len(np.unique(y_fit)) < 2:
        return p_apply.copy()
    z, za = _logit(p_fit).reshape(-1, 1), _logit(p_apply).reshape(-1, 1)
    m = LogisticRegression(max_iter=1000, C=R5B_PLATT_C, fit_intercept=not temperature_only, random_state=SEED)
    m.fit(z, y_fit, sample_weight=w)
    return m.predict_proba(za)[:, 1]


def r5b_iw_predictions(C, bank, test, rescaled=False):
    """reweights the development data with the unlabeled X of the bank (row ids) and makes five test predictions. rescaled=False is the version without rescaling (iw_*), True is the version with rescaling (iwc_*).
    (prediction dictionary, cell information dictionary)."""
    p = "iwc" if rescaled else "iw"
    Xs, ys = C["Xs_src"], C["ys_src"]
    Xt, p1t = C["Xg"][test], np.asarray(C["pb1"])[test]
    w, auc = (r5b_density_ratio_rescaled if rescaled else r5b_density_ratio)(Xs, C["Xg"][bank], r5b_disc_key(C, bank))
    m = LogisticRegression(max_iter=1000, C=R5B_LR_C, random_state=SEED).fit(Xs, ys, sample_weight=w)
    p_dev = _sigmoid(Xs @ C["w_src"] + C["b_src"])
    P = {p + "_erm": m.predict_proba(Xt)[:, 1],
         p + "_platt": r5b_platt_weighted(p_dev, ys, p1t, w=w),
         p + "_temp": r5b_platt_weighted(p_dev, ys, p1t, w=w, temperature_only=True),
         "dev_platt": r5b_platt_weighted(p_dev, ys, p1t),
         "dev_temp": r5b_platt_weighted(p_dev, ys, p1t, temperature_only=True)}
    info = {p + "_w_mean": float(w.mean()), p + "_w_sd": float(w.std()), p + "_ess": float(w.sum() ** 2 / max((w ** 2).sum(), 1e-12)),
            p + "_disc_auc": auc, p + "_share_at_lo": float((w <= R5B_IW_CLIP[0]).mean()), p + "_share_at_hi": float((w >= R5B_IW_CLIP[1]).mean())}
    if rescaled:
        info["iwc_n_ratio"] = float(len(Xs) / len(bank))
        # diagnostic variant: retraining with the correction weights divided to mean 1 (puts the relative size that regularization C = 1 has for the weighted loss on the same scale as the version without rescaling and the version with rescaling). Not a target of pattern P
        mn = LogisticRegression(max_iter=1000, C=R5B_LR_C, random_state=SEED).fit(Xs, ys, sample_weight=w / max(float(w.mean()), 1e-12))
        P["iwn_erm"] = mn.predict_proba(Xt)[:, 1]
    return P, info


# ---- B-2 CORAL (Sun et al. 2016 Algorithm 1) ----
def r5b_coral_matrix(Xs, Xt, lam):
    """A = C_s^(-1/2) C_t^(1/2), C = cov + lam I. Computed by eigendecomposition. If the test feature is one row, the target covariance is lam I."""
    d = Xs.shape[1]
    Cs = np.cov(Xs, rowvar=False) + lam * np.eye(d)
    Ct = (np.cov(Xt, rowvar=False) if len(Xt) > 1 else np.zeros((d, d))) + lam * np.eye(d)
    ws, Us = np.linalg.eigh(Cs)
    wt, Ut = np.linalg.eigh(Ct)
    if ws.min() <= R5B_EIG_REL * ws.max() or wt.min() <= R5B_EIG_REL * wt.max():
        raise R5BStop("stop , CORAL covariance + lam I is singular (minimum / maximum eigenvalue %.1e and %.1e, lam %s). Not computed." % (ws.min() / ws.max(), wt.min() / wt.max(), lam))
    return ((Us / np.sqrt(ws)) @ Us.T) @ ((Ut * np.sqrt(wt)) @ Ut.T)


def r5b_zero_sd_cols(X):
    """mask of columns whose standard deviation (sample, n - 1) is 0 or not finite, or constant columns whose values are bit-identical in all rows (all columns if there is one row). Constant columns are looked at separately because a constant column made by standardization
    may have a standard deviation that is not exactly 0 but floating-point noise (about 1e-17) (55 of 84 simulated probes). The check that there is only one value is exact and has no threshold. If there is no constant column it is the same as the previous formula."""
    X = np.asarray(X, dtype=float)
    sd = X.std(axis=0, ddof=1) if len(X) > 1 else np.zeros(X.shape[1])
    const = (np.ptp(X, axis=0) == 0) if len(X) > 1 else np.ones(X.shape[1], dtype=bool)
    return ~(np.isfinite(sd) & (sd > 0)) | const


def r5b_zstat(X, fallback_sd=None):
    """mean and standard deviation per column (sample, n - 1). For columns whose standard deviation is 0 or not finite (all columns if there is one row), the mean is subtracted and only the standard deviation is replaced:
    the value of that column in fallback_sd if given (when a finite value greater than 0), otherwise 1. coral_z passes the development standard deviation as fallback_sd to the bank statistics
    (so test rows of a categorical column absent in a small bank enter at the same size as the development scale)."""
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=0)
    sd = X.std(axis=0, ddof=1) if len(X) > 1 else np.zeros(X.shape[1])
    sub = np.ones(X.shape[1]) if fallback_sd is None else np.where(np.isfinite(fallback_sd) & (np.asarray(fallback_sd, dtype=float) > 0), fallback_sd, 1.0)
    return mu, np.where(r5b_zero_sd_cols(X), sub, sd)


def r5b_coral_aligner(Xs, Xb, lam):
    """alignment matrix A (Xb = target features. In coral_z, Zb standardized with the bank mean and standard deviation). Columns whose standard deviation is 0 in the bank (r5b_zero_sd_cols, categories absent in a small bank) are removed from the alignment (whitening and recoloring) and the row and column of A for them are set to the identity.
    reason: the target variance of that column is only 0 + lam, so if it enters the alignment, the learned coefficient becomes about sqrt((s + lam) / lam) times that of the development fit (for an uncorrelated column
    with s = 1, about 1.4 times). If removed, that column is learned on the development scale without a transform and enters the test rows with the same coefficient as the development fit. If there is no such column it is r5b_coral_matrix as it is (bit-identical)."""
    zero = r5b_zero_sd_cols(Xb)
    if not zero.any():
        return r5b_coral_matrix(Xs, Xb, lam)
    keep = ~zero
    A = np.eye(Xs.shape[1])
    if keep.any():
        A[np.ix_(keep, keep)] = r5b_coral_matrix(Xs[:, keep], Xb[:, keep], lam)
    return A


def r5b_coral_predictions(C, bank, test):
    """two versions. coral = the version without matching means (development preprocessor scale as it is, applied to the test without a transform). coral_z = development standardized with the development mean and standard deviation, bank and test with the bank mean and standard deviation
    (both domains have mean 0, the premise of the paper). Columns whose standard deviation is 0 in the bank (categories absent in a small bank) get two treatments. Standardization (coral_z): subtract the bank mean and use the development standard deviation
    (if set to 1, the test rows of that category enter much smaller than the development scale and the coefficient becomes weak). Alignment (both versions): that column is removed from the alignment (r5b_coral_aligner).
    the number of such columns is left in the cell information coral_z_zero_sd_cols (the same columns are removed from both versions)."""
    Xs, ys = C["Xs_src"], C["ys_src"]
    Xb, Xt = C["Xg"][bank], C["Xg"][test]
    A = r5b_coral_aligner(Xs, Xb, R5B_CORAL_LAMBDA)
    m = LogisticRegression(max_iter=1000, C=R5B_LR_C, random_state=SEED).fit(Xs @ A, ys)
    mu_s, sd_s = r5b_zstat(Xs)
    mu_b, sd_b = r5b_zstat(Xb, fallback_sd=sd_s)
    Zs, Zb, Zt = (Xs - mu_s) / sd_s, (Xb - mu_b) / sd_b, (Xt - mu_b) / sd_b
    Az = r5b_coral_aligner(Zs, Zb, R5B_CORAL_LAMBDA)
    mz = LogisticRegression(max_iter=1000, C=R5B_LR_C, random_state=SEED).fit(Zs @ Az, ys)
    return ({"coral": m.predict_proba(Xt)[:, 1], "coral_z": mz.predict_proba(Zt)[:, 1]},
            {"coral_shift_fro": float(np.linalg.norm(A - np.eye(A.shape[0]))), "coral_z_shift_fro": float(np.linalg.norm(Az - np.eye(Az.shape[0]))),
             "coral_z_zero_sd_cols": int(r5b_zero_sd_cols(Xb).sum())})


# ---- B-3 anchor regression ----
def r5b_dense(codes):
    u, inv = np.unique(np.asarray(codes), return_inverse=True)
    return inv.astype(int), len(u)


def r5b_env_mean(v, codes, n_env):
    """P_A v: the value replaced by the mean per environment. v is (n,) or (n, k)."""
    cnt = np.maximum(np.bincount(codes, minlength=n_env).astype(float), 1.0)
    if v.ndim == 1:
        return (np.bincount(codes, weights=v, minlength=n_env) / cnt)[codes]
    out = np.empty(v.shape, dtype=float)
    for j in range(v.shape[1]):
        out[:, j] = (np.bincount(codes, weights=v[:, j], minlength=n_env) / cnt)[codes]
    return out


def r5b_lpm_anchor_fit(X, y, codes, gamma):
    """linear anchor regression equation (2). Least squares after the transform W = I - (1 - sqrt(gamma)) P_A. Returns: [intercept, coefficients...]. With gamma = 1 it is least squares."""
    codes, n_env = r5b_dense(codes)
    Xa = np.column_stack([np.ones(len(y)), np.asarray(X, dtype=float)])
    y = np.asarray(y, dtype=float)
    c = 1.0 - np.sqrt(float(gamma))
    WX, Wy = Xa - c * r5b_env_mean(Xa, codes, n_env), y - c * r5b_env_mean(y, codes, n_env)
    beta, _res, _rk, _sv = np.linalg.lstsq(WX, Wy, rcond=None)
    if not np.isfinite(beta).all():
        raise R5BStop("stop , the linear anchor solution has a non-finite value (gamma %s)." % gamma)
    return beta


def r5b_probit_loss_grad(beta, Xa, ypm, codes, n_env, gamma):
    """mean loss (equation (4) / n) and gradient of probit anchor classification. ypm is -1 or +1."""
    f = Xa @ beta
    yf = ypm * f
    m = np.exp(scipy_stats.norm.logpdf(f) - scipy_stats.norm.logcdf(yf))
    r = ypm * m
    rp = -m * (yf + m)
    a = r5b_env_mean(r, codes, n_env)
    n = len(f)
    loss = (-scipy_stats.norm.logcdf(yf).sum() + 0.5 * (gamma - 1.0) * float(r @ a)) / n
    grad = Xa.T @ (-r + (gamma - 1.0) * rp * a) / n
    return loss, grad


def r5b_probit_anchor_fit(X, y01, codes, gamma, beta0=None):
    """probit anchor fit. (beta, information). R5BStop if it does not converge. If beta0 is absent, starts from the intercept f0 = Phi^(-1)(mean outcome) of the original paper."""
    codes, n_env = r5b_dense(codes)
    Xa = np.column_stack([np.ones(len(y01)), np.asarray(X, dtype=float)])
    ypm = 2.0 * np.asarray(y01, dtype=float) - 1.0
    if beta0 is None:
        beta0 = np.zeros(Xa.shape[1])
        beta0[0] = scipy_stats.norm.ppf(float(np.clip(np.mean(y01), 1e-6, 1 - 1e-6)))
    res = minimize(r5b_probit_loss_grad, beta0, args=(Xa, ypm, codes, n_env, float(gamma)), jac=True, method="L-BFGS-B",
                   options={"maxiter": R5B_PROBIT_MAXITER, "ftol": R5B_PROBIT_FTOL, "gtol": R5B_PROBIT_GTOL, "maxcor": 20})
    loss, g = r5b_probit_loss_grad(res.x, Xa, ypm, codes, n_env, float(gamma))
    gmax = float(np.max(np.abs(g)))
    if not (np.isfinite(loss) and np.isfinite(res.x).all()) or gmax > R5B_PROBIT_GRAD_TOL:
        raise R5BStop("stop , probit anchor did not converge (gamma %s, iterations %d, maximum gradient %.2e > %.0e, termination reason %s). No value was produced."
                      % (gamma, res.nit, gmax, R5B_PROBIT_GRAD_TOL, str(res.message)))
    if float(np.max(np.abs(res.x))) > R5B_BETA_MAX:
        raise R5BStop("stop , the absolute value of the probit anchor coefficient was %.1f and exceeded the cap %.0f (gamma %s). It looks like complete separation; the gradient approaches 0 but there is no solution. No value was produced." % (float(np.max(np.abs(res.x))), R5B_BETA_MAX, gamma))
    return res.x, {"n_iter": int(res.nit), "grad_max": gmax, "loss": float(loss)}


def r5b_fit_form(form, X, y, codes, gamma, beta0=None):
    if form == "lpm":
        return r5b_lpm_anchor_fit(X, y, codes, gamma), {"n_iter": 0, "grad_max": 0.0, "loss": float("nan")}
    if form == "probit":
        return r5b_probit_anchor_fit(X, y, codes, gamma, beta0)
    raise R5BStop("stop , unknown anchor form %s" % form)


def r5b_score(beta, X):
    return beta[0] + np.asarray(X, dtype=float) @ beta[1:]


def r5b_platt_on_score(z_fit, y_fit, z_apply):
    """Platt on the linear predictor (logistic with the same default regularization as `platt_sklearn`). (probability, coefficient a, intercept b)"""
    m = LogisticRegression(max_iter=1000, random_state=SEED).fit(np.asarray(z_fit).reshape(-1, 1), y_fit)
    a, b = float(m.coef_[0, 0]), float(m.intercept_[0])
    return _sigmoid(a * np.asarray(z_apply) + b), a, b


def r5b_ar_proba(form, beta, platt_ab, X):
    z = r5b_score(np.asarray(beta, dtype=float), X)
    return _sigmoid(platt_ab[0] * z + platt_ab[1]) if form == "lpm" else scipy_stats.norm.cdf(z)


def r5b_select_gamma(gammas, cv_auc):
    """the gamma with the largest fold-mean AUROC. Within R5B_TIE_TOL of the maximum is a tie, and on a tie the smaller gamma."""
    best = max(cv_auc[g] for g in gammas)
    return min(g for g in gammas if cv_auc[g] >= best - R5B_TIE_TOL)


def r5b_select_gamma_nll(gammas, cv_nll):
    """the gamma with the smallest fold-mean negative log likelihood (the classification selection criterion of the cited reference). Within R5B_TIE_TOL of the minimum is a tie, and on a tie the smaller gamma."""
    best = min(cv_nll[g] for g in gammas)
    return min(g for g in gammas if cv_nll[g] <= best + R5B_TIE_TOL)


def r5b_fold_nll(form, beta, X_tr, y_tr, X_te, y_te):
    """negative log likelihood (mean) of the held-out environment. For the probit version the probability Phi(score), for the linear probability version the Platt probability fitted to the training fold scores. Probabilities are clipped to [1e-15, 1 - 1e-15]."""
    z_te = r5b_score(beta, X_te)
    if form == "lpm":
        p = r5b_platt_on_score(r5b_score(beta, X_tr), y_tr, z_te)[0]
    else:
        p = scipy_stats.norm.cdf(z_te)
    p = np.clip(np.asarray(p, dtype=float), 1e-15, 1.0 - 1e-15)
    y_te = np.asarray(y_te, dtype=float)
    return float(-np.mean(y_te * np.log(p) + (1.0 - y_te) * np.log(1.0 - p)))


def r5b_anchor_fit_grid(X, y, env, form, gammas=None):
    """fits the whole gamma grid in the development environments (env: environment code per stay) and chooses gamma by environment-level folds. Target labels are not used.
    returned dictionary: gammas, betas (per gamma), platt (per gamma, linear probability version only), cv_auc, cv_nll, selected (AUROC maximum), selected_nll (negative log likelihood minimum), n_env, env_n, n_cv_folds, info (iterations and gradient per gamma)."""
    gammas = tuple(R5B_AR_GAMMAS if gammas is None else gammas)
    X, y = np.asarray(X, dtype=float), np.asarray(y).astype(int)
    codes, n_env = r5b_dense(env)
    if n_env < 2:
        raise R5BStop("stop , there are %d anchor environments. At least two environments are needed to choose gamma." % n_env)
    if len(np.unique(y)) < 2:
        raise R5BStop("stop , the development data have only one outcome value. The anchor fit cannot be defined.")
    splits = list(GroupKFold(n_splits=min(R5B_CV_FOLDS, n_env)).split(X, y, codes))
    cv = {g: [] for g in gammas}
    cvn = {g: [] for g in gammas}
    n_used = 0
    for tr, te in splits:
        if len(np.unique(y[te])) < 2 or len(np.unique(y[tr])) < 2:
            continue
        ctr = codes[tr]
        prev = None
        n_used += 1
        for g in gammas:
            beta, _inf = r5b_fit_form(form, X[tr], y[tr], ctr, g, prev)
            prev = beta if form == "probit" else None
            cv[g].append(float(roc_auc_score(y[te], r5b_score(beta, X[te]))))
            cvn[g].append(r5b_fold_nll(form, beta, X[tr], y[tr], X[te], y[te]))
    if n_used == 0:
        raise R5BStop("stop , in no environment-level fold does the held-out environment contain both outcomes. gamma cannot be chosen.")
    cv_auc = {g: float(np.mean(cv[g])) for g in gammas}
    cv_nll = {g: float(np.mean(cvn[g])) for g in gammas}
    betas, platt, info = {}, {}, {}
    prev = None
    for g in gammas:
        betas[g], info[g] = r5b_fit_form(form, X, y, codes, g, prev)
        prev = betas[g] if form == "probit" else None
        if form == "lpm":
            _p, a, b = r5b_platt_on_score(r5b_score(betas[g], X), y, r5b_score(betas[g], X[:1]))
            platt[g] = (a, b)
    return {"gammas": gammas, "betas": betas, "platt": platt, "cv_auc": cv_auc, "cv_nll": cv_nll, "selected": r5b_select_gamma(gammas, cv_auc),
            "selected_nll": r5b_select_gamma_nll(gammas, cv_nll), "n_env": n_env,
            "env_n": [int(v) for v in np.bincount(codes)], "n_cv_folds": n_used, "info": info}


def r5b_fit_rows(fit, meta):
    """one row per gamma (for resume and csv). Coefficients are kept as a JSON string."""
    rows = []
    for g in fit["gammas"]:
        a, b = fit["platt"].get(g, (float("nan"), float("nan")))
        rows.append({**meta, "gamma": g, "cv_auc": fit["cv_auc"][g], "cv_nll": fit["cv_nll"][g], "selected": bool(g == fit["selected"]), "selected_nll": bool(g == fit["selected_nll"]),
                     "n_env": fit["n_env"], "n_cv_folds": fit["n_cv_folds"],
                     "n_iter": fit["info"][g]["n_iter"], "grad_max": fit["info"][g]["grad_max"], "platt_a": a, "platt_b": b,
                     "beta_json": json.dumps([float(v) for v in fit["betas"][g]])})
    return rows


def r5b_rows_to_proba(row, X):
    """rebuilds the probability from the resume rows (r5b_fit_rows)."""
    return r5b_ar_proba(row["form"], np.array(json.loads(row["beta_json"])), (row["platt_a"], row["platt_b"]), X)


# ---- development environment construction ----
def r5b_period_envs(stay_ids_train, periods, col=None):
    """period code of the development training stays. (row mask to use, code, description of missing periods). Stays with unknown period and periods below R5B_MIN_ENV_N are removed.
    col: period rule column (strict, majority, patient_group). If absent, a table with patient_group (patient-level approximation) uses that column, otherwise the main rule column (R5B_PERIOD_MAIN_RULE)."""
    if col is None:
        col = "patient_group" if "patient_group" in periods.columns else R5B_PERIOD_MAIN_RULE
    g = periods[col].reindex(np.asarray(stay_ids_train).astype("int64")).to_numpy()
    ok = np.isfinite(g.astype(float)) & (g >= 0)
    cnt = pd.Series(g[ok]).value_counts()
    small = sorted(int(k) for k, v in cnt.items() if v < R5B_MIN_ENV_N)
    keep = ok & ~np.isin(g, small)
    return keep, g[keep].astype(int), {"period unknown": int((~ok).sum()), "small period excluded": small, "column": col}


def r5b_hosp_envs(C_dev, C_eicu, hid, sites, h):
    """environments without the target hospital h: the development training split (code 0) + each eligible hospital. (X, y, environment code)"""
    others = [g for g in sites if g != h]
    m = np.isin(hid, others)
    X = np.vstack([C_dev["Xs_src"], C_eicu["Xg"][m]])
    y = np.concatenate([C_dev["ys_src"], C_eicu["y"][m]])
    code_map = {g: i + 1 for i, g in enumerate(others)}
    env = np.concatenate([np.zeros(len(C_dev["ys_src"]), dtype=int), np.array([code_map[int(v)] for v in hid[m]], dtype=int)])
    return X, y, env


# ---- PRAM mixing weight within a bank for several base probabilities in one retrieval (same logic as inner_cv_bank) ----
def r5b_inner_cv_shared(X_bank, y_bank, bases, r_fn, key=""):
    """applies the same rule as `inner_cv_bank` to several base probabilities (bases: name -> base probability within the bank). The retrieval prediction is computed once. name -> {alpha, oof_pram, ok}."""
    n = len(y_bank)
    if n < N_INNER * 5 or len(np.unique(y_bank)) < 2:
        return {k: {"alpha": 0.0, "oof_pram": np.asarray(v).copy(), "ok": False} for k, v in bases.items()}
    skf = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=stable_seed("innercv", key))
    scores = {k: {a: [] for a in ALPHA_GRID} for k in bases}
    oof_retr = np.full(n, np.nan)
    for itr, iva in skf.split(np.zeros(n), y_bank):
        pr = r_fn({"X_tr_s": X_bank[itr], "y_tr": y_bank[itr], "X_te_s": X_bank[iva], "y_te": y_bank[iva], "mi": None, "mi_prob": None})
        oof_retr[iva] = pr
        if len(np.unique(y_bank[iva])) > 1:
            for k, pb in bases.items():
                for a in ALPHA_GRID:
                    scores[k][a].append(roc_auc_score(y_bank[iva], (1 - a) * pb[iva] + a * pr))
    out = {}
    for k, pb in bases.items():
        best_a = max(ALPHA_GRID, key=lambda a: np.nanmean(scores[k][a]) if scores[k][a] else -1)
        o = np.where(np.isnan(oof_retr), pb, oof_retr)
        out[k] = {"alpha": best_a, "oof_pram": (1 - best_a) * pb + best_a * o, "ok": True}
    return out


# ---- connection that adds the B groups to bank_predictions ----
R5B_CUR = {"key": None, "ar_p": {}}          # cell key during this call and the base probability used by pram_on_X (cell cohort all rows -> probability)


def r5b_extra(Cx, bank, test, key, groups):
    """the B groups to add to the (P, info) returned by bank_predictions of notebook 01. Group names: b_iw (version without rescaling) | b_iwc (version with rescaling) | b_coral | b_fus."""
    P, info = {}, {}
    if "b_iw" in groups:
        a, b = r5b_iw_predictions(Cx, bank, test)
        P.update(a)
        info.update(b)
    if "b_iwc" in groups:
        a, b = r5b_iw_predictions(Cx, bank, test, rescaled=True)
        P.update(a)
        info.update(b)
    if "b_coral" in groups:
        a, b = r5b_coral_predictions(Cx, bank, test)
        P.update(a)
        info.update(b)
    if "b_fus" in groups:
        Xb, yb, Xt = Cx["Xg"][bank], Cx["y"][bank], Cx["Xg"][test]
        p_retr = RETRIEVAL_FN({"X_tr_s": Xb, "y_tr": yb, "X_te_s": Xt, "y_te": Cx["y"][test], "mi": None, "mi_prob": None})
        bases = {name: np.asarray(pa)[bank] for name, pa in R5B_CUR["ar_p"].items()}
        icv = r5b_inner_cv_shared(Xb, yb, bases, RETRIEVAL_FN, key="b1|" + key)
        for name, pa in R5B_CUR["ar_p"].items():
            pt = np.asarray(pa)[test]
            P[name] = pt
            P["pram_on_" + name] = (1 - icv[name]["alpha"]) * pt + icv[name]["alpha"] * p_retr
            info["alpha_" + name] = icv[name]["alpha"]
    return P, info


def r5b_with_extra(fn, *args, **kw):
    """while fn (run_matrix_unit, hospital_unit, etc. of the stored results) is called, replaces bank_predictions with the version with the B groups added. Restores it afterwards."""
    global bank_predictions
    orig = bank_predictions

    def merged(Cx, bank, test, key, groups, cb_iter=None):
        P, info = orig(Cx, bank, test, key, groups, cb_iter)
        add, added_info = r5b_extra(Cx, bank, test, key, groups)
        P.update(add)
        info.update(added_info)
        return P, info
    bank_predictions = merged
    try:
        return fn(*args, **kw)
    finally:
        bank_predictions = orig


# ---- hospital level: method X paired with frozen and PRAM (same fold and same key as hospital_oof of notebook 01) ----
def r5b_hosp_pair_rows(C, oc, h, hid, group, methods, extra=None):
    """from one set of 3-fold test predictions of hospital h, gives per method the metrics, versus frozen, and the paired DeLong difference and variance versus PRAM.
    group: B group name (b_iw, b_coral; None if absent). extra: {method name: probability of all cohort rows} (only the hospital rows are picked. A model independent of the bank, like anchor)."""
    groups = ("pram",) + ((group,) if group else ())
    ys, oof, _fr = r5b_with_extra(hospital_oof, C, oc, h, hid, groups)
    if ys is None:
        return []
    idx = np.where(hid == h)[0]
    probs = {m: oof[m] for m in tuple(methods) + ("pram_b1",)}
    for m, pa in (extra or {}).items():
        probs[m] = np.asarray(pa)[idx]
    rows = []
    for m, v in probs.items():
        if np.isnan(v).any():
            continue
        _a, _b, d1, v1 = delong_delta_var(ys, v, oof["b1_base"])
        _a, _b, dp, vp = delong_delta_var(ys, v, oof["pram_b1"])
        rows.append({"outcome": oc, "hospitalid": int(h), "method": m, "n_site": int(len(ys)), "n_events": int(ys.sum()), "prev": float(ys.mean()), **full_metrics(ys, v),
                     "d_auroc_b1": d1, "var_b1": v1, "d_auroc_pram": dp, "var_pram": vp})
    return rows


def r5b_hosp_pram_oof(CTXd, fs, sites, hid):
    """(outcome, hospital) -> one set of PRAM (pram_b1) that collects the within-hospital 3-fold test predictions. The same fold and key as the stored results, so the same values as PRAM of the stored hospital resume (the reproduction check compares them)."""
    out = {}
    for oc in OUTCOMES:
        for h in sites[oc]:
            ys, oof, _fr = hospital_oof(CTXd[(fs, oc, "eICU")], oc, h, hid[oc], ("pram",))
            if ys is not None:
                out[(oc, int(h))] = oof["pram_b1"]
    return out


def r5b_hosp_meta(rows):
    """hospital-level meta-analysis: per (fs, outcome, method), pools the per-hospital delta AUROC (versus frozen) and the paired DeLong variance (d_auroc_b1, var_b1) with `meta_reml_hk` (the same function as A6 of notebook 01).
    columns: k, mu_random, ci_hk_trunc_lo, ci_hk_trunc_hi (Hartung-Knapp truncated), tau, I2_invvar, k_zero_var, and versus PRAM of the same hospitals mu_vs_pram, ci_vs_pram_lo, ci_vs_pram_hi (when the d_auroc_pram column exists)."""
    out = []
    for (fs, oc, m), s in rows.groupby(["fs", "outcome", "method"], sort=False):
        r = meta_reml_hk(s["d_auroc_b1"].to_numpy(float), s["var_b1"].to_numpy(float))
        rp = meta_reml_hk(s["d_auroc_pram"].to_numpy(float), s["var_pram"].to_numpy(float)) if ("d_auroc_pram" in s.columns and s["d_auroc_pram"].notna().all() and m != "pram_b1") else None
        pm = {"mu_vs_pram": rp["mu_random"] if rp else float("nan"), "ci_vs_pram_lo": rp["ci_hk_trunc_lo"] if rp else float("nan"), "ci_vs_pram_hi": rp["ci_hk_trunc_hi"] if rp else float("nan")}
        if r is None:
            out.append({"fs": fs, "outcome": oc, "method": m, "k": int(len(s)), "mu_random": float("nan"), "ci_hk_trunc_lo": float("nan"), "ci_hk_trunc_hi": float("nan"), "tau": float("nan"), "I2_invvar": float("nan"), "k_zero_var": float("nan"), **pm})
        else:
            out.append({"fs": fs, "outcome": oc, "method": m, "k": r["k"], "mu_random": r["mu_random"], "ci_hk_trunc_lo": r["ci_hk_trunc_lo"], "ci_hk_trunc_hi": r["ci_hk_trunc_hi"], "tau": r["tau"], "I2_invvar": r["I2_invvar"], "k_zero_var": r["k_zero_var"], **pm})
    return pd.DataFrame(out)


# ---- cell-level comparison and pattern ----
R5B_KEYS = ["fs", "outcome", "target", "bank", "rep"]
R5B_NB20 = nbcol(THRESH_MS)


def r5b_pair_table(xrows, comp, x_method, cmp_methods=("pram_b1", "b1_base", "platt_b1", "local_lr")):
    """per cell (fs, outcome, target, bank), the median of the paired difference between x_method and the comparison method in the same draws. xrows and comp are the cell draw rows (including the full_metrics columns).
    columns: n_draws, medians of dAUROC, dECE and dNB20 per comparison, and for vs_pram the fraction of draws with decreased ECE share_dECE_neg."""
    x = xrows[xrows["method"] == x_method].copy()
    x["bank"] = x["bank"].astype(str)
    x = x.drop_duplicates(R5B_KEYS, keep="last").set_index(R5B_KEYS)
    out = None
    for cm in cmp_methods:
        c = comp[comp["method"] == cm].copy()
        c["bank"] = c["bank"].astype(str)
        c = c.drop_duplicates(R5B_KEYS, keep="last").set_index(R5B_KEYS)
        j = x[["auroc", "ece", R5B_NB20]].join(c[["auroc", "ece", R5B_NB20]], rsuffix="_c", how="inner")
        d = pd.DataFrame({"dAUROC": j["auroc"] - j["auroc_c"], "dECE": j["ece"] - j["ece_c"], "dNB20": j[R5B_NB20] - j[R5B_NB20 + "_c"]})
        g = d.groupby(["fs", "outcome", "target", "bank"], sort=False)
        t = g.median().add_suffix("_vs_" + cm)
        t["n_draws_vs_" + cm] = g.size()
        if cm == "pram_b1":
            t["share_dECE_neg_vs_pram_b1"] = g["dECE"].apply(lambda s: float((s < 0).mean()))
        out = t if out is None else out.join(t, how="outer")
    return out.reset_index()


def r5b_flags(t, margins):
    """attaches the D+, DOM and NULL marks to the cell table t (result of r5b_pair_table). margins = (AUROC, net benefit, ECE) margins."""
    ma, mn, me = margins
    t = t.copy()
    t["D_plus"] = t["dAUROC_vs_pram_b1"] > ma
    t["DOM"] = ((t["dAUROC_vs_pram_b1"] >= -ma) & (t["dECE_vs_pram_b1"] < 0) & (t["share_dECE_neg_vs_pram_b1"] >= R5B_DOM_SHARE) & (t["dNB20_vs_pram_b1"] >= mn))
    t["NULL"] = ((t["dAUROC_vs_b1_base"].abs() <= ma) & (t["dNB20_vs_b1_base"].abs() <= mn) & (t["dECE_vs_b1_base"].abs() <= me))
    return t


def r5b_hosp_dom(xh, ph, x_method, margins):
    """hospital-level DOM. xh and ph: metric rows per hospital and method (outcome, hospitalid, method, auroc, ece, nb@0.2). Fraction of hospitals with per-hospital delta AUROC (X - PRAM) >= -margin, delta ECE < 0 and delta NB20 >= +margin.
    returns: per outcome (number of hospitals, number of DOM hospitals, fraction)."""
    ma, mn, _me = margins
    a = xh[xh["method"] == x_method].set_index(["outcome", "hospitalid"])
    b = ph[ph["method"] == "pram_b1"].set_index(["outcome", "hospitalid"])
    j = a[["auroc", "ece", R5B_NB20]].join(b[["auroc", "ece", R5B_NB20]], rsuffix="_p", how="inner")
    dom = (j["auroc"] - j["auroc_p"] >= -ma) & (j["ece"] - j["ece_p"] < 0) & (j[R5B_NB20] - j[R5B_NB20 + "_p"] >= mn)
    out = {}
    for oc in sorted(set(j.index.get_level_values(0))):
        s = dom[dom.index.get_level_values(0) == oc]
        out[oc] = (int(len(s)), int(s.sum()), float(s.mean()) if len(s) else float("nan"))
    return out


def r5b_pattern_P(cells, hosp):
    """analysis P pattern. cells: cell table of one method (result of r5b_flags, all cells). hosp: {(fs, outcome): hospital DOM fraction} or None (no hospital-level result).
    returns: dictionary (pattern, n_cells, n_eicu, n_dom_eicu, n_dplus_eicu, null_share, hosp_min_share, why)."""
    e = cells[cells["target"] == "eICU"]
    n_e = len(e)
    n_dom, n_dp = int(e["DOM"].sum()), int(e["D_plus"].sum())
    null_share = float(cells["NULL"].mean()) if len(cells) else float("nan")
    hmin = (min(hosp.values()) if hosp else float("nan"))
    need = int(np.ceil(R5B_P4_CELL_SHARE * n_e - 1e-9))
    why = ""
    if n_e and n_dom >= need and hosp and hmin >= R5B_P4_HOSP_SHARE:
        pattern = "P4 (DOM in most eICU cells and in most hospitals)"
    elif n_dp >= 1 or (1 <= n_dom <= R5B_P2_DOM_MAX):
        pattern = "P2 (eICU D+ or DOM in some cells)"
    elif n_dom > R5B_P2_DOM_MAX:
        pattern = "P0 (no listed pattern)"
        why = "eICU DOM is in %d cells (>%d) but the hospital-level condition (at least %.0f%% of hospitals in all outcomes and sets) is not met or there is no hospital-level result" % (n_dom, R5B_P2_DOM_MAX, 100 * R5B_P4_HOSP_SHARE)
    elif null_share >= R5B_NULL_SHARE:
        pattern = "P3 (no eICU DOM or D+, NULL at or above the share)"
    else:
        pattern = "P1 (no eICU DOM or D+, NULL below the share)"
    return {"pattern": pattern, "n_cells": int(len(cells)), "n_eicu": int(n_e), "n_dom_eicu": n_dom, "n_dplus_eicu": n_dp, "null_share": null_share, "hosp_min_share": hmin, "why": why}


def r5b_pattern_F(f_cells, g_std):
    """analysis F pattern (the analysis F table). f_cells: per cell G_X (dAUROC), dECE, dNB20 (paired median of pram_on_X minus X), target column.
    g_std: per cell the standard gain G_std of the same cell (median AUROC of PRAM minus frozen logistic, in the same order as f_cells)."""
    ma, mn, me = R5B_MARGIN_SETS["base"]
    g = f_cells["dAUROC"].to_numpy(float)
    ece = f_cells["dECE"].to_numpy(float)
    nb = f_cells["dNB20"].to_numpy(float)
    n = len(g)
    if n == 0:
        return {"pattern": "F0 (no listed pattern)", "n_cells": 0, "share_g_ok": float("nan"), "share_ece_down": float("nan"), "why": "no cells"}
    share_g, share_e = float((g >= -ma).mean()), float((ece < 0).mean())
    f4_share = float(((g < -ma) & (ece > 0)).mean())
    null_like = float(((g >= -ma) & (np.abs(ece) <= me) & (np.abs(nb) <= mn)).mean())
    eic = (f_cells["target"] == "eICU").to_numpy()
    half = bool(eic.any() and (g[eic] < 0.5 * np.asarray(g_std, dtype=float)[eic]).mean() > 0.5)
    why = ""
    if f4_share >= R5B_F_ECE_SHARE:
        pattern = "F4 (G_X below the margin and ECE higher in half of the cells or more)"
    elif null_like >= R5B_F_KEEP_SHARE:
        pattern = "F3 (G_X, ECE and NB20 within the margins in most cells)"
    elif share_g >= R5B_F_KEEP_SHARE and share_e >= R5B_F_ECE_SHARE:
        pattern, why = ("F2 (G_X or ECE favorable in only part of the cells)", "in many eICU cells G_X is less than half of G_std") if half else ("F1 (G_X and ECE favorable in most cells)", "")
    elif share_g >= 0.5:
        pattern = "F2 (G_X or ECE favorable in only part of the cells)"
    else:
        pattern, why = "F0 (no listed pattern)", "fewer than 50% of cells have G_X >= -margin but the F4 condition (50% or more of cells with G_X < -margin and increased ECE) is not met"
    return {"pattern": pattern, "n_cells": int(n), "share_g_ok": share_g, "share_ece_down": share_e, "share_f4": f4_share, "share_null_like": null_like, "half_of_std_eicu": half, "why": why}


# ---- B-1 comparison, check and split of the two versions (version without rescaling and version with rescaling) ----
R5B_W_KEYS = ["fs", "outcome", "target", "bank"]


def r5b_weight_compare(iw_info, iwc_info):
    """the weight summaries of the two versions side by side in the same cells (median over all folds). iw_info and iwc_info: cell information tables (including the fs column, one row per fold).
    columns: n_folds | development records / bank rows (iwc_n_ratio, the value multiplied by the version with rescaling) | mean weight of the version without rescaling (iw_) and version with rescaling (iwc_) | fraction stuck at the lower bound | fraction stuck at the upper bound | ESS | shift discrimination AUC."""
    keys = R5B_W_KEYS + ["rep", "fold"]
    a, b = iw_info.copy(), iwc_info.copy()
    a["bank"], b["bank"] = a["bank"].astype(str), b["bank"].astype(str)
    cols_a = ["n_bank", "iw_w_mean", "iw_share_at_lo", "iw_share_at_hi", "iw_ess", "iw_disc_auc"]
    cols_b = ["iwc_w_mean", "iwc_share_at_lo", "iwc_share_at_hi", "iwc_ess", "iwc_n_ratio"]
    j = a[keys + cols_a].merge(b[keys + cols_b], on=keys, how="inner")
    g = j.groupby(R5B_W_KEYS, sort=False)
    out = g[["iwc_n_ratio", "iw_w_mean", "iw_share_at_lo", "iw_share_at_hi", "iw_ess", "iwc_w_mean", "iwc_share_at_lo", "iwc_share_at_hi", "iwc_ess", "iw_disc_auc"]].median().reset_index()
    out.insert(len(R5B_W_KEYS), "n_folds", g.size().to_numpy())
    return out


def r5b_iw_impl_table(iw_f42, comp_f42, wcmp):
    """B-1 implementation check table. Two reproduction checks of the version without rescaling + two checks of the version with rescaling.
    iw_f42: 42-column draw rows of the version without rescaling (outcome, target, bank, method, slope, auroc), comp_f42: comparison target rows of the stored results (42 columns), wcmp: result of r5b_weight_compare.
    columns: target version | item | value | criterion | agreement | if outside."""
    rows = []
    if iw_f42 is not None and len(iw_f42):
        d = iw_f42.copy()
        d["bank"] = d["bank"].astype(str)
        pb = [str(b) for b in R5B_IW_CHECK["banks"]]
        sl = d[(d["method"] == "iw_platt") & (d["outcome"] == "mortality_168h") & (d["target"] == "eICU") & (d["bank"].isin(pb))].groupby("bank")["slope"].median()
        both = pd.concat([d, comp_f42], ignore_index=True)
        tb = r5b_pair_table(both, both, "iw_erm", ("b1_base",))
        tb = tb[tb["bank"].isin(pb)]
        low = int((tb["dAUROC_vs_b1_base"] < 0).sum())
        rows.append({"target version": "version without rescaling (reproduction check)", "item": "eICU death IW-Platt slope (median of bank %s)" % "·".join(pb), "value": " · ".join("%s: %.3f" % (k, v) for k, v in sl.items()),
                     "criterion": "%s to %s (reference 0.440 to 0.485)" % (R5B_IW_CHECK["slope_lo"], R5B_IW_CHECK["slope_hi"]),
                     "agreement": bool(len(sl) == len(pb) and ((sl >= R5B_IW_CHECK["slope_lo"]) & (sl <= R5B_IW_CHECK["slope_hi"])).all()), "if outside": "not used as a reproduction of the reference values but as an implementation check (unrelated to the version with rescaling)"})
        rows.append({"target version": "version without rescaling (reproduction check)", "item": "cells in which IW-ERM AUROC is lower than frozen (2 outcomes x 3 targets x 4 banks, 42 columns)", "value": "%d/%d" % (low, len(tb)),
                     "criterion": "%d/%d or more (reference 22/24)" % (R5B_IW_CHECK["min_cells"], R5B_IW_CHECK["n_cells"]),
                     "agreement": bool(len(tb) == R5B_IW_CHECK["n_cells"] and low >= R5B_IW_CHECK["min_cells"]), "if outside": "not used as a reproduction of the reference values but as an implementation check (unrelated to the version with rescaling)"})
    if wcmp is not None and len(wcmp):
        # reproduction checks 2-2 and 2-3 of the version without rescaling: comparison of the 24 reference cells (mean weight and shift discrimination AUC) with the cell medians of the same cells. Both are independent of the bank label and depend only on bank X
        w42 = wcmp[wcmp["fs"] == "F42"].copy()
        w42["bank"] = w42["bank"].astype(str)
        leg = pd.DataFrame(R5B_IW_REF, columns=["outcome", "target", "bank", "w_mean_ref", "auc_ref"])
        jl = leg.merge(w42[["outcome", "target", "bank", "iw_w_mean", "iw_disc_auc"]], on=["outcome", "target", "bank"], how="left")
        n_have = int(jl["iw_w_mean"].notna().sum())
        rel = (jl["iw_w_mean"] / jl["w_mean_ref"] - 1.0).abs()
        da = (jl["iw_disc_auc"] - jl["auc_ref"]).abs()
        in_w, in_a = int((rel <= R5B_IW_REF_TOL["w_rel"]).sum()), int((da <= R5B_IW_REF_TOL["auc_abs"]).sum())
        rows.append({"target version": "version without rescaling (reproduction check)", "item": "the mean weight is close to the values of the 24 reference cells (cell medians)",
                     "value": "outside tolerance %d/%d cells (comparable %d cells) | maximum relative difference %.3f" % (in_w, len(jl), n_have, float(rel.max()) if n_have else float("nan")),
                     "criterion": "all 24 cells within relative +-%.0f%%" % (100 * R5B_IW_REF_TOL["w_rel"]), "agreement": bool(n_have == len(jl) and in_w == len(jl)),
                     "if outside": "not used as a reproduction of the reference values but as an implementation check (unrelated to the version with rescaling)"})
        rows.append({"target version": "version without rescaling (reproduction check)", "item": "the shift discrimination AUC is close to the 24 cell values of Supplementary S8 Panel H (cell medians)",
                     "value": "outside tolerance %d/%d cells (comparable %d cells) | maximum difference %.3f" % (in_a, len(jl), n_have, float(da.max()) if n_have else float("nan")),
                     "criterion": "all 24 cells within +-%s" % R5B_IW_REF_TOL["auc_abs"], "agreement": bool(n_have == len(jl) and in_a == len(jl)),
                     "if outside": "not used as a reproduction of the reference values but as an implementation check (unrelated to the version with rescaling)"})
    if wcmp is not None and len(wcmp):
        cb = [str(b) for b in R5B_IWC_CHECK["banks"]]
        s = wcmp[wcmp["bank"].astype(str).isin(cb)]
        n = len(s)
        le = int((s["iwc_share_at_lo"] <= s["iw_share_at_lo"] + 1e-12).sum())
        lt = int((s["iwc_share_at_lo"] < s["iw_share_at_lo"] - 1e-12).sum())
        out_rng = int((~((s["iwc_w_mean"] >= R5B_IWC_CHECK["w_mean_lo"]) & (s["iwc_w_mean"] <= R5B_IWC_CHECK["w_mean_hi"]))).sum())
        rows.append({"target version": "version with rescaling", "item": "the fraction stuck at the lower bound (w = %s) is at or below the version without rescaling (all cells of bank %s)" % (R5B_IW_CLIP[0], "·".join(cb)),
                     "value": "at or below %d/%d cells | lower cells %d/%d (cell median of lower bound fraction without rescaling %.2f versus with rescaling %.2f)" % (le, n, lt, n, float(s["iw_share_at_lo"].median()) if n else float("nan"), float(s["iwc_share_at_lo"].median()) if n else float("nan")),
                     "criterion": "at or below in all cells, and more than half of the cells lower", "agreement": bool(n > 0 and le == n and lt * 2 > n), "if outside": "treated as a code error and the values of the version with rescaling must not be used"})
        rows.append({"target version": "version with rescaling", "item": "the mean weight is near 1 (cell median of bank %s)" % "·".join(cb),
                     "value": "cell range %.3f to %.3f | outside criterion %d/%d cells (cell median of version without rescaling %.3f)" % (float(s["iwc_w_mean"].min()) if n else float("nan"), float(s["iwc_w_mean"].max()) if n else float("nan"), out_rng, n, float(s["iw_w_mean"].median()) if n else float("nan")),
                     "criterion": "%s ~ %s" % (R5B_IWC_CHECK["w_mean_lo"], R5B_IWC_CHECK["w_mean_hi"]), "agreement": bool(n > 0 and out_rng == 0), "if outside": "not evidence of an implementation error. The mean weight table is shown with the pattern; the pattern is not blocked"})
    return pd.DataFrame(rows)


def r5b_split_table(cells, patterns, pairs=(("iw_erm", "iwc_erm"), ("iw_platt", "iwc_platt"), ("iw_temp", "iwc_temp"))):
    """do the two versions (version without rescaling and version with rescaling) split widely (cells: table of cells collected from r5b_cell_frame, method, fs, outcome, target, bank, D_plus, DOM, NULL),
    patterns: pattern table per method (method, pattern, pattern_strict, pattern_loose). One line per variant: number of cells in which any of the three marks differ in the same cell | three patterns | whether they split."""
    rows = []
    for a, b in pairs:
        ca, cb = cells[cells["method"] == a], cells[cells["method"] == b]
        ga, gb = patterns[patterns["method"] == a], patterns[patterns["method"] == b]
        if not (len(ca) and len(cb) and len(ga) and len(gb)):
            continue
        j = ca.merge(cb, on=R5B_W_KEYS, suffixes=("_1", "_c"))
        diff = np.zeros(len(j), dtype=bool)
        for c in ("D_plus", "DOM", "NULL"):
            diff |= (j[c + "_1"].astype(bool) != j[c + "_c"].astype(bool)).to_numpy()
        n_diff, n_diff_e = int(diff.sum()), int((diff & (j["target"] == "eICU").to_numpy()).sum())
        g1, gc = ga.iloc[0], gb.iloc[0]
        pattern_diff = any(str(g1[k]) != str(gc[k]) for k in ("pattern", "pattern_strict", "pattern_loose"))
        split = bool(n_diff >= R5B_SPLIT_CELLS or pattern_diff)
        rows.append({"variant": a.split("_", 1)[1], "n_cells": int(len(j)), "n_diff_cells": n_diff, "n_diff_eicu": n_diff_e,
                     "pattern_unscaled": g1["pattern"], "pattern_rescaled": gc["pattern"], "pattern_unscaled_strict": g1["pattern_strict"], "pattern_rescaled_strict": gc["pattern_strict"],
                     "pattern_unscaled_loose": g1["pattern_loose"], "pattern_rescaled_loose": gc["pattern_loose"], "split": split,
                     "reporting": "both versions are shown (the pattern is that of the version with rescaling, with the pattern of the version without rescaling alongside)" if split else "the version with rescaling is the main result; the version without rescaling is one line and a csv (for reproduction)"})
    return pd.DataFrame(rows)


def r5b_margin_split(cells, patterns):
    """three margin sets: per method, the number of cells whose strict and loose marks (D+, DOM, NULL) differ from base and three patterns.
    cells: table collected from r5b_cell_frame (D_plus, DOM, NULL and *_strict, *_loose columns), patterns: method, pattern, pattern_strict, pattern_loose. One line = method x sensitivity set."""
    rows = []
    for m, s in cells.groupby("method", sort=False):
        g = patterns[patterns["method"] == m]
        for nm in ("strict", "loose"):
            ch = {k: (s[k].astype(bool) != s[k + "_" + nm].astype(bool)) for k in ("D_plus", "DOM", "NULL")}
            anyc = ch["D_plus"] | ch["DOM"] | ch["NULL"]
            gb, gs = (str(g["pattern"].iloc[0]), str(g["pattern_" + nm].iloc[0])) if len(g) else ("", "")
            rows.append({"method": m, "margin_set": nm, "n_cells": int(len(s)), "n_cells_changed": int(anyc.sum()), "n_changed_eicu": int((anyc & (s["target"] == "eICU")).sum()),
                         "n_changed_D_plus": int(ch["D_plus"].sum()), "n_changed_DOM": int(ch["DOM"].sum()), "n_changed_NULL": int(ch["NULL"].sum()),
                         "pattern_base": gb, "pattern_this_set": gs, "pattern_differs": bool(gb != gs)})
    return pd.DataFrame(rows)


def r5b_hosp_boot_streams():
    """repeats the random stream of the aggregation script (main-text hospital-level interval). One generator (seed R5B_HOSP_BOOT_SEED) is used in sequence over feature set -> comparison pairs (4) -> outcome, and in each call
    draws the (B x k) replicate indices for k hospitals once. Returns: (feature set, outcome) -> the replicate indices used by the first pair (PRAM versus frozen). All methods of the same (feature set, outcome) share these replicates."""
    rng = np.random.default_rng(R5B_HOSP_BOOT_SEED)
    out = {}
    for fs in R5B_HOSP_BOOT_FS:
        for j in range(R5B_HOSP_BOOT_PAIRS):
            for oc, k in R5B_HOSP_BOOT_K:
                idx = rng.integers(0, k, size=(R5B_HOSP_BOOT_N, k))
                if j == 0:
                    out[(fs, oc)] = idx
    return out


def r5b_hosp_boot_table(rows, order=None):
    """hospital-level interval (same kind as the main text): stay-weighted mean of the per-hospital delta AUROC and the hospital bootstrap 95% interval (percentiles 2.5 and 97.5). The definition is the same as `hospital_stats` of the aggregation script
    (hospitals resampled with replacement, the same replicates for all methods). rows: fs, outcome, method, hospitalid, n_site, d_auroc_b1, and (if present) d_auroc_pram.
    order: (feature set, outcome) -> hospital id order (order of the PRAM rows of the stored hospital resume). If the number and composition of hospitals equal those of the aggregation script (R5B_HOSP_BOOT_K), the same random stream and order as that script are used so that
    the PRAM row has the same trailing digits as the main-text interval (shown in the column stream). Otherwise (including when there is no order), sorted by hospital id and drawn anew from the seed.
    columns: k, stream, wmean_vs_b1, lo_vs_b1, hi_vs_b1, wmean_vs_pram, lo_vs_pram, hi_vs_pram (nan if there is no versus-PRAM column)."""
    out = []
    exp_k = dict(R5B_HOSP_BOOT_K)
    streams = r5b_hosp_boot_streams() if order else {}
    for (fs, oc, m), s in rows.groupby(["fs", "outcome", "method"], sort=False):
        ord_ = (order or {}).get((fs, oc))
        if ord_ is not None and (fs, oc) in streams and exp_k.get(oc) == len(ord_) == len(s) and set(s["hospitalid"]) == set(ord_):
            s = s.set_index("hospitalid").loc[list(ord_)].reset_index()
            idx, stream = streams[(fs, oc)], "same random stream as the aggregation script"
        else:
            s = s.sort_values("hospitalid")
            idx, stream = np.random.default_rng(R5B_HOSP_BOOT_SEED).integers(0, len(s), size=(R5B_HOSP_BOOT_N, len(s))), ("independent random stream (the number and composition of hospitals differ from the aggregation script)" if order else "independent random stream")
        k = len(s)
        w = s["n_site"].to_numpy(float)
        rec = {"fs": fs, "outcome": oc, "method": m, "k": int(k), "stream": stream}
        for tag, col in (("b1", "d_auroc_b1"), ("pram", "d_auroc_pram")):
            if col not in s.columns or not s[col].notna().any() or m == "pram_b1" and tag == "pram":
                rec.update({"wmean_vs_" + tag: float("nan"), "lo_vs_" + tag: float("nan"), "hi_vs_" + tag: float("nan")})
                continue
            v = s[col].to_numpy(float)
            ok = np.isfinite(v)
            we, vf = np.where(ok, w, 0.0), np.where(ok, v, 0.0)
            est = float((we * vf).sum() / we.sum())
            bs = (we[idx] * vf[idx]).sum(1) / np.maximum(we[idx].sum(1), 1e-300)
            lo, hi = np.percentile(bs, [2.5, 97.5])
            rec.update({"wmean_vs_" + tag: est, "lo_vs_" + tag: float(lo), "hi_vs_" + tag: float(hi)})
        out.append(rec)
    return pd.DataFrame(out)


def r5b_hosp_boot_check(tab):
    """whether the PRAM (pram_b1) versus frozen row of the hospital-level interval table (r5b_hosp_boot_table) equals the main-text hospital-level interval (R5B_HOSP_BOOT_REF, output of the aggregation script).
    with the same random stream, hospital order and definition, they agree within R5B_HOSP_BOOT_REF_TOL. If not, that PRAM row is not used as the main-text value (the main-text value is the output of the aggregation script)."""
    rows = []
    for (fs, oc), ref in R5B_HOSP_BOOT_REF.items():
        r = tab[(tab["fs"] == fs) & (tab["outcome"] == oc) & (tab["method"] == "pram_b1")]
        if not len(r):
            continue
        r = r.iloc[0]
        now = (float(r["wmean_vs_b1"]), float(r["lo_vs_b1"]), float(r["hi_vs_b1"]))
        diff = max(abs(a - b) for a, b in zip(now, ref))
        same_stream = str(r["stream"]) == "same random stream as the aggregation script"
        rows.append({"fs": fs, "outcome": oc, "k": int(r["k"]), "stream": r["stream"], "manuscript value mean": ref[0], "manuscript value lower": ref[1], "manuscript value upper": ref[2],
                     "notebook mean": now[0], "notebook lower": now[1], "notebook upper": now[2], "maximum difference": diff, "agreement": bool(same_stream and diff <= R5B_HOSP_BOOT_REF_TOL)})
    return pd.DataFrame(rows)


## Pre-check: finished units, units to run, expected time

This cell **does not do new computation** (it measures the neighbor search speed briefly and the time of one call at bank 500; nothing is stored). It prints the following.

1. Number of units to run and units to skip (finished units of this notebook in the folder are skipped).
2. An expected number of minutes (approximate) from the cosine neighbor search cost model (the same formula as the pre-check cell of notebook 01) and the fixed time of one call. It is recounted at the actual speed after the first unit.
3. Whether stored results are read or recomputed is counted by the input-collection pre-check cell after this cell (its expected minutes are added to the total expected time).


In [ ]:
# ================================================================
# PRAM stage B  pre-check (new units | expected time)
# ================================================================
R5_PLAN = pd.DataFrame()
R5_TIMES = {}                                           # section name -> (expected minutes, actual seconds)
R5_C_CELL, R5_T_CALL = float("nan"), float("nan")
R5_EXP_MIN = {}

if not DATA_OK:
    status("part 1 - pre-check", "not available", "main inputs absent")
else:
    _c_ref = CTX[("F42", OUTCOMES[0], "eICU")]
    R5_C_CELL = r5_knn_seconds_per_cell(_c_ref)
    _bk_small = min(TAIL_BANKS)
    _yy = _c_ref["y"]
    _pool_small = np.arange(len(_yy))[: len(_yy) * (N_FOLDS - 1) // N_FOLDS]
    _bank_small = _pool_small[sample_bank(_yy[_pool_small], _bk_small, "r5plan")]
    _test_small = np.arange(len(_yy))[len(_yy) * (N_FOLDS - 1) // N_FOLDS:]
    _t0 = time.time()
    bank_predictions(_c_ref, _bank_small, _test_small, "r5plan", set(R5_TAIL_GROUPS))
    _t_call = time.time() - _t0
    R5_T_CALL = max(0.0, _t_call - r5_cells_for(len(_test_small), len(_bank_small), 2) * R5_C_CELL)       # fixed time of one call (excluding the neighbor search share)
    _Rw = Resumable(R5_RES_WHOLE)
    _rows = []
    _sec_w = sum((r5_whole_cells(len(CTX[(u[0], u[1], u[2])]["y"])) * R5_C_CELL * 1.3 + 2 * N_FOLDS * R5_T_CALL) for u in R5_WHOLE_UNITS if not _Rw.unit_done(u))
    _n_w_new = sum(1 for u in R5_WHOLE_UNITS if not _Rw.unit_done(u))
    _rows.append({"item": f"(a) {R5_WHOLE_FS} full-bank paired DeLong", "unit": len(R5_WHOLE_UNITS), "units to run": _n_w_new, "units to skip": len(R5_WHOLE_UNITS) - _n_w_new,
                  "expected minutes (approximate)": int(round(_sec_w / 60.0))})
    R5_EXP_MIN["(a) full bank"] = _sec_w / 60.0
    _pair_n = N_FOLDS * max(1, N_DRAWS_TAIL // N_FOLDS)
    _sec_t_all = 0.0
    for _grp, _units in (("(b) 42-column single bank", [u for u in R5_TAIL_UNITS if u[0] in R5_TAIL_FS]), ("(b) 33-column MIMIC supplement", [u for u in R5_TAIL_UNITS if u[0] in R5_TAIL_EXTRA_FS])):
        _Rt = Resumable(R5_RES_TAIL[_units[0][0]]) if _units else None
        _todo = [u for u in _units if not _Rt.unit_done(u)]
        _sec = sum(_pair_n * (R5_T_CALL + r5_cells_for(len(CTX[(u[0], u[1], u[2])]["y"]) / N_FOLDS, u[3], 2) * R5_C_CELL * 1.3) for u in _todo)
        _sec_t_all += _sec
        _rows.append({"item": f"{_grp} 100-draw interval (per cell {_pair_n} banks)", "unit": len(_units), "units to run": len(_todo), "units to skip": len(_units) - len(_todo), "expected minutes (approximate)": int(round(_sec / 60.0))})
        R5_EXP_MIN[_grp] = _sec / 60.0
    _Rrw, _Rrt = Resumable(R5_RES_REPRO[0]), Resumable(R5_RES_REPRO[1])
    _rw_todo = [(fs, oc, tg, "full") for fs, oc, tg in R5_REPRO_WHOLE if not _Rrw.unit_done((fs, oc, tg, "full"))]
    _rt_todo = [u for u in R5_REPRO_TAIL if not _Rrt.unit_done(u)]
    _rep_sec = (sum(r5_whole_cells(len(CTX[(u[0], u[1], u[2])]["y"])) * R5_C_CELL * 1.3 + 2 * N_FOLDS * R5_T_CALL for u in _rw_todo)
                + sum(_pair_n * (R5_T_CALL + r5_cells_for(len(CTX[(u[0], u[1], u[2])]["y"]) / N_FOLDS, u[3], 2) * R5_C_CELL * 1.3) for u in _rt_todo))
    R5_EXP_MIN["reproduction check"] = _rep_sec / 60.0
    _rows.append({"item": "reproduction check (2 full-bank units + 2 33-column eICU tail units, whether the values equal the stored results)", "unit": len(R5_REPRO_WHOLE) + len(R5_REPRO_TAIL),
                  "units to run": len(_rw_todo) + len(_rt_todo), "units to skip": len(R5_REPRO_WHOLE) + len(R5_REPRO_TAIL) - len(_rw_todo) - len(_rt_todo), "expected minutes (approximate)": int(round(_rep_sec / 60.0))})
    R5_PLAN = pd.DataFrame(_rows)
    emit_table(f"what this run will compute (neighbor search {R5_C_CELL * 1e9:.1f} ns / query x reference row, fixed time of one call {R5_T_CALL:.2f} s counted)", R5_PLAN)
    emit(f"- units to run {int(R5_PLAN['units to run'].sum())} | units to skip {int(R5_PLAN['units to skip'].sum())} | expected about {int(round(sum(R5_EXP_MIN.values())))} min (rough estimate) "
         "(data loading and frozen model fit about 75 seconds and result aggregation are separate). Whether the stored results are read or recomputed, and their time, are counted by the input-collection pre-check below. "
         "Not one line is written to existing resume files.")
    status("part 1 - write lock", "done", f"existing files {len(R5_SNAP0)} hashes | new names {len(R5_NEW_NAMES)} columns")
    status("part 1 - pre-check", "done", f"new unit {int(R5_PLAN['units to run'].sum())} columns")


## Stage B pre-check: finished units, units to run, expected time

This cell does not do new computation of B (it only measures the time of a few fits and stores nothing). It prints: (1) that the comparison targets of B (frozen, PRAM, recalibration, local retraining) are read from the stored resume or recomputed by the input-collection section; (2) hospital ids and eligible hospitals; (3) the development period rules (whether it is a MIMIC-IV patient-level approximation); (4) the time one fit takes per method and the expected minutes counted from it (approximate). After the first unit the actual speed is printed in the log.


In [ ]:
# ================================================================
# PRAM stage B  pre-check (comparison targets | hospitals | periods | expected time)
# ================================================================
R5B_HID, R5B_SITES, R5B_PERIODS, R5B_PERIODS_NOTE = {}, {}, None, ""
R5B_EXP_MIN, R5B_TIMES, R5B_STOPS = {}, {}, []
R5B_COMP = pd.DataFrame()
R5B_HOSP_COMP = pd.DataFrame()                          # hospital-level comparison targets (values read from the stored resume or recomputed by the input-collection section). Filled by input collection 2
R5B_HAVE_HOSP = False
R5B_HPRAM = {}                                          # feature set -> {(outcome, hospital): within-hospital 3-fold PRAM test predictions}. Made here so that the hospital anchor cell and the report cell do the same computation only once
R5B_N_PERIOD_ENVS = 0
R5B_PERIOD_COL = {}                                      # AR-period environment name -> period rule column. period = main rule (R5B_PERIOD_MAIN_RULE), period_alt = other rule (sensitivity). A patient-level approximation copy has only period (patient_group)
R5B_PLAN = pd.DataFrame()
R5B_ON = {"iw": RUN_B_IW, "coral": RUN_B_CORAL, "anchor": RUN_B_ANCHOR, "anchor_hosp": RUN_B_ANCHOR and RUN_B_ANCHOR_HOSP}

if not DATA_OK:
    status("part 1 - stage B pre-check", "not available", "main inputs absent")
else:
    emit("- The comparison targets of B (per-draw values of frozen, PRAM, logistic recalibration and local retraining) are collected by the input-collection section later: if finished units exist in the stored resume `matrix_*` and `hospital_*` they are read, and if absent or insufficient they are recomputed with the same function and the same draw key. This cell neither reads nor computes (it only measures the hospital ids, period rules and fit time).")
    if HAVE_HOSPITALID and EICU_TAB is not None:
        R5B_HID, R5B_SITES = prepare_sites()
        R5B_HAVE_HOSP = bool(R5B_SITES) and all(len(v) > 0 for v in R5B_SITES.values())
        emit(f"- stays linked to a hospital id | eligible hospitals: " + " · ".join(f"{oc} {int(np.isfinite(R5B_HID[oc]).sum())}/{len(R5B_HID[oc])} stays, {len(R5B_SITES[oc])} hospitals" for oc in OUTCOMES)
             + f" -> hospital-level comparison {'possible (the input-collection section reads hospital_* of the stored results or recomputes with the same function)' if R5B_HAVE_HOSP else 'not possible (there is no eligible hospital)'}")
    else:
        emit("- there is no hospital id column, so the hospital-level comparison is skipped (see the input check table of the settings cell).")
    if RUN_B_ANCHOR and HAVE_MIMIC4_YEAR:
        try:
            R5B_PERIODS, _lab = read_mimic4_periods()
            R5B_N_PERIOD_ENVS = len(_lab)
            if "patient_group" in R5B_PERIODS.columns:                      # patient-level approximation: strict and majority are the same column, so only one is computed
                R5B_PERIOD_COL = {"period": "patient_group"}
                _rule_note = "patient-level approximation, so there is one period rule (patient_group)"
            else:                                                           # unit-transfer adjustment: both rules are computed. One main rule is the main target and the other is the sensitivity
                _alt_rule = [r_ for r_ in R5B_PERIOD_ALL_RULES if r_ != R5B_PERIOD_MAIN_RULE][0]
                R5B_PERIOD_COL = {"period": R5B_PERIOD_MAIN_RULE, "period_alt": _alt_rule}
                _rule_note = f"two period rules: main {R5B_PERIOD_MAIN_RULE}(env period, main target) | sensitivity {_alt_rule}(env period_alt)"
            R5B_PERIODS_NOTE = f"MIMIC-IV periods {len(_lab)} ({', '.join(_lab)} ) | method {M4_MODE} · {_rule_note}"
        except Exception as _e:
            R5B_PERIODS_NOTE = f"could not read the period table: {type(_e).__name__}: {_e}"
            print("  failed to read development period rules:", type(_e).__name__, _e)
    elif RUN_B_ANCHOR:
        R5B_PERIODS_NOTE = "anchor_year_group not found in MIMIC-IV patients and icustays (input check table of the settings cell)"
    emit(f"- environments of the development split period version (AR-period): {R5B_PERIODS_NOTE if R5B_PERIODS_NOTE else 'not applicable (anchor off)'}. If this copy is a patient-level approximation, the periods are approximate (limitation). If AR-period is missing, the B-3 state is partial (no AR-period) and goes into the sections not completed of the last cell.")
    # time of one fit (for the expected time)
    _c = CTX[(R5B_FS[0], OUTCOMES[0], "eICU")]
    _n = len(_c["y"])
    _sizes = [min(b for b in BANK_SIZES if b != "full"), int(0.8 * _n)]
    _tt = {}
    r5b_disc_memo_clear()                               # the two versions (iw and iwc) share one shift discriminator fit: the version with rescaling measurement does not include a discriminator fit (the same as the actual run)
    for _kind, _fn in (("iw", r5b_iw_predictions), ("iwc", lambda _a1, _a2, _a3: r5b_iw_predictions(_a1, _a2, _a3, rescaled=True)), ("coral", r5b_coral_predictions)):
        _pts = []
        for _sz in _sizes:
            _bank = np.arange(_sz)
            _t0 = time.time()
            _fn(_c, _bank, np.arange(_sz, min(_sz + 2000, _n)))
            _pts.append((_sz, time.time() - _t0))
        _slope = (_pts[1][1] - _pts[0][1]) / max(_pts[1][0] - _pts[0][0], 1)
        _tt[_kind] = (max(_pts[0][1] - _slope * _pts[0][0], 0.0), max(_slope, 0.0))
    r5b_disc_memo_clear()
    _X2 = np.vstack([_c["Xs_src"], CTX[(R5B_FS[0], OUTCOMES[0], "MIMIC-III")]["Xg"]])
    _y2 = np.concatenate([_c["ys_src"], CTX[(R5B_FS[0], OUTCOMES[0], "MIMIC-III")]["y"]])
    _e2 = np.r_[np.zeros(len(_c["ys_src"]), dtype=int), np.ones(len(_y2) - len(_c["ys_src"]), dtype=int)]
    _n_rows_2db = len(_y2)
    # anchor: not one fit but the whole gamma grid (environment-level folds + final fit + Platt and negative log likelihood per fold) is timed once per form and used. It is the measurement of a two-environment grid and is scaled by the actual number of rows and folds
    _t_grid = {}
    for _form in R5B_AR_METHODS:
        _t0 = time.time()
        try:
            r5b_anchor_fit_grid(_X2, _y2, _e2, _form)
            _t_grid[_form] = time.time() - _t0
        except R5BStop as _e:
            _t_grid[_form] = float("nan")
            print(f"  the anchor fit stopped during timing ({_form} ) (the main computation also stops at the same place): {_e}")
    _t_grid_sum = sum(v for v in _t_grid.values() if np.isfinite(v))
    _rows = []
    # cell-level IW and CORAL: number of fits per bank size x time of one fit (linear in the number of bank rows)
    _lab_min = {"iw": "B-1 importance weighting", "iwc": "B-1 importance weighting", "coral": "B-2 CORAL"}
    _lab_row = {"iw": "B-1 importance weighting, version without rescaling (cells + hospitals)", "iwc": "B-1 importance weighting, version with rescaling (cells + hospitals, the same amount as the version without rescaling so B-1 is doubled)", "coral": "B-2 CORAL (cells + hospitals)"}
    _t0 = time.time()
    full_metrics(_c["y"], np.asarray(_c["pb1"], dtype=float))
    _t_fm = (time.time() - _t0) / len(_c["y"])           # time of metric computation (AUROC, ECE, slope, net benefit curve, etc.) per row. Costs as many times as the number of methods per draw
    R5B_EXP_MIN["B hospital level (importance weighting and CORAL)"] = 0.0
    for _kind, _on in (("iw", RUN_B_IW), ("iwc", RUN_B_IW), ("coral", RUN_B_CORAL)):
        _sec = 0.0
        for _fs in R5B_FS:
            for _oc in OUTCOMES:
                for _tg in FS_SPEC[_fs]["targets"]:
                    _ny = len(CTX[(_fs, _oc, _tg)]["y"])
                    for _bk in FS_SPEC[_fs]["banks"]:
                        _pool = _ny * (N_FOLDS - 1) // N_FOLDS
                        _rows_bank = _pool if _bk == "full" else min(_bk, _pool)
                        _sec += (1 if _bk == "full" else N_DRAWS_MAIN) * (N_FOLDS * (_tt[_kind][0] + _tt[_kind][1] * _rows_bank) + (len(R5B_KEEP[_kind]) + 1) * _t_fm * _ny)
        _hs = 0.0
        if R5B_HAVE_HOSP:
            for _fs in [f for f in R5B_FS if FS_SPEC[f]["hospital"]]:
                for _oc in OUTCOMES:
                    _hs += len(R5B_SITES[_oc]) * 3 * (_tt[_kind][0] + _tt[_kind][1] * 300)
        R5B_EXP_MIN[_lab_min[_kind]] = R5B_EXP_MIN.get(_lab_min[_kind], 0.0) + (_sec / 60.0 if _on else 0.0)
        R5B_EXP_MIN["B hospital level (importance weighting and CORAL)"] += _hs / 60.0 if _on else 0.0
        _rows.append({"item": _lab_row[_kind], "on": _on, "one fit [s]": round(_tt[_kind][0] + _tt[_kind][1] * 5000, 2), "expected minutes (approximate)": int(round((_sec + _hs) / 60.0)) if _on else 0})
    # anchor: the measured time of the whole two-environment grid (_t_grid_sum, sum of two forms) is scaled by the number of rows and folds. With k environment-level folds it is k fits on (k-1)/k of the training rows + 1 final fit, about k fits of the full rows (2 with two environments)
    _n_dev = len(_c["ys_src"])
    _np_env = R5B_N_PERIOD_ENVS if R5B_N_PERIOD_ENVS >= 2 else 5
    _n_period_rules = max(1, len(R5B_PERIOD_COL))                 # number of AR-period environment versions (main rule + sensitivity rule = 2, patient-level approximation = 1)

    def _grid_sec(_nrows, _nenv):
        return _t_grid_sum * (_nrows / _n_rows_2db) * (min(R5B_CV_FOLDS, _nenv) / 2.0)
    _sec = 0.0
    for _fs in R5B_FS:
        for _oc in OUTCOMES:
            _sec += _grid_sec(_n_rows_2db, 2) + _n_period_rules * _grid_sec(_n_dev, _np_env)
    _hsec = 0.0
    if R5B_HAVE_HOSP:
        for _fs in [f for f in R5B_FS if FS_SPEC[f]["hospital"]]:
            for _oc in OUTCOMES:
                _rows_h = _n_dev + int(np.isfinite(R5B_HID[_oc]).sum())
                _hsec += len(R5B_SITES[_oc]) * _grid_sec(_rows_h, R5B_CV_FOLDS)
    _fsec = 0.0
    if RUN_B_ANCHOR:
        for _fs in R5B_FS:
            for _oc in OUTCOMES:
                for _tg in FS_SPEC[_fs]["targets"]:
                    _ny = len(CTX[(_fs, _oc, _tg)]["y"])
                    _fsec += sum((1 if bk == "full" else N_DRAWS_MAIN) * N_FOLDS * (R5_T_CALL + r5_cells_for(_ny / N_FOLDS, (_ny * (N_FOLDS - 1) // N_FOLDS if bk == "full" else bk), 1) * R5_C_CELL * 1.3) for bk in FS_SPEC[_fs]["banks"])
    _pram_one = 0.0                                     # time, one amount, of recomputing PRAM per hospital with the same key as the stored results for the hospital-level paired comparison
    if R5B_HAVE_HOSP:
        for _fs in [f for f in R5B_FS if FS_SPEC[f]["hospital"]]:
            for _oc in OUTCOMES:
                _cnt = pd.Series(R5B_HID[_oc][np.isfinite(R5B_HID[_oc])]).value_counts()
                for _h in R5B_SITES[_oc]:
                    _nh = float(_cnt.get(float(_h), 0))
                    _pram_one += 3 * (R5_T_CALL + r5_cells_for(_nh / 3.0, 2.0 * _nh / 3.0, 2) * R5_C_CELL * 1.3)
    _passes_fam = 2 * int(bool(RUN_B_IW)) + int(bool(RUN_B_CORAL))        # number of times the two hospital-level IW versions and CORAL each recompute PRAM
    _hosp_anchor_on = bool(RUN_B_ANCHOR and RUN_B_ANCHOR_HOSP and R5B_HAVE_HOSP)
    _report_pram = bool(RUN_B_ANCHOR and R5B_HAVE_HOSP and not RUN_B_ANCHOR_HOSP)       # if the hospital anchor cell is off, the report cell does one hospital PRAM recomputation (if on, the hospital anchor cell does it and the report cell shares)
    R5B_EXP_MIN["B hospital level (importance weighting and CORAL)"] += _pram_one * _passes_fam / 60.0
    R5B_EXP_MIN["B-3 anchor (fit and evaluation)"] = _sec / 60.0 if RUN_B_ANCHOR else 0.0
    R5B_EXP_MIN["B-3 hospital anchor extension"] = (_hsec + _pram_one) / 60.0 if _hosp_anchor_on else 0.0
    R5B_EXP_MIN["B report (hospital-level PRAM recomputation and interval)"] = _pram_one / 60.0 if _report_pram else 0.0
    R5B_EXP_MIN["PRAM on B-3 anchor (analysis F)"] = _fsec / 60.0
    _gl = ", ".join("%s %.1f s" % (k_, v_) for k_, v_ in _t_grid.items() if np.isfinite(v_))
    _rows.append({"item": "B hospital-level PRAM recomputation (IW two versions and CORAL recompute PRAM per hospital with the same key as the stored results for the paired DeLong of X versus PRAM, separate from the hospital fit above)", "on": R5B_HAVE_HOSP, "one fit [s]": "", "expected minutes (approximate)": int(round(_pram_one * _passes_fam / 60.0))})
    _rows.append({"item": "B-3 anchor fit and evaluation (two-environment version and period version, whole grid measured once: %s)" % (_gl if _gl else "could not measure"), "on": RUN_B_ANCHOR, "one fit [s]": "", "expected minutes (approximate)": int(round(R5B_EXP_MIN["B-3 anchor (fit and evaluation)"]))})
    _rows.append({"item": "B-3 hospital anchor extension (hospital fit + one hospital PRAM recomputation. Turned off separately with RUN_B_ANCHOR_HOSP)", "on": _hosp_anchor_on, "one fit [s]": "", "expected minutes (approximate)": int(round(R5B_EXP_MIN["B-3 hospital anchor extension"]))})
    _rows.append({"item": "hospital PRAM recomputation of the B report (only when the hospital anchor extension is off)", "on": _report_pram, "one fit [s]": "", "expected minutes (approximate)": int(round(R5B_EXP_MIN["B report (hospital-level PRAM recomputation and interval)"]))})
    _rows.append({"item": "PRAM on B-3 anchor (analysis F, the retrieval cost model is the same as A)", "on": RUN_B_ANCHOR, "one fit [s]": "", "expected minutes (approximate)": int(round(R5B_EXP_MIN["PRAM on B-3 anchor (analysis F)"]))})
    R5B_PLAN = pd.DataFrame(_rows)
    emit_table("what stage B will compute (expected minutes are a short measurement on this machine multiplied by the number of fits per environment and bank size (rough estimate))", R5B_PLAN)
    emit(f"- stage B expected about {int(round(sum(R5B_EXP_MIN.values())))} min (rough estimate). B units already finished are skipped (R5_b_ resume).")
    status("part 1 - stage B pre-check", "done", f"hospital level {'possible' if R5B_HAVE_HOSP else 'not possible'}")


## Input pre-check: what to read, what to recompute, expected time

This cell **does not do new computation** (it only reads the unit markers of the resume files and measures the time of one fit call and of one network hospital; nothing measured is stored). For the six items (rescoring, 33-column eICU tail, grid, hospital level, network factor 1, TabPFN sample) each unit is classified as read from notebook 01 results, read from the resume of this notebook, or computed now, and the expected minutes of the units to compute are counted. If this folder has none of the stored results, the "computed" column below is all filled and the total expected time increases accordingly. The definitions are not changed to save time (the same function, the same draw key and the same number of draws).


In [ ]:
# ================================================================
# PRAM input-collection pre-check
# ================================================================
R5P_ARTS, R5P_PLAN = {}, pd.DataFrame()
if not DATA_OK:
    status("part 1 - input-collection pre-check", "not available", "main inputs absent")
else:
    R5P_ARTS = r5p_artifacts()
    if any(k.startswith(("matrix_", "hospital_")) for k in R5P_ARTS):                   # measures the fixed time of a grid or hospital call (it includes local retraining and recalibration fits) once. Results are not stored
        _cm = CTX[("F42", OUTCOMES[0], "eICU")]
        _ym = _cm["y"]
        _cut = len(_ym) * (N_FOLDS - 1) // N_FOLDS
        _pool_m = np.arange(len(_ym))[:_cut]
        _bank_m = _pool_m[sample_bank(_ym[_pool_m], min(TAIL_BANKS), "r5pplan")]
        _test_m = np.arange(len(_ym))[_cut:]
        _t0m = time.time()
        bank_predictions(_cm, _bank_m, _test_m, "r5pplan", set(R5P_GROUPS))
        R5P_T_CALL_M[0] = max(0.0, (time.time() - _t0m) - r5_cells_for(len(_test_m), len(_bank_m), 2) * R5_C_CELL)
    _rows, _tot = [], 0.0
    for _k, _a in R5P_ARTS.items():
        _cl = r5p_classify(_a)
        _todo = [u for u, s in _cl.items() if s == "computed"]
        if _k.startswith("network_") and _todo:
            _u0 = _todo[0]
            _t0n = time.time()
            network_unit(CTX[(_u0[0], _u0[1], "eICU")], _u0[1], _u0[2], R5B_HID[_u0[1]], R5B_SITES[_u0[1]], [R5P_NET_FACTOR], NET_ALPHA_GRID)
            R5P_NET_SEC[0] = (time.time() - _t0n) * 1.5               # hospital sizes differ, so 1.5 times the time of the first hospital as a margin (approximate)
        _sec = sum(_a["cost"](u) for u in _todo)
        _tot += _sec if np.isfinite(_sec) else 0.0
        R5_EXP_MIN["input collection: " + _a["label"]] = (_sec / 60.0) if np.isfinite(_sec) else 0.0
        _rows.append({"item": _a["label"], "units needed": len(_a["units"]), "read from stored results (count)": sum(1 for s in _cl.values() if s == "read from stored results"),
                      "read from the resume of this notebook (count)": sum(1 for s in _cl.values() if s == "resume of this notebook"), "computed anew this time (count)": len(_todo),
                      "expected minutes (approximate)": int(round(_sec / 60.0)) if np.isfinite(_sec) else -1})
    _a9_3 = (RESUME_DIR / "a9_tabpfn.jsonl").is_file()
    _rows.append({"item": "input of the TabPFN sample row count back-calculation (a9_tabpfn of the stored results)", "units needed": 4, "read from stored results (count)": 4 if _a9_3 else 0, "read from the resume of this notebook (count)": 0,
                  "computed anew this time (count)": 0, "expected minutes (approximate)": 0})
    R5P_PLAN = pd.DataFrame(_rows)
    emit_table("input collection: the stored results to be read from this folder and what this notebook will recompute (TabPFN is not rerun: if there is no resume, it is back-calculated from the aggregate constants of the stored csv)", R5P_PLAN)
    emit(f"- units newly computed by input collection {int(R5P_PLAN['computed anew this time (count)'].sum())} | units read {int(R5P_PLAN['read from stored results (count)'].sum() + R5P_PLAN['read from the resume of this notebook (count)'].sum())} | expected about {int(round(_tot / 60.0))} min (rough estimate). "
         "if opened in a folder without the stored results, it takes this much longer than with stored results. Units already finished are skipped when run again.")
    status("part 1 - input-collection pre-check", "done", f"new unit {int(R5P_PLAN['computed anew this time (count)'].sum())} | read {int(R5P_PLAN['read from stored results (count)'].sum())} columns")


## Extra computations pre-check: total expected time of A, B and the extra computations

This cell does not do new computation (it only measures the time of a few neighbor searches). The expected minutes of the extra computations (E-1 full bank 6 units, E-2 partner prevalence, and the read-only N back-calculation and S5 values) are added to the expected minutes of the A pre-check and B pre-check above, and the **total expected time** is printed within 5 minutes of the start (rough estimate). TabPFN is not rerun.


In [ ]:
# ================================================================
# PRAM extra pre-check (total expected time)
# ================================================================
X_PLAN = pd.DataFrame()
if not DATA_OK:
    status("part 1 - extra computations pre-check", "not available", "main inputs absent")
else:
    _x_rows = []
    _x_fz_sec = 0.0
    if RUN_X_FZ:
        for _oc, _tg in X_FZ_UNITS:
            if Resumable(X_NAMES["fz"]).unit_done(("F42vsF33", _oc, _tg, "full")):
                continue
            _ny = len(CTX[(X_FZ_PRAM_FS, _oc, _tg)]["y"])
            _x_fz_sec += N_FOLDS * (r5_cells_for(_ny / N_FOLDS, _ny * (N_FOLDS - 1) / N_FOLDS, 2) * R5_C_CELL * 1.3 + R5_T_CALL)
    R5_EXP_MIN["E-1 33-column frozen versus 42-column PRAM"] = _x_fz_sec / 60.0
    _x_rows.append({"item": "E-1 33-column frozen logistic versus 42 columns + PRAM (6 full-bank units)", "on": RUN_X_FZ, "expected minutes (approximate)": int(round(_x_fz_sec / 60.0))})
    _x_net_sec = 0.0
    if RUN_X_NET and R5B_SITES and X_NET_FS:
        _x_o0 = OUTCOMES[0]
        _x_h0 = R5B_SITES[_x_o0][0]
        _x_t0 = time.time()
        network_unit(CTX[(X_NET_FS[0], _x_o0, "eICU")], _x_o0, _x_h0, R5B_HID[_x_o0], R5B_SITES[_x_o0], X_NET_FACTORS, NET_ALPHA_GRID)
        _x_one = time.time() - _x_t0
        _x_n_h = sum(len(R5B_SITES[_o]) for _o in OUTCOMES) * len(X_NET_FS)
        _x_net_sec = _x_one * _x_n_h * 1.5                # hospital sizes differ, so 1.5 times the time of the first hospital as a margin (approximate)
    R5_EXP_MIN["E-2 partner prevalence"] = _x_net_sec / 60.0
    _x_rows.append({"item": "E-2 partner prevalence 0.5x and 2x (%d hospitals x %d feature sets)" % (sum(len(R5B_SITES[_o]) for _o in OUTCOMES) if R5B_SITES else 0, len(X_NET_FS)), "on": RUN_X_NET, "expected minutes (approximate)": int(round(_x_net_sec / 60.0))})
    _x_a9_min = 1 if (RESUME_DIR / "a9_tabpfn.jsonl").is_file() else 4           # back-calculating from csv aggregate constants takes a few minutes because candidates filtered by rounded fractions number several hundred per cell (approximate)
    _x_rows.append({"item": "N back-calculation and S5 Panel A values (no new model computation)", "on": bool(RUN_X_A9INV or RUN_X_S5A), "expected minutes (approximate)": _x_a9_min})
    X_PLAN = pd.DataFrame(_x_rows)
    emit_table("what the extra computations will compute (rough estimate)", X_PLAN)
    _x_a = sum(v for k, v in R5_EXP_MIN.items() if not k.startswith("E-"))
    _x_b = sum(R5B_EXP_MIN.values())
    _x_x = R5_EXP_MIN["E-1 33-column frozen versus 42-column PRAM"] + R5_EXP_MIN["E-2 partner prevalence"] + _x_a9_min
    emit(f"- **total expected about {int(round(_x_a + _x_b + _x_x))} min = about {(_x_a + _x_b + _x_x) / 60.0:.1f} hours** (rough estimate) : A and input collection about {int(round(_x_a))} min + B about {int(round(_x_b))} min + extra computation about {int(round(_x_x))} min "
         "(data loading and frozen model fit about 75 seconds and result aggregation are separate). Units already finished are skipped, so it shortens when run again. The actual speed is printed in the log after the first unit.")
    status("part 1 - extra computations pre-check", "done", f"total expected about {int(round(_x_a + _x_b + _x_x))} min")


## Input collection 1. Rescoring (42-column full bank, 10 draws) and the 33-column eICU tail

`rescore_unit` (the body of the rescoring cell of notebook 01) and `run_tail_unit` (that of the new-functions cell of notebook 01) are called with the same keys. Finished units in the resume of this folder are read and only the rest is computed and written to `R5_p_rescore_F42` and `R5_p_a2tail_F33`.


In [ ]:
# ================================================================
# PRAM input collection 1: rescoring and 33-column eICU tail
# ================================================================
R5P_RESCORE, R5P_A2T33 = pd.DataFrame(), pd.DataFrame()
if not DATA_OK:
    status("part 1 - input collection 1", "not available", "main inputs absent")
else:
    _t_p1 = time.time()
    if "rescore" in R5P_ARTS:
        R5P_RESCORE = r5p_provide("rescore", R5P_ARTS["rescore"])
    if "a2tail" in R5P_ARTS:
        R5P_A2T33 = r5p_provide("a2tail", R5P_ARTS["a2tail"])
        R5P_A2T33["bank"] = R5P_A2T33["bank"].astype(str)
    R5_TIMES["input collection 1 rescoring and 33-column eICU tail"] = (sum(v for k, v in R5_EXP_MIN.items() if k.startswith("input collection: rescoring") or k.startswith("input collection: 33-column eICU")), time.time() - _t_p1)
    status("part 1 - input collection 1", "done" if len(R5P_RESCORE) and len(R5P_A2T33) else "not available", f"rescoring {len(R5P_RESCORE)} rows | 33-column eICU tail {len(R5P_A2T33)}rows")


## Input collection 2. Strategy grid and hospital level (comparison targets of B, and the stored values for the (a) comparison and E-1)

Stage B compares each new method, in the same cells and the same draws, paired with the frozen logistic, PRAM, logistic recalibration and local retraining. Per-draw values of these four (`b1_base`, `pram_b1`, `platt_b1`, `local_lr`) are needed. If the stored resume has them they are read; otherwise `run_matrix_unit` (groups frozen, pram, recal and local) and `hospital_unit` (the same groups) are called with the same keys. The other groups of the grid (local CatBoost, anchored ridge, update hierarchy, Platt after retrieval) do not affect the values of these four and are not read here, so they are not recomputed.


In [ ]:
# ================================================================
# PRAM input collection 2: strategy grid and hospital level (comparison targets of B)
# ================================================================
R5P_MATRIX, R5P_HOSP = {}, {}
if not DATA_OK:
    status("part 1 - input collection 2", "not available", "main inputs absent")
else:
    _t_p2 = time.time()
    for _fs in R5B_FS:
        R5P_MATRIX[_fs] = r5p_provide("matrix_" + _fs, R5P_ARTS["matrix_" + _fs])
        R5P_MATRIX[_fs]["bank"] = R5P_MATRIX[_fs]["bank"].astype(str)
    if R5P_MATRIX:
        R5B_COMP = pd.concat([R5P_MATRIX[_fs] for _fs in R5B_FS], ignore_index=True)
    for _fs in [f for f in R5B_FS if ("hospital_" + f) in R5P_ARTS]:
        R5P_HOSP[_fs] = r5p_provide("hospital_" + _fs, R5P_ARTS["hospital_" + _fs])
    if R5P_HOSP:
        R5B_HOSP_COMP = pd.concat([_d.assign(fs=_fs) for _fs, _d in R5P_HOSP.items() if len(_d)], ignore_index=True)
        R5B_HAVE_HOSP = len(R5B_HOSP_COMP) > 0
    R5_TIMES["input collection 2 grid and hospital level"] = (sum(v for k, v in R5_EXP_MIN.items() if k.startswith("input collection: strategy grid") or k.startswith("input collection: hospital level")), time.time() - _t_p2)
    status("part 1 - input collection 2", "done" if len(R5B_COMP) else "not available", f"grid {len(R5B_COMP)} rows | hospital level {len(R5B_HOSP_COMP)}rows")


## Input collection 3. Network factor 1 and TabPFN sample

For the network, `network_unit` (factor 1.0) is called with the same hospitals and keys. TabPFN is not rerun: if the `a9_tabpfn` resume exists, the row count of the reduced sample is back-calculated with all digits; otherwise it is back-calculated with the aggregate constants (6 significant digits) of the stored csv. When back-calculating with constants, the values of the candidates are also rounded to the same 6 significant digits before comparing.


In [ ]:
# ================================================================
# PRAM input collection 3: network factor 1 and TabPFN reference rows
# ================================================================
R5P_NET1 = {}
R5P_A9, R5P_A9_SOURCE = pd.DataFrame(), "absent"
if not DATA_OK:
    status("part 1 - input collection 3", "not available", "main inputs absent")
else:
    _t_p3 = time.time()
    for _fs in X_NET_FS:
        if ("network_" + _fs) in R5P_ARTS:
            R5P_NET1[_fs] = r5p_provide("network_" + _fs, R5P_ARTS["network_" + _fs])
            if len(R5P_NET1[_fs]):
                R5P_NET1[_fs]["fs"] = _fs
    _a9p = RESUME_DIR / "a9_tabpfn.jsonl"
    if _a9p.is_file():
        R5P_A9, R5P_A9_SOURCE = Resumable("a9_tabpfn").df(), "a9_tabpfn resume of the stored results (all digits)"
    else:
        _rows9 = []
        for _k9, _v9 in R5_REF.get("a9", {}).items():
            if not _v9:
                continue
            _oc9, _tg9 = _k9.split("|")
            for _bk9 in _v9["banks"]:
                for _rep9 in range(_v9["n_rep"] if _bk9 != "full" else 1):
                    _rows9.append({"outcome": _oc9, "target": _tg9, "bank": _bk9, "rep": _rep9, "method": "logistic_frozen", "obs_rate": _v9["obs_rate"],
                                   **{c: _v9[c] for c in ("auroc", "brier", "ece", "slope", "intercept", "mean_pred")}, "reduced": _v9["reduced"]})
        R5P_A9 = pd.DataFrame(_rows9)
        R5P_A9_SOURCE = "stored csv aggregate constants (%d significant digits)" % R5P_A9_ROUND if len(R5P_A9) else "absent"
    emit(f"- TabPFN sample back-calculation input: {R5P_A9_SOURCE} ({len(R5P_A9)} rows). network factor 1: " + ", ".join(f"{k} {len(v)}rows" for k, v in R5P_NET1.items()))
    R5_TIMES["input collection 3 network factor 1"] = (sum(v for k, v in R5_EXP_MIN.items() if k.startswith("input collection: network")), time.time() - _t_p3)
    status("part 1 - input collection 3", "done", f"network {sum(len(v) for v in R5P_NET1.values())} rows | A9 input {R5P_A9_SOURCE}")


## Input collection 4. Input collection table and comparison with the aggregate constants

(1) What was read and what was newly computed is organized as counts (also in the summary md). (2) The secured values (read or computed now) are compared with aggregate constants taken from the stored csv files: the median over draws per grid cell x method, the median of the rescoring cells, the 100-draw median and interval fractions of the 33-column eICU tail, the distribution summary over all hospitals, the distribution summary of network factor 1, and Table 2 q. The tolerance is a relative error of 1e-4 (the csv has 6 significant digits). **The result md and csv hold only the value of this run, the difference and the agreement** (the stored value is printed only on screen). If they are mismatched, the status becomes comparison mismatch and the value must not be used.

**The number compared is matched to the expected number.** Expected number = the number of cell x column that the constants have for units in the plan of this run (the unit list of the pre-check above). If the compared number is smaller than expected (the secured table lacks the cell, or the value is empty), the status is "partially compared" even if the mismatch is 0 and is not counted as complete, and it goes into the sections not completed of the last cell. Constant cells of units that this run does not secure (the hospital level and the network when there is no hospital-level input, and bank sizes other than the full bank when B is all off) are not put into the expectation and are counted separately as outside the plan, with a reason. That section is counted as not completed in the last cell.


In [ ]:
# ================================================================
# PRAM input collection 4: input collection table and comparison with the stored csv constants
# ================================================================
R5P_TABLE, R5P_CHECK, R5P_COVER = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
R5P_CHECK_BAD = None
R5P_SHORT = None                                        # number of comparisons short of the expected count
if not DATA_OK:
    status("part 1 - input comparison", "not available", "main inputs absent")
else:
    R5P_TABLE = pd.DataFrame(R5P_LOG)
    emit_table("input collection table: units read from the stored results and units recomputed by this notebook with the same function (unit = outcome x target x bank size, hospital for the hospital level)", R5P_TABLE)
    save_csv(R5P_TABLE, "R5_input_provenance")
    R5P_CHECK_ROWS.clear()
    R5P_REF_SHOW.clear()
    R5P_EXPECT.clear()
    _med_cols = ["auroc", "ece", "slope", "brier", "nb@0.2"]
    _why_m = "bank size or feature set not in the plan of this run (if B is all off, only the full bank is secured)"
    _why_h = "there is no hospital-level input (eICU patient table) or all sections that use the hospital level are off"
    _why_n = "the network (E-2) is off or there is no hospital id"
    _pl_m = {_f: r5p_planned("matrix_" + _f) for _f in R5B_FS}
    _pl_r, _pl_t = r5p_planned("rescore"), r5p_planned("a2tail")
    # expected list: cell x column that the constants of the stored csv have for units in the plan of this run (counted regardless of whether the data exist)
    for _k, _ref in R5_REF.get("matrix", {}).items():
        _fs, _oc, _tg, _bk, _m = _k.split("|")
        r5p_expect("grid versus stored csv", _k, [_med_cols[i] for i in r5p_nn(_ref[:len(_med_cols)])], _pl_m.get(_fs) is not None and (_fs, _oc, _tg, _bk) in _pl_m[_fs], _why_m)
    for _k, _ref in R5_REF.get("rescore", {}).items():
        _oc, _tg, _bk, _var = _k.split("|")
        r5p_expect("rescoring versus stored csv", _k, [("d_auroc", "var_standard", "p_standard")[i] for i in r5p_nn(_ref[:3])], _pl_r is not None and (R5P_RESCORE_FS, _oc, _tg, _bk) in _pl_r, _why_m)
    for _k in R5_REF.get("table2_q", {}):
        r5p_expect("Table 2 q versus stored csv", _k, ["q_standard"], _pl_r is not None, _why_m)
    for _k, _ref in R5_REF.get("a2tail", {}).items():
        _fs, _oc, _tg, _bk, _m = _k.split("|")
        _cn = ["auroc median"] + (["d_auroc_base median", "var_auroc_base median"] if _m == "pram_b1" else []) + ["number of draw rows"]
        _ix = [0] + ([1, 2] if _m == "pram_b1" else []) + [3]
        r5p_expect("33-column eICU tail versus stored csv", _k, [c for c, i in zip(_cn, _ix) if _ref[i] is not None], _pl_t is not None and (_fs, _oc, _tg, _bk) in _pl_t, _why_m)
    for _k in R5_REF.get("a2tail_share", {}):
        _oc, _tg, _bk = _k.split("|")
        r5p_expect("33-column eICU interval fractions versus stored csv", _k, ["fraction of draws with the whole interval below 0", "fraction of draws with the whole interval above 0"], _pl_t is not None and (R5P_A2_FS, _oc, _tg, _bk) in _pl_t, _why_m)
    for _k, _ref in R5_REF.get("hospital", {}).items():
        _fs = _k.split("|")[0]
        r5p_expect("hospital-level distribution versus stored csv", _k, [R5_REF["hospital_stats"][i] for i in r5p_nn(_ref[:len(R5_REF["hospital_stats"])])], ("hospital_" + _fs) in R5P_ARTS, _why_h)
    for _k, _ref in R5_REF.get("network", {}).items():
        _fs = _k.split("|")[0]
        r5p_expect("network factor 1 distribution versus stored csv", _k, [R5_REF["hospital_stats"][i] for i in r5p_nn(_ref[:len(R5_REF["hospital_stats"])])], ("network_" + _fs) in R5P_ARTS, _why_n)
    # grid: median over draws per cell x method
    for _k, _ref in R5_REF.get("matrix", {}).items():
        _fs, _oc, _tg, _bk, _m = _k.split("|")
        if _fs not in R5P_MATRIX or not len(R5P_MATRIX[_fs]):
            continue
        _s = R5P_MATRIX[_fs]
        _s = _s[(_s["outcome"] == _oc) & (_s["target"] == _tg) & (_s["bank"].astype(str) == _bk) & (_s["method"] == _m)]
        if not len(_s):
            continue
        for _c, _rv in zip(_med_cols, _ref[:len(_med_cols)]):
            r5p_add("grid versus stored csv", _k, _c, _s[_c].median(), _rv)
    # rescoring: median over draws per cell x variant
    if len(R5P_RESCORE):
        for _k, _ref in R5_REF.get("rescore", {}).items():
            _oc, _tg, _bk, _var = _k.split("|")
            _s = R5P_RESCORE[(R5P_RESCORE["outcome"] == _oc) & (R5P_RESCORE["target"] == _tg) & (R5P_RESCORE["bank"].astype(str) == _bk) & (R5P_RESCORE["variant"] == _var)]
            if not len(_s):
                continue
            for _c, _rv in zip(("d_auroc", "var_standard", "p_standard"), _ref[:3]):
                r5p_add("rescoring versus stored csv", _k, _c, _s[_c].median(), _rv)
        _t2 = R5P_RESCORE[(R5P_RESCORE["variant"] == "main") & (R5P_RESCORE["bank"].astype(str) == "full")].groupby(["outcome", "target"], sort=False).agg(p=("p_standard", "median")).reset_index()
        if len(_t2) == len(R5_REF.get("table2_q", {})) and len(_t2):
            _t2["q"] = bh_adjust(_t2["p"].to_numpy())
            for _r in _t2.itertuples():
                r5p_add("Table 2 q versus stored csv", f"{_r.outcome}|{_r.target}", "q_standard", _r.q, R5_REF["table2_q"].get(f"{_r.outcome}|{_r.target}"))
    # 33-column eICU tail: 100-draw median and interval fractions per cell x method
    if len(R5P_A2T33):
        for _k, _ref in R5_REF.get("a2tail", {}).items():
            _fs, _oc, _tg, _bk, _m = _k.split("|")
            _s = R5P_A2T33[(R5P_A2T33["outcome"] == _oc) & (R5P_A2T33["target"] == _tg) & (R5P_A2T33["bank"].astype(str) == _bk) & (R5P_A2T33["method"] == _m)]
            if not len(_s):
                continue
            r5p_add("33-column eICU tail versus stored csv", _k, "auroc median", _s["auroc"].median(), _ref[0])
            if _m == "pram_b1":
                r5p_add("33-column eICU tail versus stored csv", _k, "d_auroc_base median", _s["d_auroc_base"].median(), _ref[1])
                r5p_add("33-column eICU tail versus stored csv", _k, "var_auroc_base median", _s["var_auroc_base"].median(), _ref[2])
            r5p_add("33-column eICU tail versus stored csv", _k, "number of draw rows", len(_s), _ref[3])
        _pp = R5P_A2T33[R5P_A2T33["method"] == "pram_b1"].copy()
        _pp["bank"] = _pp["bank"].astype(str)
        for _k, _ref in R5_REF.get("a2tail_share", {}).items():
            _oc, _tg, _bk = _k.split("|")
            _s = _pp[(_pp["outcome"] == _oc) & (_pp["target"] == _tg) & (_pp["bank"] == _bk)]
            if len(_s):
                _sd = np.sqrt(_s["var_auroc_base"].astype(float))
                r5p_add("33-column eICU interval fractions versus stored csv", _k, "fraction of draws with the whole interval below 0", float(((_s["d_auroc_base"] + R5_Z * _sd) < 0).mean()), _ref[0])
                r5p_add("33-column eICU interval fractions versus stored csv", _k, "fraction of draws with the whole interval above 0", float(((_s["d_auroc_base"] - R5_Z * _sd) > 0).mean()), _ref[1])
    # hospital level: distribution summary over all hospitals
    for _k, _ref in R5_REF.get("hospital", {}).items():
        _fs, _oc, _m, _c = _k.split("|")
        if _fs not in R5P_HOSP or not len(R5P_HOSP[_fs]):
            continue
        _s = R5P_HOSP[_fs]
        _s = _s[(_s["outcome"] == _oc) & (_s["method"] == _m)]
        if len(_s) and _c in _s.columns:
            for _nm, _nv, _rv in zip(R5_REF["hospital_stats"], r5p_dist(_s[_c]), _ref):
                r5p_add("hospital-level distribution versus stored csv", _k, _nm, _nv, _rv)
    # network factor 1: distribution summary over all hospitals at the mixing weight NET_ALPHA_FIXED
    for _k, _ref in R5_REF.get("network", {}).items():
        _fs, _oc, _c = _k.split("|")
        if _fs not in R5P_NET1 or not len(R5P_NET1[_fs]):
            continue
        _s = R5P_NET1[_fs]
        _s = _s[(_s["outcome"] == _oc) & np.isclose(_s["alpha"], NET_ALPHA_FIXED)]
        if len(_s):
            for _nm, _nv, _rv in zip(R5_REF["hospital_stats"], r5p_dist(_s[_c]), _ref):
                r5p_add("network factor 1 distribution versus stored csv", _k, _nm, _nv, _rv)
    R5P_CHECK = pd.DataFrame(R5P_CHECK_ROWS)
    R5P_CHECK_BAD = int((~R5P_CHECK["agreement"]).sum()) if len(R5P_CHECK) else 0
    # expected count versus compared count: if any constant cell x column in the plan was not compared, partially compared
    _have = set(zip(R5P_CHECK["item"], R5P_CHECK["cell"], R5P_CHECK["column"])) if len(R5P_CHECK) else set()
    _cov, _lost = [], []
    for _it, _e in R5P_EXPECT.items():
        _miss = [kc for kc in _e["cell columns"] if (_it, kc[0], kc[1]) not in _have]
        _lost += [(_it, kc[0], kc[1]) for kc in _miss]
        _cov.append({"item": _it, "expected count": _e["expected"], "compared count": _e["expected"] - len(_miss), "missing count": len(_miss), "outside-plan count": _e["outside plan"], "outside-plan reason": _e["reason"] if _e["outside plan"] else ""})
    R5P_COVER = pd.DataFrame(_cov)
    R5P_SHORT = len(_lost)
    _n_exp = int(R5P_COVER["expected count"].sum()) if len(R5P_COVER) else 0
    if len(R5P_CHECK):
        _sum = R5P_CHECK.groupby("item").agg(comparison=("agreement", "size"), agreement=("agreement", "sum"), max_diff=("difference", "max")).reset_index()
        emit_table("secured values versus aggregate constants of the stored csv (relative tolerance %s, csv has 6 significant digits)" % R5_CSV_RTOL, _sum, digits=6)
        save_csv(R5P_CHECK, "R5_input_check")
        emit(f"- input comparison {len(R5P_CHECK)} mismatches among **{R5P_CHECK_BAD}**. If 0, the secured values (read or recomputed) equal the stored csv.")
        if R5P_CHECK_BAD:
            print("  mismatched cells (the stored value is printed only on screen):")
            for _it, _ky, _cl, _nw, _rf in R5P_REF_SHOW[:40]:
                print(f"    {_it} | {_ky} | {_cl} | this run {_nw:.8g} | stored {_rf:.8g}")
            emit_table("mismatched cells (value of this run and difference)", R5P_CHECK[~R5P_CHECK["agreement"]].head(40), digits=8, max_rows=40)
    else:
        emit("- input comparison: there are no cells compared (the constants are empty or do not overlap with the secured cells).")
    emit_table("scope of the input comparison: the number actually matched against the stored csv constant cell x column (expected count) in the plan of this run. Outside-plan items are units this run does not secure, so they are not put in the expectation and a reason is written", R5P_COVER)
    emit(f"- input comparison scope: expected {_n_exp} of which compared {_n_exp - R5P_SHORT} , missing {R5P_SHORT} . " + ("there is nothing missing." if not R5P_SHORT else
         "**Compared only in part.** These are cells that could not be matched because the secured table lacks the cell (bank size, column name or value format) or the value is empty. The status is partially compared and is not counted as complete; values that could not be matched are not confirmed to equal the stored values."))
    if R5P_SHORT:
        print("  cells that could not be matched (first %d, item | cell | column):" % min(R5P_SHORT, 20))
        for _it, _ky, _cl in _lost[:20]:
            print(f"    {_it} | {_ky} | {_cl}")
    if len(R5P_COVER) and int(R5P_COVER["outside-plan count"].sum()):
        emit("- the count outside the plan belongs to units this run does not secure (for example when there is no hospital-level input). That section is counted separately in the sections not completed of the last cell.")
    _st_in = "comparison mismatch" if R5P_CHECK_BAD else ("partially compared" if (R5P_SHORT or _n_exp == 0) else "done")
    status("part 1 - input comparison", _st_in, f"expected {_n_exp} of which compared {_n_exp - R5P_SHORT} (missing {R5P_SHORT} ) | mismatches {R5P_CHECK_BAD}/{len(R5P_CHECK)} stays")


## (a) 33-column full-bank paired DeLong (q column of Table 3 Panel A and Whole pool of Supplementary S5 Panel B)

The same function (`rescore_unit`) as the rescoring section of notebook 01 is called on the 33-column context. There are 6 units (2 outcomes x 3 targets); one unit is one result pooling the 5-fold test predictions in the full bank. Only retrieval (`pram_b1`) is fitted. Value columns: `d_auroc` (retrieval minus frozen logistic) and `p_standard` (standard DeLong two-sided p).

**Verification**: the AUROC (retrieval and frozen) of this section must equal the value of the same cell of the grid (`matrix_F33`, read from the stored resume or recomputed by the input-collection section with the same function) to the ninth decimal. If it is different, the status becomes comparison mismatch and the value must not be used.


In [ ]:
# ================================================================
# PRAM (a) 33-column full-bank paired DeLong (resumable)
# ================================================================
R5W_RAW, R5W_TAB, R5W_CHECK, R5W_T42 = (pd.DataFrame() for _ in range(4))
R5W_BAD = None
if not DATA_OK:
    status("part 1 - (a) full bank", "not available", "main inputs absent")
else:
    _t_w0 = time.time()
    _Rw = Resumable(R5_RES_WHOLE)
    emit(f"### (a) {R5_WHOLE_FS} full bank , units {len(R5_WHOLE_UNITS)} (completed {sum(_Rw.unit_done(u) for u in R5_WHOLE_UNITS)} )")
    _w_all = {u: r5_whole_cells(len(CTX[(u[0], u[1], u[2])]["y"])) for u in R5_WHOLE_UNITS}
    _w_left = sum(w for u, w in _w_all.items() if not _Rw.unit_done(u))
    _w_done = 0.0
    for _u in R5_WHOLE_UNITS:
        if _Rw.unit_done(_u):
            print("  [SKIP]", _u)
            continue
        _t0 = time.time()
        for _r in rescore_unit(CTX[(_u[0], _u[1], _u[2])], _u[1], _u[2], _u[3], N_DRAWS_MAIN):
            _Rw.add_row({"fs": _u[0], **_r})
        _Rw.mark_unit(_u)
        _w_done += _w_all[_u]
        _eta = (time.time() - _t_w0) * (_w_left - _w_done) / max(_w_done, 1.0)
        print(f"  [DONE] (a) {_u[1]} / {_u[2]} ({time.time() - _t0:.0f} s) remaining time about {_eta / 60.0:.0f} min (estimated by cost ratio)", flush=True)
    R5W_RAW = _Rw.df().drop_duplicates(subset=["fs", "outcome", "target", "bank", "rep", "variant"], keep="last").reset_index(drop=True)
    R5W_TAB = r5_whole_frame(R5W_RAW[R5W_RAW["fs"] == R5_WHOLE_FS], len(FS_COLS[R5_WHOLE_FS]))
    # whether the AUROC of the same cell as the grid (matrix_F33) is equal (the grid values were read from the stored resume or recomputed by the input-collection section with the same function)
    _mx = R5P_MATRIX.get(R5_WHOLE_FS, pd.DataFrame())
    _mx = _mx[_mx["bank"].astype(str) == "full"] if len(_mx) else _mx
    _a = _mx[_mx["method"] == "pram_b1"].set_index(["outcome", "target", "rep"])["auroc"]
    _b = _mx[_mx["method"] == "b1_base"].set_index(["outcome", "target", "rep"])["auroc"]
    _m = R5W_RAW[(R5W_RAW["variant"] == "main") & (R5W_RAW["fs"] == R5_WHOLE_FS)].set_index(["outcome", "target", "rep"])
    _j = pd.DataFrame({"auroc_pram_new": _m["auroc_pram"], "auroc_pram_matrix": _a, "auroc_frozen_new": _m["auroc_frozen"], "auroc_frozen_matrix": _b}).dropna().reset_index()
    _j["abs_diff_pram"] = (_j["auroc_pram_new"] - _j["auroc_pram_matrix"]).abs()
    _j["abs_diff_frozen"] = (_j["auroc_frozen_new"] - _j["auroc_frozen_matrix"]).abs()
    R5W_CHECK = _j
    R5W_BAD = int(((_j["abs_diff_pram"] > R5_CHECK_TOL) | (_j["abs_diff_frozen"] > R5_CHECK_TOL)).sum()) + (len(_m) - len(_j))
    emit(f"**(a) versus grid (matrix_F33) comparison** , source of the grid values {r5p_src_text('matrix_F33')} , same cells {len(_j)}/{len(_m)} compared, mismatches {R5W_BAD} , maximum difference of retrieval AUROC {_j['abs_diff_pram'].max():.2e}, maximum difference of frozen AUROC {_j['abs_diff_frozen'].max():.2e} (tolerance {R5_CHECK_TOL})")
    # 42 columns: the same column made by the stored results (read from the stored resume or recomputed by the input-collection section with the same function)
    _r42 = R5P_RESCORE
    R5W_T42 = r5_whole_frame(_r42, len(FS_COLS["F42"]))
    # q: the main value = one set of six cells per feature set (same as 42 columns), reference = one set of twelve cells for the two sets
    R5W_TAB["q_within_set"] = bh_adjust(R5W_TAB["p"].to_numpy())
    R5W_T42["q_within_set"] = bh_adjust(R5W_T42["p"].to_numpy())
    _q12 = bh_adjust(np.concatenate([R5W_TAB["p"].to_numpy(), R5W_T42["p"].to_numpy()]))
    R5W_TAB["q_all12"], R5W_T42["q_all12"] = _q12[:len(R5W_TAB)], _q12[len(R5W_TAB):]
    for _t in (R5W_TAB, R5W_T42):
        _t["p_label"] = [r5_p_label(x) for x in _t["p"]]
        _t["q_label"] = [r5_p_label(x) for x in _t["q_within_set"]]
        _t["q12_label"] = [r5_p_label(x) for x in _t["q_all12"]]
    _cols = ["outcome", "target", "n_vars", "auroc_frozen", "d_auroc", "ci_lo", "ci_hi", "ni_at_margin", "p", "q_within_set", "q_all12", "q_label"]
    emit_table("(a) 33-column full bank , values to put in the q column of Table 3 Panel A and the Whole pool of Supplementary S5 Panel B (q_within_set is the main value: BH over the six 33-column cells, q_all12 is a reference: BH over the twelve cells of 33 and 42 columns)", R5W_TAB[_cols], digits=4)
    emit_table(f"42-column value of the same column (source of the rescoring value {r5p_src_text('rescore')}. Made again in the same way)", R5W_T42[_cols], digits=4)
    save_csv(R5W_RAW, "R5_whole_draws_F33")
    save_csv(R5W_TAB, "R5_whole_table_F33")
    save_csv(R5W_T42, "R5_whole_table_F42_reread")
    R5W_CHECK_OUT = R5W_CHECK.drop(columns=["auroc_pram_matrix", "auroc_frozen_matrix"])             # the value column of the stored results is not shown (only the value of this run and the difference)
    save_csv(R5W_CHECK_OUT, "R5_whole_check_vs_matrix_F33")
    R5_TIMES["(a) full bank"] = (R5_EXP_MIN.get("(a) full bank", float("nan")), time.time() - _t_w0)
    _done_w = len(R5W_TAB) == len(R5_WHOLE_UNITS)
    status("part 1 - (a) full bank", ("done" if (_done_w and R5W_BAD == 0) else ("comparison mismatch" if _done_w else "not available")),
           f"unit {len(R5_WHOLE_UNITS)} | AUROC mismatch with the grid {R5W_BAD}")


## (b) DeLong interval of the single-bank 100 draws (Interval below -0.01 of Supplementary S5 Panel B)

The same function (`run_tail_unit`) as the tail section of notebook 01 is called. A cell is outcome x target x bank size (500, 1,000, 2,000, 5,000), and the draws of one cell are 5 folds x 20 per fold = 100. One draw is one bank (stratified draw, key `tail|...`) and one test fold of the hospital that received that bank. The 95% interval of the difference is made with the paired DeLong (standard variance) of retrieval (`pram_b1`) and the frozen logistic. The strategy group is only retrieval and frozen (the prediction of `pram_b1` does not depend on the group composition, and the reproduction check below compares it with the value of the same cell of the stored results).

**Reference columns**: the 33-column eICU 100 draws made by notebook 01 (`a2_tail_F33`) and the 42-column 10 draws made by notebook 01 (bank other than full of `rescore_F42`, one draw = 5-fold pooling) are counted again in the same way (values read if present in this folder, otherwise recomputed by the input-collection section with the same function). The 100 draws (single bank) and the 10 draws (5-fold pooling) are different units, and the columns are separated in the table.


In [ ]:
# ================================================================
# PRAM (b) single-bank 100-draw interval (resumable)
# ================================================================
R5T_RAW, R5T_SUM, R5T_REF33, R5T_REF10 = (pd.DataFrame() for _ in range(4))
if not DATA_OK:
    status("part 1 - (b) single bank", "not available", "main inputs absent")
else:
    _t_t0 = time.time()
    _raw_parts = []
    for _fs, _name in R5_RES_TAIL.items():
        _Rt = Resumable(_name)
        _units = [u for u in R5_TAIL_UNITS if u[0] == _fs]
        emit(f"### (b) {_fs} single bank , units {len(_units)} (completed {sum(_Rt.unit_done(u) for u in _units)} ) | draws per cell {N_DRAWS_TAIL} | bank {TAIL_BANKS}")
        _pair_n = N_FOLDS * max(1, N_DRAWS_TAIL // N_FOLDS)
        _w_all = {u: _pair_n * (R5_T_CALL + r5_cells_for(len(CTX[(u[0], u[1], u[2])]["y"]) / N_FOLDS, u[3], 2) * R5_C_CELL) for u in _units}
        _w_left = sum(w for u, w in _w_all.items() if not _Rt.unit_done(u))
        _w_done, _t_loop = 0.0, time.time()
        for _u in _units:
            if _Rt.unit_done(_u):
                print("  [SKIP]", _u)
                continue
            _t0 = time.time()
            for _r in run_tail_unit(CTX[(_u[0], _u[1], _u[2])], _u[0], _u[1], _u[2], _u[3], R5_TAIL_GROUPS, R5_TAIL_KEEP, R5_TAIL_PAIRS):
                _Rt.add_row(_r)
            _Rt.mark_unit(_u)
            _w_done += _w_all[_u]
            _eta = (time.time() - _t_loop) * (_w_left - _w_done) / max(_w_done, 1e-9)
            print(f"  [DONE] (b) {_u[0]} {_u[1]} / {_u[2]} / bank={_u[3]} ({time.time() - _t0:.0f} s) remaining time about {_eta / 60.0:.0f} min (estimated by cost ratio)", flush=True)
        _part = _Rt.df()
        if len(_part):
            _raw_parts.append(_part.drop_duplicates(subset=["fs", "outcome", "target", "bank", "fold", "draw", "method"], keep="last"))
    R5T_RAW = pd.concat(_raw_parts, ignore_index=True) if _raw_parts else pd.DataFrame()
    if len(R5T_RAW):
        R5T_RAW["bank"] = R5T_RAW["bank"].astype(str)
        R5T_SUM = r5_sort_banks(r5_ci_summary(R5T_RAW[R5T_RAW["method"] == "pram_b1"], "d_auroc_base", "var_auroc_base", ["fs", "outcome", "target", "bank"]))
        R5T_SUM["unit"] = "single bank, 100 draws (new)"
    # 33-column eICU 100 draws of the stored results, 42-column 10 draws (5-fold pooling) of the stored results: values read or recomputed by the input-collection section with the same function
    _a2 = R5P_A2T33
    _a2 = _a2[_a2["method"] == "pram_b1"].drop_duplicates(subset=["fs", "outcome", "target", "bank", "fold", "draw", "method"], keep="last").copy()
    _a2["bank"] = _a2["bank"].astype(str)
    R5T_REF33 = r5_sort_banks(r5_ci_summary(_a2, "d_auroc_base", "var_auroc_base", ["fs", "outcome", "target", "bank"]))
    R5T_REF33["unit"] = "single bank, 100 draws (%s)" % r5p_src_text("a2tail", "stored results")
    _r10 = R5P_RESCORE
    _r10 = _r10[(_r10["variant"] == "main") & (_r10["bank"].astype(str) != "full")].copy()
    _r10["bank"] = _r10["bank"].astype(str)
    _r10["fs"] = "F42"
    R5T_REF10 = r5_sort_banks(r5_ci_summary(_r10, "d_auroc", "var_standard", ["fs", "outcome", "target", "bank"]))
    R5T_REF10["unit"] = "5-fold pooled, 10 draws (%s)" % r5p_src_text("rescore", "stored results")
    _show = ["fs", "outcome", "target", "bank", "n_draws", "d_median", "d_p05", "d_worst", "se_median", "share_d_below_m01", "k_ci_below_m01", "share_ci_below_m01", "cp_hi_below_m01",
             "k_ci_below0", "k_ci_above0"]
    emit_table("(b) single bank 100 draws , position of the DeLong 95% interval per cell (k_ci_below_m01 = number of draws whose whole interval is below -0.01, cp_hi = exact 95% upper bound of that fraction)", R5T_SUM[_show], digits=4, max_rows=80)
    emit_table(f"33-column eICU single bank 100 draws (value with the same definition as the stored results, source {r5p_src_text('a2tail', 'stored results')})", R5T_REF33[_show], digits=4, max_rows=20)
    emit_table(f"42-column 10 draws (one draw = one 5-fold pooled point, so not a single bank. Unit of S5 Panel C, source {r5p_src_text('rescore', 'stored results')})", R5T_REF10[_show], digits=4, max_rows=40)
    save_csv(R5T_RAW, "R5_tail_draws")
    save_csv(R5T_SUM, "R5_tail_ci_summary")
    save_csv(R5T_REF33, "R5_tail_ref_F33_eICU")
    save_csv(R5T_REF10, "R5_tail_ref_F42_10draws")
    R5_TIMES["(b) single bank"] = (R5_EXP_MIN.get("(b) 42-column single bank", 0.0) + R5_EXP_MIN.get("(b) 33-column MIMIC supplement", 0.0), time.time() - _t_t0)
    _n_cells = len(R5_TAIL_UNITS)
    _full = len(R5T_SUM) == _n_cells and bool((R5T_SUM["n_draws"] > 0).all())
    status("part 1 - (b) single bank", "done" if _full else "not available", f"cell {len(R5T_SUM)}/{_n_cells}")


## Reproduction check: does the same computation give the stored values again?

Part of what is already computed is rerun to see whether the same values come out. It runs whether or not the result folder of notebook 01 exists. This section does four things.

1. **2 full-bank units** (two small cohorts) are rerun with the same function and the same key and compared (a) line by line if the `rescore_F42` rows (variants main and key08) of notebook 01 in this folder were read, (b) with that value if the input-collection section computed it now (reproduction with the same function and the same key), and (c) in either case with the aggregate constants (constants in the cell).
2. **2 33-column eICU tail units** are rerun with the tail call of this notebook (strategy groups frozen and pram) and compared in the same three ways (line by line if the rows of `a2_tail_F33` of notebook 01 were read, with the value if computed now, with the 100-draw median of the aggregate constants).
3. **Table 2 q**: whether the 42-column q made by this notebook equals the aggregate constants and, if `RESCORE_table2_q.csv` exists in the working folder, also that file.
4. **33-column eICU fractions of whole interval below 0 and above 0**: compared with the aggregate constants in the same way (and with `A2_tail_summary.csv` if it exists in the working folder).

**The result md and csv hold only the value of this run, the difference and the agreement** (the stored value is printed only on screen). Line by line, agreement means within a relative error of 1e-9; against the aggregate constants, within a relative error of 1e-4 (the csv has 6 significant digits). If any one mismatches, the status becomes reproduction mismatch and the values of (a) and (b) must not be used.


In [ ]:
# ================================================================
# PRAM reproduction check (do the values equal the stored results)
# ================================================================
R5_REPRO_ROWS = []
R5_REPRO_SHOW = []                                       # (check, cell, column, this run, stored value) of the mismatched cells. Used only for screen output
R5_REPRO = pd.DataFrame()
R5_REPRO_BAD = None
R5_REPRO_SHORT = None                                    # number of comparisons with the aggregate constants of the stored csv short of the expected count (same definition as R5P_SHORT of the input comparison)


def r5_rep_cmp(item, cell, col, new, ref, exact):
    """matches one value. If exact, a line-by-line comparison with the same function and key (relative R5_CHECK_TOL), otherwise the aggregate constants of the stored csv (relative R5_CSV_RTOL, because there are 6 significant digits). The stored value is printed only on screen."""
    if ref is None:
        return
    new, ref = float(new), float(ref)
    ok = bool(np.isfinite(new) and np.isclose(new, ref, rtol=(R5_CHECK_TOL if exact else R5_CSV_RTOL), atol=1e-12))
    R5_REPRO_ROWS.append({"check": item, "cell": cell, "column": col, "this run": new, "difference": abs(new - ref) if np.isfinite(new) else float("nan"), "agreement": ok})
    if not ok:
        R5_REPRO_SHOW.append((item, cell, col, new, ref))


if not DATA_OK:
    status("part 1 - reproduction check", "not available", "main inputs absent")
else:
    _t_r0 = time.time()
    _Rrw, _Rrt = Resumable(R5_RES_REPRO[0]), Resumable(R5_RES_REPRO[1])
    # 1 full bank (same function and key as rescore_F42 of notebook 01)
    _src_r = R5P_SRC.get("rescore", {})
    for _fs, _oc, _tg in R5_REPRO_WHOLE:
        _u = (_fs, _oc, _tg, "full")
        if not _Rrw.unit_done(_u):
            for _r in rescore_unit(CTX[(_fs, _oc, _tg)], _oc, _tg, "full", N_DRAWS_MAIN):
                _Rrw.add_row({"fs": _fs, **_r})
            _Rrw.mark_unit(_u)
    _new_w = _Rrw.df().drop_duplicates(subset=["fs", "outcome", "target", "bank", "rep", "variant"], keep="last")
    for _fs, _oc, _tg in R5_REPRO_WHOLE:
        _from3 = _src_r.get((_fs, _oc, _tg, "full")) == "read from stored results"
        for _var in ("main", "key08"):
            _n = _new_w[(_new_w["outcome"] == _oc) & (_new_w["target"] == _tg) & (_new_w["bank"] == "full") & (_new_w["variant"] == _var)]
            _o = R5P_RESCORE[(R5P_RESCORE["outcome"] == _oc) & (R5P_RESCORE["target"] == _tg) & (R5P_RESCORE["bank"].astype(str) == "full") & (R5P_RESCORE["variant"] == _var)]
            _cell = f"{_oc} / {_tg} / full / {_var}"
            _have = len(_n) == 1 and len(_o) == 1
            for _col in ("d_auroc", "var_standard", "p_standard", "p_original"):
                r5_rep_cmp("full-bank 42-column recomputation versus stored resume (line by line)" if _from3 else "full-bank 42-column recomputation versus the value secured this time (same function and key)", _cell, _col,
                           float(_n[_col].iloc[0]) if _have else float("nan"), float(_o[_col].iloc[0]) if _have else 0.0, True)
            _rc = R5_REF.get("rescore", {}).get(f"{_oc}|{_tg}|full|{_var}")
            if _rc:
                for _col, _rv in zip(("d_auroc", "var_standard", "p_standard"), _rc[:3]):
                    r5_rep_cmp("full-bank 42-column recomputation versus aggregate constants of the stored csv", _cell, _col, float(_n[_col].iloc[0]) if len(_n) == 1 else float("nan"), _rv, False)
    # 2 33-column eICU tail (same call as a2_tail_F33 of notebook 01)
    _src_t = R5P_SRC.get("a2tail", {})
    for _fs, _oc, _tg, _bk in R5_REPRO_TAIL:
        _u = (_fs, _oc, _tg, _bk)
        if not _Rrt.unit_done(_u):
            for _r in run_tail_unit(CTX[(_fs, _oc, _tg)], _fs, _oc, _tg, _bk, R5_TAIL_GROUPS, R5_TAIL_KEEP, R5_TAIL_PAIRS):
                _Rrt.add_row(_r)
            _Rrt.mark_unit(_u)
    _new_t = _Rrt.df()
    _new_t["bank"] = _new_t["bank"].astype(str)
    for _fs, _oc, _tg, _bk in R5_REPRO_TAIL:
        _from2 = _src_t.get((_fs, _oc, _tg, _bk)) == "read from stored results"
        for _m_name, _cols in (("pram_b1", ("auroc", "d_auroc_base", "var_auroc_base")), ("b1_base", ("auroc",))):
            _n = _new_t[(_new_t["outcome"] == _oc) & (_new_t["target"] == _tg) & (_new_t["bank"] == str(_bk)) & (_new_t["method"] == _m_name)].drop_duplicates(["fold", "draw"], keep="last").set_index(["fold", "draw"]).sort_index()
            _o = R5P_A2T33[(R5P_A2T33["outcome"] == _oc) & (R5P_A2T33["target"] == _tg) & (R5P_A2T33["bank"].astype(str) == str(_bk)) & (R5P_A2T33["method"] == _m_name)].drop_duplicates(["fold", "draw"], keep="last").set_index(["fold", "draw"]).sort_index()
            _cell = f"{_oc} / {_tg} / bank {_bk} / {_m_name} ({len(_n)} draws)"
            _rc = R5_REF.get("a2tail", {}).get(f"{_fs}|{_oc}|{_tg}|{_bk}|{_m_name}")
            for _ci, _col in enumerate(_cols):
                _same_idx = len(_n) > 0 and _n.index.equals(_o.index)
                _dmax = float(np.nanmax(np.abs(_n[_col].to_numpy(float) - _o[_col].to_numpy(float)))) if _same_idx else float("nan")
                _okc = bool(_same_idx and np.allclose(_n[_col].to_numpy(float), _o[_col].to_numpy(float), rtol=R5_CHECK_TOL, atol=1e-12, equal_nan=True))
                _it = "33-column eICU tail recomputation versus stored resume (line by line)" if _from2 else "33-column eICU tail recomputation versus the value secured this time (same function and key, line by line)"
                R5_REPRO_ROWS.append({"check": _it, "cell": _cell, "column": _col, "this run": float(_n[_col].median()) if len(_n) else float("nan"), "difference": _dmax, "agreement": _okc})
                if not _okc:
                    R5_REPRO_SHOW.append((_it, _cell, _col, float(_n[_col].median()) if len(_n) else float("nan"), float(_o[_col].median()) if len(_o) else float("nan")))
                if _rc is not None and _rc[_ci] is not None:
                    r5_rep_cmp("33-column eICU tail recomputation versus aggregate constants of the stored csv (100-draw median)", _cell, _col, float(_n[_col].median()) if len(_n) else float("nan"), _rc[_ci], False)
    # 3 Table 2 q versus the stored csv
    _qn = R5W_T42.set_index(["outcome", "target"])["q_within_set"]
    _csv_q = CSV_DIR / "RESCORE_table2_q.csv"
    if _csv_q.is_file():
        for _r in pd.read_csv(_csv_q).itertuples():
            r5_rep_cmp("42-column q versus stored csv in the working folder (RESCORE_table2_q)", f"{_r.outcome} / {_r.target}", "q_standard", float(_qn.loc[(_r.outcome, _r.target)]), float(_r.q_standard), False)
    for _k, _rv in R5_REF.get("table2_q", {}).items():
        _oc, _tg = _k.split("|")
        if (_oc, _tg) in _qn.index:
            r5_rep_cmp("42-column q versus aggregate constants of the stored csv", f"{_oc} / {_tg}", "q_standard", float(_qn.loc[(_oc, _tg)]), _rv, False)
    # 4 33-column eICU interval fractions versus the stored csv
    _mine = R5T_REF33.set_index(["outcome", "target", "bank"])
    _csv_a2 = CSV_DIR / "A2_tail_summary.csv"
    if _csv_a2.is_file():
        _sa = pd.read_csv(_csv_a2)
        _sa = _sa[(_sa["fs"] == "F33") & (_sa["target"] == "eICU") & (_sa["method"] == "pram_b1") & (_sa["source"] == "new computation")].copy()
        _sa["bank"] = _sa["bank"].astype(str)
        for _r in _sa.itertuples():
            _k = (_r.outcome, _r.target, _r.bank)
            for _cn, _cm in (("draws_ci_wholly_below_zero", "share_ci_below0"), ("draws_ci_wholly_above_zero", "share_ci_above0")):
                r5_rep_cmp("33-column eICU interval fractions versus stored csv in the working folder (A2_tail_summary)", f"{_r.outcome} / {_r.target} / {_r.bank}", _cm, float(_mine.loc[_k, _cm]) if _k in _mine.index else float("nan"), float(getattr(_r, _cn)), False)
    for _k, _rv in R5_REF.get("a2tail_share", {}).items():
        _oc, _tg, _bk = _k.split("|")
        for _ci, _cm in enumerate(("share_ci_below0", "share_ci_above0")):
            r5_rep_cmp("33-column eICU interval fractions versus aggregate constants of the stored csv", f"{_oc} / {_tg} / {_bk}", _cm, float(_mine.loc[(_oc, _tg, _bk), _cm]) if (_oc, _tg, _bk) in _mine.index else float("nan"), _rv[_ci], False)
    R5_REPRO = pd.DataFrame(R5_REPRO_ROWS)
    R5_REPRO_BAD = int((~R5_REPRO["agreement"]).sum()) if len(R5_REPRO) else 0
    # expected count: cell x column to be matched with the aggregate constants of the stored csv (rescoring full-bank units x two variants x three columns, tail units x (three of pram_b1 + one of b1_base), Table 2 q). If the constants lack the cell, the comparison is silently skipped, so it is counted
    _exp_r = {"full-bank 42-column recomputation versus aggregate constants of the stored csv": 3 * 2 * len(R5_REPRO_WHOLE), "33-column eICU tail recomputation versus aggregate constants of the stored csv (100-draw median)": 4 * len(R5_REPRO_TAIL),
              "42-column q versus aggregate constants of the stored csv": len(R5_REF.get("table2_q", {}))}
    _got_r = R5_REPRO.groupby("check").size().to_dict() if len(R5_REPRO) else {}
    R5_REPRO_SHORT = int(sum(max(0, _e - int(_got_r.get(_it, 0))) for _it, _e in _exp_r.items()))
    if len(R5_REPRO):
        _sumr = R5_REPRO.groupby("check").agg(comparison=("agreement", "size"), agreement=("agreement", "sum"), max_diff=("difference", "max")).reset_index()
        emit_table("reproduction check summary (only the value of this run, the difference and agreement are shown. Line by line relative 1e-9, 1e-4 against the aggregate constants of the stored csv, csv has 6 significant digits)", _sumr, digits=3)
    emit(f"- reproduction check {len(R5_REPRO)} mismatches among **{R5_REPRO_BAD} **. " + ("if 0, the functions, keys and strategy groups used by (a) and (b) give the same values as the stored results." if R5_REPRO_BAD == 0 else "is not 0. The values of (a) and (b) must not be used. See the mismatched cells in the table below."))
    emit("- expected count to be matched with the aggregate constants of the stored csv (compared/expected) " + " · ".join(f"{_lb} {int(_got_r.get(_it, 0))}/{_e}" for (_it, _e), _lb in zip(_exp_r.items(), ("whole bank", "33-column tail", "Table 2 q"))) + f", missing count **{R5_REPRO_SHORT} **. " + ("" if not R5_REPRO_SHORT else "the constants lack the cell or the value is empty so it could not be matched. The status is partially compared and is not counted as complete."))
    if R5_REPRO_BAD:
        print("  mismatched cells (the stored value is printed only on screen):")
        for _it, _cl, _co, _nw, _rf in R5_REPRO_SHOW[:40]:
            print(f"    {_it} | {_cl} | {_co} | this run {_nw:.10g} | stored {_rf:.10g}")
        emit_table("mismatched cells (value of this run and difference)", R5_REPRO[~R5_REPRO["agreement"]], digits=6, max_rows=40)
    save_csv(R5_REPRO, "R5_repro_check")
    R5_TIMES["reproduction check"] = (R5_EXP_MIN.get("reproduction check", float("nan")), time.time() - _t_r0)
    status("part 1 - reproduction check", "reproduction mismatch" if R5_REPRO_BAD else ("partially compared" if R5_REPRO_SHORT else "done"), f"{len(R5_REPRO)} mismatches among {R5_REPRO_BAD} | constant comparison missing {R5_REPRO_SHORT}")


## Application of the pre-specified rules to (a) and (b)

This cell computes the conditions of the rules per cell, prints yes or no and the numbers, and makes the tables for Table 3 Panel A (q column) and S5 Panel B. The 42-column Single banks of 500 cells of Table 3 Panel A have the same definition as the reference values of the 100-draw tail results (fraction of single banks whose point estimate is below -0.01), but the draw keys differ, so the same patients are not drawn and the values need not be equal. **Only the one set of new 100 draws of this notebook is used for the tables.** The re-aggregation values of the stored tail results are put only in the comparison table of the S5 Panel A values section below, for the reproduction check. The two units are not mixed in the same column.


In [ ]:
# ================================================================
# PRAM application of the pre-specified rules and map of result files
# ================================================================
R5_RULES, R5_T3, R5_S5B = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
if not (DATA_OK and len(R5W_TAB) and len(R5T_SUM)):
    status("part 1 - report", "not available", "no result for (a) or (b)")
else:
    _rules = []
    # (a)
    _n_sig = int((R5W_TAB["q_within_set"] < SIG_LEVEL).sum())
    _loss = R5W_TAB[(R5W_TAB["q_within_set"] < SIG_LEVEL) & (R5W_TAB["d_auroc"] < 0)]
    _rules.append({"rule": "(a)1 number of cells with q (one set of six 33-column cells) < significance level", "value": f"{_n_sig}/{len(R5W_TAB)}", "note": "42-column same column: " + f"{int((R5W_T42['q_within_set'] < SIG_LEVEL).sum())}/{len(R5W_T42)}"})
    _rules.append({"rule": "(a)1 reference: number of 33-column cells with q (one set of twelve cells) < significance level", "value": f"{int((R5W_TAB['q_all12'] < SIG_LEVEL).sum())}/{len(R5W_TAB)}", "note": "if it splits from the main value, the main value is written in the q column and the split cells are noted"})
    _rules.append({"rule": "(a)4 cells with q < significance level and AUROC change < 0 (significant on the loss side)", "value": f"{len(_loss)}cell", "note": ("; ".join(f"{r.outcome} {r.target} d={r.d_auroc:+.4f} q={r.q_label}" for r in _loss.itertuples()) or "absent")})
    _rules.append({"rule": "(a) cells whose interval lower bound is above -margin (non-inferiority, the margin is a post hoc definition)", "value": f"{int(R5W_TAB['ni_at_margin'].sum())}/{len(R5W_TAB)}", "note": f"margin {NI_MARGIN_AUROC}"})
    # (b)
    _s42 = R5T_SUM[R5T_SUM["fs"].isin(R5_TAIL_FS)]
    _sx = R5T_SUM[R5T_SUM["fs"].isin(R5_TAIL_EXTRA_FS)]
    _k42 = int(_s42["k_ci_below_m01"].sum())
    _rules.append({"rule": f"(b)1 42 columns {len(_s42)}sum of the number of draws whose whole interval is below -margin in the cells", "value": f"{_k42} (cells {len(_s42)} x at most {int(_s42['n_draws'].max()) if len(_s42) else 0})",
                   "note": "if 0, no draw has a whole interval below -0.01 in any of the 24 cells. If 1 or more, the number of such draws is reported"})
    _rules.append({"rule": "(b)2 exact 95% upper bound for cells with k = 0 (Clopper-Pearson)", "value": ("; ".join(sorted({f"{x:.1%}" for x in _s42[_s42["k_ci_below_m01"] == 0]["cp_hi_below_m01"]})) or "not applicable"), "note": f"baseline {SAFE_UPPER_BOUND:.0%}"})
    _pos = _s42[_s42["k_ci_below_m01"] > 0]
    _rules.append({"rule": "(b)3 cells with k >= 1", "value": f"{len(_pos)}cell", "note": ("; ".join(f"{r.outcome} {r.target} {r.bank}: {int(r.k_ci_below_m01)}/{int(r.n_draws)}" for r in _pos.itertuples()) or "absent")})
    _rules.append({"rule": "(b) sum of the number of draws whose whole interval is below -margin in the 33-column MIMIC supplement cells", "value": f"{int(_sx['k_ci_below_m01'].sum())} (cells {len(_sx)} )", "note": "same rule"})
    _rules.append({"rule": "(b) number of 42-column cells that have draws whose whole interval is below 0 (one-sided 2.5% expected under the null)", "value": f"{int((_s42['k_ci_below0'] > 0).sum())}/{len(_s42)}", "note": "draws are not independent (5 folds shared), so the fraction is a descriptive value"})
    _rules.append({"rule": "(b) maximum over cells of the fraction of draws whose point estimate is below -margin (the point estimate, not the interval)", "value": f"{float(_s42['share_d_below_m01'].max()):.1%}" if len(_s42) else "not applicable", "note": "read together with k based on the interval (the interval is wide and may not have reached the threshold)"})
    R5_RULES = pd.DataFrame(_rules)
    emit_table("application of the pre-specified rules", R5_RULES, max_rows=30)
    # manuscript position table 1: Table 3 Panel A (twelve rows) q column and the single bank 500 column
    _all_sum = r5_sort_banks(pd.concat([R5T_SUM, R5T_REF33], ignore_index=True))
    _wq = pd.concat([R5W_T42.assign(fs="F42"), R5W_TAB.assign(fs=R5_WHOLE_FS)], ignore_index=True)[["fs", "outcome", "target", "n_vars", "d_auroc", "p_label", "q_label", "q12_label"]]
    _b500 = _all_sum[_all_sum["bank"] == str(TAIL_BANKS[0])][["fs", "outcome", "target", "n_draws", "share_d_below_m01", "k_ci_below_m01"]]
    R5_T3 = _wq.merge(_b500, on=["fs", "outcome", "target"], how="left")
    R5_T3["lost_more_than_m01_pct_of_draws"] = (100.0 * R5_T3["share_d_below_m01"]).round(0)
    emit_table(f"q column of the twelve rows of Table 3 Panel A (`q_label` is the main value, the 42-column is recomputed from the stored value) and Single banks of {TAIL_BANKS[0]} that lost more than {NI_MARGIN_AUROC} AUROC, % of 100 draws (by point estimate, 100-draw computation of this run. The 33-column MIMIC cells are computed by this notebook)", R5_T3, digits=4)
    # manuscript position table 2: Supplementary S5 Panel B
    _piv = _all_sum.assign(v=[f"{int(k)} of {int(n)}" for k, n in zip(_all_sum["k_ci_below_m01"], _all_sum["n_draws"])]).pivot_table(
        index=["fs", "outcome", "target"], columns="bank", values="v", aggfunc="first")
    _piv = _piv[[str(b) for b in TAIL_BANKS if str(b) in _piv.columns]].reset_index()
    _piv.columns = [str(c) for c in _piv.columns]
    R5_S5B = _piv.merge(_wq[["fs", "outcome", "target", "p_label", "q_label"]], on=["fs", "outcome", "target"], how="left")
    emit_table("values to put in Supplementary S5 Panel B: per bank size the number of draws whose whole interval is below -0.01 (of 100) and Whole pool P and q (the 33-column eICU tail reads the stored value, the 33-column Whole pool is (a), the 42-column Whole pool is the stored value recomputed)", R5_S5B, max_rows=30)
    save_csv(R5_RULES, "R5_rules_applied")
    save_csv(R5_T3, "R5_table3_PanelA_33_values")
    save_csv(R5_S5B, "R5_S5_PanelB_values")
    status("part 1 - report", "done", f"rule {len(R5_RULES)}line")


## Optional: row count of the TabPFN evaluation sample (no new computation)

TabPFN (A9) ran on a reduced sample of the eICU target. This cell only **reads** the A9 input collected by the input-collection section (the `a9_tabpfn` resume if it is in this folder, otherwise the aggregate constants of the stored csv) and checks the row count of the reduced sample. Order: (1) count by column name whether the resume rows have a row count column. (2) If not, count backwards the possible sizes from `obs_rate` (event fraction of the sample) of the rows with the `sample_bank` rule (events = round(size x cohort prevalence)). (3) Match with the size printed in the reduced line of the result md and print the values that agree. This cell counts only the **number** of candidates and the agreement with the log. The candidate N values themselves and the confirmation judgment (comparison of the frozen logistic metrics, all digits of the resume) are given by the back-calculation cell right below.


In [ ]:
# ================================================================
# PRAM optional: A9 TabPFN evaluation sample row count (read only)
# ================================================================
R5_A9_N = pd.DataFrame()
R5_A9_CAND = {}
if not (DATA_OK and len(R5P_A9)):
    status("part 1 - A9 sample", "no data", "neither the a9_tabpfn resume of the stored results nor the aggregate constants of the stored csv exist")
    emit("- A9 sample row count: could not be confirmed because the input (a9_tabpfn resume of the stored results or aggregate constants of the stored csv) is absent.")
else:
    _a9 = R5P_A9
    _csv9 = R5P_A9_SOURCE.startswith("stored csv")
    emit(f"- source of the A9 input: {R5P_A9_SOURCE}")
    _n_cols = [c for c in _a9.columns if str(c).lower() in ("n", "n_rows", "n_target", "n_sub", "n_sample", "n_test") or str(c).lower().startswith("n_")]
    emit(f"- A9 resume rows {len(_a9)} | columns {len(_a9.columns)} among them, the column holding the row count: {_n_cols if _n_cols else 'absent'}")
    _log_caps = []
    for _mdp in sorted(OUT_DIR.glob("*.md")):
        if "results" not in _mdp.name:
            continue
        for _ln_no, _ln in enumerate(_mdp.read_text(encoding="utf-8", errors="replace").splitlines(), 1):
            _mm = re.search(r"the target cohort is reduced to (\d+) rows", _ln)
            if _mm:
                _log_caps.append((_mdp.name, _ln_no, int(_mm.group(1))))
    emit(f"- the line saying reduced in the result md {len(_log_caps)} columns: " + (" · ".join(f"{n}:{k}line {c}rows" for n, k, c in _log_caps) if _log_caps else "absent"))
    _rows_n = []
    for (_oc, _tg, _bk), _s in _a9.groupby(["outcome", "target", "bank"], sort=False):
        _obs = sorted({round(float(x), 12) for x in _s["obs_rate"]})
        _yv = CTX[("F42", _oc, _tg)]["y"]
        _prev, _n_full = float(np.mean(_yv)), len(_yv)
        _sizes = np.arange(10, _n_full + 1)
        _npos = np.maximum(1, np.rint(_sizes * _prev)).astype(float)
        _cand = []
        for _o in _obs:
            if _csv9:                                    # the csv has 6 significant digits, so the fractions of the candidates are also rounded to the same digits before comparing
                _rate9 = np.array([float("%.6g" % x) for x in (_npos / _sizes)])
                _cand += [int(x) for x in _sizes[np.abs(_rate9 - _o) < 1e-12]]
            else:
                _cand += [int(x) for x in _sizes[np.abs(_npos / _sizes - _o) < 1e-11]]
        R5_A9_CAND[f"{_oc}|{_tg}|{_bk}"] = _cand
        _rows_n.append({"outcome": _oc, "target": _tg, "bank": str(_bk), "number of draws": int(_s["rep"].nunique()), "reduced marker": bool(_s["reduced"].any()) if "reduced" in _s.columns else None,
                        "all cohort rows": _n_full, "number of obs_rate values": len(_obs), "number of possible sample sizes": len(_cand),
                        "sizes consistent with the log": sorted({c for c in _cand if c in {x[2] for x in _log_caps}}), "same as all": bool(_n_full in _cand)})
    R5_A9_N = pd.DataFrame(_rows_n)
    _urows = []
    for (_oc, _tg), _s in R5_A9_N.groupby(["outcome", "target"], sort=False):
        _urows.append({"outcome": _oc, "target": _tg, "n_cells": len(_s), "reduced marker": bool(_s["reduced marker"].any()), "all cohort rows": int(_s["all cohort rows"].iloc[0]),
                       "number of possible sample sizes (maximum)": int(_s["number of possible sample sizes"].max()), "sizes consistent with the log": sorted({c for v in _s["sizes consistent with the log"] for c in v}),
                       "number of cells equal to all": int(_s["same as all"].sum())})
    _u = pd.DataFrame(_urows)
    emit_table("row count of the A9 evaluation sample (if the resume has no row count column, the candidates counted from obs_rate are matched with the reduced line of the result md. If exactly one is consistent with the log, that value is the sample size)", _u, max_rows=20)
    save_csv(R5_A9_N, "R5_A9_sample_size_candidates")
    status("part 1 - A9 sample", "done", f"column {len(_n_cols)} row counts and log lines {len(_log_caps)} columns")


## Back-calculation of the TabPFN evaluation sample row count (read only, no new computation; the `[N: ...]` of the Table 5 and Supplementary S10 notes)

The selection cell above counted only the **number** of possible sizes by `obs_rate`. This cell recomputes, for each candidate N, the AUROC, Brier score, ECE, slope, intercept and mean prediction of the frozen logistic row (`b5_base`, the same value in every cell regardless of bank size) with the same functions (`stratified_cap`, `full_metrics`) and compares them with the stored values. The stored values are, if the `a9_tabpfn` resume is in this folder, its **all-digit values** (the candidates are filtered by all digits of `obs_rate` with a tolerance of 1e-12); otherwise they are the aggregate constants of the stored csv (**6 significant digits**), and the fractions and metrics of the candidates are then also rounded to the same 6 digits before comparing (N is right if the rounded values agree). Which stored values were used is printed before the table. All candidate N values and the maximum differences are printed below and saved in `R5_A9_N_candidates.csv`.

**Three confirmation conditions (the cell judges by itself)**: (1) exactly one N matches, (2) the maximum absolute difference of the six metrics is 1e-6 or below, (3) the difference of the second candidate is 10 times or more that of the first. For each cell, confirmed or unconfirmed and the reason are printed and saved in `R5_A9_N_inverted.csv`. If confirmed, `N_hat` is the N of the Table 5 and S10 notes. If unconfirmed, no approximate value is used. TabPFN is not rerun in this notebook (the expected time is 12 hours or more).


In [ ]:
# ================================================================
# PRAM extra  back-calculation of the TabPFN evaluation sample row count (read only, no new computation)
# ================================================================
A9_INV_MIN_N = _ov("A9_INV_MIN_N", 1000)         # lower bound max(1000, ...) in the code
A9_INV_OUTCOMES = ["AKI_168h", "mortality_168h"]
A9_INV_TARGETS = ["eICU", "MIMIC-III"]
A9_INV_TOL_RATE = 1e-12
A9_INV_TOL_METRIC = 1.0e-6
A9_INV_RATIO = 10.0
A9_INV_METRICS = ["auroc", "brier", "ece", "slope", "intercept", "mean_pred"]
A9_INV_FULL_SCAN = False
A9_LOGGED_N = [22072, 16833, 15251, 15151, 15916]
A9_INV_NAMES = ["b5_base", "logistic_frozen"]


def a9_invert_sample_size(y, p_frozen, stored, key, full_scan=A9_INV_FULL_SCAN, round_sig=None):
    """y, p_frozen: the whole target cohort (y and pb1 of CTX). stored: one frozen-logistic row (dict) of the resume. key: a9cap|outcome|target.
    round_sig: if the stored value is a csv value rounded to some number of significant digits, that number (the fractions and metrics of the candidates are also rounded to the same digits before comparing). If None, it is a full-digit value of the resume.
    returns: (list of matching N, number of candidates, top 3 smallest differences [(N, maximum absolute difference)], all candidates scanned [(N, maximum absolute difference)])."""
    rd = (lambda v: float(("%." + str(int(round_sig)) + "g") % v)) if round_sig else (lambda v: float(v))
    y = np.asarray(y).astype(int)
    p_frozen = np.asarray(p_frozen, dtype=np.float32).astype(float)       # A9 collected test predictions in float32 to compute the metrics (run_matrix_unit): match with the same precision
    n_all, prev = len(y), float(y.mean())
    r = float(stored["obs_rate"])
    if full_scan:
        cand = list(range(A9_INV_MIN_N, n_all + 1))
    else:
        # the same rule as the code: number of events = max(1, int(round(N * cohort prevalence)))  (sample_bank). If N >= cohort size, all (stratified_cap)
        cand = [N for N in range(A9_INV_MIN_N, n_all + 1)
                if abs(rd(max(1, int(round(N * prev))) / N) - r) <= A9_INV_TOL_RATE or (N == n_all and abs(rd(prev) - r) <= A9_INV_TOL_RATE)]
    scored = []
    for N in cand:
        idx = stratified_cap(y, N if N < n_all else None, key)       # noqa: F821 (function of the notebook namespace)
        m = full_metrics(y[idx], p_frozen[idx])                         # noqa: F821
        d = max(abs(rd(m[c]) - float(stored[c])) for c in A9_INV_METRICS)
        scored.append((N, d))
    scored.sort(key=lambda t: t[1])
    hits = [N for N, d in scored if d <= A9_INV_TOL_METRIC]
    return hits, len(cand), scored[:3], scored


def a9_confirm(hits, top3):
    """judges the three confirmation conditions. Returns: (whether confirmed, reason string). The reason lists all unmet conditions."""
    why = []
    if len(hits) != 1:
        why.append("(1) %d matching N" % len(hits))
    best = float(top3[0][1]) if top3 else float("inf")
    if not best <= A9_INV_TOL_METRIC:
        why.append("(2) the minimum difference %.2e exceeds %.0e" % (best, A9_INV_TOL_METRIC))
    second = float(top3[1][1]) if len(top3) > 1 else float("inf")
    if not second >= A9_INV_RATIO * best:
        why.append("(3) the difference of the second candidate %.2e is less than %.0f times that of the first %.2e" % (second, best, A9_INV_RATIO))
    return (not why), ("all three conditions met" if not why else "; ".join(why))


def a9_invert_all(ctx, d, fs="F42", round_sig=None):
    """ctx: CTX of the notebook. d: the frozen-logistic row among the a9_tabpfn resume rows (DataFrame, columns: outcome, target, obs_rate, six metrics). round_sig: 6 if d is the aggregate constants of the stored csv (6 significant digits).
    returns: (judgment table per cell, table of all candidates). Printing as markdown is done by the caller (emit_table)."""
    d = d[d["method"].isin(A9_INV_NAMES)]
    rows, cand_rows = [], []
    for oc in A9_INV_OUTCOMES:
        for tg in A9_INV_TARGETS:
            s = d[(d["outcome"] == oc) & (d["target"] == tg)]
            if not len(s):
                rows.append({"outcome": oc, "target": tg, "status": "unconfirmed", "reason": "there is no frozen-logistic row in the resume", "N_hat": "not applicable"})
                continue
            if any(s[c].astype(float).nunique() > 1 for c in A9_INV_METRICS + ["obs_rate"]):
                rows.append({"outcome": oc, "target": tg, "status": "unconfirmed", "reason": "the metrics differ by bank size within the resume (two or more samples), so this back-calculation is not used", "N_hat": "not applicable"})
                continue
            row = s.iloc[0].to_dict()
            C0 = ctx[(fs, oc, tg)]
            hits, n_cand, top3, scored = a9_invert_sample_size(C0["y"], C0["pb1"], row, f"a9cap|{oc}|{tg}", round_sig=round_sig)
            ok, why = a9_confirm(hits, top3)
            for N, dd in scored:
                cand_rows.append({"outcome": oc, "target": tg, "N": int(N), "max_abs_diff": float(dd), "matches_within_tol": bool(dd <= A9_INV_TOL_METRIC), "in_logged_values": bool(N in A9_LOGGED_N)})
            rows.append({"outcome": oc, "target": tg, "cohort_n": len(C0["y"]), "stored_obs_rate": repr(float(row["obs_rate"])),
                         "candidates_after_prevalence_filter": n_cand,
                         "N_hat": hits[0] if len(hits) == 1 else (hits if hits else "no matching N"),
                         "status": "confirmed" if ok else "unconfirmed", "reason": why,
                         "uncapped": bool(len(hits) == 1 and hits[0] >= len(C0["y"])),
                         "best_diff": float(top3[0][1]) if top3 else float("nan"), "second_diff": float(top3[1][1]) if len(top3) > 1 else float("nan"),
                         "N_hat_in_logged": [N for N in (hits or []) if N in A9_LOGGED_N]})
    return pd.DataFrame(rows), pd.DataFrame(cand_rows)


X_A9_INV, X_A9_CAND = pd.DataFrame(), pd.DataFrame()
if not RUN_X_A9INV:
    status("part 1 - A9 N back-calculation", "off", "RUN_X_A9INV=False")
elif not (DATA_OK and len(R5P_A9)):
    status("part 1 - A9 N back-calculation", "no data", "neither the a9_tabpfn resume of the stored results nor the aggregate constants of the stored csv exist")
    emit("- A9 sample row count back-calculation: could not be done because the input (a9_tabpfn resume of the stored results or aggregate constants of the stored csv) is absent.")
else:
    _x_t0 = time.time()
    emit(f"- source of the stored values of the A9 back-calculation: {R5P_A9_SOURCE}")
    X_A9_INV, X_A9_CAND = a9_invert_all(CTX, R5P_A9, round_sig=(R5P_A9_ROUND if R5P_A9_SOURCE.startswith("stored csv") else None))
    emit_table("back-calculation of the A9 evaluation sample row count: comparison of the full-digit frozen-logistic metrics of the resume with the recomputation of candidate N. If status is confirmed, N_hat is the sample size (the three confirmation conditions are explained above)", X_A9_INV, digits=8, max_rows=20)
    emit_table("all candidate N values and the maximum absolute difference (candidates that passed the prevalence filter)", X_A9_CAND.sort_values(["outcome", "target", "max_abs_diff"]) if len(X_A9_CAND) else X_A9_CAND, digits=10, max_rows=60)
    save_csv(X_A9_INV, "R5_A9_N_inverted")
    save_csv(X_A9_CAND, "R5_A9_N_candidates")
    _x_nc = int((X_A9_INV["status"] == "confirmed").sum())
    emit(f"- confirmed {_x_nc}/{len(X_A9_INV)} cells | unconfirmed {len(X_A9_INV) - _x_nc}cell " + ("(the N position of the Table 5 and S10 notes is filled with N_hat)" if _x_nc == len(X_A9_INV) else "(unconfirmed cells do not use an approximate value)"))
    R5_TIMES["A9 N back-calculation"] = (float("nan"), time.time() - _x_t0)
    status("part 1 - A9 N back-calculation", "done", f"confirmed {_x_nc}/{len(X_A9_INV)}cell")


## Values of Supplementary S5 Panel A from the one set of new 100 draws (read only, no new computation)

The 42-column single-bank values (the Single banks of 500 column of Table 3 Panel A, the 42 columns of Supplementary S5 Panel A and the 42 columns of Supplementary S5 Panel B) are **taken from the one set of new 100 draws of section (b) above**. From the same draws (`R5_tail_draws`) this cell gives the columns of Supplementary S5 Panel A (median, 5th percentile, worst, fraction of draws that lost more than the margins 0.005, 0.01 and 0.02, exact 95% interval of the 0.01 margin fraction). The 33-column MIMIC rows use the same new draws; the 33-column eICU rows are read from the stored results.

The reference values of the 100-draw tail results (the rows of `A2_tail_summary.csv` that were read from the reference file; if that csv is not in this folder, the aggregate constants in the cell) are used only for the reproduction check. The rule is fixed: the new draws are the values of the tables, whatever the difference to the stored values. The result table shows only the new value and the difference.


In [ ]:
# ================================================================
# PRAM extra  Supplementary S5 Panel A values (new 100 draws) and comparison with the reference values of the 100-draw tail results (for the reproduction check)
# ================================================================
X_S5A, X_S5A_REF = pd.DataFrame(), pd.DataFrame()
X_S5A_MARGINS = (R5B_MARGIN_SETS["strict"][0], NI_MARGIN_AUROC, R5B_MARGIN_SETS["loose"][0])      # three AUROC margins (0.005, 0.01, 0.02)


def x_s5a_table(df, keys):
    """summary of the paired AUROC difference (retrieval minus frozen) per cell (keys): median | 5th percentile | worst | fraction of draws that lost more than each of the three margins | exact 95% interval of the 0.01 margin fraction."""
    rows = []
    for kk, s in df.groupby(keys, sort=False):
        kk = kk if isinstance(kk, tuple) else (kk,)
        d = s["d_auroc_base"].to_numpy(float)
        d = d[np.isfinite(d)]
        n = len(d)
        row = dict(zip(keys, kk))
        row["n_draws"] = int(n)
        if n:
            k = int((d < -NI_MARGIN_AUROC).sum())
            lo, hi = cp_interval(k, n)
            row.update({"d_median": float(np.median(d)), "d_p05": float(np.quantile(d, 0.05)), "d_worst": float(d.min()),
                        "cp_lo_lost_base": float(lo), "cp_hi_lost_base": float(hi)})
            for m in X_S5A_MARGINS:
                row["share_lost_more_than_%s" % m] = float((d < -m).mean())
        rows.append(row)
    return pd.DataFrame(rows)


if not RUN_X_S5A:
    status("part 1 - S5 Panel A", "off", "RUN_X_S5A=False")
elif not (DATA_OK and len(R5T_RAW)):
    status("part 1 - S5 Panel A", "not available", "(b) no single-bank result")
else:
    _x_t0 = time.time()
    _x_new = R5T_RAW[R5T_RAW["method"] == "pram_b1"][["fs", "outcome", "target", "bank", "fold", "draw", "d_auroc_base"]].copy()
    _x_new["unit"] = "single bank, 100 draws (new)"
    _x_old = R5P_A2T33
    _x_old = _x_old[_x_old["method"] == "pram_b1"].drop_duplicates(subset=["fs", "outcome", "target", "bank", "fold", "draw", "method"], keep="last")[["fs", "outcome", "target", "bank", "fold", "draw", "d_auroc_base"]].copy()
    _x_old["unit"] = "single bank, 100 draws (%s)" % r5p_src_text("a2tail", "stored results")
    _x_all = pd.concat([_x_new, _x_old], ignore_index=True)
    _x_all["bank"] = _x_all["bank"].astype(str)
    X_S5A = r5_sort_banks(x_s5a_table(_x_all, ["fs", "outcome", "target", "bank", "unit"]))
    emit_table("values to put in Supplementary S5 Panel A: from the same set of 100 draws (42 columns and 33-column MIMIC are the new draws, 33-column eICU is the stored value). share_lost_more_than_* = fraction of draws whose paired AUROC difference is below -margin, cp = exact 95% interval of the 0.01 margin fraction", X_S5A, digits=4, max_rows=80)
    save_csv(X_S5A, "R5_X_S5A_values")
    _x_cols54 = ["outcome", "target", "bank", "n_draws", "dAUROC_median", "dAUROC_p05", "dAUROC_worst"] + ["auroc_share_lost_more_than_%s" % m for m in X_S5A_MARGINS]
    _x_csv = CSV_DIR / "A2_tail_summary.csv"
    _x_a54 = pd.DataFrame()
    if _x_csv.is_file():
        _x_sa = pd.read_csv(_x_csv)
        _x_a54 = _x_sa[(_x_sa["fs"] == "F42") & _x_sa["source"].astype(str).str.startswith("read from the reference file") & (_x_sa["method"] == "pram_b1")][_x_cols54].copy()
        _x_a54_src = "A2_tail_summary of the stored csv in the working folder"
    elif R5_REF.get("a54"):
        _x_a54 = pd.DataFrame([dict(zip(_x_cols54, k.split("|") + v)) for k, v in R5_REF["a54"].items()])
        _x_a54_src = "stored csv aggregate constants"
    if len(_x_a54):
        _x_a54["bank"] = _x_a54["bank"].astype(str)
        _x_n42 = X_S5A[X_S5A["fs"] == "F42"]
        _x_ref = _x_n42.merge(_x_a54, on=["outcome", "target", "bank"], suffixes=("", "_a54"), how="left")
        for _m in X_S5A_MARGINS:
            _x_ref["diff_share_lost_more_than_%s" % _m] = _x_ref["share_lost_more_than_%s" % _m] - _x_ref["auroc_share_lost_more_than_%s" % _m].astype(float)
        _x_ref["diff_d_median"] = _x_ref["d_median"] - _x_ref["dAUROC_median"].astype(float)
        _x_drop = ["n_draws_a54", "dAUROC_median", "dAUROC_p05", "dAUROC_worst"] + ["auroc_share_lost_more_than_%s" % m for m in X_S5A_MARGINS]
        X_S5A_REF = _x_ref.drop(columns=[c for c in _x_drop if c in _x_ref.columns])           # the reference value column is not shown (only the value of this run and the difference)
        emit_table(f"for the reproduction check (not used for the tables): the difference in the same cell between the new 100-draw value and the reference values of the 100-draw tail results ({_x_a54_src}, rows read from the reference file). diff = value of this run minus reference value. The draw keys differ, so they may not be equal", X_S5A_REF, digits=4, max_rows=40)
        save_csv(X_S5A_REF, "R5_X_tail_ref_F42_a54")
    else:
        emit("- the reference values of the 100-draw tail results could not be read (neither the stored csv in the working folder nor the aggregate constants in the cell exist).")
    R5_TIMES["S5 Panel A values"] = (float("nan"), time.time() - _x_t0)
    status("part 1 - S5 Panel A", "done", f"cell {len(X_S5A)} columns")


## Stage B run and report functions

`run_matrix_unit` and `hospital_unit` of notebook 01 are called as they are. Only during the call, `bank_predictions` is replaced by the version with the B groups added, and afterwards it is restored (an internal check confirms that the result equals that of notebook 01 when called without the B groups). So the bank draw keys, the fold split and the metric computation are the same as in notebook 01.


In [ ]:
# ================================================================
# PRAM stage B  run and report functions
# ================================================================
R5B_GROUP = {"iw": "b_iw", "iwc": "b_iwc", "coral": "b_coral"}
R5B_X_METHODS = {"iw": ("iw_erm", "iw_platt", "iw_temp"), "iwc": ("iwc_erm", "iwc_platt", "iwc_temp"), "coral": ("coral", "coral_z")}
R5B_ROLE = {"iw": "comparison (version without rescaling)", "iwc": "main target (version with rescaling)", "coral": "main target"}


def r5b_run_matrix_families(kinds):
    """runs B-1 (iw version without rescaling and iwc version with rescaling) and B-2 (coral) per cell (resumable). For each unit, kinds are run in turn, and when the unit changes the shift discriminator memory is cleared, so the two versions in the same unit
    share one discriminator fit (the values are the same and it is computed once). Result names and rows are kept separately per version (resume compatible). Returns: version -> {feature set: (row table, information table)}."""
    out = {k: {} for k in kinds}
    for fs in R5B_FS:
        spec = FS_SPEC[fs]
        res = {k: (Resumable(r5b_rn(k, fs)), Resumable(r5b_rn(k + "info", fs))) for k in kinds}
        units = [(fs, oc, tg, bk) for oc in OUTCOMES for tg in spec["targets"] for bk in spec["banks"]]
        for kind in kinds:
            emit(f"### B {kind} {fs} , units {len(units)} (completed {sum(res[kind][0].unit_done(u) for u in units)} )")
        for u in units:
            r5b_disc_memo_clear()
            for kind in kinds:
                _R, _RI = res[kind]
                if _R.unit_done(u):
                    print("  [SKIP]", u)
                    continue
                t1 = time.time()
                rows, irows = r5b_with_extra(run_matrix_unit, CTX[(u[0], u[1], u[2])], u[0], u[1], u[2], u[3], N_DRAWS_MAIN, (R5B_GROUP[kind], "recal"), keep=R5B_KEEP[kind] + (R5B_CHECK_METHOD,))
                for r in rows:
                    _R.add_row(r)
                for r in irows:
                    _RI.add_row(r)
                _R.mark_unit(u)
                _RI.mark_unit(u)
                print(f"  [DONE] B {kind} {u[0]} {u[1]} / {u[2]} / bank={u[3]} ({time.time() - t1:.0f}s)", flush=True)
            r5b_disc_memo_clear()
        for kind in kinds:
            out[kind][fs] = (res[kind][0].df(), res[kind][1].df())
    return out


def r5b_run_hosp_families(kinds):
    """hospital level of B-1 and B-2 (the same within-hospital 3 folds and the same key as the stored results). For each hospital, kinds are run in turn and the discriminator memory is cleared when the hospital changes (the same reason as above). Version -> {feature set: hospital row table}."""
    out = {k: {} for k in kinds}
    for fs in [f for f in R5B_FS if FS_SPEC[f]["hospital"]]:
        res = {k: Resumable(r5b_rn(k + "h", fs)) for k in kinds}
        t0 = time.time()
        for oc in OUTCOMES:
            for i, h in enumerate(R5B_SITES[oc]):
                u = (fs, oc, int(h))
                r5b_disc_memo_clear()
                for kind in kinds:
                    _R = res[kind]
                    if _R.unit_done(u):
                        continue
                    rows = r5b_hosp_pair_rows(CTX[(fs, oc, "eICU")], oc, h, R5B_HID[oc], R5B_GROUP[kind], [m for m in R5B_KEEP[kind] if m != "b1_base"] + ["b1_base"])
                    for r in rows:
                        _R.add_row({"fs": fs, **r})
                    _R.mark_unit(u)
                    if i % 10 == 0:
                        print(f"  [hospital {kind} {fs}] {oc} {i + 1}/{len(R5B_SITES[oc])} ({(time.time() - t0) / 60:.1f} min)", flush=True)
                r5b_disc_memo_clear()
        for kind in kinds:
            out[kind][fs] = res[kind].df()
    return out


def r5b_cell_frame(xrows, comp, method):
    """cell table of one method (D+, DOM and NULL for all three margin sets). base is the main value, strict and loose are sensitivities."""
    tb = r5b_pair_table(xrows, comp, method)
    out = r5b_flags(tb, R5B_MARGIN_SETS["base"])
    for nm in ("strict", "loose"):
        f = r5b_flags(tb, R5B_MARGIN_SETS[nm])
        out["D_plus_" + nm], out["DOM_" + nm], out["NULL_" + nm] = f["D_plus"], f["DOM"], f["NULL"]
    out.insert(0, "method", method)
    return out


def r5b_hosp_shares(xh, ph, method, margins):
    """(fs, outcome) -> hospital DOM fraction. xh and ph are hospital rows with the fs column."""
    out, detail = {}, []
    for fs in sorted(set(xh["fs"])):
        d = r5b_hosp_dom(xh[xh["fs"] == fs], ph[ph["fs"] == fs], method, margins)
        for oc, (n, k, s) in d.items():
            out[(fs, oc)] = s
            detail.append({"method": method, "fs": fs, "outcome": oc, "n_hospitals": n, "n_dom": k, "share_dom": s})
    return out, detail


def r5b_sens_patterns(cells, hosp_x, hosp_p, method):
    """pattern assigned by the same rule for the three margin sets. cells is the result of r5b_cell_frame."""
    out = {}
    for nm, m in R5B_MARGIN_SETS.items():
        c = cells.copy()
        if nm != "base":
            c["D_plus"], c["DOM"], c["NULL"] = c["D_plus_" + nm], c["DOM_" + nm], c["NULL_" + nm]
        hs = r5b_hosp_shares(hosp_x, hosp_p, method, m)[0] if hosp_x is not None and len(hosp_x) else None
        out[nm] = r5b_pattern_P(c, hs)
    return out


def r5b_ar_cell_rows(ev, comp, env, form, which):
    """cell draw rows of AR (independent of the bank): attaches the constant metrics of X to the PRAM rows of the same cell (per draw). which: sel (gamma chosen by held-out environment AUROC) | nll (gamma chosen by held-out environment negative log likelihood).
    the two gammas are values chosen without target labels (the post hoc optimal gamma that maximizes the target AUROC per cell is not defined)."""
    e = ev[(ev["env"] == env) & (ev["form"] == form)]
    name = f"ar_{env}_{form}_{which}"
    col = "selected" if which == "sel" else "selected_nll"
    parts = []
    for (fs, oc, tg), s in e.groupby(["fs", "outcome", "target"], sort=False):
        m = s[s[col]].iloc[0]
        idx = comp[(comp["method"] == "pram_b1") & (comp["fs"] == fs) & (comp["outcome"] == oc) & (comp["target"] == tg)][R5B_KEYS]
        t = idx.copy()
        t["method"] = name
        for c in ("auroc", "ece", "slope", "intercept", R5B_NB20):
            t[c] = float(m[c])
        t["gamma"] = float(m["gamma"])
        parts.append(t)
    return (pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()), name


def r5b_ar_hosp_rows(ev_hosp, kind_name):
    """one row per hospital (gamma `sel` chosen by held-out environment AUROC and gamma `nll` chosen by negative log likelihood each). Columns: outcome, hospitalid, method, n_site, auroc, ece, nb@0.2."""
    out = []
    for (fs, oc, h, form), s in ev_hosp.groupby(["fs", "outcome", "hospitalid", "form"], sort=False):
        for which, m in (("sel", s[s["selected"]].iloc[0]), ("nll", s[s["selected_nll"]].iloc[0])):
            out.append({"fs": fs, "outcome": oc, "hospitalid": h, "method": f"{kind_name}_{form}_{which}", "n_site": int(m["n_site"]), "gamma": float(m["gamma"]), "auroc": float(m["auroc"]), "ece": float(m["ece"]), R5B_NB20: float(m[R5B_NB20]),
                        "slope": float(m["slope"]), "d_auroc_b1": float(m["d_auroc_b1"]), "var_b1": float(m["var_b1"]),
                        "d_auroc_pram": float(m["d_auroc_pram"]), "var_pram": float(m["var_pram"])})
    return pd.DataFrame(out)


## B-1. Importance-weighted retraining and recalibration (`RUN_B_IW`), two versions

The development data are reweighted with the unlabeled X of the bank to give a weighted logistic retraining (erm), a weighted Platt scaling (platt) and a weighted temperature scaling (temp). Control: the same recalibrations without weights, dev_platt and dev_temp. **The two versions run in the same cells.**

| Version | Weight | Method names | Role |
|---|---|---|---|
| version without rescaling | w = clip(q/(1-q), 0.05, 20) | iw_erm, iw_platt, iw_temp | comparison (the sample size ratio is not multiplied) |
| version with rescaling | w_c = clip(q/(1-q) x n_dev/n_bank, 0.05, 20) | iwc_erm, iwc_platt, iwc_temp | main result (estimate of the density ratio p_bank/p_dev, Bickel 2009 equations 12 to 17) |

q is the probability of the discriminator that separates development (0) from bank (1), n_dev is the number of development training records and n_bank is the number of bank rows put into the discriminator. The discriminator is the same for the two versions, and the truncation 0.05 to 20 is also the same. The version without rescaling lacks the sample size ratio, so the smaller the bank, the more the mean weight piles up near the lower bound 0.05; the fraction stuck at the lower bound is stored as cell information. **The version with rescaling also does not normalize the truncated weights to mean 1**: in cells where the mean weight is below 1, the same ridge penalty is relatively strong for the weighted loss. To show its size, the diagnostic variant `iwn_erm` with the correction weights divided to mean 1 is given in the same cells (diagnostic only, `R5_B1c_penalty_share`). The mean weight, the fractions stuck at the lower and upper bounds, ESS and the shift discrimination AUC of both versions are stored per fold in the cell information csv, and a per-cell comparison table of the two versions (`R5_B1c_weight_compare`) is printed below. **Recalibration is a monotone transform, so within one fold the AUROC equals that of frozen** (an internal check confirms). In the one result pooling the folds, the calibration differs per fold, so the AUROC differs slightly from frozen (the same holds for `platt_b1` of notebook 01). The cell AUROC change of platt and temp is therefore close to frozen, D+ hardly occurs, and DOM is decided by ECE and net benefit. This is a structural property and not a result.


In [ ]:
# ================================================================
# PRAM stage B-1  importance weighting, two versions: version without rescaling (iw) | version with rescaling (iwc) (resumable)
# ================================================================
R5B_IW, R5B_IWC, R5B_WCMP = {}, {}, pd.DataFrame()
if not DATA_OK:
    status("part 1 - stage B-1 importance weighting", "not available", "main inputs absent")
elif not RUN_B_IW:
    status("part 1 - stage B-1 importance weighting", "off", "RUN_B_IW=False")
else:
    _t0 = time.time()
    _fam_out = r5b_run_matrix_families(("iw", "iwc"))                   # the two versions in turn per unit: they share one shift discriminator fit
    R5B_IW, R5B_IWC = _fam_out["iw"], _fam_out["iwc"]
    for _fs, (_d, _i) in R5B_IW.items():
        save_csv(_d, f"R5_B1_iw_draws_{_fs}")
        save_csv(_i, f"R5_B1_iw_info_{_fs}")
    for _fs, (_d, _i) in R5B_IWC.items():
        save_csv(_d, f"R5_B1c_iwc_draws_{_fs}")
        save_csv(_i, f"R5_B1c_iwc_info_{_fs}")
    R5B_WCMP = r5b_weight_compare(pd.concat([i.assign(fs=fs) for fs, (d, i) in R5B_IW.items()], ignore_index=True), pd.concat([i.assign(fs=fs) for fs, (d, i) in R5B_IWC.items()], ignore_index=True))
    emit_table("weight comparison of the two versions (median over all folds of the cell information). n_dev/n_bank = the value multiplied by the version with rescaling, iw_ = version without rescaling, iwc_ = version with rescaling. Lower and upper = fraction of weights that reached the truncation 0.05 and 20, ESS = effective sample size, shift discrimination AUC is common to both versions",
               R5B_WCMP.rename(columns={"iwc_n_ratio": "n_dev/n_bank"}), digits=3, max_rows=60)
    save_csv(R5B_WCMP, "R5_B1c_weight_compare")
    R5_TIMES["B-1 importance weighting"] = (R5B_EXP_MIN.get("B-1 importance weighting", float("nan")), time.time() - _t0)
    status("part 1 - stage B-1 importance weighting", "done", f"fs {len(R5B_IW)} x two versions")


## B-2. CORAL (`RUN_B_CORAL`)

Sun et al. 2016, Algorithm 1 (lambda = 1). The development X is aligned to the covariance of the bank X and the logistic regression is refitted. The input is the whole `Xg` (including one-hot columns, with no per-column exception; however, a column whose standard deviation is 0 in the bank, that is a category absent in a small bank, is removed from the alignment in both versions and learned without a transform). **Two versions are given.** `coral` = the version without matching means (the scale of the development preprocessor as it is, applied to the test without a transform; the paper assumes both domains are normalized to mean 0, so this version does not satisfy the premise). `coral_z` = development standardized with the development mean and standard deviation, and bank and test features with the bank mean and standard deviation (both domains have mean 0; the version that fits the premise of the paper). Covariance alignment corrects only covariate shift, so it may not move under a large calibration departure (eICU death slope 0.41). If `coral_z`, which satisfies the premise, is NULL, it can be read as a limitation of the method; that `coral`, the version without matching means, is NULL may be because it is a weak form, so it is not read as a limitation of the method. Neither is evidence of an implementation error.


In [ ]:
# ================================================================
# PRAM stage B-2  CORAL (resumable)
# ================================================================
R5B_CORAL = {}
if not DATA_OK:
    status("part 1 - stage B-2 CORAL", "not available", "main inputs absent")
elif not RUN_B_CORAL:
    status("part 1 - stage B-2 CORAL", "off", "RUN_B_CORAL=False")
else:
    _t0 = time.time()
    R5B_CORAL = r5b_run_matrix_families(("coral",))["coral"]
    for _fs, (_d, _i) in R5B_CORAL.items():
        save_csv(_d, f"R5_B2_coral_draws_{_fs}")
        save_csv(_i, f"R5_B2_coral_info_{_fs}")
    R5_TIMES["B-2 CORAL"] = (R5B_EXP_MIN.get("B-2 CORAL", float("nan")), time.time() - _t0)
    status("part 1 - stage B-2 CORAL", "done", f"fs {len(R5B_CORAL)} columns")


## B-3. Anchor regression (`RUN_B_ANCHOR`)

**The development environments are only two (see the explanation above stage B).** Three versions are given. AR-2db (development environments: the MIMIC-IV training split and MIMIC-III). AR-period (the stay period of the MIMIC-IV training split). **There are two period rules**: in a copy with the unit-transfer adjustment (`anchor_year` and `intime` present), both strict (only stays that fall entirely inside one 3-year period; main rule, env `period`, main target) and majority (put in the period that holds most of its years; env `period_alt`, sensitivity that differs only in the environment period rule) are computed and given. Strict, which comes first in A1 of notebook 01, is the main rule. In a patient-level approximation copy the two rules are the same and only one is computed. AR-hosp (each eligible hospital other than the target hospital, plus the MIMIC-IV training split; hospital level only). There are two forms: linear probability with Platt scaling (`lpm`) and probit anchor classification (`probit`). The whole gamma grid {1, 2, 5, 10, 20, 50, 100} is given. gamma is chosen without target labels in folds that group the development environments by two criteria: the maximum held-out environment AUROC (`selected`, method name ending `_sel`) and the minimum held-out environment negative log likelihood (`selected_nll`, the classification criterion of the cited reference, column `cv_nll`, method name ending `_nll`). The post hoc optimal gamma that maximizes the target AUROC per cell is not defined (the target-cell values of the whole grid are in the csv). **Three differences from the cited reference**: (1) the gamma selection criterion (the reference uses the negative log likelihood; both criteria are given); (2) the form for a binary outcome (the reference fits a probit-link likelihood anchor by boosting and uses the linear anchor only for continuous outcomes; the probit version here uses a linear predictor with the same loss, and the linear probability version is a use not in the reference); (3) the general penalty (the reference adds an elastic-net penalty; this notebook has none). If a fit does not converge, only that form and cell stop without a value (status "not converged and stopped"). The results of other forms and cells remain in the csv and the computation continues.

The models of AR-2db and AR-period are independent of the bank, so they are fitted once per cell. The metrics of this model are the same value in all draws of the same cell.


In [ ]:
# ================================================================
# PRAM stage B-3  anchor regression fit and cell evaluation (resumable)
# ================================================================
R5B_AR_FIT, R5B_AR_EVAL = pd.DataFrame(), pd.DataFrame()
R5B_AR_OK = False
R5B_AR_STOPPED = []
if not DATA_OK:
    status("part 1 - stage B-3 anchor", "not available", "main inputs absent")
elif not RUN_B_ANCHOR:
    status("part 1 - stage B-3 anchor", "off", "RUN_B_ANCHOR=False")
else:
    _t0 = time.time()
    _Rf, _Re = Resumable(r5b_rn("arfit")), Resumable(r5b_rn("areval"))
    _envs_here = [e for e in R5B_AR_ENVS if e == "2db" or (R5B_PERIODS is not None and e in R5B_PERIOD_COL)]
    if "period" not in _envs_here:
        emit(f"- the development split period version (AR-period) cannot run: {R5B_PERIODS_NOTE}")
    for _fs in R5B_FS:
        for _oc in OUTCOMES:
            _c = CTX[(_fs, _oc, "eICU")]
            for _env in _envs_here:
                if _env == "2db":
                    _c3 = CTX[(_fs, _oc, "MIMIC-III")]
                    _X = np.vstack([_c["Xs_src"], _c3["Xg"]])
                    _y = np.concatenate([_c["ys_src"], _c3["y"]])
                    _e = np.r_[np.zeros(len(_c["ys_src"]), dtype=int), np.ones(len(_c3["y"]), dtype=int)]
                    _note = f"two development environments: MIMIC-IV training {len(_c['ys_src'])} · MIMIC-III {len(_c3['y'])}"
                else:
                    _d4 = DS[("MIMIC-IV", _oc)]
                    _keep, _e, _info_p = r5b_period_envs(_d4["stay_ids"][_d4["idx_train"]], R5B_PERIODS, R5B_PERIOD_COL[_env])
                    _X, _y = _c["Xs_src"][_keep], _c["ys_src"][_keep]
                    _note = f"development period environments {len(set(_e.tolist()))} columns {_info_p}"
                for _form in R5B_AR_METHODS:
                    _u = (_fs, _oc, _env, _form)
                    if _Rf.unit_done(_u):
                        print("  [SKIP]", _u)
                        continue
                    _t1 = time.time()
                    try:
                        _fit = r5b_anchor_fit_grid(_X, _y, _e, _form)
                    except R5BStop as _e_stop:
                        _msg = f"{_fs} {_oc} {_env} {_form}: {_e_stop}"
                        R5B_AR_STOPPED.append(_msg)
                        R5B_STOPS.append(_msg)
                        emit(f"**the anchor fit stopped (no value was produced for this form. Other forms and cells continue).** {_msg}")
                        continue
                    for _r in r5b_fit_rows(_fit, {"fs": _fs, "outcome": _oc, "env": _env, "form": _form, "n_rows": int(len(_y)), "env_n": json.dumps(_fit["env_n"])}):
                        _Rf.add_row(_r)
                    _Rf.mark_unit(_u)
                    print(f"  [DONE] B-3 fit {_fs} {_oc} {_env} {_form} ({time.time() - _t1:.0f}s) {_note} | selected gamma {_fit['selected']} (AUROC criterion) | {_fit['selected_nll']} (negative log likelihood criterion)", flush=True)
    R5B_AR_FIT = _Rf.df()
    # cell evaluation: rebuilds the probability from the finished fit rows and computes the metrics per target cohort (independent of the bank)
    for _fs in R5B_FS:
        for _oc in OUTCOMES:
            for _env in _envs_here:
                for _form in R5B_AR_METHODS:
                    _u = (_fs, _oc, _env, _form)
                    if not _Rf.unit_done(_u):
                        continue
                    if _Re.unit_done(_u):
                        print("  [SKIP] evaluation", _u)
                        continue
                    _sub = R5B_AR_FIT[(R5B_AR_FIT["fs"] == _fs) & (R5B_AR_FIT["outcome"] == _oc) & (R5B_AR_FIT["env"] == _env) & (R5B_AR_FIT["form"] == _form)]
                    for _r in _sub.to_dict("records"):
                        for _tg in R5B_TARGETS_BY_ENV[_env]:
                            _cc = CTX[(_fs, _oc, _tg)]
                            _p = r5b_rows_to_proba(_r, _cc["Xg"])
                            _Re.add_row({"fs": _fs, "outcome": _oc, "env": _env, "form": _form, "target": _tg, "gamma": _r["gamma"], "selected": bool(_r["selected"]), "selected_nll": bool(_r["selected_nll"]),
                                         "cv_auc": _r["cv_auc"], "cv_nll": _r["cv_nll"], **full_metrics(_cc["y"], _p)})
                    _Re.mark_unit(_u)
    R5B_AR_EVAL = _Re.df()
    R5B_AR_OK = len(R5B_AR_EVAL) > 0
    _stop = "; ".join(R5B_AR_STOPPED)[:200] if R5B_AR_STOPPED else None
    if len(R5B_AR_FIT):
        _show = R5B_AR_FIT.drop(columns=[c for c in ("beta_json",) if c in R5B_AR_FIT.columns])
        save_csv(_show, "R5_B3_ar_fit")
        emit_table("anchor fit: development-environment fold AUROC (cv_auc) over the whole gamma grid | fold negative log likelihood (cv_nll, the selection criterion of the cited reference) | two selection marks (selected = AUROC maximum, selected_nll = negative log likelihood minimum) | convergence (iterations and maximum gradient) | number of environments and stays per environment (env_n). The AUROC with two environments (2db) is by construction the same at all gamma and gamma = 1 is selected",
                   _show[["fs", "outcome", "env", "form", "gamma", "cv_auc", "cv_nll", "selected", "selected_nll", "n_env", "env_n", "n_cv_folds", "n_iter", "grad_max"]], max_rows=60)
    if len(R5B_AR_EVAL):
        save_csv(R5B_AR_EVAL, "R5_B3_ar_eval")
        emit_table("anchor cell evaluation (independent of the bank, whole target cohort): AUROC, ECE, slope and net benefit 0.20 over the whole gamma grid (including the two selection marks)", R5B_AR_EVAL[["fs", "outcome", "env", "form", "target", "gamma", "selected", "selected_nll", "auroc", "ece", "slope", R5B_NB20]], max_rows=60)
    R5_TIMES["B-3 anchor (fit and evaluation)"] = (R5B_EXP_MIN.get("B-3 anchor (fit and evaluation)", float("nan")), time.time() - _t0)
    _no_period = "period" not in _envs_here
    status("part 1 - stage B-3 anchor", "not converged | stopped" if _stop else (("partial (no AR-period)" if _no_period else "done") if R5B_AR_OK else "not available"),
           (_stop if _stop else f"fit {len(R5B_AR_FIT)} rows | evaluation {len(R5B_AR_EVAL)}rows" + (f" | no AR-period: {R5B_PERIODS_NOTE}" if _no_period else "")))


### B-3. eICU hospital anchor extension (AR-hosp, hospital level, reference upper bound)

Environments = the MIMIC-IV training split and each eligible hospital other than the target hospital. The labels of the target hospital are not used, but the labels of the other hospitals are used, so it is a method that needs alignment at training time (the same class as federated averaging and stacking). It does not enter the pattern of analysis P, and only the numbers are given. For each hospital the whole gamma grid is fitted and the main value is chosen by the environment-level 5 folds. There are about 60 environments.


In [ ]:
# ================================================================
# PRAM stage B-3  eICU hospital anchor extension (resumable)
# ================================================================
R5B_ARH = {}
if not DATA_OK or not RUN_B_ANCHOR or not RUN_B_ANCHOR_HOSP:
    status("part 1 - stage B-3 hospital extension", "off" if (DATA_OK and not (RUN_B_ANCHOR and RUN_B_ANCHOR_HOSP)) else "not available",
           ("RUN_B_ANCHOR=False" if not RUN_B_ANCHOR else "RUN_B_ANCHOR_HOSP=False") if DATA_OK else "main inputs absent")
elif not R5B_AR_OK or not R5B_HAVE_HOSP:
    status("part 1 - stage B-3 hospital extension", "no data" if not R5B_HAVE_HOSP else "not available", "no hospital id | no eligible hospital" if not R5B_HAVE_HOSP else "the anchor fit has not finished")
else:
    _t0 = time.time()
    _stops = []
    for _fs in [f for f in R5B_FS if FS_SPEC[f]["hospital"]]:
        _Rh = Resumable(r5b_rn("arh", _fs))
        _todo_any = any(not _Rh.unit_done((_fs, _oc, int(_h), _form)) for _oc in OUTCOMES for _h in R5B_SITES[_oc] for _form in R5B_AR_METHODS)
        if _todo_any and _fs not in R5B_HPRAM:                           # the hospital PRAM recomputation is done only once here and the report cell shares the same values
            R5B_HPRAM[_fs] = r5b_hosp_pram_oof(CTX, _fs, R5B_SITES, R5B_HID)
        _hp = R5B_HPRAM.get(_fs, {})
        for _oc in OUTCOMES:
            _ce = CTX[(_fs, _oc, "eICU")]
            for _i, _h in enumerate(R5B_SITES[_oc]):
                _idx = np.where(R5B_HID[_oc] == _h)[0]
                _env_data = None
                for _form in R5B_AR_METHODS:
                    _u = (_fs, _oc, int(_h), _form)
                    if _Rh.unit_done(_u):
                        continue
                    if _env_data is None:
                        _env_data = r5b_hosp_envs(_ce, _ce, R5B_HID[_oc], R5B_SITES[_oc], _h)
                    _X, _y, _e = _env_data
                    try:
                        _fit = r5b_anchor_fit_grid(_X, _y, _e, _form)
                    except R5BStop as _e_stop:
                        _msg = f"hospital anchor {_fs} {_oc} hospital {_i + 1} {_form}: {_e_stop}"
                        _stops.append(_msg)
                        R5B_STOPS.append(_msg)
                        emit(f"**the hospital anchor fit stopped (no value was produced for this hospital and form, other hospitals and forms continue).** {_msg}")
                        continue
                    for _g in _fit["gammas"]:
                        _fr = {"fs": _fs, "form": _form, "gamma": _g, "platt_a": _fit["platt"].get(_g, (0.0, 0.0))[0], "platt_b": _fit["platt"].get(_g, (0.0, 0.0))[1], "beta_json": json.dumps([float(v) for v in _fit["betas"][_g]])}
                        _p = r5b_rows_to_proba(_fr, _ce["Xg"][_idx])
                        _Rh.add_row({"fs": _fs, "outcome": _oc, "hospitalid": int(_h), "form": _form, "gamma": _g, "selected": bool(_g == _fit["selected"]), "selected_nll": bool(_g == _fit["selected_nll"]),
                                     "cv_auc": _fit["cv_auc"][_g], "cv_nll": _fit["cv_nll"][_g], "n_env": _fit["n_env"], "n_site": int(len(_idx)), "n_iter": _fit["info"][_g]["n_iter"], "grad_max": _fit["info"][_g]["grad_max"],
                                     **full_metrics(_ce["y"][_idx], _p),
                                     **dict(zip(("d_auroc_b1", "var_b1"), delong_delta_var(_ce["y"][_idx], _p, _ce["pb1"][_idx])[2:])),
                                     **dict(zip(("d_auroc_pram", "var_pram"), delong_delta_var(_ce["y"][_idx], _p, _hp[(_oc, int(_h))])[2:]))})
                    _Rh.mark_unit(_u)
                if _i % 5 == 0:
                    print(f"  [hospital anchor {_fs}] {_oc} {_i + 1}/{len(R5B_SITES[_oc])} ({(time.time() - _t0) / 60:.1f} min)", flush=True)
        R5B_ARH[_fs] = _Rh.df()
    for _fs, _d in R5B_ARH.items():
        if len(_d):
            _x = _d.copy()
            _x.insert(2, "site", _x["hospitalid"].map(SITE_CODE))
            save_csv(_x.drop(columns=["hospitalid"]), f"R5_B3_ar_hosp_{_fs}")
    R5_TIMES["B-3 hospital anchor extension"] = (R5B_EXP_MIN.get("B-3 hospital anchor extension", float("nan")), time.time() - _t0)
    status("part 1 - stage B-3 hospital extension", "not converged | stopped" if _stops else "done", ("; ".join(_stops)[:200] if _stops else f"fs {len(R5B_ARH)} columns"))


### B-3. PRAM on the frozen model trained by anchor (analysis F)

The AR probabilities of the two gammas (chosen by held-out environment AUROC and by negative log likelihood) are each taken as the frozen model (variant names `ar_<environment version>_<form>` and `ar_<environment version>_<form>_nll`), and PRAM is put on top (the same construction as PRAM on TabPFN in A9 of notebook 01). Several AR variants of the same bank share one retrieval (an internal check confirms that it gives the same mixing weight as `inner_cv_bank`). Columns: `pram_on_<variant>` and `<variant>` (that AR model alone).


In [ ]:
# ================================================================
# PRAM stage B-3  PRAM on anchor (resumable)
# ================================================================
R5B_FUS = {}
if not DATA_OK or not RUN_B_ANCHOR:
    status("part 1 - stage B-3 PRAM on anchor", "off" if (DATA_OK and not RUN_B_ANCHOR) else "not available", "RUN_B_ANCHOR=False" if DATA_OK else "main inputs absent")
elif not R5B_AR_OK:
    status("part 1 - stage B-3 PRAM on anchor", "not available", "the anchor fit has not finished")
else:
    _t0 = time.time()
    _sel = R5B_AR_FIT[R5B_AR_FIT["selected"]]
    _sel_nll = R5B_AR_FIT[R5B_AR_FIT["selected_nll"]]
    for _fs in R5B_FS:
        _spec = FS_SPEC[_fs]
        _R, _RI = Resumable(r5b_rn("fus", _fs)), Resumable(r5b_rn("fusinfo", _fs))
        _units = [(_fs, _oc, _tg, _bk) for _oc in OUTCOMES for _tg in _spec["targets"] for _bk in _spec["banks"]]
        emit(f"### B-3 PRAM on anchor {_fs} , units {len(_units)} (completed {sum(_R.unit_done(u) for u in _units)} )")
        for _u in _units:
            if _R.unit_done(_u):
                print("  [SKIP]", _u)
                continue
            _t1 = time.time()
            _cc = CTX[(_u[0], _u[1], _u[2])]
            _ap = {}
            for _sfx, _sl in (("", _sel), ("_nll", _sel_nll)):
                for _r in _sl[(_sl["fs"] == _u[0]) & (_sl["outcome"] == _u[1])].to_dict("records"):
                    if _u[2] in R5B_TARGETS_BY_ENV[_r["env"]]:
                        _ap[f"ar_{_r['env']}_{_r['form']}{_sfx}"] = r5b_rows_to_proba(_r, _cc["Xg"])
            R5B_CUR["ar_p"] = _ap
            try:
                _rows, _irows = r5b_with_extra(run_matrix_unit, _cc, _u[0], _u[1], _u[2], _u[3], N_DRAWS_MAIN, ("b_fus",), keep=tuple(["b1_base"] + list(_ap) + ["pram_on_" + k for k in _ap]))
            finally:
                R5B_CUR["ar_p"] = {}
            for _r in _rows:
                _R.add_row(_r)
            for _r in _irows:
                _RI.add_row(_r)
            _R.mark_unit(_u)
            _RI.mark_unit(_u)
            print(f"  [DONE] B-3 PRAM on {_u[0]} {_u[1]} / {_u[2]} / bank={_u[3]} ({time.time() - _t1:.0f}s)", flush=True)
        R5B_FUS[_fs] = _R.df()
        save_csv(R5B_FUS[_fs], f"R5_B3_fusion_draws_{_fs}")
    R5_TIMES["B-3 PRAM on anchor"] = (R5B_EXP_MIN.get("PRAM on B-3 anchor (analysis F)", float("nan")), time.time() - _t0)
    status("part 1 - stage B-3 PRAM on anchor", "done", f"fs {len(R5B_FUS)} columns")


## B hospital level. Importance weighting and CORAL (within-hospital 3 folds, the same key as the grid)

B-1 (two versions) and B-2 are run with the same split and the same bank key as the hospital level (`hospital_unit`) of notebook 01. The unlabeled covariates of the target hospital are only the training-fold X of that hospital. Per hospital, the AUROC, ECE and net benefit at 0.20 are compared with the PRAM row of the same hospital to give the hospital-level DOM fraction (the hospital condition of pattern P4).


In [ ]:
# ================================================================
# PRAM stage B  hospital level (resumable)
# ================================================================
R5B_HOSP = {}
if not DATA_OK:
    status("part 1 - stage B hospital level", "not available", "main inputs absent")
elif not R5B_HAVE_HOSP:
    status("part 1 - stage B hospital level", "no data", "no hospital id column or no eligible hospital")
elif not (RUN_B_IW or RUN_B_CORAL):
    status("part 1 - stage B hospital level", "off", "RUN_B_IW and RUN_B_CORAL are both False")
else:
    _t0 = time.time()
    _kinds_on = [_k for _k, _on in (("iw", RUN_B_IW), ("iwc", RUN_B_IW), ("coral", RUN_B_CORAL)) if _on]
    R5B_HOSP = r5b_run_hosp_families(_kinds_on)                         # versions in turn per hospital: they share one shift discriminator fit
    for _kind in _kinds_on:
        for _fs, _d in R5B_HOSP[_kind].items():
            if len(_d):
                _x = _d.copy()
                _x.insert(2, "site", _x["hospitalid"].map(SITE_CODE))
                save_csv(_x.drop(columns=["hospitalid"]), f"R5_B_hosp_{_kind}_{_fs}")
    R5_TIMES["B hospital level (importance weighting and CORAL)"] = (R5B_EXP_MIN.get("B hospital level (importance weighting and CORAL)", float("nan")), time.time() - _t0)      # the expectation is the value counted by the pre-check as hospital fit + hospital PRAM recomputation
    status("part 1 - stage B hospital level", "done", f"method {list(R5B_HOSP)}")


## B reproduction check: the same bank draws and the same splits?

The new methods of B use the same draw key as the grid (so the comparison target values are read from the stored resume or recomputed with the same function). This is checked directly with the methods that use the bank label (logistic recalibration and local retraining). Two small-cohort units are rerun with the same function, and it is checked whether the per-draw values of `platt_b1`, `local_lr` and `b1_base` equal the comparison targets (values read from the stored resume or recomputed by the input-collection section) and also the aggregate constants (median over draws). The hospital level is compared in the same way with two hospitals. Then it is counted whether the `platt_b1` made by B (fitted with the bank label, so the values differ if the bank draws and splits differ from the stored ones) equals the stored value in **all units and all draws** (AUROC, ECE, slope, Brier). `b1_base` is the whole frozen probability and is always the same regardless of split and draws (equal by construction), so only one line is kept separately. The evidence for the same draws is on the `platt_b1` side.


In [ ]:
# ================================================================
# PRAM stage B  reproduction check (do the values equal the stored grid and hospital level)
# ================================================================
R5B_REPRO, R5B_ALIGN = pd.DataFrame(), pd.DataFrame()
R5B_REPRO_BAD = None
R5B_REPRO_SHORT = None                                  # number of comparisons short of the expected count to be matched with the aggregate constants of the stored csv (same definition as R5P_SHORT of the input comparison)
if not (DATA_OK and (RUN_B_IW or RUN_B_CORAL or RUN_B_ANCHOR)):
    status("part 1 - stage B reproduction check", "off" if DATA_OK else "not available", "all three B methods are off" if DATA_OK else "main inputs absent")
else:
    _t0 = time.time()
    _Rr = Resumable(r5b_rn("repro"))
    _rb = tuple(dict.fromkeys([BANK_SIZES[0], BANK_SIZES[-1]]))
    _rep_methods = ("platt_b1", "local_lr", "b1_base")
    _rows = []
    for _fs, _oc, _tg in R5B_REPRO_UNITS:
        if _fs not in R5B_FS:
            continue
        for _bk in _rb:
            _u = (_fs, _oc, _tg, _bk)
            if not _Rr.unit_done(_u):
                _rr, _ = run_matrix_unit(CTX[(_fs, _oc, _tg)], _fs, _oc, _tg, _bk, N_DRAWS_MAIN, ("frozen", "recal", "local"), keep=_rep_methods)
                for _r in _rr:
                    _Rr.add_row(_r)
                _Rr.mark_unit(_u)
    _new = _Rr.df()
    _new["bank"] = _new["bank"].astype(str)
    _old = R5B_COMP[R5B_COMP["method"].isin(_rep_methods)]
    _j = _new.merge(_old, on=R5B_KEYS + ["method"], suffixes=("_new", "_old"), how="left")
    for _col in ("auroc", "ece", "slope", "brier"):
        _a, _b = _j[_col + "_new"].to_numpy(float), _j[_col + "_old"].to_numpy(float)
        for _m in _rep_methods:
            _s = (_j["method"] == _m).to_numpy()
            _okc = bool(np.isfinite(_b[_s]).all() and np.allclose(_a[_s], _b[_s], rtol=R5B_REPRO_TOL, atol=1e-12)) if _s.any() else False
            _rows.append({"check": "cell draws versus grid comparison target (%s)" % r5p_src_text("matrix_" + R5B_FS[0]), "method": _m, "column": _col, "rows compared": int(_s.sum()), "maximum difference": float(np.nanmax(np.abs(_a[_s] - _b[_s]))) if _s.any() and np.isfinite(_b[_s]).any() else float("nan"), "agreement": _okc})
    # against the aggregate constants of the stored csv: median over draws of the same cell and method. Expected count = cells x methods x four columns; if the constants lack the cell or the value is empty it is counted as missing
    _n_ref_exp, _n_ref_got, _ref_lost = 0, 0, []
    for _fs, _oc, _tg in R5B_REPRO_UNITS:
        if _fs not in R5B_FS:
            continue
        for _bk in _rb:
            for _m in _rep_methods:
                _ref = R5_REF.get("matrix", {}).get(f"{_fs}|{_oc}|{_tg}|{_bk}|{_m}")
                _s = _new[(_new["fs"] == _fs) & (_new["outcome"] == _oc) & (_new["target"] == _tg) & (_new["bank"] == str(_bk)) & (_new["method"] == _m)]
                _n_ref_exp += 4
                if _ref is None or not len(_s) or any(_v is None for _v in _ref[:4]):
                    _ref_lost.append(f"{_fs}|{_oc}|{_tg}|{_bk}|{_m}")
                    continue
                _n_ref_got += 4
                for _ci, _col in enumerate(("auroc", "ece", "slope", "brier")):
                    _vn, _vr = float(_s[_col].median()), float(_ref[_ci])
                    _rows.append({"check": "cell draws versus aggregate constants of the stored csv (median over draws)", "method": _m, "column": f"{_col} {_oc}/{_tg}/{_bk}", "rows compared": int(len(_s)), "maximum difference": abs(_vn - _vr),
                                  "agreement": bool(np.isclose(_vn, _vr, rtol=R5_CSV_RTOL, atol=1e-12))})
    # hospital level: two hospitals
    if R5B_HAVE_HOSP:
        _Rrh = Resumable(r5b_rn("reproh", R5B_FS[0]))
        for _oc in OUTCOMES:
            for _h in R5B_SITES[_oc][:2]:
                _u = (R5B_FS[0], _oc, int(_h))
                if not _Rrh.unit_done(_u):
                    _hr, _ = hospital_unit(CTX[(R5B_FS[0], _oc, "eICU")], _oc, _h, R5B_HID[_oc], ("frozen", "recal"))
                    for _r in (_hr or []):
                        if _r["method"] in ("platt_b1", "b1_base"):
                            _Rrh.add_row({"fs": R5B_FS[0], **_r})
                    _Rrh.mark_unit(_u)
        _hn = _Rrh.df()
        _ho = R5B_HOSP_COMP[R5B_HOSP_COMP["method"].isin(["platt_b1", "b1_base"]) & (R5B_HOSP_COMP["fs"] == R5B_FS[0])]
        _jh = _hn.merge(_ho, on=["outcome", "hospitalid", "method"], suffixes=("_new", "_old"), how="left")
        for _m in ("platt_b1", "b1_base"):
            _s = (_jh["method"] == _m).to_numpy()
            for _col in ("auroc", "ece"):
                _a, _b = _jh[_col + "_new"].to_numpy(float), _jh[_col + "_old"].to_numpy(float)
                _rows.append({"check": "hospital level versus hospital comparison target (%s)" % r5p_src_text("hospital_" + R5B_FS[0]), "method": _m, "column": _col, "rows compared": int(_s.sum()), "maximum difference": float(np.nanmax(np.abs(_a[_s] - _b[_s]))) if _s.any() else float("nan"),
                              "agreement": bool(_s.any() and np.isfinite(_b[_s]).all() and np.allclose(_a[_s], _b[_s], rtol=R5B_REPRO_TOL, atol=1e-12))})
    # compares platt_b1 made by B (fitted with the bank label, depends on draws and split) and b1_base (the whole frozen probability, equal by construction) with the stored results in all units and draws
    _al = []
    for _kind, _fam in (("iw", R5B_IW), ("iwc", R5B_IWC), ("coral", R5B_CORAL)):
        for _fs, (_d, _i) in _fam.items():
            for _mname, _label, _cols in (("platt_b1", "platt_b1 direct comparison (depends on draws and split)", ("auroc", "ece", "slope", "brier")), ("b1_base", "b1_base split check (equal by construction)", ("auroc",))):
                _x = _d[_d["method"] == _mname].copy()
                _x["bank"] = _x["bank"].astype(str)
                _jj = _x.merge(R5B_COMP[R5B_COMP["method"] == _mname], on=R5B_KEYS, suffixes=("_new", "_old"), how="left")
                _dmax = max((float(np.nanmax(np.abs(_jj[_c + "_new"] - _jj[_c + "_old"]))) if len(_jj) else float("nan")) for _c in _cols)
                _al.append({"method": _kind, "comparison": _label, "fs": _fs, "rows compared": int(len(_jj)), "rows not in the stored results": int(_jj["auroc_old"].isna().sum()), "maximum difference": _dmax,
                            "agreement": bool(len(_jj) > 0 and _jj["auroc_old"].notna().all() and all(np.allclose(_jj[_c + "_new"], _jj[_c + "_old"], rtol=R5B_REPRO_TOL, atol=1e-12) for _c in _cols))})
    for _kind, _fam in R5B_HOSP.items():
        for _fs, _d in _fam.items():
            _x = _d[_d["method"] == "pram_b1"]
            _jj = _x.merge(R5B_HOSP_COMP[R5B_HOSP_COMP["method"] == "pram_b1"], on=["fs", "outcome", "hospitalid"], suffixes=("_new", "_old"), how="left")
            _al.append({"method": f"{_kind} hospital-level pram_b1", "comparison": "hospital-level pram_b1 (same fold and same key, source of the comparison target %s)" % r5p_src_text("hospital_" + _fs), "fs": _fs, "rows compared": int(len(_jj)), "rows not in the stored results": int(_jj["auroc_old"].isna().sum()),
                        "maximum difference": float(np.nanmax(np.abs(_jj["auroc_new"] - _jj["auroc_old"]))) if len(_jj) else float("nan"),
                        "agreement": bool(len(_jj) > 0 and _jj["auroc_old"].notna().all() and np.allclose(_jj["auroc_new"], _jj["auroc_old"], rtol=R5B_REPRO_TOL, atol=1e-12))})
    R5B_ALIGN = pd.DataFrame(_al)
    R5B_REPRO = pd.DataFrame(_rows)
    R5B_REPRO_BAD = int((~R5B_REPRO["agreement"]).sum()) + (int((~R5B_ALIGN["agreement"]).sum()) if len(R5B_ALIGN) else 0)
    R5B_REPRO_SHORT = _n_ref_exp - _n_ref_got
    emit_table("B reproduction check summary (agreement = within relative error 1e-9)", R5B_REPRO.groupby(["check", "method"]).agg(comparison=("agreement", "size"), agreement=("agreement", "sum"), max_diff=("maximum difference", "max")).reset_index(), digits=3)
    if len(R5B_ALIGN):
        emit_table("comparison-target comparison of platt_b1 made by B (cell level, direct comparison of all draws because it depends on the bank label), b1_base (equal by construction) and pram_b1 (hospital level) (the comparison targets are values read from the stored resume or recomputed by the input-collection section)", R5B_ALIGN, digits=3)
    emit(f"- B reproduction check mismatches **{R5B_REPRO_BAD} **. If not 0, the values of B must not be used.")
    emit(f"- B reproduction check against the aggregate constants of the stored csv: expected {_n_ref_exp} of which {_n_ref_got} compared, missing count **{R5B_REPRO_SHORT} **." + ("" if not R5B_REPRO_SHORT else " the constants lack the cell or the value is empty so it could not be matched (cell: " + ", ".join(_ref_lost[:6]) + "). The status is partially compared and is not counted as complete."))
    save_csv(R5B_REPRO, "R5_B_repro_check")
    if len(R5B_ALIGN):
        save_csv(R5B_ALIGN, "R5_B_align_check")
    R5_TIMES["B reproduction check"] = (float("nan"), time.time() - _t0)
    status("part 1 - stage B reproduction check", "reproduction mismatch" if R5B_REPRO_BAD else ("partially compared" if R5B_REPRO_SHORT else "done"), f"mismatch {R5B_REPRO_BAD} | constant comparison missing {R5B_REPRO_SHORT}")


## Application of the stage B pre-specified rules

This cell counts D+, DOM and NULL per cell, assigns the cell patterns of analyses P and F (P0 to P4 and F0 to F4), and also gives the sensitivity where the margins are changed to strict and loose. The **B-1 implementation check** (two reproduction checks of the version without rescaling and two checks of the version with rescaling) is done first, and whether the two versions of B-1 split widely (number of split cells and pattern) is examined after the decision table.


In [ ]:
# ================================================================
# PRAM stage B  recheck of stored values | cell table | analysis P and analysis F patterns
# ================================================================
R5B_CELLS, R5B_PATTERNS, R5B_PATTERNS_F, R5B_HDOM, R5B_IMPL, R5B_HMETA, R5B_SPLIT, R5B_MSPLIT, R5B_HBOOT, R5B_HBOOT_CHK = (pd.DataFrame() for _ in range(10))
R5B_MAP_ROWS = []
R5B_NOHOSP_TEXT = "cannot be judged without the hospital-level condition (pattern P4 cannot be judged)"      # notation attached to a method whose pattern was printed with only cell-level conditions although there is no hospital-level result. The pattern definition is not changed
R5B_NOHOSP_LINE = ""                                    # line at the top of the summary md (filled only when there is no hospital-level result)
R5B_PERIOD_CMP = pd.DataFrame()                         # pattern comparison of the two AR-period rules (main versus sensitivity)


def r5b_headline_lines():
    out = [f"- B reproduction check mismatches {R5B_REPRO_BAD} | constant comparison missing {R5B_REPRO_SHORT} (all must be 0 for the values of B to be used)"]
    if len(R5B_HBOOT_CHK):
        out.append(f"- B hospital-level interval reproduction check (is the PRAM row on the same random stream as the main-text hospital-level interval): mismatches {int((~R5B_HBOOT_CHK['agreement']).sum())}/{len(R5B_HBOOT_CHK)}cells (0 means the same)")
    for r in (R5B_PATTERNS.itertuples() if len(R5B_PATTERNS) else []):
        out.append(f"- B analysis P {r.method}: pattern {r.pattern}(strict {r.pattern_strict} · loose {r.pattern_loose}) · eICU {r.n_eicu}cells of which DOM {r.n_dom_eicu} · D+ {r.n_dplus_eicu} | NULL fraction {r.null_share:.2f} | minimum hospital DOM fraction {r.hosp_min_share:.2f} · {'reference upper bound' if r.reference else r.role}"
                   + (f" · **{R5B_NOHOSP_TEXT}**" if str(r.p4_status).startswith("cannot be judged") else ""))
    for r in (R5B_PERIOD_CMP.itertuples() if len(R5B_PERIOD_CMP) else []):
        out.append(f"- B AR-period environment period rules, two kinds ({r.form} · {r.which}): rule {r.main_rule}(main) pattern {r.main_pattern} · eICU DOM {r.main_n_dom_eicu} · D+ {r.main_n_dplus_eicu} / rule {r.alt_rule}(sensitivity) pattern {r.alt_pattern} · eICU DOM {r.alt_n_dom_eicu} · D+ {r.alt_n_dplus_eicu} / {'the patterns are the same' if r.same_pattern else 'the patterns differ: both patterns are shown'}")
    for r in (R5B_SPLIT.itertuples() if len(R5B_SPLIT) else []):
        out.append(f"- B-1 comparison of the two versions {r.variant}: of the same {r.n_cells}cells, the cells whose decision (D+, DOM, NULL) differs {r.n_diff_cells}(eICU {r.n_diff_eicu}) | analysis P pattern version with rescaling {r.pattern_rescaled} versus version without rescaling {r.pattern_unscaled} · {'splits widely' if r.split else 'does not split'} , {r.reporting}")
    for r in (R5B_MSPLIT.itertuples() if len(R5B_MSPLIT) else []):
        out.append(f"- B three margin sets {r.method} ({r.margin_set}): cells whose marks differ from base {r.n_cells_changed}/{r.n_cells}(eICU {r.n_changed_eicu}) | pattern base {r.pattern_base} versus {r.margin_set} {r.pattern_this_set}")
    for r in (R5B_PATTERNS_F.itertuples() if len(R5B_PATTERNS_F) else []):
        out.append(f"- B analysis F {r.variant}: pattern {r.pattern} | G_X >= -margin {r.share_g_ok:.2f} | cells with decreased ECE {r.share_ece_down:.2f} | cells {r.n_cells}")
    for _ix, r in (R5B_IMPL.iterrows() if len(R5B_IMPL) else []):
        out.append(f"- B-1 implementation check [{r['target version']}] {r['item']}: {r['value']} ({'agreement' if r['agreement'] else 'outside'})")
    return out


if not DATA_OK:
    status("part 1 - stage B report", "not available", "main inputs absent")
elif not (R5B_IW or R5B_CORAL or R5B_AR_OK):
    status("part 1 - stage B report", "off" if not (RUN_B_IW or RUN_B_CORAL or RUN_B_ANCHOR) else "not available", "no B result")
else:
    _t_rep = time.time()
    _comp = R5B_COMP
    _cell_parts, _g_rows, _hd_rows = [], [], []
    _hosp_on = R5B_HAVE_HOSP and bool(R5B_HOSP or R5B_ARH or R5B_AR_OK)           # even if the hospital extension is off, the hospital-level DOM of the two-environment version and the period version is computed

    # (1) B-1 implementation check: two reproduction checks of the version without rescaling + two checks of the version with rescaling
    if R5B_IW and "F42" in R5B_IW:
        R5B_IMPL = r5b_iw_impl_table(R5B_IW["F42"][0], _comp[_comp["fs"] == "F42"], R5B_WCMP)
        emit_table("B-1 implementation check. Whether the version without rescaling reproduces the reference values and whether the version with rescaling has weights that reflect the sample size ratio. The handling when outside is in the last column", R5B_IMPL, max_rows=10)
        save_csv(R5B_IMPL, "R5_B_impl_check_B1")
        status("part 1 - stage B-1 implementation check", "done" if bool(R5B_IMPL["agreement"].all()) else "outside", "; ".join(f"{r['target version'][:6]} {r['item'][:14]}: {r['value'][:30]}" for _ix, r in R5B_IMPL.iterrows()))
    # (2) cell table and analysis P
    _classified = []
    for _kind, _fam in (("iw", R5B_IW), ("iwc", R5B_IWC), ("coral", R5B_CORAL)):
        if not _fam:
            continue
        _x = pd.concat([_d for _fs, (_d, _i) in _fam.items()], ignore_index=True)
        _x["bank"] = _x["bank"].astype(str)
        _hx = pd.concat([_d for _d in R5B_HOSP.get(_kind, {}).values() if len(_d)], ignore_index=True) if R5B_HOSP.get(_kind) else pd.DataFrame()
        for _m in R5B_X_METHODS[_kind]:
            _classified.append((_m, _x, _hx, False, R5B_ROLE[_kind]))
    if R5B_AR_OK:
        for _env in [e for e in R5B_AR_ENVS if (R5B_AR_EVAL["env"] == e).any()]:
            for _form in R5B_AR_METHODS:
                for _which in ("sel", "nll"):
                    _xr, _nm = r5b_ar_cell_rows(R5B_AR_EVAL, _comp, _env, _form, _which)
                    if len(_xr):
                        _hx = pd.DataFrame()
                        if _hosp_on:
                            _rows_h = []
                            for _fs in [f for f in R5B_FS if FS_SPEC[f]["hospital"]]:
                                if _fs not in R5B_HPRAM:
                                    R5B_HPRAM[_fs] = r5b_hosp_pram_oof(CTX, _fs, R5B_SITES, R5B_HID)
                                for _oc in OUTCOMES:
                                    _ev = R5B_AR_EVAL[(R5B_AR_EVAL["fs"] == _fs) & (R5B_AR_EVAL["outcome"] == _oc) & (R5B_AR_EVAL["env"] == _env) & (R5B_AR_EVAL["form"] == _form) & (R5B_AR_EVAL["target"] == "eICU")]
                                    if not len(_ev):
                                        continue
                                    _g = _ev[_ev["selected" if _which == "sel" else "selected_nll"]]["gamma"].iloc[0]
                                    _fr = R5B_AR_FIT[(R5B_AR_FIT["fs"] == _fs) & (R5B_AR_FIT["outcome"] == _oc) & (R5B_AR_FIT["env"] == _env) & (R5B_AR_FIT["form"] == _form) & (R5B_AR_FIT["gamma"] == _g)].iloc[0].to_dict()
                                    _ce = CTX[(_fs, _oc, "eICU")]
                                    _p = r5b_rows_to_proba(_fr, _ce["Xg"])
                                    for _h in R5B_SITES[_oc]:
                                        _ix = np.where(R5B_HID[_oc] == _h)[0]
                                        _mm = full_metrics(_ce["y"][_ix], _p[_ix])
                                        _dv = delong_delta_var(_ce["y"][_ix], _p[_ix], _ce["pb1"][_ix])
                                        _rows_h.append({"fs": _fs, "outcome": _oc, "hospitalid": int(_h), "method": _nm, "n_site": int(len(_ix)), "auroc": _mm["auroc"], "ece": _mm["ece"], R5B_NB20: _mm[R5B_NB20], "d_auroc_b1": _dv[2], "var_b1": _dv[3],
                                                        **dict(zip(("d_auroc_pram", "var_pram"), delong_delta_var(_ce["y"][_ix], _p[_ix], R5B_HPRAM[_fs][(_oc, int(_h))])[2:]))})
                            _hx = pd.DataFrame(_rows_h)
                        _classified.append((_nm, _xr, _hx, _env in ("2db",), "reference upper bound" if _env == "2db" else ("main target" if _env == "period" else f"sensitivity (only the AR-period environment period rule {R5B_PERIOD_COL.get('period_alt', '')} changed, main rule {R5B_PERIOD_COL.get('period', '')}. Not included in the overall pattern; both patterns are shown if they differ)")))
    for _m, _x, _hx, _ref, _role in _classified:
        _cf = r5b_cell_frame(_x, _comp, _m)
        _cell_parts.append(_cf)
        _hs, _det = (r5b_hosp_shares(_hx, R5B_HOSP_COMP, _m, R5B_MARGIN_SETS["base"]) if (_hosp_on and len(_hx)) else (None, []))
        _hd_rows += _det
        _sg = r5b_sens_patterns(_cf, _hx if (_hosp_on and len(_hx)) else None, R5B_HOSP_COMP, _m)
        _row = {"method": _m, "reference": _ref, "role": _role, **_sg["base"], "pattern_strict": _sg["strict"]["pattern"], "pattern_loose": _sg["loose"]["pattern"]}
        _g_rows.append(_row)
    if R5B_ARH and _hosp_on:
        _hh = r5b_ar_hosp_rows(pd.concat([_d for _d in R5B_ARH.values() if len(_d)], ignore_index=True), "ar_hosp")
        for _nm in sorted(set(_hh["method"])):
            _hs, _det = r5b_hosp_shares(_hh, R5B_HOSP_COMP, _nm, R5B_MARGIN_SETS["base"])
            _hd_rows += _det
            _g_rows.append({"method": _nm, "reference": True, "role": "reference upper bound", "pattern": "reference upper bound (hospital level only)", "n_cells": 0, "n_eicu": 0, "n_dom_eicu": 0, "n_dplus_eicu": 0, "null_share": float("nan"),
                            "hosp_min_share": min(_hs.values()) if _hs else float("nan"), "why": "AR-hosp gives the hospital level only", "pattern_strict": "", "pattern_loose": ""})
    if _cell_parts:
        R5B_CELLS = pd.concat(_cell_parts, ignore_index=True)
        R5B_PATTERNS = pd.DataFrame(_g_rows)
        _nh = (~R5B_PATTERNS["reference"].astype(bool)) & R5B_PATTERNS["hosp_min_share"].isna()
        R5B_PATTERNS["p4_status"] = np.where(_nh, "cannot be judged (no hospital level)", np.where(R5B_PATTERNS["reference"].astype(bool), "not applicable (reference upper bound)", "can be judged"))
        if _nh.any():
            R5B_PATTERNS.loc[_nh, "why"] = (R5B_PATTERNS.loc[_nh, "why"].fillna("").astype(str) + " " + R5B_NOHOSP_TEXT).str.strip()
            R5B_NOHOSP_LINE = (f"**analysis P pattern {int(_nh.sum())} lines are {R5B_NOHOSP_TEXT}.** This notebook did not find the hospital-level input (eICU patient table). The pattern was assigned with cell-level conditions only, and P4 is judged only with the hospital-level condition. "
                               "these sections are in the sections not completed of the last cell. If the notebook is opened in the folder where the stored results were done, the hospital level comes out.")
            emit("- " + R5B_NOHOSP_LINE)
        R5B_HDOM = pd.DataFrame(_hd_rows)
        R5B_SPLIT = r5b_split_table(R5B_CELLS, R5B_PATTERNS)
        _pc = []
        for _form_ in R5B_AR_METHODS:
            for _which_ in ("sel", "nll"):
                _gm = R5B_PATTERNS[R5B_PATTERNS["method"] == f"ar_period_{_form_}_{_which_}"]
                _ga = R5B_PATTERNS[R5B_PATTERNS["method"] == f"ar_period_alt_{_form_}_{_which_}"]
                if len(_gm) and len(_ga):
                    _pc.append({"form": _form_, "which": _which_, "main_rule": R5B_PERIOD_COL.get("period"), "main_pattern": _gm["pattern"].iloc[0], "alt_rule": R5B_PERIOD_COL.get("period_alt"), "alt_pattern": _ga["pattern"].iloc[0],
                                "same_pattern": bool(_gm["pattern"].iloc[0] == _ga["pattern"].iloc[0]), "main_n_dom_eicu": int(_gm["n_dom_eicu"].iloc[0]), "alt_n_dom_eicu": int(_ga["n_dom_eicu"].iloc[0]),
                                "main_n_dplus_eicu": int(_gm["n_dplus_eicu"].iloc[0]), "alt_n_dplus_eicu": int(_ga["n_dplus_eicu"].iloc[0])})
        R5B_PERIOD_CMP = pd.DataFrame(_pc)
        _show = ["method", "fs", "outcome", "target", "bank", "n_draws_vs_pram_b1", "dAUROC_vs_pram_b1", "dECE_vs_pram_b1", "dNB20_vs_pram_b1", "dAUROC_vs_b1_base", "dECE_vs_b1_base", "dNB20_vs_b1_base", "D_plus", "DOM", "NULL"]
        emit_table("cell table (method x feature set x outcome x target x bank size): median differences between the same draws paired, and D+, DOM, NULL (first 40 rows, full table in csv `R5_B_cells_all`)", R5B_CELLS[_show], digits=4, max_rows=40)
        emit_table("application of the analysis P pattern rule (for anchor, the gamma chosen without target labels by two criteria, sel and nll, each give a method; the post hoc optimum is not defined. reference = the reference upper bound is not included in the pattern)",
                   R5B_PATTERNS[["method", "role", "reference", "pattern", "p4_status", "pattern_strict", "pattern_loose", "n_eicu", "n_dom_eicu", "n_dplus_eicu", "null_share", "hosp_min_share", "why"]], digits=3, max_rows=40)
        if len(R5B_PERIOD_CMP):
            emit_table("pattern comparison of the two environment period rules (main rule versus sensitivity rule, differing in only the environment period rule). If the patterns differ, both are shown", R5B_PERIOD_CMP, digits=3, max_rows=10)
            save_csv(R5B_PERIOD_CMP, "R5_B_ar_period_rules")
        if len(R5B_SPLIT):
            emit_table("do the two B-1 versions (version without rescaling and version with rescaling) split widely: number of cells whose D+, DOM and NULL differ in the same cell (at least %d cells as the criterion) and the analysis P patterns of the three margin sets. If they split, both versions are shown" % R5B_SPLIT_CELLS, R5B_SPLIT, digits=3, max_rows=10)
            save_csv(R5B_SPLIT, "R5_B1c_split_check")
        R5B_MSPLIT = r5b_margin_split(R5B_CELLS, R5B_PATTERNS)
        emit_table("for each method, under the three margin sets (strict, base, loose), the number of cells whose marks (D+, DOM, NULL) differ from base and the pattern (base is the main decision and the number of split cells is given as it is)", R5B_MSPLIT, digits=3, max_rows=60)
        save_csv(R5B_MSPLIT, "R5_B_margin_split")
        if len(R5B_HDOM):
            emit_table("hospital-level DOM (number and fraction of hospitals with per-hospital delta AUROC >= -0.01, delta ECE < 0, delta NB20 >= +0.001)", R5B_HDOM, digits=3, max_rows=40)
        if _hosp_on:
            _hm_src = [R5B_HOSP_COMP[R5B_HOSP_COMP["method"].isin(["pram_b1", "platt_b1", "local_lr"])].drop(columns=[c for c in ("d_auroc_pram", "var_pram") if c in R5B_HOSP_COMP.columns])]
            for _m, _x, _hx, _ref, _role in _classified:
                if len(_hx) and "d_auroc_b1" in _hx.columns:
                    _hm_src.append(_hx[_hx["method"] == _m])
            if R5B_ARH:
                _hh2 = r5b_ar_hosp_rows(pd.concat([_d for _d in R5B_ARH.values() if len(_d)], ignore_index=True), "ar_hosp")
                _hm_src.append(_hh2)
            R5B_HMETA = r5b_hosp_meta(pd.concat(_hm_src, ignore_index=True))
            emit_table("hospital-level meta-analysis (delta AUROC versus the frozen logistic, per-hospital paired DeLong variance, REML + Hartung-Knapp truncated interval, the same function `meta_reml_hk` as A6 of notebook 01). PRAM, recalibration and local retraining use the stored values", R5B_HMETA, digits=4, max_rows=60)
            save_csv(R5B_HMETA, "R5_B_hosp_meta")
            _hb_order = {_k: list(dict.fromkeys(int(v) for v in _g["hospitalid"])) for _k, _g in R5B_HOSP_COMP[R5B_HOSP_COMP["method"] == "pram_b1"].groupby(["fs", "outcome"], sort=False)}      # order of the PRAM rows of the stored hospital resume (the hospital order read by the aggregation script)
            R5B_HBOOT = r5b_hosp_boot_table(pd.concat(_hm_src, ignore_index=True), _hb_order)
            emit_table("hospital-level interval (same kind as the main text: stay-weighted mean of the per-hospital delta AUROC + hospital bootstrap 95%% interval, B = %d | seed %d, the same random stream and hospital order as the aggregation script, shown in the column stream). vs_b1 = versus the frozen logistic, vs_pram = versus PRAM of the same hospitals. The REML + Hartung-Knapp table above is a reference" % (R5B_HOSP_BOOT_N, R5B_HOSP_BOOT_SEED), R5B_HBOOT, digits=4, max_rows=60)
            save_csv(R5B_HBOOT, "R5_B_hosp_boot")
            R5B_HBOOT_CHK = r5b_hosp_boot_check(R5B_HBOOT)
            if len(R5B_HBOOT_CHK):
                emit_table("hospital-level interval reproduction check: does the notebook PRAM (pram_b1) versus frozen interval equal the main-text hospital-level interval (output of the aggregation script) (tolerance %s). If equal, the intervals of the new methods of this notebook are on the same random stream as the manuscript interval" % R5B_HOSP_BOOT_REF_TOL, R5B_HBOOT_CHK, digits=6, max_rows=10)
                save_csv(R5B_HBOOT_CHK, "R5_B_hosp_boot_check")
                emit(f"- hospital-level interval reproduction check mismatches **{int((~R5B_HBOOT_CHK['agreement']).sum())} **/{len(R5B_HBOOT_CHK)} . If not 0, the PRAM row of R5_B_hosp_boot is on a random stream different from the manuscript value, so it is not used and the output of the aggregation script is used as it is.")
        save_csv(R5B_CELLS, "R5_B_cells_all")
        save_csv(R5B_PATTERNS, "R5_B_patterns_P")
        save_csv(R5B_HDOM, "R5_B_hosp_dom")
    # (3) gain over the control (recalibration without weights) (two versions)
    for _pfx, _fam_ in (("iw", R5B_IW), ("iwc", R5B_IWC)):
        if not _fam_:
            continue
        _d = pd.concat([_x for _x in (v[0] for v in _fam_.values())], ignore_index=True)
        _d["bank"] = _d["bank"].astype(str)
        _c1 = r5b_pair_table(_d, _d, _pfx + "_platt", ("dev_platt",))
        _c2 = r5b_pair_table(_d, _d, _pfx + "_temp", ("dev_temp",))
        _ctl = _c1[["fs", "outcome", "target", "bank", "dAUROC_vs_dev_platt", "dECE_vs_dev_platt"]].merge(_c2[["fs", "outcome", "target", "bank", "dAUROC_vs_dev_temp", "dECE_vs_dev_temp"]], on=["fs", "outcome", "target", "bank"])
        emit_table("what the weights added: %s (paired median versus the same recalibration without weights)" % ("version without rescaling" if _pfx == "iw" else "version with rescaling"), _ctl, digits=4, max_rows=30)
        if _pfx == "iw":
            save_csv(_ctl, "R5_B1_iw_vs_unweighted_recal")
        else:
            save_csv(_ctl, "R5_B1c_iwc_vs_unweighted_recal")
    # (3-2) diagnosis: retraining with the correction weights divided to mean 1 (iwn_erm). The difference in penalty size arising because the mean weight is not 1. Diagnostic only
    if R5B_IWC:
        _d = pd.concat([_x for _x in (v[0] for v in R5B_IWC.values())], ignore_index=True)
        _d["bank"] = _d["bank"].astype(str)
        _pn = r5b_pair_table(_d, _d, "iwn_erm", ("iwc_erm", "b1_base"))
        _pc = r5b_pair_table(_d, _d, "iwc_erm", ("b1_base",))
        _pen = _pn[["fs", "outcome", "target", "bank", "dAUROC_vs_iwc_erm", "dECE_vs_iwc_erm", "dAUROC_vs_b1_base", "dECE_vs_b1_base"]].rename(columns={
            "dAUROC_vs_iwc_erm": "iwn_minus_iwc_dAUROC", "dECE_vs_iwc_erm": "iwn_minus_iwc_dECE", "dAUROC_vs_b1_base": "iwn_minus_frozen_dAUROC", "dECE_vs_b1_base": "iwn_minus_frozen_dECE"}).merge(
            _pc[["fs", "outcome", "target", "bank", "dAUROC_vs_b1_base"]].rename(columns={"dAUROC_vs_b1_base": "iwc_minus_frozen_dAUROC"}), on=["fs", "outcome", "target", "bank"])
        emit_table("diagnosis: retraining with the correction weights divided to mean 1 (iwn_erm). iwn - iwc = the effect of the penalty size arising because the weights are not normalized, iwn - frozen = the effect of importance weighting remaining after normalization (paired median, diagnostic only)", _pen, digits=4, max_rows=60)
        save_csv(_pen, "R5_B1c_penalty_share")
    # (4) analysis F: PRAM on anchor
    if R5B_FUS:
        _fx = pd.concat([_d for _d in R5B_FUS.values() if len(_d)], ignore_index=True)
        _fx["bank"] = _fx["bank"].astype(str)
        _gstd = r5b_pair_table(_comp, _comp, "pram_b1", ("b1_base",))
        _fr = []
        _gf_cells = []
        for _v in sorted({m[len("pram_on_"):] for m in set(_fx["method"]) if m.startswith("pram_on_")}):
            _tb = r5b_pair_table(_fx, _fx, "pram_on_" + _v, (_v,)).rename(columns={f"dAUROC_vs_{_v}": "dAUROC", f"dECE_vs_{_v}": "dECE", f"dNB20_vs_{_v}": "dNB20"})
            _tb = _tb.merge(_gstd[["fs", "outcome", "target", "bank", "dAUROC_vs_b1_base"]].rename(columns={"dAUROC_vs_b1_base": "G_std"}), on=["fs", "outcome", "target", "bank"], how="left")
            _gr = r5b_pattern_F(_tb, _tb["G_std"].to_numpy())
            _fr.append({"variant": _v, **_gr})
            _gf_cells.append(_tb.assign(variant=_v))
        R5B_PATTERNS_F = pd.DataFrame(_fr)
        emit_table("application of the analysis F pattern rule (PRAM on the frozen model trained by anchor versus that model alone, main gamma)", R5B_PATTERNS_F[["variant", "pattern", "n_cells", "share_g_ok", "share_ece_down", "share_f4", "share_null_like", "half_of_std_eicu", "why"]], digits=3)
        save_csv(R5B_PATTERNS_F, "R5_B_patterns_F")
        save_csv(pd.concat(_gf_cells, ignore_index=True), "R5_B3_fusion_cells")
    R5B_MAP_ROWS = [
        ["domain adaptation and anchor regression rows of Table 2 and Supplementary S16", "R5_B_cells_all · R5_B_patterns_P · R5_B_patterns_F · R5_B_margin_split", "values of the methods. If the patterns split under the three margin sets, the number of split cells is given with the base pattern"],
        ["B-1 importance weighting (the cell version of Supplementary S8 Panels G and H, two versions: without rescaling and with rescaling)", "R5_B1_iw_draws_* · R5_B1c_iwc_draws_* · R5_B1c_weight_compare · R5_B1c_split_check · R5_B1c_penalty_share · R5_B1_iw_vs_unweighted_recal · R5_B1c_iwc_vs_unweighted_recal · R5_B_impl_check_B1", "the version with rescaling is the main result and the version without rescaling is for reproduction. The version with rescaling also does not normalize the weights to mean 1 (the difference of the diagnostic variant iwn_erm is the effect of the penalty size). The implementation check must agree"],
        ["B-2 CORAL (two versions)", "R5_B2_coral_draws_*", "coral = the version without matching means (a weak form that does not satisfy the premise of the paper), coral_z = the version re-standardized with the bank mean and standard deviation (the version that satisfies the premise of the paper). Both versions are reported"],
        ["B-3 anchor regression (2db, period, hosp and PRAM on them)", "R5_B3_ar_fit · R5_B3_ar_eval · R5_B3_ar_hosp_* · R5_B3_fusion_* · R5_B_ar_period_rules", "AR-period lists both environment period rules (strict main and majority sensitivity). With two development environments AR-2db resolves to gamma = 1. Differences from the cited reference: gamma selection criterion, form for a binary outcome, general penalty. gamma has two criteria: AUROC and negative log likelihood"],
        ["hospital-level interval (versus frozen, X and PRAM)", "R5_B_hosp_boot (same kind as the main text) | R5_B_hosp_boot_check (is the PRAM row on the same random stream as the main-text interval) | R5_B_hosp_meta (REML + Hartung-Knapp, reference) | R5_B_hosp_dom", "the hospital bootstrap interval (R5_B_hosp_boot). The random stream and hospital order are the same as in the aggregation script, so the PRAM row (pram_b1) has the same trailing digits as the PRAM hospital interval of the main text (R5_B_hosp_boot_check confirms agreement). The meta-analysis interval is a reference value of the same function as A6 of notebook 01"],
        ["B reproduction evidence", "R5_B_repro_check · R5_B_align_check", "valid only if the mismatch is 0. Whether platt_b1 equals the stored value in all units and draws is the evidence of draw agreement"]]
    R5_TIMES["B report (hospital-level PRAM recomputation and interval)"] = (R5B_EXP_MIN.get("B report (hospital-level PRAM recomputation and interval)", float("nan")), time.time() - _t_rep)
    status("part 1 - stage B report", "done", f"method {len(R5B_PATTERNS)} lines | analysis F {len(R5B_PATTERNS_F)}line")


## E-1. Paired DeLong of the 33-column frozen logistic versus 42 columns + PRAM

For death in eICU, the AUROC of the frozen logistic with nine variables removed (33 columns, 0.8117) is higher than the 0.8103 of 42 columns + PRAM. **A paired DeLong is possible by construction**: the two probabilities are predictions for the same patients and the same outcome vector (the cohort is the same and only the variable columns differ), and the fold split depends only on the outcome vector. The notebook checks per unit that the outcome vectors of the two contexts are the same and stops before computing if they differ.

- (Y) 33-column frozen logistic = the frozen probability (`pb1`) of the F33 context; (X) 42 columns + PRAM = the full-bank `pram_b1` of the F42 context (one set collecting the 5-fold test predictions, the same fold and the same key as the rescoring of notebook 01). Difference = AUROC(Y) minus AUROC(X).
- Units = outcome 2 x target 3 = 6. **The pre-specified comparison is death in eICU alone.** The other 5 units are descriptive. The 6-unit Benjamini-Hochberg q is a reference column.
- Reproduction check: the two AUROC of this section must equal those of the grid (the full bank `b1_base` of `matrix_F33` and the full bank `pram_b1` of `matrix_F42`; read from the stored resume or recomputed by the input-collection section with the same function) (tolerance 1e-9), and also the aggregate constants (relative tolerance 1e-4, the csv has 6 significant digits). If they differ, the values of this section must not be used.
- Reading (rule E-1; three readings): p below 0.05 and a positive difference; p at or above 0.05; p below 0.05 and a negative difference. A small p may reflect a large sample and not a large difference (eICU 68,041 rows, point-estimate difference 0.0014), and a large p may reflect lack of power and not equality. The full bank is one draw, so the variation between bank draws is not included.


In [ ]:
# ================================================================
# PRAM extra  E-1 33-column frozen logistic versus 42 columns + PRAM paired DeLong (resumable)
# ================================================================
X_FZ_TAB, X_FZ_RULE = pd.DataFrame(), pd.DataFrame()
X_FZ_BAD = None
X_FZ_SHORT = None                                        # number of cases that could not be matched in the comparison with the aggregate constants of the stored csv because the constants are absent (same definition as R5P_SHORT of the input comparison: unit x two columns expected)


def x_fz_pram_oof(C, outcome, target):
    """one set collecting the 5-fold test predictions of the full-bank PRAM (pram_b1) in the 42-column context. The same fold, the same key and the same storage precision (float32) as the full-bank path of rescore_unit of the rescoring of notebook 01."""
    y = C["y"]
    n = len(y)
    oof = np.full(n, np.nan, dtype=np.float32)
    for fi, (pool_idx, test_idx) in enumerate(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(n), y)):
        P, _info = bank_predictions(C, pool_idx, test_idx, f"{outcome}|{target}|full|{fi}|0", {"pram"})
        oof[test_idx] = P["pram_b1"]
    return oof.astype(float)


def x_fz_rule(row):
    """reading of rule E-1 (numbers only): p at or above the level | p below the level with a positive difference | p below the level with a negative difference."""
    if not (np.isfinite(row["p"]) and np.isfinite(row["d_auroc"])):
        return "cannot compute"
    if row["p"] >= SIG_LEVEL:
        return "p at or above the level"
    return "p below the level, positive difference" if row["d_auroc"] > 0 else "p below the level, negative difference"


if not RUN_X_FZ:
    status("part 1 - extra E-1 frozen 33 columns versus PRAM 42 columns", "off", "RUN_X_FZ=False")
elif not (DATA_OK and (X_FZ_FROZEN_FS, OUTCOMES[0], "eICU") in CTX and (X_FZ_PRAM_FS, OUTCOMES[0], "eICU") in CTX):
    status("part 1 - extra E-1 frozen 33 columns versus PRAM 42 columns", "not available", "main inputs absent or no feature set")
else:
    _x_t0 = time.time()
    _Rz = Resumable(X_NAMES["fz"])
    _x_units = [("F42vsF33", oc, tg, "full") for oc, tg in X_FZ_UNITS]
    emit(f"### E-1 , 33-column frozen logistic versus 42 columns + full-bank PRAM , units {len(_x_units)} (completed {sum(_Rz.unit_done(u) for u in _x_units)} )")
    _x_w_all = {u: N_FOLDS * r5_cells_for(len(CTX[(X_FZ_PRAM_FS, u[1], u[2])]["y"]) / N_FOLDS, len(CTX[(X_FZ_PRAM_FS, u[1], u[2])]["y"]) * (N_FOLDS - 1) / N_FOLDS, 2) for u in _x_units}
    _x_w_left = sum(w for u, w in _x_w_all.items() if not _Rz.unit_done(u))
    _x_w_done = 0.0
    for _u in _x_units:
        if _Rz.unit_done(_u):
            print("  [SKIP]", _u)
            continue
        _t1 = time.time()
        _c42, _c33 = CTX[(X_FZ_PRAM_FS, _u[1], _u[2])], CTX[(X_FZ_FROZEN_FS, _u[1], _u[2])]
        if not np.array_equal(_c42["y"], _c33["y"]):
            raise R5Stop("stop , E-1: %s %s the outcome vectors of the 33-column context and the 42-column context are not the same (not the same patients or in a different order). The paired DeLong was not computed." % (_u[1], _u[2]))
        _pm = x_fz_pram_oof(_c42, _u[1], _u[2])
        _py = np.asarray(_c33["pb1"], dtype=np.float32).astype(float)         # same storage precision as b1_base of the stored grid
        _a_y, _a_x, _dd, _vv = delong_delta_var(_c42["y"], _py, _pm)
        _Rz.add_row({"outcome": _u[1], "target": _u[2], "n": int(len(_c42["y"])), "events": int(_c42["y"].sum()), "auroc_frozen_33": _a_y, "auroc_pram_42": _a_x,
                     "d_auroc": _dd, "var_standard": _vv, "p": delong_p(_dd, _vv)})
        _Rz.mark_unit(_u)
        _x_w_done += _x_w_all[_u]
        _eta = (time.time() - _x_t0) * (_x_w_left - _x_w_done) / max(_x_w_done, 1.0)
        print(f"  [DONE] E-1 {_u[1]} / {_u[2]} ({time.time() - _t1:.0f} s) remaining time about {_eta / 60.0:.0f} min (estimated by cost ratio)", flush=True)
    _x_raw = _Rz.df().drop_duplicates(subset=["outcome", "target"], keep="last").reset_index(drop=True)
    _x_raw["_o"] = [OUTCOMES.index(o) * 10 + TARGETS.index(g) for o, g in zip(_x_raw["outcome"], _x_raw["target"])]
    X_FZ_TAB = _x_raw.sort_values("_o").drop(columns=["_o"]).reset_index(drop=True)
    X_FZ_TAB["se"] = np.sqrt(X_FZ_TAB["var_standard"])
    X_FZ_TAB["ci_lo"], X_FZ_TAB["ci_hi"] = X_FZ_TAB["d_auroc"] - R5_Z * X_FZ_TAB["se"], X_FZ_TAB["d_auroc"] + R5_Z * X_FZ_TAB["se"]
    X_FZ_TAB["q_bh6"] = bh_adjust(X_FZ_TAB["p"].to_numpy())
    X_FZ_TAB["primary"] = [(o, g) == X_FZ_PRIMARY for o, g in zip(X_FZ_TAB["outcome"], X_FZ_TAB["target"])]
    X_FZ_TAB["p_label"] = [r5_p_label(x) for x in X_FZ_TAB["p"]]
    X_FZ_TAB["q_label"] = [r5_p_label(x) for x in X_FZ_TAB["q_bh6"]]
    # reproduction check: the grid value of the same cell (read from the stored resume or recomputed by the input-collection section)
    _x_m33 = R5P_MATRIX[X_FZ_FROZEN_FS]
    _x_m42 = R5P_MATRIX[X_FZ_PRAM_FS]
    _x_m33 = _x_m33[(_x_m33["method"] == "b1_base") & (_x_m33["bank"].astype(str) == "full")].set_index(["outcome", "target"])["auroc"]
    _x_m42 = _x_m42[(_x_m42["method"] == "pram_b1") & (_x_m42["bank"].astype(str) == "full")].set_index(["outcome", "target"])["auroc"]
    X_FZ_TAB["diff_vs_matrix_33"] = [abs(a - float(_x_m33.get((o, g), np.nan))) for a, o, g in zip(X_FZ_TAB["auroc_frozen_33"], X_FZ_TAB["outcome"], X_FZ_TAB["target"])]
    X_FZ_TAB["diff_vs_matrix_42"] = [abs(a - float(_x_m42.get((o, g), np.nan))) for a, o, g in zip(X_FZ_TAB["auroc_pram_42"], X_FZ_TAB["outcome"], X_FZ_TAB["target"])]
    _x_r33 = [R5_REF.get("matrix", {}).get(f"{X_FZ_FROZEN_FS}|{o}|{g}|full|b1_base") for o, g in zip(X_FZ_TAB["outcome"], X_FZ_TAB["target"])]
    _x_r42 = [R5_REF.get("matrix", {}).get(f"{X_FZ_PRAM_FS}|{o}|{g}|full|pram_b1") for o, g in zip(X_FZ_TAB["outcome"], X_FZ_TAB["target"])]
    X_FZ_TAB["diff_vs_csv_33"] = [abs(a - r[0]) if r else float("nan") for a, r in zip(X_FZ_TAB["auroc_frozen_33"], _x_r33)]
    X_FZ_TAB["diff_vs_csv_42"] = [abs(a - r[0]) if r else float("nan") for a, r in zip(X_FZ_TAB["auroc_pram_42"], _x_r42)]
    _x_csv_ok = [(not r1 or bool(np.isclose(a1, r1[0], rtol=R5_CSV_RTOL, atol=1e-12))) and (not r2 or bool(np.isclose(a2, r2[0], rtol=R5_CSV_RTOL, atol=1e-12)))
                 for a1, r1, a2, r2 in zip(X_FZ_TAB["auroc_frozen_33"], _x_r33, X_FZ_TAB["auroc_pram_42"], _x_r42)]
    X_FZ_TAB["repro_ok"] = (X_FZ_TAB["diff_vs_matrix_33"] <= X_FZ_TOL) & (X_FZ_TAB["diff_vs_matrix_42"] <= X_FZ_TOL) & np.array(_x_csv_ok)
    X_FZ_BAD = int((~X_FZ_TAB["repro_ok"]).sum()) + (len(X_FZ_UNITS) - len(X_FZ_TAB))
    X_FZ_SHORT = sum(1 for r in _x_r33 if not r) + sum(1 for r in _x_r42 if not r) + 2 * (len(X_FZ_UNITS) - len(X_FZ_TAB))
    X_FZ_TAB["rule_reading"] = [x_fz_rule(r) if r["primary"] else "descriptive (not a pre-specified comparison)" for _ix, r in X_FZ_TAB.iterrows()]
    _x_cols = ["outcome", "target", "primary", "n", "auroc_frozen_33", "auroc_pram_42", "d_auroc", "ci_lo", "ci_hi", "p_label", "q_label", "repro_ok", "rule_reading"]
    emit_table("E-1 , 33-column frozen logistic minus 42 columns + PRAM (full-bank paired DeLong, standard variance). The pre-specified comparison is the line with primary = True (death eICU). q is the 6-unit BH reference column (csv). The reading column is the number substitution of rule E-1", X_FZ_TAB[_x_cols], digits=4)
    X_FZ_RULE = X_FZ_TAB[X_FZ_TAB["primary"]][["outcome", "target", "auroc_frozen_33", "auroc_pram_42", "d_auroc", "ci_lo", "ci_hi", "p", "rule_reading"]]
    emit(f"- E-1 reproduction check mismatches **{X_FZ_BAD} ** | comparison with aggregate constants of the stored csv missing **{X_FZ_SHORT} ** (within the tolerance for the AUROC of the same grid cell, {X_FZ_TOL} , relative tolerance against aggregate constants of the stored csv {R5_CSV_RTOL} , must be 0 for the values of this section to be used). Reading of the pre-specified comparison (death eICU): " + (X_FZ_RULE["rule_reading"].iloc[0] if len(X_FZ_RULE) else "cannot compute"))
    save_csv(X_FZ_TAB, "R5_X_fz_table")
    R5_TIMES["E-1 33-column frozen versus 42-column PRAM"] = (R5_EXP_MIN.get("E-1 33-column frozen versus 42-column PRAM", float("nan")), time.time() - _x_t0)
    status("part 1 - extra E-1 frozen 33 columns versus PRAM 42 columns", "reproduction mismatch" if (X_FZ_BAD or len(X_FZ_TAB) != len(X_FZ_UNITS)) else ("partially compared" if X_FZ_SHORT else "done"), f"unit {len(X_FZ_TAB)} | reproduction mismatches {X_FZ_BAD} | constant comparison missing {X_FZ_SHORT}")


## E-2. Partner prevalence halved and doubled

The network analysis A7 (`network_unit`) of notebook 01 is a setting in which a hospital without local labels queries partner hospitals (at most 20) and mixes the mean of the returned probabilities into the frozen output. Notebook 01 uses the partner prevalence as observed (factor 1). This section redraws the prevalence of the partner bank at **0.5x and 2x** (to raise: all events and some non-events; to lower: all non-events and some events) and reruns with the same querying hospitals and the same partner selection (the same random stream as A7 of notebook 01). The feature sets are both 42 and 33 columns, with two outcomes and all eligible hospitals.

- Factor 1 (observed) is the value collected by the input-collection section (read if the `network_*` resume exists, otherwise recomputed by this notebook with the same function). A reproduction check reruns the factor 1 of a few hospitals with the same function and checks that it equals that value (per column, the values agree if |difference| is at or below absolute tolerance + relative tolerance x |stored value|. The relative tolerance is 1e-4, as in the other comparisons with the stored csv; the absolute tolerance is 1e-7 for delta AUROC and delta ECE and 1e-4 for the calibration intercept and slope. If rerun on the same computer with the same number of threads, the values are bit-identical. If the number of BLAS threads differs, the last digits of the single-precision distances of the neighbor search differ, and the calibration intercept and slope may differ by about 1e-5, so this tolerance is used). The comparison with the aggregate constants (distribution summary over all hospitals) is done by input collection 4.
- The unit of decision is 8 cells (feature set x outcome x multiplier), and per cell the median over hospitals is used (the mixing weight is 0.30). Rule E-2: **E2-1** = in all 8 cells median delta AUROC > 0, median delta ECE < 0 and 50% or more of hospitals with negative delta ECE. **E2-2** = otherwise (does not reach E2-3). **E2-3** = in half or more of the 8 cells median delta AUROC < 0 and median delta ECE > 0.
- If the partner prevalence is doubled, the neighbor probabilities move up and the mean predicted risk rises. If the prevalence of the querying hospital is low, a worse ECE is a structurally expected result, and it measures the condition that the partner prevalence must be known. 0.5x is the opposite direction. The redrawn banks also shrink in size. Delta AUROC uses only the ranking of the partner probabilities, so it is less sensitive to a prevalence shift.


In [ ]:
# ================================================================
# PRAM extra  E-2 partner bank prevalence 0.5x and 2x (unlabeled hospital network, resumable)
# ================================================================
X_NET_RAW, X_NET_SUM, X_NET_ALPHA, X_NET_COR, X_NET_PATTERN, X_NET_REPRO = (pd.DataFrame() for _ in range(6))
X_NET_BAD = None


def x_net_summary(d, label):
    """hospital median summary at the mixing weight NET_ALPHA_FIXED per (feature set, outcome, multiplier). The improvement fraction is a fraction of hospitals."""
    rows = []
    s0 = d[np.isclose(d["alpha"], NET_ALPHA_FIXED)]
    for (fs, oc, f), s in s0.groupby(["fs", "outcome", "factor"], sort=True):
        rows.append({"fs": fs, "outcome": oc, "factor": float(f), "source": label, "n_hospitals": int(s["hospitalid"].nunique()), "prev_diff_median": float(s["prev_diff"].median()),
                     "d_auroc_median": float(s["d_auroc"].median()), "pct_auroc_improved": float((s["d_auroc"] > 0).mean()),
                     "d_ece_median": float(s["d_ece"].median()), "pct_ece_improved": float((s["d_ece"] < 0).mean()),
                     "d_nb_rep_median": float(s["d_nb_rep"].median()), "slope_median": float(s["slope"].median()), "base_slope_median": float(s["base_slope"].median()),
                     "intercept_median": float(s["intercept"].median())})
    return pd.DataFrame(rows)


def x_net_repro_ratio(jj):
    """E-2 reproduction check: in the table matched between the same (mixing weight) rows (columns `<column>_new` and `<column>_old`), the maximum ratio to the tolerance per column = |difference| / (absolute tolerance + relative tolerance x |stored value|). At or below 1 is within tolerance."""
    return max(float(np.nanmax(np.abs(jj[c + "_new"] - jj[c + "_old"]) / (X_NET_REPRO_ATOL[c] + X_NET_REPRO_RTOL * np.abs(jj[c + "_old"])))) for c in X_NET_REPRO_ATOL)


def x_net_pattern(summ):
    """rule E-2: substitute numbers in 8 cells (feature set x outcome x multiplier). Positions: E2-1 all cells favorable | E2-2 not all cells favorable | E2-3 harmful cells in half or more | E2-0 cells insufficient. The cell table is also returned."""
    c = summ[(summ["source"] == "this run") & summ["factor"].isin(X_NET_FACTORS)].copy()
    c["keep_cell"] = (c["d_auroc_median"] > 0) & (c["d_ece_median"] < 0) & (c["pct_ece_improved"] >= X_NET_ECE_SHARE)
    c["harmful_cell"] = (c["d_auroc_median"] < 0) & (c["d_ece_median"] > 0)
    n_need = len(X_NET_FS) * len(OUTCOMES) * len(X_NET_FACTORS)
    if len(c) < n_need:
        return "E2-0 (only %d of %d cells present)" % (n_need, len(c)), c
    if int(c["harmful_cell"].sum()) >= int(np.ceil(X_NET_E23_SHARE * len(c) - 1e-9)):
        return "E2-3 (harmful cells in half or more)", c
    return ("E2-1 (all cells favorable)" if bool(c["keep_cell"].all()) else "E2-2 (not all cells favorable)"), c


if not RUN_X_NET:
    status("part 1 - extra E-2 partner prevalence", "off", "RUN_X_NET=False")
elif not (DATA_OK and R5B_SITES and R5B_HID and X_NET_FS):
    status("part 1 - extra E-2 partner prevalence", "no data", "no hospital id column or no network feature set")
else:
    _x_t0 = time.time()
    _x_parts = []
    for _fs in X_NET_FS:
        _Rn = Resumable(f"{X_NAMES['net']}_{_fs}")
        for _oc in OUTCOMES:
            for _i, _h in enumerate(R5B_SITES[_oc]):
                _u = (_fs, _oc, int(_h))
                if _Rn.unit_done(_u):
                    continue
                for _r in (network_unit(CTX[(_fs, _oc, "eICU")], _oc, _h, R5B_HID[_oc], R5B_SITES[_oc], X_NET_FACTORS, NET_ALPHA_GRID) or []):
                    _Rn.add_row({"fs": _fs, **_r})
                _Rn.mark_unit(_u)
                if _i % 20 == 0:
                    print(f"  [DONE] E-2 {_fs} {_oc} {_i + 1}/{len(R5B_SITES[_oc])} ({(time.time() - _x_t0) / 60:.1f} min)", flush=True)
        _x_parts.append(_Rn.df())
    X_NET_RAW = pd.concat([p for p in _x_parts if len(p)], ignore_index=True) if any(len(p) for p in _x_parts) else pd.DataFrame()
    if len(X_NET_RAW):
        X_NET_SUM = x_net_summary(X_NET_RAW, "this run")
        # factor 1 (observed): values read from the stored resume or recomputed by the input-collection section with the same function
        _x_ref = []
        for _fs in X_NET_FS:
            if _fs in R5P_NET1 and len(R5P_NET1[_fs]):
                _x_ref.append(x_net_summary(R5P_NET1[_fs][R5P_NET1[_fs]["factor"] == 1.0].assign(fs=_fs), "factor 1 observed values, " + r5p_src_text("network_" + _fs)))
        if _x_ref:
            X_NET_SUM = pd.concat([X_NET_SUM] + _x_ref, ignore_index=True).sort_values(["fs", "outcome", "factor", "source"]).reset_index(drop=True)
        else:
            emit("- the factor 1 (observed) column could not be collected (no hospital id or no network feature set).")
        _x_ag = X_NET_RAW[X_NET_RAW["alpha"].between(X_NET_ALPHA_LO - 1e-9, X_NET_ALPHA_HI + 1e-9)]
        X_NET_ALPHA = _x_ag.groupby(["fs", "outcome", "factor", "alpha"], sort=True).agg(n_hospitals=("hospitalid", "nunique"), d_auroc_median=("d_auroc", "median"),
                                                                                        d_ece_median=("d_ece", "median"), pct_ece_improved=("d_ece", lambda s: float((s < 0).mean()))).reset_index()
        _x_cor = []
        for (_fs, _oc, _f), _s in X_NET_RAW[np.isclose(X_NET_RAW["alpha"], NET_ALPHA_FIXED)].groupby(["fs", "outcome", "factor"], sort=True):
            if len(_s) < 5:
                continue
            _r1, _p1 = scipy_stats.spearmanr(_s["prev_diff"], _s["intercept"])
            _r2, _p2 = scipy_stats.spearmanr(_s["prev_diff"], _s["slope"])
            _x_cor.append({"fs": _fs, "outcome": _oc, "factor": float(_f), "n_points": int(len(_s)), "rho_prevdiff_intercept": float(_r1), "p_intercept": float(_p1),
                           "rho_prevdiff_slope": float(_r2), "p_slope": float(_p2)})
        X_NET_COR = pd.DataFrame(_x_cor)
        _x_g, X_NET_PATTERN = x_net_pattern(X_NET_SUM)
        emit_table(f"E-2 , hospital median per partner prevalence multiplier (mixing weight {NET_ALPHA_FIXED}, versus frozen). pct_* = fraction of improved hospitals. If source is factor 1 observed values, it is the value that used the partner prevalence as it is (the source is in the source column)", X_NET_SUM, digits=4, max_rows=40)
        emit_table(f"E-2 , mixing weight {X_NET_ALPHA_LO} ~ {X_NET_ALPHA_HI} hospital median over the whole grid (the same range as A7 of notebook 01)", X_NET_ALPHA, digits=4, max_rows=120)
        if len(X_NET_COR):
            emit_table("E-2 , rank correlation across hospitals between the partner prevalence difference (partner mean minus own) and the calibration (intercept and slope)", X_NET_COR, digits=3, max_rows=20)
        emit_table("E-2 rule substitution in 8 cells: keep_cell = median delta AUROC > 0, median delta ECE < 0, 50% or more of hospitals with negative delta ECE, harmful_cell = delta AUROC negative and delta ECE positive", X_NET_PATTERN, digits=4, max_rows=12)
        emit(f"- E-2 rule substitution result: **{_x_g}** (one of E2-0 to E2-3)")
        # reproduction check: whether rerunning factor 1 with the same function equals the factor 1 value of the input-collection section (the value read from stored results or computed with the same function)
        _x_rep = []
        for _fs in X_NET_FS:
            _o3 = R5P_NET1.get(_fs, pd.DataFrame())
            _note = r5p_src_text("network_" + _fs)
            if not len(_o3):
                _x_rep.append({"fs": _fs, "outcome": "", "hospital_k": 0, "rows compared": 0, "maximum difference": float("nan"), "maximum ratio to the tolerance": float("nan"), "agreement": False, "note": "there is no factor 1 value"})
                continue
            for _oc in OUTCOMES:
                for _h in R5B_SITES[_oc][:X_NET_REPRO_N]:
                    _new = pd.DataFrame(network_unit(CTX[(_fs, _oc, "eICU")], _oc, _h, R5B_HID[_oc], R5B_SITES[_oc], [1.0], NET_ALPHA_GRID) or [])
                    _old = _o3[(_o3["outcome"] == _oc) & (_o3["hospitalid"] == int(_h)) & (_o3["factor"] == 1.0)]
                    _jj = _new.merge(_old, on=["alpha"], suffixes=("_new", "_old"), how="left") if len(_new) else pd.DataFrame()
                    _dm = max((float(np.nanmax(np.abs(_jj[c + "_new"] - _jj[c + "_old"]))) for c in X_NET_REPRO_ATOL), default=float("nan")) if len(_jj) else float("nan")
                    _rt = x_net_repro_ratio(_jj) if len(_jj) else float("nan")
                    _x_rep.append({"fs": _fs, "outcome": _oc, "hospital_k": int(R5B_SITES[_oc].index(_h)), "rows compared": int(len(_jj)), "maximum difference": _dm, "maximum ratio to the tolerance": _rt,
                                   "agreement": bool(len(_jj) > 0 and _jj["d_auroc_old"].notna().all() and np.isfinite(_rt) and _rt <= 1.0), "note": _note})
        X_NET_REPRO = pd.DataFrame(_x_rep)
        X_NET_BAD = int((~X_NET_REPRO["agreement"]).sum())
        emit_table(f"E-2 reproduction check: does the value of factor 1 rerun with the same function equal the factor 1 value of the input-collection section (read from the stored network resume or recomputed with the same function, note column) (the first few hospitals, per column |difference| <= absolute tolerance {X_NET_REPRO_ATOL} + relative tolerance {X_NET_REPRO_RTOL} x |stored value|, maximum ratio to the tolerance <= 1)", X_NET_REPRO, digits=9, max_rows=20)
        for _fs in X_NET_FS:
            _xd = X_NET_RAW[X_NET_RAW["fs"] == _fs].copy()
            _xd.insert(1, "site", _xd["hospitalid"].map(SITE_CODE))
            save_csv(_xd.drop(columns=["hospitalid"]), f"R5_X_net_{_fs}")
        save_csv(X_NET_SUM, "R5_X_net_summary")
        save_csv(X_NET_ALPHA, "R5_X_net_alpha_grid")
        save_csv(X_NET_COR, "R5_X_net_corr")
        save_csv(X_NET_PATTERN, "R5_X_net_pattern_cells")
        save_csv(X_NET_REPRO, "R5_X_net_repro_check")
    R5_TIMES["E-2 partner prevalence"] = (R5_EXP_MIN.get("E-2 partner prevalence", float("nan")), time.time() - _x_t0)
    status("part 1 - extra E-2 partner prevalence", "done" if (len(X_NET_RAW) and not X_NET_BAD) else ("reproduction mismatch" if len(X_NET_RAW) else "not available"), f"rows {len(X_NET_RAW)} | reproduction mismatches {X_NET_BAD}")


## Extra computations report: key number lines and map of result files

This cell makes the lines (N back-calculation, E-1, E-2) and the map rows for the summary md of the closing cell.


In [ ]:
# ================================================================
# PRAM extra  summary lines | map to manuscript positions
# ================================================================
def r5x_headline_lines():
    out = []
    for r in (X_A9_INV.itertuples() if len(X_A9_INV) else []):
        out.append(f"- extra N back-calculation {r.outcome} {r.target}: {r.status} · N_hat {r.N_hat} · {r.reason}")
    for r in (X_FZ_TAB.itertuples() if len(X_FZ_TAB) else []):
        out.append(f"- extra E-1 {r.outcome} {r.target}: 33-column frozen {r.auroc_frozen_33:.4f} versus 42 columns + PRAM {r.auroc_pram_42:.4f} , difference {r.d_auroc:+.4f} (95% interval {r.ci_lo:+.4f} ~ {r.ci_hi:+.4f}), p {r.p_label}, q (6 units) {r.q_label} · {r.rule_reading} | reproduction {'agreement' if r.repro_ok else 'mismatch'}")
    for r in (X_NET_SUM.itertuples() if len(X_NET_SUM) else []):
        out.append(f"- extra E-2 {r.fs} {r.outcome} multiplier {r.factor} ({r.source}): median delta AUROC {r.d_auroc_median:+.4f} (improved hospitals {r.pct_auroc_improved:.0%}) | median delta ECE {r.d_ece_median:+.4f} (improved hospitals {r.pct_ece_improved:.0%}) | slope {r.base_slope_median:.2f} at {r.slope_median:.2f}")
    if len(X_NET_PATTERN):
        out.append("- extra E-2 rule substitution: " + x_net_pattern(X_NET_SUM)[0])
    return out


R5X_MAP_ROWS = [
    ["`[N: ...]` of the Table 5 note and Supplementary S10 note (number of eICU rows of the TabPFN evaluation sample)", "R5_A9_N_inverted(status · N_hat) · R5_A9_N_candidates", "if confirmed, N_hat. If unconfirmed, no approximate value is used"],
    ["42-column 12 rows of Supplementary S5 Panel A, 33-column MIMIC, and the Single banks of 500 column of Table 3 Panel A (fraction by point estimate)", "R5_X_S5A_values | R5_table3_PanelA_33_values | R5_X_tail_ref_F42_a54 (for the reproduction check)", "the 42-column values are one set of the new 100 draws. The reference values of the 100-draw tail results are used only for the reproduction check"],
    ["paired comparison of the 33-column frozen logistic (0.8117) with 42 columns + PRAM (0.8103), eICU death", "R5_X_fz_table (the death eICU line is the pre-specified comparison, p, interval, rule_reading)", "valid only if the reproduction mismatch is 0 (reading: rule E-1)"],
    ["partner prevalence halved or doubled (0.5x and 2x)", "R5_X_net_summary · R5_X_net_alpha_grid · R5_X_net_corr · R5_X_net_pattern_cells · R5_X_net_repro_check · R5_X_net_F42 · R5_X_net_F33", "the rule substitution is the cell table (pattern_cells)"]]
status("part 1 - extra report", "done", f"summary lines {len(r5x_headline_lines())} columns")


## Closing: summary, map of result files, hashes of existing files, completion check, actual versus expected time, result zip

(1) The key numbers and all tables shown on the screen are written to `PRAM-extra-summary.md` and `PRAM-extra-results.md` (the md of notebooks 01 and 02 are not touched). (2) The hashes of the files of notebooks 01 and 02 are recomputed and compared with those at the start. (3) A zip holding only the `R5_` csv files and the stage md is made (if there is a patient or hospital identifier column, the zip is not made and the run stops; the resume folder is not included).

**The zip is created in the current working folder.**


In [ ]:
# ================================================================
# PRAM closing . summary | map of outputs | hashes | was it finished | time | zip
# ================================================================
def r5_not_done():
    """sections not completed. No data means not complete when that section is on (a section that needs the hospital-level input)."""
    _needs = {"part 1 - stage B hospital level": bool(RUN_B_IW or RUN_B_CORAL), "part 1 - stage B-3 hospital extension": bool(RUN_B_ANCHOR and RUN_B_ANCHOR_HOSP), "part 1 - extra E-2 partner prevalence": bool(RUN_X_NET)}
    return [k for k, v in STATUS.items() if k.startswith("part 1") and (v[0] not in ("done", "off", "skipped", "no data") or (v[0] == "no data" and _needs.get(k, True)))]


if not DATA_OK:
    emit("**Not computed because the input is missing.** In the folder where the notebook was opened (" + str(Path(".").resolve()) + ") `prepared_data` was not found. Open the notebook in the folder where the stored results were done and run again (do not change `DATA_ROOT` of the settings cell). "
         "No result file (summary md and zip) was made, so there is nothing to move.")
    emit_table("status by section (was it finished)", pd.DataFrame([{"section": k, "status": v[0], "note": v[1]} for k, v in STATUS.items() if k.startswith("part 1")]), max_rows=30)
    emit(f"- sections not completed {len(r5_not_done())} columns: {r5_not_done()}")
    status("part 1 - closing", "not available", "main inputs absent: not computed because the input is missing")
else:
    R5_MAP = pd.DataFrame([
        ["q column of Table 3 Panel A (33-column six cells)", "R5_table3_PanelA_33_values (q_label is the main value, q12_label is a reference)", "the 42-column q is the RESCORE_table2_q of notebook 01 (this notebook recomputes the same value: R5_whole_table_F42_reread)"],
        ["33-column MIMIC cells of Single banks of 500 that lost more than 0.01 AUROC of Table 3 Panel A", "R5_table3_PanelA_33_values (lost_more_than_m01_pct_of_draws)", "fraction by point estimate (k by interval is R5_tail_ci_summary)"],
        ["42-column Interval below -0.01 24 cells and 33-column MIMIC rows of Supplementary S5 Panel B", "R5_S5_PanelB_values · R5_tail_ci_summary (k_ci_below_m01, n_draws)", "100-draw single bank. The 10 draws of notebook 01 are separate as R5_tail_ref_F42_10draws"],
        ["33-column Whole pool P and q of Supplementary S5 Panel B", "R5_whole_table_F33 (p, q_within_set, q_all12)", ""],
        ["reproduction evidence", "R5_repro_check · R5_whole_check_vs_matrix_F33", "valid only if the mismatch is 0"],
        ["input collection table (units read from the stored results and units recomputed with the same function)", "R5_input_provenance", "not paper numbers. The counts of which values were read from stored results and which were recomputed this time"],
        ["input comparison (secured values versus aggregate constants of the stored csv)", "R5_input_check", "valid only if the mismatch is 0"],
        ["application of the pre-specified rules", "R5_rules_applied", ""],
    ] + R5B_MAP_ROWS + R5X_MAP_ROWS, columns=["place in the paper", "csv (github_csv folder)", "note"])


    def r5_headline_lines():
        out = []
        for r in R5W_TAB.itertuples():
            out.append(f"- (a) {r.n_vars} columns full bank {r.outcome} {r.target}: AUROC change {r.d_auroc:+.4f} (95% interval {r.ci_lo:+.4f} ~ {r.ci_hi:+.4f}), p {r.p_label}, q {r.q_label} (one set of twelve cells q {r.q12_label})")
        for r in R5T_SUM.itertuples():
            out.append(f"- (b) {r.fs} {r.outcome} {r.target} bank {r.bank}: whole interval below -0.01 {int(r.k_ci_below_m01)}/{int(r.n_draws)} (exact 95% upper bound {r.cp_hi_below_m01:.1%} ) | below 0 {int(r.k_ci_below0)} | above 0 {int(r.k_ci_above0)} | point estimate below -0.01 {r.share_d_below_m01:.0%} | median {r.d_median:+.4f}")
        for r in R5_RULES.itertuples():
            out.append(f"- rule {r.rule}: {r.value}" + (f" ({r.note})" if r.note else ""))
        return out


    _post = r5_snapshot()
    R5_CHANGED = sorted(k for k in R5_SNAP0 if _post.get(k) != R5_SNAP0[k])
    R5_ADDED = sorted(k for k in _post if k not in R5_SNAP0)
    R5_HASH = pd.DataFrame([{"item": "files of the stored results collected at the start", "count": len(R5_SNAP0)}, {"item": "files whose hash now differs", "count": len(R5_CHANGED)},
                            {"item": "files newly created now outside the existing names", "count": len(R5_ADDED)}])
    emit_table("hash comparison of the files of the stored results (before the start versus now)", R5_HASH)
    emit(f"- files of the stored results whose hash changed **{len(R5_CHANGED)} ** {R5_CHANGED if R5_CHANGED else ''} | files newly made by this notebook outside the existing names {R5_ADDED if R5_ADDED else 'absent'}")
    status("part 1 - hashes of existing files", "done" if (not R5_CHANGED and not R5_ADDED) else "existing file changes", f"changed {len(R5_CHANGED)} | added {len(R5_ADDED)}")

    _time_rows = [{"section": k, "expected minutes (approximate)": round(v[0], 1) if np.isfinite(v[0]) else None, "actual minutes": round(v[1] / 60.0, 1)} for k, v in R5_TIMES.items()]
    R5_TIME = pd.DataFrame(_time_rows)
    emit_table("actual time versus expected time", R5_TIME)

    _not5 = r5_not_done()                                    # counted before writing the summary md and the result md (so that they are included in the two md and the zip)
    _NOT5_LINE = f"sections not completed {len(_not5)} columns: {_not5 if _not5 else 'absent'}"
    emit_table("status by section (was it finished)", pd.DataFrame([{"section": k, "status": v[0], "note": v[1]} for k, v in STATUS.items() if k.startswith("part 1")]), max_rows=20)
    emit("- " + _NOT5_LINE)
    _HEAD5 = [f"# PRAM extra-analysis summary (stage A: 33-column full-bank test and 42-column single-bank 100-draw interval, stage B: importance weighting, CORAL and anchor regression, extra computations: N back-calculation, frozen 33 columns versus PRAM 42 columns, partner prevalence, run {time.strftime('%Y-%m-%d %H:%M')})", "",
              "**The numbers in this file are notebook run results.**", "",
              f"**reproduction check mismatches {R5_REPRO_BAD} | reproduction check missing {R5_REPRO_SHORT} | input comparison mismatches {R5P_CHECK_BAD} | input comparison missing {R5P_SHORT} | (a) versus grid AUROC mismatches {R5W_BAD} | files of the stored results changed {len(R5_CHANGED)}** (all must be 0).", ""] + ([R5B_NOHOSP_LINE, ""] if R5B_NOHOSP_LINE else []) + [
              f"**{_NOT5_LINE}** (0 means all sections are finished).", "",
              "## Key numbers", ""] + (r5_headline_lines() if len(R5W_TAB) and len(R5T_SUM) and len(R5_RULES) else ["- (no section was computed)"]) + ["", "## Stage B key numbers (importance weighting, CORAL, anchor regression)", ""] + (r5b_headline_lines() if (len(R5B_PATTERNS) or len(R5B_IMPL) or len(R5B_PATTERNS_F)) else ["- (no section was computed in stage B)"]) + ["", "## Extra computation key numbers (TabPFN sample row count back-calculation, 33-column frozen versus 42-column PRAM test, partner prevalence halved and doubled)", ""] + (r5x_headline_lines() or ["- (no section was computed in the extra computations)"]) + [
              "", "## Input collection (were the stored results read, or recomputed by this notebook with the same function)", "", (md_table(R5P_TABLE) if len(R5P_TABLE) else "(no input collection)"), "",
              "## Scope of the input comparison (expected count of the aggregate constants of the stored csv versus the actual matched count)", "", (md_table(R5P_COVER) if len(R5P_COVER) else "(no comparison)"), "",
              "## Status by section", "", "| section | status | note |", "|---|---|---|"] + [f"| {k} | {v[0]} | {v[1]} |" for k, v in STATUS.items() if k.startswith("part 1")] + [
              "", "## Actual time versus expected", "", md_table(R5_TIME), "", "## Map to manuscript table positions", "", md_table(R5_MAP), ""]
    save_csv(R5_MAP, "R5_manuscript_table_map")
    (OUT_DIR / R5_SUM_NAME).write_text("\n".join(_HEAD5), encoding="utf-8")
    (OUT_DIR / R5_RES_NAME).write_text("\n".join(REPORT_LINES), encoding="utf-8")

    # ---- zip: only the R5_ csv files and the stage md. Stops if there is a patient or hospital identifier column ----
    ID_LIKE_R5 = {"stay_id", "patientunitstayid", "subject_id", "hadm_id", "icustay_id", "hospitalid", "uniquepid", "patienthealthsystemstayid", "pt_idx", "patient_id"}
    _bad_r5 = []
    for _p in sorted(CSV_DIR.glob(R5_CSV_PREFIX + "*.csv")):
        _h = pd.read_csv(_p, nrows=2)
        if ID_LIKE_R5 & {c.lower() for c in _h.columns}:
            _bad_r5.append((_p.name, "identifier column"))
    if _bad_r5:
        emit(f"**The zip was not made.** There are files with identifier columns: {_bad_r5}")
        status("part 1 - zip", "not available", str(_bad_r5))
    else:
        with zipfile.ZipFile(R5_ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as _z:
            for _p in sorted(CSV_DIR.glob(R5_CSV_PREFIX + "*.csv")) + [OUT_DIR / R5_SUM_NAME, OUT_DIR / R5_RES_NAME]:
                _z.write(_p, arcname=_p.relative_to(OUT_DIR.parent).as_posix())
        with zipfile.ZipFile(R5_ZIP_NAME) as _z:
            _names_r5 = _z.namelist()
        emit(f"- zip `{Path(R5_ZIP_NAME).resolve()}` ({Path(R5_ZIP_NAME).stat().st_size / 1e6:.2f} MB, files {len(_names_r5)} files, only the R5_ csv files and the stage md)")
        emit("**Keep this zip with the results.** There is only this one file (" + Path(R5_ZIP_NAME).name + ").")
        status("part 1 - zip", "done", f"{len(_names_r5)} files")


## Part 2. Four S8 cells (Supplementary S8 within-hospital bootstrap, 5,000 repetitions)

They continue in the same kernel in which the part above finished. `out_pram_s8.zip` is written again to the working folder. In a folder with stored results they pass quickly. Notebook 01 (A6 diagnosis cell) computes the same quantities with the same bootstrap stream.


In [ ]:
# ================================================================
# PRAM within-hospital bootstrap, cell 1/4 . check (the same kernel that ran the preceding cells | is there a hospital-level input | settings | hashes of the stored files)
# ================================================================
import hashlib
import json
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

S8_STOP = ""                          # reason for not computing. The cells below compute only when it is empty
S8_FS = "F42"                         # the same feature set as the A6 cell of notebook 01 (42 columns)
S8_RES = "R5S8_a6"                    # resume name of this cell (does not overlap with the stored resume)
S8_PREFIX = "R5S8_"                   # prefix of all files written by this cell (csv, md, resume)
S8_REF_N_BOOT, S8_REF_SEED = 5000, 20261003      # number of repetitions and seed of the hospital-level interval (the N_BOOT and BOOT_SEED of the settings cell are these values)
S8_NEED = ("DATA_OK", "HAVE_HOSPITALID", "EICU_TAB", "CTX", "OUTCOMES", "FS_RUN", "R5B_HID", "R5B_SITES", "R5B_HPRAM", "prepare_sites", "hospital_oof",
           "delong_delta_var", "meta_reml_hk", "Resumable", "RESUME_DIR", "CSV_DIR", "OUT_DIR", "OUT_NAME", "R5_NEW_NAMES", "REPORT_LINES", "emit", "emit_table",
           "status", "STATUS", "SITE_CODE", "N_BOOT", "BOOT_SEED", "SEED")
S8_MISSING = [n for n in S8_NEED if n not in globals()]


def s8_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as _f:
        for _blk in iter(lambda: _f.read(1 << 20), b""):
            h.update(_blk)
    return h.hexdigest()


def s8_snapshot():
    """path -> sha256 of the resume, github_csv, md and zip left by the preceding cells. Files of this cell (R5S8_ prefix) are not included."""
    snap = {}
    for _d, _pat in ((RESUME_DIR, "*"), (CSV_DIR, "*"), (OUT_DIR, "*.md"), (Path("."), OUT_NAME + "*.zip")):
        for _p in sorted(_d.glob(_pat)):
            if _p.is_file() and not _p.name.startswith(S8_PREFIX) and _p.name != OUT_NAME + "_s8.zip":
                snap[_p.as_posix()] = s8_sha(_p)
    return snap


def s8_is_r5_zip(key):
    """is the snapshot key a out_pram*.zip directly under the working folder (zip of the stored results)? These are files moved out of the folder."""
    return "/" not in key and key.startswith(OUT_NAME) and key.endswith(".zip")


S8_HID, S8_SITES, S8_SNAP0, S8_REPORT0, S8_LINES1, S8_T0 = {}, {}, {}, 0, [], time.time()
if S8_MISSING:
    S8_STOP = "not the same kernel that ran the preceding cells (%d names absent from the kernel: %s)" % (len(S8_MISSING), S8_MISSING[:6])
    print("**Not computed because it is not the same kernel that ran the preceding cells.** " + S8_STOP + ". Run the preceding cells again from start to end and then run the four cells of this file attached. No result file was made.")
else:
    S8_ZIP_NAME = OUT_NAME + "_s8.zip"
    S8_REPORT0 = len(REPORT_LINES)
    if S8_RES not in R5_NEW_NAMES:
        R5_NEW_NAMES += (S8_RES,)             # the write lock of the preceding cells opens only one more name, the resume name of this cell (other names stay blocked)
    emit("### within-hospital bootstrap, Supplementary S8: %d repetitions (reference %d | seed %d)" % (N_BOOT, S8_REF_N_BOOT, S8_REF_SEED))
    emit_table("settings (one number of bootstrap repetitions and one seed for the whole analysis)", pd.DataFrame([
        {"item": "number of within-hospital bootstrap repetitions N_BOOT", "this cell": N_BOOT, "manuscript": S8_REF_N_BOOT, "same?": "same" if N_BOOT == S8_REF_N_BOOT else "different"},
        {"item": "bootstrap random stream BOOT_SEED (split by outcome index and hospital index)", "this cell": BOOT_SEED, "manuscript": S8_REF_SEED, "same?": "same" if BOOT_SEED == S8_REF_SEED else "different"},
        {"item": "model fitting and splitting SEED (not changed)", "this cell": SEED, "manuscript": 42, "same?": "same" if SEED == 42 else "different"}]))
    if (N_BOOT, BOOT_SEED, SEED) != (S8_REF_N_BOOT, S8_REF_SEED, 42):
        emit("- warning: the value of the settings cell differs from the reference. This cell uses N_BOOT and BOOT_SEED of the kernel as they are.")
    if not DATA_OK:
        S8_STOP = "not computed by the preceding cells because the main input (prepared_data) is absent"
    elif not (HAVE_HOSPITALID and EICU_TAB is not None):
        S8_STOP = "the preceding cells could not read the eICU patient table with the hospital id column"
    elif S8_FS not in FS_RUN or any((S8_FS, _oc, "eICU") not in CTX for _oc in OUTCOMES):
        S8_STOP = "the 42-column eICU context is not in the kernel of the preceding cells"
    else:
        if R5B_SITES and all(len(R5B_SITES.get(_oc, [])) > 0 and _oc in R5B_HID for _oc in OUTCOMES):
            S8_HID, S8_SITES = R5B_HID, R5B_SITES
        else:
            S8_HID, S8_SITES = prepare_sites()
        if not all(len(S8_SITES.get(_oc, [])) > 0 for _oc in OUTCOMES):
            S8_STOP = "there is no eligible hospital (at least 300 stays, at least 15 events and 15 non-events each)"
    if S8_STOP:
        emit("**Not computed because the hospital-level input is absent.** Reason: " + S8_STOP + ". No result file (csv, md, zip) was made.")
        status("S8 within-hospital bootstrap", "not available", "not computed because the hospital-level input is absent: " + S8_STOP)
    else:
        S8_SNAP0 = s8_snapshot()
        emit("- eligible hospitals: " + " · ".join("%s %d hospitals" % (_oc, len(S8_SITES[_oc])) for _oc in OUTCOMES) + " (expected: AKI 67 | death 59)")
        emit("- hashes of %d files of the preceding cells were collected (resume, github_csv, md, stored zip). After the last cell finishes they are compared." % len(S8_SNAP0))
        status("S8 within-hospital bootstrap", "in progress", "check passed")
    S8_LINES1 = [str(_l) for _l in REPORT_LINES[S8_REPORT0:]]          # screen output of this cell (for the summary md). Kept separately so it is not duplicated when the cell is run again

In [ ]:
# ================================================================
# PRAM within-hospital bootstrap, cell 2/4 . functions (the computation of the A6 cell of notebook 01; the number of repetitions and the random stream are N_BOOT and BOOT_SEED)
# ================================================================
def s8_boot_rng(outcome, h):
    """bootstrap random stream of one hospital: made by splitting BOOT_SEED by (outcome index, hospital index). It does not mix with SEED of fitting and splitting and does not depend on the hospital order."""
    return np.random.default_rng([int(BOOT_SEED), OUTCOMES.index(outcome), int(h)])


def s8_boot_var(ys, pa, pb, rs, n_boot):
    """same formula as the A6 cell of notebook 01: resample the test patients with replacement n_boot times and recompute the difference of the two AUROCs. Samples with only one class are discarded. The variance uses ddof=1, and nan if 10 or fewer samples were used."""
    bs = []
    for _ in range(n_boot):
        i = rs.integers(0, len(ys), len(ys))
        if len(np.unique(ys[i])) > 1:
            bs.append(roc_auc_score(ys[i], pa[i]) - roc_auc_score(ys[i], pb[i]))
    return (float(np.var(bs, ddof=1)) if len(bs) > 10 else np.nan), len(bs)


def s8_oof_pair(oc, h):
    """from one set of 3-fold test predictions of hospital h: (truth, PRAM, frozen, source). (None, None, None, source) if not eligible.
    if the hospital PRAM test predictions left in the kernel by the preceding cells (R5B_HPRAM, the same function and key as hospital_oof of notebook 01) exist they are read, otherwise recomputed with the same function and the same group as the A6 cell of notebook 01."""
    C = CTX[(S8_FS, oc, "eICU")]
    hp = R5B_HPRAM.get(S8_FS) if S8_SITES is R5B_SITES else None
    if hp:
        idx = np.where(S8_HID[oc] == h)[0]
        if (oc, int(h)) not in hp:
            return None, None, None, "read stored hospital PRAM"
        if len(hp[(oc, int(h))]) == len(idx):
            return C["y"][idx], np.asarray(hp[(oc, int(h))], dtype=float), C["pb1"][idx], "read stored hospital PRAM"
    ys, oof, _fr = hospital_oof(C, oc, h, S8_HID[oc], ("frozen", "pram"))
    if ys is None or "pram_b1" not in oof:
        return None, None, None, "recomputed"
    return ys, oof["pram_b1"], oof["b1_base"], "recomputed"


def s8_save_csv(df, name):
    """writes only the files of this cell (R5S8_ prefix). The same format as save_csv of the preceding cells (utf-8-sig, 6 significant digits)."""
    assert name.startswith(S8_PREFIX), "names not starting with R5S8_ are not written: " + name
    path = CSV_DIR / (name + ".csv")
    df.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.6g")
    emit("- saved `%s` (%d rows)" % (path.as_posix(), len(df)))
    return path

In [ ]:
# ================================================================
# PRAM within-hospital bootstrap, cell 3/4 . within-hospital bootstrap variance and three-variance meta-analysis (resumable)
# ================================================================
S8_H, S8_META = pd.DataFrame(), pd.DataFrame()
S8_HOW = {"read stored hospital PRAM": 0, "recomputed": 0, "resumed": 0, "not eligible": 0}
S8_VERDICT, S8_LINES3 = [], []
if not S8_STOP:
    S8_REPORT3 = len(REPORT_LINES)
    S8_R = Resumable(S8_RES)
    S8_ORDER = [(_oc, int(_h)) for _oc in OUTCOMES for _h in S8_SITES[_oc]]
    _t_loop, _n_new = time.time(), 0
    for _k, (_oc, _h) in enumerate(S8_ORDER):
        _u = (_oc, _h, int(N_BOOT), int(BOOT_SEED))
        if S8_R.unit_done(_u):
            S8_HOW["resumed"] += 1
            continue
        _ys, _pa, _pb, _how = s8_oof_pair(_oc, _h)
        S8_HOW["not eligible" if _ys is None else _how] += 1
        if _ys is not None:
            _d, _v_std = delong_delta_var(_ys, _pa, _pb)[2:]
            _v_org = delong_delta_var(_ys, _pa, _pb, original=True)[3]
            _vb, _nb = s8_boot_var(_ys, _pa, _pb, s8_boot_rng(_oc, _h), int(N_BOOT))
            S8_R.add_row({"outcome": _oc, "hospitalid": _h, "n_boot": int(N_BOOT), "boot_seed": int(BOOT_SEED), "n_site": int(len(_ys)), "d_auroc": _d,
                          "var_delong_original": _v_org, "var_delong_standard": _v_std, "var_boot": _vb, "n_boot_used": int(_nb)})
        S8_R.mark_unit(_u)
        _n_new += 1
        if _n_new % 10 == 1 or _k == len(S8_ORDER) - 1:
            _left = len(S8_ORDER) - _k - 1
            print("  [S8] %s hospital %d/%d done, remaining time about %.0f min (estimated by the number of hospitals)" % (_oc, _k + 1, len(S8_ORDER), (time.time() - _t_loop) / max(_n_new, 1) * _left / 60.0), flush=True)
    _df = pd.DataFrame([_r for _r in S8_R.rows if _r.get("n_boot") == int(N_BOOT) and _r.get("boot_seed") == int(BOOT_SEED)])
    if len(_df):
        _df = _df.drop_duplicates(["outcome", "hospitalid"], keep="last")
        _pos = {_key: _i for _i, _key in enumerate(S8_ORDER)}
        _df["_ord"] = [_pos.get((_o, int(_i)), -1) for _o, _i in zip(_df["outcome"], _df["hospitalid"])]
        S8_H = _df[_df["_ord"] >= 0].sort_values("_ord").drop(columns="_ord").reset_index(drop=True)
    emit("- of %d hospitals, computed this time %d | resumed %d | not eligible (skipped as in the A6 cell of notebook 01) %d | source of the hospital PRAM test predictions: read from the kernel %d | recomputed with the same function %d"
         % (len(S8_ORDER), _n_new, S8_HOW["resumed"], S8_HOW["not eligible"], S8_HOW["read stored hospital PRAM"], S8_HOW["recomputed"]))
    if len(S8_H):
        S8_H["ratio_original_over_boot"] = S8_H["var_delong_original"] / S8_H["var_boot"]
        S8_H["ratio_standard_over_boot"] = S8_H["var_delong_standard"] / S8_H["var_boot"]
        _x = S8_H.copy()
        _x.insert(1, "site", _x["hospitalid"].map(SITE_CODE))
        s8_save_csv(_x.drop(columns=["hospitalid"])[["outcome", "site", "n_site", "d_auroc", "var_delong_original", "var_delong_standard", "var_boot", "n_boot_used",
                                                      "ratio_original_over_boot", "ratio_standard_over_boot", "n_boot", "boot_seed"]], "R5S8_A6_variance_diagnosis_by_hospital")
        _out = []
        for _oc, _s in S8_H.dropna(subset=["var_boot"]).groupby("outcome"):
            for _src, _col in (("DeLong original implementation", "var_delong_original"), ("DeLong standard", "var_delong_standard"), ("bootstrap", "var_boot")):
                _mt = meta_reml_hk(_s["d_auroc"], _s[_col])
                if _mt:
                    _out.append({"outcome": _oc, "variance_source": _src, "k": _mt["k"], "k_zero_var": _mt["k_zero_var"],
                                 "top1_weight_share": _mt["top1_weight_share"], "df": _mt["df"], "Q": _mt["Q"], "Q_p": _mt["Q_p"],
                                 "I2_invvar": _mt["I2_invvar"], "tau": _mt["tau"], "mu_random": _mt["mu_random"],
                                 "ci_hk_lo": _mt["ci_hk_lo"], "ci_hk_hi": _mt["ci_hk_hi"], "pi_lo": _mt["pi_lo"], "pi_hi": _mt["pi_hi"],
                                 "ratio_orig_over_boot_median": float(_s["ratio_original_over_boot"].median()),
                                 "ratio_std_over_boot_median": float(_s["ratio_standard_over_boot"].median()),
                                 "n_boot": int(N_BOOT), "boot_seed": int(BOOT_SEED)})
        S8_META = pd.DataFrame(_out)
        if len(S8_META):
            emit_table("S8 within-hospital bootstrap %d repetitions , meta-analysis using each of the three variances on the same per-hospital gains (ratio = hospital median of each DeLong variance / bootstrap variance)" % N_BOOT, S8_META, digits=4)
            s8_save_csv(S8_META, "R5S8_A6_variance_diagnosis_meta")
            for _oc, _s in S8_META[S8_META["variance_source"] == "bootstrap"].groupby("outcome"):
                _ro, _rs_ = float(_s["ratio_orig_over_boot_median"].iloc[0]), float(_s["ratio_std_over_boot_median"].iloc[0])
                if _ro > 1.5 and 0.8 <= _rs_ <= 1.25:
                    _verdict = "The original DeLong implementation overestimates the variance (the standard implementation agrees with the bootstrap). The Q and the per-draw DeLong significance fractions based on the original implementation were conservative"
                elif _rs_ > 1.25:
                    _verdict = "The standard DeLong is also larger than the bootstrap (this may mean the within-hospital variation is smaller than the independent-sample assumption of DeLong)"
                elif _ro <= 1.5:
                    _verdict = "The variances of the original and standard implementations are similar (the effect of the defect is small in these hospitals)"
                else:
                    _verdict = "The standard DeLong is smaller than the bootstrap (a bootstrap that does not reflect the within-hospital fold structure may be too small)"
                S8_VERDICT.append((_oc, _verdict))
                emit("- %s: median variance ratio original/bootstrap %.2f , standard/bootstrap %.2f , %s (the decision thresholds 1.5, 0.8 and 1.25 are the same as in the A6 cell of notebook 01. They are post hoc values)" % (_oc, _ro, _rs_, _verdict))
    else:
        emit("**There is no within-hospital bootstrap row.** No hospital exceeded the eligibility or no hospital test predictions could be made. No result file was made.")
    S8_LINES3 = [str(_l) for _l in REPORT_LINES[S8_REPORT3:]]

In [ ]:
# ================================================================
# PRAM within-hospital bootstrap, cell 4/4 . comparison with the stored values | hashes of the stored files | status | summary md | zip
# ================================================================
# 6 aggregate rows of the within-hospital bootstrap diagnosis computed with 200 repetitions per hospital (reference values for the comparison below; the values of Supplementary S8 Panel F come from the 5,000-repetition file A6_variance_diagnosis_by_hospital.csv)
S8_REF_3RD = {
    ("AKI_168h", "DeLong original implementation"): {"k": 66, "df": 65, "Q": 4.43395, "I2_invvar": 0.0, "tau": 0.0, "mu_random": 0.000686157, "ci_hk_lo": -0.0010615, "ci_hk_hi": 0.00243382, "ratio_orig_over_boot_median": 21.8159, "ratio_std_over_boot_median": 0.989278},
    ("AKI_168h", "DeLong standard"): {"k": 66, "df": 65, "Q": 90.5364, "I2_invvar": 28.2057, "tau": 0.000688222, "mu_random": 0.000877656, "ci_hk_lo": -0.00126195, "ci_hk_hi": 0.00301726, "ratio_orig_over_boot_median": 21.8159, "ratio_std_over_boot_median": 0.989278},
    ("AKI_168h", "bootstrap"): {"k": 66, "df": 65, "Q": 92.1792, "I2_invvar": 29.4852, "tau": 0.000899622, "mu_random": 0.00110789, "ci_hk_lo": -0.00126288, "ci_hk_hi": 0.00347866, "ratio_orig_over_boot_median": 21.8159, "ratio_std_over_boot_median": 0.989278},
    ("mortality_168h", "DeLong original implementation"): {"k": 59, "df": 58, "Q": 3.77574, "I2_invvar": 0.0, "tau": 0.0, "mu_random": 0.015131, "ci_hk_lo": 0.00971024, "ci_hk_hi": 0.0205517, "ratio_orig_over_boot_median": 35.3929, "ratio_std_over_boot_median": 0.992369},
    ("mortality_168h", "DeLong standard"): {"k": 59, "df": 58, "Q": 136.983, "I2_invvar": 57.6589, "tau": 0.0136531, "mu_random": 0.0113108, "ci_hk_lo": 0.00591714, "ci_hk_hi": 0.0167044, "ratio_orig_over_boot_median": 35.3929, "ratio_std_over_boot_median": 0.992369},
    ("mortality_168h", "bootstrap"): {"k": 59, "df": 58, "Q": 136.672, "I2_invvar": 57.5625, "tau": 0.0135667, "mu_random": 0.0111654, "ci_hk_lo": 0.00579703, "ci_hk_hi": 0.0165337, "ratio_orig_over_boot_median": 35.3929, "ratio_std_over_boot_median": 0.992369},
}


def s8_panel_f(r, src):
    """words printed in Supplementary S8 Panel F (including decimal places)."""
    ratio = {"DeLong original implementation": r["ratio_orig_over_boot_median"], "DeLong standard": r["ratio_std_over_boot_median"], "bootstrap": 1.0}[src]
    return {"ratio": "%.2f" % ratio, "Q": "%.1f" % r["Q"], "degrees of freedom": "%d" % int(r["df"]), "I2(%)": "%.1f" % r["I2_invvar"], "tau": "%.4f" % r["tau"],
            "pooled change (95% interval)": "%+.4f (%+.4f to %+.4f)" % (r["mu_random"], r["ci_hk_lo"], r["ci_hk_hi"])}


S8_CMP, S8_CHG, S8_CHECK = pd.DataFrame(), [], pd.DataFrame()
S8_FILES_CHANGED, S8_FILES_LOST, S8_ZIPS_MOVED, S8_CHG_DET, S8_ZIP_OK = None, None, None, [], False
S8_DET_ROWS, S8_DET_FIELDS = ("DeLong original implementation", "DeLong standard"), ("Q", "degrees of freedom", "I2(%)", "tau", "pooled change (95% interval)")      # rows and columns that do not use the bootstrap (must equal the stored results)
S8_TODO = ("What to do: if the reason is that stored files changed or disappeared, restore those files and run only the third and fourth cells again (the third cell resumes the finished hospitals and ends quickly. Do not run the first cell again). "
           "For any other reason, or if it is the same after restoring, save the notebook file (.ipynb) holding this screen.")
if not S8_STOP and len(S8_META):
    S8_REPORT4 = len(REPORT_LINES)
    _rows = []
    for _r in S8_META.to_dict("records"):
        _key = (_r["outcome"], _r["variance_source"])
        if _key not in S8_REF_3RD:
            continue
        _old, _new = s8_panel_f(S8_REF_3RD[_key], _key[1]), s8_panel_f(_r, _key[1])
        _row = {"outcome": _key[0], "variance_source": _key[1], "k stored": S8_REF_3RD[_key]["k"], "k this run": int(_r["k"])}
        if _key[1] in S8_DET_ROWS and _row["k stored"] != _row["k this run"]:
            S8_CHG_DET.append((_key[0], _key[1], "k", _row["k stored"], _row["k this run"]))
        for _f in _new:
            _row[_f + " reference (200 repetitions)"], _row[_f + " this run"] = _old[_f], _new[_f]
            if _old[_f] != _new[_f]:
                S8_CHG.append((_key[0], _key[1], _f, _old[_f], _new[_f]))
                if _key[1] in S8_DET_ROWS and _f in S8_DET_FIELDS:
                    S8_CHG_DET.append((_key[0], _key[1], _f, _old[_f], _new[_f]))
        _rows.append(_row)
    S8_CMP = pd.DataFrame(_rows)
    if len(S8_CMP):
        emit_table("reference (200 repetitions) versus this run (%d repetitions) in the notation of Supplementary S8 Panel F (decimal places)" % N_BOOT, S8_CMP, max_rows=20)
        emit("- cells whose notation changed %d / cells compared %d: %s" % (len(S8_CHG), 6 * len(S8_CMP), "; ".join("%s %s %s %s -> %s" % _c for _c in S8_CHG) if S8_CHG else "absent"))
        s8_save_csv(S8_CMP, "R5S8_A6_vs_stored_panelF")
        _n_det = int(S8_CMP["variance_source"].isin(S8_DET_ROWS).sum()) * (1 + len(S8_DET_FIELDS))
        emit("- change versus the stored results in the deterministic rows (k, Q, degrees of freedom, I2, tau and pooled change of the two DeLong rows, cells that do not use the bootstrap) **%d cells** / compared %d cells (must be 0 on real data)%s"
             % (len(S8_CHG_DET), _n_det, "" if not S8_CHG_DET else ": " + "; ".join("%s %s %s %s -> %s" % _c for _c in S8_CHG_DET)))
        if S8_CHG_DET:
            emit("- warning: if the deterministic rows differ from the stored results, the data or eligible hospitals of this run differ from the stored results. Do not use the bootstrap values of this cell.")
    # reproduction of already existing values: the per-hospital gain (d_auroc) and the DeLong variance do not depend on the bootstrap. If the hospital-level results of the preceding cells (R5_p_ or stored resume) and the A6 csv of the stored results exist, they are checked for equality.
    _chk = []
    for _nm in ("R5_p_hospital_F42", "hospital_F42"):
        _rf = Resumable(_nm).df()
        if len(_rf) and {"method", "d_auroc_b1", "var_b1"} <= set(_rf.columns):
            _m = S8_H.merge(_rf[_rf["method"] == "pram_b1"].drop_duplicates(["outcome", "hospitalid"], keep="last")[["outcome", "hospitalid", "d_auroc_b1", "var_b1"]], on=["outcome", "hospitalid"], how="inner")
            _bad = int((~(np.isclose(_m["d_auroc"], _m["d_auroc_b1"], rtol=1e-6, atol=1e-10) & np.isclose(_m["var_delong_standard"], _m["var_b1"], rtol=1e-6, atol=1e-12))).sum())
            _chk.append({"comparison target": "resume " + _nm + " (hospital-level pram_b1)", "column": "d_auroc · var_delong_standard", "hospitals matched": int(len(_m)), "mismatch": _bad})
    _f3 = CSV_DIR / "A6_variance_diagnosis_by_hospital.csv"
    if _f3.is_file():
        _o = pd.read_csv(_f3)
        _n = S8_H.assign(site=S8_H["hospitalid"].map(SITE_CODE)).merge(_o, on=["outcome", "site"], how="inner", suffixes=("", "_3rd"))
        _bad = 0
        for _c in ("d_auroc", "var_delong_original", "var_delong_standard"):
            _bad += int((~np.isclose(_n[_c], _n[_c + "_3rd"], rtol=1e-4, atol=1e-12)).sum())
        _chk.append({"comparison target": "github_csv/A6_variance_diagnosis_by_hospital.csv (stored results, 6 significant digits)", "column": "d_auroc · var_delong_original · var_delong_standard", "hospitals matched": int(len(_n)), "mismatch": _bad})
    S8_CHECK = pd.DataFrame(_chk)
    if len(S8_CHECK):
        emit_table("reproduction of already existing values (columns independent of the bootstrap, the mismatch must be 0)", S8_CHECK)
    else:
        emit("- the hospital-level resume and the A6 csv of the stored results are not in this folder, so the reproduction comparison with already existing values was not made (this cell made the hospital test predictions with the same function).")
    _post = s8_snapshot()
    S8_FILES_CHANGED = sorted(_k for _k in S8_SNAP0 if _k in _post and _post[_k] != S8_SNAP0[_k])                  # files that exist but whose hash differs
    _gone = sorted(_k for _k in S8_SNAP0 if _k not in _post)                                                        # files that disappeared (moved or deleted. This cell has no code that deletes)
    S8_ZIPS_MOVED = [_k for _k in _gone if s8_is_r5_zip(_k)]                                                        # the stored zip moved out of the folder is normal
    S8_FILES_LOST = [_k for _k in _gone if not s8_is_r5_zip(_k)]
    _added = sorted(_k for _k in _post if _k not in S8_SNAP0)
    emit("- hash comparison of the stored files: at the start %d , changed (hash mismatch) **%d** %s , disappeared (excluding the stored zip) **%d** %s , newly created files outside the stored names %d %s"
         % (len(S8_SNAP0), len(S8_FILES_CHANGED), S8_FILES_CHANGED or "", len(S8_FILES_LOST), S8_FILES_LOST or "", len(_added), _added or ""))
    if S8_ZIPS_MOVED:
        emit("- the stored zip has been moved (normal): %s" % S8_ZIPS_MOVED)
    _nb_all = bool((S8_H["n_boot_used"] == int(N_BOOT)).all())
    emit("- %d hospitals , is n_boot_used %d in all: %s (minimum %d maximum %d)" % (len(S8_H), N_BOOT, "yes" if _nb_all else "no (some samples contained only one class, so some hospitals have fewer)", int(S8_H["n_boot_used"].min()), int(S8_H["n_boot_used"].max())))
    _zero_cmp = [_r["comparison target"] for _r in _chk if _r["hospitals matched"] == 0]                                                  # zero matched hospitals is not a pass (the comparison could not be made)
    if _zero_cmp:
        emit("- targets with zero hospitals matched in the reproduction comparison: %s (not counted as a pass)" % _zero_cmp)
    _bad_cmp = int(S8_CHECK["mismatch"].sum()) if len(S8_CHECK) else 0
    _why = ([] if len(S8_H) > 0 and set(S8_META["outcome"]) == set(OUTCOMES) else ["hospital rows or meta rows of the two outcomes are insufficient"])         + (["the hashes of %d stored files changed" % len(S8_FILES_CHANGED)] if S8_FILES_CHANGED else []) + (["%d stored files disappeared (excluding the stored zip)" % len(S8_FILES_LOST)] if S8_FILES_LOST else [])         + (["%d files were created outside the stored names" % len(_added)] if _added else []) + (["reproduction comparison mismatches %d" % _bad_cmp] if _bad_cmp else []) + (["there are targets with zero hospitals matched in the reproduction comparison"] if _zero_cmp else [])
    _ok = not _why
    status("S8 within-hospital bootstrap", "done" if _ok else "not available",
           ("%d hospitals | repetitions %d | seed %d | stored file changes %d | stored zip moved %d | deterministic rows changed versus the stored results %d cells" % (len(S8_H), N_BOOT, BOOT_SEED, len(S8_FILES_CHANGED), len(S8_ZIPS_MOVED), len(S8_CHG_DET))) if _ok
           else "reason: " + " , ".join(_why) + " (see the line above). " + S8_TODO)
    emit("- status: **%s** , %s , elapsed time %.1f min" % (STATUS["S8 within-hospital bootstrap"][0], STATUS["S8 within-hospital bootstrap"][1], (time.time() - S8_T0) / 60.0))
    if _ok:
        _md = ["# PRAM within-hospital bootstrap . Supplementary S8 within-hospital bootstrap %d repetitions (run %s)" % (N_BOOT, time.strftime("%Y-%m-%d %H:%M")), "",
               "**The numbers in this file are notebook run results.**", "",
               "- status: %s , %s" % STATUS["S8 within-hospital bootstrap"], ""] + S8_LINES1 + S8_LINES3 + [str(_l) for _l in REPORT_LINES[S8_REPORT4:]]
        (OUT_DIR / (S8_PREFIX + "summary.md")).write_text("\n".join(_md), encoding="utf-8")
        _id_like = {"stay_id", "patientunitstayid", "subject_id", "hadm_id", "icustay_id", "hospitalid", "uniquepid", "patienthealthsystemstayid", "pt_idx", "patient_id"}
        _bad_id = [_p.name for _p in sorted(CSV_DIR.glob(S8_PREFIX + "*.csv")) if _id_like & {_c.lower() for _c in pd.read_csv(_p, nrows=2).columns}]
        if _bad_id:
            emit("**The zip was not made.** Files with identifier columns: %s" % _bad_id)
            status("S8 zip", "not available", str(_bad_id))
        else:
            with zipfile.ZipFile(S8_ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as _z:
                for _p in sorted(CSV_DIR.glob(S8_PREFIX + "*.csv")) + [OUT_DIR / (S8_PREFIX + "summary.md")]:
                    _z.write(_p, arcname=_p.relative_to(OUT_DIR.parent).as_posix())
            with zipfile.ZipFile(S8_ZIP_NAME) as _z:
                _names = _z.namelist()
            S8_ZIP_OK = True
            status("S8 zip", "done", "%d files" % len(_names))
            emit("- zip `%s` (%.2f MB, %d files: %s)" % (Path(S8_ZIP_NAME).resolve(), Path(S8_ZIP_NAME).stat().st_size / 1e6, len(_names), _names))
            emit("**Keep this zip with the results.** There is only this one file (%s). It is separate from the zip of the preceding cells." % Path(S8_ZIP_NAME).name)
    else:
        emit("**The zip was not made.** The status is not complete (see the line above). " + S8_TODO)
elif not S8_STOP:
    status("S8 within-hospital bootstrap", "not available", "there is no meta-analysis result")
    emit("**Not available.** Too few hospitals could be meta-analyzed, so the result file (zip) was not made.")

## Part 3. Thirteen supplementary cells (N5, N1, N2 and the federated analyses; the new computation of this notebook is the mixing-weight grid up to 1.00)

The settings are in one place above supplementary cell 1/13. Keep the `pram_results_notebook03.zip` reported by the last cell (13/13) in the working folder.


In [ ]:
# ================================================================
# PRAM supplementary cell 1/13 . check (the same kernel that ran the preceding cells and the within-hospital bootstrap | inputs | settings | hashes of the stored files)
# ================================================================
import hashlib
import json
import os
import platform
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

R6_STOP = ""                           # reason for not computing. The cells below compute only when it is empty
R6_PREFIX = "R6_"                      # prefix of all files written by this cell (csv, md, resume)
CODE_VER_MIX = "cv6b"           # code version. Change this string when the computation method changes. It is in the unit key, so stored results of another version are not read and are recomputed
SUPP_ZIP_NAME = "pram_results_notebook03.zip"    # the one result zip. It does not start with out_pram, so it is not in the hash comparison of the preceding cells and the S8 cells
R6_REF_N_BOOT, R6_REF_SEED = 5000, 20261003      # number of bootstrap repetitions and seed (the N_BOOT and BOOT_SEED of the settings cell are these values)
CODE_VER_FED = "cv8c"           # code version for fingerprints of the stored results of the three-fold supplementary analyses (independent of CODE_VER_MIX and CODE_VER_ARMS)
R8_RUN_A, R8_RUN_B, R8_RUN_C = True, True, True      # switches of the federation analyses: A federated retrieval, eight modes | B federated learning | C retrieval network
R8_K_RETR, R8_EPS_W, R8_MAX_PARTNERS = 50, 1e-6, 20  # settings of the federated retrieval analyses (K_RETR, EPS_W, MAX_PARTNERS). If they differ from K_NEIGHBORS and MAX_PARTNERS of the main settings, A is not computed
R8_NET_SIZES, R8_N_PROTO_SITE, R8_DP_EPS, R8_A_TEST_FRAC = (1, 2, 4, 8, 16), 100, (1.0, 4.0), 0.30      # network size curve | number of prototypes per hospital | aggregation noise parameter | test fraction of the original split
R8_B_TRAIN_FRAC, R8_B_WEIGHT_FRAC, R8_B_POOL_NET_CAP = 0.60, 0.20, 20000    # training and weighting shares of the original split (the remaining 20% is test) | pooled_net cap
R8_B_REST_TRAIN_SHARE = 0.75           # three folds: of what remains after removing test 1/3, the training share (base model and bank); the rest is weighting. 60/20/20 is the split when test is 20%, so 0.75 keeps the training : weighting ratio of 3 : 1. If changed to None, the B three folds are not computed and only the original split is given
R8_WORKING_MEMORY_MB = 1024              # upper limit of memory for the distance chunks of the neighbor search (brute) (MiB, the same as the sklearn default). If memory is insufficient, lower it to 256 (the search is only a little slower and the values are the same. This value is in the unit fingerprint of the federation analyses, so changing it recomputes those units from the start)
R8_NB_T = (0.10, 0.20)                 # thresholds at which net benefit is measured
R8_C_SHOW_ALPHA = (0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00)      # mixing weights printed on screen (all in csv)
R8_C_FACTORS = (1.0, 0.5, 2.0)         # partner bank prevalence multipliers (1.0 = the whole partner cohort, 0.5 and 2.0 = the two scenarios of Supplementary S11 Panel E)
R8_TOL = {"dmed": 0.0005, "pct": 2.0, "slope": 0.01, "exact_err": 1e-6, "exact_flip_share": 0.001, "exact_auc_gap": 1e-4, "agg_err": 0.002, "agg_corr": 0.02}      # reproduction tolerance: hospital median change | fraction of improved hospitals (percentage points, one hospital = 1.5 to 1.7) | calibration slope | fraction of queries whose exact-method probability difference exceeds exact_err and the AUROC difference | mean absolute difference of aggregates | correlation (the data are float32, so neighbors whose distances are alike at the 1e-7 level can swap depending on the search method; therefore a single maximum difference is not used)
R8_RULE_MARGIN, R8_RULE_FRAME = 0.001, 0.002   # minimum difference for reporting an ordering | criterion for tabulating the split-frame effect
CODE_VER_ARMS = "cv7b"           # version for fingerprints of the stored results
N_THREADS = 0                         # upper limit of BLAS threads. 0 = no limit (default). If changed to a value such as 4, the fingerprint of the stored results changes, they are recomputed, and they match version CODE_VER_MIX only within the tolerance below. The number of worker threads of the neighbor search is decided by n_jobs=-1 of the neighbor search function and is not reduced by this value (measured on a simulated 200,000 patients: from 4 to 2 the peak working set 2.56 -> 2.53 GiB)
R7_TOL_EXACT, R7_TOL_NOISE = 1e-12, 1e-6   # metric tolerance of the reproduction check. A difference <= 1e-12 is shown as within 1e-12, over 1e-12 and up to 1e-6 as within thread-order noise, and over 1e-6 as not read. The per-fold alpha lists must be equal without tolerance. Place of 1e-6: the maximum difference of a computation differing only in threads is delta ECE 4.1e-10, and the minimum difference of a truly mismatched computation is delta ECE 1.4e-4 and delta AUROC 2.7e-3, so 1e-6 lies between the two. This mark is attached regardless of the kind of metric cell, so within thread-order noise does not assert a cause (the number of cells per metric is written on the first line of the reading)
R8_FP_THR = 4                          # fixed thread value in the fingerprint of the federation analyses units. Unrelated to N_THREADS. If changed to 0, the federation analyses are recomputed from the start (the fingerprint changes)
RUN_MARKER = "v1"      # marker of this code version (printed in the environment line of the summary md together with CODE_VER_MIX, CODE_VER_ARMS and CODE_VER_FED)
R6_N5_CHUNK, R6_N5_CK_EVERY = 500000, 40         # number of rows per chunk when reading the N5 raw tables (memory: if insufficient lower to 200000 and set R6_N5_CK_EVERY to 100) | how many chunks per save (= every 20 million rows). Decided in this one place only
try:
    from threadpoolctl import threadpool_limits
    if int(N_THREADS) > 0:
        threadpool_limits(limits=int(N_THREADS))
except Exception as _exc:  # noqa: BLE001
    print("could not limit the number of threads with threadpoolctl (%s). The computation continues." % type(_exc).__name__)
R6_RUN_N1_WHOLE = True                 # four full-bank eICU cells of N1. If off, only the four hospital-level cells
R6_FS_ALL = ("F42", "F33")
R6_EICU_RAW_DIRS = [EICU_ROOT]      # confirmed folder holding the N5 raw tables (csv.gz of vitalPeriodic and nurseCharting, directly under this folder). Searched first
R6_N5_VER = "n5v1"                                                  # version for fingerprints of the stored N5 results (independent of CODE_VER_MIX, so a change recomputes only N5)
R6_EXTRA_RAW_PATHS = {"eicu_vitalperiodic": [EICU_ROOT / "vitalPeriodic.csv.gz"], "eicu_nursecharting": [EICU_ROOT / "nurseCharting.csv.gz"]}      # file in the confirmed folder. Used only when it cannot be found in the folder list above. If it is elsewhere, change this path
R6_NEED = ("DATA_OK", "HAVE_HOSPITALID", "HAVE_EICU_YEAR", "HAVE_MIMIC4_YEAR", "EICU_TAB", "CTX", "DS", "OUTCOMES", "FS_RUN", "FS_COLS", "CAT_REF", "NUM_REF",
           "R5B_HID", "R5B_SITES", "R5B_HPRAM", "R5B_PERIODS", "R5B_HOSP_BOOT_REF", "R5B_HOSP_BOOT_K", "R5P_MATRIX",
           "R5B_HOSP_COMP", "R5B_HOSP_BOOT_REF_TOL", "R5B_HOSP_BOOT_FS", "R5B_HOSP_BOOT_PAIRS", "R5B_HOSP_BOOT_N", "R5B_HOSP_BOOT_SEED", "r5b_hosp_boot_streams",
           "prepare_sites", "inner_cv_bank", "bank_predictions", "run_temporal", "build_dev", "make_ctx", "split_56_14_30", "map_to_stays", "read_mimic4_periods",
           "full_metrics", "delong_delta_var", "compute_ece", "nb_curve", "nbcol", "_logloss", "stable_seed", "RETRIEVAL_FN", "ALPHA_GRID", "N_INNER", "N_SITE_SPLITS",
           "N_FOLDS", "SEED", "N_BOOT", "BOOT_SEED", "T_SITE", "K_NEIGHBORS", "N_TIME_SPLITS", "A1_POLICIES", "TARGETS", "EICU_PATIENT_PATH", "EICU_PATIENT_CANDIDATES", "DATA_ROOT", "find_first",
           "OUT_DIR", "CSV_DIR", "RESUME_DIR", "OUT_NAME", "REPORT_LINES", "emit", "emit_table", "status", "STATUS", "SITE_CODE",
           "R5_TAIL_UNITS", "sample_bank", "TAIL_BANKS", "N_DRAWS_TAIL",
           "network_unit", "_resample_prev", "_site_reply", "NET_ALPHA_GRID", "NET_ALPHA_FIXED", "MAX_PARTNERS", "TEST_FRAC_NET")
R6_MISSING = [n for n in R6_NEED if n not in globals()]


def r6_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as _f:
        for _blk in iter(lambda: _f.read(1 << 20), b""):
            h.update(_blk)
    return h.hexdigest()


def r6_snapshot():
    """path -> sha256 of the resume, github_csv, md and zip left by the preceding cells. Files of the S8 cells (R5S8_) and of these cells (R6_ prefix, folder out_pram/r6/) are not included."""
    snap = {}
    for _d, _pat in ((RESUME_DIR, "*"), (CSV_DIR, "*"), (OUT_DIR, "*.md"), (Path("."), OUT_NAME + "*.zip")):
        for _p in sorted(_d.glob(_pat)):
            if _p.is_file() and not _p.name.startswith(("R5S8_", R6_PREFIX, "R7_", "R8_")) and _p.name not in (OUT_NAME + "_s8.zip", SUPP_ZIP_NAME):
                snap[_p.as_posix()] = r6_sha(_p)
    return snap


def r6_is_root_zip(key):
    """is the snapshot key a out_pram*.zip directly under the working folder. These are files moved out of the folder, so it is normal if they disappear."""
    return "/" not in key and key.startswith(OUT_NAME) and key.endswith(".zip")


R6_LOG = []                            # (resume name, "reused" | "computed") . Units this group of cells loaded and units it computed in this run
R6_STATE = {"hid": {}, "sites": {}, "plan": None, "fs_hosp": [], "snap0": {}, "rep0": 0, "t0": time.time(), "n1_why": "", "n2": (False, False), "n2_per": None}
R6_LINES = {"1": [], "n1": [], "n2": [], "n56": []}
if R6_MISSING:
    R6_STOP = "not the same kernel that ran the preceding cells (and the S8 cells) (%d names absent from the kernel: %s)" % (len(R6_MISSING), R6_MISSING[:6])
    print("**Not computed because it is not the same kernel that ran the preceding cells.** " + R6_STOP + ". Run the preceding cells again from start to end, run the four S8 cells, and then run the thirteen cells of this file attached. No result file was made.")
else:
    R6_STATE["rep0"] = len(REPORT_LINES)
    R6_DIR = OUT_DIR / "r6"
    R6_RES_DIR, R6_CSV_DIR = R6_DIR / "resume", R6_DIR / "csv"
    for _dd in (R6_DIR, R6_RES_DIR, R6_CSV_DIR):
        _dd.mkdir(parents=True, exist_ok=True)
    R6_CFG1 = "alpha%s|inner%d|site%d|seed%d|k%d|t%s|code%s" % ("-".join("%g" % a for a in ALPHA_GRID), N_INNER, N_SITE_SPLITS, SEED, K_NEIGHBORS, sorted(T_SITE.items()), CODE_VER_MIX)       # fingerprint of the settings and code version of the stored N1 results (recomputed if different)
    R6_CFG1_H = hashlib.sha256(R6_CFG1.encode("utf-8")).hexdigest()[:8]                # short fingerprint put in the name of the full-bank fold npz
    emit("### supplementary cells . new computations: N1 mixing weight selection criterion | N2 interval for the difference between banks of different periods | N5 fraction of temperature records")
    emit("- run environment: Python %s , numpy %s , pandas %s , scikit-learn %s . code version %s ." % (platform.python_version(), np.__version__, pd.__version__, sklearn.__version__, CODE_VER_MIX))
    emit("- code version markers: marker %s , CODE_VER_MIX %s , CODE_VER_ARMS %s , CODE_VER_FED %s . Settings: N_THREADS=%d (0 = no limit) , R8_FP_THR=%d , tolerance of the reproduction check %g (over 1e-12 and up to 1e-6 is within thread-order noise)" % (RUN_MARKER, CODE_VER_MIX, CODE_VER_ARMS, CODE_VER_FED, int(N_THREADS), int(R8_FP_THR), R7_TOL_NOISE))
    try:
        from threadpoolctl import threadpool_info
        _tpi = ["%s/%s version %s threads %s%s" % (_d.get("user_api"), _d.get("internal_api"), _d.get("version") or "unknown", _d.get("num_threads"), (" (" + str(_d["threading_layer"]) + ")") if _d.get("threading_layer") else "") for _d in threadpool_info()]
    except Exception as _exc:  # noqa: BLE001
        _tpi = ["could not call threadpool_info() (%s)" % type(_exc).__name__]
    emit("- math library threads (threadpool_info): %s . environment variables: %s . CPU logical cores %s" % (" ; ".join(_tpi) or "library absent", " , ".join("%s=%s" % (_k, os.environ.get(_k, "(not set)")) for _k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS")), os.cpu_count()))
    _n_st5 = sum(1 for _k in STATUS if str(_k).startswith("part 1"))
    _n_st8 = sum(1 for _k in STATUS if str(_k).startswith("S8"))
    emit("- number of status lines of the preceding cells in this kernel %d , status lines of the S8 cells %d . If these cells were attached after the preceding cells and the S8 cells ran in this kernel, neither is 0. If the S8 line is 0, the four S8 cells were skipped (this does not affect these computations, and the S8 files are excluded from the hash comparison of the stored files)." % (_n_st5, _n_st8))
    emit_table("settings (one number of bootstrap repetitions and one seed for the whole analysis)", pd.DataFrame([
        {"item": "number of bootstrap repetitions N_BOOT", "this cell": N_BOOT, "manuscript": R6_REF_N_BOOT, "same?": "same" if N_BOOT == R6_REF_N_BOOT else "different"},
        {"item": "bootstrap random stream seed BOOT_SEED", "this cell": BOOT_SEED, "manuscript": R6_REF_SEED, "same?": "same" if BOOT_SEED == R6_REF_SEED else "different"},
        {"item": "model fitting and splitting SEED (not changed)", "this cell": SEED, "manuscript": 42, "same?": "same" if SEED == 42 else "different"},
        {"item": "mixing weight grid ALPHA_GRID (the value of the preceding cells as it is)", "this cell": "%g to %g , %d values" % (min(ALPHA_GRID), max(ALPHA_GRID), len(ALPHA_GRID)), "manuscript": "0 to 0.5 , 11 values", "same?": "same" if len(ALPHA_GRID) == 11 else "different"},
        {"item": "inner cross-validation folds N_INNER | hospital folds N_SITE_SPLITS", "this cell": "%d · %d" % (N_INNER, N_SITE_SPLITS), "manuscript": "3 · 3", "same?": "same" if (N_INNER, N_SITE_SPLITS) == (3, 3) else "different"},
        {"item": "resampling generator method (one for the whole analysis)", "this cell": "hospital-level N1: r5b_hosp_boot_streams of the preceding cells (one generator used in sequence) . full-bank N1 and N2: per-cell generators keyed by analysis number and cell number (the same values even when stopped and resumed)",
         "manuscript": "hospital-level interval: method of the preceding cells . patient-level interval: no interval corresponds to a reference value", "same?": "hospital level same"}]))
    if (N_BOOT, BOOT_SEED, SEED) != (R6_REF_N_BOOT, R6_REF_SEED, 42):
        emit("- warning: the value of the settings cell differs from the reference. This cell uses N_BOOT and BOOT_SEED of the kernel as they are.")

    def r6_plan():
        rows = []
        n1_ok, why = True, ""
        fs_hosp = [f for f in R6_FS_ALL if f in FS_RUN and all((f, oc, "eICU") in CTX for oc in OUTCOMES)]
        if not DATA_OK:
            n1_ok, why = False, "not computed by the preceding cells because the main input (prepared_data) is absent"
        elif not (HAVE_HOSPITALID and EICU_TAB is not None):
            n1_ok, why = False, "the preceding cells could not read the eICU patient table with the hospital id column"
        elif not fs_hosp:
            n1_ok, why = False, "the eICU context (CTX) is not in the kernel of the preceding cells"
        else:
            if R5B_SITES and all(len(R5B_SITES.get(oc, [])) > 0 and oc in R5B_HID for oc in OUTCOMES):
                R6_STATE["hid"], R6_STATE["sites"] = R5B_HID, R5B_SITES
            else:
                R6_STATE["hid"], R6_STATE["sites"] = prepare_sites()
            if not all(len(R6_STATE["sites"].get(oc, [])) > 0 for oc in OUTCOMES):
                n1_ok, why = False, "there is no eligible hospital (at least 300 stays, at least 15 events and 15 non-events each)"
        R6_STATE["fs_hosp"] = fs_hosp if n1_ok else []
        R6_STATE["n1_why"] = why
        rows.append({"cell": "N1 hospital level four cells", "input": "hospital id of the eICU patient table and CTX of the preceding cells", "status": "computed (done)" if n1_ok else "not computed because the input is missing: " + why})
        rows.append({"cell": "N1 eICU full bank four cells", "input": "CTX (eICU) of the preceding cells", "status": ("computed (done)" if (n1_ok and R6_RUN_N1_WHOLE) else ("off (R6_RUN_N1_WHOLE=False)" if n1_ok else "not computed because the input is missing: " + why))})
        n2_eicu = bool(DATA_OK and HAVE_EICU_YEAR and EICU_TAB is not None and "year" in getattr(EICU_TAB, "columns", []) and all(("F42", oc, "eICU") in CTX for oc in OUTCOMES))
        per_ok = R5B_PERIODS is not None or HAVE_MIMIC4_YEAR
        n2_m4 = bool(DATA_OK and per_ok and "F42" in FS_COLS and all((d, oc) in DS for d in ("MIMIC-III", "MIMIC-IV") for oc in OUTCOMES))
        rows.append({"cell": "N2 eICU two rows (2014 versus 2015)", "input": "hospitaldischargeyear of the eICU patient table", "status": "computed (done)" if n2_eicu else "not computed because the input is missing: no eICU discharge year column or CTX"})
        rows.append({"cell": "N2 MIMIC-IV four rows (strict and majority rules)", "input": "MIMIC-IV patients and icustays (the period table of the preceding cells)", "status": "computed (done)" if n2_m4 else "not computed because the input is missing: no MIMIC-IV period table or MIMIC-III data"})
        R6_STATE["n2"] = (n2_eicu, n2_m4)
        n5_ok = bool(DATA_OK and ("eICU", OUTCOMES[0]) in DS)
        rows.append({"cell": "N5 fraction of temperature records", "input": "eICU raw tables vitalPeriodic and nurseCharting (searched in the same folders as the eICU candidate paths of the preceding cells)", "status": "computed if the raw tables are found" if n5_ok else "not computed because the input is missing: no eICU cohort"})
        return pd.DataFrame(rows)

    R6_STATE["plan"] = r6_plan()
    emit_table("what these cells do and their inputs (for an input not found, only that cell is skipped)", R6_STATE["plan"])
    emit("- the readings of this file carry the rule numbers in brackets")
    R6_STATE["snap0"] = r6_snapshot()
    emit("- hashes of %d stored files were collected (resume, github_csv, md, the stored zip before it is moved out of the folder). After the last cell finishes they are compared." % len(R6_STATE["snap0"]))
    status("additional: check", "done", "check passed")
    R6_LINES["1"] = [str(_l) for _l in REPORT_LINES[R6_STATE["rep0"]:]]

In [ ]:
# ================================================================
# PRAM supplementary cell 2/13 . functions (the functions of the preceding cells are called as they are, and only the selection criterion score, storage, resampling and raw-table reading are newly written)
# ================================================================
R6_CRIT = ("auroc", "logloss", "brier")
R6_CRIT_LABEL = {"auroc": "AUROC (rule of the main analysis)", "logloss": "log loss", "brier": "Brier"}
R6_BINS = np.linspace(0, 1, 11)


def r6_small(v, lim=5):
    """count columns that go to aggregate files: if 1 to lim-1, masked as <5 (0 is kept). Only counts are masked; fractions and estimates are not touched."""
    return "<%d" % lim if 0 < int(v) < lim else int(v)


def r6_json(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


class R6Resumable:
    """writes the rows of one unit at once and writes a completion marker last. Rows without a completion marker (a unit interrupted midway) are ignored on reading. Resumable of the preceding cells has a write lock and is not used. There is no code that deletes."""

    def __init__(self, name):
        self.name = name
        self.rows_path = R6_RES_DIR / (name + ".jsonl")
        self.units_path = R6_RES_DIR / (name + ".units.jsonl")
        self.tok, self.by_unit, self.n_dropped = {}, {}, 0
        if self.units_path.exists():
            _txt = self.units_path.read_text(encoding="utf-8")
            _lines = _txt.split("\n")
            if not _txt.endswith("\n"):
                _lines = _lines[:-1]                          # a line without a newline at the end is a line interrupted while writing
            for _ln in _lines:
                _ln = _ln.strip()
                if _ln and "\t" in _ln:
                    _k, _t = _ln.split("\t", 1)
                    self.tok[_k] = _t
        if self.rows_path.exists():
            for _ln in self.rows_path.read_text(encoding="utf-8").splitlines():
                _ln = _ln.strip()
                if not _ln:
                    continue
                try:
                    _r = json.loads(_ln)
                except json.JSONDecodeError:
                    self.n_dropped += 1
                    continue
                _k = _r.get("_unit")
                if _k in self.tok and _r.get("_tok") == self.tok[_k]:
                    self.by_unit.setdefault(_k, []).append(_r)
        if self.n_dropped:
            print("  [%s] %d incomplete lines were discarded (the last line at the time of interruption)" % (name, self.n_dropped))

    @staticmethod
    def key(unit):
        return "::".join(str(x) for x in unit)

    def unit_done(self, unit):
        return self.key(unit) in self.tok

    def get(self, unit):
        return [{k: v for k, v in r.items() if k not in ("_unit", "_tok")} for r in self.by_unit.get(self.key(unit), [])]

    def commit(self, unit, rows):
        k, t = self.key(unit), "%d" % time.time_ns()
        norm = [json.loads(json.dumps(dict(r, _unit=k, _tok=t), default=r6_json)) for r in rows]      # a round trip is made once so that the row just computed and the row read again are identical
        for _pp in (self.rows_path, self.units_path):
            if _pp.exists() and _pp.stat().st_size:
                with open(_pp, "rb") as f:
                    f.seek(-1, 2)
                    _last = f.read(1)
                if _last != b"\n":
                    with open(_pp, "a", encoding="utf-8") as f:       # closes the line first so that a new line attached after an interrupted line does not break the new row
                        f.write("\n")
        with open(self.rows_path, "a", encoding="utf-8") as f:
            for r in norm:
                f.write(json.dumps(r) + "\n")
            f.flush()
        with open(self.units_path, "a", encoding="utf-8") as f:
            f.write(k + "\t" + t + "\n")
            f.flush()
        self.tok[k] = t
        self.by_unit[k] = norm


def r6_unit(res, unit, fn, extra_ok=None):
    """if a stored result with the same settings exists it is loaded (skipped), otherwise computed and stored. extra_ok is a function that checks whether files other than the stored result are needed."""
    if res.unit_done(unit) and (extra_ok is None or extra_ok(res.get(unit))):
        R6_LOG.append((res.name, "reused"))
        return res.get(unit)
    rows = fn()
    res.commit(unit, rows)
    R6_LOG.append((res.name, "computed"))
    return res.get(unit)


def r6_save_csv(df, name):
    assert name.startswith(R6_PREFIX), "names not starting with R6_ are not written: " + name
    path = R6_CSV_DIR / (name + ".csv")
    df.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.6g")
    emit("- saved `%s` (%d rows)" % (path.as_posix(), len(df)))
    return path


def r6_wmean_boot(vals, w, idx):
    """weighted mean of per-hospital (or per-row) values and the percentile 95% interval over the resample idx (B x k). The same formula as the hospital-level interval of the preceding cells (r5b_hosp_boot_table)."""
    v, we = np.asarray(vals, dtype=float), np.asarray(w, dtype=float)
    ok = np.isfinite(v)
    we, vf = np.where(ok, we, 0.0), np.where(ok, v, 0.0)
    est = float((we * vf).sum() / we.sum())
    bs = (we[idx] * vf[idx]).sum(1) / np.maximum(we[idx].sum(1), 1e-300)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return est, float(lo), float(hi)


# ---- N1: selection criterion of the mixing weight a ---------------------------------------------------
def r6_brier(y, p):
    return float(np.mean((np.asarray(p, dtype=float) - np.asarray(y, dtype=float)) ** 2))


def r6_inner_scores(Xb, yb, p1b, key):
    """in the same folds as inner_cv_bank of the preceding cells (StratifiedKFold N_INNER, shuffle, random_state = stable_seed(innercv, b1| + key)), the same grid and the same retrieval predictions,
    collects the AUROC, log loss and Brier per alpha per fold. Only folds in which AUROC is defined (folds whose validation fold has both classes) enter the three criteria. None if the bank is too small."""
    n = len(yb)
    if n < N_INNER * 5 or len(np.unique(yb)) < 2:
        return None
    skf = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=stable_seed("innercv", "b1|" + key))
    sc = {a: {"auroc": [], "logloss": [], "brier": []} for a in ALPHA_GRID}
    for itr, iva in skf.split(np.zeros(n), yb):
        pr = RETRIEVAL_FN({"X_tr_s": Xb[itr], "y_tr": yb[itr], "X_te_s": Xb[iva], "y_te": yb[iva], "mi": None, "mi_prob": None})
        if len(np.unique(yb[iva])) > 1:
            for a in ALPHA_GRID:
                p = (1 - a) * p1b[iva] + a * pr
                sc[a]["auroc"].append(roc_auc_score(yb[iva], p))
                sc[a]["logloss"].append(_logloss(yb[iva], p))
                sc[a]["brier"].append(r6_brier(yb[iva], p))
    return sc


def r6_pick(sc, crit):
    """the a with the best criterion. AUROC follows the formula of inner_cv_bank of the preceding cells as it is (largest mean, the earlier a on ties), log loss and Brier have the smallest mean (the earlier a on ties)."""
    if sc is None:
        return 0.0
    if crit == "auroc":
        return max(ALPHA_GRID, key=lambda a: np.nanmean(sc[a]["auroc"]) if sc[a]["auroc"] else -1)
    return min(ALPHA_GRID, key=lambda a: np.nanmean(sc[a][crit]) if sc[a][crit] else np.inf)


def r6_pick_fold(C, bank, test, key, xcheck):
    """chooses the a of the three criteria with one bank and makes three sets of test predictions (the PRAM formula of bank_predictions (1 - a) * frozen + a * retrieval). If xcheck, it also checks whether it equals the AUROC a chosen by inner_cv_bank of the preceding cells."""
    Xb, yb, p1b = C["Xg"][bank], C["y"][bank], C["pb1"][bank]
    Xt, yt, p1t = C["Xg"][test], C["y"][test], C["pb1"][test]
    p_retr = RETRIEVAL_FN({"X_tr_s": Xb, "y_tr": yb, "X_te_s": Xt, "y_te": yt, "mi": None, "mi_prob": None})
    sc = r6_inner_scores(Xb, yb, p1b, key)
    al = {c: r6_pick(sc, c) for c in R6_CRIT}
    ok = True
    if xcheck:
        icv = inner_cv_bank(Xb, yb, p1b, RETRIEVAL_FN, key="b1|" + key)
        ok = bool(icv["alpha"] == al["auroc"])
    return {c: (1 - al[c]) * p1t + al[c] * p_retr for c in R6_CRIT}, al, ok


def r6_hospital_unit(fs, oc, h):
    """makes the 3-fold test predictions of one hospital (feature set fs, outcome oc) with the three criteria and keeps only the metrics (predictions are not stored)."""
    C = CTX[(fs, oc, "eICU")]
    y, hid = C["y"], R6_STATE["hid"][oc]
    idx = np.where(hid == h)[0]
    ys = y[idx]
    base = {"fs": fs, "outcome": oc, "hospitalid": int(h)}
    n_sp = min(N_SITE_SPLITS, int(ys.sum()), int(len(ys) - ys.sum()))
    if n_sp < 2:
        return [dict(base, status="not eligible")]
    skf = StratifiedKFold(n_splits=n_sp, shuffle=True, random_state=stable_seed("a29", oc, int(h)))
    pr = {c: np.full(len(ys), np.nan) for c in R6_CRIT}
    alphas = {c: [] for c in R6_CRIT}
    xok = True
    for fi, (bank_i, test_i) in enumerate(skf.split(np.zeros(len(ys)), ys)):
        bank, test = idx[bank_i], idx[test_i]
        if len(np.unique(y[bank])) < 2 or len(np.unique(y[test])) < 2:
            return [dict(base, status="not eligible")]
        P, al, ok = r6_pick_fold(C, bank, test, "a29|%s|%d|%d" % (oc, int(h), fi), True)
        xok = xok and ok
        for c in R6_CRIT:
            pr[c][test_i] = P[c]
            alphas[c].append(al[c])
    t = float(T_SITE[oc])
    row = dict(base, status="ok", n_site=int(len(ys)), n_events=int(ys.sum()), n_folds=int(n_sp), xcheck_inner=bool(xok))
    hp = (R5B_HPRAM.get(fs) or {}).get((oc, int(h)))
    row["hpram_match"] = None if (hp is None or len(hp) != len(ys)) else bool(np.max(np.abs(np.asarray(hp, dtype=float) - pr["auroc"])) <= 1e-12)
    for name, p in (("frozen", C["pb1"][idx]),) + tuple((c, pr[c]) for c in R6_CRIT):
        row["auroc_" + name] = float(roc_auc_score(ys, p))
        row["ece_" + name] = float(compute_ece(ys, p))
        row["nb_" + name] = float(nb_curve(ys, p, np.array([t]))[0])
    for c in R6_CRIT:
        row["alpha_" + c] = [float(a) for a in alphas[c]]
    return [row]


R6_STREAM_SAME = "same random stream as the main text"
R6_STREAM_IND = "independent random stream (the number and composition of hospitals differ from the preceding cells)"


def r6_hosp_orders():
    """order of the PRAM rows of the stored hospital resume: (feature set, outcome) -> hospital id order. The same formula as `_hb_order` of the preceding cells (the pram_b1 rows of R5B_HOSP_COMP in order of appearance)."""
    if not (len(R5B_HOSP_COMP) and {"method", "fs", "outcome", "hospitalid"} <= set(R5B_HOSP_COMP.columns)):
        return {}
    return {_k: list(dict.fromkeys(int(v) for v in _g["hospitalid"])) for _k, _g in R5B_HOSP_COMP[R5B_HOSP_COMP["method"] == "pram_b1"].groupby(["fs", "outcome"], sort=False)}


def r6_hosp_boot_plan(g, fs, oc, streams, orders):
    """returns the hospital rows g of one cell (feature set, outcome) together with the bootstrap replicate indices idx (N x k). Under the same conditions as r5b_hosp_boot_table of the preceding cells (the number of hospitals equals R5B_HOSP_BOOT_K and the hospital composition equals the hospital order of the stored results)
    it uses the replicates of the random stream r5b_hosp_boot_streams() of the preceding cells and the hospital order of notebook 01 (the same trailing digits as the main-text hospital-level interval). Otherwise, as in the preceding cells, it sorts by hospital id and draws anew from BOOT_SEED. The three selection criteria and the pair differences use the same replicates."""
    ord_ = orders.get((fs, oc))
    exp_k = dict(R5B_HOSP_BOOT_K)
    if ord_ is not None and (fs, oc) in streams and exp_k.get(oc) == len(ord_) == len(g) and set(int(h) for h in g["hospitalid"]) == set(ord_):
        return g.set_index("hospitalid").loc[list(ord_)].reset_index(), streams[(fs, oc)], R6_STREAM_SAME
    g = g.sort_values("hospitalid").reset_index(drop=True)
    k = len(g)
    return g, np.random.default_rng(R5B_HOSP_BOOT_SEED).integers(0, k, size=(int(R5B_HOSP_BOOT_N), k)), R6_STREAM_IND


def r6_n1_cells(df):
    """collects the hospital rows (status ok) into three tables per (feature set, outcome): change per criterion (versus frozen) | pair differences between selection criteria | distribution of the selected a. The hospital bootstrap replicates follow the method of the preceding cells (r6_hosp_boot_plan), so the same values come out even when stopped and resumed."""
    a_rows, p_rows, alpha_rows = [], [], []
    streams = r5b_hosp_boot_streams()
    orders = r6_hosp_orders()
    for fs in R6_FS_ALL:
        for oc in OUTCOMES:
            g = df[(df["fs"] == fs) & (df["outcome"] == oc) & (df["status"] == "ok")].reset_index(drop=True)
            if len(g) < 3:
                continue
            g, idx, stream = r6_hosp_boot_plan(g, fs, oc, streams, orders)
            k = len(g)
            w = g["n_site"].to_numpy(float)
            d, fr = {}, {}
            for c in R6_CRIT:
                d[c] = {m: g[m + "_" + c].to_numpy(float) - g[m + "_frozen"].to_numpy(float) for m in ("auroc", "ece", "nb")}
            for c in R6_CRIT:
                al = np.concatenate([np.asarray(x, dtype=float) for x in g["alpha_" + c]])
                zero_h = float(np.mean([all(float(a) == 0.0 for a in x) for x in g["alpha_" + c]]))
                fr[c] = (float(np.mean(al == 0.0)), zero_h)
                row = {"fs": fs, "outcome": oc, "k": k, "stream": stream, "criterion": R6_CRIT_LABEL[c], "a0_share_fold": fr[c][0], "a0_share_hospital_all_folds": zero_h, "alpha_median": float(np.median(al))}
                for m, nm in (("auroc", "dAUROC"), ("ece", "dECE"), ("nb", "dNB")):
                    est, lo, hi = r6_wmean_boot(d[c][m], w, idx)
                    row[nm], row[nm + "_lo"], row[nm + "_hi"] = est, lo, hi
                a_rows.append(row)
                for av in ALPHA_GRID:
                    alpha_rows.append({"fs": fs, "outcome": oc, "criterion": R6_CRIT_LABEL[c], "alpha": float(av), "folds": int(np.sum(np.isclose(al, av)))})  # the number of times alpha was chosen is not a person-level count and is not masked
            for c in ("logloss", "brier"):
                row = {"fs": fs, "outcome": oc, "k": k, "stream": stream, "difference": "%s selection minus AUROC selection" % R6_CRIT_LABEL[c], "criterion": c, "a0_share_fold": fr[c][0], "a0_share_fold_auroc": fr["auroc"][0]}
                for m, nm in (("auroc", "dAUROC"), ("ece", "dECE"), ("nb", "dNB")):
                    est, lo, hi = r6_wmean_boot(d[c][m] - d["auroc"][m], w, idx)
                    row[nm], row[nm + "_lo"], row[nm + "_hi"] = est, lo, hi
                    row[nm + "_half"] = (hi - lo) / 2.0
                p_rows.append(row)
    return pd.DataFrame(a_rows), pd.DataFrame(p_rows), pd.DataFrame(alpha_rows)


def r6_cells_text(x):
    return ", ".join("%s %s" % (a, "AKI" if str(b).startswith("AKI") else "death") for a, b in zip(x["fs"], x["outcome"])) or "absent"


def r6_ece_text(x):
    return "; ".join("%s %s ECE pair difference %+.4f (%+.4f to %+.4f)" % (a, "AKI" if str(b).startswith("AKI") else "death", e, lo, hi) for a, b, e, lo, hi in zip(x["fs"], x["outcome"], x["dECE"], x["dECE_lo"], x["dECE_hi"]))


def r6_n1_cells_names(txt):
    """returns only the cell names from text of the form of 2 cells: 42 AKI, 33 death."""
    return txt.split(": ", 1)[1] if ": " in txt else txt


R6_STATES = ("1 insensitive", "1 and 2 overlap", "2 calibration improves", "3 a discrimination loss of 0.005 cannot be excluded", "2 + 3 cells split", "4a insensitivity cannot be judged (interval width)", "4b mixed")


def r6_n1_reading(pairs):
    """applies reading rules 1 to 4 to each selection criterion (log loss, Brier). Cell = feature set x outcome, four cells. If rules 1 and 2 hold together, it is read as one overlap of 1 and 2,
    if none of rules 1 to 3 applies, 4a and 4b are separated by interval width. One criterion has one state, so opposite readings do not appear at the same time."""
    out = []
    for c in ("logloss", "brier"):
        s = pairs[pairs["criterion"] == c]
        if len(s) == 0:
            continue
        r1 = bool(len(s) == 4 and ((s["dAUROC_lo"] >= -0.005) & (s["dAUROC_hi"] <= 0.005) & (s["dECE_lo"] >= -0.005) & (s["dECE_hi"] <= 0.005)).all())
        ece_in = bool(len(s) == 4 and ((s["dECE_lo"] >= -0.005) & (s["dECE_hi"] <= 0.005)).all())      # only the ECE pair-difference interval is looked at (the AUROC upper bound is not looked at)
        r2 = s[(s["dECE_hi"] < -0.0025) & (s["dAUROC_lo"] >= -0.005)]
        r3 = s[s["dAUROC_lo"] < -0.005]
        r3w = r3[(r3["dAUROC_hi"] - r3["dAUROC_lo"]) > 0.010]                      # among rule 3 cells, cells whose whole AUROC interval width is wider than 0.010
        wide = s[((s["dAUROC_hi"] - s["dAUROC_lo"]) > 0.010) | ((s["dECE_hi"] - s["dECE_lo"]) > 0.010)]
        if r1 and len(r2):
            state = "1 and 2 overlap"
        elif r1:
            state = "1 insensitive"
        elif len(r2) and len(r3):
            state = "2 + 3 cells split"
        elif len(r3):
            state = "3 a discrimination loss of 0.005 cannot be excluded"
        elif len(r2):
            state = "2 calibration improves"
        else:
            state = "4a insensitivity cannot be judged (interval width)" if len(wide) else "4b mixed"
        z = s[s["a0_share_fold"] > 2 * s["a0_share_fold_auroc"]]
        out.append({"selection criterion": R6_CRIT_LABEL[c], "reading (priority applied)": state, "rule 1 all four cells inside the band": "yes" if r1 else "no", "ECE pair-difference interval inside the band in all four cells": "yes" if ece_in else "no", "rule 2 calibration improves (cells)": "%d cells: %s" % (len(r2), r6_cells_text(r2)),
                    "rule 3 a discrimination loss of 0.005 cannot be excluded (cells)": "%d cells: %s" % (len(r3), r6_cells_text(r3)), "among rule 3 cells, cells whose AUROC interval width is wider than 0.010": "%d cells: %s" % (len(r3w), r6_cells_text(r3w)), "cells whose whole interval width is wider than 0.010": "%d cells: %s" % (len(wide), r6_cells_text(wide)),
                    "cells where the a=0 fraction is more than twice that of AUROC selection": "%d cells: %s" % (len(z), r6_cells_text(z)), "value of the cells with lower ECE": r6_ece_text(r2)})
    return pd.DataFrame(out)


def r6_n1_lines(rd):
    """makes readings for each selection criterion of the reading table (r6_n1_reading). One criterion has one state, so opposite readings do not appear at the same time."""
    out = []
    for r_ in rd.to_dict("records"):
        lab, st = r_["selection criterion"], r_["reading (priority applied)"]
        n2_ = int(r_["rule 2 calibration improves (cells)"].split("cell")[0])
        n3_ = int(r_["rule 3 a discrimination loss of 0.005 cannot be excluded (cells)"].split("cell")[0])
        n3w_ = int(r_["among rule 3 cells, cells whose AUROC interval width is wider than 0.010"].split("cell")[0])
        # the discussion reading has one condition independent of the state (three or more cells under rule 2, no cell under rule 3); the same ECE result does not change whether it applies because of the AUROC upper bound of another cell
        disc = (n2_ >= 3 and n3_ == 0)
        # if the ECE pair-difference interval is inside -0.005 to +0.005 in all four cells, an ECE note is added (the condition looks only at the ECE interval, so the same ECE result gives the same note)
        ece_cl = " and the ECE pair-difference interval is inside -0.005 to +0.005 in all four cells" if r_["ECE pair-difference interval inside the band in all four cells"] == "yes" else ""
        disc_txt = ("three or more of the four cells and no rule 3 cell (two facts: ECE was lower, and the lower bound of the AUROC pair difference was above -0.005;%s) when the selection criterion is changed to %s." % (ece_cl if ece_cl else " there are cells where the ECE pair-difference interval is outside -0.005 to +0.005, so the ECE phrase is not added", lab)) if disc else "the discussion reading does not apply (fewer than three cells or there is a rule 3 cell)."
        if st == "1 insensitive":
            out.append("- [rule 1] %s: in all four cells the AUROC pair-difference interval and the ECE pair-difference interval are inside -0.005 to +0.005 (selection by %s versus selection by AUROC)." % (lab, lab))
        elif st == "1 and 2 overlap":
            out.append("- [rules 1 and 2 overlap] %s: the AUROC and ECE pair-difference intervals are inside -0.005 to +0.005 in all four cells, and there are cells where the ECE pair-difference interval is wholly below -0.0025 (%s). Both facts apply, and neither insensitive nor improved summarizes them: "
                 "when the mixing weight is chosen by %s, hospital-level ECE was lower than when chosen by AUROC in %d cells (%s), and the pair-difference intervals of AUROC and ECE were inside +-0.005 in all four cells. %s" % (lab, r_["value of the cells with lower ECE"], lab, n2_, r6_n1_cells_names(r_["rule 2 calibration improves (cells)"]), disc_txt))
        elif st in ("2 calibration improves", "2 + 3 cells split"):
            out.append("- [rule 2] %s: %s . %s" % (lab, r_["rule 2 calibration improves (cells)"], disc_txt + ("" if disc else "")))
        if n3_ > 0:
            out.append("- [rule 3] %s: %s . The reading name is: a discrimination loss of 0.005 cannot be excluded. It means that the lower bound of the AUROC pair-difference interval is below -0.005; it is not evidence that there is a loss." % (lab, r_["rule 3 a discrimination loss of 0.005 cannot be excluded (cells)"]))
            if n3w_ > 0:
                out.append("- [rule 3, interval is wide] %s: among rule 3 cells there are cells whose whole AUROC pair-difference interval width is wider than 0.010 (%s). It cannot be decided with this sample; the per-cell values and the interval half width (the _half column of the pair-difference table above) are given." % (lab, r_["among rule 3 cells, cells whose AUROC interval width is wider than 0.010"]))
        if st.startswith("4a"):
            out.append("- [rule 4a] %s: there are cells whose whole interval width is wider than 0.010 (%s). Whether it is insensitive cannot be decided with this sample; the per-cell values and the interval half width (the _half column of the pair-difference table above) are given." % (lab, r_["cells whose whole interval width is wider than 0.010"]))
        if st.startswith("4b"):
            out.append("- [rule 4b] %s: mixed; the per-cell values are given." % lab)
        if int(r_["cells where the a=0 fraction is more than twice that of AUROC selection"].split("cell")[0]) > 0:
            out.append("- [note] %s: there are cells where the a=0 fraction is more than twice that of AUROC selection (%s)." % (lab, r_["cells where the a=0 fraction is more than twice that of AUROC selection"]))
    return out


# ---- N1 full bank (whole eICU cohort, 5-fold batch) ------------------------------------------
def r6_ece_fast(y, p):
    b = np.searchsorted(R6_BINS, p, side="right") - 1
    m = (b >= 0) & (b < 10)
    cnt = np.bincount(b[m], minlength=10)
    sy = np.bincount(b[m], weights=y[m], minlength=10)
    sp = np.bincount(b[m], weights=p[m], minlength=10)
    nz = cnt > 0
    return float(np.abs(sy[nz] - sp[nz]).sum() / len(y))


def r6_nb_fast(y, p, t):
    k = p >= t
    return float(y[k].sum() / len(y) - (k.sum() - y[k].sum()) / len(y) * (t / (1 - t)))


def r6_whole_fold(fs, oc, fi):
    """makes the test predictions of the three criteria with the same bank, test fold and key as one fold of the full bank (full) of run_matrix_unit of the preceding cells, and stores them as npz (arrays in patient order, no identifier, only inside the resume folder)."""
    C = CTX[(fs, oc, "eICU")]
    y = C["y"]
    pool_idx, test_idx = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))[fi]
    P, al, _ok = r6_pick_fold(C, pool_idx, test_idx, "%s|eICU|full|%d|0" % (oc, fi), False)
    path = R6_RES_DIR / ("R6_n1w_%s_%s_f%d_%s.npz" % (fs, oc, fi, R6_CFG1_H))
    np.savez_compressed(path, test_idx=test_idx.astype(np.int64), **{c: P[c].astype(np.float32) for c in R6_CRIT})
    return [{"fs": fs, "outcome": oc, "fold": int(fi), "n_bank": int(len(pool_idx)), "n_test": int(len(test_idx)), "npz": path.name, **{"alpha_" + c: float(al[c]) for c in R6_CRIT}}]


def r6_whole_pooled(fs, oc, rows):
    """collects the five-fold npz to make the test predictions of the whole cohort (stored as float32 and read as float: the same as run_matrix_unit of the preceding cells)."""
    n = len(CTX[(fs, oc, "eICU")]["y"])
    oof = {c: np.full(n, np.nan, dtype=np.float32) for c in R6_CRIT}
    for r in rows:
        z = np.load(R6_RES_DIR / r["npz"])
        for c in R6_CRIT:
            oof[c][z["test_idx"]] = z[c]
    if any(np.isnan(v).any() for v in oof.values()):
        return None
    return {c: v.astype(float) for c, v in oof.items()}


def r6_whole_boot(fs, oc, fi_, oi_, c, pooled, y, t):
    """pair difference between the prediction of selection criterion c and the AUROC selection prediction. The AUROC difference is the paired DeLong interval, and the ECE and net benefit differences are percentile intervals of the patient-paired bootstrap with N_BOOT repetitions."""
    _a, _b, dd, var = delong_delta_var(y, pooled[c], pooled["auroc"])
    rng = np.random.default_rng([int(BOOT_SEED), 63, fi_, oi_, 1 if c == "logloss" else 2])
    n = len(y)
    be, bn = np.empty(int(N_BOOT)), np.empty(int(N_BOOT))
    for r in range(int(N_BOOT)):
        i = rng.integers(0, n, n)
        yi = y[i]
        be[r] = r6_ece_fast(yi, pooled[c][i]) - r6_ece_fast(yi, pooled["auroc"][i])
        bn[r] = r6_nb_fast(yi, pooled[c][i], t) - r6_nb_fast(yi, pooled["auroc"][i], t)
    de = r6_ece_fast(y, pooled[c]) - r6_ece_fast(y, pooled["auroc"])
    dn = r6_nb_fast(y, pooled[c], t) - r6_nb_fast(y, pooled["auroc"], t)
    se = float(np.sqrt(var))
    return [{"fs": fs, "outcome": oc, "difference": "%s selection minus AUROC selection" % R6_CRIT_LABEL[c], "criterion": c, "n": int(n), "dAUROC": float(dd), "dAUROC_lo": float(dd - 1.96 * se), "dAUROC_hi": float(dd + 1.96 * se),
             "dECE": float(de), "dECE_lo": float(np.percentile(be, 2.5)), "dECE_hi": float(np.percentile(be, 97.5)),
             "dNB": float(dn), "dNB_lo": float(np.percentile(bn, 2.5)), "dNB_hi": float(np.percentile(bn, 97.5)), "n_boot": int(N_BOOT)}]


# ---- N2: difference between banks of different periods ---------------------------------------------------------------
R6_TABLE4 = {   # (Gain earlier bank, Gain same-period bank, Difference) of Table 4. Compared only on screen, and the csv holds only the difference and the agreement mark
    ("eicu", "year", "AKI_168h"): (0.0152, 0.0173, -0.0021), ("eicu", "year", "mortality_168h"): (0.0387, 0.0376, 0.0011),
    ("mimic4", "strict", "AKI_168h"): (0.0146, 0.0095, 0.0051), ("mimic4", "strict", "mortality_168h"): (0.0067, 0.0101, -0.0034),
    ("mimic4", "majority", "AKI_168h"): (0.0167, 0.0119, 0.0048), ("mimic4", "majority", "mortality_168h"): (0.0094, 0.0065, 0.0029)}
R6_N2_ROW = {("eicu", "year", "AKI_168h"): 0, ("eicu", "year", "mortality_168h"): 1, ("mimic4", "strict", "AKI_168h"): 2, ("mimic4", "strict", "mortality_168h"): 3,
             ("mimic4", "majority", "AKI_168h"): 4, ("mimic4", "majority", "mortality_168h"): 5, ("mimic4", "patient_group", "AKI_168h"): 6, ("mimic4", "patient_group", "mortality_168h"): 7}
R6_DEV3 = {}


def r6_n2_specs():
    n2_eicu, n2_m4 = R6_STATE["n2"]
    specs = []
    if n2_eicu:
        specs += [("eicu", "year", oc) for oc in OUTCOMES]
    if n2_m4:
        per = R5B_PERIODS if R5B_PERIODS is not None else read_mimic4_periods()[0]
        R6_STATE["n2_per"] = per
        rules = ("patient_group",) if "patient_group" in per.columns else ("strict", "majority")
        specs += [("mimic4", rule, oc) for rule in rules for oc in OUTCOMES]
    return specs


def r6_n2_inputs(spec):
    """(context C, past mask, test-period mask, analysis name, reason). The same definition as the A1a and A1b cells of the preceding cells."""
    kind, rule, oc = spec
    if kind == "eicu":
        C = CTX[("F42", oc, "eICU")]
        yr = map_to_stays(EICU_TAB, "year", C["stay_ids"])
        ys_ = sorted(np.unique(yr[np.isfinite(yr)]))
        if len(ys_) < 2:
            return None, None, None, "", "only one year, so no temporal split is possible"
        return C, yr == ys_[0], yr == ys_[-1], "A1a_eICU_%d_to_%d" % (int(ys_[0]), int(ys_[-1])), ""
    d4 = DS[("MIMIC-IV", oc)]
    per = pd.Series(d4["stay_ids"]).map(R6_STATE["n2_per"][rule]).fillna(-1).to_numpy(dtype=int)
    present = [g for g in range(int(per.max()) + 1) if (per == g).any()]
    if len(present) < 3:
        return None, None, None, "", "periods with cohort stays number %d (fewer than 3)" % len(present)
    last = present[-1]
    n_te, ev_te, n_pa = int((per == last).sum()), int(d4["y"][per == last].sum()), int(((per >= 0) & (per < last)).sum())
    if n_te < 4 * N_TIME_SPLITS or ev_te == 0 or ev_te == n_te:
        return None, None, None, "", "test period is empty (stays %d | events %d)" % (n_te, ev_te)
    if n_pa < 50:
        return None, None, None, "", "past bank pool insufficient (past period stays %d < 50)" % n_pa
    if oc not in R6_DEV3:
        d3 = DS[("MIMIC-III", oc)]
        tr3, _va3, _te3 = split_56_14_30(d3["y"])
        R6_DEV3[oc] = build_dev(d3["X_df"].iloc[tr3], d3["y"][tr3], FS_COLS["F42"], CAT_REF, NUM_REF, "A1b|%s" % oc)
    C = make_ctx(R6_DEV3[oc], d4["X_df"], d4["y"], d4["stay_ids"], oc)
    return C, (per >= 0) & (per < last), per == last, "A1b_MIMIC3dev_%s" % rule, ""


def r6_n2_boot(yt, pp, pf, rng, n_boot):
    """list of AUROC differences of (PRAM of P - frozen of P) - (PRAM of S - frozen of S) over n_boot resamples of the test patients with replacement. If the frozen predictions are bit-identical in the two policies, the frozen term cancels and
    equals the difference of the two PRAM AUROCs, otherwise it is computed with four AUROCs (the same quantity as the gain difference of the point estimate). The replicate indices are the same either way."""
    same = bool(np.array_equal(pf["P"], pf["S"]))
    n = len(yt)
    out = []
    for _r in range(int(n_boot)):
        i = rng.integers(0, n, n)
        if len(np.unique(yt[i])) > 1:
            if same:
                out.append(roc_auc_score(yt[i], pp["P"][i]) - roc_auc_score(yt[i], pp["S"][i]))
            else:
                out.append((roc_auc_score(yt[i], pp["P"][i]) - roc_auc_score(yt[i], pf["P"][i])) - (roc_auc_score(yt[i], pp["S"][i]) - roc_auc_score(yt[i], pf["S"][i])))
    return out


def r6_n2_unit(spec):
    kind, rule, oc = spec
    base = {"kind": kind, "rule": rule, "outcome": oc}
    C, past, test, name, why = r6_n2_inputs(spec)
    if C is None:
        return [dict(base, status="not computed", why=why)]
    cap = []
    orig = bank_predictions

    def wrap(C_, bank, test_, key, groups, cb_iter=None):
        P, info = orig(C_, bank, test_, key, groups) if cb_iter is None else orig(C_, bank, test_, key, groups, cb_iter)
        cap.append((key, np.asarray(test_), {m: np.asarray(P[m]) for m in ("pram_b1", "b1_base") if m in P}))
        return P, info

    globals()["bank_predictions"] = wrap            # intercepts the result of bank_predictions called by run_temporal (test predictions per patient). The computation is the same function and the same key as the preceding cells
    try:
        rows, _irows, notes = run_temporal(C, past, test, name, oc, ["full"], 1, ("frozen", "pram"))
    finally:
        globals()["bank_predictions"] = orig
    if not rows or len(cap) != 4:
        return [dict(base, status="not computed", why="the results of policies P and S of the largest bank were not produced: %s (intercepted calls %d)" % (notes[:1], len(cap)))]
    y = C["y"]
    test_ids = np.where(test)[0]
    oof = {pol: {m: np.full(len(y), np.nan, dtype=np.float32) for m in ("pram_b1", "b1_base")} for pol in ("P", "S")}
    for key, te, P in cap:
        pol = key.split("|")[2]
        for m, v in P.items():
            oof[pol][m][te] = v
    yt = y[test_ids]
    pp = {pol: oof[pol]["pram_b1"][test_ids].astype(float) for pol in ("P", "S")}
    pf = {pol: oof[pol]["b1_base"][test_ids].astype(float) for pol in ("P", "S")}
    a_pP, a_pS = float(roc_auc_score(yt, pp["P"])), float(roc_auc_score(yt, pp["S"]))
    a_fP, a_fS = float(roc_auc_score(yt, pf["P"])), float(roc_auc_score(yt, pf["S"]))
    rr = {(r["policy"], r["method"]): r["auroc"] for r in rows}
    gap = max(abs(rr[("P", "pram_b1")] - a_pP), abs(rr[("S", "pram_b1")] - a_pS), abs(rr[("P", "b1_base")] - a_fP), abs(rr[("S", "b1_base")] - a_fS))
    rng = np.random.default_rng([int(BOOT_SEED), 62, R6_N2_ROW[spec]])
    n = len(yt)
    bs = r6_n2_boot(yt, pp, pf, rng, N_BOOT)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return [dict(base, status="ok", name=name, n_past=int(past.sum()), n_test=int(n), n_bank=int(rows[0]["n_bank"]), gain_earlier=a_pP - a_fP, gain_same=a_pS - a_fS,
                 diff=(a_pP - a_fP) - (a_pS - a_fS), lo=float(lo), hi=float(hi), n_boot_used=int(len(bs)), same_as_run_temporal=float(gap), frozen_equal=bool(np.array_equal(pf["P"], pf["S"])))]


# ---- N5: fraction of temperature records -----------------------------------------------------------------
R6_TEMP_LO, R6_TEMP_HI = 30.0, 43.0               # valid range of body temperature (Celsius), the same range as for the temp_c columns of the prepared input tables
R6_WIN_MAIN, R6_WIN_FB = (0.0, 1440.0), (-1440.0, 1440.0)      # 24 hours after admission [0, 1440) minutes . The fallback window starts 24 hours before admission
R6_N5_RX = {"temp": r"\btemp", "not_loc": "locat|site|source|route|method", "fahr_nam": r"\(f\)|fahrenheit|deg f|degf", "fahr_lab": r"\(f\)|fahrenheit",
            "cels_nam": r"\(c\)|celsius|deg c|degc", "cels_lab": r"\(c\)|celsius"}       # rule for judging nurseCharting temperature items (regular expression, applied after lowercasing). It is in the fingerprint of the stored results
R6_N5_FAHR_AMB = (80.0, 115.0)                    # if the unit is not in the name and the value is in this range, it is treated as Fahrenheit
R6_N5_NAMES = {"vital": ["vitalPeriodic.csv.gz", "vitalPeriodic.csv", "vitalperiodic.csv.gz", "vitalperiodic.csv"],
               "nurse": ["nurseCharting.csv.gz", "nurseCharting.csv", "nursecharting.csv.gz", "nursecharting.csv"]}


def r6_n5_find(kind):
    """finds the raw tables. Folder order: the confirmed folder (R6_EICU_RAW_DIRS) , the folders of the candidate paths with which the preceding cells search for the eICU patient table (parent folders of EICU_PATIENT_CANDIDATES | the folder of the patient table found | DATA_ROOT/eicu).
    in each folder, first look at the file name candidates (csv.gz and csv, two case forms), and if absent look for a name that differs only in case in the folder listing. Returns: (path found or None, list of file paths searched)."""
    dirs = [Path(d) for d in R6_EICU_RAW_DIRS]
    if EICU_PATIENT_PATH:
        dirs.append(Path(EICU_PATIENT_PATH).parent)
    dirs += [Path(p).parent for p in EICU_PATIENT_CANDIDATES] + [DATA_ROOT / "eicu", DATA_ROOT / "eICU"]
    seen, cands = set(), []
    for d in dirs:
        if str(d) not in seen:
            seen.add(str(d))
            cands += [d / nm for nm in R6_N5_NAMES[kind]]
    cands += [Path(p) for p in R6_EXTRA_RAW_PATHS["eicu_vitalperiodic" if kind == "vital" else "eicu_nursecharting"]]
    hit, tried = find_first(cands)
    if hit is not None:
        return hit, tried
    want = {nm.lower() for nm in R6_N5_NAMES[kind]}
    for d in dirs:                                                         # names that differ only in case (Linux and some shared folders)
        try:
            for f in sorted(d.iterdir()):
                if f.name.lower() in want and f.is_file():
                    return f, tried
        except OSError:
            continue
    return None, tried


def r6_n5_dirs_report(*tried_lists):
    """counts each folder searched once per folder (not one line per file name candidate). Writes whether the folder opens and the file names that contain vital or nurse (if any). Only file names are written and contents are not read."""
    folders = list(dict.fromkeys(str(Path(t).parent) for tl in tried_lists for t in tl))
    rows = []
    for f in folders:
        try:
            names = sorted(x.name for x in Path(f).iterdir() if ("vital" in x.name.lower() or "nurse" in x.name.lower()))
            rows.append("%s (opened , vital and nurse files: %s)" % (f, names[:6] if names else "absent"))
        except OSError as exc:
            rows.append("%s (did not open: %s)" % (f, type(exc).__name__))
    return rows


def r6_mark(flags, ids_sorted, stay, mask):
    """if the stay of a row where mask is true is in the cohort (ids_sorted), sets flags of that stay to true."""
    s = stay[mask]
    p = np.searchsorted(ids_sorted, s)
    inb = (p < len(ids_sorted)) & (ids_sorted[np.minimum(p, len(ids_sorted) - 1)] == s)
    flags[p[inb]] = True


def r6_n5_save(kind, n_ck, sig, st):
    np.savez_compressed(R6_RES_DIR / ("R6_n5_%s.ck%04d.npz" % (kind, n_ck)), sig=np.array(sig), fm=st["fm"], ff=st["ff"], chunks=np.array(st["chunks"]), rows=np.array(st["rows"]),
                        rows_cohort=np.array(st["rows_cohort"]), done=np.array(bool(st["done"])), names=np.array(json.dumps(st["names"])))


def r6_n5_scan(kind, path, ids_sorted):
    """reads the raw table in chunks and counts, for each cohort stay, the presence of temperature records (valid range) in two windows. Saves in a new file (ck number) every ten chunks to resume reading. Row-level values are not written on screen or in files."""
    hdr = {str(c).lower(): c for c in pd.read_csv(path, nrows=0, encoding_errors="replace").columns}
    need = ["patientunitstayid", "observationoffset", "temperature"] if kind == "vital" else \
        ["patientunitstayid", "nursingchartoffset", "nursingchartcelltypevallabel", "nursingchartcelltypevalname", "nursingchartvalue"]
    miss = [n for n in need if n not in hdr]
    if miss:
        return {"error": "no column: %s . column names of this raw table: %s" % (miss, list(hdr.values())[:30])}
    sig = json.dumps([Path(path).name, int(Path(path).stat().st_size), int(len(ids_sorted)), int(ids_sorted[0]), int(ids_sorted[-1]), R6_N5_CHUNK, R6_TEMP_LO, R6_TEMP_HI, list(R6_WIN_MAIN), list(R6_WIN_FB), R6_N5_RX, list(R6_N5_FAHR_AMB), CODE_VER_MIX, R6_N5_VER], sort_keys=True)
    st = {"fm": np.zeros(len(ids_sorted), dtype=bool), "ff": np.zeros(len(ids_sorted), dtype=bool), "chunks": 0, "rows": 0, "rows_cohort": 0, "done": False, "names": {}}
    cks = sorted(R6_RES_DIR.glob("R6_n5_%s.ck*.npz" % kind))
    for p in reversed(cks):
        try:
            z = np.load(p, allow_pickle=False)
            if str(z["sig"]) != sig:
                continue
            st = {"fm": z["fm"].copy(), "ff": z["ff"].copy(), "chunks": int(z["chunks"]), "rows": int(z["rows"]), "rows_cohort": int(z["rows_cohort"]), "done": bool(z["done"]), "names": json.loads(str(z["names"]))}
            print("  [N5 %s] resumed from stored result %s (%d chunks read%s)" % (kind, p.name, st["chunks"], ", finished" if st["done"] else ""))
            break
        except Exception as exc:  # noqa: BLE001
            print("  [N5 %s] could not read stored result %s and skipped: %s %s" % (kind, p.name, type(exc).__name__, str(exc)[:80]))
    if st["done"]:
        R6_LOG.append(("R6_n5_" + kind, "reused"))
        return st
    n_ck = len(cks)
    cols = [hdr[n] for n in need]
    if kind == "vital":
        it = pd.read_csv(path, usecols=cols, chunksize=R6_N5_CHUNK, encoding_errors="replace")             # does not stop even if there are bytes that are not UTF-8 (the temperature item judgment uses only ASCII characters)
    else:
        it = pd.read_csv(path, usecols=cols, chunksize=R6_N5_CHUNK, encoding_errors="replace", dtype={hdr["nursingchartcelltypevallabel"]: str, hdr["nursingchartcelltypevalname"]: str, hdr["nursingchartvalue"]: str})
    t0 = time.time()
    for ci, ch in enumerate(it):
        if ci < st["chunks"]:
            continue
        stay = pd.to_numeric(ch[hdr["patientunitstayid"]], errors="coerce").to_numpy(dtype=float)
        okid = np.isfinite(stay)
        stay_i = np.where(okid, stay, -1).astype(np.int64)
        pos = np.searchsorted(ids_sorted, stay_i)
        inc = okid & (pos < len(ids_sorted)) & (ids_sorted[np.minimum(pos, len(ids_sorted) - 1)] == stay_i)
        st["rows"] += int(len(ch))
        st["rows_cohort"] += int(inc.sum())
        if inc.any():
            sub = ch[inc]
            s_i = stay_i[inc]
            if kind != "vital":                                                    # keeps only the temperature item rows and does the remaining string computation only on those rows (rows without the text temp are not caught by the \btemp rule below either, so the result is the same)
                _pre = (sub[hdr["nursingchartcelltypevallabel"]].str.contains("temp", case=False, regex=False, na=False)
                        | sub[hdr["nursingchartcelltypevalname"]].str.contains("temp", case=False, regex=False, na=False)).to_numpy()
                sub, s_i = sub[_pre], s_i[_pre]
            if kind == "vital":
                off = pd.to_numeric(sub[hdr["observationoffset"]], errors="coerce").to_numpy(dtype=float)
                val = pd.to_numeric(sub[hdr["temperature"]], errors="coerce").to_numpy(dtype=float)
                good = np.isfinite(val) & (val >= R6_TEMP_LO) & (val <= R6_TEMP_HI)
            else:
                off = pd.to_numeric(sub[hdr["nursingchartoffset"]], errors="coerce").to_numpy(dtype=float)
                lab = sub[hdr["nursingchartcelltypevallabel"]].fillna("").astype(str).str.lower()
                nam = sub[hdr["nursingchartcelltypevalname"]].fillna("").astype(str).str.lower()
                val = pd.to_numeric(sub[hdr["nursingchartvalue"]], errors="coerce").to_numpy(dtype=float)
                has_t = lab.str.contains(R6_N5_RX["temp"]) | nam.str.contains(R6_N5_RX["temp"])            # word boundary: a name such as attempt is not a temperature
                not_loc = ~(lab.str.contains(R6_N5_RX["not_loc"]) | nam.str.contains(R6_N5_RX["not_loc"]))
                is_t = (has_t & not_loc).to_numpy()
                fahr = (nam.str.contains(R6_N5_RX["fahr_nam"]) | lab.str.contains(R6_N5_RX["fahr_lab"])).to_numpy()
                cels = (nam.str.contains(R6_N5_RX["cels_nam"]) | lab.str.contains(R6_N5_RX["cels_lab"])).to_numpy()
                amb_f = ~fahr & ~cels & (val >= R6_N5_FAHR_AMB[0]) & (val <= R6_N5_FAHR_AMB[1])           # if the unit is not in the name and the value is in the Fahrenheit range (80 to 115), it is treated as Fahrenheit
                vc = np.where(fahr | amb_f, (val - 32.0) / 1.8, val)
                good = is_t & np.isfinite(vc) & (vc >= R6_TEMP_LO) & (vc <= R6_TEMP_HI)
                if is_t.any():
                    tab = pd.DataFrame({"lb": lab[is_t].to_numpy(), "nm": nam[is_t].to_numpy(), "num": np.isfinite(val[is_t]), "good": good[is_t]})
                    for (lb, nm), grp in tab.groupby(["lb", "nm"]):
                        e = st["names"].setdefault(lb + " | " + nm, [0, 0, 0])
                        e[0] += int(len(grp))
                        e[1] += int(grp["num"].sum())
                        e[2] += int(grp["good"].sum())
            r6_mark(st["fm"], ids_sorted, s_i, good & (off >= R6_WIN_MAIN[0]) & (off < R6_WIN_MAIN[1]))
            r6_mark(st["ff"], ids_sorted, s_i, good & (off >= R6_WIN_FB[0]) & (off < R6_WIN_FB[1]))
        st["chunks"] = ci + 1
        if st["chunks"] % R6_N5_CK_EVERY == 0:
            n_ck += 1
            r6_n5_save(kind, n_ck, sig, st)
            print("  [N5 %s] %d chunks read (rows %s) , %.0f s" % (kind, st["chunks"], format(st["rows"], ","), time.time() - t0), flush=True)
    st["done"] = True
    r6_n5_save(kind, n_ck + 1, sig, st)
    R6_LOG.append(("R6_n5_" + kind, "computed"))
    return st

In [ ]:
# ================================================================
# PRAM supplementary cell 3/13 . N1 selection criterion of the mixing weight a: AUROC (rule of the main analysis) versus log loss versus Brier (4 hospital-level cells + 4 full-bank eICU cells, resumable)
# ================================================================
R6_N1 = {"state": "", "hosp": pd.DataFrame(), "cells": pd.DataFrame(), "pairs": pd.DataFrame(), "alpha": pd.DataFrame(), "reading": pd.DataFrame(), "checks": pd.DataFrame(),
         "w_cells": pd.DataFrame(), "w_pairs": pd.DataFrame(), "w_checks": pd.DataFrame(), "w_state": ""}


def r6_run_n1():
    if not R6_STATE["fs_hosp"]:
        R6_N1["state"] = "not computed because the input is missing: " + R6_STATE["n1_why"]
        emit("**N1 is not computed because the input is missing.** Reason: " + R6_STATE["n1_why"] + ". What to do: check the eICU patient table (hospital id) line of the input check table of the settings cell.")
        status("additional: N1 hospital level", "not available", "not computed because the input is missing: " + R6_STATE["n1_why"])
        return
    sites, fs_list = R6_STATE["sites"], R6_STATE["fs_hosp"]
    order = [(fs, oc, int(h)) for fs in fs_list for oc in OUTCOMES for h in sites[oc]]
    emit("### N1 hospital level . selection criterion of the mixing weight a (feature set %s , hospitals %s , units %d)" % (" · ".join(fs_list), " · ".join("%s %d hospitals" % (oc, len(sites[oc])) for oc in OUTCOMES), len(order)))
    res = {fs: R6Resumable("R6_n1_" + fs) for fs in fs_list}
    t_loop, n_new, rows_all, bad = time.time(), 0, [], []
    for k, (fs, oc, h) in enumerate(order):
        unit = ("n1", oc, h, R6_CFG1)
        was = res[fs].unit_done(unit)
        rows = r6_unit(res[fs], unit, lambda: r6_hospital_unit(fs, oc, h))
        rows_all += rows
        if not was:
            n_new += 1
            if n_new % 10 == 1 or k == len(order) - 1:
                print("  [N1] %s %s hospital %d/%d done, remaining time about %.0f min (estimated by the number of hospitals)" % (fs, oc, k + 1, len(order), (time.time() - t_loop) / max(n_new, 1) * (len(order) - k - 1) / 60.0), flush=True)
        if rows[0].get("status") == "ok" and (rows[0].get("xcheck_inner") is False or rows[0].get("hpram_match") is False):
            bad.append((fs, oc, h))
            break
    df = pd.DataFrame(rows_all)
    R6_N1["hosp"] = df
    n_ok = int((df["status"] == "ok").sum())
    hm = df["hpram_match"] if "hpram_match" in df else pd.Series([], dtype=object)
    chk = [{"check": "number of hospitals whose AUROC a equals that chosen by inner_cv_bank of the preceding cells", "ok": int(df.loc[df["status"] == "ok", "xcheck_inner"].astype(bool).sum()), "all": n_ok},
           {"check": "number of hospitals equal within 1e-12 to the hospital PRAM test predictions of the preceding cells (R5B_HPRAM)", "ok": int((hm == True).sum()), "all": int(hm.notna().sum())}]  # noqa: E712
    emit("- of %d hospitals, computed this time %d | loaded %d | not eligible %d" % (len(df), n_new, len(df) - n_new, int((df["status"] != "ok").sum())))
    if bad:
        R6_N1["state"] = "reproduction mismatch"
        emit("**N1 reproduction mismatch , stopped.** The AUROC a chosen by the preceding cells or the hospital PRAM test predictions of the preceding cells differ from the values this cell made again (feature set %s , outcome %s). The new values are not used. Save this screen (.ipynb)." % (bad[0][0], bad[0][1]))
        status("additional: N1 hospital level", "reproduction mismatch", "the AUROC selection version differs from the preceding cells")
        R6_N1["checks"] = pd.DataFrame(chk)
        return
    cells, pairs, alpha = r6_n1_cells(df)
    exp_k = dict(R5B_HOSP_BOOT_K)
    for r_ in cells[cells["criterion"] == R6_CRIT_LABEL["auroc"]].itertuples():
        rf = R5B_HOSP_BOOT_REF.get((r_.fs, r_.outcome))
        if rf is None:
            continue
        if r_.stream == R6_STREAM_SAME:
            dmax = max(abs(r_.dAUROC - rf[0]), abs(r_.dAUROC_lo - rf[1]), abs(r_.dAUROC_hi - rf[2]))
            chk.append({"check": "the mean, lower and upper bound of the AUROC (rule of the main analysis) row are within %g of the main-text hospital-level interval of the preceding cells (same random stream) %s %s" % (R5B_HOSP_BOOT_REF_TOL, r_.fs, r_.outcome), "ok": int(dmax <= R5B_HOSP_BOOT_REF_TOL), "all": 1, "maximum difference": dmax})
        elif exp_k.get(r_.outcome) == r_.k:
            chk.append({"check": "the mean of the AUROC (rule of the main analysis) row is within 1e-4 of the main-text hospital-level value (the hospital composition differs so an independent random stream , the interval is not compared) %s %s" % (r_.fs, r_.outcome), "ok": int(abs(r_.dAUROC - rf[0]) <= 1e-4), "all": 1, "maximum difference": abs(r_.dAUROC - rf[0])})
    n_same = int((cells["stream"] == R6_STREAM_SAME).sum() / max(len(R6_CRIT), 1))
    chk.append({"check": "number of cells where the hospital bootstrap random stream is the same as the main text of the preceding cells (the same only when the number and composition of hospitals equal the stored values)", "ok": n_same, "all": int(len(cells) / max(len(R6_CRIT), 1))})
    R6_N1["checks"] = pd.DataFrame(chk)
    emit_table("N1 reproduction check (ok = all is required to use the new values. The last line is the random stream marker and does not stop even if short)", R6_N1["checks"], digits=8)
    if any(r["ok"] != r["all"] for r in chk[:-1]):
        R6_N1["state"] = "reproduction mismatch"
        emit("**N1 reproduction mismatch , the new values are not used.** See the lines where ok differs from all in the table above. Save this screen (.ipynb).")
        status("additional: N1 hospital level", "reproduction mismatch", "see the reproduction check table")
        return
    R6_N1["cells"], R6_N1["pairs"], R6_N1["alpha"] = cells, pairs, alpha
    emit_table("N1 hospital level: change versus frozen per selection criterion (hospital mean is stay-weighted, hospital bootstrap %d repetitions 95%% interval, random stream by the same method as the main-text hospital-level interval of the preceding cells (column stream), a0 = fraction of (hospital, fold) pairs in which a was drawn as 0)" % N_BOOT,
               cells[["fs", "outcome", "k", "stream", "criterion", "dAUROC", "dAUROC_lo", "dAUROC_hi", "dECE", "dECE_lo", "dECE_hi", "dNB", "dNB_lo", "dNB_hi", "a0_share_fold", "a0_share_hospital_all_folds", "alpha_median"]], digits=4, max_rows=40)
    emit_table("N1 hospital level: pair differences between selection criteria (log loss selection minus AUROC selection, Brier selection minus AUROC selection) , 95% interval with the same hospital replicate indices . _half = interval half width", pairs[["fs", "outcome", "k", "difference", "dAUROC", "dAUROC_lo", "dAUROC_hi", "dAUROC_half", "dECE", "dECE_lo", "dECE_hi", "dECE_half", "dNB", "dNB_lo", "dNB_hi", "dNB_half", "a0_share_fold", "a0_share_fold_auroc"]], digits=4, max_rows=40)
    emit("- [note] log loss is sensitive to extreme probabilities, so in hospitals with few events (median events per fold 11 to 13) the selection can be unstable and a can pile up at 0. The a0 column above is that check. "
         "A result of insensitive means the selection criterion does not matter and is not evidence that choosing by net benefit or calibration criteria is better. "
         "A cell whose interval is wider than the band of -0.005 to +0.005 cannot be decided as insensitive and is not evidence that the selection criterion matters.")
    rd = r6_n1_reading(pairs)
    R6_N1["reading"] = rd
    emit_table("N1 reading (rules 1 to 4 applied to each selection criterion; one reading per selection criterion)", rd, digits=4)
    for _ln in r6_n1_lines(rd):
        emit(_ln)
    emit("- [note] The reading above gives the position of the intervals only.")
    r6_save_csv(cells, "R6_N1_hospital_criteria")
    r6_save_csv(pairs, "R6_N1_hospital_pairdiff")
    r6_save_csv(alpha, "R6_N1_hospital_alpha_distribution")
    R6_N1["state"] = "done"
    status("additional: N1 hospital level", "done", "%d hospitals | units %d (computed this time %d | loaded %d)" % (n_ok, len(order), n_new, len(order) - n_new))
    if not R6_RUN_N1_WHOLE:
        R6_N1["w_state"] = "off"
        status("additional: N1 full bank", "off", "R6_RUN_N1_WHOLE=False")
        return
    emit("### N1 eICU full bank . comparison of the same selection criteria (5-fold batch, feature set %s)" % " · ".join(fs_list))
    w_cells, w_pairs, w_chk = [], [], []
    t_w, n_w_new = time.time(), 0
    for fs in fs_list:
        fi_ = R6_FS_ALL.index(fs)
        res_w = R6Resumable("R6_n1w_" + fs)
        res_b = R6Resumable("R6_n1wb_" + fs)
        for oi_, oc in enumerate(OUTCOMES):
            C = CTX[(fs, oc, "eICU")]
            y, t = C["y"], float(T_SITE[oc])
            rows = []
            for fo in range(N_FOLDS):
                was = res_w.unit_done(("n1w", oc, fo, R6_CFG1, N_FOLDS))
                rows += r6_unit(res_w, ("n1w", oc, fo, R6_CFG1, N_FOLDS), lambda: r6_whole_fold(fs, oc, fo), extra_ok=lambda rr: bool(rr) and (R6_RES_DIR / rr[0]["npz"]).is_file())
                if not was:
                    n_w_new += 1
                    print("  [N1 full bank] %s %s fold %d/%d done , elapsed %.0f min" % (fs, oc, fo + 1, N_FOLDS, (time.time() - t_w) / 60.0), flush=True)
            pooled = r6_whole_pooled(fs, oc, rows)
            if pooled is None:
                emit("**N1 full bank %s %s: skipped because the test predictions are empty.**" % (fs, oc))
                continue
            fro = C["pb1"].astype(np.float32).astype(float)
            met = {nm: full_metrics(y, p) for nm, p in (("frozen", fro),) + tuple((c, pooled[c]) for c in R6_CRIT)}
            tcol = nbcol(t)
            mat = R5P_MATRIX.get(fs)
            if mat is not None and len(mat):
                mm = mat[(mat["outcome"] == oc) & (mat["target"] == "eICU") & (mat["bank"].astype(str) == "full") & (mat["method"] == "pram_b1")]
                if len(mm):
                    w_chk.append({"check": "%s %s full-bank AUROC selection PRAM AUROC and ECE are within 1e-9 of the grid value of the preceding cells" % (fs, oc), "ok": int(abs(float(mm["auroc"].iloc[0]) - met["auroc"]["auroc"]) <= 1e-9 and abs(float(mm["ece"].iloc[0]) - met["auroc"]["ece"]) <= 1e-9), "all": 1})
            for c in R6_CRIT:
                al = np.array([r["alpha_" + c] for r in rows], dtype=float)
                w_cells.append({"fs": fs, "outcome": oc, "criterion": R6_CRIT_LABEL[c], "n": int(len(y)), "dAUROC": met[c]["auroc"] - met["frozen"]["auroc"], "dECE": met[c]["ece"] - met["frozen"]["ece"],
                                "dNB": met[c][tcol] - met["frozen"][tcol], "a0_share_fold": float(np.mean(al == 0.0)), "alpha_median": float(np.median(al)), "alphas": " ".join("%g" % a for a in al)})
            for c in ("logloss", "brier"):
                was_b = res_b.unit_done(("n1wb", oc, c, int(N_BOOT), int(BOOT_SEED), R6_CFG1))
                w_pairs += r6_unit(res_b, ("n1wb", oc, c, int(N_BOOT), int(BOOT_SEED), R6_CFG1), lambda: r6_whole_boot(fs, oc, fi_, oi_, c, pooled, y, t))
                if not was_b:
                    print("  [N1 full bank] %s %s %s pair-difference interval done , elapsed %.0f min" % (fs, oc, c, (time.time() - t_w) / 60.0), flush=True)
    if w_cells:
        wc, wp = pd.DataFrame(w_cells), pd.DataFrame(w_pairs)
        R6_N1["w_cells"], R6_N1["w_pairs"], R6_N1["w_checks"] = wc, wp, pd.DataFrame(w_chk)
        if len(w_chk):
            emit_table("N1 full-bank reproduction check (ok = all is required for use)", R6_N1["w_checks"])
        else:
            emit("- there is no full-bank PRAM row in the grid of the preceding cells (R5P_MATRIX), so no reproduction comparison was made (the AUROC selection version was matched with the preceding cells and checked in the four hospital-level cells).")
        if any(r["ok"] != r["all"] for r in w_chk):
            R6_N1["w_state"] = "reproduction mismatch"
            emit("**N1 full-bank reproduction mismatch , the new values are not used.**")
            status("additional: N1 full bank", "reproduction mismatch", "the AUROC selection version differs from the grid of the preceding cells")
        else:
            emit_table("N1 eICU full bank: change versus frozen per selection criterion (single estimate, the intervals are in the pair differences below)", wc, digits=4, max_rows=30)
            emit_table("N1 eICU full bank: pair differences between selection criteria (AUROC with the paired DeLong 95%% interval, ECE and net benefit with the patient-paired bootstrap %d repetitions 95%% interval)" % N_BOOT, wp[["fs", "outcome", "difference", "n", "dAUROC", "dAUROC_lo", "dAUROC_hi", "dECE", "dECE_lo", "dECE_hi", "dNB", "dNB_lo", "dNB_hi"]], digits=4, max_rows=30)
            emit("- hospital-level reading rules (1 to 4) are not applied to the four full-bank cells and only values are written.")
            r6_save_csv(wc, "R6_N1_whole_criteria")
            r6_save_csv(wp, "R6_N1_whole_pairdiff")
            R6_N1["w_state"] = "done"
            status("additional: N1 full bank", "done", "feature sets %d x outcomes %d (folds computed this time %d)" % (len(fs_list), len(OUTCOMES), n_w_new))
    else:
        R6_N1["w_state"] = "not available"
        status("additional: N1 full bank", "not available", "the test predictions are empty")


if not R6_STOP:
    _rep3 = len(REPORT_LINES)
    r6_run_n1()
    R6_LINES["n1"] = [str(_l) for _l in REPORT_LINES[_rep3:]]
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)

In [ ]:
# ================================================================
# PRAM supplementary cell 4/13 . N2 six rows of Table 4: earlier-period bank gain minus same-period bank gain, test-patient paired bootstrap interval (resumable)
# ================================================================
R6_N2 = {"state": "", "rows": pd.DataFrame(), "notes": []}


def r6_n2_lines(known):
    """N2 reading: takes only the computed rows (known) that have the same row in Table 4, and returns the notes first and the readings next."""
    out = []
    low = known[known["lo"] <= -0.01]
    out.append("- [note] (i) This interval is a conditional interval that includes only the variation of the test patient sample. It does not include the variation of the bank composition and of the selection of the mixing weight a, so it is narrower than the real uncertainty. "
         "(ii) Even if the interval includes 0, it does not mean equivalence (the non-inferiority margin 0.01 is a post hoc definition). "
         "(iii) A row whose lower bound is -0.01 or below may mean that this test sample cannot show it is within -0.01, and not that the earlier bank is lower (the interval half width is in the half column of the table above). "
         "for the death row, whose direction split by rule (strict versus majority), the intervals of the two rules are read separately (the mortality_168h row of the table above).")
    if len(known) == 6 and len(low) == 0:
        out.append("- [rule N2] the interval lower bounds of all six rows are greater than -0.01 (lowest lower bound %+.4f)." % float(known["lo"].min()))
    elif len(low):
        out.append("- [rule N2] %d rows whose interval lower bound is -0.01 or below: %s . For these rows the earlier-period bank may be lower than the same-period bank. The lower bounds of the remaining %d computed rows are greater than -0.01."
             % (len(low), "; ".join("%s %s %s (lower bound %+.4f)" % (a, b, c, d) for a, b, c, d in zip(low["kind"], low["rule"], low["outcome"], low["lo"])), len(known) - len(low)))
    elif len(known):
        out.append("- [rule N2] the interval lower bounds of the %d computed rows are all greater than -0.01 (lowest lower bound %+.4f). The remaining rows were not computed because the input is missing." % (len(known), float(known["lo"].min())))
    return out


def r6_run_n2():
    specs = r6_n2_specs()
    emit("### N2 bootstrap interval of the difference between banks of different periods (largest bank of Table 4, test patients resampled with replacement %d times)" % N_BOOT)
    if not specs:
        R6_N2["state"] = "not computed because the input is missing"
        emit("**N2 is not computed because the input is missing.** There is neither an eICU discharge year column nor a MIMIC-IV period table. What to do: check the eICU patient table (hospitaldischargeyear) and the MIMIC-IV patients and icustays lines of the input check table of the settings cell.")
        status("additional: N2", "not available", "not computed because the input is missing")
        return
    res = R6Resumable("R6_n2")
    rows_all, t_n2 = [], time.time()
    for spec in specs:
        unit = ("n2",) + spec + (int(N_BOOT), int(BOOT_SEED), int(SEED), int(N_TIME_SPLITS), "".join(A1_POLICIES), CODE_VER_MIX)
        was = res.unit_done(unit)
        rows_all += r6_unit(res, unit, lambda: r6_n2_unit(spec))
        if not was:
            print("  [N2] %s done , elapsed %.0f min" % (spec, (time.time() - t_n2) / 60.0), flush=True)
    df = pd.DataFrame(rows_all)
    skipped = df[df["status"] != "ok"]
    ok = df[df["status"] == "ok"].copy()
    for r_ in skipped.itertuples():
        emit("- N2 %s %s %s: not computed , reason %s" % (r_.kind, r_.rule, r_.outcome, r_.why))
    if not len(ok):
        R6_N2["state"] = "not available"
        status("additional: N2", "not available", "there is no computed row")
        return
    ref = [R6_TABLE4.get((k_, r_, o_)) for k_, r_, o_ in zip(ok["kind"], ok["rule"], ok["outcome"])]
    ok["table4_known"] = [r is not None for r in ref]
    ok["abs_err_gain_earlier_vs_table4"] = [abs(g - r[0]) if r is not None else np.nan for g, r in zip(ok["gain_earlier"], ref)]
    ok["abs_err_gain_same_vs_table4"] = [abs(g - r[1]) if r is not None else np.nan for g, r in zip(ok["gain_same"], ref)]
    ok["abs_err_diff_vs_table4"] = [abs(g - r[2]) if r is not None else np.nan for g, r in zip(ok["diff"], ref)]
    ok["reproduced"] = [bool(r is not None and a <= 1e-4 and b <= 1e-4 and c <= 1e-4) if r is not None else None for r, a, b, c in zip(ref, ok["abs_err_gain_earlier_vs_table4"], ok["abs_err_gain_same_vs_table4"], ok["abs_err_diff_vs_table4"])]
    ok["half"] = (ok["hi"] - ok["lo"]) / 2.0
    show = ok[["kind", "rule", "outcome", "name", "n_past", "n_test", "n_bank", "gain_earlier", "gain_same", "diff", "lo", "hi", "half", "n_boot_used", "abs_err_diff_vs_table4", "reproduced"]]
    bad = ok[ok["reproduced"] == False]  # noqa: E712
    emit_table("N2 six rows of Table 4: earlier-period bank gain (gain_earlier) , same-period bank gain (gain_same) , difference (diff) and the test-patient paired bootstrap 95% interval (lo , hi , half = interval half width) . reproduced = within 1e-4 of the Table 4 value", show, digits=4, max_rows=20)
    for r_ in ok[ok["table4_known"]].itertuples():
        t4 = R6_TABLE4[(r_.kind, r_.rule, r_.outcome)]
        print("  [screen comparison] %s %s %s: this run gain_earlier %+.4f gain_same %+.4f diff %+.4f , Table 4 %+.4f %+.4f %+.4f" % (r_.kind, r_.rule, r_.outcome, r_.gain_earlier, r_.gain_same, r_.diff, t4[0], t4[1], t4[2]))
    if len(bad):
        R6_N2["state"] = "reproduction mismatch"
        emit("**N2 reproduction mismatch , the interval is not used.** %d rows whose point estimate is not equal to the Difference of Table 4 within 1e-4. Save this screen (.ipynb)." % len(bad))
        status("additional: N2", "reproduction mismatch", "%d rows not equal to Table 4 within 1e-4" % len(bad))
        R6_N2["rows"] = show
        return
    unknown = ok[~ok["table4_known"]]
    if len(unknown):
        emit("- note: %d rows have no row of the same rule in Table 4 (%s), so the reproduction of the point estimate could not be checked. The intervals of these rows are not used." % (len(unknown), sorted(set(unknown["rule"]))))
    R6_N2["rows"] = show
    emit("- the degree to which the per-policy AUROC the same as run_temporal of the preceding cells came out again from the test predictions intercepted by this cell (maximum difference): %.2e . The frozen model prediction is the same in the two policies: %s (if there are rows that are not the same, the interval of that row was computed with four AUROCs)" % (float(ok["same_as_run_temporal"].max()), bool(ok["frozen_equal"].all())))
    known = ok[ok["table4_known"]]
    for _ln in r6_n2_lines(known):
        emit(_ln)
    emit("- [note] The reading above gives the position of the intervals only.")
    r6_save_csv(show, "R6_N2_period_pairs")
    R6_N2["state"] = "done"
    status("additional: N2", "done", "%d rows (Table 4 reproduced %d)" % (len(ok), int(ok["reproduced"].fillna(False).sum())))


if not R6_STOP:
    _rep4 = len(REPORT_LINES)
    r6_run_n2()
    R6_LINES["n2"] = [str(_l) for _l in REPORT_LINES[_rep4:]]
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)

In [ ]:
# ================================================================
# PRAM supplementary cell 5/13 . N5 fraction of temperature records (vitalPeriodic versus nurseCharting) (resumable)
# ================================================================
def r6_n5_pieces(tab, lim=5):
    """pieces to check whether they are 1 to lim-1 in the N5 fraction table: (name, value, formula, cohort coefficient). The formula is {(window number, table): coefficient}. Four pieces per window (only in vitalPeriodic | only in nurseCharting | in both | in neither) ,
    the difference of the two windows per table , the difference of the two windows of the in-both piece. The formula is written down to compute which cell, if shown, resolves a piece (linear relation)."""
    c = "stays with a temperature record"
    wins = list(dict.fromkeys(tab["window"]))
    pieces = []
    for wi, w in enumerate(wins):
        d = tab[tab["window"] == w].set_index("table")
        n = int(d["cohort stays"].iloc[0])
        v = int(d.loc["vitalPeriodic", c])
        if "nurseCharting" in d.index:
            u, un = int(d.loc["nurseCharting", c]), int(d.loc["union", c])
            pieces += [("%s: only in vitalPeriodic" % w, un - u, {(wi, "union"): 1, (wi, "nurseCharting"): -1}, 0),
                       ("%s: only in nurseCharting" % w, un - v, {(wi, "union"): 1, (wi, "vitalPeriodic"): -1}, 0),
                       ("%s: in both" % w, v + u - un, {(wi, "vitalPeriodic"): 1, (wi, "nurseCharting"): 1, (wi, "union"): -1}, 0),
                       ("%s: in neither" % w, n - un, {(wi, "union"): -1}, 1)]
        else:
            pieces.append(("%s: no record in vitalPeriodic" % w, n - v, {(wi, "vitalPeriodic"): -1}, 1))
    for wi, (a_, b_) in enumerate(zip(wins[:-1], wins[1:])):                  # the later window contains the earlier window (from 24 hours before admission includes 24 hours after admission): monotonicity in which counts can only drop out of the window
        da, db = tab[tab["window"] == a_].set_index("table"), tab[tab["window"] == b_].set_index("table")
        for t_ in da.index:
            if t_ in db.index:
                pieces.append(("%s: difference of the two windows" % t_, int(db.loc[t_, c]) - int(da.loc[t_, c]), {(wi + 1, t_): 1, (wi, t_): -1}, 0))
        if "nurseCharting" in da.index and "nurseCharting" in db.index:      # the difference of the two windows of the in-both piece
            both_ = lambda d_: int(d_.loc["vitalPeriodic", c]) + int(d_.loc["nurseCharting", c]) - int(d_.loc["union", c])
            f = {}
            for wk, sg in ((wi + 1, 1), (wi, -1)):
                for t_, cf in (("vitalPeriodic", 1), ("nurseCharting", 1), ("union", -1)):
                    f[(wk, t_)] = f.get((wk, t_), 0) + sg * cf
            pieces.append(("in both: difference of the two windows", both_(db) - both_(da), f, 0))
    return pieces, wins


def r6_n5_plan(tab, lim=5):
    """plan for masking the count cells of the N5 fraction table. The set of cells to mask is one rule independent of values: if any cell whose value is 1 to lim-1 or any piece whose value is 1 to lim-1 (sum and difference relations, r6_n5_pieces) exists, all count cells of the table are written as masked, and otherwise the numbers are given as they are.
    So the only thing the masking pattern tells is that a value of 1 to lim-1 exists somewhere (which cell or piece is not in the pattern). The returned value is a dict: need (is there anything to mask) | masked (masked cells) | verified (whether the check that the fractions do not narrow the cell values passed).
    verified is an interval computation and not an enumeration: for each cell it finds the minimum and maximum integers consistent with the fraction on screen (first decimal place) and checks whether the pieces involved (and cells whose value is 1 to lim-1) cover 1 and lim-1 on the product of those intervals (if the cohort is small and the fractions narrow the cell values, it does not cover). The interval computation does not use the non-negativity conditions between pieces, so a reader who uses those conditions as well could narrow further (limitation)."""
    c = "stays with a temperature record"
    pieces, wins = r6_n5_pieces(tab, lim)
    cells = [(wi, t_) for wi, w in enumerate(wins) for t_ in tab[tab["window"] == w]["table"]]
    n = int(tab["cohort stays"].iloc[0])
    true = {k: int(tab[(tab["window"] == wins[k[0]]) & (tab["table"] == k[1])][c].iloc[0]) for k in cells}
    hit_p = [p for p in pieces if 0 < int(p[1]) < lim]
    forced = [k for k in cells if 0 < true[k] < lim]
    plan = {"need": bool(hit_p or forced), "masked": [], "verified": True}
    if not plan["need"]:
        return plan
    plan["masked"] = list(cells)
    grid = np.arange(n + 1, dtype=np.int64)
    exact = np.round(100.0 * (grid / n), 1)
    box = {}
    for k in cells:
        pct = float(np.round(float(tab[(tab["window"] == wins[k[0]]) & (tab["table"] == k[1])]["fraction (%)"].iloc[0]), 1))
        d = np.union1d(grid[np.abs(exact - pct) < 1e-9], [true[k]])
        box[k] = (int(d.min()), int(d.max()))
    ok = all(box[k][0] <= 1 and box[k][1] >= lim - 1 for k in forced)
    for p in hit_p:
        lo = hi = int(p[3]) * n
        for k, cf in p[2].items():
            lo += cf * (box[k][0] if cf > 0 else box[k][1])
            hi += cf * (box[k][1] if cf > 0 else box[k][0])
        ok = ok and lo <= 1 and hi >= lim - 1
    plan["verified"] = bool(ok)
    return plan


def r6_n5_guard(tab, lim=5):
    """masking of the count cells of the N5 fraction table. If there is something to mask according to the plan (r6_n5_plan), all count cells are masked (a cell whose value is 1 to lim-1 gets the same word), the fractions are reduced to the first decimal place, and a masking check column is added. If there is nothing to mask it is as it is.
    returns: (table to export, info dict). If there is nothing to mask the info is an empty dict, and otherwise {verified: whether the check passed}. The names, number and positions of the pieces involved are not returned. The caller decides the fractions used for the decision (reproduction check and the 20-point threshold) (for a table with masking, the rounded fractions put on screen)."""
    c = "stays with a temperature record"
    plan = r6_n5_plan(tab, lim)
    out = tab.copy()
    if not plan["need"]:
        return out, {}
    out[c] = "masked"
    out["fraction (%)"] = out["fraction (%)"].round(1)
    out["masking check"] = "passed" if plan["verified"] else "not passed"
    return out, {"verified": bool(plan["verified"])}


R6_N5 = {"state": "", "table": pd.DataFrame(), "items": pd.DataFrame(), "window": ""}


def r6_run_n5():
    emit("### N5 fraction of stays with a temperature record (descriptive statistics: the model input does not change and the model is not rerun)")
    if not ("eICU", OUTCOMES[0]) in DS:
        R6_N5["state"] = "not computed because the input is missing"
        emit("**N5 is not computed because the input is missing.** The eICU cohort is not in the kernel of the preceding cells.")
        status("additional: N5", "not available", "not computed because the input is missing: no eICU cohort")
        return
    ids = np.unique(np.asarray(DS[("eICU", OUTCOMES[0])]["stay_ids"], dtype=np.int64))
    ids_m = np.unique(np.asarray(DS[("eICU", OUTCOMES[1])]["stay_ids"], dtype=np.int64)) if ("eICU", OUTCOMES[1]) in DS else ids
    kern = DS[("eICU", OUTCOMES[0])]["X_df"]
    kmiss = 100.0 * float(kern["temp_c_last"].isna().mean()) if "temp_c_last" in kern.columns else float("nan")
    emit("- cohort stays %s (by the stay ids of the AKI cohort, with the death cohort %s) . Missingness of temperature (temp_c_last) of the input X of the preceding cells %.1f%% (reference 88.9%%)" % (format(len(ids), ","), "same" if np.array_equal(ids, ids_m) else "different (%d versus %d)" % (len(ids), len(ids_m)), kmiss))
    vpath, tried_v = r6_n5_find("vital")
    npath, tried_n = r6_n5_find("nurse")
    emit("- raw table locations: vitalPeriodic %s , nurseCharting %s" % (Path(vpath).as_posix() if vpath else "not found", Path(npath).as_posix() if npath else "not found"))
    if vpath is None and npath is None:
        R6_N5["state"] = "not computed because the input is missing"
        _fr = r6_n5_dirs_report(tried_v, tried_n)
        emit("**N5 is not computed because the input is missing.** Folders searched: %d: %s . What to do: write the folder holding the vitalPeriodic and nurseCharting raw tables (csv.gz) in `R6_EICU_RAW_DIRS` in the first cell of this file, or write the file path in `R6_EXTRA_RAW_PATHS`, and run again from the first cell." % (len(_fr), " ; ".join(_fr)))
        status("additional: N5", "not available", "not computed because the input is missing: raw table not found (R6_EXTRA_RAW_PATHS)")
        return
    st = {}
    for kind, path in (("vital", vpath), ("nurse", npath)):
        if path is None:
            emit("- the %s raw table was not found, so that table was not computed." % kind)
            continue
        t0 = time.time()
        st[kind] = r6_n5_scan(kind, path, ids)
        if "error" in st[kind]:
            emit("**N5 could not read the %s raw table.** %s" % (kind, st[kind]["error"]))
            continue
        emit("- %s raw table: rows %s , rows of cohort stays %s , %.0f s" % (kind, format(st[kind]["rows"], ","), format(st[kind]["rows_cohort"], ","), time.time() - t0))
    good = {k: v for k, v in st.items() if "error" not in v}
    if "vital" not in good:
        R6_N5["state"] = "reproduction check not possible"
        emit("**N5 reproduction check not possible.** The vitalPeriodic raw table is absent or could not be read, so it cannot be checked against the cohort temperature input missingness 88.9%. The value is not used. What to do: write the vitalPeriodic path in `R6_EXTRA_RAW_PATHS` and run again from the first cell.")
        status("additional: N5", "not available", "no vitalPeriodic: reproduction check not possible")
        return
    n = len(ids)
    rows = []
    for win, key in (("after admission, first 24 hours [0, 1440) minutes", "fm"), ("before admission, from 24 hours before [-1440, 1440) minutes (fallback window)", "ff")):
        v = good["vital"][key]
        rows.append({"window": win, "table": "vitalPeriodic", "cohort stays": n, "stays with a temperature record": int(v.sum()), "fraction (%)": 100.0 * float(v.mean())})
        if "nurse" in good:
            u = good["nurse"][key]
            rows.append({"window": win, "table": "nurseCharting", "cohort stays": n, "stays with a temperature record": int(u.sum()), "fraction (%)": 100.0 * float(u.mean())})
            rows.append({"window": win, "table": "union", "cohort stays": n, "stays with a temperature record": int((v | u).sum()), "fraction (%)": 100.0 * float((v | u).mean())})
    tab = pd.DataFrame(rows)
    tab_out, n5_hit = r6_n5_guard(tab)                                      # tables to export (md and csv). The judgment below uses tab before masking
    dg = 1 if n5_hit else 2
    if n5_hit:
        emit("- the fractions were reduced to the first decimal place and all six count cells were masked: a rule independent of values that masks all count cells of the table if any cell whose value is 1 to 4 exists or any piece whose value is 1 to 4 exists in the sum relations or the difference of the two windows (if no piece is involved, the numbers are given as they are). The names and positions of the pieces involved are not written. What the masking pattern tells is only that a value of 1 to 4 exists somewhere. The 20-point threshold judgment and the choice of the judgment window were made with the rounded fractions put on screen.")
        if not n5_hit["verified"]:
            emit("- **the masking check did not pass**: the cohort is small and the first-decimal fractions narrow the cell values, so it could not be confirmed by the interval computation that all of the 1 to 4 of the pieces involved remain. The fractions of this table must not be used or published (also marked in the masking check column of the csv).")
    rv_ = (lambda x_: float(np.round(float(x_), 1))) if n5_hit else float             # for a table with masking, only the rounded fractions put on screen are used for the judgment window choice, the threshold judgment and the gap line (if the gap were computed with the integer fractions before masking, the masked cells would narrow)
    pm = rv_(tab[(tab["table"] == "vitalPeriodic")]["fraction (%)"].iloc[0])
    pf = rv_(tab[(tab["table"] == "vitalPeriodic")]["fraction (%)"].iloc[1])
    ref_k = (100.0 - kmiss) if np.isfinite(kmiss) else 11.1                 # reference value: the fraction with temperature counted by the kernel from the same input. Only when there is no counted value, the reference value 11.1 (= 100 - 88.9)
    ref_src = "value counted from the temp_c_last missingness of the input X of the preceding cells" if np.isfinite(kmiss) else "manuscript value 11.1 (there is no temp_c_last column in the kernel, so there is no counted value)"
    if np.isfinite(kmiss) and abs(ref_k - 11.1) > 0.1:
        emit("- **to note first**: the fraction with temperature counted from the input of the preceding cells, %.2f%%, differs by more than 0.1 points from the 11.1%% (= 100 - 88.9) of the reference value. The denominator of the reference value 88.9%% may differ from this input. The reproduction check uses the value counted in the kernel." % ref_k)
    use = "fm" if abs(pm - ref_k) <= 0.5 else ("ff" if abs(pf - ref_k) <= 0.5 else "")
    chk = pd.DataFrame([{"check": "the vitalPeriodic fraction is within +-0.5 points of the reference value (%s): window of 24 hours after admission" % ref_src, "reference value (%)": ref_k, "value (%)": pm, "ok": int(abs(pm - ref_k) <= 0.5)},
                        {"check": "the same check: fallback window from 24 hours before admission", "reference value (%)": ref_k, "value (%)": pf, "ok": int(abs(pf - ref_k) <= 0.5)},
                        {"check": "for reference (not a judgment): is the reference value the same as 11.1 (= 100 - 88.9) within 0.1 points", "reference value (%)": ref_k, "value (%)": 11.1, "ok": int(abs(ref_k - 11.1) <= 0.1)}])
    R6_N5["table"] = tab
    items = pd.DataFrame([{"item (label / valname)": " / ".join(k.split(" | ")), "number of rows": r6_small(v[0]), "numeric rows": r6_small(v[1]), "rows in the valid range": r6_small(v[2])} for k, v in sorted(good.get("nurse", {"names": {}})["names"].items(), key=lambda kv: -kv[1][0])]) if "nurse" in good else pd.DataFrame()
    R6_N5["items"] = items
    emit_table("N5 reproduction check (does the vitalPeriodic fraction match the fraction with temperature counted from the same input)", chk, digits=dg)
    if "nurse" in good:
        emit_table("names of items counted as temperature in nurseCharting (with counts, position and site items excluded , only %d to %d degrees valid after Celsius conversion)" % (R6_TEMP_LO, R6_TEMP_HI), items, digits=0, max_rows=40)
    if not use:
        R6_N5["state"] = "reproduction mismatch"
        fp_ = lambda x_: ("%." + str(dg) + "f%%") % x_                           # a masked run uses the first decimal place
        emit("**N5 reproduction mismatch , the values are not used.** The vitalPeriodic fraction is outside +-0.5 points of the reference value %s in both windows (%s | %s). Save this screen (.ipynb)." % (fp_(ref_k), fp_(pm), fp_(pf)))
        emit_table("N5 fractions (not used because the reproduction check did not pass)", tab_out, digits=dg)
        status("additional: N5", "reproduction mismatch", "vitalPeriodic %s differs from the reference value %s" % (fp_(pm), fp_(ref_k)))
        return
    R6_N5["window"] = "after admission, first 24 hours" if use == "fm" else "before admission, from 24 hours before (fallback window)"
    emit_table("N5 fraction of stays with a temperature record (cohort %s stays)" % format(n, ","), tab_out, digits=dg)
    sel = tab[tab["window"].str.startswith("after admission") if use == "fm" else tab["window"].str.startswith("before admission")]
    vv = rv_(sel[sel["table"] == "vitalPeriodic"]["fraction (%)"].iloc[0])
    if "nurse" in good:
        nn = rv_(sel[sel["table"] == "nurseCharting"]["fraction (%)"].iloc[0])
        uu = rv_(sel[sel["table"] == "union"]["fraction (%)"].iloc[0])
        gap = float(np.round(max(nn, uu) - vv, 1)) if n5_hit else max(nn, uu) - vv
        emit("- [rule N5, window: %s] vitalPeriodic %.1f%% , nurseCharting %.1f%% , union %.1f%% . nurseCharting or the union is %.1f points higher than vitalPeriodic (threshold 20 points)." % (R6_N5["window"], vv, nn, uu, gap))
        if gap >= 20.0:
            emit("- reading A (20 points or more): temperature was taken only from vitalPeriodic of the two tables; the fraction of stays with a record within 24 hours is vitalPeriodic %.1f%%, nurseCharting %.1f%%, union %.1f%%." % (vv, nn, uu))
        else:
            emit("- reading B (less than 20 points): vitalPeriodic %.1f%%, nurseCharting %.1f%%, union %.1f%%." % (vv, nn, uu))
    else:
        emit("- the nurseCharting raw table is absent, so only vitalPeriodic %.1f%% was computed. What to do: write the nurseCharting path in `R6_EXTRA_RAW_PATHS` and run again from the first cell." % vv)
    emit("- [note] this fraction is not evidence of an artifact. The model is not refitted.")
    emit("- [note] if the union fraction is high, merging the two tables as the input would be a separate analysis (not done here).")
    r6_save_csv(tab_out, "R6_N5_temperature_records")
    if len(items):
        r6_save_csv(items, "R6_N5_nurse_items")
    R6_N5["state"] = "done" if "nurse" in good else "partial (no nurseCharting)"
    status("additional: N5", "done" if "nurse" in good else "partial", "window %s . vitalPeriodic %.1f%%" % (R6_N5["window"], vv))


if not R6_STOP:
    _rep5 = len(REPORT_LINES)
    r6_run_n5()
    R6_LINES["n56"] = [str(_l) for _l in REPORT_LINES[_rep5:]]
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)

In [ ]:
# ================================================================
# PRAM supplementary cell 6/13 (A) . functions + hospital level (mixing weight grid 0 to 1.00 | fixed 0.50 arm, resumable)
# reading rules of this part. All new names start with R7_.
# ================================================================
R7_ARMS = ("A50", "A100", "L50", "L100", "F50")
R7_SEL_ARMS = ("A50", "A100", "L50", "L100")
R7_ARM_LABEL = {"A50": "AUROC selection (0 to 0.50)", "A100": "AUROC selection (0 to 1.00)", "L50": "log loss selection (0 to 0.50)", "L100": "log loss selection (0 to 1.00)", "F50": "fixed weight 0.50"}
R7_ARM_CRIT = {"A50": "auroc", "A100": "auroc", "L50": "logloss", "L100": "logloss"}      # names of the selection criteria in the stored table (used for the reproduction check)
R7_PAIRS = (("P1", "A100", "A50", "upper end of the grid, AUROC selection"), ("P2", "L100", "L50", "upper end of the grid, log loss selection"),
            ("P3", "L50", "F50", "selection rule, log loss selection versus fixed 0.50"), ("P4", "A50", "F50", "selection rule, AUROC selection versus fixed 0.50, descriptive"))
if R6_STOP:                                                         # if it is not the kernel of the preceding cells (if a required name is absent), the definitions below that use the stored names are skipped (the cell ends without computing)
    R7_GRID, R7_GRID50, R7_CFG1, R7_CFG1_H = [], [], "", ""
else:
    R7_GRID = [float(a) for a in ALPHA_GRID] + [round(0.05 * i, 2) for i in range(11, 21)]          # the first 11 are the ALPHA_GRID of the preceding cells as it is, the last 10 are 0.55 to 1.00
    R7_GRID50 = R7_GRID[:11]
    R7_CFG1 = "alpha21|inner%d|site%d|seed%d|k%d|t%s|code%s|thr%d" % (N_INNER, N_SITE_SPLITS, SEED, K_NEIGHBORS, sorted(T_SITE.items()), CODE_VER_ARMS, int(N_THREADS))      # if the number of threads changes, the stored results of this version are not read and are recomputed (to compare the 1e-12 reproduction with the stored results under the same condition)
    R7_CFG1_H = hashlib.sha256(R7_CFG1.encode("utf-8")).hexdigest()[:8]
R7_LINES = {"n1": [], "w": [], "sb": []}
R7_CHECK_GRID = bool(not R6_STOP and len(ALPHA_GRID) == 11 and abs(R7_GRID[10] - 0.5) < 1e-12 and abs(R7_GRID[20] - 1.0) < 1e-12 and all(abs(R7_GRID[i] - 0.05 * i) < 1e-12 for i in range(21)))


def r7_save_csv(df, name):
    assert name.startswith("R7_"), "names not starting with R7_ are not written: " + name
    path = R6_CSV_DIR / (name + ".csv")
    df.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.6g")
    emit("- saved `%s` (%d rows)" % (path.as_posix(), len(df)))
    return path


R7_REPRO = {"cells": [], "q": {}}      # per-cell differences of the reproduction check: (level, feature set, outcome, metric, arm, unit (hospital id or fold number: only counted and not written to files), absolute difference)


def r7_label(dmax):
    """judgment of the difference of a metric cell."""
    return "within 1e-12" if dmax <= R7_TOL_EXACT else ("within thread-order noise" if dmax <= R7_TOL_NOISE else "over 1e-6: not read")


def r7_repro_table():
    """collects the per-cell differences of the reproduction check by (level, feature set, outcome, metric, arm) (number of mismatched cells, number of mismatched units, maximum difference, median). Hospital ids and fold numbers are only counted and not exported."""
    cols = ["level", "fs", "outcome", "quantity", "arm", "n_cells", "cells_diff_gt_1e-12", "cells_diff_gt_1e-6", "queries_diff_gt_1e-6", "units_differing", "max_diff", "median_diff_of_differing_cells", "median_diff_all_cells", "verdict"]
    if not R7_REPRO["cells"]:
        return pd.DataFrame(columns=cols)
    df = pd.DataFrame(R7_REPRO["cells"], columns=["level", "fs", "outcome", "metric", "arm", "unit", "d"])
    rows = []
    for (lv, fs, oc, me, arm), g in df.groupby(["level", "fs", "outcome", "metric", "arm"], sort=False):
        d = g["d"].to_numpy(float)
        over = d > R7_TOL_EXACT
        row = {"level": lv, "fs": fs, "outcome": oc, "quantity": me, "arm": arm, "n_cells": int(len(d)), "cells_diff_gt_1e-12": int(over.sum()), "cells_diff_gt_1e-6": int((d > R7_TOL_NOISE).sum()),
               "queries_diff_gt_1e-6": sum(v_ for k_, v_ in R7_REPRO["q"].items() if k_[:3] == (fs, oc, arm)) if (lv == "whole bank" and me == "test predictions (largest difference by fold)") else "",
               "units_differing": int(g.loc[over, "unit"].nunique()), "max_diff": float(d.max()), "median_diff_of_differing_cells": float(np.median(d[over])) if over.any() else 0.0, "median_diff_all_cells": float(np.median(d))}
        row["verdict"] = ("identical" if not over.any() else "different: not read") if me == "alpha_list" else r7_label(row["max_diff"])
        rows.append(row)
    return pd.DataFrame(rows, columns=cols)


def r7_repro_verdict(level):
    t = r7_repro_table()
    t = t[t["level"] == level]
    if not len(t):
        return "no comparison"
    if t["verdict"].isin(["over 1e-6: not read", "different: not read"]).any():
        return "over 1e-6: not read"
    return "within thread-order noise" if (t["verdict"] == "within thread-order noise").any() else "within 1e-12"


def r7_repro_report(level, show=True):
    """prints the mismatch summary on screen and rewrites it as R7_repro_diff_summary.csv (all levels combined). There are no hospital ids or patient-level values."""
    t = r7_repro_table()
    shown = t[t["level"] == level].copy()
    for c_ in ("max_diff", "median_diff_of_differing_cells", "median_diff_all_cells"):
        shown[c_] = ["%.2e" % v for v in shown[c_]]
    if show:
        emit_table("reproduction check per-cell difference summary (%s . stored results versus the values of this run . 1e-12 or below = same , over 1e-12 and up to 1e-6 = within thread-order noise , over 1e-6 = not read . number of mismatched units = number of hospitals with a cell over 1e-12 (hospital level) or number of folds (full bank) . there are no hospital ids)" % level, shown, digits=0, max_rows=80)
    r7_save_csv(t, "R7_repro_diff_summary")
    v = r7_repro_verdict(level)
    emit("- reproduction judgment (%s): **%s** (maximum difference %.2e). Within thread-order noise is a marker that it exceeds 1e-12 but is within 1e-6 and the value is read as it is. Over 1e-6 is not read." % (level, v, float(t[t["level"] == level]["max_diff"].max()) if len(t[t["level"] == level]) else float("nan")))
    return v


def r7_inner_scores(Xb, yb, p1b, key):
    """in the same folds as r6_inner_scores (StratifiedKFold N_INNER, random_state = stable_seed(innercv, b1| + key)) and the same retrieval predictions, collects the AUROC and log loss per fold at each of the 21 grid points. None if the bank is too small."""
    n = len(yb)
    if n < N_INNER * 5 or len(np.unique(yb)) < 2:
        return None
    skf = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=stable_seed("innercv", "b1|" + key))
    sc = {a: {"auroc": [], "logloss": []} for a in R7_GRID}
    for itr, iva in skf.split(np.zeros(n), yb):
        pr = RETRIEVAL_FN({"X_tr_s": Xb[itr], "y_tr": yb[itr], "X_te_s": Xb[iva], "y_te": yb[iva], "mi": None, "mi_prob": None})
        if len(np.unique(yb[iva])) > 1:
            for a in R7_GRID:
                p = (1 - a) * p1b[iva] + a * pr
                sc[a]["auroc"].append(roc_auc_score(yb[iva], p))
                sc[a]["logloss"].append(_logloss(yb[iva], p))
    return sc


def r7_pick(sc, crit, grid):
    """the same formula as r6_pick (AUROC maximum, log loss minimum, the earlier a on ties) within the given grid."""
    if sc is None:
        return 0.0
    if crit == "auroc":
        return max(grid, key=lambda a: np.nanmean(sc[a]["auroc"]) if sc[a]["auroc"] else -1)
    return min(grid, key=lambda a: np.nanmean(sc[a][crit]) if sc[a][crit] else np.inf)


def r7_alphas(sc):
    return {"A50": r7_pick(sc, "auroc", R7_GRID50), "A100": r7_pick(sc, "auroc", R7_GRID), "L50": r7_pick(sc, "logloss", R7_GRID50), "L100": r7_pick(sc, "logloss", R7_GRID), "F50": 0.5}


def r7_hospital_unit(fs, oc, h):
    """makes the 3-fold test predictions of one hospital with five arms and keeps only the metrics (predictions are not stored). The folds, keys and inner folds within a bank are the same as in r6_hospital_unit."""
    C = CTX[(fs, oc, "eICU")]
    y, hid = C["y"], R6_STATE["hid"][oc]
    idx = np.where(hid == h)[0]
    ys = y[idx]
    base = {"fs": fs, "outcome": oc, "hospitalid": int(h)}
    n_sp = min(N_SITE_SPLITS, int(ys.sum()), int(len(ys) - ys.sum()))
    if n_sp < 2:
        return [dict(base, status="not eligible")]
    skf = StratifiedKFold(n_splits=n_sp, shuffle=True, random_state=stable_seed("a29", oc, int(h)))
    pr = {a: np.full(len(ys), np.nan) for a in R7_ARMS}
    alphas = {a: [] for a in R7_ARMS}
    for fi, (bank_i, test_i) in enumerate(skf.split(np.zeros(len(ys)), ys)):
        bank, test = idx[bank_i], idx[test_i]
        if len(np.unique(y[bank])) < 2 or len(np.unique(y[test])) < 2:
            return [dict(base, status="not eligible")]
        key = "a29|%s|%d|%d" % (oc, int(h), fi)
        Xb, yb, p1b = C["Xg"][bank], y[bank], C["pb1"][bank]
        p1t = C["pb1"][test]
        p_retr = RETRIEVAL_FN({"X_tr_s": Xb, "y_tr": yb, "X_te_s": C["Xg"][test], "y_te": y[test], "mi": None, "mi_prob": None})
        al = r7_alphas(r7_inner_scores(Xb, yb, p1b, key))
        for a in R7_ARMS:
            pr[a][test_i] = (1 - al[a]) * p1t + al[a] * p_retr
            alphas[a].append(al[a])
    t = float(T_SITE[oc])
    row = dict(base, status="ok", n_site=int(len(ys)), n_events=int(ys.sum()), n_folds=int(n_sp))
    for name, p in (("frozen", C["pb1"][idx]),) + tuple((a, pr[a]) for a in R7_ARMS):
        row["auroc_" + name] = float(roc_auc_score(ys, p))
        row["ece_" + name] = float(compute_ece(ys, p))
        row["nb_" + name] = float(nb_curve(ys, p, np.array([t]))[0])
    for a in R7_ARMS:
        row["alpha_" + a] = [float(v) for v in alphas[a]]
    return [row]


def r7_share(al, v):
    return float(np.mean(np.isclose(al, v)))


def r7_n1_tables(df):
    """collects the hospital rows (status ok) by (feature set, outcome) and gives the change per arm | pair differences (P1 to P4) | alpha distribution | number of hospitals with an AUROC decrease over 0.01. The hospital bootstrap replicates are the same as in the preceding cells (r6_hosp_boot_plan)."""
    a_rows, p_rows, d_rows = [], [], []
    streams = r5b_hosp_boot_streams()
    orders = r6_hosp_orders()
    for fs in R6_FS_ALL:
        for oc in OUTCOMES:
            g = df[(df["fs"] == fs) & (df["outcome"] == oc) & (df["status"] == "ok")].reset_index(drop=True)
            if len(g) < 3:
                continue
            g, idx, stream = r6_hosp_boot_plan(g, fs, oc, streams, orders)
            k = len(g)
            w = g["n_site"].to_numpy(float)
            M = {a: {m: g[m + "_" + a].to_numpy(float) for m in ("auroc", "ece", "nb")} for a in R7_ARMS}
            M["frozen"] = {m: g[m + "_frozen"].to_numpy(float) for m in ("auroc", "ece", "nb")}
            for a in R7_ARMS:
                row = {"fs": fs, "outcome": oc, "k": k, "stream": stream, "arm": a, "arm_label": R7_ARM_LABEL[a]}
                for m, nm in (("auroc", "dAUROC"), ("ece", "dECE"), ("nb", "dNB")):
                    est, lo, hi = r6_wmean_boot(M[a][m] - M["frozen"][m], w, idx)
                    row[nm], row[nm + "_lo"], row[nm + "_hi"] = est, lo, hi
                lost_f = int(np.sum((M[a]["auroc"] - M["frozen"]["auroc"]) < -0.01))             # counter-example check: number of hospitals whose AUROC dropped by more than 0.01 versus frozen (per arm)
                row["n_hospitals_auroc_drop_gt001"], row["share_hospitals_auroc_drop_gt001"] = lost_f, float(lost_f / k)
                if a in R7_SEL_ARMS:
                    al = np.concatenate([np.asarray(x, dtype=float) for x in g["alpha_" + a]])
                    row.update({"n_hospital_folds": int(len(al)), "a0_share": r7_share(al, 0.0), "a050_share": r7_share(al, 0.5), "a_gt050_share": float(np.mean(al > 0.5 + 1e-12)),
                                "a100_share": r7_share(al, 1.0), "alpha_median": float(np.median(al))})
                    for av in R7_GRID:
                        d_rows.append({"fs": fs, "outcome": oc, "arm": a, "alpha": float(av), "hospital_folds": int(np.sum(np.isclose(al, av)))})      # not masked
                a_rows.append(row)
            for pid, new, ref, desc in R7_PAIRS:
                row = {"fs": fs, "outcome": oc, "k": k, "stream": stream, "pair": pid, "difference": "%s - %s (%s)" % (R7_ARM_LABEL[new], R7_ARM_LABEL[ref], desc)}
                for m, nm in (("auroc", "dAUROC"), ("ece", "dECE"), ("nb", "dNB")):
                    est, lo, hi = r6_wmean_boot(M[new][m] - M[ref][m], w, idx)
                    row[nm], row[nm + "_lo"], row[nm + "_hi"], row[nm + "_half"] = est, lo, hi, (hi - lo) / 2.0
                lost = int(np.sum((M[new]["auroc"] - M[ref]["auroc"]) < -0.01))
                row["n_hospitals_auroc_drop_gt001"] = lost
                row["share_hospitals_auroc_drop_gt001"] = float(lost / k)
                p_rows.append(row)
    return pd.DataFrame(a_rows), pd.DataFrame(p_rows), pd.DataFrame(d_rows)


def r7_band(lo, hi, b=0.005):
    return bool(lo >= -b and hi <= b)


def r7_read(pairs):
    """applies the pre-specified rules to P1 and P2 (grid effect) and P3 (share specific to the criterion). Cell = feature set x outcome, four cells. P4 is not read."""
    out = []
    for pid in ("P1", "P2", "P3"):
        s = pairs[pairs["pair"] == pid]
        if len(s) == 0:
            continue
        full = len(s) == 4
        band = bool(full and all(r7_band(r.dAUROC_lo, r.dAUROC_hi) and r7_band(r.dECE_lo, r.dECE_hi) for r in s.itertuples()))
        ece_good = s[(s["dECE_hi"] < -0.0025) & (s["dAUROC_lo"] >= -0.005)]
        auc_lose = s[s["dAUROC_lo"] < -0.005]
        auc_gain = s[s["dAUROC_lo"] >= 0.005]
        wide = s[((s["dAUROC_hi"] - s["dAUROC_lo"]) > 0.010) | ((s["dECE_hi"] - s["dECE_lo"]) > 0.010)]
        if pid in ("P1", "P2"):
            if band:
                st = "grid insensitive"
            elif len(ece_good) and len(auc_lose):
                st = "no summarizing word: rule 2 cells and rule 3 cells are present together (cells in the next column)"                     # no summarizing word is attached
            elif len(auc_lose):
                st = "widening cannot exclude a discrimination loss of 0.005 (there are cells whose AUROC lower bound is below -0.005)"
            elif len(ece_good):
                st = "widening improves calibration"
            elif len(auc_gain):
                st = "widening increases discrimination"
            else:
                st = "mixed" if not len(wide) else "cannot be decided (interval is wide)"
        else:
            fixed_better = s[(s["dECE_lo"] > 0.0025) | (s["dAUROC_hi"] < -0.005)]
            if band:
                st = "no share specific to the criterion"
            elif len(fixed_better):
                st = "fixed 0.50 is better (cells present)" + (" + cells that also have a calibration share" if len(ece_good) else "")
            elif len(ece_good):
                st = "share specific to the criterion (calibration)"
            else:
                st = "mixed"
        out.append({"pair difference": pid, "reading": st, "all four cells inside the band": "yes" if band else "no", "cells where the ECE pair-difference interval is below -0.0025 and the AUROC lower bound is -0.005 or above": ", ".join("%s %s" % (r.fs, r.outcome) for r in ece_good.itertuples()) or "absent",
                    "cells where the AUROC lower bound is below -0.005": ", ".join("%s %s" % (r.fs, r.outcome) for r in auc_lose.itertuples()) or "absent",
                    "cells where the AUROC lower bound is +0.005 or above (rule 4)": ", ".join("%s %s" % (r.fs, r.outcome) for r in auc_gain.itertuples()) or "absent",
                    "cells whose interval width is wider than 0.010": ", ".join("%s %s" % (r.fs, r.outcome) for r in wide.itertuples()) or "absent"})
    return pd.DataFrame(out)


def r7_pattern_zone(arms, pairs):
    """the interval position of the pre-specified rules. Uses only the two cells of the 42-column set (F42), AKI and death, and the hospital-level values (hospital bootstrap 95% interval).
    order checked: both upper bounds below 0 -> lower bounds above the limits -> upper bound below the limit or interval including 0 -> otherwise no listed position (only values are reported). Returns: (table of input values, position name)."""
    rows = []
    for oc in OUTCOMES:
        a = arms[(arms["fs"] == "F42") & (arms["outcome"] == oc) & (arms["arm"] == "A100")]
        l = arms[(arms["fs"] == "F42") & (arms["outcome"] == oc) & (arms["arm"] == "L100")]
        p = pairs[(pairs["fs"] == "F42") & (pairs["outcome"] == oc) & (pairs["pair"] == "P1")]
        if len(a) != 1 or len(l) != 1 or len(p) != 1:
            return pd.DataFrame(), "cannot be computed (no 42-column cell)"
        rows.append({"outcome": oc, "A100 dAUROC lower bound": float(a["dAUROC_lo"].iloc[0]), "A100 dAUROC upper bound": float(a["dAUROC_hi"].iloc[0]), "L100 dAUROC upper bound": float(l["dAUROC_hi"].iloc[0]),
                     "P1 dAUROC lower bound": float(p["dAUROC_lo"].iloc[0]), "P1 dAUROC upper bound": float(p["dAUROC_hi"].iloc[0])})
    t = pd.DataFrame(rows)
    nul = bool(((t["A100 dAUROC upper bound"] < 0) & (t["L100 dAUROC upper bound"] < 0)).all())
    keep = bool(((t["A100 dAUROC lower bound"] > 0) & (t["P1 dAUROC lower bound"] >= -0.005)).all())
    down = bool(((t["P1 dAUROC upper bound"] < -0.005) | ((t["A100 dAUROC lower bound"] <= 0) & (t["A100 dAUROC upper bound"] >= 0))).any())
    return t, ("both A100 and L100 upper bounds below 0" if nul else "A100 lower bound above 0 and P1 lower bound at or above -0.005" if keep else "P1 upper bound below -0.005 or A100 interval includes 0" if down else "no listed position (values only)")


R7_N1 = {"state": "", "repro": "", "arms": pd.DataFrame(), "pairs": pd.DataFrame(), "dist": pd.DataFrame(), "reading": pd.DataFrame(), "checks": pd.DataFrame(), "zone": ""}


def r7_run_n1():
    if not R6_STATE["fs_hosp"]:
        R7_N1["state"] = "not computed because the input is missing: " + R6_STATE["n1_why"]
        emit("**The hospital level is not computed because the input is missing.** Reason: " + R6_STATE["n1_why"])
        status("additional: arms hospital level", "not available", R7_N1["state"])
        return
    if R6_N1["state"] != "done":
        R7_N1["state"] = "the hospital level of the preceding cells is not in a completed state, so the reproduction check could not be made and it is not computed (state of the preceding cells: %s)" % (R6_N1["state"] or "not run")
        emit("**The hospital level is not computed.** " + R7_N1["state"])
        status("additional: arms hospital level", "not available", R7_N1["state"])
        return
    sites, fs_list = R6_STATE["sites"], R6_STATE["fs_hosp"]
    order = [(fs, oc, int(h)) for fs in fs_list for oc in OUTCOMES for h in sites[oc]]
    emit("### hospital level . grid 0 to 1.00 (21 points) versus 0 to 0.50 (11 points), fixed 0.50 arm (feature set %s , units %d, 3 folds per hospital)" % (" · ".join(fs_list), len(order)))
    res = {fs: R6Resumable("R7_n1_" + fs) for fs in fs_list}
    t_loop, n_new, rows_all = time.time(), 0, []
    for k, (fs, oc, h) in enumerate(order):
        unit = ("n1", oc, h, R7_CFG1)
        was = res[fs].unit_done(unit)
        rows_all += r6_unit(res[fs], unit, lambda: r7_hospital_unit(fs, oc, h))
        if not was:
            n_new += 1
            if n_new % 10 == 1 or k == len(order) - 1:
                print("  [hospital level] %s %s hospital %d/%d done, remaining time about %.0f min (estimated by the number of hospitals)" % (fs, oc, k + 1, len(order), (time.time() - t_loop) / max(n_new, 1) * (len(order) - k - 1) / 60.0), flush=True)
    df = pd.DataFrame(rows_all)
    ok6 = R6_N1["hosp"][R6_N1["hosp"]["status"] == "ok"].set_index(["fs", "outcome", "hospitalid"])
    ok7 = df[df["status"] == "ok"].set_index(["fs", "outcome", "hospitalid"])
    same_set = bool(set(ok6.index) == set(ok7.index))
    n_bad_a, n_all = 0, 0
    R7_REPRO["cells"] = [c_ for c_ in R7_REPRO["cells"] if c_[0] != "hospital level"]          # so that they do not overlap even if run again in the same kernel
    if same_set:
        for ix in ok7.index:
            n_all += 1
            r6_, r7_ = ok6.loc[ix], ok7.loc[ix]
            for a in R7_SEL_ARMS[0::2]:          # only A50 and L50 exist in the preceding cells
                c = R7_ARM_CRIT[a]
                for m in ("auroc", "ece", "nb"):
                    R7_REPRO["cells"].append(("hospital level", ix[0], ix[1], m, a, ix[2], abs(float(r6_[m + "_" + c]) - float(r7_[m + "_" + a]))))
                same_al = [float(v) for v in r6_["alpha_" + c]] == [float(v) for v in r7_["alpha_" + a]]          # the per-fold alpha lists must be equal without tolerance
                R7_REPRO["cells"].append(("hospital level", ix[0], ix[1], "alpha_list", a, ix[2], 0.0 if same_al else 1.0))
                if not same_al:
                    n_bad_a += 1
            for m in ("auroc", "ece", "nb"):
                R7_REPRO["cells"].append(("hospital level", ix[0], ix[1], m, "frozen", ix[2], abs(float(r6_[m + "_frozen"]) - float(r7_[m + "_frozen"]))))
    _dd = [c_[6] for c_ in R7_REPRO["cells"] if c_[0] == "hospital level" and c_[3] != "alpha_list"]
    n_bad_m, n_hard = sum(1 for d_ in _dd if d_ > R7_TOL_EXACT), sum(1 for d_ in _dd if d_ > R7_TOL_NOISE)
    chk = [{"check": "the set of hospitals counted by this run is the same as in the preceding cells", "ok": int(same_set), "all": 1},
           {"check": "the per-hospital AUROC, ECE and net benefit of A50, L50 and frozen are within 1e-6 of the stored hospital rows (cells over 1e-12 are marked within thread-order noise . the cells over 1e-6 must be 0)", "ok": int(n_hard == 0 and same_set), "all": 1},
           {"check": "the per-fold alpha lists of A50 and L50 equal those of the preceding cells (no tolerance . mismatched hospitals must be 0)", "ok": int(n_bad_a == 0 and same_set), "all": 1},
           {"check": "the first 11 of the 21 grid points equal the ALPHA_GRID of the preceding cells and the end is 1.00", "ok": int(R7_CHECK_GRID), "all": 1}]
    R7_N1["checks"] = pd.DataFrame(chk)
    emit("- of %d hospitals, computed this time %d | loaded %d | hospitals compared %d | of %d metric cells over 1e-12 %d (of which over 1e-6 %d) | mismatched alpha lists %d" % (len(df), n_new, len(df) - n_new, n_all, len(_dd), n_bad_m, n_hard, n_bad_a))
    R7_N1["repro"] = r7_repro_report("hospital level")
    emit_table("hospital-level reproduction check (ok = all is required to use the new values)", R7_N1["checks"], digits=0)
    if any(r["ok"] != r["all"] for r in chk):
        R7_N1["state"] = "reproduction mismatch"
        emit("**hospital-level reproduction mismatch , the new values are not used.** Save this screen (.ipynb).")
        status("additional: arms hospital level", "reproduction mismatch", "A50 and L50 differ from the preceding cells")
        return
    arms, pairs, dist = r7_n1_tables(df)
    ref = R6_N1["cells"][R6_N1["cells"]["criterion"] == R6_CRIT_LABEL["auroc"]].set_index(["fs", "outcome"])
    bad_b, bad_b_hard = 0, 0
    for r_ in arms[arms["arm"] == "A50"].itertuples():
        rr = ref.loc[(r_.fs, r_.outcome)]
        dm_ = max(abs(r_.dAUROC - rr["dAUROC"]), abs(r_.dAUROC_lo - rr["dAUROC_lo"]), abs(r_.dAUROC_hi - rr["dAUROC_hi"]), abs(r_.dECE - rr["dECE"]), abs(r_.dNB - rr["dNB"]))
        R7_REPRO["cells"].append(("hospital level", r_.fs, r_.outcome, "hospital-weighted mean and interval (A50 row versus the stored AUROC row)", "A50", None, float(dm_)))
        bad_b += int(dm_ > R7_TOL_EXACT)
        bad_b_hard += int(dm_ > R7_TOL_NOISE)
    emit("- cells where the hospital bootstrap replicate indices are the same as the preceding cells: %s . Number of cells where the mean and interval of the stored AUROC (rule of the main analysis) row and the A50 row differ by more than 1e-12 %d (of which over 1e-6 %d, over 1e-6 must be 0)" % (arms["stream"].value_counts().to_dict(), bad_b, bad_b_hard))
    R7_N1["repro"] = r7_repro_report("hospital level", show=False)           # the table was printed above . only the hospital-weighted mean and interval rows are added and the csv and the judgment are rewritten
    if bad_b_hard:
        R7_N1["state"] = "reproduction mismatch"
        status("additional: arms hospital level", "reproduction mismatch", "the A50 bootstrap rows differ from the preceding cells by more than 1e-6")
        return
    R7_N1["arms"], R7_N1["pairs"], R7_N1["dist"] = arms, pairs, dist
    emit_table("hospital level: change versus frozen per arm (hospital mean is stay-weighted, hospital bootstrap %d repetitions 95%% interval, same replicate indices as in the preceding cells) . a0 , a050 and a100 = fraction of (hospital, fold) pairs in which that alpha was chosen" % N_BOOT,
               arms[["fs", "outcome", "k", "arm_label", "dAUROC", "dAUROC_lo", "dAUROC_hi", "dECE", "dECE_lo", "dECE_hi", "dNB", "dNB_lo", "dNB_hi", "a0_share", "a050_share", "a_gt050_share", "a100_share", "alpha_median", "n_hospitals_auroc_drop_gt001", "share_hospitals_auroc_drop_gt001"]], digits=4, max_rows=60)
    pv = arms.pivot_table(index=["fs", "outcome", "k"], columns="arm", values="share_hospitals_auroc_drop_gt001").reset_index()
    pv["L50 - A50"], pv["L100 - A100"] = pv["L50"] - pv["A50"], pv["L100"] - pv["A100"]
    emit_table("hospital level: fraction of hospitals whose AUROC is lower than frozen by more than 0.01 (per arm , counter-example check: did log loss selection increase more than AUROC selection = the last two columns)", pv[["fs", "outcome", "k"] + list(R7_ARMS) + ["L50 - A50", "L100 - A100"]], digits=4, max_rows=20)
    emit_table("hospital level: pair differences P1 to P4 (earlier arm - later arm , 95% interval with the same hospital replicate indices , _half = half width , AUROC decrease over 0.01 per hospital = hospitals where the pair difference (earlier arm - later arm) is smaller than -0.01)",
               pairs[["fs", "outcome", "k", "pair", "difference", "dAUROC", "dAUROC_lo", "dAUROC_hi", "dECE", "dECE_lo", "dECE_hi", "dNB", "dNB_lo", "dNB_hi", "n_hospitals_auroc_drop_gt001", "share_hospitals_auroc_drop_gt001"]], digits=4, max_rows=60)
    # audit of sums and inequalities between tables
    aud = []
    for (fs_, oc_, a_), s_ in dist.groupby(["fs", "outcome", "arm"]):
        n_tot = int(arms[(arms["fs"] == fs_) & (arms["outcome"] == oc_) & (arms["arm"] == a_)]["n_hospital_folds"].iloc[0])
        vals = [None if isinstance(v, str) else int(v) for v in s_["hospital_folds"]]
        aud.append({"fs": fs_, "outcome": oc_, "arm": a_, "sum of the distribution table (excluding masked cells)": sum(v for v in vals if v is not None), "number of masked cells": sum(1 for v in vals if v is None), "denominator of the summary table": n_tot})
    aud = pd.DataFrame(aud)
    ineq = bool(((arms["a0_share"] + arms["a050_share"]) <= 1 + 1e-12).where(arms["arm"].isin(R7_SEL_ARMS), True).all()) and bool((arms["a100_share"] <= arms["a_gt050_share"] + 1e-12).where(arms["arm"].isin(R7_SEL_ARMS), True).all())
    emit_table("sum audit: is the sum of the distribution table cells equal to the denominator of the summary table (the number of times alpha was chosen is not masked, so it must be equal . the masked-cell-count column is 0)", aud, digits=0, max_rows=40)
    ok_sum = all((r["number of masked cells"] == 0 and r["sum of the distribution table (excluding masked cells)"] == r["denominator of the summary table"]) or (r["number of masked cells"] > 0 and r["number of masked cells"] <= r["denominator of the summary table"] - r["sum of the distribution table (excluding masked cells)"] <= 4 * r["number of masked cells"]) for r in aud.to_dict("records"))
    emit("- sum audit result: %s . inequalities (a0 fraction + a050 fraction <= 1 , a100 fraction <= a>0.50 fraction): %s" % ("passed" if ok_sum else "mismatch present (table above)", "passed" if ineq else "mismatch"))
    rd = r7_read(pairs)
    R7_N1["reading"] = rd
    emit_table("hospital-level reading (the pre-specified rules , P1 and P2 are the grid effect , P3 is the share specific to the criterion . P4 is not read . only one reading word is attached and the applicable cells are all written in the next column)", rd, digits=4)
    zt, zone = r7_pattern_zone(arms, pairs)
    R7_N1["zone"] = zone
    if len(zt):
        emit_table("hospital level: input values of the pre-specified rules (42 columns, two outcomes , hospital-level values)", zt, digits=4)
    emit("- interval position: **%s** (values only; no further label is assigned)." % zone)
    emit("- [note] widening may either decrease or increase the PRAM gain . The main analysis grid 0 to 0.50 is not changed . Even if P1 and P2 are 0 it is not evidence of insensitivity and may mean that the selected alpha did not stick to 0.50 (read the fractions a050 and a_gt050 together) . "
         "if P3 is 0, the gain of the selection-criterion comparison comes from setting alpha large, and it does not mean that log loss is useless .")
    r7_save_csv(arms, "R7_N1_hospital_arms")
    r7_save_csv(pairs, "R7_N1_hospital_pairdiff")
    r7_save_csv(dist, "R7_N1_hospital_alpha_distribution")
    R7_N1["state"] = "done"
    status("additional: arms hospital level", "done", "%d hospitals | computed this time %d | loaded %d | reproduction judgment %s" % (len(df), n_new, len(df) - n_new, R7_N1["repro"]))


if not R6_STOP:
    _rep7 = len(REPORT_LINES)
    if not R7_CHECK_GRID:
        R7_N1["state"] = "grid check failed"
        emit("**The grid check failed , not computed.** ALPHA_GRID of the preceding cells is not 11 points from 0 to 0.50: %s" % (list(ALPHA_GRID),))
        status("additional: arms hospital level", "not available", "ALPHA_GRID of the preceding cells differs from expectation")
    else:
        r7_run_n1()
    R7_LINES["n1"] = [str(_l) for _l in REPORT_LINES[_rep7:]]
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)

In [ ]:
# ================================================================
# PRAM supplementary cell 7/13 (B) . eICU full bank (5 folds) . grid 0 to 1.00 versus 0 to 0.50 , fixed 0.50 arm (resumable)
# ================================================================
R7_N1W = {"state": "", "repro": "", "cells": pd.DataFrame(), "pairs": pd.DataFrame(), "checks": pd.DataFrame()}


def r7_whole_fold(fs, oc, fi):
    """from the same bank, test fold and key as one fold of the full bank (full) of run_matrix_unit of the preceding cells, stores the test predictions of the five arms as npz (arrays in patient order, no identifier, only inside the resume folder)."""
    C = CTX[(fs, oc, "eICU")]
    y = C["y"]
    pool_idx, test_idx = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))[fi]
    Xb, yb, p1b = C["Xg"][pool_idx], y[pool_idx], C["pb1"][pool_idx]
    p1t = C["pb1"][test_idx]
    key = "%s|eICU|full|%d|0" % (oc, fi)
    p_retr = RETRIEVAL_FN({"X_tr_s": Xb, "y_tr": yb, "X_te_s": C["Xg"][test_idx], "y_te": y[test_idx], "mi": None, "mi_prob": None})
    al = r7_alphas(r7_inner_scores(Xb, yb, p1b, key))
    path = R6_RES_DIR / ("R7_n1w_%s_%s_f%d_%s.npz" % (fs, oc, fi, R7_CFG1_H))
    np.savez_compressed(path, test_idx=test_idx.astype(np.int64), **{a: ((1 - al[a]) * p1t + al[a] * p_retr).astype(np.float32) for a in R7_ARMS})
    return [{"fs": fs, "outcome": oc, "fold": int(fi), "n_bank": int(len(pool_idx)), "n_test": int(len(test_idx)), "npz": path.name, **{"alpha_" + a: float(al[a]) for a in R7_ARMS}}]


def r7_whole_pooled(fs, oc, rows):
    n = len(CTX[(fs, oc, "eICU")]["y"])
    oof = {a: np.full(n, np.nan, dtype=np.float32) for a in R7_ARMS}
    for r in rows:
        z = np.load(R6_RES_DIR / r["npz"])
        for a in R7_ARMS:
            oof[a][z["test_idx"]] = z[a]
    if any(np.isnan(v).any() for v in oof.values()):
        return None
    return {a: v.astype(float) for a, v in oof.items()}


def r7_whole_boot(fs, oc, fi_, oi_, pooled, y, t, fro):
    """pair differences P1 to P4 and the change versus frozen per arm. The AUROC is the paired DeLong 95% interval, and ECE and net benefit are patient-paired bootstrap N_BOOT repetitions percentile 95% interval.
    there is one replicate set and frozen and all five arms use the same replicate (one generator, drawing the index once per replicate). If kind of the row is pair, it is a pair difference, if arm it is the change versus frozen of that arm."""
    rng = np.random.default_rng([int(BOOT_SEED), 73, fi_, oi_])
    n, B = len(y), int(N_BOOT)
    allp = dict(pooled)
    allp["frozen"] = fro
    names = ("frozen",) + R7_ARMS
    be = {a: np.empty(B) for a in names}
    bn = {a: np.empty(B) for a in names}
    for r in range(B):
        i = rng.integers(0, n, n)
        yi = y[i]
        for a in names:
            pi_ = allp[a][i]
            be[a][r] = r6_ece_fast(yi, pi_)
            bn[a][r] = r6_nb_fast(yi, pi_, t)

    def one(new, ref):
        _a, _b, dd, var = delong_delta_var(y, allp[new], allp[ref])
        de = r6_ece_fast(y, allp[new]) - r6_ece_fast(y, allp[ref])
        dn = r6_nb_fast(y, allp[new], t) - r6_nb_fast(y, allp[ref], t)
        se = float(np.sqrt(var))
        ce, cn = be[new] - be[ref], bn[new] - bn[ref]
        return {"n": int(n), "dAUROC": float(dd), "dAUROC_lo": float(dd - 1.96 * se), "dAUROC_hi": float(dd + 1.96 * se), "dECE": float(de), "dECE_lo": float(np.percentile(ce, 2.5)), "dECE_hi": float(np.percentile(ce, 97.5)),
                "dNB": float(dn), "dNB_lo": float(np.percentile(cn, 2.5)), "dNB_hi": float(np.percentile(cn, 97.5)), "n_boot": B}

    out = []
    for pid, new, ref, desc in R7_PAIRS:
        out.append({"kind": "pair", "fs": fs, "outcome": oc, "pair": pid, "difference": "%s - %s (%s)" % (R7_ARM_LABEL[new], R7_ARM_LABEL[ref], desc), **one(new, ref)})
    for a in R7_ARMS:
        out.append({"kind": "arm", "fs": fs, "outcome": oc, "arm": a, "difference": "%s - frozen" % R7_ARM_LABEL[a], **one(a, "frozen")})
    return out


def r7_run_w():
    if R7_N1["state"] != "done":
        R7_N1W["state"] = "not computed because the hospital level is not in a completed state (%s)" % (R7_N1["state"] or "not run")
        emit("**The full bank is not computed.** " + R7_N1W["state"])
        status("additional: arms full bank", "not available", R7_N1W["state"])
        return
    if not R6_RUN_N1_WHOLE or R6_N1["w_state"] != "done":
        R7_N1W["state"] = "the full bank of the preceding cells is not in a completed state or is off, so the reproduction check could not be made and it is not computed (preceding cells: %s)" % (R6_N1["w_state"] or "not run")
        emit("**The full bank is not computed.** " + R7_N1W["state"])
        status("additional: arms full bank", "not available", R7_N1W["state"])
        return
    emit("### eICU full bank . 5-fold batch (feature set %s) , grid 0 to 1.00 versus 0 to 0.50 , fixed 0.50" % " · ".join(R6_STATE["fs_hosp"]))
    w_cells, w_pairs, w_chk = [], [], []
    t_w, n_w_new = time.time(), 0
    for fs in R6_STATE["fs_hosp"]:
        fi_ = R6_FS_ALL.index(fs)
        res_w, res_b = R6Resumable("R7_n1w_" + fs), R6Resumable("R7_n1wb_" + fs)
        res_6 = R6Resumable("R6_n1w_" + fs)
        for oi_, oc in enumerate(OUTCOMES):
            C = CTX[(fs, oc, "eICU")]
            y, t = C["y"], float(T_SITE[oc])
            rows = []
            for fo in range(N_FOLDS):
                u = ("n1w", oc, fo, R7_CFG1, N_FOLDS)
                was = res_w.unit_done(u)
                rows += r6_unit(res_w, u, lambda: r7_whole_fold(fs, oc, fo), extra_ok=lambda rr: bool(rr) and (R6_RES_DIR / rr[0]["npz"]).is_file())
                if not was:
                    n_w_new += 1
                    print("  [full bank] %s %s fold %d/%d done , elapsed %.0f min" % (fs, oc, fo + 1, N_FOLDS, (time.time() - t_w) / 60.0), flush=True)
            pooled = r7_whole_pooled(fs, oc, rows)
            if pooled is None:
                emit("**full bank %s %s: skipped because the test predictions are empty.**" % (fs, oc))
                continue
            # reproduction check: A50 and L50 equal the per-fold alpha of the preceding cells (no tolerance) and the test predictions (float32, within 1e-6 . over 1e-12 is within thread-order noise)
            bad = 0
            R7_REPRO["cells"] = [c_ for c_ in R7_REPRO["cells"] if not (c_[0] == "whole bank" and c_[1] == fs and c_[2] == oc)]
            R7_REPRO["q"] = {k_: v_ for k_, v_ in R7_REPRO["q"].items() if not (k_[0] == fs and k_[1] == oc)}
            for r in rows:
                r6u = res_6.get(("n1w", oc, r["fold"], R6_CFG1, N_FOLDS))
                if not r6u:
                    bad += 1
                    R7_REPRO["cells"].append(("whole bank", fs, oc, "stored results present", "A50 · L50", r["fold"], 1.0))
                    continue
                z6, z7 = np.load(R6_RES_DIR / r6u[0]["npz"]), np.load(R6_RES_DIR / r["npz"])
                same_idx = bool(np.array_equal(z6["test_idx"], z7["test_idx"]))
                for a in ("A50", "L50"):
                    c = R7_ARM_CRIT[a]
                    same_al = bool(float(r6u[0]["alpha_" + c]) == float(r["alpha_" + a]))
                    dmax = float(np.max(np.abs(z6[c].astype(float) - z7[a].astype(float)))) if same_idx else 1.0
                    R7_REPRO["q"][(fs, oc, a, r["fold"])] = int(np.sum(np.abs(z6[c].astype(float) - z7[a].astype(float)) > R7_TOL_NOISE)) if same_idx else int(len(z7[a]))      # number of queries over 1e-6 within a fold . separates whether it is a swap of neighbors of a single query or a mismatch of the whole
                    R7_REPRO["cells"].append(("whole bank", fs, oc, "alpha_list", a, r["fold"], 0.0 if same_al else 1.0))
                    R7_REPRO["cells"].append(("whole bank", fs, oc, "test predictions (largest difference by fold)", a, r["fold"], dmax))
                    if (not same_al) or dmax > R7_TOL_NOISE:
                        bad += 1
            w_chk.append({"check": "%s %s full-bank A50 and L50 per-fold alpha and test predictions equal the stored results (alpha without tolerance . predictions within 1e-6 . over 1e-12 marked within thread-order noise . mismatched fold x arm must be 0)" % (fs, oc), "ok": int(bad == 0), "all": 1})
            fro = C["pb1"].astype(np.float32).astype(float)
            met = {nm: full_metrics(y, p) for nm, p in (("frozen", fro),) + tuple((a, pooled[a]) for a in R7_ARMS)}
            tcol = nbcol(t)
            for a in R7_ARMS:
                row = {"fs": fs, "outcome": oc, "arm": a, "arm_label": R7_ARM_LABEL[a], "n": int(len(y)), "dAUROC": met[a]["auroc"] - met["frozen"]["auroc"], "dECE": met[a]["ece"] - met["frozen"]["ece"], "dNB": met[a][tcol] - met["frozen"][tcol]}
                if a in R7_SEL_ARMS:
                    al = np.array([r["alpha_" + a] for r in rows], dtype=float)
                    row.update({"a0_share_fold": r7_share(al, 0.0), "a050_share_fold": r7_share(al, 0.5), "a100_share_fold": r7_share(al, 1.0), "alpha_median": float(np.median(al)), "alphas": " ".join("%g" % v for v in al)})
                w_cells.append(row)
            ub = ("n1wb", oc, int(N_BOOT), int(BOOT_SEED), R7_CFG1)
            was_b = res_b.unit_done(ub)
            w_pairs += r6_unit(res_b, ub, lambda: r7_whole_boot(fs, oc, fi_, oi_, pooled, y, t, fro))
            if not was_b:
                print("  [full bank] %s %s pair-difference interval done , elapsed %.0f min" % (fs, oc, (time.time() - t_w) / 60.0), flush=True)
            del pooled
    if not w_cells:
        R7_N1W["state"] = "not available"
        status("additional: arms full bank", "not available", "the test predictions are empty")
        return
    wc, wpa = pd.DataFrame(w_cells), pd.DataFrame(w_pairs)
    wp = wpa[wpa["kind"] == "pair"].drop(columns=["kind", "arm"], errors="ignore").reset_index(drop=True)
    wc = wc.merge(wpa[wpa["kind"] == "arm"][["fs", "outcome", "arm", "dAUROC_lo", "dAUROC_hi", "dECE_lo", "dECE_hi", "dNB_lo", "dNB_hi", "n_boot"]], on=["fs", "outcome", "arm"], how="left", validate="one_to_one")
    R7_N1W["checks"] = pd.DataFrame(w_chk)
    emit_table("full-bank reproduction check (ok = all is required for use)", R7_N1W["checks"], digits=0)
    R7_N1W["repro"] = r7_repro_report("whole bank")
    if any(r["ok"] != r["all"] for r in w_chk):
        R7_N1W["state"] = "reproduction mismatch"
        emit("**full-bank reproduction mismatch , the new values are not used.**")
        status("additional: arms full bank", "reproduction mismatch", "A50 and L50 differ from the stored results")
        return
    R7_N1W["cells"], R7_N1W["pairs"] = wc, wp
    emit_table("eICU full bank: change versus frozen per arm (AUROC with the paired DeLong 95%% interval, ECE and net benefit with the patient-paired bootstrap %d repetitions 95%% interval)" % N_BOOT, wc.drop(columns=["alphas"], errors="ignore"), digits=4, max_rows=30)
    emit_table("eICU full bank: pair differences P1 to P4 (AUROC with the paired DeLong 95%% interval, ECE and net benefit with the patient-paired bootstrap %d repetitions 95%% interval)" % N_BOOT, wp[["fs", "outcome", "pair", "difference", "n", "dAUROC", "dAUROC_lo", "dAUROC_hi", "dECE", "dECE_lo", "dECE_hi", "dNB", "dNB_lo", "dNB_hi"]], digits=4, max_rows=30)
    emit("- the hospital-level reading rules are not applied to the four full-bank cells and only values are written (as at the hospital level above).")
    r7_save_csv(wc, "R7_N1_whole_arms")
    r7_save_csv(wp, "R7_N1_whole_pairdiff")
    R7_N1W["state"] = "done"
    status("additional: arms full bank", "done", "feature sets %d x outcomes %d (folds computed this time %d) | reproduction judgment %s" % (len(R6_STATE["fs_hosp"]), len(OUTCOMES), n_w_new, R7_N1W["repro"]))


if not R6_STOP:
    _rep7 = len(REPORT_LINES)
    r7_run_w()
    R7_LINES["w"] = [str(_l) for _l in REPORT_LINES[_rep7:]]
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)

In [ ]:
# ================================================================
# PRAM supplementary cell 8/13 (C) . storing the alpha columns of the single-bank 100 draws (same bank and key as run_tail_unit of the preceding cells, no predictions are made, resumable)
# ================================================================
R7_SB = {"state": "", "rows": pd.DataFrame(), "summary": pd.DataFrame()}


def r7_sb_unit(fs, oc, tg, bk):
    C = CTX[(fs, oc, tg)]
    y = C["y"]
    folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
    per_fold = max(1, N_DRAWS_TAIL // N_FOLDS)
    rows = []
    for fi, (pool_idx, _test_idx) in enumerate(folds):
        if bk > len(pool_idx):
            continue
        for dr in range(per_fold):
            key = "%s|%s|%s|%d|%d" % (oc, tg, bk, fi, dr)
            bank = pool_idx[sample_bank(y[pool_idx], bk, "tail|" + key)]
            if len(np.unique(y[bank])) < 2:
                continue
            Xb, yb, p1b = C["Xg"][bank], y[bank], C["pb1"][bank]
            al = r7_alphas(r7_inner_scores(Xb, yb, p1b, key))
            row = {"fs": fs, "outcome": oc, "target": tg, "bank": str(bk), "fold": int(fi), "draw": int(dr), **{"alpha_" + a: float(al[a]) for a in R7_SEL_ARMS}, "xcheck_inner": None}
            if dr == 0:
                row["xcheck_inner"] = bool(inner_cv_bank(Xb, yb, p1b, RETRIEVAL_FN, key="b1|" + key)["alpha"] == al["A50"])
            rows.append(row)
    return rows


def r7_run_sb():
    if not DATA_OK:
        R7_SB["state"] = "not computed because the input is missing: main inputs absent"
        status("additional: single-bank alpha", "not available", R7_SB["state"])
        return
    units = [u for u in R5_TAIL_UNITS if (u[0], u[1], u[2]) in CTX and isinstance(u[3], (int, np.integer))]
    emit("### single bank 100 draws . stores per draw the alpha of A50, A100, L50 and L100 from the 3-fold inner cross-validation scores within the bank (units %d , draws per cell %d)" % (len(units), N_DRAWS_TAIL))
    res = {fs: R6Resumable("R7_sb_" + fs) for fs in sorted({u[0] for u in units})}
    t_loop, n_new, rows_all = time.time(), 0, []
    for k, (fs, oc, tg, bk) in enumerate(units):
        unit = ("sb", oc, tg, int(bk), R7_CFG1, int(N_DRAWS_TAIL))
        was = res[fs].unit_done(unit)
        rows_all += r6_unit(res[fs], unit, lambda: r7_sb_unit(fs, oc, tg, int(bk)))
        if not was:
            n_new += 1
            print("  [single bank] %s %s / %s / bank %s done (%d/%d) , remaining time about %.0f min (estimated by the number of units)" % (fs, oc, tg, bk, k + 1, len(units), (time.time() - t_loop) / max(n_new, 1) * (len(units) - k - 1) / 60.0), flush=True)
    df = pd.DataFrame(rows_all)
    if not len(df):
        R7_SB["state"] = "not available"
        status("additional: single-bank alpha", "not available", "there are no draw rows")
        return
    xc = df["xcheck_inner"].dropna()
    n_bad = int((~xc.astype(bool)).sum())
    emit("- draws in which A50 equals the alpha chosen by inner_cv_bank of the preceding cells: %d/%d (the mismatch must be 0)" % (int(xc.astype(bool).sum()), len(xc)))
    if n_bad:
        R7_SB["state"] = "reproduction mismatch"
        emit("**single-bank reproduction mismatch , the new values are not used.**")
        status("additional: single-bank alpha", "reproduction mismatch", "A50 differs from inner_cv_bank of the preceding cells")
        return
    srows, drows = [], []
    for (fs, oc, tg, bk), s in df.groupby(["fs", "outcome", "target", "bank"], sort=False):
        n = len(s)
        row = {"fs": fs, "outcome": oc, "target": tg, "bank": bk, "n_draws": int(n)}
        for a in R7_SEL_ARMS:
            al = s["alpha_" + a].to_numpy(float)
            row.update({"%s_a0_share" % a: r7_share(al, 0.0), "%s_a050_share" % a: r7_share(al, 0.5), "%s_a_gt050_share" % a: float(np.mean(al > 0.5 + 1e-12)), "%s_a100_share" % a: r7_share(al, 1.0), "%s_median" % a: float(np.median(al))})
            for av in (R7_GRID if a in ("A100", "L100") else R7_GRID50):
                drows.append({"fs": fs, "outcome": oc, "target": tg, "bank": bk, "arm": a, "alpha": float(av), "draws": int(np.sum(np.isclose(al, av)))})
        srows.append(row)
    summ, dist = pd.DataFrame(srows), pd.DataFrame(drows)
    emit_table("single bank 100 draws: alpha selection fraction per cell (a0 = fraction of draws in which alpha was chosen as 0 . reading it side by side with the fraction below 0 of the stored single-bank result is a job for the interpretation)",
               summ[["fs", "outcome", "target", "bank", "n_draws"] + ["%s_%s" % (a, c) for a in ("A50", "A100", "L50", "L100") for c in ("a0_share", "a050_share", "a_gt050_share", "a100_share")]], digits=3, max_rows=60)
    R7_SB["rows"], R7_SB["summary"] = df, summ
    r7_save_csv(summ, "R7_single_bank_alpha_summary")
    r7_save_csv(dist, "R7_single_bank_alpha_distribution")
    R7_SB["state"] = "done"
    status("additional: single-bank alpha", "done", "units %d (computed this time %d | loaded %d)" % (len(units), n_new, len(units) - n_new))


if not R6_STOP:
    _rep7 = len(REPORT_LINES)
    r7_run_sb()
    R7_LINES["sb"] = [str(_l) for _l in REPORT_LINES[_rep7:]]
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)

In [ ]:
# ================================================================
# PRAM supplementary cell 9/13 (A) . supplementary analyses, three folds: federated retrieval, eight modes
# reading rules of this part. All new names start with R8_. All settings are in the first cell.
# runs two methods in the same cell: original split (one random split per hospital, test 30%: the original calculation as is = for the reproduction check) and three folds (the same folds as the hospital level of the preceding cells, all stays of the hospital with test 1/3 x 3 combined).
# the two methods have the same partner list, the same retrieval and the same mixing weight selection, and differ only in test composition and bank size (one factor at a time).
# ================================================================
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.neighbors import NearestNeighbors
from scipy import stats as r8_stats

R8_A_SCHEMES = ("FR0_local", "FR1_fed_exact", "FR2_fed_aggregate", "FR3_fed_agg_dp1", "FR3_fed_agg_dp4", "FR4_proto_network", "FR5_local_plus_proto", "FR9_pooled_raw")
R8_FS_AB = ("F42",)                    # Supplementary S11 Panels A and B use 42 columns
R8_LINES = {"a": [], "b": [], "c": [], "s": []}
R8_A = {"state": "", "rows": pd.DataFrame(), "summary": pd.DataFrame(), "struct": pd.DataFrame(), "repro": pd.DataFrame(), "sweep": pd.DataFrame()}
R8_B = {"state": "", "rows": pd.DataFrame(), "summary": pd.DataFrame(), "repro": pd.DataFrame()}
R8_C = {"state": "", "rows": pd.DataFrame(), "summary": pd.DataFrame(), "repro": pd.DataFrame(), "prev": pd.DataFrame(), "corr": pd.DataFrame()}
R8_STRUCT_ITEM = "structure (mean absolute difference of the aggregate / fraction of queries with an exact-method probability difference / correlation)"           # name of the A structure row of the reproduction table (the gate of rule N8-2 searches for the same string)
R8_ANCHOR_ITEM = "anchor: three-fold own-bank PRAM versus the hospital-level PRAM of the preceding cells (R5B_HPRAM)"          # name of the A anchor row of the reproduction table (the gate of rule N8-1 searches for the same string)
R8_PROTO = {}                          # stores in memory the prototypes of one partner hospital (independent of the target hospital). The same seed, so it is the same even if rebuilt


if not R6_STOP:
    import sklearn as _skl8
    _skl8.set_config(working_memory=int(R8_WORKING_MEMORY_MB))      # upper limit of memory for the distance chunks of the neighbor search (brute) (MiB). A setting of one place in the first cell and the default equals the sklearn default (1024)
    emit_table("settings of the supplementary analyses (one setting for the whole analysis: the number of bootstrap repetitions, seed and model fitting seed read the same values as the preceding cells, and the supplementary computation has no resampling)", pd.DataFrame([
        {"item": "number of neighbors k , distance weighting , partner cap", "value": "%d , 1/(d + %g) , %d" % (R8_K_RETR, R8_EPS_W, R8_MAX_PARTNERS), "basis": "K_RETR, EPS_W and MAX_PARTNERS, computed only if equal to the main settings"},
        {"item": "test fraction of the original split (A) , training and weighting shares (B)", "value": "%g , %g · %g" % (R8_A_TEST_FRAC, R8_B_TRAIN_FRAC, R8_B_WEIGHT_FRAC), "basis": "settings of the original split"},
        {"item": "three folds B: training share of the rest", "value": str(R8_B_REST_TRAIN_SHARE), "basis": "there is no reference value, so the value that keeps the training : weighting ratio of 3 : 1 of the original split is used. If None, the B three folds are not computed"},
        {"item": "number of hospital folds , number of inner folds , SEED", "value": "%d , %d , %d" % (N_SITE_SPLITS, N_INNER, SEED), "basis": "values of the preceding cells (the same folds as the hospital-level within-hospital bootstrap)"},
        {"item": "mixing weight grid (A and B selection) , unlabeled hospital grid (C)", "value": "%g to %g (%d points) , %g to %g (%d points)" % (min(ALPHA_GRID), max(ALPHA_GRID), len(ALPHA_GRID), min(NET_ALPHA_GRID), max(NET_ALPHA_GRID), len(NET_ALPHA_GRID)), "basis": "ALPHA_GRID and NET_ALPHA_GRID of the preceding cells"},
        {"item": "random number method", "value": "one method of the per-cell generators rng_of(key) of the preceding cells (the same values even when stopped and resumed)", "basis": "no bootstrap or resampling in the supplementary analyses"},
        {"item": "neighbor search chunk memory working_memory (MiB)", "value": int(R8_WORKING_MEMORY_MB), "basis": "R8_WORKING_MEMORY_MB of the first cell. If insufficient, lower to 256, but this value is in the unit fingerprint of the supplementary analyses so changing it recomputes those units from the start (the stored results of the other versions are kept)"}]))
# reference values: (hospital median change, improved hospitals %, calibration slope median)
R8_REF_A = {("AKI_168h", "FR0_local"): (0.0000, 47.7612, 1.0850), ("AKI_168h", "FR1_fed_exact"): (0.0000, 49.2537, 1.0899), ("AKI_168h", "FR2_fed_aggregate"): (0.0058, 68.6567, 1.1347),
            ("AKI_168h", "FR3_fed_agg_dp1"): (0.0052, 67.1642, 1.1275), ("AKI_168h", "FR3_fed_agg_dp4"): (0.0063, 67.1642, 1.1340), ("AKI_168h", "FR4_proto_network"): (0.0028, 52.2388, 1.1858),
            ("AKI_168h", "FR5_local_plus_proto"): (0.0044, 53.7313, 1.1491), ("AKI_168h", "FR9_pooled_raw"): (0.0000, 49.2537, 1.0899),
            ("mortality_168h", "FR0_local"): (0.0067, 67.7966, 0.8014), ("mortality_168h", "FR1_fed_exact"): (0.0190, 79.6610, 0.7824), ("mortality_168h", "FR2_fed_aggregate"): (0.0095, 81.3559, 0.8172),
            ("mortality_168h", "FR3_fed_agg_dp1"): (0.0101, 81.3559, 0.8608), ("mortality_168h", "FR3_fed_agg_dp4"): (0.0098, 81.3559, 0.8102), ("mortality_168h", "FR4_proto_network"): (0.0102, 69.4915, 0.8261),
            ("mortality_168h", "FR5_local_plus_proto"): (0.0130, 74.5763, 0.7974), ("mortality_168h", "FR9_pooled_raw"): (0.0190, 79.6610, 0.7824)}
R8_REF_SWEEP = {("AKI_168h", 1): (0.0020, 52.2388), ("AKI_168h", 2): (0.0057, 58.2090), ("AKI_168h", 4): (0.0047, 61.1940), ("AKI_168h", 8): (0.0043, 65.6716), ("AKI_168h", 16): (0.0049, 62.6866),
                ("mortality_168h", 1): (0.0088, 72.8814), ("mortality_168h", 2): (0.0104, 77.9661), ("mortality_168h", 4): (0.0082, 77.9661), ("mortality_168h", 8): (0.0098, 79.6610), ("mortality_168h", 16): (0.0090, 79.6610)}
R8_REF_A_STRUCT = {"exact_err_max": 2.545e-07, "agg_err_median": 0.0270, "agg_corr_median": 0.795}            # reference values of the exact and aggregate schemes
R8_N_HOSP_REF = {"AKI_168h": 67, "mortality_168h": 59}                                                          # reference number of hospitals

if R6_STOP:
    R8_CFG_A = R8_CFG_B0 = R8_CFG_B = R8_CFG_C = ""
else:
    R8_CFG_A = "A|code%s|k%d|eps%g|part%d|proto%d|dp%s|sizes%s|test%g|inner%d|seed%d|alpha%s|thr%d|wm%d" % (CODE_VER_FED, R8_K_RETR, R8_EPS_W, R8_MAX_PARTNERS, R8_N_PROTO_SITE, R8_DP_EPS, R8_NET_SIZES, R8_A_TEST_FRAC, N_INNER, SEED, "-".join("%g" % a for a in ALPHA_GRID), int(R8_FP_THR), int(R8_WORKING_MEMORY_MB))
    R8_CFG_B0 = "B|code%s|tr%g|wt%g|pool%d|inner%d|seed%d|k%d|alpha%s|thr%d|wm%d" % (CODE_VER_FED, R8_B_TRAIN_FRAC, R8_B_WEIGHT_FRAC, R8_B_POOL_NET_CAP, N_INNER, SEED, R8_K_RETR, "-".join("%g" % a for a in ALPHA_GRID), int(R8_FP_THR), int(R8_WORKING_MEMORY_MB))       # original-split unit fingerprint (independent of the three-fold share)
    R8_CFG_B = R8_CFG_B0 + "|rest%s" % R8_B_REST_TRAIN_SHARE                                                                                                          # three-fold unit fingerprint (if the share changes only the three folds are recomputed)
    R8_CFG_C = "C|code%s|part%d|test%g|factors%s|alpha%s|k%d|seed%d|thr%d|wm%d" % (CODE_VER_FED, R8_MAX_PARTNERS, TEST_FRAC_NET, R8_C_FACTORS, "-".join("%g" % a for a in NET_ALPHA_GRID), K_NEIGHBORS, SEED, int(R8_FP_THR), int(R8_WORKING_MEMORY_MB))


def r8_save_csv(df, name):
    assert name.startswith("R8_"), "names not starting with R8_ are not written: " + name
    path = R6_CSV_DIR / (name + ".csv")
    df.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.6g")
    emit("- saved `%s` (%d rows)" % (path.as_posix(), len(df)))
    return path


def r8_metrics(y, p):
    """metrics (AUROC, AUPRC, Brier, ECE, calibration slope, net benefit at the threshold R8_NB_T)."""
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    out = {"auroc": np.nan, "ece": np.nan, "cal_slope": np.nan}
    try:
        out["auroc"] = float(roc_auc_score(y, p))
        out["auprc"] = float(average_precision_score(y, p))
        out["brier"] = float(brier_score_loss(y, p))
        out["ece"] = float(compute_ece(y, p))
        pc = np.clip(p, 1e-6, 1 - 1e-6)
        lp = np.log(pc / (1 - pc))
        if np.isfinite(lp).all() and lp.std() > 1e-9:
            out["cal_slope"] = float(LogisticRegression(max_iter=1000, C=1e6).fit(lp.reshape(-1, 1), y).coef_[0][0])
    except ValueError:
        pass
    for t in R8_NB_T:
        pred = p >= t
        tp = float(((pred == 1) & (y == 1)).sum())
        fp = float(((pred == 1) & (y == 0)).sum())
        out["nb@%s" % t] = tp / len(y) - (fp / len(y)) * (t / (1 - t))
    return out


def r8_topk_cos(Xb, yb, Xq, k):
    """_topk_cos: (distance, label) of the top k by cosine distance."""
    kk = int(min(k, len(Xb)))
    nn = NearestNeighbors(n_neighbors=kk, metric="cosine").fit(Xb)
    d, i = nn.kneighbors(Xq)
    return d, yb[i]


def r8_p_from(d, yk):
    w = 1.0 / (d + R8_EPS_W)
    return (w * yk).sum(axis=1) / np.maximum(w.sum(axis=1), 1e-12)


def r8_proto(Xb, yb, n_p, seed):
    """_proto: KMeans cluster centers and the event fraction within the clusters."""
    k = int(min(n_p, max(5, len(Xb) // 2)))
    km = KMeans(n_clusters=k, n_init=4, random_state=seed).fit(Xb)
    lab = km.labels_
    yp = np.array([yb[lab == c].mean() if (lab == c).any() else yb.mean() for c in range(k)])
    return km.cluster_centers_, yp


def r8_a_schemes(fs, C, oc, h, bk, te, partners, part_idx, a_key, dp_tag, rng_raw):
    """body of _run_fed_site: from the bank bk and test te gives the retrieval estimates of the eight modes, the network size curve and the mixing weight selection.
    if dp_tag is None, the noise generator key of the original split, otherwise appended to the end of the key (different noise per fold of the three folds). Returns: (retrieval estimates dict, size curve dict, selected alpha, whether subsampled)."""
    Xg, y, pb1 = C["Xg"], C["y"], C["pb1"]
    Xte, Xbk, ybk = Xg[te], Xg[bk], y[bk]
    a_local = inner_cv_bank(Xbk, ybk, pb1[bk], RETRIEVAL_FN, key=a_key)["alpha"]           # the mixing weight is chosen only from the own bank (the test is not used)
    d0, y0 = r8_topk_cos(Xbk, ybk.astype(float), Xte, R8_K_RETR)
    P = {"FR0_local": r8_p_from(d0, y0)}
    per_d, per_y = {}, {}
    for g in partners:
        gi = part_idx[g]
        per_d[g], per_y[g] = r8_topk_cos(Xg[gi], y[gi].astype(float), Xte, R8_K_RETR)

    def merge_exact(gs):
        D = np.hstack([d0] + [per_d[g] for g in gs])
        Y = np.hstack([y0] + [per_y[g] for g in gs])
        take = np.argsort(D, axis=1)[:, :R8_K_RETR]
        r = np.arange(len(D))[:, None]
        return r8_p_from(D[r, take], Y[r, take])

    def merge_agg(gs, eps=None, rng_=None):
        num = (1.0 / (d0 + R8_EPS_W) * y0).sum(axis=1)
        den = (1.0 / (d0 + R8_EPS_W)).sum(axis=1)
        for g in gs:
            w = 1.0 / (per_d[g] + R8_EPS_W)
            n_g, d_g = (w * per_y[g]).sum(axis=1), w.sum(axis=1)
            if eps is not None:
                sc = float(np.median(d_g)) / max(R8_K_RETR, 1)
                n_g = n_g + rng_.laplace(0.0, sc / eps, n_g.shape)
                d_g = np.maximum(d_g + rng_.laplace(0.0, sc / eps, d_g.shape), 1e-9)
            num, den = num + n_g, den + d_g
        return num / np.maximum(den, 1e-12)

    P["FR1_fed_exact"] = merge_exact(partners)
    P["FR2_fed_aggregate"] = merge_agg(partners)
    for e in R8_DP_EPS:
        rr = rng_of("dp", oc, int(h), e) if dp_tag is None else rng_of("dp", oc, int(h), e, dp_tag)
        P["FR3_fed_agg_dp%d" % int(e)] = merge_agg(partners, eps=e, rng_=rr)
    Xp, yp = [], []
    for g in partners:
        k_ = (fs, oc, int(g))
        if k_ not in R8_PROTO:
            gi = part_idx[g]
            R8_PROTO[k_] = r8_proto(Xg[gi], y[gi].astype(float), R8_N_PROTO_SITE, stable_seed("proto", oc, int(g)))
        cx, cy = R8_PROTO[k_]
        Xp.append(cx)
        yp.append(cy)
    Xp, yp = np.vstack(Xp), np.concatenate(yp)
    dp_, yp_ = r8_topk_cos(Xp, yp, Xte, R8_K_RETR)
    P["FR4_proto_network"] = r8_p_from(dp_, yp_)
    Dm, Ym = np.hstack([d0, dp_]), np.hstack([y0, yp_])
    tk = np.argsort(Dm, axis=1)[:, :R8_K_RETR]
    rr_ = np.arange(len(Dm))[:, None]
    P["FR5_local_plus_proto"] = r8_p_from(Dm[rr_, tk], Ym[rr_, tk])
    raw = np.concatenate([bk] + [part_idx[g] for g in partners])
    subsampled = len(raw) > 60000
    if subsampled:
        raw = rng_raw.choice(raw, 60000, replace=False)
    dr, yr = r8_topk_cos(Xg[raw], y[raw].astype(float), Xte, R8_K_RETR)
    P["FR9_pooled_raw"] = r8_p_from(dr, yr)
    SW = {ns: merge_agg(partners[:ns]) for ns in R8_NET_SIZES if ns <= len(partners)}
    return P, SW, float(a_local), bool(subsampled)


def r8_a_rows(mode, oc, h, ys, base_p, retr, mix, sweep_mix, n_site, n_bank, n_test, n_part, alphas, subsampled, hp=None):
    """one row of a hospital: change versus frozen, calibration slope and net benefit per mode. retr and mix are dicts of arrays in test patient order."""
    exact_err = np.nan if subsampled else float(np.max(np.abs(retr["FR1_fed_exact"] - retr["FR9_pooled_raw"])))
    agg_err = float(np.mean(np.abs(retr["FR2_fed_aggregate"] - retr["FR1_fed_exact"])))
    agg_corr = float(np.corrcoef(retr["FR2_fed_aggregate"], retr["FR1_fed_exact"])[0, 1])
    n_flip = -1 if subsampled else int(np.sum(np.abs(retr["FR1_fed_exact"] - retr["FR9_pooled_raw"]) > R8_TOL["exact_err"]))      # number of queries whose probability difference exceeds exact_err (swaps of near ties in float32)
    base = r8_metrics(ys, base_p)
    hp_diff = np.nan if hp is None else float(np.max(np.abs(mix["FR0_local"] - hp)))                         # anchor: three-fold own-bank PRAM test predictions versus the hospital-level PRAM test predictions of the preceding cells (R5B_HPRAM)
    hp_over = -1 if hp is None else int(np.sum(np.abs(mix["FR0_local"] - hp) > R8_TOL["exact_err"]))
    hp_auc = np.nan if hp is None else abs(r8_metrics(ys, mix["FR0_local"])["auroc"] - float(roc_auc_score(ys, hp)))
    allm = list(mix.items()) + [("SWEEP_FR2_n%02d" % ns, v) for ns, v in sweep_mix.items()]
    mets = {m_: r8_metrics(ys, pm) for m_, pm in allm}
    auc_gap = np.nan if subsampled else abs(mets["FR1_fed_exact"]["auroc"] - mets["FR9_pooled_raw"]["auroc"])
    rows = []
    for m_, pm in allm:
        met = mets[m_]
        main = not m_.startswith("SWEEP")
        rows.append({"mode": mode, "outcome": oc, "hospitalid": int(h), "scheme": m_, "n_site": int(n_site), "n_bank": int(n_bank), "n_test": int(n_test), "n_partners": int(n_part if main else int(m_[-2:])),
                     "alpha": [float(a) for a in alphas], "exact_err": exact_err if main else np.nan, "exact_flips": n_flip if main else -1, "n_queries": int(len(ys)), "exact_auc_gap": auc_gap if main else np.nan, "agg_err": agg_err if main else np.nan, "agg_corr": agg_corr if main else np.nan,
                     "hp_max_diff": hp_diff if main else np.nan, "hp_n_over": hp_over if main else -1, "hp_auc_gap": hp_auc if main else np.nan,
                     "auroc_base": base["auroc"], "auroc": met["auroc"], "delta_auroc": met["auroc"] - base["auroc"], "cal_slope": met["cal_slope"],
                     "nb02": met["nb@0.2"], "delta_nb02": met["nb@0.2"] - base["nb@0.2"], "ece": met["ece"], "delta_ece": met["ece"] - base["ece"], "status": "ok"})
    return rows


def r8_a_partners(oc, h, rng):
    """uses random numbers in a fixed order (random permutation within the hospital -> partner shuffle). The three folds also keep this order so that the two methods have the same partner list."""
    sites, hid = R6_STATE["sites"][oc], R6_STATE["hid"][oc]
    partners = [g for g in sites if g != h]
    rng.shuffle(partners)
    partners = partners[:R8_MAX_PARTNERS]
    return partners, {g: np.where(hid == g)[0] for g in partners}


def r8_a_unit_orig(fs, oc, h):
    C = CTX[(fs, oc, "eICU")]
    y, pb1, hid = C["y"], C["pb1"], R6_STATE["hid"][oc]
    idx = np.where(hid == h)[0]
    rng = rng_of("a29c", oc, int(h))
    perm = rng.permutation(len(idx))
    n_te = max(50, int(R8_A_TEST_FRAC * len(idx)))
    te, bk = idx[perm[:n_te]], idx[perm[n_te:]]
    if len(np.unique(y[te])) < 2 or len(np.unique(y[bk])) < 2:
        return [{"mode": "orig", "outcome": oc, "hospitalid": int(h), "status": "not eligible"}]
    partners, part_idx = r8_a_partners(oc, h, rng)
    if len(partners) < 2:
        return [{"mode": "orig", "outcome": oc, "hospitalid": int(h), "status": "not eligible"}]
    P, SW, a_local, subs = r8_a_schemes(fs, C, oc, h, bk, te, partners, part_idx, "a29c|%s|%d" % (oc, int(h)), None, rng)
    mix = {m_: (1 - a_local) * pb1[te] + a_local * P[m_] for m_ in R8_A_SCHEMES}
    smix = {ns: (1 - a_local) * pb1[te] + a_local * v for ns, v in SW.items()}
    return r8_a_rows("orig", oc, h, y[te], pb1[te], P, mix, smix, len(idx), len(bk), len(te), len(partners), [a_local], subs)


def r8_a_unit_cv3(fs, oc, h):
    C = CTX[(fs, oc, "eICU")]
    y, pb1, hid = C["y"], C["pb1"], R6_STATE["hid"][oc]
    idx = np.where(hid == h)[0]
    ys = y[idx]
    rng = rng_of("a29c", oc, int(h))
    rng.permutation(len(idx))                                   # keeps the same random number order as the original split method so that the partner list is the same (the permutation itself is not used)
    partners, part_idx = r8_a_partners(oc, h, rng)
    n_sp = min(N_SITE_SPLITS, int(ys.sum()), int(len(ys) - ys.sum()))
    if n_sp < 2 or len(partners) < 2:
        return [{"mode": "cv3", "outcome": oc, "hospitalid": int(h), "status": "not eligible"}]
    skf = StratifiedKFold(n_splits=n_sp, shuffle=True, random_state=stable_seed("a29", oc, int(h)))       # the same folds as the hospital level of the preceding cells (hospital_unit)
    retr = {m_: np.full(len(ys), np.nan) for m_ in R8_A_SCHEMES}
    mix = {m_: np.full(len(ys), np.nan) for m_ in R8_A_SCHEMES}
    smix = {ns: np.full(len(ys), np.nan) for ns in R8_NET_SIZES if ns <= len(partners)}
    alphas, n_banks, any_subs = [], [], False
    for fi, (bank_i, test_i) in enumerate(skf.split(np.zeros(len(ys)), ys)):
        bk, te = idx[bank_i], idx[test_i]
        if len(np.unique(y[bk])) < 2 or len(np.unique(y[te])) < 2:
            return [{"mode": "cv3", "outcome": oc, "hospitalid": int(h), "status": "not eligible"}]
        P, SW, a_local, subs = r8_a_schemes(fs, C, oc, h, bk, te, partners, part_idx, "b1|a29|%s|%d|%d" % (oc, int(h), fi), "cv%d" % fi, rng_of("a29c3raw", oc, int(h), fi))        # the mixing weight key is the same as the hospital level of the preceding cells (b1| + a29|outcome|hospital|fold of bank_predictions): one seed for the whole analysis
        any_subs = any_subs or subs
        alphas.append(a_local)
        n_banks.append(len(bk))
        for m_ in R8_A_SCHEMES:
            retr[m_][test_i] = P[m_]
            mix[m_][test_i] = (1 - a_local) * pb1[te] + a_local * P[m_]
        for ns, v in SW.items():
            smix[ns][test_i] = (1 - a_local) * pb1[te] + a_local * v
    hp = (R5B_HPRAM.get(fs) or {}).get((oc, int(h)))
    hp = None if (hp is None or len(hp) != len(ys)) else np.asarray(hp, dtype=float)
    return r8_a_rows("cv3", oc, h, ys, pb1[idx], retr, mix, smix, len(idx), int(round(float(np.mean(n_banks)))), len(idx), len(partners), alphas, any_subs, hp)


def r8_summary_rows(df, keys):
    """summary: number of hospitals | median change | improved hospitals % | median calibration slope | median net benefit (hospital level, not stay-weighted)."""
    out = []
    for k, g in df.groupby(keys, sort=False):
        d = g["delta_auroc"].to_numpy(float)
        row = dict(zip(keys, k if isinstance(k, tuple) else (k,)))
        row.update({"n_hospitals": int(g["hospitalid"].nunique()), "delta_median": float(np.nanmedian(d)), "delta_p10": float(np.nanquantile(d, 0.10)), "delta_p90": float(np.nanquantile(d, 0.90)),
                    "pct_improved": float(100.0 * np.nanmean(d > 0)), "cal_slope_median": float(np.nanmedian(g["cal_slope"])), "nb02_median": float(np.nanmedian(g["nb02"])),
                    "delta_ece_median": float(np.nanmedian(g["delta_ece"])), "pct_ece_lower": float(100.0 * np.nanmean(g["delta_ece"].to_numpy(float) < 0))})
        out.append(row)
    return pd.DataFrame(out)


def r8_compare(panel, summ, ref, item_col, with_slope=True):
    """compares the values from the original split method with the reference values. The tolerance is R8_TOL of the first cell."""
    rows = []
    for r_ in summ.itertuples():
        key = (r_.outcome, getattr(r_, item_col))
        if key not in ref:
            continue
        v = ref[key]
        rd, rp = v[0], v[1]
        rs_ = v[2] if (with_slope and len(v) > 2) else np.nan
        ok_d, ok_p = abs(r_.delta_median - rd) <= R8_TOL["dmed"], abs(r_.pct_improved - rp) <= R8_TOL["pct"]
        ok_s = True if np.isnan(rs_) else bool((not np.isnan(r_.cal_slope_median)) and abs(r_.cal_slope_median - rs_) <= R8_TOL["slope"])
        rows.append({"panel": panel, "outcome": key[0], "item": str(key[1]), "orig_delta_median": r_.delta_median, "ref_delta_median": rd, "orig_pct": r_.pct_improved, "ref_pct": rp, "orig_slope": r_.cal_slope_median, "ref_slope": rs_,
                     "ok_delta": bool(ok_d), "ok_pct": bool(ok_p), "ok_slope": bool(ok_s), "repro_ok": bool(ok_d and ok_p and ok_s)})
    return pd.DataFrame(rows)


def r8_count_row(panel, oc, n_got):
    ok = bool(n_got == R8_N_HOSP_REF[oc])
    return {"panel": panel, "outcome": oc, "item": "number of hospitals (reference 67 and 59)", "orig_delta_median": float(n_got), "ref_delta_median": float(R8_N_HOSP_REF[oc]), "orig_pct": np.nan, "ref_pct": np.nan, "orig_slope": np.nan, "ref_slope": np.nan,
            "ok_delta": ok, "ok_pct": True, "ok_slope": True, "repro_ok": ok}


def r8_run_a():
    if not R6_STATE["fs_hosp"] or "F42" not in R6_STATE["fs_hosp"]:
        R8_A["state"] = "not computed because the input is missing: " + (R6_STATE["n1_why"] or "there is no 42-column eICU context")
        emit("**The supplementary analysis A (federated retrieval) is not computed because the input is missing.** Reason: " + R8_A["state"])
        status("supplementary A federated retrieval", "not available", R8_A["state"])
        return
    sites = R6_STATE["sites"]
    order = [(fs, oc, int(h)) for fs in R8_FS_AB for oc in OUTCOMES for h in sites[oc]]
    emit("### supplementary A . federated retrieval, eight modes . original split (test 30%%, reproduction check) versus three folds (all stays of the hospital) , %d hospitals" % len(order))
    res = {"orig": R6Resumable("R8_a_orig"), "cv3": R6Resumable("R8_a_cv3")}
    t_loop, n_new, rows_all = time.time(), 0, []
    for k, (fs, oc, h) in enumerate(order):
        for mode, fn in (("orig", r8_a_unit_orig), ("cv3", r8_a_unit_cv3)):
            unit = ("a", mode, fs, oc, h, R8_CFG_A)
            was = res[mode].unit_done(unit)
            rows_all += r6_unit(res[mode], unit, lambda: fn(fs, oc, h))
            if not was:
                n_new += 1
        if (k + 1) % 10 == 1 or k == len(order) - 1:
            print("  [supplementary A] hospital %d/%d done, units computed this time %d , remaining time about %.0f min (estimated by the number of hospitals)" % (k + 1, len(order), n_new, ((time.time() - t_loop) / max(n_new, 1) * 2 * (len(order) - k - 1) / 60.0) if n_new else 0.0), flush=True)
    df = pd.DataFrame(rows_all)
    df = df[df["status"] == "ok"].reset_index(drop=True)
    R8_A["rows"] = df
    if not len(df):
        R8_A["state"] = "not available: there is no eligible hospital"
        status("supplementary A federated retrieval", "not available", R8_A["state"])
        return
    main = df[~df["scheme"].str.startswith("SWEEP")]
    summ = r8_summary_rows(main, ["mode", "outcome", "scheme"])
    sw = df[df["scheme"].str.startswith("SWEEP")].copy()
    sw["net_size"] = sw["scheme"].str[-2:].astype(int)
    sws = r8_summary_rows(sw, ["mode", "outcome", "net_size"])
    st = []
    for (mode, oc), g in main.groupby(["mode", "outcome"], sort=False):
        g1 = g.drop_duplicates("hospitalid")
        okq = g1["exact_flips"] >= 0
        st.append({"mode": mode, "outcome": oc, "n_hospitals": int(g["hospitalid"].nunique()), "exact_err_max": float(np.nanmax(g["exact_err"])) if g["exact_err"].notna().any() else np.nan,
                   "exact_flip_share": float(g1.loc[okq, "exact_flips"].sum() / max(g1.loc[okq, "n_queries"].sum(), 1)), "exact_auc_gap_max": float(np.nanmax(g1["exact_auc_gap"])) if g1["exact_auc_gap"].notna().any() else np.nan,
                   "agg_err_median": float(np.nanmedian(g1["agg_err"])), "agg_corr_median": float(np.nanmedian(g1["agg_corr"])), "n_partners_median": float(np.median(g1["n_partners"]))})
    st = pd.DataFrame(st)
    R8_A["summary"], R8_A["sweep"], R8_A["struct"] = summ, sws, st
    emit_table("supplementary A: summary per mode (hospital level . delta = AUROC change versus the frozen logistic . mode orig = original split test 30% (original calculation as is) , cv3 = three folds all stays of the hospital)", summ, digits=4, max_rows=40)
    emit_table("supplementary A: structural property (maximum difference between the exact method and the pooled bank . mean absolute difference and correlation between the aggregate method and the exact method)", st, digits=6)
    rp = r8_compare("A", summ[summ["mode"] == "orig"], R8_REF_A, "scheme")
    rs = r8_compare("A", sws[sws["mode"] == "orig"], R8_REF_SWEEP, "net_size", with_slope=False)
    if len(rs):
        rs["item"] = rs["item"].map(lambda v: "SWEEP_FR2_n%02d" % int(v))
    srows = []
    for r_ in st[st["mode"] == "orig"].itertuples():
        ok_d, ok_p, ok_s = bool(abs(r_.agg_err_median - R8_REF_A_STRUCT["agg_err_median"]) <= R8_TOL["agg_err"]), bool(r_.exact_flip_share <= R8_TOL["exact_flip_share"] and r_.exact_auc_gap_max <= R8_TOL["exact_auc_gap"]), bool(abs(r_.agg_corr_median - R8_REF_A_STRUCT["agg_corr_median"]) <= R8_TOL["agg_corr"])
        srows.append({"panel": "A", "outcome": r_.outcome, "item": R8_STRUCT_ITEM, "orig_delta_median": r_.agg_err_median, "ref_delta_median": R8_REF_A_STRUCT["agg_err_median"], "orig_pct": r_.exact_flip_share,
                      "ref_pct": 0.0, "orig_slope": r_.agg_corr_median, "ref_slope": R8_REF_A_STRUCT["agg_corr_median"], "ok_delta": ok_d, "ok_pct": ok_p, "ok_slope": ok_s, "repro_ok": bool(ok_d and ok_p and ok_s)})
    nh = [r8_count_row("A", oc, g["hospitalid"].nunique()) for oc, g in main[main["mode"] == "orig"].groupby("outcome", sort=False)]
    arows = []                                                    # anchor row: are the test predictions of three-fold FR0_local the same as the test predictions of the hospital-level PRAM of the preceding cells. Column meaning: orig_delta_median = maximum over hospitals of the maximum probability difference within the hospital , orig_pct = fraction of queries whose probability difference exceeds R8_TOL[exact_err] , orig_slope = maximum over hospitals of the AUROC difference
    for oc, g in main[main["mode"] == "cv3"].groupby("outcome", sort=False):
        g1 = g.drop_duplicates("hospitalid")
        has = g1["hp_n_over"] >= 0
        n_q = float(g1.loc[has, "n_queries"].sum())
        sh_ = float(g1.loc[has, "hp_n_over"].sum() / n_q) if n_q > 0 else np.nan
        mx_ = float(g1.loc[has, "hp_max_diff"].max()) if has.any() else np.nan
        au_ = float(g1.loc[has, "hp_auc_gap"].max()) if has.any() else np.nan
        ok_d, ok_p, ok_s = bool(has.any() and int(has.sum()) == len(g1)), bool(np.isfinite(sh_) and sh_ <= R8_TOL["exact_flip_share"]), bool(np.isfinite(au_) and au_ <= R8_TOL["exact_auc_gap"])
        arows.append({"panel": "A", "outcome": oc, "item": R8_ANCHOR_ITEM, "orig_delta_median": mx_, "ref_delta_median": 0.0, "orig_pct": sh_, "ref_pct": 0.0, "orig_slope": au_, "ref_slope": 0.0,
                      "ok_delta": ok_d, "ok_pct": ok_p, "ok_slope": ok_s, "repro_ok": bool(ok_d and ok_p and ok_s)})
    rep = pd.concat([rp, rs, pd.DataFrame(srows), pd.DataFrame(nh), pd.DataFrame(arows)], ignore_index=True)
    R8_A["repro"] = rep
    emit_table("supplementary A reproduction check: values from the original split method versus the reference values (tolerance: median %g , improved hospitals %g percentage points , slope %g . The package versions of this PC may differ from those of the reference values)" % (R8_TOL["dmed"], R8_TOL["pct"], R8_TOL["slope"]),
               rep[["outcome", "item", "orig_delta_median", "ref_delta_median", "orig_pct", "ref_pct", "ok_delta", "ok_pct", "ok_slope", "repro_ok"]], digits=4, max_rows=70)
    n_bad = int((~rep["repro_ok"]).sum())
    emit("- A reproduction check: of %d rows ok %d , mismatch %d . The three-fold values of mismatched rows are not read. Judged per row." % (len(rep), int(rep["repro_ok"].sum()), n_bad))
    r8_save_csv(summ, "R8_A_scheme_summary")
    r8_save_csv(sws, "R8_A_network_size_curve")
    r8_save_csv(st, "R8_A_structure")
    R8_A["state"] = "done" if n_bad == 0 else "reproduction partly mismatched (%d rows)" % n_bad
    status("supplementary A federated retrieval", "done" if n_bad == 0 else "partial", "hospitals %d x two methods (computed this time %d)" % (len(order), n_new))


if not R6_STOP:
    _rep8 = len(REPORT_LINES)
    if not R8_RUN_A:
        R8_A["state"] = "off"
    elif R8_K_RETR != K_NEIGHBORS or R8_MAX_PARTNERS != MAX_PARTNERS:
        R8_A["state"] = "settings check failed"
        emit("**The settings check of supplementary A failed , not computed.** R8_K_RETR(%s) must equal K_NEIGHBORS(%s) of the preceding cells and R8_MAX_PARTNERS(%s) must equal MAX_PARTNERS(%s) of the preceding cells." % (R8_K_RETR, K_NEIGHBORS, R8_MAX_PARTNERS, MAX_PARTNERS))
        status("supplementary A federated retrieval", "not available", R8_A["state"])
    else:
        r8_run_a()
    R8_LINES["a"] = [str(_l) for _l in REPORT_LINES[_rep8:]]
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)

In [ ]:
# ================================================================
# PRAM supplementary cell 10/13 (B) . supplementary analyses, three folds: federated learning comparison
# original split: per hospital training 60% | weighting 20% | test 20% once (R8_B_TRAIN_FRAC and R8_B_WEIGHT_FRAC). Three folds: the share R8_B_REST_TRAIN_SHARE with which what remains after removing test 1/3 is divided into training and weighting
#   has no reference value. If None in the first cell, the three folds are not computed and only the reproduction of the original split is given.
# ================================================================
R8_B_METHODS = ("frozen_lr", "local_lr", "pooled_dev", "pooled_net", "fed_avg", "fed_stack", "fed_stack_meta", "pram_frozen", "pram_fedstack")
R8_REF_B = {("AKI_168h", "fed_avg"): (0.0071, 56.7164, 1.0165), ("AKI_168h", "fed_stack"): (0.0111, 55.2239, 1.0457), ("AKI_168h", "fed_stack_meta"): (0.0086, 53.7313, 1.0613),
            ("AKI_168h", "frozen_lr"): (0.0000, 0.0000, 0.8429), ("AKI_168h", "local_lr"): (-0.0663, 29.8507, 0.3207), ("AKI_168h", "pooled_dev"): (0.0043, 64.1791, 0.9084),
            ("AKI_168h", "pooled_net"): (0.0363, 73.1343, 0.9922), ("AKI_168h", "pram_fedstack"): (0.0032, 52.2388, 1.1690), ("AKI_168h", "pram_frozen"): (0.0016, 52.2388, 1.0274),
            ("mortality_168h", "fed_avg"): (0.0400, 83.0508, 0.9149), ("mortality_168h", "fed_stack"): (0.0357, 66.1017, 0.9941), ("mortality_168h", "fed_stack_meta"): (0.0338, 66.1017, 0.9920),
            ("mortality_168h", "frozen_lr"): (0.0000, 0.0000, 0.3979), ("mortality_168h", "local_lr"): (-0.0322, 44.0678, 0.4171), ("mortality_168h", "pooled_dev"): (0.0196, 86.4407, 0.7228),
            ("mortality_168h", "pooled_net"): (0.0389, 83.0508, 1.0002), ("mortality_168h", "pram_fedstack"): (0.0352, 67.7966, 1.1239), ("mortality_168h", "pram_frozen"): (0.0110, 66.1017, 0.8091)}


def r8_b_prepare(fs, oc, mode):
    """prepares the folds. Returns: list of folds. One fold for the original split and three folds for the three-fold method (per fold per hospital the training, weighting and test split and the base model)."""
    C = CTX[(fs, oc, "eICU")]
    Xg, y = C["Xg"], C["y"]
    hid = R6_STATE["hid"][oc]
    elig = set(int(g) for g in R6_STATE["sites"][oc])
    cand = [int(h) for h in pd.unique(hid[pd.notna(hid)]) if int(h) in elig]            # in order of appearance
    if mode == "orig":
        folds = [{"split": {}, "base": {}}]
        for h in cand:
            idx = np.where(hid == h)[0]
            rng = rng_of("a30split", oc, int(h))
            perm = rng.permutation(len(idx))
            n_tr, n_wt = int(R8_B_TRAIN_FRAC * len(idx)), int(R8_B_WEIGHT_FRAC * len(idx))
            tr, wt, te = idx[perm[:n_tr]], idx[perm[n_tr:n_tr + n_wt]], idx[perm[n_tr + n_wt:]]
            if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2 or len(np.unique(y[wt])) < 2:
                continue
            folds[0]["split"][h] = (tr, wt, te)
            folds[0]["base"][h] = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xg[tr], y[tr])
        keep = list(folds[0]["split"])
    else:
        sp3 = {}
        for h in cand:
            idx = np.where(hid == h)[0]
            ys = y[idx]
            skf = StratifiedKFold(n_splits=N_SITE_SPLITS, shuffle=True, random_state=stable_seed("a29", oc, int(h)))        # the same folds as the hospital level of the preceding cells
            per = []
            for fi, (rest_i, test_i) in enumerate(skf.split(np.zeros(len(ys)), ys)):
                rest, te = idx[rest_i], idx[test_i]
                perm = rng_of("a30split3", oc, int(h), fi).permutation(len(rest))
                n_tr = int(R8_B_REST_TRAIN_SHARE * len(rest))
                per.append((rest[perm[:n_tr]], rest[perm[n_tr:]], te))
            if all(len(np.unique(y[a])) > 1 and len(np.unique(y[b])) > 1 and len(np.unique(y[c])) > 1 for a, b, c in per):
                sp3[h] = per
        keep = list(sp3)
        folds = []
        for fi in range(N_SITE_SPLITS):
            f = {"split": {h: sp3[h][fi] for h in keep}, "base": {}}
            for h in keep:
                f["base"][h] = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xg[f["split"][h][0]], y[f["split"][h][0]])
            folds.append(f)
    for f in folds:
        f["sites"] = sorted(keep, key=lambda g: -len(f["split"][g][0]))
    return folds


def r8_b_fit(C, oc, h, f, tag):
    """body of _run_fed_site. In fold f, hospital h is the target and the rest are used as the network. Returns: (test patient indices, predictions per method, two selected alphas, number of network hospitals) or None."""
    Xg, y, pb1 = C["Xg"], C["y"], C["pb1"]
    Xs_src, ys_src = C["Xs_src"], C["ys_src"]
    tr, wt, te = f["split"][h]
    others = [g for g in f["sites"] if g != h]
    if len(others) < 3:
        return None
    ytr = y[tr]
    Xte, Xtr = Xg[te], Xg[tr]
    base, split = f["base"], f["split"]
    R = {"frozen_lr": pb1[te]}
    R["local_lr"] = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    R["pooled_dev"] = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(np.vstack([Xs_src, Xtr]), np.concatenate([ys_src, ytr])).predict_proba(Xte)[:, 1]
    net = np.concatenate([split[g][0] for g in others])
    if len(net) > R8_B_POOL_NET_CAP:
        net = (rng_of("a30net", oc, int(h)) if tag is None else rng_of("a30net", oc, int(h), tag)).choice(net, R8_B_POOL_NET_CAP, replace=False)
    R["pooled_net"] = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(np.vstack([Xg[net], Xtr]), np.concatenate([y[net], ytr])).predict_proba(Xte)[:, 1]
    W = np.array([len(split[g][0]) for g in others], dtype=float)
    W = W / W.sum()
    coef = np.zeros_like(base[others[0]].coef_)
    icpt = np.zeros_like(base[others[0]].intercept_)
    for w_, g in zip(W, others):
        coef += w_ * base[g].coef_
        icpt += w_ * base[g].intercept_
    R["fed_avg"] = 1.0 / (1.0 + np.exp(-(Xte @ coef.ravel() + icpt[0])))
    meta_idx = np.concatenate([split[g][1] for g in others])
    stack = lambda Xrows: np.column_stack([base[g].predict_proba(Xrows)[:, 1] for g in others])
    Zm, ym = stack(Xg[meta_idx]), y[meta_idx]
    meta = LogisticRegression(max_iter=2000, C=1.0, random_state=SEED).fit(Zm, ym)
    Zte = stack(Xte)
    R["fed_stack"] = meta.predict_proba(Zte)[:, 1]
    meta_site = np.concatenate([np.repeat([[np.log10(len(split[g][0])), float(y[split[g][0]].mean())]], len(split[g][1]), axis=0) for g in others])
    meta2 = LogisticRegression(max_iter=2000, C=1.0, random_state=SEED).fit(np.hstack([Zm, meta_site]), ym)
    te_site = np.repeat([[np.log10(len(tr)), float(ytr.mean())]], len(te), axis=0)
    R["fed_stack_meta"] = meta2.predict_proba(np.hstack([Zte, te_site]))[:, 1]
    p_retr = RETRIEVAL_FN({"X_tr_s": Xtr, "y_tr": ytr, "X_te_s": Xte, "y_te": y[te], "mi": None, "mi_prob": None})
    kf = "a30f|%s|%d" % (oc, int(h)) if tag is None else "a30f|%s|%d|%s" % (oc, int(h), tag)
    ks = "a30s|%s|%d" % (oc, int(h)) if tag is None else "a30s|%s|%d|%s" % (oc, int(h), tag)
    a_f = inner_cv_bank(Xtr, ytr, pb1[tr], RETRIEVAL_FN, key=kf)["alpha"]
    R["pram_frozen"] = (1 - a_f) * pb1[te] + a_f * p_retr
    p_stack_tr = meta.predict_proba(stack(Xtr))[:, 1]
    a_s = inner_cv_bank(Xtr, ytr, p_stack_tr, RETRIEVAL_FN, key=ks)["alpha"]                     # the mixing weight is chosen by within-bank stacking predictions (avoids self-prediction)
    R["pram_fedstack"] = (1 - a_s) * R["fed_stack"] + a_s * p_retr
    return te, R, float(a_f), float(a_s), len(others)


def r8_b_unit(fs, oc, h, mode, folds):
    C = CTX[(fs, oc, "eICU")]
    y = C["y"]
    fail = [{"mode": mode, "outcome": oc, "hospitalid": int(h), "method": "-", "status": "not eligible"}]
    ys_l, pr_l, af_l, as_l, n_net = [], {m_: [] for m_ in R8_B_METHODS}, [], [], 0
    n_tr = n_all = 0
    for fi, f in enumerate(folds):
        if h not in f["split"]:
            return fail
        out = r8_b_fit(C, oc, h, f, None if mode == "orig" else "cv%d" % fi)
        if out is None:
            return fail
        te, R, a_f, a_s, nn = out
        ys_l.append(y[te])
        for m_ in R8_B_METHODS:
            pr_l[m_].append(R[m_])
        af_l.append(a_f)
        as_l.append(a_s)
        n_net = nn
        tr, wt, _t = f["split"][h]
        n_tr += len(tr)
        n_all += len(tr) + len(wt) + len(te)
    ys = np.concatenate(ys_l)
    met = {m_: r8_metrics(ys, np.concatenate(pr_l[m_])) for m_ in R8_B_METHODS}
    b = met["frozen_lr"]
    rows = []
    for m_ in R8_B_METHODS:
        rows.append({"mode": mode, "outcome": oc, "hospitalid": int(h), "method": m_, "n_site": int(n_all // len(folds)), "n_train": int(n_tr // len(folds)), "n_test": int(len(ys)), "n_network": int(n_net),
                     "alpha_frozen": float(np.mean(af_l)), "alpha_fedstack": float(np.mean(as_l)), "auroc": met[m_]["auroc"], "delta_auroc": met[m_]["auroc"] - b["auroc"], "cal_slope": met[m_]["cal_slope"], "nb02": met[m_]["nb@0.2"],
                     "delta_nb02": met[m_]["nb@0.2"] - b["nb@0.2"], "ece": met[m_]["ece"], "delta_ece": met[m_]["ece"] - b["ece"], "status": "ok"})
    return rows


def r8_run_b():
    if not R6_STATE["fs_hosp"] or "F42" not in R6_STATE["fs_hosp"]:
        R8_B["state"] = "not computed because the input is missing: " + (R6_STATE["n1_why"] or "there is no 42-column eICU context")
        emit("**The supplementary analysis B (federated learning) is not computed because the input is missing.** Reason: " + R8_B["state"])
        status("supplementary B federated learning", "not available", R8_B["state"])
        return
    modes = ["orig"] + (["cv3"] if R8_B_REST_TRAIN_SHARE is not None else [])
    if R8_B_REST_TRAIN_SHARE is None:
        emit("- **The supplementary B three folds are not computed**: R8_B_REST_TRAIN_SHARE in the first cell is None (no reference value: how to divide what remains after removing test 1/3 into training and weighting). Only the reproduction of the original split (training %g | weighting %g | test %g) is given." % (R8_B_TRAIN_FRAC, R8_B_WEIGHT_FRAC, 1 - R8_B_TRAIN_FRAC - R8_B_WEIGHT_FRAC))
    else:
        emit("- supplementary B three folds: training share of what remains after removing test 1/3 R8_B_REST_TRAIN_SHARE = %g (the rest is weighting) . This value is fixed in the reading rules." % R8_B_REST_TRAIN_SHARE)
    emit("### supplementary B . federated learning comparison . method %s" % " · ".join(modes))
    rows_all, n_new, t_loop = [], 0, time.time()
    for mode in modes:
        res = R6Resumable("R8_b_" + mode)
        for oc in OUTCOMES:
            folds = None
            sites_oc = [int(h) for h in R6_STATE["sites"][oc]]
            for k, h in enumerate(sites_oc):
                unit = ("b", mode, "F42", oc, h, R8_CFG_B if mode == "cv3" else R8_CFG_B0)
                was = res.unit_done(unit)
                if not was and folds is None:
                    folds = r8_b_prepare("F42", oc, mode)
                rows_all += r6_unit(res, unit, lambda: r8_b_unit("F42", oc, h, mode, folds))
                if not was:
                    n_new += 1
                    if n_new % 10 == 1:
                        print("  [supplementary B] %s %s hospital %d/%d done , remaining time about %.0f min (estimated by the number of hospitals)" % (mode, oc, k + 1, len(sites_oc), (time.time() - t_loop) / max(n_new, 1) * max(len(sites_oc) - k - 1, 0) / 60.0), flush=True)
    df = pd.DataFrame(rows_all)
    df = df[df["status"] == "ok"].reset_index(drop=True) if len(df) else df
    R8_B["rows"] = df
    if not len(df):
        R8_B["state"] = "not available: there is no eligible hospital"
        status("supplementary B federated learning", "not available", R8_B["state"])
        return
    summ = r8_summary_rows(df, ["mode", "outcome", "method"])
    R8_B["summary"] = summ
    emit_table("supplementary B: summary per method (hospital level . delta = AUROC change versus the frozen logistic on the same test patients . orig = original split training 60 | weighting 20 | test 20 , cv3 = three folds all stays of the hospital)", summ, digits=4, max_rows=40)
    rep = r8_compare("B", summ[summ["mode"] == "orig"], R8_REF_B, "method")
    nh = [r8_count_row("B", oc, g["hospitalid"].nunique()) for oc, g in df[(df["mode"] == "orig") & (df["method"] == "frozen_lr")].groupby("outcome", sort=False)]
    rep = pd.concat([rep, pd.DataFrame(nh)], ignore_index=True)
    R8_B["repro"] = rep
    emit_table("supplementary B reproduction check: values from the original split method versus the reference values (tolerance the same as A)", rep[["outcome", "item", "orig_delta_median", "ref_delta_median", "orig_pct", "ref_pct", "ok_delta", "ok_pct", "ok_slope", "repro_ok"]], digits=4, max_rows=40)
    n_bad = int((~rep["repro_ok"]).sum())
    emit("- B reproduction check: of %d rows ok %d , mismatch %d . Judged per row (the three-fold values of mismatched rows are not read)." % (len(rep), int(rep["repro_ok"].sum()), n_bad))
    r8_save_csv(summ, "R8_B_method_summary")
    R8_B["state"] = ("done" if n_bad == 0 else "reproduction partly mismatched (%d rows)" % n_bad) + ("" if "cv3" in modes else " . three folds not computed (no split ratio value)")
    status("supplementary B federated learning", "done" if (n_bad == 0 and "cv3" in modes) else "partial", R8_B["state"])


if not R6_STOP:
    _rep8 = len(REPORT_LINES)
    if not R8_RUN_B:
        R8_B["state"] = "off"
    else:
        r8_run_b()
    R8_LINES["b"] = [str(_l) for _l in REPORT_LINES[_rep8:]]
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)

In [ ]:
# ================================================================
# PRAM supplementary cell 11/13 (C) . supplementary analyses, three folds: retrieval network of unlabeled hospitals
# original split = network_unit of the preceding cells as it is (test 30%, reproduction check) . three folds = the same partner list and the same mixing weight grid with the test being all stays of the hospital (this analysis does not use a hospital bank, so combining test 1/3 x 3 gives all stays)
# ================================================================
R8_REF_C = {0.05: (0.003, 77.6, 0.004, 81.4, 0.0018, 73.1, 0.001, 67.8), 0.1: (0.0052, 80.6, 0.0064, 89.8, 0.0042, 73.1, 0.0014, 62.7), 0.15: (0.0077, 83.6, 0.009, 89.8, 0.0059, 80.6, 0.0024, 72.9), 0.2: (0.0106, 85.1, 0.0133, 88.1, 0.0077, 82.1, 0.0029, 71.2), 0.25: (0.0133, 83.6, 0.0153, 88.1, 0.0103, 80.6, 0.0042, 69.5), 0.3: (0.0168, 83.6, 0.0174, 88.1, 0.0101, 76.1, 0.0043, 69.5), 0.35: (0.0193, 80.6, 0.0191, 89.8, 0.0123, 76.1, 0.0043, 66.1), 0.4: (0.022, 80.6, 0.0206, 88.1, 0.0125, 80.6, 0.0056, 67.8), 0.45: (0.024, 80.6, 0.022, 89.8, 0.0165, 79.1, 0.0043, 64.4), 0.5: (0.0263, 79.1, 0.0245, 89.8, 0.0201, 79.1, 0.0037, 64.4), 0.75: (0.0297, 77.6, 0.0369, 88.1, 0.0191, 67.2, 0.0045, 54.2), 1.0: (0.0156, 58.2, 0.0359, 76.3, -0.0021, 46.3, -0.0072, 44.1)}      # table values of Supplementary S11 Panel C (values of Supplementary S11 Panel C): alpha -> (AKI42 median, %, death42 median, %, AKI33 median, %, death33 median, %)
R8_C_FS_COLS = (("F42", "AKI_168h"), ("F42", "mortality_168h"), ("F33", "AKI_168h"), ("F33", "mortality_168h"))


def r8_c_unit_all(C, oc, h, factors, alphas):
    """the same formula as network_unit of the preceding cells with the test changed to all stays of the hospital. Random numbers are used in the same order so that the partner list is the same (permutation -> partner shuffle)."""
    hid, sites = R6_STATE["hid"][oc], R6_STATE["sites"][oc]
    Xg, y, pb1 = C["Xg"], C["y"], C["pb1"]
    idx = np.where(hid == h)[0]
    rng = rng_of("a32", oc, int(h))
    rng.permutation(len(idx))
    n_te = max(60, int(TEST_FRAC_NET * len(idx)))
    yte = y[idx]
    if len(np.unique(yte)) < 2 or len(idx) - n_te < 60:
        return None
    Xte, p1te = Xg[idx], pb1[idx]
    partners = [g for g in sites if g != h]
    rng.shuffle(partners)
    partners = partners[:R8_MAX_PARTNERS]
    if len(partners) < 4:
        return None
    own_prev = float(y[idx].mean())
    base = full_metrics(yte, p1te)
    nb_c = "nb@%s" % float(T_SITE[oc])
    rows = []
    for r in factors:
        P_list, prevs = [], []
        for g in partners:
            gi = np.where(hid == g)[0]
            if r != 1.0:
                gi = _resample_prev(gi, y, r, stable_seed("a7", oc, int(h), int(g), r))
            d, yk = r8_topk_cos(Xg[gi], y[gi].astype(float), Xte, K_NEIGHBORS)
            P_list.append(_site_reply(d, yk))
            prevs.append(float(y[gi].mean()))
        p_net = np.column_stack(P_list).mean(axis=1)
        for a in alphas:
            m = full_metrics(yte, (1 - a) * p1te + a * p_net)
            rows.append({"outcome": oc, "hospitalid": int(h), "factor": float(r), "alpha": float(a), "n_test": int(len(idx)), "n_partners": int(len(partners)), "prev_own": own_prev, "prev_partner": float(np.mean(prevs)),
                         "prev_diff": float(np.mean(prevs)) - own_prev, "d_auroc": m["auroc"] - base["auroc"], "base_auroc": base["auroc"], "d_ece": m["ece"] - base["ece"], "slope": m["slope"], "intercept": m["intercept"],
                         "d_nb_rep": m[nb_c] - base[nb_c]})
    return rows


def r8_c_slim(rows):
    """keeps only the columns R8 uses from the rows of network_unit of the preceding cells (original split method)."""
    return [{"outcome": r["outcome"], "hospitalid": r["hospitalid"], "factor": r["factor"], "alpha": r["alpha"], "n_test": r["n_test"], "n_partners": r["n_partners"], "prev_own": r["prev_own"], "prev_partner": r["prev_partner"],
             "prev_diff": r["prev_diff"], "d_auroc": r["d_auroc"], "base_auroc": r["base_auroc"], "d_ece": r["d_ece"], "slope": r["slope"], "intercept": r["intercept"], "d_nb_rep": r["d_nb_rep"]} for r in (rows or [])]


def r8_sign_p(d):
    """hospital-level sign test: two-sided exact binomial test (P = 0.5) of the number of positive hospitals among hospitals whose change is not 0. The definition used for the reference values is not verified here."""
    d = np.asarray(d, dtype=float)
    d = d[np.isfinite(d) & (d != 0)]
    if not len(d):
        return np.nan
    bt = getattr(r8_stats, "binomtest", None)           # scipy 1.7 or later. For a lower version, binom_test
    return float(bt(int((d > 0).sum()), len(d), 0.5).pvalue) if bt is not None else float(r8_stats.binom_test(int((d > 0).sum()), len(d), 0.5))


def r8_c_summary(df):
    out = []
    for (mode, fs, oc, fa, al), g in df.groupby(["mode", "fs", "outcome", "factor", "alpha"], sort=False):
        d = g["d_auroc"].to_numpy(float)
        out.append({"mode": mode, "fs": fs, "outcome": oc, "factor": fa, "alpha": al, "n_hospitals": int(g["hospitalid"].nunique()), "d_auroc_median": float(np.nanmedian(d)), "pct_improved": float(100.0 * np.nanmean(d > 0)),
                    "sign_test_p": r8_sign_p(d), "d_ece_median": float(np.nanmedian(g["d_ece"])), "pct_ece_lower": float(100.0 * np.nanmean(g["d_ece"].to_numpy(float) < 0)), "slope_median": float(np.nanmedian(g["slope"])),
                    "d_nb_rep_median": float(np.nanmedian(g["d_nb_rep"]))})
    return pd.DataFrame(out)


def r8_run_c():
    fsl = [fs for fs in R6_STATE["fs_hosp"]]
    if not fsl:
        R8_C["state"] = "not computed because the input is missing: " + R6_STATE["n1_why"]
        emit("**The supplementary analysis C (retrieval network) is not computed because the input is missing.** Reason: " + R8_C["state"])
        status("supplementary C retrieval network", "not available", R8_C["state"])
        return
    sites = R6_STATE["sites"]
    order = [(fs, oc, int(h)) for fs in fsl for oc in OUTCOMES for h in sites[oc]]
    emit("### supplementary C . retrieval network of unlabeled hospitals (Supplementary S11 Panel C to E and one row of Table 6) . original split (test 30%%, network_unit of the preceding cells) versus all stays of the hospital , hospital units %d x two methods x prevalence multipliers %s x mixing weights %d" % (len(order), list(R8_C_FACTORS), len(NET_ALPHA_GRID)))
    res = {fs: {"orig": R6Resumable("R8_c_orig_" + fs), "cv3": R6Resumable("R8_c_cv3_" + fs)} for fs in fsl}
    t_loop, n_new, rows_all = time.time(), 0, []
    for k, (fs, oc, h) in enumerate(order):
        C = CTX[(fs, oc, "eICU")]
        for mode in ("orig", "cv3"):
            unit = ("c", mode, fs, oc, h, R8_CFG_C)
            was = res[fs][mode].unit_done(unit)
            if mode == "orig":
                fn = lambda: r8_c_slim(network_unit(C, oc, h, R6_STATE["hid"][oc], R6_STATE["sites"][oc], list(R8_C_FACTORS), NET_ALPHA_GRID))
            else:
                fn = lambda: r8_c_unit_all(C, oc, h, list(R8_C_FACTORS), NET_ALPHA_GRID) or []
            rr = r6_unit(res[fs][mode], unit, fn)
            n_site_h = int(np.sum(R6_STATE["hid"][oc] == h))
            rows_all += [dict(r, mode=mode, fs=fs, n_site=n_site_h) for r in rr]
            if not was:
                n_new += 1
        if (k + 1) % 10 == 1 or k == len(order) - 1:
            print("  [supplementary C] %s %s hospital %d/%d done , remaining time about %.0f min (estimated by the number of hospitals)" % (fs, oc, k + 1, len(order), ((time.time() - t_loop) / max(n_new, 1) * 2 * (len(order) - k - 1) / 60.0) if n_new else 0.0), flush=True)
    df = pd.DataFrame(rows_all)
    R8_C["rows"] = df
    if not len(df):
        R8_C["state"] = "not available: there is no eligible hospital"
        status("supplementary C retrieval network", "not available", R8_C["state"])
        return
    summ = r8_c_summary(df)
    R8_C["summary"] = summ
    s1 = summ[np.isclose(summ["factor"], 1.0)]
    wide = s1[s1["alpha"].apply(lambda a: any(np.isclose(a, v) for v in R8_C_SHOW_ALPHA))].pivot_table(index=["mode", "alpha"], columns=["fs", "outcome"], values="d_auroc_median").reset_index()
    wide.columns = ["_".join(str(x) for x in c if x != "") if isinstance(c, tuple) else c for c in wide.columns]
    emit_table("supplementary C: hospital median AUROC change per mixing weight (partner prevalence factor 1 . only some weights on screen . all in R8_C_network_summary.csv)", wide, digits=4, max_rows=30)
    rows_r = []
    o1 = s1[s1["mode"] == "orig"]
    for al, v in R8_REF_C.items():
        for j, (fs, oc) in enumerate(R8_C_FS_COLS):
            g = o1[(o1["fs"] == fs) & (o1["outcome"] == oc) & np.isclose(o1["alpha"], al)]
            if not len(g):
                continue
            g = g.iloc[0]
            ok_d, ok_p = abs(g.d_auroc_median - v[2 * j]) <= R8_TOL["dmed"], abs(g.pct_improved - v[2 * j + 1]) <= R8_TOL["pct"]
            rows_r.append({"panel": "C", "outcome": oc, "item": "%s weight %.2f" % (fs, al), "orig_delta_median": g.d_auroc_median, "ref_delta_median": v[2 * j], "orig_pct": g.pct_improved, "ref_pct": v[2 * j + 1], "orig_slope": np.nan, "ref_slope": np.nan,
                           "ok_delta": bool(ok_d), "ok_pct": bool(ok_p), "ok_slope": True, "repro_ok": bool(ok_d and ok_p)})
    rep = pd.DataFrame(rows_r)
    R8_C["repro"] = rep
    emit_table("supplementary C reproduction check: values from the original split method (network_unit of the preceding cells) versus the table values of Supplementary S11 Panel C (tolerance: median %g , improved hospitals %g percentage points)" % (R8_TOL["dmed"], R8_TOL["pct"]),
               rep[["outcome", "item", "orig_delta_median", "ref_delta_median", "orig_pct", "ref_pct", "ok_delta", "ok_pct", "repro_ok"]] if len(rep) else rep, digits=4, max_rows=60)
    n_bad = int((~rep["repro_ok"]).sum()) if len(rep) else 0
    emit("- C reproduction check: of %d cells ok %d , mismatch %d (cell = feature set x outcome x weight)." % (len(rep), int(rep["repro_ok"].sum()) if len(rep) else 0, n_bad))
    pv = summ[np.isclose(summ["alpha"], NET_ALPHA_FIXED) & (summ["factor"] != 1.0)][["mode", "fs", "outcome", "factor", "n_hospitals", "d_auroc_median", "d_ece_median", "pct_ece_lower", "d_nb_rep_median"]].reset_index(drop=True)
    R8_C["prev"] = pv
    emit_table("supplementary C: per partner prevalence multiplier (weight %.2f , hospital median versus frozen)" % NET_ALPHA_FIXED, pv, digits=4, max_rows=30)
    cr = []
    for mode in ("orig", "cv3"):
        for fs, oc in R8_C_FS_COLS:
            g = df[(df["mode"] == mode) & (df["fs"] == fs) & (df["outcome"] == oc) & np.isclose(df["factor"], 1.0) & np.isclose(df["alpha"], NET_ALPHA_FIXED)]
            if len(g) > 3:
                r1, p1 = r8_stats.spearmanr(g["prev_diff"], g["intercept"])
                r2, p2 = r8_stats.spearmanr(g["prev_diff"], g["slope"])
                cr.append({"mode": mode, "fs": fs, "outcome": oc, "n_hospitals": int(len(g)), "rho_intercept": float(r1), "p_intercept": float(p1), "rho_slope": float(r2), "p_slope": float(p2)})
    cr = pd.DataFrame(cr)
    R8_C["corr"] = cr
    emit_table("supplementary C: rank correlation between the prevalence difference and the calibration intercept and slope (weight %.2f . P is for reference: independence between hospitals was not checked)" % NET_ALPHA_FIXED, cr, digits=3, max_rows=20)
    r8_save_csv(summ, "R8_C_network_summary")
    r8_save_csv(pv, "R8_C_prevalence_scenarios")
    r8_save_csv(cr, "R8_C_prevalence_correlation")
    R8_C["state"] = "done" if n_bad == 0 else "reproduction partly mismatched (%d cells)" % n_bad
    status("supplementary C retrieval network", "done" if n_bad == 0 else "partial", "units %d x two methods (computed this time %d)" % (len(order), n_new))


if not R6_STOP:
    _rep8 = len(REPORT_LINES)
    if not R8_RUN_C:
        R8_C["state"] = "off"
    else:
        r8_run_c()
    R8_LINES["c"] = [str(_l) for _l in REPORT_LINES[_rep8:]]
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)

In [ ]:
# ================================================================
# PRAM supplementary cell 12/13 . one-to-one table of Supplementary S11 Panel A and B rows and the mode list | summary of the reproduction checks | input values of the readings
# ================================================================
R8_MAP = pd.DataFrame([
    {"panel": "A", "S11 row": "Own bank", "mode name": "FR0_local", "scheme of the R8 csv": "FR0_local", "what the partner returns": "nothing"},
    {"panel": "A", "S11 row": "Exact scheme (merge of the nearest neighbours)", "mode name": "FR1_fed_exact", "scheme of the R8 csv": "FR1_fed_exact", "what the partner returns": "(distance, outcome) of the k nearest"},
    {"panel": "A", "S11 row": "Aggregate scheme (two sums)", "mode name": "FR2_fed_aggregate", "scheme of the R8 csv": "FR2_fed_aggregate", "what the partner returns": "two scalars"},
    {"panel": "A", "S11 row": "Aggregate scheme with noise, parameter 1", "mode name": "FR3_fed_agg_dp1", "scheme of the R8 csv": "FR3_fed_agg_dp1", "what the partner returns": "two scalars, noise parameter 1 (DP_EPS_FR = 1.0)"},
    {"panel": "A", "S11 row": "Aggregate scheme with noise, parameter 4", "mode name": "FR3_fed_agg_dp4", "scheme of the R8 csv": "FR3_fed_agg_dp4", "what the partner returns": "two scalars, noise parameter 4 (DP_EPS_FR = 4.0)"},
    {"panel": "A", "S11 row": "Prototype network", "mode name": "FR4_proto_network", "scheme of the R8 csv": "FR4_proto_network", "what the partner returns": "prototypes (100 per hospital)"},
    {"panel": "A", "S11 row": "Own bank and prototypes", "mode name": "FR5_local_plus_proto", "scheme of the R8 csv": "FR5_local_plus_proto", "what the partner returns": "prototypes"},
    {"panel": "A", "S11 row": "Pooled raw records", "mode name": "FR9_pooled_raw", "scheme of the R8 csv": "FR9_pooled_raw", "what the partner returns": "records"},
    {"panel": "A+", "S11 row": "(no row in S11: an additional output outside the FR list)", "mode name": "SWEEP_FR2_n01 to n16 (network size curve)", "scheme of the R8 csv": "R8_A_network_size_curve.csv", "what the partner returns": "two scalars"},
    {"panel": "B", "S11 row": "PRAM with the hospital's own bank", "mode name": "pram_frozen", "scheme of the R8 csv": "pram_frozen", "what the partner returns": "nothing"},
    {"panel": "B", "S11 row": "Federated averaging", "mode name": "fed_avg", "scheme of the R8 csv": "fed_avg", "what the partner returns": "model coefficients"},
    {"panel": "B", "S11 row": "Federated model stacking", "mode name": "fed_stack", "scheme of the R8 csv": "fed_stack", "what the partner returns": "models and predictions"},
    {"panel": "B", "S11 row": "Federated stacking with metadata", "mode name": "fed_stack_meta", "scheme of the R8 csv": "fed_stack_meta", "what the partner returns": "models and predictions"},
    {"panel": "B", "S11 row": "PRAM on the federated stack", "mode name": "pram_fedstack", "scheme of the R8 csv": "pram_fedstack", "what the partner returns": "models and predictions"},
    {"panel": "B", "S11 row": "Local logistic", "mode name": "local_lr", "scheme of the R8 csv": "local_lr", "what the partner returns": "nothing"},
    {"panel": "B", "S11 row": "Pooled, development data", "mode name": "pooled_dev", "scheme of the R8 csv": "pooled_dev", "what the partner returns": "records"},
    {"panel": "B", "S11 row": "Pooled, whole network", "mode name": "pooled_net", "scheme of the R8 csv": "pooled_net", "what the partner returns": "records"},
    {"panel": "B+", "S11 row": "(no row in S11: baseline)", "mode name": "frozen_lr", "scheme of the R8 csv": "frozen_lr", "what the partner returns": "the development model as it is . the reference of all changes (delta = 0)"},
    {"panel": "C", "S11 row": "Panel C mixing weight 12 rows | Panel D 6 weights | Panel E correlation | prevalence multipliers", "mode name": "_run_zero_label (as network_unit of the preceding cells)", "scheme of the R8 csv": "R8_C_network_summary.csv", "what the partner returns": "one probability"}])


def r8_wmean_table():
    """the same summary as the hospital level of the preceding cells: stay-weighted mean of the per-hospital changes and the hospital bootstrap 95% interval (the same replicate indices as in the preceding cells: r6_hosp_boot_plan). If the hospital unit rows equal the hospital set of the preceding cells, the random stream of the preceding cells is used as it is, otherwise an independent random stream from BOOT_SEED (written in the stream column).
    per mode of panel A | method of B | mixing weight of C (prevalence factor 1), per method (original split and three folds). Only values are written and they are not used in the reading rules (the summary of S11 is a median)."""
    streams, orders = r5b_hosp_boot_streams(), r6_hosp_orders()
    out = []

    def add(panel, mode, fs, oc, item, g, col):
        g = g[["hospitalid", "n_site", col]].dropna().drop_duplicates("hospitalid").reset_index(drop=True)
        if len(g) < 3:
            return
        g, idx, stream = r6_hosp_boot_plan(g, fs, oc, streams, orders)
        est, lo, hi = r6_wmean_boot(g[col].to_numpy(float), g["n_site"].to_numpy(float), idx)
        out.append({"panel": panel, "mode": mode, "fs": fs, "outcome": oc, "item": item, "k": int(len(g)), "stream": stream, "dAUROC_wmean": est, "dAUROC_lo": lo, "dAUROC_hi": hi})

    for panel, df_, col in (("A", R8_A["rows"], "scheme"), ("B", R8_B["rows"], "method")):
        if len(df_):
            for (mode, oc, it), g in df_.groupby(["mode", "outcome", col], sort=False):
                add(panel, mode, "F42", oc, str(it), g, "delta_auroc")
    dC = R8_C["rows"]
    if len(dC):
        d1 = dC[np.isclose(dC["factor"], 1.0)]
        for (mode, fs, oc, al), g in d1.groupby(["mode", "fs", "outcome", "alpha"], sort=False):
            add("C", mode, fs, oc, "weight %.2f" % al, g, "d_auroc")
    return pd.DataFrame(out, columns=["panel", "mode", "fs", "outcome", "item", "k", "stream", "dAUROC_wmean", "dAUROC_lo", "dAUROC_hi"])


def r8_split_effect():
    """split-frame effect: three-fold value minus original-split value in the same cell (hospital median change and improved hospital fraction). Panel A mode | B method | C (feature set, outcome, mixing weight; prevalence factor 1). The difference between two methods that differ in only one factor."""
    rows = []
    for nm, s_, col in (("A", R8_A["summary"], "scheme"), ("B", R8_B["summary"], "method")):
        if len(s_) and (s_["mode"] == "cv3").any():
            o, n = s_[s_["mode"] == "orig"].set_index(["outcome", col]), s_[s_["mode"] == "cv3"].set_index(["outcome", col])
            for k in o.index.intersection(n.index):
                rows.append({"panel": nm, "fs": "F42", "outcome": k[0], "item": str(k[1]), "orig_delta_median": float(o.loc[k, "delta_median"]), "cv3_delta_median": float(n.loc[k, "delta_median"]),
                             "orig_pct_improved": float(o.loc[k, "pct_improved"]), "cv3_pct_improved": float(n.loc[k, "pct_improved"])})
    sC = R8_C["summary"]
    if len(sC):
        c_ = sC[np.isclose(sC["factor"], 1.0)]
        o, n = c_[c_["mode"] == "orig"].set_index(["fs", "outcome", "alpha"]), c_[c_["mode"] == "cv3"].set_index(["fs", "outcome", "alpha"])
        for k in o.index.intersection(n.index):
            rows.append({"panel": "C", "fs": k[0], "outcome": k[1], "item": "weight %.2f" % k[2], "orig_delta_median": float(o.loc[k, "d_auroc_median"]), "cv3_delta_median": float(n.loc[k, "d_auroc_median"]),
                         "orig_pct_improved": float(o.loc[k, "pct_improved"]), "cv3_pct_improved": float(n.loc[k, "pct_improved"])})
    df = pd.DataFrame(rows, columns=["panel", "fs", "outcome", "item", "orig_delta_median", "cv3_delta_median", "orig_pct_improved", "cv3_pct_improved"])
    df["diff_delta_median"] = df["cv3_delta_median"] - df["orig_delta_median"]
    df["diff_pct_improved"] = df["cv3_pct_improved"] - df["orig_pct_improved"]
    df["over_frame"] = df["diff_delta_median"].abs() > R8_RULE_FRAME

    def _ok(r):                                                # does the original-split value of this row match the reference values and the S11 table values (reproduction table row). False if there is no reference row
        if r.panel == "A":
            return r8_rep_ok(R8_A["repro"], r.outcome, r.item)
        if r.panel == "B":
            return r8_rep_ok(R8_B["repro"], r.outcome, r.item)
        return r8_rep_ok(R8_C["repro"], r.outcome, "%s %s" % (r.fs, r.item))
    df["repro_ok"] = [bool(_ok(r)) for r in df.itertuples()]
    return df


R8_NOREF_P = "reproduction check not possible (there is no reference value for the hospital-level sign test P): only the P value is reported"
R8_NOREF_E = "reproduction check not possible (there is no reference value for the prevalence 2x and 0.5x scenarios): only values are reported"
R8_NOT_READ = "not read because there are rows without a reproduction check"


def r8_q(d):
    """the difference compared with the threshold is a value rounded at the sixth decimal place (prevents a difference equal to the threshold from splitting due to floating point. The threshold numbers are not changed)."""
    return round(float(d), 6)


def r8_rep_ok(rep, oc, item):
    """True if a (outcome, item) row exists in the reproduction table and matches. False if there is no row (reproduction check not possible)."""
    g = rep[(rep["outcome"] == oc) & (rep["item"] == item)] if len(rep) else rep
    return bool(g["repro_ok"].iloc[0]) if len(g) else False


def r8_zone_r83(diffs, margin=None):
    """rule N8-3: list of per-cell (federated mean and stacking - PRAM own bank) differences. Cells higher by the threshold or more u, lower by the threshold or more r, others z are counted. The zones do not overlap and have no gap (u + r + z = number of cells)."""
    m = R8_RULE_MARGIN if margin is None else margin
    q = [r8_q(d) for d in diffs]
    n = len(q)
    u, r = sum(1 for d in q if d >= m - 1e-12), sum(1 for d in q if d <= -m + 1e-12)
    z = n - u - r
    if n == 0:
        return "not computed (no cells)"
    if u == n:
        return "higher in all cells (federated mean and stacking are both higher than PRAM own bank by %.3f or more in both outcomes)" % m
    if r == n:
        return "lower in all cells (PRAM own bank is higher by %.3f or more in all %d cells)" % (n, m)
    if z == n:
        return "no difference (the absolute value of all differences is below %.3f)" % m
    if u >= 1:
        return "partly higher (higher cells %d , lower cells %d , other cells %d)" % (u, r, z)
    return "partly lower (PRAM own bank is higher by %.3f or more in only some cells and there is no higher cell. Lower cells %d , other cells %d)" % (m, r, z)


def r8_zone_r84(meds, gate_ok):
    """rule N8-4: median change in the four cells (feature set x outcome) at the fixed weight (NET_ALPHA_FIXED). meds is {(fs, outcome): value}."""
    if not gate_ok:
        return R8_NOT_READ
    q = {k: r8_q(v) for k, v in meds.items()}
    pos = [k for k, v in q.items() if v > 0]
    if len(q) == 4 and len(pos) == 4:
        return "all four cells positive (only the values are replaced by the three-fold values)"
    if len(q) == 4 and not pos:
        return "not all four cells positive"
    return "there are cells that are not positive: " + " · ".join("%s %s" % (k[0], k[1][:3]) for k, v in q.items() if v <= 0)


def r8_rule_inputs():
    """collects the input values of the pre-specified rules (the numbers to substitute the rules with three-fold values) into one table. Only the position name is given.
    every rule reads the three-fold values only when the reproduction check of the rows it uses (repro_ok of the corresponding reproduction table row) matches. For the two that have no reference value (P of N8-5 and N8-6), only values are reported."""
    rows = []
    sA, rA = R8_A["summary"], R8_A["repro"]

    def a_val(oc, sc):
        g = sA[(sA["mode"] == "cv3") & (sA["outcome"] == oc) & (sA["scheme"] == sc)]
        return float(g["delta_median"].iloc[0]) if len(g) else np.nan

    z1, v1 = "skipped (not computed)", ""
    if len(sA):
        diffs = {oc: a_val(oc, "FR2_fed_aggregate") - a_val(oc, "FR1_fed_exact") for oc in OUTCOMES}
        okr = all(r8_rep_ok(rA, oc, s_) for oc in OUTCOMES for s_ in ("FR2_fed_aggregate", "FR1_fed_exact", R8_ANCHOR_ITEM))
        pat = bool(r8_q(diffs.get("AKI_168h", np.nan)) >= R8_RULE_MARGIN - 1e-12 and r8_q(diffs.get("mortality_168h", np.nan)) <= -R8_RULE_MARGIN + 1e-12)
        z1 = ("order kept" if pat else "order reversed or unclear") if okr else R8_NOT_READ
        v1 = " ; ".join("FR2-FR1 %s=%+.4f" % (oc[:3], diffs[oc]) for oc in OUTCOMES)
    rows.append({"rule": "N8-1 order of aggregate versus exact method (AKI aggregate > exact , death aggregate < exact , difference %.3f or more . including the anchor row)" % R8_RULE_MARGIN, "input values": v1, "zone": z1})
    z2, v2 = "skipped (not computed)", ""
    st = R8_A["struct"]
    if len(st):
        g = st[st["mode"] == "cv3"]
        fs_, ag_ = float(g["exact_flip_share"].max()), float(g["exact_auc_gap_max"].max())
        okr2 = all(r8_rep_ok(rA, oc, R8_STRUCT_ITEM) for oc in OUTCOMES)
        if not okr2:
            z2 = R8_NOT_READ
        else:
            z2 = "exact method = pooled bank kept" if (np.isfinite(fs_) and np.isfinite(ag_) and fs_ <= R8_TOL["exact_flip_share"] and ag_ <= R8_TOL["exact_auc_gap"]) else "the difference between the exact method and the pooled bank needs checking (implementation check)"
        v2 = "maximum probability difference %.3e ; fraction of queries whose difference exceeds %g %.5f ; maximum AUROC difference %.2e ; median mean absolute difference of the aggregate %s ; median correlation %s" % (float(g["exact_err_max"].max()), R8_TOL["exact_err"], fs_, ag_, " / ".join("%.4f" % v for v in g["agg_err_median"]), " / ".join("%.3f" % v for v in g["agg_corr_median"]))
    rows.append({"rule": "N8-2 (S11 A note) structural property", "input values": v2, "zone": z2})
    z3, v3 = "not computed (no three-fold B)", ""
    sB, rB = R8_B["summary"], R8_B["repro"]
    if len(sB) and (sB["mode"] == "cv3").any():
        diff_by_outcome = {}
        for oc in OUTCOMES:
            g = sB[(sB["mode"] == "cv3") & (sB["outcome"] == oc)].set_index("method")["delta_median"]
            diff_by_outcome[oc] = [(m_, float(g[m_] - g["pram_frozen"])) for m_ in ("fed_avg", "fed_stack")]
        okr = all(r8_rep_ok(rB, oc, m_) for oc in OUTCOMES for m_ in ("fed_avg", "fed_stack", "pram_frozen"))
        z3 = R8_NOT_READ if not okr else r8_zone_r83([d for oc in OUTCOMES for _m, d in diff_by_outcome[oc]])
        v3 = " ; ".join("%s %s=%+.4f" % (oc[:3], m_, d) for oc in OUTCOMES for m_, d in diff_by_outcome[oc])
    rows.append({"rule": "N8-3 federated mean and stacking - PRAM own bank (difference %.3f or more)" % R8_RULE_MARGIN, "input values": v3, "zone": z3})
    sC, rC = R8_C["summary"], R8_C["repro"]
    z4, v4, z5, v5, zp, vp, z5b, v5b = "skipped (not computed)", "", "skipped (not computed)", "", "skipped (not computed)", "", "skipped (not computed)", ""

    def c_ok(fs, oc, al):
        return r8_rep_ok(rC, oc, "%s weight %.2f" % (fs, al))

    if len(sC):
        c = sC[(sC["mode"] == "cv3") & np.isclose(sC["factor"], 1.0)]
        a30 = c[np.isclose(c["alpha"], NET_ALPHA_FIXED)]
        meds = {(r_.fs, r_.outcome): r_.d_auroc_median for r_ in a30.itertuples()}
        gate4 = len(meds) == 4 and all(c_ok(fs, oc, NET_ALPHA_FIXED) for (fs, oc) in meds)
        z4 = r8_zone_r84(meds, gate4)
        v4 = " ; ".join("%s %s=%+.4f" % (k[0], k[1][:3], v) for k, v in meds.items())
        rng_ = c[(c["alpha"] >= 0.05 - 1e-9) & (c["alpha"] <= 0.50 + 1e-9)]
        nneg = int((rng_["d_auroc_median"].map(r8_q) <= 0).sum())
        pmax = float(rng_["sign_test_p"].max()) if len(rng_) else np.nan
        gate5 = len(rng_) > 0 and all(c_ok(r_.fs, r_.outcome, r_.alpha) for r_ in rng_.itertuples())
        if not gate5:
            z5 = R8_NOT_READ
        else:
            z5 = ("all %d cells of 0.05 to 0.50 positive" % len(rng_)) if nneg == 0 else ("%d cells not positive (list of cells)" % nneg)
        v5 = "cells not positive %d / %d" % (nneg, len(rng_))
        zp, vp = R8_NOREF_P, (("maximum sign test P %.3f" % pmax) if np.isfinite(pmax) else "no sign test P")
        c1 = c[np.isclose(c["alpha"], 1.0) & (c["fs"] == "F33")]
        gate5b = len(c1) == 2 and all(c_ok(r_.fs, r_.outcome, 1.0) for r_ in c1.itertuples())
        z5b = ("outcomes negative at 33 columns 1.00: " + (" · ".join(r_.outcome[:3] for r_ in c1.itertuples() if r8_q(r_.d_auroc_median) < 0) or "absent")) if gate5b else R8_NOT_READ
        v5b = " ; ".join("%s=%+.4f" % (r_.outcome[:3], r_.d_auroc_median) for r_ in c1.itertuples())
    rows.append({"rule": "N8-4 (one row of Table 6) gain of unlabeled hospitals at weight %.2f" % NET_ALPHA_FIXED, "input values": v4, "zone": z4})
    rows.append({"rule": "N8-5 median positive at weights 0.05 to 0.50 (40 cells whose reproduction was confirmed)", "input values": v5, "zone": z5})
    rows.append({"rule": "N8-5 sign test P (no reference value)", "input values": vp, "zone": zp})
    rows.append({"rule": "N8-5b negative median at 33-column weight 1.00", "input values": v5b, "zone": z5b})
    z6, v6 = "skipped (not computed)", ""
    pv = R8_C["prev"]
    if len(pv):
        p3 = pv[pv["mode"] == "cv3"]
        dbl, hlf = p3[np.isclose(p3["factor"], 2.0)], p3[np.isclose(p3["factor"], 0.5)]
        z6 = R8_NOREF_E
        v6 = "2x: ECE median positive %d/%d , 0.5x: ECE median negative %d/%d , AUROC median positive %d/%d (reference counts 3/4 , 4/4 , 8/8) ; 2x delta ECE median %s ; 0.5x delta ECE median %s" % (
            int((dbl["d_ece_median"] > 0).sum()), len(dbl), int((hlf["d_ece_median"] < 0).sum()), len(hlf), int((p3["d_auroc_median"] > 0).sum()), len(p3),
            " / ".join("%+.4f" % v for v in dbl["d_ece_median"]), " / ".join("%+.4f" % v for v in hlf["d_ece_median"]))
    rows.append({"rule": "N8-6 partner prevalence 2x and 0.5x", "input values": v6, "zone": z6})
    se = r8_split_effect()
    zz, vv = "skipped (not computed)", ""
    if len(se):
        se_ok = se[se["repro_ok"]]
        mx = [(nm, float(g["diff_delta_median"].abs().max()), int(len(g)), int(g["over_frame"].sum())) for nm, g in se_ok.groupby("panel", sort=True)]
        vv = " ; ".join("%s maximum %.4f (of the %d rows whose reproduction was confirmed, %d rows exceed)" % t for t in mx) + " ; the %d rows whose reproduction was not confirmed are not read" % int((~se["repro_ok"]).sum())
        if not mx:
            zz = R8_NOT_READ + "(there is no row whose reproduction was confirmed)"
        else:
            zz = ("all rows whose reproduction was confirmed are %.4f or below" % R8_RULE_FRAME) if all(t[3] == 0 for t in mx) else ("%d rows exceed %.4f (they are in the table R8_split_frame_effect.csv)" % (R8_RULE_FRAME, sum(t[3] for t in mx)))
    rows.append({"rule": "N8-7 split-frame effect (three-fold value - original-split value, maximum over the rows whose reproduction was confirmed of the absolute hospital median change . criterion %.4f)" % R8_RULE_FRAME, "input values": vv, "zone": zz})
    return pd.DataFrame(rows)


def r8_run_summary():
    states = {"A": R8_A["state"], "B": R8_B["state"], "C": R8_C["state"]}
    if not any(v and v != "off" and not v.startswith(("not computed because the input is missing", "not available")) for v in states.values()):
        emit("- supplementary summary: no cell was computed, so no summary table is made (A %s , B %s , C %s)" % (states["A"] or "not run", states["B"] or "not run", states["C"] or "not run"))
        status("supplementary summary", "not available", "no cell computed")
        return
    emit_table("one-to-one of the rows of Supplementary S11 Panel A and B and the mode list (A 8 rows = 8 FR list entries . B 8 rows = 8 federated learning methods + baseline frozen_lr)", R8_MAP, digits=0, max_rows=30)
    n_a, n_b = int((R8_MAP["panel"] == "A").sum()), int((R8_MAP["panel"] == "B").sum())
    ok_map = bool(n_a == len(R8_A_SCHEMES) == 8 and n_b == len(R8_B_METHODS) - 1 == 8 and set(R8_MAP[R8_MAP["panel"] == "A"]["mode name"]) == set(R8_A_SCHEMES)
                  and set(R8_MAP[R8_MAP["panel"].isin(["B", "B+"])]["mode name"]) == set(R8_B_METHODS))
    emit("- one-to-one check: Panel A rows %d = FR list %d , Panel B rows %d = M30 list %d minus one baseline . Are the name sets also equal: %s" % (n_a, len(R8_A_SCHEMES), n_b, len(R8_B_METHODS), "ok" if ok_map else "mismatch"))
    r8_save_csv(R8_MAP, "R8_mapping_S11_vs_source")
    reps = [x["repro"] for x in (R8_A, R8_B, R8_C) if len(x["repro"])]
    if reps:
        rep = pd.concat(reps, ignore_index=True)
        tab = rep.groupby("panel").agg(rows=("repro_ok", "size"), ok=("repro_ok", "sum")).reset_index()
        tab["mismatch"] = tab["rows"] - tab["ok"]
        emit_table("summary of the reproduction checks (original-split values versus reference values and S11 table values, per row . the three-fold values of mismatched rows are not read)", tab, digits=0)
        r8_save_csv(rep, "R8_repro_check")
        bad = rep[~rep["repro_ok"]][["panel", "outcome", "item", "orig_delta_median", "ref_delta_median", "orig_pct", "ref_pct"]]
        if len(bad):
            emit_table("mismatched reproduction rows (rows whose three-fold values are not read)", bad, digits=4, max_rows=60)
    wm = r8_wmean_table()
    if len(wm):
        emit_table("stay-weighted mean change and hospital bootstrap %d repetitions 95%% interval (the same summary as the hospital level of the preceding cells . the same replicate indices as in the preceding cells . all of panels A and B . C in csv . only values are written and not used in the reading rules)" % N_BOOT, wm[wm["panel"].isin(["A", "B"])].head(40), digits=4, max_rows=40)
        emit("- stay-weighted mean table: %d rows , replicate index method %s" % (len(wm), wm["stream"].value_counts().to_dict()))
        r8_save_csv(wm, "R8_hospital_weighted_means")
    se = r8_split_effect()
    if len(se):
        emit_table("split-frame effect: three-fold value - original-split value (hospital median AUROC change and improved hospital fraction . only rows exceeding the criterion %.4f on screen, all in R8_split_frame_effect.csv)" % R8_RULE_FRAME, se[se["over_frame"]].head(40), digits=4, max_rows=40)
        emit("- split-frame effect: of %d rows %d with reproduction confirmed , %d rows exceed the criterion (of which %d with reproduction confirmed) , maximum absolute value %.4f . (This table contains together the effect of test patients 30%% versus 100%% and the effect of bank and training size)" % (len(se), int(se["repro_ok"].sum()), int(se["over_frame"].sum()), int((se["over_frame"] & se["repro_ok"]).sum()), float(se["diff_delta_median"].abs().max())))
        r8_save_csv(se, "R8_split_frame_effect")
    ri = r8_rule_inputs()
    emit_table("input values of the pre-specified rules of the supplementary analyses and position names (a position name says where the values fall)", ri, digits=4)
    r8_save_csv(ri, "R8_rule_inputs")
    emit("- [note] (1) A difference of the three-fold value from the original-split value is not an effect of the split frame alone: it also contains the effect of the test patients increasing from 30% to 100% reducing the noise of the per-hospital AUROC (A: own bank changes from 70% to 67% , B: training from 60% to 50% . weighting from 20% to 17%, so the effect of bank and training size is also contained) . (2) The improved hospital fraction moves mechanically because hospitals whose change is exactly 0 (mixing weight 0 chosen) are not counted as improved . "
         "(3) That the original-split values are reproduced is evidence that the transfer equals the original calculation and not evidence that the three-fold values are right . (4) Rule N8-7 reports only the maximum of the differences.")
    status("supplementary summary", "done", "one-to-one table | reproduction check | rule input values")


if not R6_STOP:
    _rep8 = len(REPORT_LINES)
    r8_run_summary()
    R8_LINES["s"] = [str(_l) for _l in REPORT_LINES[_rep8:]]
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)

In [ ]:
# ================================================================
# PRAM supplementary cell 13/13 (last) . hash comparison of the stored files | status | summary md | zip
# ================================================================
R6_FINAL = {"changed": None, "lost": None, "added": None, "zip_ok": False, "moved": []}
if not R6_STOP:
    _rep6 = len(REPORT_LINES)
    _post = r6_snapshot()
    R6_FINAL["changed"] = sorted(k for k in R6_STATE["snap0"] if k in _post and _post[k] != R6_STATE["snap0"][k])
    _gone = sorted(k for k in R6_STATE["snap0"] if k not in _post)
    R6_FINAL["moved"] = [k for k in _gone if r6_is_root_zip(k)]
    R6_FINAL["lost"] = [k for k in _gone if not r6_is_root_zip(k)]
    R6_FINAL["added"] = sorted(k for k in _post if k not in R6_STATE["snap0"])
    emit("- hash comparison of the stored files: at the start %d , changed (hash mismatch) **%d** %s , disappeared (excluding the zip moved out of the folder) **%d** %s , newly created files outside the stored names %d %s"
         % (len(R6_STATE["snap0"]), len(R6_FINAL["changed"]), R6_FINAL["changed"] or "", len(R6_FINAL["lost"]), R6_FINAL["lost"] or "", len(R6_FINAL["added"]), R6_FINAL["added"] or ""))
    if R6_FINAL["moved"]:
        emit("- zip moved out of the folder (normal): %s" % R6_FINAL["moved"])
    _cnt = pd.DataFrame(R6_LOG, columns=["stage (resume name)", "mode"]).groupby(["stage (resume name)", "mode"]).size().unstack(fill_value=0) if R6_LOG else pd.DataFrame()
    if len(_cnt):
        emit_table("unit counts of this run (loaded | newly computed per resume name)", _cnt.reset_index(), digits=0, max_rows=40)
    _n_re = int(sum(1 for _s, _w in R6_LOG if _w == "reused"))
    _n_co = int(sum(1 for _s, _w in R6_LOG if _w == "computed"))
    emit("- units of this run: loaded %d , newly computed %d (unit = one hospital | one fold of the full bank | one bootstrap row | one raw table)" % (_n_re, _n_co))
    _u = lambda pre_, w_: int(sum(1 for _s, _w in R6_LOG if str(_s).startswith(pre_) and _w == w_))
    emit("- **thread condition marker**: of the units of this version, loaded %d and newly computed %d, and the newly computed units were computed under N_THREADS=%d of this run (0 = no limit) (the number of threads is in the fingerprint of the stored results, so stored results computed under another N_THREADS are not loaded). "
         "**The supplementary-analysis units are loaded %d and newly computed %d. The loaded values were computed under a BLAS thread limit of %d (the number of threads is not part of the supplementary unit fingerprint: R8_FP_THR=%d , CODE_VER_FED=%s). If there are supplementary units newly computed in this run, they were computed under N_THREADS=%d of this run and the fingerprint string is the same.**" % (
             _u("R7_", "reused"), _u("R7_", "computed"), int(N_THREADS), _u("R8_", "reused"), _u("R8_", "computed"), int(R8_FP_THR), int(R8_FP_THR), CODE_VER_FED, int(N_THREADS)))
    emit("- reproduction judgment: hospital level **%s** , full bank **%s** (aggregate file R7_repro_diff_summary.csv)" % (R7_N1["repro"] or "no comparison", R7_N1W["repro"] or "no comparison"))
    _states = {"N1 hospital level": R6_N1["state"], "N1 full bank": R6_N1["w_state"], "N2": R6_N2["state"], "N5": R6_N5["state"], "additional: arms hospital level": R7_N1["state"], "additional: arms full bank": R7_N1W["state"], "additional: single-bank alpha": R7_SB["state"], "supplementary A federated retrieval": R8_A["state"], "supplementary B federated learning": R8_B["state"], "supplementary C retrieval network": R8_C["state"]}
    emit_table("state per supplementary cell", pd.DataFrame([{"cell": k, "status": v or "not run"} for k, v in _states.items()]))
    _ok = all(v in ("done", "off") for v in _states.values()) and not R6_FINAL["changed"] and not R6_FINAL["lost"] and not R6_FINAL["added"]
    _dones = [k for k, v in _states.items() if v == "done"]
    _skips = {k: v for k, v in _states.items() if v not in ("done", "off")}
    _why = ([] if not R6_FINAL["changed"] else ["the hashes of %d stored files changed" % len(R6_FINAL["changed"])]) + ([] if not R6_FINAL["lost"] else ["%d stored files disappeared (excluding the zip moved out of the folder)" % len(R6_FINAL["lost"])]) \
        + ([] if not R6_FINAL["added"] else ["%d files were created outside the stored names" % len(R6_FINAL["added"])])
    _todo = "What to do: in the per-cell state table above, for a cell that is not computed because the input is missing follow the one line of what to do for that cell, and for a reproduction mismatch save the notebook file (.ipynb) holding this screen."
    status("supplementary cells", "done" if _ok else "partial", ("cell %s done | stored file changes 0" % " · ".join(_dones)) if _ok else
           " . ".join(x for x in ("done " + (" · ".join(_dones) or "absent"), ("cells not available " + " · ".join("%s(%s)" % (k, v or "not run") for k, v in _skips.items())) if _skips else "", " , ".join(_why)) if x))
    emit("- status: **%s** , %s , elapsed time %.1f min" % (STATUS["supplementary cells"][0], STATUS["supplementary cells"][1], (time.time() - R6_STATE["t0"]) / 60.0))
    try:
        import psutil
        emit("- peak working set memory of this kernel (including the preceding cells) %.2f GiB (psutil peak_wset , a measurement to decide the memory settings of the next run)" % (psutil.Process().memory_info().peak_wset / 2 ** 30))
    except Exception as _exc:  # noqa: BLE001
        emit("- could not measure the peak memory (psutil: %s). It is unrelated to the computed results." % type(_exc).__name__)
    if not _ok:
        emit("- " + _todo)
    if _dones and not (_why):
        _md = ["# PRAM supplementary computations (run %s)" % time.strftime("%Y-%m-%d %H:%M"), "",
               "**The numbers in this file are notebook run results.**", "",
               "- status: %s , %s" % STATUS["supplementary cells"], ""] + R6_LINES["1"] + R6_LINES["n1"] + R6_LINES["n2"] + R6_LINES["n56"] + R7_LINES["n1"] + R7_LINES["w"] + R7_LINES["sb"] + R8_LINES["a"] + R8_LINES["b"] + R8_LINES["c"] + R8_LINES["s"] + [str(_l) for _l in REPORT_LINES[_rep6:]]
        _md_path = R6_DIR / "R6_summary.md"
        _md_path.write_text("\n".join(_md), encoding="utf-8")
        _id_like = {"stay_id", "patientunitstayid", "subject_id", "hadm_id", "icustay_id", "hospitalid", "uniquepid", "patienthealthsystemstayid", "pt_idx", "patient_id"}
        _bad_id = [_p.name for _p in sorted(R6_CSV_DIR.glob("R[678]_*.csv")) if _id_like & {str(_c).lower() for _c in pd.read_csv(_p, nrows=2).columns}]
        if _bad_id:
            emit("**The zip was not made.** Files with identifier columns: %s" % _bad_id)
            status("supplementary zip", "not available", str(_bad_id))
        else:
            _stale = sorted([_p.name for _p in R6_CSV_DIR.glob("R7_N1_hospital_*.csv") if R7_N1["state"] != "done"] + [_p.name for _p in R6_CSV_DIR.glob("R7_N1_whole_*.csv") if R7_N1W["state"] != "done"])
            if _stale:
                emit("- the %d csv files with the R7_ prefix left from stored results were not put in the zip because the state of this run is not complete (hospital level %s , full bank %s): %s" % (len(_stale), R7_N1["state"] or "not run", R7_N1W["state"] or "not run", _stale))
            with zipfile.ZipFile(SUPP_ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as _z:
                for _p in [_q for _q in sorted(R6_CSV_DIR.glob("R[678]_*.csv")) if _q.name not in _stale] + [_md_path]:
                    _z.write(_p, arcname=_p.relative_to(OUT_DIR.parent).as_posix())
            with zipfile.ZipFile(SUPP_ZIP_NAME) as _z:
                _names = _z.namelist()
            R6_FINAL["zip_ok"] = True
            status("supplementary zip", "done", "%d files" % len(_names))
            emit("- zip `%s` (%.2f MB, %d files: %s)" % (Path(SUPP_ZIP_NAME).resolve(), Path(SUPP_ZIP_NAME).stat().st_size / 1e6, len(_names), _names))
            emit("**Keep this zip with the results.** There is only this one file (%s). Do not touch the stored zip and S8 zip already in the root (they are separate). Patient-level values (the test prediction npz of the full bank) and hospital ids were not put in the zip." % Path(SUPP_ZIP_NAME).name)
    else:
        emit("**The zip was not made.** There is no completed cell or the stored files changed (see the line above).")
else:
    print("the first cell ended without computing, so this cell is also skipped: " + R6_STOP)